In [ ]:
# Install deps (run once per fresh runtime)
!pip -q install -U transformers datasets accelerate bitsandbytes triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.3 MB/s eta 0:00:00


In [ ]:
import sys
!pip -q install -U transformers datasets accelerate bitsandbytes triton peft
!{sys.executable} -m pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 11.4 MB/s eta 0:00:00


In [ ]:
import os, math, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.utils.cpp_extension import load_inline
from transformers import TrainingArguments, Trainer, TrainerCallback

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", device)

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

device: cuda:0


In [ ]:
torch.autograd.set_detect_anomaly(True)

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [ ]:
from dataclasses import replace

def get_batch_seq_len(batch):
    if "attention_mask" in batch and batch["attention_mask"] is not None:
        return int(batch["attention_mask"].shape[-1])
    return int(batch["input_ids"].shape[-1])

def sanitize_protocol_for_seq_len(p, seq_len):
    max_off = max(int(seq_len) - 1, 0)

    w = min(int(p.w), max_off)
    num_anchors = min(int(p.num_anchors), int(seq_len))

    mode = int(p.scaffold_mode)
    ring_inner = int(p.ring_inner)
    ring_outer = int(p.ring_outer)

    if mode in (1, 2):
        ring_outer = min(ring_outer, max_off)
        ring_inner = min(ring_inner, ring_outer)

        if ring_outer < 1:
            ring_inner = 0
            ring_outer = 0
            mode = 0
    else:
        ring_inner = 0
        ring_outer = 0

    return replace(
        p,
        w=w,
        num_anchors=num_anchors,
        scaffold_mode=mode,
        ring_inner=ring_inner,
        ring_outer=ring_outer,
    )

def sanitize_protocol_table_for_seq_len(protocol_specs, seq_len):
    return [sanitize_protocol_for_seq_len(p, seq_len) for p in protocol_specs]

In [ ]:
from datasets import DatasetDict

# ----------------------------
# Robust "get text" for a *batch* (RAM-safe)
# ----------------------------
def _batch_extract_text(batch):
    """
    Returns list[str] named 'text' for tokenization.
    Works for common schemas without iterating whole dataset in Python.
    """
    # 1) If already has "text" and it's strings -> use it
    if "text" in batch and isinstance(batch["text"], list) and (len(batch["text"]) == 0 or isinstance(batch["text"][0], str)):
        return {"text": batch["text"]}

    # 2) Chat format: messages = list[list[dict]]
    if "messages" in batch:
        out = []
        for msgs in batch["messages"]:
            if isinstance(msgs, list):
                parts = []
                for m in msgs:
                    if isinstance(m, dict):
                        c = m.get("content", None)
                        if isinstance(c, str) and c.strip():
                            parts.append(c.strip())
                out.append("\n".join(parts) if parts else "")
            else:
                out.append("")
        return {"text": out}

    # 3) Common pair formats
    pair_candidates = [
        ("prompt", "completion"),
        ("instruction", "output"),
        ("question", "answer"),
        ("input", "output"),
    ]
    for a, b in pair_candidates:
        if a in batch or b in batch:
            A = batch.get(a, [""] * len(next(iter(batch.values()))))
            B = batch.get(b, [""] * len(A))
            out = []
            for aa, bb in zip(A, B):
                aa = aa if isinstance(aa, str) else ""
                bb = bb if isinstance(bb, str) else ""
                s = (aa + "\n" + bb).strip()
                out.append(s)
            return {"text": out}

    # 4) Fallback: pick the first column that looks like list[str]
    for k, v in batch.items():
        if isinstance(v, list) and (len(v) == 0 or isinstance(v[0], str)):
            return {"text": v}

    # If we get here, we don't know how to build text for this dataset
    # Return empty strings so tokenization doesn't crash; you should inspect schema.
    n = len(next(iter(batch.values()))) if len(batch) else 0
    return {"text": [""] * n}


def build_lm_dataset(ds, tokenizer, block_size, batch_size: int = 1000):
    """
    ds: HF DatasetDict or Dataset; schema can be text/chat/pairs.
    Returns tokenized, block-chunked LM dataset with labels=input_ids.
    RAM-safe: uses datasets.map(batched=True) like your original.
    """
    def tokenize_fn(examples):
        # Build a clean 'text' list[str] for this batch
        text_batch = _batch_extract_text(examples)["text"]
        # Drop empty strings to reduce junk (but keep alignment simple: replace empties with "")
        # Tokenizer requires list[str]
        return tokenizer(
            text_batch,
            return_attention_mask=False,
            truncation=False,
            add_special_tokens=False,
        )

    def group_texts(examples):
        # Same as your original: concatenate then chunk
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_len = len(concatenated["input_ids"])
        total_len = (total_len // block_size) * block_size
        result = {
            k: [t[i : i + block_size] for i in range(0, total_len, block_size)]
            for k, t in concatenated.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    if isinstance(ds, DatasetDict):
        out = {}
        for split in ds.keys():
            tokenized = ds[split].map(
                tokenize_fn,
                batched=True,
                batch_size=batch_size,
                remove_columns=ds[split].column_names,
                desc=f"Tokenizing {split}",
            )
            out[split] = tokenized.map(
                group_texts,
                batched=True,
                batch_size=batch_size,
                desc=f"Grouping {split}",
            )
        return out

    tokenized = ds.map(
        tokenize_fn,
        batched=True,
        batch_size=batch_size,
        remove_columns=ds.column_names,
        desc="Tokenizing",
    )
    return tokenized.map(
        group_texts,
        batched=True,
        batch_size=batch_size,
        desc="Grouping",
    )


In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def make_collate_fixedlen(tokenizer, seq_len: int):
    """
    Pads/truncates to EXACT seq_len.
    Masks padding in labels with -100 so loss/PPL ignore pads.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id  # standard fallback for tokenizers

    def collate(batch):
        input_ids = [x["input_ids"] for x in batch]

        # Directly pad to max_length=seq_len (no variable-length batches)
        out = tokenizer.pad(
            {"input_ids": input_ids},
            padding="max_length",
            max_length=seq_len,
            return_tensors="pt",
        )

        # Ensure attention_mask exists and is exactly seq_len
        if "attention_mask" not in out:
            out["attention_mask"] = (out["input_ids"] != pad_id).long()

        labels = out["input_ids"].clone()
        labels[out["attention_mask"] == 0] = -100
        out["labels"] = labels
        return out

    return collate

def make_loaders(lm_ds, tokenizer, train_bs, eval_bs, seq_len_train, seq_len_eval=None, num_workers=0):
    if seq_len_eval is None:
        seq_len_eval = seq_len_train

    collate_train = make_collate_fixedlen(tokenizer, seq_len_train)
    collate_eval  = make_collate_fixedlen(tokenizer, seq_len_eval)

    # Colab-safe: num_workers=0 avoids worker exits
    train_loader = DataLoader(
        lm_ds["train"],
        batch_size=train_bs,
        shuffle=True,
        drop_last=True,
        collate_fn=collate_train,
        num_workers=num_workers,
        pin_memory=(num_workers == 0),     # harmless either way
        persistent_workers=False,
    )

    val_loader = DataLoader(
        lm_ds["validation"],
        batch_size=eval_bs,
        shuffle=False,
        drop_last=True,
        collate_fn=collate_eval,
        num_workers=num_workers,
        pin_memory=(num_workers == 0),
        persistent_workers=False,
    )

    return train_loader, val_loader

In [ ]:
# ============================================================
# CONFIG CELL
# Put this right before the dataset/loader build cell
# ============================================================
SEQ_LEN = 2048      # choose the sequence length you actually want
TRAIN_BS = 4
EVAL_BS = 4
GRAD_ACCUM = 4
LR = 3e-4

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer:", TOKENIZER_NAME)
print("Vocab size:", len(tokenizer))
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer loaded: mistralai/Mistral-7B-v0.1
pad_token: </s> 2
eos_token: </s> 2


In [ ]:
# 1) Build LM dataset
lm_ds = build_lm_dataset(ds, tokenizer, block_size=SEQ_LEN)

# 2) Build loaders (single call)
train_loader, val_loader = make_loaders(
    lm_ds,
    tokenizer,
    train_bs=TRAIN_BS,
    eval_bs=EVAL_BS,
    seq_len_train=SEQ_LEN,
    seq_len_eval=SEQ_LEN,   # or different if you want
)


Tokenizing test:   0%|          | 0/4358 [00:00<?, ? examples/s]

Grouping test:   0%|          | 0/4358 [00:00<?, ? examples/s]

Tokenizing train:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Grouping train:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/3760 [00:00<?, ? examples/s]

Grouping validation:   0%|          | 0/3760 [00:00<?, ? examples/s]

**Stage A: CUDA candidate builder**

In [ ]:
# ============================================================
# DCSFC missing components:
#   bounded family pool -> learnable logits -> softmax
#   -> soft K/V family summaries -> hard top-k tables
#
# Reuse AFTER this:
#   your existing Stage-A family compiler that takes:
#     precomputed_window_idx/ok
#     precomputed_ring_idx/ok
#     precomputed_anchor_idx/ok
#
# This code is family-agnostic:
#   call it once for window pool
#   call it once for ring pool
#   call it once for anchor pool
#
# Inputs per call:
#   H_bnd        : [B,N,Dh] float32/float16/bfloat16 CUDA
#   deficit_bn   : [B,N]    float32/float16/bfloat16 CUDA
#   K_bnd        : [B,N,Dv] float32/float16/bfloat16 CUDA
#   V_bnd        : [B,N,Dv] float32/float16/bfloat16 CUDA
#   pool_idx_tc  : [T,Cpool] int32/int64 CUDA   (LOCAL token ids j in [0,N))
#   pool_ok_tc   : [T,Cpool] uint8/bool CUDA
#   cand_ctx_tc  : [T,Cpool] float32 CUDA       (candidate spectral/local/band-fit context; pass zeros if none)
#   router_w_f32 : [7] float32 CUDA:
#       [w_sim, w_di, w_dj, w_pos, w_blk, w_ctx, bias]
#
# Outputs:
#   logits_tc    : [T,Cpool] float32
#   probs_tc     : [T,Cpool] float32
#   softK_tdv    : [T,Dv] float32
#   softV_tdv    : [T,Dv] float32
#   hard_mask_tc : [T,Cpool] uint8
#   hard_idx_tk  : [T,topk] int32    (LOCAL token ids)
#   hard_ok_tk   : [T,topk] uint8
#
# Notes:
# - This is the "remaining router core" only.
# - It does NOT replace your existing compiler / family meta outputs.
# - It uses stable bounded-pool softmax, not entmax.
# - Straight-through bridge is added in the Python wrapper.
# ============================================================

import hashlib
import torch
from torch.utils.cpp_extension import load_inline

FUSED_VERBOSE_BUILD = False

CUDA_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <c10/cuda/CUDAGuard.h>
#include <vector>
#include <cstdint>
#include <cmath>
#include <limits>
#include <climits>

#define CHECK_CUDA(x) TORCH_CHECK((x).is_cuda(), #x " must be CUDA")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK((x).is_contiguous(), #x " must be contiguous")
#define CHECK_2D(x) TORCH_CHECK((x).dim() == 2, #x " must be 2D")
#define CHECK_3D(x) TORCH_CHECK((x).dim() == 3, #x " must be 3D")

static constexpr int MAX_POOL = 128;
static constexpr int MAX_TOPK = 64;

__device__ __forceinline__ float neg_inf_f() { return -1.0e30f; }

__device__ __forceinline__ void insert_topk_desc(
    int cand_c,
    float score,
    int* best_c,
    float* best_s,
    int K
) {
    int pos = -1;
    for (int t = 0; t < K; ++t) {
        if (score > best_s[t]) {
            pos = t;
            break;
        }
    }
    if (pos < 0) return;
    for (int t = K - 1; t > pos; --t) {
        best_c[t] = best_c[t - 1];
        best_s[t] = best_s[t - 1];
    }
    best_c[pos] = cand_c;
    best_s[pos] = score;
}

__global__ void dcsfc_router_forward_kernel(
    const float* __restrict__ H_t_d,        // [T,Dh]
    const float* __restrict__ deficit_bn,   // [B,N]
    const float* __restrict__ K_t_dv,       // [T,Dv]
    const float* __restrict__ V_t_dv,       // [T,Dv]

    const int32_t* __restrict__ pool_idx_tc,// [T,C]
    const uint8_t* __restrict__ pool_ok_tc, // [T,C]
    const float* __restrict__ cand_ctx_tc,  // [T,C]
    const float* __restrict__ router_w,     // [7]

    int B,
    int N,
    int Dh,
    int Dv,
    int C,
    float tau,
    int block_size,
    bool causal_only,

    float* __restrict__ logits_tc,          // [T,C]
    float* __restrict__ probs_tc,           // [T,C]
    float* __restrict__ softK_t_dv,         // [T,Dv]
    float* __restrict__ softV_t_dv          // [T,Dv]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;

    const float w_sim  = router_w[0];
    const float w_di   = router_w[1];
    const float w_dj   = router_w[2];
    const float w_pos  = router_w[3];
    const float w_blk  = router_w[4];
    const float w_ctx  = router_w[5];
    const float bias   = router_w[6];

    const float di = deficit_bn[b * N + i];
    const float inv_sqrt_Dh = rsqrtf(fmaxf((float)Dh, 1.0f));
    const int num_blocks = (N + block_size - 1) / block_size;
    const int blk_i = i / block_size;

    float zmax = neg_inf_f();
    bool any_valid = false;

    // pass 1: logits
    for (int c = 0; c < C; ++c) {
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        logits_tc[off] = neg_inf_f();
        probs_tc[off] = 0.0f;

        if (!pool_ok_tc[off]) continue;
        const int j = pool_idx_tc[off];
        if (j < 0 || j >= N) continue;
        if (j == i) continue;
        if (causal_only && j > i) continue;

        const int gj = b * N + j;
        const float dj = deficit_bn[b * N + j];

        float sim = 0.0f;
        const int64_t hi = (int64_t)t * (int64_t)Dh;
        const int64_t hj = (int64_t)gj * (int64_t)Dh;
        for (int d = 0; d < Dh; ++d) {
            sim += H_t_d[hi + d] * H_t_d[hj + d];
        }
        sim *= inv_sqrt_Dh;

        const float pos_norm = (N > 1) ? (float)abs(i - j) / (float)(N - 1) : 0.0f;
        const int blk_j = j / block_size;
        const float blk_norm = (num_blocks > 1) ? (float)abs(blk_i - blk_j) / (float)(num_blocks - 1) : 0.0f;
        const float ctx = cand_ctx_tc[off];

        const float raw =
            w_sim * sim
          + w_di  * di
          + w_dj  * dj
          + w_pos * pos_norm
          + w_blk * blk_norm
          + w_ctx * ctx
          + bias;

        const float z = raw / fmaxf(tau, 1.0e-6f);
        logits_tc[off] = z;
        zmax = fmaxf(zmax, z);
        any_valid = true;
    }

    // init summaries
    const int64_t out_base = (int64_t)t * (int64_t)Dv;
    for (int d = 0; d < Dv; ++d) {
        softK_t_dv[out_base + d] = 0.0f;
        softV_t_dv[out_base + d] = 0.0f;
    }
    if (!any_valid) return;

    // pass 2: exp/sum
    float denom = 0.0f;
    for (int c = 0; c < C; ++c) {
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        const float z = logits_tc[off];
        if (z <= -1.0e20f) continue;
        float e = expf(z - zmax);
        probs_tc[off] = e;
        denom += e;
    }
    denom = fmaxf(denom, 1.0e-12f);

    // pass 3: normalize + summaries
    for (int c = 0; c < C; ++c) {
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        if (!pool_ok_tc[off]) continue;
        const int j = pool_idx_tc[off];
        if (j < 0 || j >= N) {
            probs_tc[off] = 0.0f;
            continue;
        }
        if (j == i || (causal_only && j > i)) {
            probs_tc[off] = 0.0f;
            continue;
        }
        const float p = probs_tc[off] / denom;
        probs_tc[off] = p;

        const int gj = b * N + j;
        const int64_t kv_base = (int64_t)gj * (int64_t)Dv;
        for (int d = 0; d < Dv; ++d) {
            softK_t_dv[out_base + d] += p * K_t_dv[kv_base + d];
            softV_t_dv[out_base + d] += p * V_t_dv[kv_base + d];
        }
    }
}

__global__ void dcsfc_hard_topk_kernel(
    const int32_t* __restrict__ pool_idx_tc, // [T,C]
    const uint8_t* __restrict__ pool_ok_tc,  // [T,C]
    const float* __restrict__ probs_tc,      // [T,C]
    int B,
    int N,
    int C,
    int topk,
    bool causal_only,
    uint8_t* __restrict__ hard_mask_tc,      // [T,C]
    int32_t* __restrict__ hard_idx_tk,       // [T,K]
    uint8_t* __restrict__ hard_ok_tk         // [T,K]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;

    for (int c = 0; c < C; ++c) hard_mask_tc[(int64_t)t * C + c] = 0;
    for (int k = 0; k < topk; ++k) {
        hard_idx_tk[(int64_t)t * topk + k] = -1;
        hard_ok_tk[(int64_t)t * topk + k] = 0;
    }

    int best_c[MAX_TOPK];
    float best_s[MAX_TOPK];
    for (int k = 0; k < topk; ++k) {
        best_c[k] = -1;
        best_s[k] = neg_inf_f();
    }

    for (int c = 0; c < C; ++c) {
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        if (!pool_ok_tc[off]) continue;
        const int j = pool_idx_tc[off];
        if (j < 0 || j >= N) continue;
        if (j == i) continue;
        if (causal_only && j > i) continue;
        const float s = probs_tc[off];
        insert_topk_desc(c, s, best_c, best_s, topk);
    }

    for (int k = 0; k < topk; ++k) {
        const int c = best_c[k];
        if (c < 0) continue;
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        const int j = pool_idx_tc[off];
        hard_mask_tc[off] = 1;
        hard_idx_tk[(int64_t)t * topk + k] = j;
        hard_ok_tk[(int64_t)t * topk + k] = 1;
    }
}

__global__ void dcsfc_router_backward_kernel(
    const float* __restrict__ H_t_d,         // [T,Dh]
    const float* __restrict__ deficit_bn,    // [B,N]
    const float* __restrict__ K_t_dv,        // [T,Dv]
    const float* __restrict__ V_t_dv,        // [T,Dv]

    const int32_t* __restrict__ pool_idx_tc, // [T,C]
    const uint8_t* __restrict__ pool_ok_tc,  // [T,C]
    const float* __restrict__ cand_ctx_tc,   // [T,C]
    const float* __restrict__ probs_tc,      // [T,C]
    const float* __restrict__ router_w,      // [7]

    const float* __restrict__ grad_logits_tc,// [T,C]
    const float* __restrict__ grad_probs_tc, // [T,C]
    const float* __restrict__ grad_softK_tdv,// [T,Dv]
    const float* __restrict__ grad_softV_tdv,// [T,Dv]

    int B,
    int N,
    int Dh,
    int Dv,
    int C,
    float tau,
    int block_size,
    bool causal_only,

    float* __restrict__ grad_H_t_d,          // [T,Dh]
    float* __restrict__ grad_deficit_bn,     // [B,N]
    float* __restrict__ grad_K_t_dv,         // [T,Dv]
    float* __restrict__ grad_V_t_dv,         // [T,Dv]
    float* __restrict__ grad_cand_ctx_tc,    // [T,C]
    float* __restrict__ grad_router_w        // [7]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;

    const float w_sim  = router_w[0];
    const float w_di   = router_w[1];
    const float w_dj   = router_w[2];
    const float w_pos  = router_w[3];
    const float w_blk  = router_w[4];
    const float w_ctx  = router_w[5];

    const float di = deficit_bn[b * N + i];
    const float inv_sqrt_Dh = rsqrtf(fmaxf((float)Dh, 1.0f));
    const int num_blocks = (N + block_size - 1) / block_size;
    const int blk_i = i / block_size;

    float gp[MAX_POOL];
    float sum_pgp = 0.0f;

    // build grad wrt probs from:
    //   direct grad on probs output
    //   soft summary paths
    for (int c = 0; c < C; ++c) {
        gp[c] = 0.0f;
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        if (!pool_ok_tc[off]) continue;
        const int j = pool_idx_tc[off];
        if (j < 0 || j >= N) continue;
        if (j == i) continue;
        if (causal_only && j > i) continue;

        const int gj = b * N + j;
        const int64_t kv_base = (int64_t)gj * (int64_t)Dv;
        const int64_t out_base = (int64_t)t * (int64_t)Dv;

        float g = grad_probs_tc[off];
        for (int d = 0; d < Dv; ++d) {
            g += grad_softK_tdv[out_base + d] * K_t_dv[kv_base + d];
            g += grad_softV_tdv[out_base + d] * V_t_dv[kv_base + d];
        }
        gp[c] = g;
        sum_pgp += probs_tc[off] * g;
    }

    // backprop through summaries + logits
    for (int c = 0; c < C; ++c) {
        const int64_t off = (int64_t)t * (int64_t)C + (int64_t)c;
        if (!pool_ok_tc[off]) continue;
        const int j = pool_idx_tc[off];
        if (j < 0 || j >= N) continue;
        if (j == i) continue;
        if (causal_only && j > i) continue;

        const int gj = b * N + j;
        const int64_t hi = (int64_t)t * (int64_t)Dh;
        const int64_t hj = (int64_t)gj * (int64_t)Dh;
        const int64_t kv_base = (int64_t)gj * (int64_t)Dv;
        const int64_t out_base = (int64_t)t * (int64_t)Dv;

        const float p = probs_tc[off];

        // direct aggregation grads
        for (int d = 0; d < Dv; ++d) {
            atomicAdd(&grad_K_t_dv[kv_base + d], p * grad_softK_tdv[out_base + d]);
            atomicAdd(&grad_V_t_dv[kv_base + d], p * grad_softV_tdv[out_base + d]);
        }

        // softmax backward + direct logits grad
        const float inv_tau = 1.0f / fmaxf(tau, 1.0e-6f);
        const float gz_soft = p * (gp[c] - sum_pgp) * inv_tau;
        const float gz = gz_soft + grad_logits_tc[off] * inv_tau;

        // features
        float sim = 0.0f;
        for (int d = 0; d < Dh; ++d) {
            sim += H_t_d[hi + d] * H_t_d[hj + d];
        }
        sim *= inv_sqrt_Dh;

        const float dj = deficit_bn[b * N + j];
        const float pos_norm = (N > 1) ? (float)abs(i - j) / (float)(N - 1) : 0.0f;
        const int blk_j = j / block_size;
        const float blk_norm = (num_blocks > 1) ? (float)abs(blk_i - blk_j) / (float)(num_blocks - 1) : 0.0f;
        const float ctx = cand_ctx_tc[off];

        // grad router_w
        atomicAdd(&grad_router_w[0], gz * sim);
        atomicAdd(&grad_router_w[1], gz * di);
        atomicAdd(&grad_router_w[2], gz * dj);
        atomicAdd(&grad_router_w[3], gz * pos_norm);
        atomicAdd(&grad_router_w[4], gz * blk_norm);
        atomicAdd(&grad_router_w[5], gz * ctx);
        atomicAdd(&grad_router_w[6], gz);

        // grad cand_ctx
        atomicAdd(&grad_cand_ctx_tc[off], gz * w_ctx);

        // grad deficit
        atomicAdd(&grad_deficit_bn[b * N + i], gz * w_di);
        atomicAdd(&grad_deficit_bn[b * N + j], gz * w_dj);

        // grad H through sim
        const float gsim = gz * w_sim * inv_sqrt_Dh;
        for (int d = 0; d < Dh; ++d) {
            atomicAdd(&grad_H_t_d[hi + d], gsim * H_t_d[hj + d]);
            atomicAdd(&grad_H_t_d[hj + d], gsim * H_t_d[hi + d]);
        }
    }
}

std::vector<torch::Tensor> dcsfc_router_family_forward_cuda(
    torch::Tensor H_bnd,
    torch::Tensor deficit_bn,
    torch::Tensor K_bnd,
    torch::Tensor V_bnd,
    torch::Tensor pool_idx_tc,
    torch::Tensor pool_ok_tc,
    torch::Tensor cand_ctx_tc,
    torch::Tensor router_w_f32,
    double tau,
    int64_t topk,
    int64_t block_size,
    bool causal_only
) {
    CHECK_CUDA(H_bnd);
    CHECK_CUDA(deficit_bn);
    CHECK_CUDA(K_bnd);
    CHECK_CUDA(V_bnd);
    CHECK_CUDA(pool_idx_tc);
    CHECK_CUDA(pool_ok_tc);
    CHECK_CUDA(cand_ctx_tc);
    CHECK_CUDA(router_w_f32);

    c10::cuda::CUDAGuard device_guard(H_bnd.device());
    TORCH_CHECK(deficit_bn.device() == H_bnd.device(), "deficit_bn device mismatch");
    TORCH_CHECK(K_bnd.device() == H_bnd.device(), "K_bnd device mismatch");
    TORCH_CHECK(V_bnd.device() == H_bnd.device(), "V_bnd device mismatch");
    TORCH_CHECK(pool_idx_tc.device() == H_bnd.device(), "pool_idx_tc device mismatch");
    TORCH_CHECK(pool_ok_tc.device() == H_bnd.device(), "pool_ok_tc device mismatch");
    TORCH_CHECK(cand_ctx_tc.device() == H_bnd.device(), "cand_ctx_tc device mismatch");
    TORCH_CHECK(router_w_f32.device() == H_bnd.device(), "router_w_f32 device mismatch");

    CHECK_3D(H_bnd);
    CHECK_2D(deficit_bn);
    CHECK_3D(K_bnd);
    CHECK_3D(V_bnd);
    CHECK_2D(pool_idx_tc);
    CHECK_2D(pool_ok_tc);
    CHECK_2D(cand_ctx_tc);

    TORCH_CHECK(H_bnd.size(0) == deficit_bn.size(0) && H_bnd.size(1) == deficit_bn.size(1),
                "H_bnd and deficit_bn shape mismatch");
    TORCH_CHECK(K_bnd.size(0) == H_bnd.size(0) && K_bnd.size(1) == H_bnd.size(1),
                "K_bnd shape mismatch");
    TORCH_CHECK(V_bnd.size(0) == H_bnd.size(0) && V_bnd.size(1) == H_bnd.size(1),
                "V_bnd shape mismatch");
    TORCH_CHECK(V_bnd.size(2) == K_bnd.size(2),
                "V_bnd.size(2) must equal K_bnd.size(2)");

    const int64_t B64 = H_bnd.size(0);
    const int64_t N64 = H_bnd.size(1);
    const int64_t Dh64 = H_bnd.size(2);
    const int64_t Dv64 = K_bnd.size(2);
    TORCH_CHECK(B64 > 0 && N64 > 0 && Dh64 > 0 && Dv64 > 0,
                "B, N, Dh, Dv must be positive");
    TORCH_CHECK(N64 == 0 || B64 <= std::numeric_limits<int64_t>::max() / N64,
                "B*N overflows int64");
    const int64_t T64 = B64 * N64;
    TORCH_CHECK(B64 <= INT_MAX && N64 <= INT_MAX &&
                Dh64 <= INT_MAX && Dv64 <= INT_MAX &&
                T64 <= INT_MAX,
                "B/N/Dh/Dv/T exceed int32 kernel indexing limits");

    TORCH_CHECK(pool_idx_tc.size(0) == T64, "pool_idx_tc rows must equal T=B*N");
    TORCH_CHECK(pool_ok_tc.size(0) == T64,  "pool_ok_tc rows must equal T=B*N");
    TORCH_CHECK(cand_ctx_tc.size(0) == T64, "cand_ctx_tc rows must equal T=B*N");

    TORCH_CHECK(pool_idx_tc.size(1) == pool_ok_tc.size(1), "pool idx/ok width mismatch");
    TORCH_CHECK(pool_idx_tc.size(1) == cand_ctx_tc.size(1), "pool idx/cand_ctx width mismatch");

    TORCH_CHECK(router_w_f32.numel() == 7, "router_w_f32 must have 7 elements");
    TORCH_CHECK(block_size > 0, "block_size must be > 0");

    const int C = (int)pool_idx_tc.size(1);
    TORCH_CHECK(C > 0 && C <= MAX_POOL, "Cpool must be in [1, MAX_POOL]");
    TORCH_CHECK(topk > 0 && topk <= C && topk <= MAX_TOPK, "topk must be in [1, min(Cpool, MAX_TOPK)]");

    auto H_f32 = H_bnd.to(torch::kFloat32).contiguous();
    auto deficit_f32 = deficit_bn.to(torch::kFloat32).contiguous();
    auto K_f32 = K_bnd.to(torch::kFloat32).contiguous();
    auto V_f32 = V_bnd.to(torch::kFloat32).contiguous();
    auto idx_i32 = pool_idx_tc.to(torch::kInt32).contiguous();
    auto ok_u8 = pool_ok_tc.ne(0).to(torch::kUInt8).contiguous();
    auto ctx_f32 = cand_ctx_tc.to(torch::kFloat32).contiguous();
    auto w_f32 = router_w_f32.to(torch::kFloat32).contiguous();

    CHECK_CONTIGUOUS(H_f32);
    CHECK_CONTIGUOUS(deficit_f32);
    CHECK_CONTIGUOUS(K_f32);
    CHECK_CONTIGUOUS(V_f32);
    CHECK_CONTIGUOUS(idx_i32);
    CHECK_CONTIGUOUS(ok_u8);
    CHECK_CONTIGUOUS(ctx_f32);
    CHECK_CONTIGUOUS(w_f32);

    auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(H_bnd.device());
    auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(H_bnd.device());
    auto opts_u8  = torch::TensorOptions().dtype(torch::kUInt8).device(H_bnd.device());

    torch::Tensor logits_tc = torch::empty({T64, C}, opts_f32);
    torch::Tensor probs_tc  = torch::empty({T64, C}, opts_f32);
    torch::Tensor softK_tdv = torch::empty({T64, Dv64}, opts_f32);
    torch::Tensor softV_tdv = torch::empty({T64, Dv64}, opts_f32);

    torch::Tensor hard_mask_tc = torch::empty({T64, C}, opts_u8);
    torch::Tensor hard_idx_tk  = torch::empty({T64, topk}, opts_i32);
    torch::Tensor hard_ok_tk   = torch::empty({T64, topk}, opts_u8);

    const int B = (int)B64;
    const int N = (int)N64;
    const int Dh = (int)Dh64;
    const int Dv = (int)Dv64;
    const int T = (int)T64;

    const int threads = 128;
    const int blocks = (T + threads - 1) / threads;
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    dcsfc_router_forward_kernel<<<blocks, threads, 0, stream>>>(
        H_f32.data_ptr<float>(),
        deficit_f32.data_ptr<float>(),
        K_f32.data_ptr<float>(),
        V_f32.data_ptr<float>(),
        idx_i32.data_ptr<int32_t>(),
        ok_u8.data_ptr<uint8_t>(),
        ctx_f32.data_ptr<float>(),
        w_f32.data_ptr<float>(),
        B, N, Dh, Dv, C, (float)tau, (int)block_size, causal_only,
        logits_tc.data_ptr<float>(),
        probs_tc.data_ptr<float>(),
        softK_tdv.data_ptr<float>(),
        softV_tdv.data_ptr<float>()
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    dcsfc_hard_topk_kernel<<<blocks, threads, 0, stream>>>(
        idx_i32.data_ptr<int32_t>(),
        ok_u8.data_ptr<uint8_t>(),
        probs_tc.data_ptr<float>(),
        B, N, C, (int)topk, causal_only,
        hard_mask_tc.data_ptr<uint8_t>(),
        hard_idx_tk.data_ptr<int32_t>(),
        hard_ok_tk.data_ptr<uint8_t>()
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return {
        logits_tc, probs_tc, softK_tdv, softV_tdv,
        hard_mask_tc, hard_idx_tk, hard_ok_tk
    };
}

std::vector<torch::Tensor> dcsfc_router_family_backward_cuda(
    torch::Tensor H_bnd,
    torch::Tensor deficit_bn,
    torch::Tensor K_bnd,
    torch::Tensor V_bnd,
    torch::Tensor pool_idx_tc,
    torch::Tensor pool_ok_tc,
    torch::Tensor cand_ctx_tc,
    torch::Tensor probs_tc,
    torch::Tensor router_w_f32,
    torch::Tensor grad_logits_tc,
    torch::Tensor grad_probs_tc,
    torch::Tensor grad_softK_tdv,
    torch::Tensor grad_softV_tdv,
    double tau,
    int64_t block_size,
    bool causal_only
) {
    CHECK_CUDA(H_bnd);
    CHECK_CUDA(deficit_bn);
    CHECK_CUDA(K_bnd);
    CHECK_CUDA(V_bnd);
    CHECK_CUDA(pool_idx_tc);
    CHECK_CUDA(pool_ok_tc);
    CHECK_CUDA(cand_ctx_tc);
    CHECK_CUDA(probs_tc);
    CHECK_CUDA(router_w_f32);
    CHECK_CUDA(grad_logits_tc);
    CHECK_CUDA(grad_probs_tc);
    CHECK_CUDA(grad_softK_tdv);
    CHECK_CUDA(grad_softV_tdv);

    c10::cuda::CUDAGuard device_guard(H_bnd.device());
    TORCH_CHECK(deficit_bn.device() == H_bnd.device(), "deficit_bn device mismatch");
    TORCH_CHECK(K_bnd.device() == H_bnd.device(), "K_bnd device mismatch");
    TORCH_CHECK(V_bnd.device() == H_bnd.device(), "V_bnd device mismatch");
    TORCH_CHECK(pool_idx_tc.device() == H_bnd.device(), "pool_idx_tc device mismatch");
    TORCH_CHECK(pool_ok_tc.device() == H_bnd.device(), "pool_ok_tc device mismatch");
    TORCH_CHECK(cand_ctx_tc.device() == H_bnd.device(), "cand_ctx_tc device mismatch");
    TORCH_CHECK(probs_tc.device() == H_bnd.device(), "probs_tc device mismatch");
    TORCH_CHECK(router_w_f32.device() == H_bnd.device(), "router_w_f32 device mismatch");
    TORCH_CHECK(grad_logits_tc.device() == H_bnd.device(), "grad_logits_tc device mismatch");
    TORCH_CHECK(grad_probs_tc.device() == H_bnd.device(), "grad_probs_tc device mismatch");
    TORCH_CHECK(grad_softK_tdv.device() == H_bnd.device(), "grad_softK_tdv device mismatch");
    TORCH_CHECK(grad_softV_tdv.device() == H_bnd.device(), "grad_softV_tdv device mismatch");

    CHECK_3D(H_bnd);
    CHECK_2D(deficit_bn);
    CHECK_3D(K_bnd);
    CHECK_3D(V_bnd);
    CHECK_2D(pool_idx_tc);
    CHECK_2D(pool_ok_tc);
    CHECK_2D(cand_ctx_tc);
    CHECK_2D(probs_tc);
    CHECK_2D(grad_logits_tc);
    CHECK_2D(grad_probs_tc);
    CHECK_2D(grad_softK_tdv);
    CHECK_2D(grad_softV_tdv);

    TORCH_CHECK(H_bnd.size(0) == deficit_bn.size(0) && H_bnd.size(1) == deficit_bn.size(1),
                "H_bnd and deficit_bn shape mismatch");
    TORCH_CHECK(K_bnd.size(0) == H_bnd.size(0) && K_bnd.size(1) == H_bnd.size(1),
                "K_bnd shape mismatch");
    TORCH_CHECK(V_bnd.size(0) == H_bnd.size(0) && V_bnd.size(1) == H_bnd.size(1),
                "V_bnd shape mismatch");
    TORCH_CHECK(V_bnd.size(2) == K_bnd.size(2),
                "V_bnd.size(2) must equal K_bnd.size(2)");

    const int64_t B64 = H_bnd.size(0);
    const int64_t N64 = H_bnd.size(1);
    const int64_t Dh64 = H_bnd.size(2);
    const int64_t Dv64 = K_bnd.size(2);
    TORCH_CHECK(B64 > 0 && N64 > 0 && Dh64 > 0 && Dv64 > 0,
                "B, N, Dh, Dv must be positive");
    TORCH_CHECK(N64 == 0 || B64 <= std::numeric_limits<int64_t>::max() / N64,
                "B*N overflows int64");
    const int64_t T64 = B64 * N64;
    TORCH_CHECK(B64 <= INT_MAX && N64 <= INT_MAX &&
                Dh64 <= INT_MAX && Dv64 <= INT_MAX &&
                T64 <= INT_MAX,
                "B/N/Dh/Dv/T exceed int32 kernel indexing limits");

    TORCH_CHECK(pool_idx_tc.size(0) == T64, "pool_idx_tc rows must equal T=B*N");
    TORCH_CHECK(pool_ok_tc.size(0) == T64,  "pool_ok_tc rows must equal T=B*N");
    TORCH_CHECK(cand_ctx_tc.size(0) == T64, "cand_ctx_tc rows must equal T=B*N");
    TORCH_CHECK(pool_idx_tc.size(1) == pool_ok_tc.size(1), "pool idx/ok width mismatch");
    TORCH_CHECK(pool_idx_tc.size(1) == cand_ctx_tc.size(1), "pool idx/cand_ctx width mismatch");
    TORCH_CHECK(probs_tc.sizes() == pool_idx_tc.sizes(), "probs_tc shape mismatch");
    TORCH_CHECK(grad_logits_tc.sizes() == pool_idx_tc.sizes(), "grad_logits_tc shape mismatch");
    TORCH_CHECK(grad_probs_tc.sizes() == pool_idx_tc.sizes(), "grad_probs_tc shape mismatch");
    TORCH_CHECK(grad_softK_tdv.size(0) == T64 && grad_softK_tdv.size(1) == Dv64,
                "grad_softK_tdv shape mismatch");
    TORCH_CHECK(grad_softV_tdv.size(0) == T64 && grad_softV_tdv.size(1) == Dv64,
                "grad_softV_tdv shape mismatch");
    TORCH_CHECK(router_w_f32.numel() == 7, "router_w_f32 must have 7 elements");
    TORCH_CHECK(block_size > 0, "block_size must be > 0");

    const int C = (int)pool_idx_tc.size(1);
    TORCH_CHECK(C > 0 && C <= MAX_POOL, "Cpool must be in [1, MAX_POOL]");

    auto H_f32 = H_bnd.to(torch::kFloat32).contiguous();
    auto deficit_f32 = deficit_bn.to(torch::kFloat32).contiguous();
    auto K_f32 = K_bnd.to(torch::kFloat32).contiguous();
    auto V_f32 = V_bnd.to(torch::kFloat32).contiguous();
    auto idx_i32 = pool_idx_tc.to(torch::kInt32).contiguous();
    auto ok_u8 = pool_ok_tc.ne(0).to(torch::kUInt8).contiguous();
    auto ctx_f32 = cand_ctx_tc.to(torch::kFloat32).contiguous();
    auto probs_f32 = probs_tc.to(torch::kFloat32).contiguous();
    auto w_f32 = router_w_f32.to(torch::kFloat32).contiguous();
    auto g_logits = grad_logits_tc.to(torch::kFloat32).contiguous();
    auto g_probs  = grad_probs_tc.to(torch::kFloat32).contiguous();
    auto g_softK  = grad_softK_tdv.to(torch::kFloat32).contiguous();
    auto g_softV  = grad_softV_tdv.to(torch::kFloat32).contiguous();

    auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(H_bnd.device());

    torch::Tensor grad_H = torch::zeros_like(H_f32);
    torch::Tensor grad_deficit = torch::zeros_like(deficit_f32);
    torch::Tensor grad_K = torch::zeros_like(K_f32);
    torch::Tensor grad_V = torch::zeros_like(V_f32);
    torch::Tensor grad_cand_ctx = torch::zeros_like(ctx_f32);
    torch::Tensor grad_router_w = torch::zeros({7}, opts_f32);

    const int B = (int)B64;
    const int N = (int)N64;
    const int Dh = (int)Dh64;
    const int Dv = (int)Dv64;
    const int T = (int)T64;

    const int threads = 128;
    const int blocks = (T + threads - 1) / threads;
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    dcsfc_router_backward_kernel<<<blocks, threads, 0, stream>>>(
        H_f32.data_ptr<float>(),
        deficit_f32.data_ptr<float>(),
        K_f32.data_ptr<float>(),
        V_f32.data_ptr<float>(),
        idx_i32.data_ptr<int32_t>(),
        ok_u8.data_ptr<uint8_t>(),
        ctx_f32.data_ptr<float>(),
        probs_f32.data_ptr<float>(),
        w_f32.data_ptr<float>(),
        g_logits.data_ptr<float>(),
        g_probs.data_ptr<float>(),
        g_softK.data_ptr<float>(),
        g_softV.data_ptr<float>(),
        B, N, Dh, Dv, C, (float)tau, (int)block_size, causal_only,
        grad_H.data_ptr<float>(),
        grad_deficit.data_ptr<float>(),
        grad_K.data_ptr<float>(),
        grad_V.data_ptr<float>(),
        grad_cand_ctx.data_ptr<float>(),
        grad_router_w.data_ptr<float>()
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return {grad_H, grad_deficit, grad_K, grad_V, grad_cand_ctx, grad_router_w};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("dcsfc_router_family_forward_cuda",  &dcsfc_router_family_forward_cuda,  "DCSFC router family forward (CUDA)");
    m.def("dcsfc_router_family_backward_cuda", &dcsfc_router_family_backward_cuda, "DCSFC router family backward (CUDA)");
}
"""

extra_cuda_cflags = ["-O3", "--use_fast_math"]
extra_cflags = ["-O3"]

_build_sig = "dcsfc_router_family_softmax_st_cuda_v1"
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
ext_name = f"dcsfc_router_family_ext_{_build_hash}"

dcsfc_router_family_ext = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=CUDA_SRC,
    functions=None,
    extra_cuda_cflags=extra_cuda_cflags,
    extra_cflags=extra_cflags,
    with_cuda=True,
    verbose=bool(FUSED_VERBOSE_BUILD),
)

print("Built:", ext_name)

In [ ]:
# ============================================================
# Autograd wrapper for the CUDA router core
# ============================================================

import torch
import torch.nn as nn


class DCSFCRouterFamilyFn(torch.autograd.Function):
    @staticmethod
    def forward(
        ctx,
        H_bnd,
        deficit_bn,
        K_bnd,
        V_bnd,
        pool_idx_tc,
        pool_ok_tc,
        cand_ctx_tc,
        router_w_f32,
        tau: float,
        topk: int,
        block_size: int,
        causal_only: bool,
    ):
        (
            logits_tc,
            probs_tc,
            softK_tdv,
            softV_tdv,
            hard_mask_tc,
            hard_idx_tk,
            hard_ok_tk,
        ) = dcsfc_router_family_ext.dcsfc_router_family_forward_cuda(
            H_bnd,
            deficit_bn,
            K_bnd,
            V_bnd,
            pool_idx_tc,
            pool_ok_tc,
            cand_ctx_tc,
            router_w_f32,
            float(tau),
            int(topk),
            int(block_size),
            bool(causal_only),
        )

        ctx.save_for_backward(
            H_bnd,
            deficit_bn,
            K_bnd,
            V_bnd,
            pool_idx_tc,
            pool_ok_tc,
            cand_ctx_tc,
            probs_tc,
            router_w_f32,
        )
        ctx.tau = float(tau)
        ctx.block_size = int(block_size)
        ctx.causal_only = bool(causal_only)

        return (
            logits_tc,
            probs_tc,
            softK_tdv,
            softV_tdv,
            hard_mask_tc,
            hard_idx_tk,
            hard_ok_tk,
        )

    @staticmethod
    def backward(
        ctx,
        grad_logits_tc,
        grad_probs_tc,
        grad_softK_tdv,
        grad_softV_tdv,
        grad_hard_mask_tc,
        grad_hard_idx_tk,
        grad_hard_ok_tk,
    ):
        (
            H_bnd,
            deficit_bn,
            K_bnd,
            V_bnd,
            pool_idx_tc,
            pool_ok_tc,
            cand_ctx_tc,
            probs_tc,
            router_w_f32,
        ) = ctx.saved_tensors

        if grad_logits_tc is None:
            grad_logits_tc = torch.zeros_like(probs_tc)
        if grad_probs_tc is None:
            grad_probs_tc = torch.zeros_like(probs_tc)
        if grad_softK_tdv is None:
            grad_softK_tdv = torch.zeros(
                probs_tc.size(0),
                K_bnd.size(-1),
                device=probs_tc.device,
                dtype=torch.float32,
            )
        if grad_softV_tdv is None:
            grad_softV_tdv = torch.zeros(
                probs_tc.size(0),
                V_bnd.size(-1),
                device=probs_tc.device,
                dtype=torch.float32,
            )

        (
            grad_H,
            grad_deficit,
            grad_K,
            grad_V,
            grad_cand_ctx,
            grad_router_w,
        ) = dcsfc_router_family_ext.dcsfc_router_family_backward_cuda(
            H_bnd,
            deficit_bn,
            K_bnd,
            V_bnd,
            pool_idx_tc,
            pool_ok_tc,
            cand_ctx_tc,
            probs_tc,
            router_w_f32,
            grad_logits_tc,
            grad_probs_tc,
            grad_softK_tdv,
            grad_softV_tdv,
            float(ctx.tau),
            int(ctx.block_size),
            bool(ctx.causal_only),
        )

        return (
            grad_H.to(H_bnd.dtype),
            grad_deficit.to(deficit_bn.dtype),
            grad_K.to(K_bnd.dtype),
            grad_V.to(V_bnd.dtype),
            None,                            # pool_idx_tc
            None,                            # pool_ok_tc
            grad_cand_ctx.to(cand_ctx_tc.dtype),
            grad_router_w.to(router_w_f32.dtype),
            None,                            # tau
            None,                            # topk
            None,                            # block_size
            None,                            # causal_only
        )


class DCSFCRouterFamily(nn.Module):
    """
    Family-agnostic learnable bounded-pool router.

    router_w layout:
      [w_sim, w_di, w_dj, w_pos, w_blk, w_ctx, bias]

    cand_ctx_tc:
      - window family: local-fit / local summary score
      - ring family  : band-fit / bridge score
      - anchor family: spectral centrality / landmark score
      - if unavailable, pass zeros
    """
    def __init__(
        self,
        topk: int,
        block_size: int = 64,
        tau: float = 0.35,
        causal_only: bool = True,
        init_w = None,
    ):
        super().__init__()
        if int(topk) <= 0:
            raise ValueError(f"topk must be > 0, got {topk}")
        if int(block_size) <= 0:
            raise ValueError(f"block_size must be > 0, got {block_size}")
        if float(tau) <= 0:
            raise ValueError(f"tau must be > 0, got {tau}")
        if init_w is None:
            init_w = [1.0, 0.25, 0.75, -0.25, 0.10, 1.00, 0.0]
        if len(init_w) != 7:
            raise ValueError(f"init_w must have length 7, got {len(init_w)}")
        w = torch.tensor(init_w, dtype=torch.float32)
        self.router_w = nn.Parameter(w)
        self.topk = int(topk)
        self.block_size = int(block_size)
        self.tau = float(tau)
        self.causal_only = bool(causal_only)

    def forward(
        self,
        H_bnd,        # [B,N,Dh]
        deficit_bn,   # [B,N]
        K_bnd,        # [B,N,Dv]
        V_bnd,        # [B,N,Dv]
        pool_idx_tc,  # [T,Cpool]
        pool_ok_tc,   # [T,Cpool]
        cand_ctx_tc=None,  # [T,Cpool]
    ):
        T, Cpool = pool_idx_tc.shape
        if cand_ctx_tc is None:
            cand_ctx_tc = torch.zeros(T, Cpool, device=H_bnd.device, dtype=torch.float32)

        (
            logits_tc,
            probs_tc,
            softK_tdv,
            softV_tdv,
            hard_mask_tc,
            hard_idx_tk,
            hard_ok_tk,
        ) = DCSFCRouterFamilyFn.apply(
            H_bnd,
            deficit_bn,
            K_bnd,
            V_bnd,
            pool_idx_tc,
            pool_ok_tc,
            cand_ctx_tc,
            self.router_w,
            self.tau,
            self.topk,
            self.block_size,
            self.causal_only,
        )

        # Straight-through hard mask bridge:
        # use hard_mask_ste in auxiliary losses / soft-hard consistency losses
        hard_mask_f32 = hard_mask_tc.float()
        hard_mask_ste = hard_mask_f32 + probs_tc - probs_tc.detach()

        return {
            "logits_tc": logits_tc,          # [T,Cpool]
            "probs_tc": probs_tc,            # [T,Cpool]
            "softK_tdv": softK_tdv,          # [T,Dv]
            "softV_tdv": softV_tdv,          # [T,Dv]
            "hard_mask_tc": hard_mask_tc,    # [T,Cpool] uint8
            "hard_mask_ste": hard_mask_ste,  # [T,Cpool] float32
            "hard_idx_tk": hard_idx_tk,      # [T,topk] LOCAL token ids
            "hard_ok_tk": hard_ok_tk,        # [T,topk] uint8
        }

In [ ]:
# ============================================================
# Minimal DCSFC router losses - safer version
# ============================================================

def dcsfc_router_aux_loss(
    probs_tc: torch.Tensor,
    pool_ok_tc: torch.Tensor,
    lambda_entropy: float = 1e-3,
    lambda_util: float = 1e-3,
    eps: float = 1e-8,
):
    """
    probs_tc   : [T,C] float tensor, usually CUDA float32
    pool_ok_tc : [T,C] uint8/bool/int tensor

    Loss behavior under minimization:
      + lambda_entropy * row_entropy  -> encourages sharper / lower-entropy row routing
      - lambda_util    * util_entropy -> encourages broader utilization across columns
    """
    if probs_tc.dim() != 2:
        raise ValueError(f"probs_tc must be [T,C], got shape {tuple(probs_tc.shape)}")
    if pool_ok_tc.shape != probs_tc.shape:
        raise ValueError(
            f"pool_ok_tc shape must match probs_tc, got {tuple(pool_ok_tc.shape)} vs {tuple(probs_tc.shape)}"
        )

    # Keep computation stable and device-aligned
    probs = probs_tc.float()
    mask = pool_ok_tc.to(device=probs.device, dtype=probs.dtype)

    # Mask invalid pool slots
    p_raw = probs * mask
    row_mass = p_raw.sum(dim=-1, keepdim=True)

    # Only rows with nonzero valid mass should contribute to row entropy
    valid_rows = row_mass.squeeze(-1) > eps

    # If every row is invalid, return differentiable zero instead of hiding a NaN/crash
    if not valid_rows.any():
        return probs.sum() * 0.0

    p = p_raw / row_mass.clamp_min(eps)

    p_safe = p.clamp_min(eps)

    # Row entropy over valid rows only
    row_entropy_per_row = -(p_safe * p_safe.log()).sum(dim=-1)
    row_entropy = row_entropy_per_row[valid_rows].mean()

    # Column/slot utilization entropy
    col_mass = p[valid_rows].sum(dim=0)
    col_mass = col_mass / col_mass.sum().clamp_min(eps)
    col_safe = col_mass.clamp_min(eps)
    util_entropy = -(col_safe * col_safe.log()).sum()

    return lambda_entropy * row_entropy - lambda_util * util_entropy

In [ ]:
def dcsfc_router_aux_loss_token_util(
    probs_tc,
    pool_ok_tc,
    pool_idx_tc,
    num_tokens: int,
    lambda_entropy: float = 1e-3,
    lambda_util: float = 1e-3,
    eps: float = 1e-8,
):
    if probs_tc.dim() != 2:
        raise ValueError(f"probs_tc must be [T,C], got {tuple(probs_tc.shape)}")
    if pool_ok_tc.shape != probs_tc.shape:
        raise ValueError("pool_ok_tc shape must match probs_tc")
    if pool_idx_tc.shape != probs_tc.shape:
        raise ValueError("pool_idx_tc shape must match probs_tc")

    probs = probs_tc.float()
    mask = pool_ok_tc.to(device=probs.device, dtype=probs.dtype)
    idx = pool_idx_tc.to(device=probs.device, dtype=torch.long)

    valid = (mask > 0) & (idx >= 0) & (idx < num_tokens)

    p_raw = probs * valid.to(probs.dtype)
    row_mass = p_raw.sum(dim=-1, keepdim=True)
    valid_rows = row_mass.squeeze(-1) > eps

    if not valid_rows.any():
        return probs.sum() * 0.0

    p = p_raw / row_mass.clamp_min(eps)
    p_safe = p.clamp_min(eps)

    row_entropy = -(p_safe * p_safe.log()).sum(dim=-1)[valid_rows].mean()

    token_mass = torch.zeros(num_tokens, device=probs.device, dtype=probs.dtype)
    token_mass.scatter_add_(
        0,
        idx[valid].reshape(-1),
        p[valid].reshape(-1),
    )

    token_mass = token_mass / token_mass.sum().clamp_min(eps)
    token_safe = token_mass.clamp_min(eps)
    token_util_entropy = -(token_safe * token_safe.log()).sum()

    return lambda_entropy * row_entropy - lambda_util * token_util_entropy

In [ ]:
# ============================================================
# DCSFC bounded family pool builders (forward-only)
#
# Builds:
#   window_pool_idx_tc, window_pool_ok_tc, window_ctx_tc
#   ring_pool_idx_tc,   ring_pool_ok_tc,   ring_ctx_tc
#   anchor_pool_idx_tc, anchor_pool_ok_tc, anchor_ctx_tc
#
# Inputs:
#   H_bnd        : [B,N,Dh]   CUDA
#   deficit_bn   : [B,N]      CUDA
#   attn_mask_bn : [B,N]      CUDA
#
# Outputs:
#   window_pool_idx_tc  : [T,Cw_pool] int32   (LOCAL ids j in [0,N))
#   window_pool_ok_tc   : [T,Cw_pool] uint8
#   window_ctx_tc       : [T,Cw_pool] float32
#
#   ring_pool_idx_tc    : [T,Cr_pool] int32
#   ring_pool_ok_tc     : [T,Cr_pool] uint8
#   ring_ctx_tc         : [T,Cr_pool] float32
#
#   anchor_pool_idx_tc  : [T,Ca_pool] int32
#   anchor_pool_ok_tc   : [T,Ca_pool] uint8
#   anchor_ctx_tc       : [T,Ca_pool] float32
#
# Notes:
# - This is NOT the learnable router. It only builds bounded pools.
# - These outputs are meant to feed your DCSFCRouterFamily module.
# - LOCAL token ids are returned because your downstream router/compiler
#   already expects local j in [0,N).
# - This version uses:
#     * block reps from top-deficit tokens
#     * block mean deficit
#     * coarse spectral centrality from block rep cosine similarity
#     * anchor block selection from deficit + centrality + boundary bonus
# ============================================================

import hashlib
import torch
from torch.utils.cpp_extension import load_inline

FUSED_VERBOSE_BUILD = False

CUDA_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <vector>
#include <cstdint>
#include <cmath>

#define CHECK_CUDA(x) TORCH_CHECK((x).is_cuda(), #x " must be CUDA")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK((x).is_contiguous(), #x " must be contiguous")
#define CHECK_2D(x) TORCH_CHECK((x).dim() == 2, #x " must be 2D")
#define CHECK_3D(x) TORCH_CHECK((x).dim() == 3, #x " must be 3D")

static constexpr int MAX_POOL = 128;
static constexpr int MAX_REPS = 8;
static constexpr int MAX_ANCHOR_BLOCKS = 64;

__device__ __forceinline__ float neg_inf_f() { return -1.0e30f; }

__device__ __forceinline__ float clamp01(float x) {
    return x < 0.0f ? 0.0f : (x > 1.0f ? 1.0f : x);
}

__device__ __forceinline__ bool already_present_idx(int idx, const int* best_idx, int K) {
    for (int t = 0; t < K; ++t) {
        if (best_idx[t] == idx) return true;
    }
    return false;
}

__device__ __forceinline__ void insert_topk_unique(
    int idx,
    float score,
    int* best_idx,
    float* best_score,
    int K
) {
    if (idx < 0) return;
    if (!isfinite(score)) return;
    if (already_present_idx(idx, best_idx, K)) return;

    int pos = -1;
    for (int t = 0; t < K; ++t) {
        if (score > best_score[t]) {
            pos = t;
            break;
        }
    }
    if (pos < 0) return;

    for (int t = K - 1; t > pos; --t) {
        best_idx[t] = best_idx[t - 1];
        best_score[t] = best_score[t - 1];
    }
    best_idx[pos] = idx;
    best_score[pos] = score;
}

__device__ __forceinline__ float dot_row_f32(
    const float* __restrict__ H_t_d,
    int row_a,
    int row_b,
    int D
) {
    const int64_t off_a = (int64_t)row_a * (int64_t)D;
    const int64_t off_b = (int64_t)row_b * (int64_t)D;
    float s = 0.0f;
    for (int d = 0; d < D; ++d) {
        s += H_t_d[off_a + d] * H_t_d[off_b + d];
    }
    return s;
}

__device__ __forceinline__ float l2_row_f32(
    const float* __restrict__ H_t_d,
    int row,
    int D
) {
    const int64_t off = (int64_t)row * (int64_t)D;
    float s = 0.0f;
    for (int d = 0; d < D; ++d) {
        float v = H_t_d[off + d];
        s += v * v;
    }
    return sqrtf(fmaxf(s, 1.0e-12f));
}

__device__ __forceinline__ float cosine_row_f32(
    const float* __restrict__ H_t_d,
    int row_a,
    int row_b,
    int D
) {
    float num = dot_row_f32(H_t_d, row_a, row_b, D);
    float da = l2_row_f32(H_t_d, row_a, D);
    float db = l2_row_f32(H_t_d, row_b, D);
    return num / fmaxf(da * db, 1.0e-12f);
}

__global__ void select_block_reps_kernel(
    const uint8_t* __restrict__ mask_bn_u8,   // [B,N]
    const float*   __restrict__ deficit_bn,   // [B,N]
    int B,
    int N,
    int M,
    int block_size,
    int reps_per_block,
    int32_t* __restrict__ rep_idx_bmr,        // [B,M,R]
    uint8_t* __restrict__ rep_ok_bmr,         // [B,M,R]
    float*   __restrict__ block_mean_def_bm,  // [B,M]
    uint8_t* __restrict__ block_valid_bm      // [B,M]
) {
    const int bm = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    if (bm >= B * M) return;

    const int b = bm / M;
    const int m = bm - b * M;

    const int64_t start64 = (int64_t)m * (int64_t)block_size;
    const int64_t end64 = ((start64 + (int64_t)block_size) < (int64_t)N) ? (start64 + (int64_t)block_size) : (int64_t)N;
    const int start = (int)start64;
    const int end = (int)end64;

    int top_idx[MAX_REPS];
    float top_score[MAX_REPS];
    for (int r = 0; r < reps_per_block; ++r) {
        top_idx[r] = -1;
        top_score[r] = neg_inf_f();
    }

    float sum_def = 0.0f;
    int count = 0;

    for (int j = start; j < end; ++j) {
        if (!mask_bn_u8[b * N + j]) continue;
        float s = deficit_bn[b * N + j];
        if (!isfinite(s)) s = 0.0f;
        sum_def += s;
        count += 1;
        insert_topk_unique(j, s, top_idx, top_score, reps_per_block);
    }

    const int64_t rep_base = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
    for (int r = 0; r < reps_per_block; ++r) {
        rep_idx_bmr[rep_base + r] = top_idx[r];
        rep_ok_bmr[rep_base + r] = (top_idx[r] >= 0) ? 1 : 0;
    }

    float mean_def = (count > 0) ? (sum_def / (float)count) : 0.0f;
    if (!isfinite(mean_def)) mean_def = 0.0f;
    block_mean_def_bm[b * M + m] = mean_def;
    block_valid_bm[b * M + m] = (count > 0) ? 1 : 0;
}

__global__ void compute_block_centrality_kernel(
    const float*   __restrict__ H_t_d,        // [T,D]
    const int32_t* __restrict__ rep_idx_bmr,  // [B,M,R]
    const uint8_t* __restrict__ rep_ok_bmr,   // [B,M,R]
    const uint8_t* __restrict__ block_valid_bm,
    int B,
    int N,
    int D,
    int M,
    int reps_per_block,
    float* __restrict__ block_centrality_bm   // [B,M]
) {
    const int b = (int)blockIdx.x;
    if (b >= B) return;
    if (threadIdx.x != 0) return;

    for (int m = 0; m < M; ++m) {
        block_centrality_bm[b * M + m] = 0.0f;
    }

    for (int m = 0; m < M; ++m) {
        if (!block_valid_bm[b * M + m]) continue;

        const int64_t rep_base_m = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
        if (!rep_ok_bmr[rep_base_m + 0]) continue;

        const int rep_m_local = rep_idx_bmr[rep_base_m + 0];
        if (rep_m_local < 0 || rep_m_local >= N) continue;
        const int rep_m_global = b * N + rep_m_local;

        float cent = 0.0f;
        int cent_count = 0;
        for (int n = 0; n < M; ++n) {
            if (!block_valid_bm[b * M + n]) continue;

            const int64_t rep_base_n = ((int64_t)b * (int64_t)M + (int64_t)n) * (int64_t)reps_per_block;
            if (!rep_ok_bmr[rep_base_n + 0]) continue;

            const int rep_n_local = rep_idx_bmr[rep_base_n + 0];
            if (rep_n_local < 0 || rep_n_local >= N) continue;
            const int rep_n_global = b * N + rep_n_local;

            float sim = cosine_row_f32(H_t_d, rep_m_global, rep_n_global, D);
            if (!isfinite(sim)) sim = 0.0f;
            cent += fmaxf(sim, 0.0f);
            cent_count += 1;
        }
        block_centrality_bm[b * M + m] = (cent_count > 0) ? (cent / (float)cent_count) : 0.0f;
    }
}

__global__ void select_anchor_blocks_kernel(
    const float*   __restrict__ block_mean_def_bm,   // [B,M]
    const float*   __restrict__ block_centrality_bm, // [B,M]
    const uint8_t* __restrict__ block_valid_bm,      // [B,M]
    int B,
    int M,
    int anchor_block_k,
    uint8_t* __restrict__ anchor_mask_bm,            // [B,M]
    float*   __restrict__ anchor_score_bm            // [B,M]
) {
    const int b = (int)blockIdx.x;
    if (b >= B) return;
    if (threadIdx.x != 0) return;

    for (int m = 0; m < M; ++m) {
        anchor_mask_bm[b * M + m] = 0;
        anchor_score_bm[b * M + m] = 0.0f;
    }

    int top_blk[MAX_ANCHOR_BLOCKS];
    float top_score[MAX_ANCHOR_BLOCKS];
    for (int k = 0; k < anchor_block_k; ++k) {
        top_blk[k] = -1;
        top_score[k] = neg_inf_f();
    }

    for (int m = 0; m < M; ++m) {
        if (!block_valid_bm[b * M + m]) continue;

        float def = block_mean_def_bm[b * M + m];
        if (!isfinite(def)) def = 0.0f;
        float cent = block_centrality_bm[b * M + m];
        if (!isfinite(cent)) cent = 0.0f;
        const float boundary = (m == 0 || m == (M - 1)) ? 1.0f : 0.0f;

        const float score =
            1.25f * def +
            1.00f * cent +
            0.10f * boundary;

        anchor_score_bm[b * M + m] = score;
        insert_topk_unique(m, score, top_blk, top_score, anchor_block_k);
    }

    for (int k = 0; k < anchor_block_k; ++k) {
        const int m = top_blk[k];
        if (m >= 0 && m < M) {
            anchor_mask_bm[b * M + m] = 1;
        }
    }
}

__global__ void build_window_pool_kernel(
    const float*   __restrict__ deficit_bn,         // [B,N]
    const uint8_t* __restrict__ mask_bn_u8,         // [B,N]
    const float*   __restrict__ block_mean_def_bm,  // [B,M]
    int B,
    int N,
    int M,
    int block_size,
    int window_radius_tokens,
    int Cw_pool,
    bool causal_only,
    int32_t* __restrict__ window_pool_idx_tc,       // [T,Cw]
    uint8_t* __restrict__ window_pool_ok_tc,        // [T,Cw]
    float*   __restrict__ window_ctx_tc             // [T,Cw]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;

    const int64_t row_base = (int64_t)t * (int64_t)Cw_pool;
    for (int c = 0; c < Cw_pool; ++c) {
        window_pool_idx_tc[row_base + c] = -1;
        window_pool_ok_tc[row_base + c] = 0;
        window_ctx_tc[row_base + c] = 0.0f;
    }
    if (!mask_bn_u8[b * N + i]) return;

    int best_idx[MAX_POOL];
    float best_score[MAX_POOL];
    for (int c = 0; c < Cw_pool; ++c) {
        best_idx[c] = -1;
        best_score[c] = neg_inf_f();
    }

    const int64_t j_lo64 = (int64_t)i - (int64_t)window_radius_tokens;
    const int64_t j_hi64 = (int64_t)i + (int64_t)window_radius_tokens;
    const int j_lo = (j_lo64 > 0) ? (int)j_lo64 : 0;
    const int j_hi = causal_only ? i : ((j_hi64 < (int64_t)(N - 1)) ? (int)j_hi64 : (N - 1));

    for (int j = j_lo; j <= j_hi; ++j) {
        if (j == i) continue;
        if (!mask_bn_u8[b * N + j]) continue;

        const float dist_norm = (window_radius_tokens > 0) ? ((float)abs(i - j) / (float)window_radius_tokens) : 0.0f;
        const int block_j = j / block_size;
        float blk_def = block_mean_def_bm[b * M + block_j];
        if (!isfinite(blk_def)) blk_def = 0.0f;
        float dj = deficit_bn[b * N + j];
        if (!isfinite(dj)) dj = 0.0f;

        // local support context
        const float ctx =
            0.60f * (1.0f - clamp01(dist_norm)) +
            0.25f * blk_def +
            0.15f * dj;

        insert_topk_unique(j, ctx, best_idx, best_score, Cw_pool);
    }

    for (int c = 0; c < Cw_pool; ++c) {
        const int j = best_idx[c];
        if (j >= 0) {
            window_pool_idx_tc[row_base + c] = j;
            window_pool_ok_tc[row_base + c] = 1;
            window_ctx_tc[row_base + c] = best_score[c];
        }
    }
}

__global__ void build_ring_pool_kernel(
    const int32_t* __restrict__ rep_idx_bmr,        // [B,M,R]
    const uint8_t* __restrict__ rep_ok_bmr,         // [B,M,R]
    const float*   __restrict__ block_mean_def_bm,  // [B,M]
    const float*   __restrict__ block_centrality_bm,// [B,M]
    const uint8_t* __restrict__ block_valid_bm,     // [B,M]
    const uint8_t* __restrict__ anchor_mask_bm,     // [B,M]
    const uint8_t* __restrict__ mask_bn_u8,         // [B,N]
    int B,
    int N,
    int M,
    int reps_per_block,
    int block_size,
    int ring_min_blocks,
    int ring_max_blocks,
    int Cr_pool,
    bool causal_only,
    int32_t* __restrict__ ring_pool_idx_tc,         // [T,Cr]
    uint8_t* __restrict__ ring_pool_ok_tc,          // [T,Cr]
    float*   __restrict__ ring_ctx_tc               // [T,Cr]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;
    const int block_i = i / block_size;

    const int64_t row_base = (int64_t)t * (int64_t)Cr_pool;
    for (int c = 0; c < Cr_pool; ++c) {
        ring_pool_idx_tc[row_base + c] = -1;
        ring_pool_ok_tc[row_base + c] = 0;
        ring_ctx_tc[row_base + c] = 0.0f;
    }
    if (!mask_bn_u8[b * N + i]) return;

    int best_idx[MAX_POOL];
    float best_score[MAX_POOL];
    for (int c = 0; c < Cr_pool; ++c) {
        best_idx[c] = -1;
        best_score[c] = neg_inf_f();
    }

    const float band_target = 0.5f * (float)((int64_t)ring_min_blocks + (int64_t)ring_max_blocks);
    const float band_den = (float)((ring_max_blocks > 0) ? ring_max_blocks : 1);

    for (int m = 0; m < M; ++m) {
        if (!block_valid_bm[b * M + m]) continue;
        if (m == block_i) continue;
        if (anchor_mask_bm[b * M + m]) continue; // ring avoids global landmark blocks

        const int block_dist = abs(m - block_i);
        if (block_dist < ring_min_blocks || block_dist > ring_max_blocks) continue;
        if (causal_only && m > block_i) continue;

        const float band_fit = 1.0f - clamp01(fabsf((float)block_dist - band_target) / band_den);
        float blk_def = block_mean_def_bm[b * M + m];
        if (!isfinite(blk_def)) blk_def = 0.0f;
        float blk_cent = block_centrality_bm[b * M + m];
        if (!isfinite(blk_cent)) blk_cent = 0.0f;

        const int64_t rep_base = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
        for (int r = 0; r < reps_per_block; ++r) {
            if (!rep_ok_bmr[rep_base + r]) continue;
            const int j = rep_idx_bmr[rep_base + r];
            if (j < 0 || j >= N) continue;
            if (j == i) continue;
            if (!mask_bn_u8[b * N + j]) continue;
            if (causal_only && j > i) continue;

            // ring bridge context
            const float ctx =
                0.45f * band_fit +
                0.35f * blk_def +
                0.20f * blk_cent;

            insert_topk_unique(j, ctx, best_idx, best_score, Cr_pool);
        }
    }

    for (int c = 0; c < Cr_pool; ++c) {
        const int j = best_idx[c];
        if (j >= 0) {
            ring_pool_idx_tc[row_base + c] = j;
            ring_pool_ok_tc[row_base + c] = 1;
            ring_ctx_tc[row_base + c] = best_score[c];
        }
    }
}

__global__ void build_anchor_pool_kernel(
    const int32_t* __restrict__ rep_idx_bmr,        // [B,M,R]
    const uint8_t* __restrict__ rep_ok_bmr,         // [B,M,R]
    const float*   __restrict__ block_centrality_bm,// [B,M]
    const float*   __restrict__ anchor_score_bm,    // [B,M]
    const uint8_t* __restrict__ anchor_mask_bm,     // [B,M]
    const uint8_t* __restrict__ mask_bn_u8,         // [B,N]
    int B,
    int N,
    int M,
    int reps_per_block,
    int block_size,
    int ring_min_blocks,
    int Ca_pool,
    bool causal_only,
    int32_t* __restrict__ anchor_pool_idx_tc,       // [T,Ca]
    uint8_t* __restrict__ anchor_pool_ok_tc,        // [T,Ca]
    float*   __restrict__ anchor_ctx_tc             // [T,Ca]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;
    const int block_i = i / block_size;

    const int64_t row_base = (int64_t)t * (int64_t)Ca_pool;
    for (int c = 0; c < Ca_pool; ++c) {
        anchor_pool_idx_tc[row_base + c] = -1;
        anchor_pool_ok_tc[row_base + c] = 0;
        anchor_ctx_tc[row_base + c] = 0.0f;
    }
    if (!mask_bn_u8[b * N + i]) return;

    int best_idx[MAX_POOL];
    float best_score[MAX_POOL];
    for (int c = 0; c < Ca_pool; ++c) {
        best_idx[c] = -1;
        best_score[c] = neg_inf_f();
    }

    for (int m = 0; m < M; ++m) {
        if (!anchor_mask_bm[b * M + m]) continue;

        const int block_dist = abs(m - block_i);
        const float far_bonus = (block_dist >= ring_min_blocks) ? 1.0f : 0.0f;
        float a_score = anchor_score_bm[b * M + m];
        if (!isfinite(a_score)) a_score = 0.0f;
        float cent = block_centrality_bm[b * M + m];
        if (!isfinite(cent)) cent = 0.0f;

        const int64_t rep_base = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
        for (int r = 0; r < reps_per_block; ++r) {
            if (!rep_ok_bmr[rep_base + r]) continue;
            const int j = rep_idx_bmr[rep_base + r];
            if (j < 0 || j >= N) continue;
            if (j == i) continue;
            if (!mask_bn_u8[b * N + j]) continue;
            if (causal_only && j > i) continue;

            // global landmark context
            const float ctx =
                0.55f * a_score +
                0.25f * far_bonus +
                0.20f * cent;

            insert_topk_unique(j, ctx, best_idx, best_score, Ca_pool);
        }
    }

    for (int c = 0; c < Ca_pool; ++c) {
        const int j = best_idx[c];
        if (j >= 0) {
            anchor_pool_idx_tc[row_base + c] = j;
            anchor_pool_ok_tc[row_base + c] = 1;
            anchor_ctx_tc[row_base + c] = best_score[c];
        }
    }
}

std::vector<torch::Tensor> build_dcsfc_bounded_pools_cuda(
    torch::Tensor H_bnd,         // [B,N,D]
    torch::Tensor deficit_bn,    // [B,N]
    torch::Tensor attn_mask_bn,  // [B,N]

    int64_t block_size,
    int64_t reps_per_block,

    int64_t window_radius_tokens,
    int64_t Cw_pool,

    int64_t ring_min_blocks,
    int64_t ring_max_blocks,
    int64_t Cr_pool,

    int64_t Ca_pool,

    bool causal_only
) {
    CHECK_CUDA(H_bnd);
    CHECK_CUDA(deficit_bn);
    CHECK_CUDA(attn_mask_bn);
    CHECK_3D(H_bnd);
    CHECK_2D(deficit_bn);
    CHECK_2D(attn_mask_bn);

    TORCH_CHECK(H_bnd.size(0) == deficit_bn.size(0) && H_bnd.size(1) == deficit_bn.size(1),
                "H_bnd and deficit_bn shape mismatch");
    TORCH_CHECK(H_bnd.size(0) == attn_mask_bn.size(0) && H_bnd.size(1) == attn_mask_bn.size(1),
                "H_bnd and attn_mask_bn shape mismatch");
    TORCH_CHECK(deficit_bn.device() == H_bnd.device(),
                "deficit_bn must be on same CUDA device as H_bnd");
    TORCH_CHECK(attn_mask_bn.device() == H_bnd.device(),
                "attn_mask_bn must be on same CUDA device as H_bnd");

    c10::cuda::CUDAGuard device_guard(H_bnd.device());

    TORCH_CHECK(block_size > 0, "block_size must be > 0");
    TORCH_CHECK(block_size <= INT32_MAX, "block_size must fit int32");
    TORCH_CHECK(reps_per_block > 0 && reps_per_block <= MAX_REPS,
                "reps_per_block must be in [1, MAX_REPS]");
    TORCH_CHECK(reps_per_block <= INT32_MAX, "reps_per_block must fit int32");
    TORCH_CHECK(Cw_pool > 0 && Cw_pool <= MAX_POOL, "Cw_pool must be in [1, MAX_POOL]");
    TORCH_CHECK(Cr_pool > 0 && Cr_pool <= MAX_POOL, "Cr_pool must be in [1, MAX_POOL]");
    TORCH_CHECK(Ca_pool > 0 && Ca_pool <= MAX_POOL, "Ca_pool must be in [1, MAX_POOL]");
    TORCH_CHECK(Cw_pool <= INT32_MAX && Cr_pool <= INT32_MAX && Ca_pool <= INT32_MAX,
                "pool sizes must fit int32");
    TORCH_CHECK(window_radius_tokens >= 1, "window_radius_tokens must be >= 1");
    TORCH_CHECK(window_radius_tokens <= INT32_MAX, "window_radius_tokens must fit int32");
    TORCH_CHECK(ring_min_blocks >= 1, "ring_min_blocks must be >= 1");
    TORCH_CHECK(ring_min_blocks <= INT32_MAX, "ring_min_blocks must fit int32");
    TORCH_CHECK(ring_max_blocks >= ring_min_blocks, "ring_max_blocks must be >= ring_min_blocks");
    TORCH_CHECK(ring_max_blocks <= INT32_MAX, "ring_max_blocks must fit int32");

    const int64_t B64 = H_bnd.size(0);
    const int64_t N64 = H_bnd.size(1);
    const int64_t D64 = H_bnd.size(2);
    TORCH_CHECK(B64 > 0 && N64 > 0 && D64 > 0, "H_bnd must be non-empty");
    TORCH_CHECK(B64 <= INT32_MAX && N64 <= INT32_MAX && D64 <= INT32_MAX, "B/N/D must fit int32");

    const int64_t T64 = B64 * N64;
    TORCH_CHECK(T64 <= INT32_MAX, "B*N must fit int32 because kernels use int t/T");

    const int64_t M64 = (N64 + block_size - 1) / block_size;
    TORCH_CHECK(M64 > 0 && M64 <= INT32_MAX, "M must fit int32");
    TORCH_CHECK(B64 * M64 <= INT32_MAX, "B*M must fit int32 because kernels use int bm");

    const int B = (int)B64;
    const int N = (int)N64;
    const int D = (int)D64;
    const int T = (int)T64;
    const int M = (int)M64;

    // anchor blocks chosen to roughly cover Ca_pool with reps_per_block reps each
    int anchor_block_k = (int)((Ca_pool + reps_per_block - 1) / reps_per_block);
    if (anchor_block_k < 1) anchor_block_k = 1;
    if (anchor_block_k > M) anchor_block_k = M;
    TORCH_CHECK(anchor_block_k <= MAX_ANCHOR_BLOCKS, "anchor_block_k exceeds MAX_ANCHOR_BLOCKS");

    auto H_f32 = H_bnd.to(torch::kFloat32).contiguous();
    auto deficit_f32 = deficit_bn.to(torch::kFloat32).contiguous();
    auto mask_u8 = attn_mask_bn.ne(0).to(torch::kUInt8).contiguous();

    CHECK_CONTIGUOUS(H_f32);
    CHECK_CONTIGUOUS(deficit_f32);
    CHECK_CONTIGUOUS(mask_u8);

    auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(H_bnd.device());
    auto opts_u8  = torch::TensorOptions().dtype(torch::kUInt8).device(H_bnd.device());
    auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(H_bnd.device());

    // scratch block-level tensors
    torch::Tensor rep_idx_bmr        = torch::empty({B, M, (int)reps_per_block}, opts_i32);
    torch::Tensor rep_ok_bmr         = torch::empty({B, M, (int)reps_per_block}, opts_u8);
    torch::Tensor block_mean_def_bm  = torch::empty({B, M}, opts_f32);
    torch::Tensor block_valid_bm     = torch::empty({B, M}, opts_u8);
    torch::Tensor block_centrality_bm= torch::empty({B, M}, opts_f32);
    torch::Tensor anchor_mask_bm     = torch::empty({B, M}, opts_u8);
    torch::Tensor anchor_score_bm    = torch::empty({B, M}, opts_f32);

    // outputs
    torch::Tensor window_pool_idx_tc = torch::empty({T, (int)Cw_pool}, opts_i32);
    torch::Tensor window_pool_ok_tc  = torch::empty({T, (int)Cw_pool}, opts_u8);
    torch::Tensor window_ctx_tc      = torch::empty({T, (int)Cw_pool}, opts_f32);

    torch::Tensor ring_pool_idx_tc   = torch::empty({T, (int)Cr_pool}, opts_i32);
    torch::Tensor ring_pool_ok_tc    = torch::empty({T, (int)Cr_pool}, opts_u8);
    torch::Tensor ring_ctx_tc        = torch::empty({T, (int)Cr_pool}, opts_f32);

    torch::Tensor anchor_pool_idx_tc = torch::empty({T, (int)Ca_pool}, opts_i32);
    torch::Tensor anchor_pool_ok_tc  = torch::empty({T, (int)Ca_pool}, opts_u8);
    torch::Tensor anchor_ctx_tc      = torch::empty({T, (int)Ca_pool}, opts_f32);

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    // 1) block reps and mean deficits
    {
        const int threads = 128;
        const int blocks = (int)(((int64_t)B * (int64_t)M + (int64_t)threads - 1) / (int64_t)threads);
        select_block_reps_kernel<<<blocks, threads, 0, stream>>>(
            mask_u8.data_ptr<uint8_t>(),
            deficit_f32.data_ptr<float>(),
            B, N, M, (int)block_size, (int)reps_per_block,
            rep_idx_bmr.data_ptr<int32_t>(),
            rep_ok_bmr.data_ptr<uint8_t>(),
            block_mean_def_bm.data_ptr<float>(),
            block_valid_bm.data_ptr<uint8_t>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 2) block centrality from rep cosine similarity
    compute_block_centrality_kernel<<<B, 1, 0, stream>>>(
        H_f32.data_ptr<float>(),
        rep_idx_bmr.data_ptr<int32_t>(),
        rep_ok_bmr.data_ptr<uint8_t>(),
        block_valid_bm.data_ptr<uint8_t>(),
        B, N, D, M, (int)reps_per_block,
        block_centrality_bm.data_ptr<float>()
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    // 3) select anchor blocks
    select_anchor_blocks_kernel<<<B, 1, 0, stream>>>(
        block_mean_def_bm.data_ptr<float>(),
        block_centrality_bm.data_ptr<float>(),
        block_valid_bm.data_ptr<uint8_t>(),
        B, M, anchor_block_k,
        anchor_mask_bm.data_ptr<uint8_t>(),
        anchor_score_bm.data_ptr<float>()
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    // 4) window pool
    {
        const int threads = 128;
        const int blocks = (int)(((int64_t)T + (int64_t)threads - 1) / (int64_t)threads);
        build_window_pool_kernel<<<blocks, threads, 0, stream>>>(
            deficit_f32.data_ptr<float>(),
            mask_u8.data_ptr<uint8_t>(),
            block_mean_def_bm.data_ptr<float>(),
            B, N, M, (int)block_size,
            (int)window_radius_tokens,
            (int)Cw_pool,
            causal_only,
            window_pool_idx_tc.data_ptr<int32_t>(),
            window_pool_ok_tc.data_ptr<uint8_t>(),
            window_ctx_tc.data_ptr<float>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 5) ring pool
    {
        const int threads = 128;
        const int blocks = (int)(((int64_t)T + (int64_t)threads - 1) / (int64_t)threads);
        build_ring_pool_kernel<<<blocks, threads, 0, stream>>>(
            rep_idx_bmr.data_ptr<int32_t>(),
            rep_ok_bmr.data_ptr<uint8_t>(),
            block_mean_def_bm.data_ptr<float>(),
            block_centrality_bm.data_ptr<float>(),
            block_valid_bm.data_ptr<uint8_t>(),
            anchor_mask_bm.data_ptr<uint8_t>(),
            mask_u8.data_ptr<uint8_t>(),
            B, N, M, (int)reps_per_block, (int)block_size,
            (int)ring_min_blocks,
            (int)ring_max_blocks,
            (int)Cr_pool,
            causal_only,
            ring_pool_idx_tc.data_ptr<int32_t>(),
            ring_pool_ok_tc.data_ptr<uint8_t>(),
            ring_ctx_tc.data_ptr<float>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 6) anchor pool
    {
        const int threads = 128;
        const int blocks = (int)(((int64_t)T + (int64_t)threads - 1) / (int64_t)threads);
        build_anchor_pool_kernel<<<blocks, threads, 0, stream>>>(
            rep_idx_bmr.data_ptr<int32_t>(),
            rep_ok_bmr.data_ptr<uint8_t>(),
            block_centrality_bm.data_ptr<float>(),
            anchor_score_bm.data_ptr<float>(),
            anchor_mask_bm.data_ptr<uint8_t>(),
            mask_u8.data_ptr<uint8_t>(),
            B, N, M, (int)reps_per_block, (int)block_size,
            (int)ring_min_blocks,
            (int)Ca_pool,
            causal_only,
            anchor_pool_idx_tc.data_ptr<int32_t>(),
            anchor_pool_ok_tc.data_ptr<uint8_t>(),
            anchor_ctx_tc.data_ptr<float>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    return {
        window_pool_idx_tc, window_pool_ok_tc, window_ctx_tc,
        ring_pool_idx_tc,   ring_pool_ok_tc,   ring_ctx_tc,
        anchor_pool_idx_tc, anchor_pool_ok_tc, anchor_ctx_tc
    };
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def(
        "build_dcsfc_bounded_pools_cuda",
        &build_dcsfc_bounded_pools_cuda,
        "Build DCSFC bounded family pools (CUDA)"
    );
}
"""

extra_cuda_cflags = ["-O3", "--use_fast_math"]
extra_cflags = ["-O3"]

_build_sig = "dcsfc_bounded_family_pools_cuda_v1"
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
ext_name = f"dcsfc_bounded_family_pools_ext_{_build_hash}"

dcsfc_bounded_family_pools_ext = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=CUDA_SRC,
    functions=None,
    extra_cuda_cflags=extra_cuda_cflags,
    extra_cflags=extra_cflags,
    with_cuda=True,
    verbose=bool(FUSED_VERBOSE_BUILD),
)

print("Built:", ext_name)

In [ ]:
# ============================================================
# Python wrapper for the compiled CUDA bounded-pool extension
# Place this immediately AFTER the CUDA extension build cell.
# ============================================================

def build_dcsfc_bounded_pools_cuda(
    H_bnd,
    deficit_bn,
    attn_mask_bn,
    *,
    block_size=64,
    reps_per_block=2,
    window_radius_tokens=32,
    Cw_pool=32,
    ring_min_blocks=2,
    ring_max_blocks=8,
    Cr_pool=16,
    Ca_pool=16,
    causal_only=True,
):
    """
    Returns:
      window_pool_idx_tc, window_pool_ok_tc, window_ctx_tc,
      ring_pool_idx_tc,   ring_pool_ok_tc,   ring_ctx_tc,
      anchor_pool_idx_tc, anchor_pool_ok_tc, anchor_ctx_tc
    """
    return dcsfc_bounded_family_pools_ext.build_dcsfc_bounded_pools_cuda(
        H_bnd,
        deficit_bn,
        attn_mask_bn,
        int(block_size),
        int(reps_per_block),
        int(window_radius_tokens),
        int(Cw_pool),
        int(ring_min_blocks),
        int(ring_max_blocks),
        int(Cr_pool),
        int(Ca_pool),
        bool(causal_only),
    )

In [ ]:
# ============================================================
# End-to-end C-CUDA:
#   hidden states + deficit
#      -> CUDA family proposal
#      -> precomputed_*_idx/ok
#      -> CUDA family compiler
#
# Notes:
# - This replaces the PyTorch proposal modules with CUDA kernels.
# - It is a deterministic/scored proposer in CUDA.
# - Supports:
#     * window family
#     * ring family
#     * anchor family
#     * final Stage-A style materialization
# - Hidden states are converted to float32 internally for stability.
#
# Returned:
#   precomputed_window_idx, precomputed_window_ok
#   precomputed_ring_idx,   precomputed_ring_ok
#   precomputed_anchor_idx, precomputed_anchor_ok
#   cand_out, ok_out, valid_T, family_out
#   family_bounds, family_order_caps, dispatch_meta
# ============================================================

import hashlib
import torch
from torch.utils.cpp_extension import load_inline

FUSED_VERBOSE_BUILD = False

CUDA_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <vector>
#include <cstdint>
#include <cmath>
#include <algorithm>

#define CHECK_CUDA(x) TORCH_CHECK((x).is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK((x).is_contiguous(), #x " must be contiguous")
#define CHECK_2D(x) TORCH_CHECK((x).dim() == 2, #x " must be 2D")
#define CHECK_3D(x) TORCH_CHECK((x).dim() == 3, #x " must be 3D")

static constexpr int MAX_CAP  = 64;  // max Cw, Cr, Ca
static constexpr int MAX_REPS = 8;   // max reps per block

enum FamilyId : uint8_t {
    FAM_INVALID  = 0,
    FAM_SELF     = 1,
    FAM_WINDOW   = 2,
    FAM_RING     = 3,
    FAM_ANCHOR   = 4,
    FAM_PROMOTED = 5
};

enum ProtocolId : int32_t {
    PROTO_ECONOMY        = 0,
    PROTO_HYBRID         = 1,
    PROTO_MULTISCALE     = 2,
    PROTO_GLOBAL_REFRESH = 3
};

__device__ __forceinline__ float neg_inf() {
    return -1.0e30f;
}

__device__ __forceinline__ int ceil_div_int(int a, int b) {
    return (a + b - 1) / b;
}

__device__ __forceinline__ float dot_row_f32(
    const float* __restrict__ H, // [T,D]
    int row_a,
    int row_b,
    int D
) {
    const int64_t off_a = (int64_t)row_a * (int64_t)D;
    const int64_t off_b = (int64_t)row_b * (int64_t)D;
    float s = 0.0f;
    for (int d = 0; d < D; ++d) {
        s += H[off_a + d] * H[off_b + d];
    }
    return s;
}

__device__ __forceinline__ float l2_row_f32(
    const float* __restrict__ H,
    int row,
    int D
) {
    const int64_t off = (int64_t)row * (int64_t)D;
    float s = 0.0f;
    for (int d = 0; d < D; ++d) {
        float v = H[off + d];
        s += v * v;
    }
    return sqrtf(fmaxf(s, 1.0e-12f));
}

__device__ __forceinline__ float cosine_row_f32(
    const float* __restrict__ H,
    int row_a,
    int row_b,
    int D
) {
    float num = dot_row_f32(H, row_a, row_b, D);
    float da = l2_row_f32(H, row_a, D);
    float db = l2_row_f32(H, row_b, D);
    return num / fmaxf(da * db, 1.0e-12f);
}

__device__ __forceinline__ bool already_present(
    int idx,
    const int* best_idx,
    int K
) {
    for (int t = 0; t < K; ++t) {
        if (best_idx[t] == idx) return true;
    }
    return false;
}

__device__ __forceinline__ void insert_topk_unique(
    int idx,
    float score,
    int* best_idx,
    float* best_score,
    int K
) {
    if (idx < 0) return;
    if (!isfinite(score)) return;
    if (already_present(idx, best_idx, K)) return;

    int pos = -1;
    for (int t = 0; t < K; ++t) {
        if (score > best_score[t]) {
            pos = t;
            break;
        }
    }
    if (pos < 0) return;

    for (int t = K - 1; t > pos; --t) {
        best_idx[t] = best_idx[t - 1];
        best_score[t] = best_score[t - 1];
    }
    best_idx[pos] = idx;
    best_score[pos] = score;
}

__device__ __forceinline__ uint8_t family_code_for_slot(
    int c,
    int Cself,
    int Cw,
    int Cr,
    int Ca
) {
    const int self_s = 0;
    const int self_e = self_s + Cself;
    const int win_s  = self_e;
    const int win_e  = win_s + Cw;
    const int ring_s = win_e;
    const int ring_e = ring_s + Cr;
    const int anch_s = ring_e;
    const int anch_e = anch_s + Ca;

    if (c >= self_s && c < self_e) return FAM_SELF;
    if (c >= win_s  && c < win_e ) return FAM_WINDOW;
    if (c >= ring_s && c < ring_e) return FAM_RING;
    if (c >= anch_s && c < anch_e) return FAM_ANCHOR;
    return FAM_PROMOTED;
}

__device__ __forceinline__ int compute_stage_order_device(
    int protocol_id,
    int layer_idx,
    int num_layers,
    int refresh_period
) {
    if (protocol_id == PROTO_ECONOMY) return 0;
    if (protocol_id == PROTO_HYBRID) return 1;
    if (protocol_id == PROTO_MULTISCALE) return 1;

    if (protocol_id == PROTO_GLOBAL_REFRESH) {
        int periodic_refresh = (refresh_period > 0) && ((layer_idx % refresh_period) == 0);
        int later_layer = (num_layers > 0) && (layer_idx >= (num_layers / 2));
        return (periodic_refresh || later_layer) ? 2 : 1;
    }
    return 0;
}

__global__ void select_block_reps_kernel(
    const uint8_t* __restrict__ mask_bn_u8,   // [B,N]
    const float*   __restrict__ deficit_bn,   // [B,N]
    int B,
    int N,
    int M,
    int block_size,
    int reps_per_block,
    int32_t* __restrict__ rep_idx_bmr,        // [B,M,R], local j
    uint8_t* __restrict__ rep_ok_bmr,         // [B,M,R]
    float*   __restrict__ block_mean_def_bm,  // [B,M]
    uint8_t* __restrict__ block_valid_bm      // [B,M]
) {
    const int bm = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    if (bm >= B * M) return;

    const int b = bm / M;
    const int m = bm - b * M;

    const int start = m * block_size;
    const int end   = min(N, start + block_size);

    int top_idx[MAX_REPS];
    float top_score[MAX_REPS];
    for (int r = 0; r < reps_per_block; ++r) {
        top_idx[r] = -1;
        top_score[r] = neg_inf();
    }

    float sum_def = 0.0f;
    int count = 0;

    for (int j = start; j < end; ++j) {
        if (!mask_bn_u8[b * N + j]) continue;
        float s = deficit_bn[b * N + j];
        if (!isfinite(s)) s = 0.0f;
        sum_def += s;
        count += 1;
        insert_topk_unique(j, s, top_idx, top_score, reps_per_block);
    }

    const int64_t rep_base = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
    for (int r = 0; r < reps_per_block; ++r) {
        rep_idx_bmr[rep_base + r] = top_idx[r];
        rep_ok_bmr[rep_base + r] = (top_idx[r] >= 0) ? 1 : 0;
    }

    block_mean_def_bm[b * M + m] = (count > 0) ? (sum_def / (float)count) : 0.0f;
    block_valid_bm[b * M + m] = (count > 0) ? 1 : 0;
}

__global__ void select_anchor_blocks_kernel(
    const float*   __restrict__ H_t_d,         // [T,D]
    const int32_t* __restrict__ rep_idx_bmr,   // [B,M,R]
    const uint8_t* __restrict__ rep_ok_bmr,    // [B,M,R]
    const float*   __restrict__ block_mean_def_bm,
    const uint8_t* __restrict__ block_valid_bm,
    int B,
    int N,
    int D,
    int M,
    int reps_per_block,
    int anchor_block_k,
    float alpha_def,
    float beta_cent,
    float gamma_boundary,
    int32_t* __restrict__ anchor_block_ids_bk, // [B,K]
    uint8_t* __restrict__ anchor_block_ok_bk   // [B,K]
) {
    const int b = (int)blockIdx.x;
    if (b >= B) return;
    if (threadIdx.x != 0) return;

    int top_blk[MAX_CAP];
    float top_score[MAX_CAP];
    for (int k = 0; k < anchor_block_k; ++k) {
        top_blk[k] = -1;
        top_score[k] = neg_inf();
    }

    for (int m = 0; m < M; ++m) {
        if (!block_valid_bm[b * M + m]) continue;

        const int64_t rep_base_m = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
        if (!rep_ok_bmr[rep_base_m + 0]) continue;
        const int rep_m_local = rep_idx_bmr[rep_base_m + 0];
        if (rep_m_local < 0 || rep_m_local >= N) continue;

        const int rep_m_global = b * N + rep_m_local;

        float centrality = 0.0f;
        int centrality_count = 0;
        for (int n = 0; n < M; ++n) {
            if (!block_valid_bm[b * M + n]) continue;

            const int64_t rep_base_n = ((int64_t)b * (int64_t)M + (int64_t)n) * (int64_t)reps_per_block;
            if (!rep_ok_bmr[rep_base_n + 0]) continue;
            const int rep_n_local = rep_idx_bmr[rep_base_n + 0];
            if (rep_n_local < 0 || rep_n_local >= N) continue;

            const int rep_n_global = b * N + rep_n_local;
            float sim = cosine_row_f32(H_t_d, rep_m_global, rep_n_global, D);
            if (!isfinite(sim)) sim = 0.0f;
            centrality += fmaxf(sim, 0.0f);
            centrality_count += 1;
        }

        centrality = (centrality_count > 0) ? centrality / (float)centrality_count : 0.0f;

        const float boundary = (m == 0 || m == (M - 1)) ? 1.0f : 0.0f;
        const float score =
            alpha_def * block_mean_def_bm[b * M + m]
          + beta_cent * centrality
          + gamma_boundary * boundary;

        insert_topk_unique(m, score, top_blk, top_score, anchor_block_k);
    }

    for (int k = 0; k < anchor_block_k; ++k) {
        anchor_block_ids_bk[b * anchor_block_k + k] = top_blk[k];
        anchor_block_ok_bk[b * anchor_block_k + k] = (top_blk[k] >= 0) ? 1 : 0;
    }
}

__global__ void propose_family_candidates_kernel(
    const float*   __restrict__ H_t_d,         // [T,D]
    const float*   __restrict__ deficit_bn,    // [B,N]
    const uint8_t* __restrict__ mask_bn_u8,    // [B,N]

    const int32_t* __restrict__ rep_idx_bmr,   // [B,M,R]
    const uint8_t* __restrict__ rep_ok_bmr,    // [B,M,R]
    const float*   __restrict__ block_mean_def_bm,
    const uint8_t* __restrict__ block_valid_bm,

    const int32_t* __restrict__ anchor_block_ids_bk, // [B,K]
    const uint8_t* __restrict__ anchor_block_ok_bk,  // [B,K]

    int B,
    int N,
    int D,
    int M,
    int block_size,
    int reps_per_block,
    int anchor_block_k,

    int window_radius_tokens,
    int ring_min_blocks,
    int ring_max_blocks,

    int Cw,
    int Cr,
    int Ca,

    bool causal_only,

    float w_def,
    float w_sim,
    float w_dist,

    float r_def,
    float r_sim,
    float r_band,

    float a_def,
    float a_sim,
    float a_far,

    int32_t* __restrict__ win_idx_tc,   // [T,Cw]
    uint8_t* __restrict__ win_ok_tc,    // [T,Cw]
    int32_t* __restrict__ ring_idx_tc,  // [T,Cr]
    uint8_t* __restrict__ ring_ok_tc,   // [T,Cr]
    int32_t* __restrict__ anch_idx_tc,  // [T,Ca]
    uint8_t* __restrict__ anch_ok_tc    // [T,Ca]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;
    const int block_i = i / block_size;

    // init outputs
    for (int c = 0; c < Cw; ++c) {
        win_idx_tc[(int64_t)t * Cw + c] = -1;
        win_ok_tc[(int64_t)t * Cw + c] = 0;
    }
    for (int c = 0; c < Cr; ++c) {
        ring_idx_tc[(int64_t)t * Cr + c] = -1;
        ring_ok_tc[(int64_t)t * Cr + c] = 0;
    }
    for (int c = 0; c < Ca; ++c) {
        anch_idx_tc[(int64_t)t * Ca + c] = -1;
        anch_ok_tc[(int64_t)t * Ca + c] = 0;
    }

    if (!mask_bn_u8[b * N + i]) return;

    // -------------------------
    // WINDOW family
    // -------------------------
    int w_best_idx[MAX_CAP];
    float w_best_score[MAX_CAP];
    for (int c = 0; c < Cw; ++c) {
        w_best_idx[c] = -1;
        w_best_score[c] = neg_inf();
    }

    const int j_lo = max(0, i - window_radius_tokens);
    const int j_hi = causal_only ? i : min(N - 1, i + window_radius_tokens);

    for (int j = j_lo; j <= j_hi; ++j) {
        if (j == i) continue;
        if (!mask_bn_u8[b * N + j]) continue;

        const int gj = b * N + j;
        float sim = cosine_row_f32(H_t_d, t, gj, D);
        if (!isfinite(sim)) sim = 0.0f;
        float deficit_j = deficit_bn[b * N + j];
        if (!isfinite(deficit_j)) deficit_j = 0.0f;
        float dist_pen = (float)abs(i - j) / (float)max(window_radius_tokens, 1);
        float score =
            w_def  * deficit_j
          + w_sim  * sim
          - w_dist * dist_pen;

        insert_topk_unique(j, score, w_best_idx, w_best_score, Cw);
    }

    for (int c = 0; c < Cw; ++c) {
        win_idx_tc[(int64_t)t * Cw + c] = w_best_idx[c];
        win_ok_tc[(int64_t)t * Cw + c] = (w_best_idx[c] >= 0) ? 1 : 0;
    }

    // -------------------------
    // RING family
    // -------------------------
    int r_best_idx[MAX_CAP];
    float r_best_score[MAX_CAP];
    for (int c = 0; c < Cr; ++c) {
        r_best_idx[c] = -1;
        r_best_score[c] = neg_inf();
    }

    const float band_target = 0.5f * (float)(ring_min_blocks + ring_max_blocks);

    for (int m = 0; m < M; ++m) {
        if (!block_valid_bm[b * M + m]) continue;
        if (m == block_i) continue;

        int block_dist = abs(m - block_i);
        if (block_dist < ring_min_blocks || block_dist > ring_max_blocks) continue;

        if (causal_only && m > block_i) continue;

        const int64_t rep_base = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;

        for (int r = 0; r < reps_per_block; ++r) {
            if (!rep_ok_bmr[rep_base + r]) continue;
            int j = rep_idx_bmr[rep_base + r];
            if (j < 0 || j >= N) continue;
            if (j == i) continue;
            if (!mask_bn_u8[b * N + j]) continue;
            if (causal_only && j > i) continue;

            const int gj = b * N + j;
            float sim = cosine_row_f32(H_t_d, t, gj, D);
            if (!isfinite(sim)) sim = 0.0f;
            float band_pen = fabsf((float)block_dist - band_target) / fmaxf((float)ring_max_blocks, 1.0f);

            float score =
                r_def  * block_mean_def_bm[b * M + m]
              + r_sim  * sim
              - r_band * band_pen;

            insert_topk_unique(j, score, r_best_idx, r_best_score, Cr);
        }
    }

    for (int c = 0; c < Cr; ++c) {
        ring_idx_tc[(int64_t)t * Cr + c] = r_best_idx[c];
        ring_ok_tc[(int64_t)t * Cr + c] = (r_best_idx[c] >= 0) ? 1 : 0;
    }

    // -------------------------
    // ANCHOR family
    // -------------------------
    int a_best_idx[MAX_CAP];
    float a_best_score[MAX_CAP];
    for (int c = 0; c < Ca; ++c) {
        a_best_idx[c] = -1;
        a_best_score[c] = neg_inf();
    }

    for (int k = 0; k < anchor_block_k; ++k) {
        if (!anchor_block_ok_bk[b * anchor_block_k + k]) continue;
        int m = anchor_block_ids_bk[b * anchor_block_k + k];
        if (m < 0 || m >= M) continue;

        const int64_t rep_base = ((int64_t)b * (int64_t)M + (int64_t)m) * (int64_t)reps_per_block;
        for (int r = 0; r < reps_per_block; ++r) {
            if (!rep_ok_bmr[rep_base + r]) continue;
            int j = rep_idx_bmr[rep_base + r];
            if (j < 0 || j >= N) continue;
            if (j == i) continue;
            if (!mask_bn_u8[b * N + j]) continue;
            if (causal_only && j > i) continue;

            const int gj = b * N + j;
            float sim = cosine_row_f32(H_t_d, t, gj, D);
            if (!isfinite(sim)) sim = 0.0f;
            float far_bonus = (abs(m - block_i) >= ring_min_blocks) ? 1.0f : 0.0f;

            float score =
                a_def * block_mean_def_bm[b * M + m]
              + a_sim * sim
              + a_far * far_bonus;

            insert_topk_unique(j, score, a_best_idx, a_best_score, Ca);
        }
    }

    for (int c = 0; c < Ca; ++c) {
        anch_idx_tc[(int64_t)t * Ca + c] = a_best_idx[c];
        anch_ok_tc[(int64_t)t * Ca + c] = (a_best_idx[c] >= 0) ? 1 : 0;
    }
}

__global__ void compile_family_kernel(
    const uint8_t* __restrict__ mask_bn_u8, // [B,N]

    const int32_t* __restrict__ win_idx_tc,   // [T,Cw], local j
    const uint8_t* __restrict__ win_ok_tc,    // [T,Cw]
    int Cw,

    const int32_t* __restrict__ ring_idx_tc,  // [T,Cr], local j
    const uint8_t* __restrict__ ring_ok_tc,   // [T,Cr]
    int Cr,

    const int32_t* __restrict__ anch_idx_tc,  // [T,Ca], local j
    const uint8_t* __restrict__ anch_ok_tc,   // [T,Ca]
    int Ca,

    int B,
    int N,
    bool include_self,
    bool causal_only,
    bool dedup_across_families,

    int Cself,
    int Cmax,

    int32_t* __restrict__ cand_out_tc,   // [T,Cmax], global flattened ids
    uint8_t* __restrict__ ok_out_tc,     // [T,Cmax]
    uint8_t* __restrict__ valid_T_t,     // [T]
    uint8_t* __restrict__ family_out_tc  // [T,Cmax]
) {
    const int t = (int)blockIdx.x * (int)blockDim.x + (int)threadIdx.x;
    const int T = B * N;
    if (t >= T) return;

    const int b = t / N;
    const int i = t - b * N;

    const int64_t base = (int64_t)t * (int64_t)Cmax;
    const uint8_t q_valid = mask_bn_u8[b * N + i] ? 1 : 0;

    valid_T_t[t] = q_valid;

    for (int c = 0; c < Cmax; ++c) {
        cand_out_tc[base + c] = -1;
        ok_out_tc[base + c] = 0;
        family_out_tc[base + c] = family_code_for_slot(c, Cself, Cw, Cr, Ca);
    }

    if (!q_valid) return;

    if (Cself > 0 && include_self) {
        cand_out_tc[base + 0] = b * N + i;
        ok_out_tc[base + 0] = 1;
        family_out_tc[base + 0] = FAM_SELF;
    }

    const int win_s = Cself;
    for (int c = 0; c < Cw; ++c) {
        int j = win_idx_tc[(int64_t)t * Cw + c];
        uint8_t ok = win_ok_tc[(int64_t)t * Cw + c];
        bool good = (ok != 0);
        good = good && (j >= 0) && (j < N);
        good = good && (mask_bn_u8[b * N + j] != 0);
        good = good && (!causal_only || (j <= i));
        good = good && (j != i);
        if (good) {
            cand_out_tc[base + win_s + c] = b * N + j;
            ok_out_tc[base + win_s + c] = 1;
        }
    }

    const int ring_s = Cself + Cw;
    for (int c = 0; c < Cr; ++c) {
        int j = ring_idx_tc[(int64_t)t * Cr + c];
        uint8_t ok = ring_ok_tc[(int64_t)t * Cr + c];
        bool good = (ok != 0);
        good = good && (j >= 0) && (j < N);
        good = good && (mask_bn_u8[b * N + j] != 0);
        good = good && (!causal_only || (j <= i));
        good = good && (j != i);
        if (good) {
            cand_out_tc[base + ring_s + c] = b * N + j;
            ok_out_tc[base + ring_s + c] = 1;
        }
    }

    const int anch_s = Cself + Cw + Cr;
    for (int c = 0; c < Ca; ++c) {
        int j = anch_idx_tc[(int64_t)t * Ca + c];
        uint8_t ok = anch_ok_tc[(int64_t)t * Ca + c];
        bool good = (ok != 0);
        good = good && (j >= 0) && (j < N);
        good = good && (mask_bn_u8[b * N + j] != 0);
        good = good && (!causal_only || (j <= i));
        good = good && (j != i);
        if (good) {
            cand_out_tc[base + anch_s + c] = b * N + j;
            ok_out_tc[base + anch_s + c] = 1;
        }
    }

    if (dedup_across_families) {
        for (int c = 0; c < Cmax; ++c) {
            if (!ok_out_tc[base + c]) continue;
            int g = cand_out_tc[base + c];
            bool dup = false;
            for (int p = 0; p < c; ++p) {
                if (ok_out_tc[base + p] && cand_out_tc[base + p] == g) {
                    dup = true;
                    break;
                }
            }
            if (dup) {
                cand_out_tc[base + c] = -1;
                ok_out_tc[base + c] = 0;
            }
        }
    }

    bool any_ok = false;
    for (int c = 0; c < Cmax; ++c) {
        if (ok_out_tc[base + c]) {
            any_ok = true;
            break;
        }
    }
    if (!any_ok && include_self) {
        cand_out_tc[base + 0] = b * N + i;
        ok_out_tc[base + 0] = 1;
        family_out_tc[base + 0] = FAM_SELF;
    }
}

__global__ void fill_stagea_meta_kernel(
    int32_t* __restrict__ family_bounds,      // [10]
    int32_t* __restrict__ family_order_caps,  // [5]
    int32_t* __restrict__ dispatch_meta,      // [10]
    int protocol_id,
    int layer_idx,
    int num_layers,
    int refresh_period,
    int Cself,
    int Cw,
    int Cr,
    int Ca,
    int Cmax
) {
    if (blockIdx.x != 0 || threadIdx.x != 0) return;

    family_bounds[0] = 0;
    family_bounds[1] = Cself;
    family_bounds[2] = Cself;
    family_bounds[3] = Cself + Cw;
    family_bounds[4] = Cself + Cw;
    family_bounds[5] = Cself + Cw + Cr;
    family_bounds[6] = Cself + Cw + Cr;
    family_bounds[7] = Cself + Cw + Cr + Ca;
    family_bounds[8] = Cself + Cw + Cr + Ca;
    family_bounds[9] = Cself + Cw + Cr + Ca;

    family_order_caps[0] = 0;
    family_order_caps[1] = 0;
    family_order_caps[2] = 0;
    family_order_caps[3] = 0;
    family_order_caps[4] = 0;

    if (protocol_id == PROTO_HYBRID || protocol_id == PROTO_MULTISCALE) {
        family_order_caps[1] = (Cw > 0) ? 1 : 0;
        family_order_caps[2] = (Cr > 0) ? 1 : 0;
        family_order_caps[3] = (Ca > 0) ? 1 : 0;
    } else if (protocol_id == PROTO_GLOBAL_REFRESH) {
        family_order_caps[1] = (Cw > 0) ? 1 : 0;
        family_order_caps[2] = (Cr > 0) ? 2 : 0;
        family_order_caps[3] = (Ca > 0) ? 2 : 0;
    }

    dispatch_meta[0] = protocol_id;
    dispatch_meta[1] = layer_idx;
    dispatch_meta[2] = num_layers;
    dispatch_meta[3] = refresh_period;
    dispatch_meta[4] = compute_stage_order_device(protocol_id, layer_idx, num_layers, refresh_period);
    dispatch_meta[5] = Cself;
    dispatch_meta[6] = Cw;
    dispatch_meta[7] = Cr;
    dispatch_meta[8] = Ca;
    dispatch_meta[9] = Cmax;
}

std::vector<torch::Tensor> build_candidates_spectral_family_end2end_cuda(
    torch::Tensor H_bnd,         // [B,N,D]
    torch::Tensor deficit_bn,    // [B,N]
    torch::Tensor attn_mask_bn,  // [B,N]

    int64_t block_size,
    int64_t reps_per_block,

    int64_t window_radius_tokens,
    int64_t ring_min_blocks,
    int64_t ring_max_blocks,

    int64_t Cw,
    int64_t Cr,
    int64_t Ca,

    bool include_self,
    bool causal_only,
    bool dedup_across_families,

    int64_t protocol_id,
    int64_t layer_idx,
    int64_t num_layers,
    int64_t refresh_period
) {
    CHECK_CUDA(H_bnd);
    CHECK_CUDA(deficit_bn);
    CHECK_CUDA(attn_mask_bn);
    CHECK_3D(H_bnd);
    CHECK_2D(deficit_bn);
    CHECK_2D(attn_mask_bn);

    TORCH_CHECK(deficit_bn.device() == H_bnd.device(),
                "deficit_bn must be on same CUDA device as H_bnd");
    TORCH_CHECK(attn_mask_bn.device() == H_bnd.device(),
                "attn_mask_bn must be on same CUDA device as H_bnd");
    c10::cuda::CUDAGuard device_guard(H_bnd.device());

    TORCH_CHECK(H_bnd.size(0) == deficit_bn.size(0) && H_bnd.size(1) == deficit_bn.size(1),
                "H_bnd and deficit_bn shape mismatch");
    TORCH_CHECK(H_bnd.size(0) == attn_mask_bn.size(0) && H_bnd.size(1) == attn_mask_bn.size(1),
                "H_bnd and attn_mask_bn shape mismatch");

    TORCH_CHECK(block_size > 0, "block_size must be > 0");
    TORCH_CHECK(block_size <= INT32_MAX, "block_size must fit int32");
    TORCH_CHECK(reps_per_block > 0 && reps_per_block <= MAX_REPS, "reps_per_block must be in [1, MAX_REPS]");
    TORCH_CHECK(Cw >= 0 && Cw <= MAX_CAP, "Cw must be in [0, MAX_CAP]");
    TORCH_CHECK(Cr >= 0 && Cr <= MAX_CAP, "Cr must be in [0, MAX_CAP]");
    TORCH_CHECK(Ca >= 0 && Ca <= MAX_CAP, "Ca must be in [0, MAX_CAP]");
    TORCH_CHECK(window_radius_tokens >= 0, "window_radius_tokens must be >= 0");
    TORCH_CHECK(window_radius_tokens <= INT32_MAX, "window_radius_tokens must fit int32");
    TORCH_CHECK(ring_min_blocks >= 1, "ring_min_blocks must be >= 1");
    TORCH_CHECK(ring_min_blocks <= INT32_MAX, "ring_min_blocks must fit int32");
    TORCH_CHECK(ring_max_blocks >= ring_min_blocks, "ring_max_blocks must be >= ring_min_blocks");
    TORCH_CHECK(ring_max_blocks <= INT32_MAX, "ring_max_blocks must fit int32");
    TORCH_CHECK(protocol_id >= 0 && protocol_id <= 3, "protocol_id must be in [0,3]");
    TORCH_CHECK(layer_idx >= 0 && layer_idx <= INT32_MAX, "layer_idx must fit int32");
    TORCH_CHECK(num_layers >= 0 && num_layers <= INT32_MAX, "num_layers must fit int32");
    TORCH_CHECK(refresh_period >= 0 && refresh_period <= INT32_MAX, "refresh_period must fit int32");

    const int64_t B64 = H_bnd.size(0);
    const int64_t N64 = H_bnd.size(1);
    const int64_t D64 = H_bnd.size(2);

    TORCH_CHECK(B64 > 0 && N64 > 0 && D64 > 0, "H_bnd must be non-empty");
    TORCH_CHECK(B64 <= INT32_MAX && N64 <= INT32_MAX && D64 <= INT32_MAX, "B,N,D must fit in int32");

    const int64_t T64 = B64 * N64;
    TORCH_CHECK(T64 > 0 && T64 <= INT32_MAX, "B*N must fit int32 because kernels use int t/T");

    const int64_t M64 = (N64 + block_size - 1) / block_size;
    TORCH_CHECK(M64 > 0 && M64 <= INT32_MAX, "M must fit int32");
    TORCH_CHECK(B64 * M64 <= INT32_MAX, "B*M must fit int32 because kernels use int bm");

    const int B = (int)B64;
    const int N = (int)N64;
    const int D = (int)D64;
    const int T = (int)T64;
    const int M = (int)M64;

    const int R = (int)reps_per_block;
    const int KanchorBlocks = (Ca > 0)
        ? std::min(M, (int)(((int64_t)Ca + (int64_t)R - 1) / (int64_t)R))
        : 0;

    auto H_f32 = H_bnd.to(torch::kFloat32).contiguous();
    auto deficit_f32 = deficit_bn.to(torch::kFloat32).contiguous();
    auto mask_u8 = attn_mask_bn.ne(0).to(torch::kUInt8).contiguous();

    CHECK_CONTIGUOUS(H_f32);
    CHECK_CONTIGUOUS(deficit_f32);
    CHECK_CONTIGUOUS(mask_u8);

    auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(H_bnd.device());
    auto opts_u8  = torch::TensorOptions().dtype(torch::kUInt8).device(H_bnd.device());
    auto opts_f32 = torch::TensorOptions().dtype(torch::kFloat32).device(H_bnd.device());

    // block-level structures
    torch::Tensor rep_idx_bmr = torch::empty({B, M, R}, opts_i32);
    torch::Tensor rep_ok_bmr  = torch::empty({B, M, R}, opts_u8);
    torch::Tensor block_mean_def_bm = torch::empty({B, M}, opts_f32);
    torch::Tensor block_valid_bm    = torch::empty({B, M}, opts_u8);

    torch::Tensor anchor_block_ids_bk;
    torch::Tensor anchor_block_ok_bk;
    if (KanchorBlocks > 0) {
        anchor_block_ids_bk = torch::empty({B, KanchorBlocks}, opts_i32);
        anchor_block_ok_bk  = torch::empty({B, KanchorBlocks}, opts_u8);
    } else {
        anchor_block_ids_bk = torch::empty({B, 1}, opts_i32);
        anchor_block_ok_bk  = torch::empty({B, 1}, opts_u8);
        anchor_block_ids_bk.zero_();
        anchor_block_ok_bk.zero_();
    }

    // precomputed family tables
    torch::Tensor win_idx_tc  = torch::empty({T, (int)Cw}, opts_i32);
    torch::Tensor win_ok_tc   = torch::empty({T, (int)Cw}, opts_u8);
    torch::Tensor ring_idx_tc = torch::empty({T, (int)Cr}, opts_i32);
    torch::Tensor ring_ok_tc  = torch::empty({T, (int)Cr}, opts_u8);
    torch::Tensor anch_idx_tc = torch::empty({T, (int)Ca}, opts_i32);
    torch::Tensor anch_ok_tc  = torch::empty({T, (int)Ca}, opts_u8);

    // final compiled outputs
    const int Cself = 1;
    const int Cmax = Cself + (int)Cw + (int)Cr + (int)Ca;

    torch::Tensor cand_out_tc  = torch::empty({T, Cmax}, opts_i32);
    torch::Tensor ok_out_tc    = torch::empty({T, Cmax}, opts_u8);
    torch::Tensor valid_T_t    = torch::empty({T}, opts_u8);
    torch::Tensor family_out_tc = torch::empty({T, Cmax}, opts_u8);

    torch::Tensor family_bounds = torch::empty({10}, opts_i32);
    torch::Tensor family_order_caps = torch::empty({5}, opts_i32);
    torch::Tensor dispatch_meta = torch::empty({10}, opts_i32);

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    // 1) select representative tokens per block
    {
        int threads = 128;
        int blocks = (int)(((int64_t)B * (int64_t)M + threads - 1) / threads);
        select_block_reps_kernel<<<blocks, threads, 0, stream>>>(
            mask_u8.data_ptr<uint8_t>(),
            deficit_f32.data_ptr<float>(),
            B, N, M, (int)block_size, R,
            rep_idx_bmr.data_ptr<int32_t>(),
            rep_ok_bmr.data_ptr<uint8_t>(),
            block_mean_def_bm.data_ptr<float>(),
            block_valid_bm.data_ptr<uint8_t>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 2) select anchor blocks globally per batch
    if (KanchorBlocks > 0) {
        select_anchor_blocks_kernel<<<B, 1, 0, stream>>>(
            H_f32.data_ptr<float>(),
            rep_idx_bmr.data_ptr<int32_t>(),
            rep_ok_bmr.data_ptr<uint8_t>(),
            block_mean_def_bm.data_ptr<float>(),
            block_valid_bm.data_ptr<uint8_t>(),
            B, N, D, M, R, KanchorBlocks,
            1.25f, 1.00f, 0.10f,  // alpha_def, beta_cent, gamma_boundary
            anchor_block_ids_bk.data_ptr<int32_t>(),
            anchor_block_ok_bk.data_ptr<uint8_t>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 3) build precomputed window/ring/anchor family tables
    {
        int threads = 128;
        int blocks = (int)(((int64_t)T + threads - 1) / threads);
        propose_family_candidates_kernel<<<blocks, threads, 0, stream>>>(
            H_f32.data_ptr<float>(),
            deficit_f32.data_ptr<float>(),
            mask_u8.data_ptr<uint8_t>(),

            rep_idx_bmr.data_ptr<int32_t>(),
            rep_ok_bmr.data_ptr<uint8_t>(),
            block_mean_def_bm.data_ptr<float>(),
            block_valid_bm.data_ptr<uint8_t>(),

            anchor_block_ids_bk.data_ptr<int32_t>(),
            anchor_block_ok_bk.data_ptr<uint8_t>(),

            B, N, D, M, (int)block_size, R, KanchorBlocks,

            (int)window_radius_tokens,
            (int)ring_min_blocks,
            (int)ring_max_blocks,

            (int)Cw,
            (int)Cr,
            (int)Ca,

            causal_only,

            1.00f, 1.00f, 0.25f,  // window weights: def, sim, dist
            1.00f, 1.00f, 0.30f,  // ring weights: def, sim, band
            1.00f, 1.00f, 0.50f,  // anchor weights: def, sim, far
            win_idx_tc.data_ptr<int32_t>(),
            win_ok_tc.data_ptr<uint8_t>(),
            ring_idx_tc.data_ptr<int32_t>(),
            ring_ok_tc.data_ptr<uint8_t>(),
            anch_idx_tc.data_ptr<int32_t>(),
            anch_ok_tc.data_ptr<uint8_t>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 4) compile into Stage-A style layout
    {
        int threads = 128;
        int blocks = (int)(((int64_t)T + threads - 1) / threads);
        compile_family_kernel<<<blocks, threads, 0, stream>>>(
            mask_u8.data_ptr<uint8_t>(),

            win_idx_tc.data_ptr<int32_t>(),
            win_ok_tc.data_ptr<uint8_t>(),
            (int)Cw,

            ring_idx_tc.data_ptr<int32_t>(),
            ring_ok_tc.data_ptr<uint8_t>(),
            (int)Cr,

            anch_idx_tc.data_ptr<int32_t>(),
            anch_ok_tc.data_ptr<uint8_t>(),
            (int)Ca,

            B, N,
            include_self,
            causal_only,
            dedup_across_families,

            Cself,
            Cmax,

            cand_out_tc.data_ptr<int32_t>(),
            ok_out_tc.data_ptr<uint8_t>(),
            valid_T_t.data_ptr<uint8_t>(),
            family_out_tc.data_ptr<uint8_t>()
        );
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    }

    // 5) metadata
    fill_stagea_meta_kernel<<<1, 1, 0, stream>>>(
        family_bounds.data_ptr<int32_t>(),
        family_order_caps.data_ptr<int32_t>(),
        dispatch_meta.data_ptr<int32_t>(),
        (int)protocol_id,
        (int)layer_idx,
        (int)num_layers,
        (int)refresh_period,
        Cself,
        (int)Cw,
        (int)Cr,
        (int)Ca,
        Cmax
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return {
        win_idx_tc,  win_ok_tc,
        ring_idx_tc, ring_ok_tc,
        anch_idx_tc, anch_ok_tc,
        cand_out_tc, ok_out_tc, valid_T_t, family_out_tc,
        family_bounds, family_order_caps, dispatch_meta
    };
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def(
        "build_candidates_spectral_family_end2end_cuda",
        &build_candidates_spectral_family_end2end_cuda,
        "End-to-end CUDA family proposal + compiler"
    );
}
"""

extra_cuda_cflags = ["-O3", "--use_fast_math"]
extra_cflags = ["-O3"]

_build_sig = "stageA_spectral_family_end2end_cuda_v1"
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
ext_name = f"stageA_spectral_family_end2end_ext_{_build_hash}"

stageA_spectral_family_end2end_ext = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=CUDA_SRC,
    functions=None,
    extra_cuda_cflags=extra_cuda_cflags,
    extra_cflags=extra_cflags,
    with_cuda=True,
    verbose=bool(FUSED_VERBOSE_BUILD),
)

build_candidates_spectral_family_end2end_cuda_raw = (
    stageA_spectral_family_end2end_ext.build_candidates_spectral_family_end2end_cuda
)

def build_candidates_spectral_family_end2end_cuda(
    H_bnd,
    deficit_bn,
    attn_mask_bn,
    *,
    block_size=64,
    reps_per_block=2,
    window_radius_tokens=32,
    ring_min_blocks=2,
    ring_max_blocks=8,
    Cw=32,
    Cr=16,
    Ca=16,
    include_self=True,
    causal_only=True,
    dedup_across_families=True,
    protocol_id=1,
    layer_idx=0,
    num_layers=1,
    refresh_period=4,
):
    return build_candidates_spectral_family_end2end_cuda_raw(
        H_bnd,
        deficit_bn,
        attn_mask_bn,
        int(block_size),
        int(reps_per_block),
        int(window_radius_tokens),
        int(ring_min_blocks),
        int(ring_max_blocks),
        int(Cw),
        int(Cr),
        int(Ca),
        bool(include_self),
        bool(causal_only),
        bool(dedup_across_families),
        int(protocol_id),
        int(layer_idx),
        int(num_layers),
        int(refresh_period),
    )

print("Built:", ext_name)

In [ ]:
def build_candidates_spectral_family_end2end_cuda(
    H_bnd,
    deficit_bn,
    attn_mask_bn,
    *,
    block_size=32,
    reps_per_block=2,
    window_radius_tokens=32,
    ring_min_blocks=2,
    ring_max_blocks=8,
    Cw=16,
    Cr=8,
    Ca=8,
    include_self=True,
    causal_only=True,
    dedup_across_families=True,
    protocol_id=1,   # 0=ECONOMY,1=HYBRID,2=MULTISCALE,3=GLOBAL_REFRESH
    layer_idx=0,
    num_layers=1,
    refresh_period=0,
):
    """
    Inputs:
      H_bnd        : [B,N,D]
      deficit_bn   : [B,N]
      attn_mask_bn : [B,N]

    Returns:
      precomputed_window_idx, precomputed_window_ok,
      precomputed_ring_idx,   precomputed_ring_ok,
      precomputed_anchor_idx, precomputed_anchor_ok,
      cand_out, ok_out, valid_T, family_out,
      family_bounds, family_order_caps, dispatch_meta
    """
    block_size = int(block_size)
    reps_per_block = int(reps_per_block)
    window_radius_tokens = int(window_radius_tokens)
    ring_min_blocks = int(ring_min_blocks)
    ring_max_blocks = int(ring_max_blocks)
    Cw = int(Cw)
    Cr = int(Cr)
    Ca = int(Ca)
    protocol_id = int(protocol_id)
    layer_idx = int(layer_idx)
    num_layers = int(num_layers)
    refresh_period = int(refresh_period)

    if block_size <= 0:
        raise ValueError("block_size must be > 0")
    if reps_per_block <= 0:
        raise ValueError("reps_per_block must be > 0")
    if window_radius_tokens < 0:
        raise ValueError("window_radius_tokens must be >= 0")
    if ring_min_blocks < 1:
        raise ValueError("ring_min_blocks must be >= 1")
    if ring_max_blocks < ring_min_blocks:
        raise ValueError("ring_max_blocks must be >= ring_min_blocks")
    if Cw < 0 or Cr < 0 or Ca < 0:
        raise ValueError("Cw, Cr, and Ca must be >= 0")
    if protocol_id not in (0, 1, 2, 3):
        raise ValueError("protocol_id must be one of {0,1,2,3}")
    if layer_idx < 0:
        raise ValueError("layer_idx must be >= 0")
    if num_layers < 0:
        raise ValueError("num_layers must be >= 0")
    if refresh_period < 0:
        raise ValueError("refresh_period must be >= 0")

    if not H_bnd.is_cuda or not deficit_bn.is_cuda or not attn_mask_bn.is_cuda:
        raise ValueError("H_bnd, deficit_bn, and attn_mask_bn must be CUDA tensors")

    if H_bnd.device != deficit_bn.device or H_bnd.device != attn_mask_bn.device:
        raise ValueError("H_bnd, deficit_bn, and attn_mask_bn must be on the same CUDA device")

    return build_candidates_spectral_family_end2end_cuda_raw(
        H_bnd,
        deficit_bn,
        attn_mask_bn,
        block_size,
        reps_per_block,
        window_radius_tokens,
        ring_min_blocks,
        ring_max_blocks,
        Cw,
        Cr,
        Ca,
        bool(include_self),
        bool(causal_only),
        bool(dedup_across_families),
        protocol_id,
        layer_idx,
        num_layers,
        refresh_period,
    )

In [ ]:
# ============================================================
# Stage-A FAMILY-AWARE + PROTOCOL-META Candidate Builder
# Resilient version: keeps old Stage-A safety pattern and
# adds only Stage-A metadata outputs for later Stage-E-LCHA use.
# ============================================================

import os, hashlib, torch
from torch.utils.cpp_extension import load_inline

FUSED_VERBOSE_BUILD = False

CUDA_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <vector>
#include <cstdint>
#include <algorithm>

#define CHECK_CUDA(x) TORCH_CHECK((x).is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK((x).is_contiguous(), #x " must be contiguous")
#define CHECK_2D(x) TORCH_CHECK((x).dim() == 2, #x " must be 2D [B,N]")

enum FamilyId : uint8_t {
    FAM_INVALID  = 0,
    FAM_SELF     = 1,
    FAM_WINDOW   = 2,
    FAM_RING     = 3,
    FAM_ANCHOR   = 4,
    FAM_PROMOTED = 5
};

enum ProtocolId : int32_t {
    PROTO_ECONOMY        = 0,
    PROTO_HYBRID         = 1,
    PROTO_MULTISCALE     = 2,
    PROTO_GLOBAL_REFRESH = 3
};

__device__ __forceinline__ int64_t row_base_tc(int t, int Cmax) {
    return (int64_t)t * (int64_t)Cmax;
}

__device__ __forceinline__ uint8_t family_code_for_slot(
    int c,
    int Cself,
    int Cw,
    int Cr,
    int Ca
) {
    const int self_s = 0;
    const int self_e = self_s + Cself;
    const int win_s  = self_e;
    const int win_e  = win_s + Cw;
    const int ring_s = win_e;
    const int ring_e = ring_s + Cr;
    const int anch_s = ring_e;
    const int anch_e = anch_s + Ca;

    if (c >= self_s && c < self_e) return FAM_SELF;
    if (c >= win_s  && c < win_e ) return FAM_WINDOW;
    if (c >= ring_s && c < ring_e) return FAM_RING;
    if (c >= anch_s && c < anch_e) return FAM_ANCHOR;
    return FAM_PROMOTED;
}

__device__ __forceinline__ int map_window_slot_to_offset(int c_local, int w) {
    // self excluded from window family
    // c_local in [0, 2w)
    if (c_local < w) return c_local - w;      // [-w .. -1]
    return (c_local - w) + 1;                 // [ +1 .. +w ]
}

__device__ __forceinline__ int compute_stage_order_device(
    int protocol_id,
    int layer_idx,
    int num_layers,
    int refresh_period
) {
    if (protocol_id == PROTO_ECONOMY) return 0;
    if (protocol_id == PROTO_HYBRID) return 1;
    if (protocol_id == PROTO_MULTISCALE) return 1;

    if (protocol_id == PROTO_GLOBAL_REFRESH) {
        int periodic_refresh = (refresh_period > 0) && ((layer_idx % refresh_period) == 0);
        int later_layer = (num_layers > 0) && (layer_idx >= (num_layers / 2));
        return (periodic_refresh || later_layer) ? 2 : 1;
    }
    return 0;
}

// ------------------------------------------------------------
// Main Stage-A kernel
// Emits per-row:
//   cand_out [T,Cmax]
//   ok_out   [T,Cmax]
//   valid_T  [T]
//   family_out [T,Cmax]
// ------------------------------------------------------------
__global__ void build_candidates_family_kernel_u8(
    const uint8_t* __restrict__ mask_bn_u8, // [B,N]
    int B,
    int N,
    int w,
    int ring_inner,
    int ring_outer,
    int mode,              // 0=window-only, 1=ring-only, 2=window+ring
    int A,                 // min(num_anchors, N)
    bool include_self,
    int anchor_strategy,   // 0=uniform,1=periodic,2=boundary,3=global_tokens
    int anchor_param0,
    int Cself,             // always 1
    int Cw,                // 2*w if window enabled else 0 (self excluded)
    int Cr,                // ring slots
    int Ca,                // anchor slots
    int Cmax,
    int32_t* __restrict__ cand_out,   // [T,Cmax]
    uint8_t* __restrict__ ok_out,     // [T,Cmax]
    uint8_t* __restrict__ valid_T,    // [T]
    uint8_t* __restrict__ family_out  // [T,Cmax]
) {
    int t = (int)blockIdx.x;
    int T = B * N;
    if (t >= T) return;

    int b = t / N;
    int i = t - b * N;

    __shared__ uint8_t qv;
    if (threadIdx.x == 0) {
        qv = mask_bn_u8[b * N + i] ? 1 : 0;
        valid_T[t] = qv;
    }
    __syncthreads();

    const uint8_t q_valid = qv;
    const int64_t base = row_base_tc(t, Cmax);

    // Always stamp family ids so downstream code sees fixed slot semantics.
    for (int c = threadIdx.x; c < Cmax; c += blockDim.x) {
        family_out[base + (int64_t)c] = family_code_for_slot(c, Cself, Cw, Cr, Ca);
    }
    __syncthreads();

    if (!q_valid) {
        for (int c = threadIdx.x; c < Cmax; c += blockDim.x) {
            cand_out[base + (int64_t)c] = -1;
            ok_out[base + (int64_t)c] = 0;
        }
        return;
    }

    int any_ok = 0;

    // SELF family: slot 0
    if (Cself > 0) {
        if (threadIdx.x == 0) {
            if (include_self && mask_bn_u8[b * N + i]) {
                cand_out[base + 0] = (int32_t)(b * N + i);
                ok_out[base + 0] = 1;
                any_ok = 1;
            } else {
                cand_out[base + 0] = -1;
                ok_out[base + 0] = 0;
            }
        }
    }

    // WINDOW family: [Cself .. Cself+Cw)
    if (Cw > 0) {
        const int win_s = Cself;
        for (int c_local = threadIdx.x; c_local < Cw; c_local += blockDim.x) {
            const int off = map_window_slot_to_offset(c_local, w);
            const int j = i + off;
            const int c = win_s + c_local;

            bool in_range = (j >= 0) && (j < N);
            bool causal   = (j <= i);
            bool self_ok  = (j != i);

            if (!in_range || !causal || !self_ok) {
                cand_out[base + (int64_t)c] = -1;
                ok_out[base + (int64_t)c] = 0;
            } else {
                uint8_t cand_valid = mask_bn_u8[b * N + j] ? 1 : 0;
                if (!cand_valid) {
                    cand_out[base + (int64_t)c] = -1;
                    ok_out[base + (int64_t)c] = 0;
                } else {
                    cand_out[base + (int64_t)c] = (int32_t)(b * N + j);
                    ok_out[base + (int64_t)c] = 1;
                    any_ok = 1;
                }
            }
        }
    }

    // RING family: [Cself+Cw .. Cself+Cw+Cr)
    if (Cr > 0) {
        const int ring_s = Cself + Cw;
        for (int c_local = threadIdx.x; c_local < Cr; c_local += blockDim.x) {
            const int d = ring_inner + c_local;
            const int j = i - d;
            const int c = ring_s + c_local;

            // suppress overlap with window when both enabled
            bool overlaps_window = (Cw > 0) && (d <= w);
            bool in_range = (j >= 0) && (j < N);
            bool causal   = true;
            bool self_ok  = (j != i);

            if (overlaps_window || !in_range || !causal || !self_ok) {
                cand_out[base + (int64_t)c] = -1;
                ok_out[base + (int64_t)c] = 0;
            } else {
                uint8_t cand_valid = mask_bn_u8[b * N + j] ? 1 : 0;
                if (!cand_valid) {
                    cand_out[base + (int64_t)c] = -1;
                    ok_out[base + (int64_t)c] = 0;
                } else {
                    cand_out[base + (int64_t)c] = (int32_t)(b * N + j);
                    ok_out[base + (int64_t)c] = 1;
                    any_ok = 1;
                }
            }
        }
    }

    // ANCHOR family: [Cself+Cw+Cr .. Cself+Cw+Cr+Ca)
    if (Ca > 0) {
        const int anch_s = Cself + Cw + Cr;

        int lo_win = i - w; if (lo_win < 0) lo_win = 0;
        int hi_win = i;     if (hi_win > (N - 1)) hi_win = N - 1;

        int lo_ring = i - ring_outer; if (lo_ring < 0) lo_ring = 0;
        int hi_ring = i - ring_inner; if (hi_ring < 0) hi_ring = -1;

        for (int a_idx = threadIdx.x; a_idx < Ca; a_idx += blockDim.x) {
            const int c = anch_s + a_idx;
            int a_local = 0;

            if (anchor_strategy == 1) {
                int p = (anchor_param0 > 0) ? anchor_param0 : (N / (A > 0 ? A : 1));
                if (p < 1) p = 1;
                a_local = a_idx * p;
                if (a_local > (N - 1)) a_local = (N - 1);
            } else if (anchor_strategy == 2) {
                int k = (anchor_param0 > 0) ? anchor_param0 : (A / 2);
                if (k < 0) k = 0;
                if (k > A) k = A;
                if (a_idx < k) a_local = a_idx;
                else a_local = N - (A - a_idx);
                if (a_local < 0) a_local = 0;
                if (a_local > (N - 1)) a_local = (N - 1);
            } else if (anchor_strategy == 3) {
                int start = (anchor_param0 > 0) ? anchor_param0 : 0;
                if (start < 0) start = 0;
                a_local = start + a_idx;
                if (a_local > (N - 1)) a_local = (N - 1);
            } else {
                a_local = (int)(((long long)a_idx * (long long)N) / (long long)A);
            }

            bool causal = (a_local <= i);
            bool self_ok = (a_local != i);
            bool in_window = (Cw > 0) && (a_local >= lo_win && a_local <= hi_win);
            bool in_ring   = (Cr > 0) && (a_local >= lo_ring && a_local <= hi_ring);

            if (!causal || !self_ok || in_window || in_ring) {
                cand_out[base + (int64_t)c] = -1;
                ok_out[base + (int64_t)c] = 0;
            } else {
                uint8_t cand_valid = mask_bn_u8[b * N + a_local] ? 1 : 0;
                if (!cand_valid) {
                    cand_out[base + (int64_t)c] = -1;
                    ok_out[base + (int64_t)c] = 0;
                } else {
                    cand_out[base + (int64_t)c] = (int32_t)(b * N + a_local);
                    ok_out[base + (int64_t)c] = 1;
                    any_ok = 1;
                }
            }
        }
    }

    // Guarantee at least one valid key
    int any_ok_all = __syncthreads_or(any_ok);
    if (any_ok_all == 0) {
        if (threadIdx.x == 0) {
            cand_out[base + 0] = (int32_t)(b * N + i);
            ok_out[base + 0] = 1;
            family_out[base + 0] = FAM_SELF;
        }
    }
}

// Metadata-only CUDA kernel
__global__ void fill_stagea_meta_kernel_i32(
    int32_t* __restrict__ family_bounds,      // [10]
    int32_t* __restrict__ family_order_caps,  // [5]
    int32_t* __restrict__ dispatch_meta,      // [10]
    int protocol_id,
    int layer_idx,
    int num_layers,
    int refresh_period,
    int Cself,
    int Cw,
    int Cr,
    int Ca,
    int Cmax
) {
    if (blockIdx.x != 0 || threadIdx.x != 0) return;

    // family_bounds = [self_s,self_e, win_s,win_e, ring_s,ring_e, anch_s,anch_e, prom_s,prom_e]
    family_bounds[0] = 0;
    family_bounds[1] = Cself;
    family_bounds[2] = Cself;
    family_bounds[3] = Cself + Cw;
    family_bounds[4] = Cself + Cw;
    family_bounds[5] = Cself + Cw + Cr;
    family_bounds[6] = Cself + Cw + Cr;
    family_bounds[7] = Cself + Cw + Cr + Ca;
    family_bounds[8] = Cself + Cw + Cr + Ca;
    family_bounds[9] = Cself + Cw + Cr + Ca;

    const int stage_order = compute_stage_order_device(protocol_id, layer_idx, num_layers, refresh_period);

    // family_order_caps = [self, window, ring, anchor, promoted]
    family_order_caps[0] = 0;
    family_order_caps[1] = 0;
    family_order_caps[2] = 0;
    family_order_caps[3] = 0;
    family_order_caps[4] = 0;

    if (protocol_id == PROTO_HYBRID || protocol_id == PROTO_MULTISCALE) {
        family_order_caps[1] = (Cw > 0) ? 1 : 0;
        family_order_caps[2] = (Cr > 0) ? 1 : 0;
        family_order_caps[3] = (Ca > 0) ? 1 : 0;
    } else if (protocol_id == PROTO_GLOBAL_REFRESH) {
        family_order_caps[1] = (Cw > 0) ? 1 : 0;
        family_order_caps[2] = (Cr > 0) ? 2 : 0;
        family_order_caps[3] = (Ca > 0) ? 2 : 0;
    }
    // ECONOMY stays all-zero

    // dispatch_meta = [protocol_id, layer_idx, num_layers, refresh_period, stage_order, Cself, Cw, Cr, Ca, Cmax]
    dispatch_meta[0] = protocol_id;
    dispatch_meta[1] = layer_idx;
    dispatch_meta[2] = num_layers;
    dispatch_meta[3] = refresh_period;
    dispatch_meta[4] = stage_order;
    dispatch_meta[5] = Cself;
    dispatch_meta[6] = Cw;
    dispatch_meta[7] = Cr;
    dispatch_meta[8] = Ca;
    dispatch_meta[9] = Cmax;
}

std::vector<torch::Tensor> build_candidates_family_cuda(
    torch::Tensor attn_mask_bn, // [B,N], any dtype, CUDA
    int64_t w,
    int64_t ring_inner,
    int64_t ring_outer,
    int64_t mode,               // 0=window-only, 1=ring-only, 2=window+ring
    int64_t num_anchors,
    bool include_self,
    int64_t anchor_strategy,
    int64_t anchor_param0,
    int64_t protocol_id,        // 0=ECONOMY,1=HYBRID,2=MULTISCALE,3=GLOBAL_REFRESH
    int64_t layer_idx,
    int64_t num_layers,
    int64_t refresh_period
) {
    CHECK_CUDA(attn_mask_bn);
    // CHECK_CONTIGUOUS(attn_mask_bn);
    CHECK_2D(attn_mask_bn);

    c10::cuda::CUDAGuard device_guard(attn_mask_bn.device());

    TORCH_CHECK(w >= 0, "w must be >= 0");
    TORCH_CHECK(num_anchors >= 0, "num_anchors must be >= 0");
    TORCH_CHECK(mode == 0 || mode == 1 || mode == 2, "mode must be 0 (window), 1 (ring), or 2 (window+ring)");
    TORCH_CHECK(protocol_id >= 0 && protocol_id <= 3, "protocol_id must be in [0,3]");
    TORCH_CHECK(include_self, "include_self must be true for causal scaffolds.");

    TORCH_CHECK(anchor_strategy >= 0 && anchor_strategy <= 3, "anchor_strategy must be in [0,3]");
    TORCH_CHECK(layer_idx >= 0, "layer_idx must be >= 0");
    TORCH_CHECK(num_layers >= 0, "num_layers must be >= 0");
    TORCH_CHECK(refresh_period >= 0, "refresh_period must be >= 0");

    const int64_t B64 = attn_mask_bn.size(0);
    const int64_t N64 = attn_mask_bn.size(1);
    TORCH_CHECK(B64 > 0 && N64 > 0, "attn_mask_bn must be non-empty");
    TORCH_CHECK(B64 <= INT32_MAX && N64 <= INT32_MAX, "B and N must fit in int32");

    // old hardening re-applied
    TORCH_CHECK(w <= N64, "w must be <= sequence length N");
    TORCH_CHECK(w <= (int64_t)INT32_MAX, "w too large for int32");
    TORCH_CHECK(ring_inner <= (int64_t)INT32_MAX, "ring_inner too large for int32");
    TORCH_CHECK(ring_outer <= (int64_t)INT32_MAX, "ring_outer too large for int32");
    TORCH_CHECK(mode <= (int64_t)INT32_MAX, "mode too large for int32");
    TORCH_CHECK(anchor_strategy <= (int64_t)INT32_MAX, "anchor_strategy too large for int32");
    TORCH_CHECK(anchor_param0 <= (int64_t)INT32_MAX, "anchor_param0 too large for int32");
    TORCH_CHECK(layer_idx <= (int64_t)INT32_MAX, "layer_idx too large for int32");
    TORCH_CHECK(num_layers <= (int64_t)INT32_MAX, "num_layers too large for int32");
    TORCH_CHECK(refresh_period <= (int64_t)INT32_MAX, "refresh_period too large for int32");

    int64_t Cself64 = 1;

    int64_t Cw64 = 0;
    if (mode == 0 || mode == 2) {
        Cw64 = 2 * w; // self excluded from window family
        TORCH_CHECK(Cw64 >= 0, "Cw overflow");
    }

    int64_t Cr64 = 0;
    if (mode == 1 || mode == 2) {
        TORCH_CHECK(ring_inner >= 1, "ring_inner must be >= 1 when ring is enabled");
        TORCH_CHECK(ring_outer >= ring_inner, "ring_outer must be >= ring_inner when ring is enabled");
        TORCH_CHECK(ring_outer <= (N64 - 1), "ring_outer must be <= N-1 when ring is enabled");
        Cr64 = (ring_outer - ring_inner + 1);
        TORCH_CHECK(Cr64 > 0, "Cr must be positive");
    } else {
        ring_inner = 0;
        ring_outer = 0;
    }

    if (mode == 1) {
        TORCH_CHECK(num_anchors > 0, "ring-only requires num_anchors > 0 to avoid empty candidate sets for early tokens.");
    }

    int64_t Ca64 = 0;
    if (num_anchors > 0) Ca64 = std::min<int64_t>(num_anchors, N64);

    const int64_t Cmax64 = Cself64 + Cw64 + Cr64 + Ca64;
    TORCH_CHECK(Cmax64 > 0, "Cmax must be positive");
    TORCH_CHECK(Cmax64 <= (int64_t)INT32_MAX, "Cmax must fit in int32");

    const int64_t T64 = B64 * N64;
    TORCH_CHECK(T64 <= INT32_MAX, "T=B*N must fit in int32");
    const int64_t total_elems = T64 * Cmax64;
    TORCH_CHECK(total_elems > 0, "T*Cmax must be positive");

    const int B = (int)B64;
    const int N = (int)N64;
    const int ww = (int)w;
    const int rin = (int)ring_inner;
    const int rout = (int)ring_outer;
    const int imod = (int)mode;
    const int Cself = (int)Cself64;
    const int Cw = (int)Cw64;
    const int Cr = (int)Cr64;
    const int Ca = (int)Ca64;
    const int Cmax = (int)Cmax64;
    const int T = (int)T64;

    auto mask_u8 = attn_mask_bn.ne(0).to(torch::kUInt8).contiguous();
    TORCH_CHECK(mask_u8.is_contiguous(), "mask_u8 must be contiguous");

    auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(attn_mask_bn.device());
    auto opts_u8  = torch::TensorOptions().dtype(torch::kUInt8).device(attn_mask_bn.device());

    torch::Tensor cand = torch::empty({T, Cmax}, opts_i32);
    torch::Tensor ok   = torch::empty({T, Cmax}, opts_u8);
    torch::Tensor valid_T = torch::empty({T}, opts_u8);
    torch::Tensor family = torch::empty({T, Cmax}, opts_u8);

    torch::Tensor family_bounds = torch::empty({10}, opts_i32);
    torch::Tensor family_order_caps = torch::empty({5}, opts_i32);
    torch::Tensor dispatch_meta = torch::empty({10}, opts_i32);

    int threads = 256;
    if (Cmax <= 128) threads = 128;
    if (Cmax <= 64)  threads = 64;

    const dim3 block(threads);
    const dim3 grid((unsigned int)T);
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    build_candidates_family_kernel_u8<<<grid, block, 0, stream>>>(
        (const uint8_t*)mask_u8.data_ptr<uint8_t>(),
        B,
        N,
        ww,
        rin,
        rout,
        imod,
        Ca,
        include_self,
        (int)anchor_strategy,
        (int)anchor_param0,
        Cself,
        Cw,
        Cr,
        Ca,
        Cmax,
        cand.data_ptr<int32_t>(),
        ok.data_ptr<uint8_t>(),
        valid_T.data_ptr<uint8_t>(),
        family.data_ptr<uint8_t>()
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    fill_stagea_meta_kernel_i32<<<1, 1, 0, stream>>>(
        family_bounds.data_ptr<int32_t>(),
        family_order_caps.data_ptr<int32_t>(),
        dispatch_meta.data_ptr<int32_t>(),
        (int)protocol_id,
        (int)layer_idx,
        (int)num_layers,
        (int)refresh_period,
        Cself,
        Cw,
        Cr,
        Ca,
        Cmax
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return {cand, ok, valid_T, family, family_bounds, family_order_caps, dispatch_meta};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def(
        "build_candidates_family_cuda",
        &build_candidates_family_cuda,
        "Stage-A family-aware candidates + protocol-aware metadata (CUDA)"
    );
}
"""

extra_cuda_cflags = ["-O3", "--use_fast_math"]
extra_cflags = ["-O3"]

_build_sig = "stageA_family_proto_meta_v3_resilient_cuda"
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
ext_name = f"stageA_family_proto_meta_ext_{_build_hash}"

stageA_family_proto_meta_ext = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=CUDA_SRC,
    functions=None,
    extra_cuda_cflags=extra_cuda_cflags,
    extra_cflags=extra_cflags,
    with_cuda=True,
    verbose=bool(FUSED_VERBOSE_BUILD),
)

build_candidates_family_cuda_raw = stageA_family_proto_meta_ext.build_candidates_family_cuda

def build_candidates_family_cuda(
    attn_mask_bn,
    w,
    ring_inner,
    ring_outer,
    mode,
    num_anchors,
    include_self,
    anchor_strategy=0,
    anchor_param0=0,
    protocol_id=0,      # 0=ECONOMY,1=HYBRID,2=MULTISCALE,3=GLOBAL_REFRESH
    layer_idx=0,
    num_layers=0,
    refresh_period=0,
):
    if not include_self:
        raise ValueError("include_self must be True for this Stage-A causal scaffold builder.")

    return build_candidates_family_cuda_raw(
        attn_mask_bn,
        int(w),
        int(ring_inner),
        int(ring_outer),
        int(mode),
        int(num_anchors),
        bool(include_self),
        int(anchor_strategy),
        int(anchor_param0),
        int(protocol_id),
        int(layer_idx),
        int(num_layers),
        int(refresh_period),
    )

print("Built:", ext_name)

\**Stage-B-MH projector + RoPE + Stage-E attention**


In [ ]:
import os, shutil
import torch
from torch.utils.cpp_extension import load_inline

import hashlib

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# 2) Better C++ stack traces when extensions throw
os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "1"

# Toggle these as needed:
FUSED_ENABLE_HOST_LOGS   = False    # C++ host-side logs via std::cout
FUSED_ENABLE_DEVICE_LOGS = False   # CUDA printf inside kernel
FUSED_SYNC_AFTER_KERNEL  = False    # device sync after kernel for accurate failure location
FUSED_VERBOSE_BUILD      = False    # verbose compile logs

# V-tiling toggle
FUSED_USE_SMEM_V         = True    # introduce V-tiling

STAGEB_MH_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cuda_fp16.h>   // <-- ADDED (for half)
#include <cuda_bf16.h>
#include <c10/cuda/CUDAGuard.h>
#include <vector>
#include <stdint.h>
#include <cublas_v2.h>
#include <limits>

#define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be CUDA")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_DTYPE(x, dt) TORCH_CHECK(x.scalar_type() == dt, #x " wrong dtype")

// -----------------------------------------
// dtype helpers
// -----------------------------------------
template<typename T> __device__ __forceinline__ float to_f32(T x);
template<> __device__ __forceinline__ float to_f32<float>(float x){ return x; }
template<> __device__ __forceinline__ float to_f32<half>(half x){ return __half2float(x); }
template<> __device__ __forceinline__ float to_f32<__nv_bfloat16>(__nv_bfloat16 x){
#if defined(__CUDA_ARCH__) && (__CUDA_ARCH__ < 800)
    asm("trap;");
#endif
    return __bfloat162float(x);
}

template<typename T> __device__ __forceinline__ T from_f32(float x);
template<> __device__ __forceinline__ float from_f32<float>(float x){ return x; }
template<> __device__ __forceinline__ half from_f32<half>(float x){ return __float2half_rn(x); }
template<> __device__ __forceinline__ __nv_bfloat16 from_f32<__nv_bfloat16>(float x){
#if defined(__CUDA_ARCH__) && (__CUDA_ARCH__ < 800)
    asm("trap;");
#endif
    return __float2bfloat16_rn(x);
}

// -----------------------------------------
// RoPE (forward/backward) on [T,H,D]
// We assume full rotary on D (head_dim), and D is even.
// cos/sin are float tensors shaped [T, D/2] (token-major).
// -----------------------------------------
template<typename scalar_t>
__global__ void rope_fwd_kernel(
    scalar_t* __restrict__ Q,   // [T,H,D]
    scalar_t* __restrict__ K,   // [T,H,D]
    const float* __restrict__ cos, // [T, D/2]
    const float* __restrict__ sin, // [T, D/2]
    int T, int H, int D
){
    int t = (int)blockIdx.x;
    int h = (int)blockIdx.y;
    int tid = (int)threadIdx.x;
    if (t >= T || h >= H) return;

    int halfD = D >> 1;
    // each thread handles multiple pairs
    for (int i = tid; i < halfD; i += (int)blockDim.x){
        int d0 = i;
        int d1 = i + halfD;

        float c = cos[t * halfD + i];
        float s = sin[t * halfD + i];

        // Q
        scalar_t* q = Q + ((int64_t)t * H + h) * D;
        float q0 = to_f32<scalar_t>(q[d0]);
        float q1 = to_f32<scalar_t>(q[d1]);
        float rq0 = q0 * c - q1 * s;
        float rq1 = q0 * s + q1 * c;
        q[d0] = from_f32<scalar_t>(rq0);
        q[d1] = from_f32<scalar_t>(rq1);

        // K
        scalar_t* k = K + ((int64_t)t * H + h) * D;
        float k0 = to_f32<scalar_t>(k[d0]);
        float k1 = to_f32<scalar_t>(k[d1]);
        float rk0 = k0 * c - k1 * s;
        float rk1 = k0 * s + k1 * c;
        k[d0] = from_f32<scalar_t>(rk0);
        k[d1] = from_f32<scalar_t>(rk1);
    }
}

template<typename scalar_t>
__global__ void rope_bwd_kernel(
    scalar_t* __restrict__ dQ,  // [T,H,D]
    scalar_t* __restrict__ dK,  // [T,H,D]
    const float* __restrict__ cos, // [T, D/2]
    const float* __restrict__ sin, // [T, D/2]
    int T, int H, int D
){
    // Backward for rotation:
    // [x'; y'] = [ x*c - y*s ; x*s + y*c ]
    // gradient wrt [x;y] given [dx';dy']:
    // dx = dx'*c + dy'*s
    // dy = -dx'*s + dy'*c
    int t = (int)blockIdx.x;
    int h = (int)blockIdx.y;
    int tid = (int)threadIdx.x;
    if (t >= T || h >= H) return;

    int halfD = D >> 1;
    for (int i = tid; i < halfD; i += (int)blockDim.x){
        int d0 = i;
        int d1 = i + halfD;

        float c = cos[t * halfD + i];
        float s = sin[t * halfD + i];

        scalar_t* dq = dQ + ((int64_t)t * H + h) * D;
        float dxp = to_f32<scalar_t>(dq[d0]);
        float dyp = to_f32<scalar_t>(dq[d1]);
        float dx  = dxp * c + dyp * s;
        float dy  = -dxp * s + dyp * c;
        dq[d0] = from_f32<scalar_t>(dx);
        dq[d1] = from_f32<scalar_t>(dy);

        scalar_t* dk = dK + ((int64_t)t * H + h) * D;
        float kxp = to_f32<scalar_t>(dk[d0]);
        float kyp = to_f32<scalar_t>(dk[d1]);
        float kx  = kxp * c + kyp * s;
        float ky  = -kxp * s + kyp * c;
        dk[d0] = from_f32<scalar_t>(kx);
        dk[d1] = from_f32<scalar_t>(ky);
    }
}

// -----------------------------------------
// GEMM helpers (cuBLAS, float32 path)
//
// We keep the wrapper policy the same as your current code:
//   - cast inputs/grads to FP32
//   - run GEMMs in FP32
//   - keep outputs/grads in FP32
//
// Row-major helpers:
//
// 1) C[T,O]   = A[T,Dm]   @ B[O,Dm]^T
// 2) C[O,Dm]  = A[T,O]^T  @ B[T,Dm]
// 3) C[T,Dm]  = A[T,O]    @ B[O,Dm]
//
// Implemented via cuBLAS column-major mapping.
// -----------------------------------------
#define CUBLAS_CHECK(stmt) \
    do { \
        cublasStatus_t _s = (stmt); \
        TORCH_CHECK(_s == CUBLAS_STATUS_SUCCESS, #stmt " failed with cuBLAS status ", (int)_s); \
    } while (0)

static inline void check_same_device(const torch::Tensor& a, const torch::Tensor& b,
                                     const char* an, const char* bn) {
    TORCH_CHECK(a.device() == b.device(), an, " and ", bn, " must be on the same device");
}

struct CublasHandleGuard {
    cublasHandle_t h;
    CublasHandleGuard() : h(nullptr) {
        CUBLAS_CHECK(cublasCreate(&h));
    }
    ~CublasHandleGuard() {
        if (h) {
            cublasDestroy(h);
            h = nullptr;
        }
    }
    operator cublasHandle_t() const { return h; }
};

static inline void gemm_rowmajor_A_Bt_f32(
    cublasHandle_t handle,
    const float* A,   // row-major [T, Dm]
    const float* B,   // row-major [O, Dm]
    float* C,         // row-major [T, O]
    int T, int Dm, int O,
    cudaStream_t stream
){
    TORCH_CHECK(T > 0 && Dm > 0 && O > 0, "gemm_rowmajor_A_Bt_f32: invalid dims");
    CUBLAS_CHECK(cublasSetStream(handle, stream));

    // Row-major mapping:
    // C_row(T,O) = A_row(T,Dm) @ B_row(O,Dm)^T
    // <=> C_col(O,T) = B_col(Dm,O)^T @ A_col(Dm,T)
    const float alpha = 1.0f;
    const float beta  = 0.0f;

    CUBLAS_CHECK(cublasSgemm(
        handle,
        CUBLAS_OP_T, CUBLAS_OP_N,
        O, T, Dm,
        &alpha,
        B, Dm,   // B memory interpreted as col-major [Dm, O], opT -> [O, Dm]
        A, Dm,   // A memory interpreted as col-major [Dm, T], opN -> [Dm, T]
        &beta,
        C, O     // C memory interpreted as col-major [O, T] => row-major [T, O]
    ));
}

static inline void gemm_rowmajor_At_B_f32(
    cublasHandle_t handle,
    const float* A,   // row-major [T, O]
    const float* B,   // row-major [T, Dm]
    float* C,         // row-major [O, Dm]
    int T, int O, int Dm,
    cudaStream_t stream
){
    TORCH_CHECK(T > 0 && O > 0 && Dm > 0, "gemm_rowmajor_At_B_f32: invalid dims");
    CUBLAS_CHECK(cublasSetStream(handle, stream));

    // Row-major mapping:
    // C_row(O,Dm) = A_row(T,O)^T @ B_row(T,Dm)
    // <=> C_col(Dm,O) = B_col(Dm,T) @ A_col(O,T)^T
    const float alpha = 1.0f;
    const float beta  = 0.0f;

    CUBLAS_CHECK(cublasSgemm(
        handle,
        CUBLAS_OP_N, CUBLAS_OP_T,
        Dm, O, T,
        &alpha,
        B, Dm,   // B memory interpreted as col-major [Dm, T], opN -> [Dm, T]
        A, O,    // A memory interpreted as col-major [O, T], opT -> [T, O]
        &beta,
        C, Dm    // C memory interpreted as col-major [Dm, O] => row-major [O, Dm]
    ));
}

static inline void gemm_rowmajor_A_B_f32(
    cublasHandle_t handle,
    const float* A,   // row-major [T, O]
    const float* B,   // row-major [O, Dm]
    float* C,         // row-major [T, Dm]
    int T, int O, int Dm,
    cudaStream_t stream,
    float beta = 0.0f
){
    TORCH_CHECK(T > 0 && O > 0 && Dm > 0, "gemm_rowmajor_A_B_f32: invalid dims");
    CUBLAS_CHECK(cublasSetStream(handle, stream));

    // Row-major mapping:
    // C_row(T,Dm) = A_row(T,O) @ B_row(O,Dm)
    // <=> C_col(Dm,T) = B_col(Dm,O) @ A_col(O,T)
    const float alpha = 1.0f;

    CUBLAS_CHECK(cublasSgemm(
        handle,
        CUBLAS_OP_N, CUBLAS_OP_N,
        Dm, T, O,
        &alpha,
        B, Dm,   // B memory interpreted as col-major [Dm, O], opN -> [Dm, O]
        A, O,    // A memory interpreted as col-major [O, T], opN -> [O, T]
        &beta,
        C, Dm    // C memory interpreted as col-major [Dm, T] => row-major [T, Dm]
    ));
}

// -----------------------------------------
// Forward/Backward interface
// Fully cuBLAS-backed FP32 projection/grads.
// -----------------------------------------

std::vector<torch::Tensor> stageb_mh_project_forward(
    torch::Tensor H,   // [T, Dm]
    torch::Tensor Wq,  // [H*D, Dm]
    torch::Tensor Wk,  // [H*D, Dm]
    torch::Tensor Wv,  // [H*DV, Dm]
    torch::Tensor cos, // [T, D/2] float
    torch::Tensor sin  // [T, D/2] float
){
    CHECK_CUDA(H); CHECK_CUDA(Wq); CHECK_CUDA(Wk); CHECK_CUDA(Wv);
    CHECK_CONTIGUOUS(H); CHECK_CONTIGUOUS(Wq); CHECK_CONTIGUOUS(Wk); CHECK_CONTIGUOUS(Wv);
    CHECK_CUDA(cos); CHECK_CUDA(sin);
    CHECK_CONTIGUOUS(cos); CHECK_CONTIGUOUS(sin);
    TORCH_CHECK(cos.scalar_type() == torch::kFloat && sin.scalar_type() == torch::kFloat,
                "cos/sin must be float32");

    TORCH_CHECK(H.dim() == 2,  "H must be [T, Dm]");
    TORCH_CHECK(Wq.dim() == 2 && Wk.dim() == 2 && Wv.dim() == 2,
                "Wq/Wk/Wv must be 2D");
    TORCH_CHECK(cos.dim() == 2 && sin.dim() == 2, "cos/sin must be 2D");

    c10::cuda::CUDAGuard guard(H.device());
    check_same_device(H, Wq, "H", "Wq");
    check_same_device(H, Wk, "H", "Wk");
    check_same_device(H, Wv, "H", "Wv");
    check_same_device(H, cos, "H", "cos");
    check_same_device(H, sin, "H", "sin");

    TORCH_CHECK(H.is_floating_point() && Wq.is_floating_point() &&
                Wk.is_floating_point() && Wv.is_floating_point(),
                "H/Wq/Wk/Wv must be floating point");

    const int64_t T64  = H.size(0);
    const int64_t Dm64 = H.size(1);
    TORCH_CHECK(T64 > 0 && Dm64 > 0, "H must be non-empty");

    TORCH_CHECK(Wq.size(1) == Dm64 && Wk.size(1) == Dm64 && Wv.size(1) == Dm64,
                "Wq/Wk/Wv second dim must match Dm");

    const int64_t qk_out64 = Wq.size(0);
    const int64_t v_out64  = Wv.size(0);
    TORCH_CHECK(qk_out64 == Wk.size(0), "Wq/Wk out dims mismatch");
    TORCH_CHECK(qk_out64 > 0 && v_out64 > 0, "invalid out dims");

    const int64_t halfD64 = cos.size(1);
    TORCH_CHECK(halfD64 > 0, "cos/sin halfD must be > 0");
    TORCH_CHECK(halfD64 <= std::numeric_limits<int64_t>::max() / 2,
                "cos/sin halfD too large; D64 would overflow int64");
    const int64_t D64 = halfD64 * 2;
    TORCH_CHECK(cos.size(0) == T64 && sin.size(0) == T64 && sin.size(1) == halfD64,
                "cos/sin shape must be [T, D/2]");
    TORCH_CHECK(D64 > 0 && (D64 % 2 == 0), "D must be positive and even");
    TORCH_CHECK(qk_out64 % D64 == 0, "Wq out dim must be multiple of head_dim D");

    const int64_t Hh64 = qk_out64 / D64;
    TORCH_CHECK(Hh64 > 0, "number of heads inferred as <= 0");
    TORCH_CHECK(v_out64 % Hh64 == 0, "Wv out dim must be multiple of n_heads");

    const int64_t DV64 = v_out64 / Hh64;

    TORCH_CHECK(T64  <= (int64_t)std::numeric_limits<int>::max(), "T too large");
    TORCH_CHECK(Dm64 <= (int64_t)std::numeric_limits<int>::max(), "Dm too large");
    TORCH_CHECK(D64  <= (int64_t)std::numeric_limits<int>::max(), "D too large");
    TORCH_CHECK(Hh64 <= (int64_t)std::numeric_limits<int>::max(), "Hh too large");
    TORCH_CHECK(DV64 <= (int64_t)std::numeric_limits<int>::max(), "DV too large");
    TORCH_CHECK(qk_out64 <= (int64_t)std::numeric_limits<int>::max(), "qk_out too large");
    TORCH_CHECK(v_out64  <= (int64_t)std::numeric_limits<int>::max(), "v_out too large");

    const int T   = (int)T64;
    const int Dm  = (int)Dm64;
    const int D   = (int)D64;
    const int Hh  = (int)Hh64;
    const int DV  = (int)DV64;
    const int qk_out = (int)qk_out64;
    const int v_out  = (int)v_out64;

    auto Hf  = H.to(torch::kFloat).contiguous();
    auto Wqf = Wq.to(torch::kFloat).contiguous();
    auto Wkf = Wk.to(torch::kFloat).contiguous();
    auto Wvf = Wv.to(torch::kFloat).contiguous();

    auto Qflat = torch::empty({T64, qk_out64}, Hf.options());
    auto Kflat = torch::empty({T64, qk_out64}, Hf.options());
    auto Vflat = torch::empty({T64, v_out64},  Hf.options());

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();
    CublasHandleGuard handle;

    gemm_rowmajor_A_Bt_f32(handle, (const float*)Hf.data_ptr(), (const float*)Wqf.data_ptr(),
                           (float*)Qflat.data_ptr(), T, Dm, qk_out, stream);
    gemm_rowmajor_A_Bt_f32(handle, (const float*)Hf.data_ptr(), (const float*)Wkf.data_ptr(),
                           (float*)Kflat.data_ptr(), T, Dm, qk_out, stream);
    gemm_rowmajor_A_Bt_f32(handle, (const float*)Hf.data_ptr(), (const float*)Wvf.data_ptr(),
                           (float*)Vflat.data_ptr(), T, Dm, v_out, stream);

    auto Q = Qflat.view({T64, Hh64, D64}).contiguous();
    auto K = Kflat.view({T64, Hh64, D64}).contiguous();
    auto V = Vflat.view({T64, Hh64, DV64}).contiguous();

    const dim3 grid(T, Hh);
    const dim3 block(128);

    rope_fwd_kernel<float><<<grid, block, 0, stream>>>(
        (float*)Q.data_ptr(), (float*)K.data_ptr(),
        (const float*)cos.data_ptr(), (const float*)sin.data_ptr(),
        T, Hh, D
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    return {Q, K, V};
}

std::vector<torch::Tensor> stageb_mh_project_backward(
    torch::Tensor dQ,  // [T,H,D]
    torch::Tensor dK,  // [T,H,D]
    torch::Tensor dV,  // [T,H,DV]
    torch::Tensor H,   // [T,Dm]
    torch::Tensor Wq,  // [H*D, Dm]
    torch::Tensor Wk,  // [H*D, Dm]
    torch::Tensor Wv,  // [H*DV, Dm]
    torch::Tensor cos, // [T, D/2] float
    torch::Tensor sin  // [T, D/2] float
){
    CHECK_CUDA(dQ); CHECK_CUDA(dK); CHECK_CUDA(dV);
    CHECK_CUDA(H);  CHECK_CUDA(Wq); CHECK_CUDA(Wk); CHECK_CUDA(Wv);
    CHECK_CUDA(cos); CHECK_CUDA(sin);

    CHECK_CONTIGUOUS(dQ); CHECK_CONTIGUOUS(dK); CHECK_CONTIGUOUS(dV);
    CHECK_CONTIGUOUS(H);  CHECK_CONTIGUOUS(Wq); CHECK_CONTIGUOUS(Wk); CHECK_CONTIGUOUS(Wv);
    CHECK_CONTIGUOUS(cos); CHECK_CONTIGUOUS(sin);

    TORCH_CHECK(cos.scalar_type() == torch::kFloat && sin.scalar_type() == torch::kFloat,
                "cos/sin must be float32");

    TORCH_CHECK(dQ.dim() == 3 && dK.dim() == 3 && dV.dim() == 3,
                "dQ/dK/dV must be [T,H,D] / [T,H,D] / [T,H,DV]");
    TORCH_CHECK(H.dim() == 2, "H must be [T, Dm]");
    TORCH_CHECK(Wq.dim() == 2 && Wk.dim() == 2 && Wv.dim() == 2,
                "Wq/Wk/Wv must be 2D");
    TORCH_CHECK(cos.dim() == 2 && sin.dim() == 2, "cos/sin must be 2D");

    c10::cuda::CUDAGuard guard(H.device());
    check_same_device(dQ, dK, "dQ", "dK");
    check_same_device(dQ, dV, "dQ", "dV");
    check_same_device(H, dQ, "H", "dQ");
    check_same_device(H, Wq, "H", "Wq");
    check_same_device(H, Wk, "H", "Wk");
    check_same_device(H, Wv, "H", "Wv");
    check_same_device(H, cos, "H", "cos");
    check_same_device(H, sin, "H", "sin");

    TORCH_CHECK(dQ.is_floating_point() && dK.is_floating_point() && dV.is_floating_point() &&
                H.is_floating_point()  && Wq.is_floating_point() && Wk.is_floating_point() && Wv.is_floating_point(),
                "dQ/dK/dV/H/Wq/Wk/Wv must be floating point");

    const int64_t T64  = H.size(0);
    const int64_t Dm64 = H.size(1);
    TORCH_CHECK(T64 > 0 && Dm64 > 0, "H must be non-empty");

    TORCH_CHECK(Wq.size(1) == Dm64 && Wk.size(1) == Dm64 && Wv.size(1) == Dm64,
                "Wq/Wk/Wv second dim must match Dm");

    TORCH_CHECK(dQ.size(0) == T64 && dK.size(0) == T64 && dV.size(0) == T64,
                "dQ/dK/dV T mismatch");

    const int64_t Hh64 = dQ.size(1);
    const int64_t D64  = dQ.size(2);
    const int64_t DV64 = dV.size(2);

    TORCH_CHECK(Hh64 > 0 && D64 > 0 && DV64 > 0, "invalid dQ/dV dims");
    TORCH_CHECK((D64 % 2) == 0, "head dim D must be even for RoPE");
    TORCH_CHECK(dK.size(1) == Hh64 && dK.size(2) == D64, "dK shape mismatch");
    TORCH_CHECK(dV.size(1) == Hh64, "dV shape mismatch");
    TORCH_CHECK(Hh64 <= std::numeric_limits<int64_t>::max() / D64,
                "Hh*D would overflow int64");
    const int64_t qk_out64 = Hh64 * D64;
    TORCH_CHECK(Hh64 <= std::numeric_limits<int64_t>::max() / DV64,
                "Hh*DV would overflow int64");
    const int64_t v_out64 = Hh64 * DV64;
    TORCH_CHECK(Wq.size(0) == qk_out64 && Wk.size(0) == qk_out64 && Wv.size(0) == v_out64,
                "W dims mismatch");
    TORCH_CHECK(cos.size(0) == T64 && sin.size(0) == T64 &&
                cos.size(1) == (D64 >> 1) && sin.size(1) == (D64 >> 1),
                "cos/sin shape must be [T, D/2] matching dQ head dim");

    TORCH_CHECK(T64  <= (int64_t)std::numeric_limits<int>::max(), "T too large");
    TORCH_CHECK(Dm64 <= (int64_t)std::numeric_limits<int>::max(), "Dm too large");
    TORCH_CHECK(Hh64 <= (int64_t)std::numeric_limits<int>::max(), "Hh too large");
    TORCH_CHECK(D64  <= (int64_t)std::numeric_limits<int>::max(), "D too large");
    TORCH_CHECK(DV64 <= (int64_t)std::numeric_limits<int>::max(), "DV too large");
    TORCH_CHECK(qk_out64 <= (int64_t)std::numeric_limits<int>::max(), "Hh*D too large");
    TORCH_CHECK(v_out64  <= (int64_t)std::numeric_limits<int>::max(), "Hh*DV too large");

    const int T   = (int)T64;
    const int Dm  = (int)Dm64;
    const int Hh  = (int)Hh64;
    const int D   = (int)D64;
    const int DV  = (int)DV64;
    const int qk_out = (int)qk_out64;
    const int v_out  = (int)v_out64;

    auto Hf  = H.to(torch::kFloat).contiguous();
    auto Wqf = Wq.to(torch::kFloat).contiguous();
    auto Wkf = Wk.to(torch::kFloat).contiguous();
    auto Wvf = Wv.to(torch::kFloat).contiguous();

    auto dQf = dQ.to(torch::kFloat).contiguous();
    auto dKf = dK.to(torch::kFloat).contiguous();
    auto dVf = dV.to(torch::kFloat).contiguous();

    // do not mutate caller-provided grads
    auto dQ_work = dQf.clone();
    auto dK_work = dKf.clone();

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    const dim3 grid(T, Hh);
    const dim3 block(128);
    rope_bwd_kernel<float><<<grid, block, 0, stream>>>(
        (float*)dQ_work.data_ptr(), (float*)dK_work.data_ptr(),
        (const float*)cos.data_ptr(), (const float*)sin.data_ptr(),
        T, Hh, D
    );
    C10_CUDA_KERNEL_LAUNCH_CHECK();

    auto dQflat = dQ_work.reshape({T64, (int64_t)qk_out}).contiguous();
    auto dKflat = dK_work.reshape({T64, (int64_t)qk_out}).contiguous();
    auto dVflat = dVf.reshape({T64, (int64_t)v_out}).contiguous();

    auto dWq = torch::empty({(int64_t)qk_out, Dm64}, Hf.options());
    auto dWk = torch::empty({(int64_t)qk_out, Dm64}, Hf.options());
    auto dWv = torch::empty({(int64_t)v_out,  Dm64}, Hf.options());
    auto dH  = torch::empty({T64, Dm64}, Hf.options());

    CublasHandleGuard handle;

    // dW = dOut^T @ H
    gemm_rowmajor_At_B_f32(handle, (const float*)dQflat.data_ptr(), (const float*)Hf.data_ptr(),
                           (float*)dWq.data_ptr(), T, qk_out, Dm, stream);
    gemm_rowmajor_At_B_f32(handle, (const float*)dKflat.data_ptr(), (const float*)Hf.data_ptr(),
                           (float*)dWk.data_ptr(), T, qk_out, Dm, stream);
    gemm_rowmajor_At_B_f32(handle, (const float*)dVflat.data_ptr(), (const float*)Hf.data_ptr(),
                           (float*)dWv.data_ptr(), T, v_out, Dm, stream);

    // dH = dQ @ Wq + dK @ Wk + dV @ Wv
    gemm_rowmajor_A_B_f32(handle, (const float*)dQflat.data_ptr(), (const float*)Wqf.data_ptr(),
                          (float*)dH.data_ptr(), T, qk_out, Dm, stream, 0.0f);
    gemm_rowmajor_A_B_f32(handle, (const float*)dKflat.data_ptr(), (const float*)Wkf.data_ptr(),
                          (float*)dH.data_ptr(), T, qk_out, Dm, stream, 1.0f);
    gemm_rowmajor_A_B_f32(handle, (const float*)dVflat.data_ptr(), (const float*)Wvf.data_ptr(),
                          (float*)dH.data_ptr(), T, v_out, Dm, stream, 1.0f);

    return {dH, dWq, dWk, dWv};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("proj_forward",  &stageb_mh_project_forward,  "StageB-MH project forward (QKV + RoPE)");
    m.def("proj_backward", &stageb_mh_project_backward, "StageB-MH project backward (RoPE-bwd + GEMM grads)");
}
"""

# ext_name = "stageB_MH_projector_rope_stageE_bridge_ext"
import hashlib

# ---- build-hash (forces rebuild when code / flags / linkage change) ----
_src_hash = hashlib.md5(STAGEB_MH_SRC.encode("utf-8")).hexdigest()[:10]
_build_sig = (
    "stageB_MH_projector_rope_stageE_bridge|"
    "O3_fastmath|"
    "cxx17|"
    "cublas|"
    f"src={_src_hash}"
)
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
ext_name = f"stageB_MH_projector_rope_stageE_bridge_ext_{_build_hash}"

stageB_MH_projector_rope_stageE_bridge = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=STAGEB_MH_SRC,
    functions=None,          # we use PYBIND11_MODULE
    extra_cuda_cflags=["-O3", "--use_fast_math"],
    extra_cflags=["-O3", "-std=c++17"],
    with_cuda=True,
    verbose=False,
    extra_ldflags=["-lcublas"],
)

print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))
print("detect_anomaly =", torch.is_anomaly_enabled())

_proj_forward_cuda  = stageB_MH_projector_rope_stageE_bridge.proj_forward
_proj_backward_cuda = stageB_MH_projector_rope_stageE_bridge.proj_backward

# Debug-only finite checks. Set False for speed after safety validation.
STAGEB_DEBUG_FINITE_CHECKS = True

def _stageb_assert_finite(name, x):
    if not torch.isfinite(x).all():
        raise ValueError(f"{name} contains NaN/Inf")

def proj_forward(H, Wq, Wk, Wv, cos, sin):
    if STAGEB_DEBUG_FINITE_CHECKS:
        for name, x in {"H": H, "Wq": Wq, "Wk": Wk, "Wv": Wv, "cos": cos, "sin": sin}.items():
            _stageb_assert_finite(name, x)
    return _proj_forward_cuda(H, Wq, Wk, Wv, cos, sin)

def proj_backward(dQ, dK, dV, H, Wq, Wk, Wv, cos, sin):
    if STAGEB_DEBUG_FINITE_CHECKS:
        for name, x in {
            "dQ": dQ, "dK": dK, "dV": dV,
            "H": H, "Wq": Wq, "Wk": Wk, "Wv": Wv,
            "cos": cos, "sin": sin,
        }.items():
            _stageb_assert_finite(name, x)
    return _proj_backward_cuda(dQ, dK, dV, H, Wq, Wk, Wv, cos, sin)

**Stage E: Multi-head fusion**

In [ ]:
# # ============================================================
# # CELL: Stage-E FULL family-segmented LCHA (hardened)
# # - Keeps old FP32 bridge + frozen saved tensor pattern
# # - Adds runtime gate, head mask, explicit K/V alpha0/1/2 config
# # - Adds V-side harmonic path
# # - Uses Stage-A metadata:
# #     family_bounds_i32 [10]
# #     family_order_caps_i32 [5]
# #     dispatch_meta_i32 [10]
# #
# # Runtime tensors:
# #   head_mask_u8 : [H] uint8
# #   runtime_i32  : [4] int32
# #       [0]=lcha_enabled
# #       [1]=use_v_lcha
# #       [2]=trap_on_bad_cand
# #       [3]=reserved
# #
# # Coeff tensors (float32, CUDA, contiguous):
# #   k_cfg_f32 : [7] = [self0, win1, win2, ring1, ring2, anch1, anch2]
# #   v_cfg_f32 : [7] = [self0, win1, win2, ring1, ring2, anch1, anch2]
# # ============================================================

# import os, hashlib, torch
# from torch.utils.cpp_extension import load_inline

# os.environ.setdefault("TORCH_EXTENSIONS_DIR", "/content/torch_extensions")
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "1"

# FUSED_VERBOSE_BUILD = False

# CUDA_SRC = r"""
# #include <torch/extension.h>
# #include <ATen/cuda/CUDAContext.h>
# #include <c10/cuda/CUDAGuard.h>
# #include <cuda.h>
# #include <cuda_runtime.h>
# #include <cuda_fp16.h>
# #include <cuda_bf16.h>
# #include <vector>
# #include <limits>
# #include <cmath>
# #include <cstdint>

# #ifndef BLOCK_X
# #define BLOCK_X 256
# #endif

# #ifndef V_TILE
# #define V_TILE 256
# #endif

# #if (BLOCK_X & (BLOCK_X - 1)) != 0
# #error "BLOCK_X must be a power of two."
# #endif

# static inline void check_cuda(const torch::Tensor& t, const char* name) {
#     TORCH_CHECK(t.is_cuda(), name, " must be CUDA");
# }
# static inline void check_contig(const torch::Tensor& t, const char* name) {
#     TORCH_CHECK(t.is_contiguous(), name, " must be contiguous");
# }
# static inline void check_dtype_eq(const torch::Tensor& a, const torch::Tensor& b, const char* an, const char* bn) {
#     TORCH_CHECK(a.scalar_type() == b.scalar_type(), an, " and ", bn, " dtype must match");
# }
# static inline void check_same_device(const torch::Tensor& a, const torch::Tensor& b,
#                                      const char* an, const char* bn) {
#     TORCH_CHECK(a.device() == b.device(), an, " and ", bn, " must be on the same device");
# }

# __device__ __forceinline__ void trap_if(bool cond) { if (cond) asm("trap;"); }
# __device__ __forceinline__ float expf_safe(float x) { return expf(x); }
# __device__ __forceinline__ int isfinite_f(float x) { return isfinite(x); }

# // ------------------------------------------------------------
# // float-only kernel path
# // wrappers cast Q/K/V/dOut -> FP32 before launch
# // ------------------------------------------------------------
# __device__ __forceinline__ float warp_reduce_sum(float v) {
# #pragma unroll
#     for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(0xffffffff, v, off);
#     return v;
# }

# template <int BLOCK>
# __device__ __forceinline__ float block_reduce_sum(float v, float* __restrict__ smem_warp_sums) {
#     const int lane   = (int)threadIdx.x & 31;
#     const int warp   = (int)threadIdx.x >> 5;
#     const int nwarps = BLOCK >> 5;

#     v = warp_reduce_sum(v);
#     if (lane == 0) smem_warp_sums[warp] = v;
#     __syncthreads();

#     if (warp == 0) {
#         float out = (lane < nwarps) ? smem_warp_sums[lane] : 0.0f;
#         out = warp_reduce_sum(out);
#         if (lane == 0) smem_warp_sums[0] = out;
#     }
#     __syncthreads();
#     return smem_warp_sums[0];
# }

# __device__ __forceinline__ float dot_vec_float(const float* __restrict__ q,
#                                                const float* __restrict__ k,
#                                                int D) {
#     const uintptr_t qa = (uintptr_t)q;
#     const uintptr_t ka = (uintptr_t)k;
#     const bool aligned8 = ((qa & 0x7u) == 0u) && ((ka & 0x7u) == 0u);
#     const bool even = ((D & 1) == 0);

#     float acc = 0.0f;
#     if (aligned8 && even) {
#         const float2* q2 = reinterpret_cast<const float2*>(q);
#         const float2* k2 = reinterpret_cast<const float2*>(k);
#         const int n2 = D >> 1;
#         for (int i2 = (int)threadIdx.x; i2 < n2; i2 += BLOCK_X) {
#             float2 a = q2[i2];
#             float2 b = k2[i2];
#             acc += a.x * b.x + a.y * b.y;
#         }
#     } else {
#         for (int d = (int)threadIdx.x; d < D; d += BLOCK_X) acc += q[d] * k[d];
#     }
#     return acc;
# }

# __device__ __forceinline__ int64_t cand_row_base(int t, int h, int H, int C, int cand_is_per_head) {
#     return (cand_is_per_head
#             ? ((int64_t)t * (int64_t)H + (int64_t)h)
#             : (int64_t)t) * (int64_t)C;
# }

# // ------------------------------------------------------------
# // family helpers
# // family index layout:
# //   0=self, 1=window, 2=ring, 3=anchor, 4=promoted
# // family_bounds layout:
# //   [self_s,self_e, win_s,win_e, ring_s,ring_e, anch_s,anch_e, prom_s,prom_e]
# // cfg layout for K/V:
# //   [self0, win1, win2, ring1, ring2, anch1, anch2]
# // ------------------------------------------------------------
# __device__ __forceinline__ int slot_to_family_idx(int slot, const int* __restrict__ bounds) {
#     if (slot < bounds[1]) return 0;
#     if (slot < bounds[3]) return 1;
#     if (slot < bounds[5]) return 2;
#     if (slot < bounds[7]) return 3;
#     return 4;
# }

# __device__ __forceinline__ int family_start(const int* __restrict__ bounds, int fam_idx) {
#     return bounds[2 * fam_idx];
# }
# __device__ __forceinline__ int family_end(const int* __restrict__ bounds, int fam_idx) {
#     return bounds[2 * fam_idx + 1];
# }
# __device__ __forceinline__ int min_i(int a, int b) { return (a < b) ? a : b; }

# __device__ __forceinline__ float coeff1_for_family(const float* __restrict__ cfg, int fam_idx) {
#     if (fam_idx == 1) return cfg[1];
#     if (fam_idx == 2) return cfg[3];
#     if (fam_idx == 3) return cfg[5];
#     return 0.0f;
# }
# __device__ __forceinline__ float coeff2_for_family(const float* __restrict__ cfg, int fam_idx) {
#     if (fam_idx == 1) return cfg[2];
#     if (fam_idx == 2) return cfg[4];
#     if (fam_idx == 3) return cfg[6];
#     return 0.0f;
# }

# __device__ __forceinline__ int valid_family_degree(
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     int src_t,
#     int h,
#     int H,
#     int C,
#     int T,
#     int cand_is_per_head,
#     int fam_s,
#     int fam_e
# ) {
#     const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
#     int deg = 0;
#     for (int kk = fam_s; kk < fam_e; ++kk) {
#         if (!mask[row + (int64_t)kk]) continue;
#         const int32_t j = cand[row + (int64_t)kk];
#         if (j < 0 || j >= T) continue;
#         deg++;
#     }
#     return deg;
# }

# // q dot mean(X_family)
# __device__ __forceinline__ float dot_family_mean(
#     const float* __restrict__ q_ptr,
#     const float* __restrict__ X,   // [T,H,DX]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     int src_t,
#     int h,
#     int H,
#     int DX,
#     int T,
#     int C,
#     int cand_is_per_head,
#     int fam_s,
#     int fam_e,
#     float* __restrict__ warp_sums
# ) {
#     const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
#     const int deg = valid_family_degree(cand, mask, src_t, h, H, C, T, cand_is_per_head, fam_s, fam_e);
#     if (deg <= 0) {
#         float z = 0.0f;
#         return block_reduce_sum<BLOCK_X>(z, warp_sums) * 0.0f;
#     }

#     float local = 0.0f;
#     const float w = 1.0f / (float)deg;

#     for (int kk = fam_s; kk < fam_e; ++kk) {
#         if (!mask[row + (int64_t)kk]) continue;
#         const int32_t j2 = cand[row + (int64_t)kk];
#         if (j2 < 0 || j2 >= T) continue;
#         const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
#         const float* xg = X + j2h * (int64_t)DX;
#         local += w * dot_vec_float(q_ptr, xg, DX);
#     }
#     return block_reduce_sum<BLOCK_X>(local, warp_sums);
# }

# // q dot second-order mean(X_family)
# __device__ __forceinline__ float dot_family_second_mean(
#     const float* __restrict__ q_ptr,
#     const float* __restrict__ X,   // [T,H,DX]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     int src_t,
#     int h,
#     int H,
#     int DX,
#     int T,
#     int C,
#     int cand_is_per_head,
#     int fam_s,
#     int fam_e,
#     float* __restrict__ warp_sums
# ) {
#     const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
#     const int deg1 = valid_family_degree(cand, mask, src_t, h, H, C, T, cand_is_per_head, fam_s, fam_e);
#     if (deg1 <= 0) {
#         float z = 0.0f;
#         return block_reduce_sum<BLOCK_X>(z, warp_sums) * 0.0f;
#     }

#     float local = 0.0f;
#     const float inv_deg1 = 1.0f / (float)deg1;

#     for (int kk = fam_s; kk < fam_e; ++kk) {
#         if (!mask[row + (int64_t)kk]) continue;
#         const int32_t j1 = cand[row + (int64_t)kk];
#         if (j1 < 0 || j1 >= T) continue;

#         const int deg2 = valid_family_degree(cand, mask, j1, h, H, C, T, cand_is_per_head, fam_s, fam_e);
#         if (deg2 <= 0) continue;

#         const float w = inv_deg1 / (float)deg2;
#         const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);

#         for (int uu = fam_s; uu < fam_e; ++uu) {
#             if (!mask[row1 + (int64_t)uu]) continue;
#             const int32_t j2 = cand[row1 + (int64_t)uu];
#             if (j2 < 0 || j2 >= T) continue;
#             const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
#             const float* xg = X + j2h * (int64_t)DX;
#             local += w * dot_vec_float(q_ptr, xg, DX);
#         }
#     }
#     return block_reduce_sum<BLOCK_X>(local, warp_sums);
# }

# __device__ __forceinline__ float family_mean_at_d(
#     const float* __restrict__ X,   // [T,H,DX]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     int src_t,
#     int h,
#     int H,
#     int DX,
#     int T,
#     int C,
#     int cand_is_per_head,
#     int fam_s,
#     int fam_e,
#     int d
# ) {
#     const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
#     int deg = 0;
#     float sum = 0.0f;

#     for (int kk = fam_s; kk < fam_e; ++kk) {
#         if (!mask[row + (int64_t)kk]) continue;
#         const int32_t j2 = cand[row + (int64_t)kk];
#         if (j2 < 0 || j2 >= T) continue;
#         deg++;
#         const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
#         sum += X[j2h * (int64_t)DX + (int64_t)d];
#     }
#     return (deg > 0) ? (sum / (float)deg) : 0.0f;
# }

# __device__ __forceinline__ float family_second_mean_at_d(
#     const float* __restrict__ X,   // [T,H,DX]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     int src_t,
#     int h,
#     int H,
#     int DX,
#     int T,
#     int C,
#     int cand_is_per_head,
#     int fam_s,
#     int fam_e,
#     int d
# ) {
#     const int deg1 = valid_family_degree(cand, mask, src_t, h, H, C, T, cand_is_per_head, fam_s, fam_e);
#     if (deg1 <= 0) return 0.0f;

#     const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
#     const float inv_deg1 = 1.0f / (float)deg1;
#     float sum = 0.0f;

#     for (int kk = fam_s; kk < fam_e; ++kk) {
#         if (!mask[row + (int64_t)kk]) continue;
#         const int32_t j1 = cand[row + (int64_t)kk];
#         if (j1 < 0 || j1 >= T) continue;

#         const int deg2 = valid_family_degree(cand, mask, j1, h, H, C, T, cand_is_per_head, fam_s, fam_e);
#         if (deg2 <= 0) continue;

#         const float w = inv_deg1 / (float)deg2;
#         const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);

#         for (int uu = fam_s; uu < fam_e; ++uu) {
#             if (!mask[row1 + (int64_t)uu]) continue;
#             const int32_t j2 = cand[row1 + (int64_t)uu];
#             if (j2 < 0 || j2 >= T) continue;
#             const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
#             sum += w * X[j2h * (int64_t)DX + (int64_t)d];
#         }
#     }
#     return sum;
# }

# __device__ __forceinline__ float segmented_k_score_unscaled(
#     const float* __restrict__ smem_q,      // [D]
#     const float* __restrict__ smem_k,      // [D]
#     const float* __restrict__ K,           // [T,H,D]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     int t,
#     int h,
#     int H,
#     int D,
#     int T,
#     int C,
#     int cand_is_per_head,
#     int slot_jj,
#     const int* __restrict__ bounds,
#     const int* __restrict__ caps,
#     int stage_order,
#     int lcha_enabled_for_head,
#     const float* __restrict__ k_cfg,
#     float* __restrict__ warp_sums,
#     int* family_idx_out,
#     int* eff_order_out
# ) {
#     const float qk0 = block_reduce_sum<BLOCK_X>(dot_vec_float(smem_q, smem_k, D), warp_sums);
#     const float self0 = k_cfg[0];

#     const int fam_idx = slot_to_family_idx(slot_jj, bounds);
#     int eff_order = 0;
#     if (lcha_enabled_for_head && fam_idx >= 1 && fam_idx <= 3) {
#         const int fam_cap = caps[fam_idx];
#         eff_order = min_i(stage_order, fam_cap);
#     }

#     if (family_idx_out) *family_idx_out = fam_idx;
#     if (eff_order_out) *eff_order_out = eff_order;

#     float score_unscaled = self0 * qk0;

#     if (lcha_enabled_for_head && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#         const int fam_s = family_start(bounds, fam_idx);
#         const int fam_e = family_end(bounds, fam_idx);

#         const float c1 = coeff1_for_family(k_cfg, fam_idx);
#         const float c2 = coeff2_for_family(k_cfg, fam_idx);

#         if (c1 != 0.0f) {
#             const float qk1 = dot_family_mean(
#                 smem_q, K, cand, mask,
#                 t, h, H, D, T, C, cand_is_per_head,
#                 fam_s, fam_e, warp_sums
#             );
#             score_unscaled += c1 * qk1;
#         }

#         if (eff_order >= 2 && c2 != 0.0f) {
#             const float qk2 = dot_family_second_mean(
#                 smem_q, K, cand, mask,
#                 t, h, H, D, T, C, cand_is_per_head,
#                 fam_s, fam_e, warp_sums
#             );
#             score_unscaled += c2 * qk2;
#         }
#     }

#     return score_unscaled;
# }

# // ============================================================================
# // Forward kernel
# // ============================================================================

# __global__ void fused_fwd_streaming_mh_family_full_kernel(
#     const float* __restrict__ Q,      // [T,H,D]
#     const float* __restrict__ K,      // [T,H,D]
#     const float* __restrict__ V,      // [T,H,DV]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     const int32_t* __restrict__ family_bounds,      // [10]
#     const int32_t* __restrict__ family_order_caps,  // [5]
#     const int32_t* __restrict__ dispatch_meta,      // [10]
#     const uint8_t* __restrict__ head_mask,          // [H]
#     const int32_t* __restrict__ runtime_i32,        // [4]
#     const float* __restrict__ k_cfg,                // [7]
#     const float* __restrict__ v_cfg,                // [7]
#     float* __restrict__ m_out,       // [T*H]
#     float* __restrict__ l_out,       // [T*H]
#     float* __restrict__ out,         // [T,H,DV]
#     int T, int H, int D, int DV, int C,
#     float scale,
#     int cand_is_per_head
# ) {
#     trap_if(blockDim.x != BLOCK_X);

#     const int t = (int)blockIdx.x;
#     const int h = (int)blockIdx.y;
#     const int tid = (int)threadIdx.x;
#     if (t >= T || h >= H) return;

#     const int64_t th = (int64_t)t * (int64_t)H + (int64_t)h;
#     const int64_t row = cand_row_base(t, h, H, C, cand_is_per_head);

#     __shared__ int sbounds[10];
#     __shared__ int scaps[5];
#     __shared__ int sruntime[4];
#     __shared__ float skcfg[7];
#     __shared__ float svcfg[7];
#     __shared__ int sstage_order;
#     __shared__ int s_head_lcha_on;
#     __shared__ int s_use_v;

#     if (tid < 10) sbounds[tid] = (int)family_bounds[tid];
#     if (tid < 5)  scaps[tid]   = (int)family_order_caps[tid];
#     if (tid < 4)  sruntime[tid]= (int)runtime_i32[tid];
#     if (tid < 7)  skcfg[tid]   = k_cfg[tid];
#     if (tid < 7)  svcfg[tid]   = v_cfg[tid];
#     if (tid == 0) {
#         sstage_order = (int)dispatch_meta[4];
#         s_head_lcha_on = (sruntime[0] != 0) && (head_mask[h] != 0) && (sstage_order > 0);
#         s_use_v = (sruntime[1] != 0);
#     }
#     __syncthreads();

#     extern __shared__ unsigned char smem_raw[];
#     float* smem_q = reinterpret_cast<float*>(smem_raw);
#     float* smem_k = reinterpret_cast<float*>(smem_raw + (size_t)256 * sizeof(float));
#     float* smem_warp = reinterpret_cast<float*>(smem_raw + (size_t)512 * sizeof(float));
#     float* smem_v = reinterpret_cast<float*>(
#         smem_raw + (size_t)512 * sizeof(float) + (size_t)(BLOCK_X / 32) * sizeof(float));

#     const float* qg = Q + th * (int64_t)D;
#     for (int d = tid; d < D; d += BLOCK_X) smem_q[d] = qg[d];
#     __syncthreads();

#     for (int dv = tid; dv < DV; dv += BLOCK_X) out[th * (int64_t)DV + (int64_t)dv] = 0.0f;
#     __syncthreads();

#     float m = -INFINITY;
#     float l = 0.0f;

#     for (int jj = 0; jj < C; ++jj) {
#         const uint8_t m_ij = mask[row + (int64_t)jj];
#         if (!m_ij) continue;

#         const int32_t j = cand[row + (int64_t)jj];
#         if (j < 0 || j >= T) {
#             if (sruntime[2]) trap_if(true);
#             continue;
#         }

#         const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;
#         const float* kg = K + jh * (int64_t)D;
#         for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
#         __syncthreads();

#         int fam_idx = 0;
#         int eff_order = 0;
#         float score_unscaled = segmented_k_score_unscaled(
#             smem_q, smem_k, K, cand, mask,
#             t, h, H, D, T, C, cand_is_per_head,
#             jj, sbounds, scaps, sstage_order,
#             s_head_lcha_on, skcfg, smem_warp,
#             &fam_idx, &eff_order
#         );

#         float score = score_unscaled * scale;
#         if (!isfinite_f(score)) { __syncthreads(); continue; }

#         float m_new = fmaxf(m, score);
#         float exp_m = (m == -INFINITY) ? 0.0f : expf_safe(m - m_new);
#         float exp_s = expf_safe(score - m_new);
#         if (!isfinite_f(exp_s)) exp_s = 0.0f;

#         float l_new = l * exp_m + exp_s;
#         if (!isfinite_f(l_new) || l_new <= 0.0f) { __syncthreads(); continue; }

#         float beta  = (l * exp_m) / l_new;
#         float alpha = exp_s / l_new;

#         const float* vg = V + jh * (int64_t)DV;
#         const float v_self0 = svcfg[0];

#         int fam_s = 0;
#         int fam_e = 0;
#         float v1 = 0.0f;
#         float v2 = 0.0f;
#         if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#             fam_s = family_start(sbounds, fam_idx);
#             fam_e = family_end(sbounds, fam_idx);
#             v1 = coeff1_for_family(svcfg, fam_idx);
#             v2 = coeff2_for_family(svcfg, fam_idx);
#         }

#         for (int dv0 = 0; dv0 < DV; dv0 += V_TILE) {
#             const int dv = dv0 + tid;
#             float vt = 0.0f;

#             if (dv < DV) {
#                 vt = v_self0 * vg[dv];
#                 if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#                     if (v1 != 0.0f) {
#                         vt += v1 * family_mean_at_d(
#                             V, cand, mask,
#                             t, h, H, DV, T, C, cand_is_per_head,
#                             fam_s, fam_e, dv
#                         );
#                     }
#                     if (eff_order >= 2 && v2 != 0.0f) {
#                         vt += v2 * family_second_mean_at_d(
#                             V, cand, mask,
#                             t, h, H, DV, T, C, cand_is_per_head,
#                             fam_s, fam_e, dv
#                         );
#                     }
#                 }
#             }

#             if (tid < V_TILE) smem_v[tid] = (dv < DV) ? vt : 0.0f;
#             __syncthreads();

#             if (dv < DV) {
#                 float old = out[th * (int64_t)DV + (int64_t)dv];
#                 float upd = beta * old + alpha * smem_v[tid];
#                 out[th * (int64_t)DV + (int64_t)dv] = upd;
#             }
#             __syncthreads();
#         }

#         m = m_new;
#         l = l_new;
#     }

#     if (tid == 0) {
#         m_out[th] = m;
#         l_out[th] = l;
#     }
# }

# // ============================================================================
# // Backward kernel
# // ============================================================================

# __global__ void fused_bwd_mh_family_full_kernel(
#     const float* __restrict__ dOut,    // [T,H,DV]
#     const float* __restrict__ Q,       // [T,H,D]
#     const float* __restrict__ K,       // [T,H,D]
#     const float* __restrict__ V,       // [T,H,DV]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     const int32_t* __restrict__ family_bounds,      // [10]
#     const int32_t* __restrict__ family_order_caps,  // [5]
#     const int32_t* __restrict__ dispatch_meta,      // [10]
#     const uint8_t* __restrict__ head_mask,          // [H]
#     const int32_t* __restrict__ runtime_i32,        // [4]
#     const float* __restrict__ k_cfg,                // [7]
#     const float* __restrict__ v_cfg,                // [7]
#     const float* __restrict__ m_in,   // [T*H]
#     const float* __restrict__ l_in,   // [T*H]
#     float* __restrict__ dQ,           // [T*H,D]
#     float* __restrict__ dK,           // [T*H,D]
#     float* __restrict__ dV,           // [T*H,DV]
#     int T, int H, int D, int DV, int C,
#     float scale,
#     int cand_is_per_head
# ) {
#     trap_if(blockDim.x != BLOCK_X);

#     const int t = (int)blockIdx.x;
#     const int h = (int)blockIdx.y;
#     const int tid = (int)threadIdx.x;
#     if (t >= T || h >= H) return;

#     const int64_t th = (int64_t)t * (int64_t)H + (int64_t)h;
#     const int64_t row = cand_row_base(t, h, H, C, cand_is_per_head);

#     float m = m_in[th];
#     float l = l_in[th];
#     if (!(isfinite_f(m)) || !(isfinite_f(l)) || l <= 0.0f) return;

#     __shared__ int sbounds[10];
#     __shared__ int scaps[5];
#     __shared__ int sruntime[4];
#     __shared__ float skcfg[7];
#     __shared__ float svcfg[7];
#     __shared__ int sstage_order;
#     __shared__ int s_head_lcha_on;
#     __shared__ int s_use_v;

#     __shared__ float k_acc1[5];
#     __shared__ float k_acc2[5];
#     __shared__ float v_acc1[5];
#     __shared__ float v_acc2[5];

#     if (tid < 10) sbounds[tid] = (int)family_bounds[tid];
#     if (tid < 5)  scaps[tid]   = (int)family_order_caps[tid];
#     if (tid < 4)  sruntime[tid]= (int)runtime_i32[tid];
#     if (tid < 7)  skcfg[tid]   = k_cfg[tid];
#     if (tid < 7)  svcfg[tid]   = v_cfg[tid];
#     if (tid < 5) {
#         k_acc1[tid] = 0.0f;
#         k_acc2[tid] = 0.0f;
#         v_acc1[tid] = 0.0f;
#         v_acc2[tid] = 0.0f;
#     }
#     if (tid == 0) {
#         sstage_order = (int)dispatch_meta[4];
#         s_head_lcha_on = (sruntime[0] != 0) && (head_mask[h] != 0) && (sstage_order > 0);
#         s_use_v = (sruntime[1] != 0);
#     }
#     __syncthreads();

#     __shared__ float smem_q[256];
#     __shared__ float smem_k[256];
#     __shared__ float red[BLOCK_X];
#     __shared__ float warp_sums[BLOCK_X / 32];

#     const float* qg = Q + th * (int64_t)D;
#     for (int d = tid; d < D; d += BLOCK_X) smem_q[d] = qg[d];
#     __syncthreads();

#     const float* dout = dOut + th * (int64_t)DV;

#     // --------------------------------------------------------
#     // Pass 1: do_out = sum_j p_j * (dOut · Vtilde_j)
#     // --------------------------------------------------------
#     float local_do_out = 0.0f;

#     for (int jj = 0; jj < C; ++jj) {
#         const uint8_t m_ij = mask[row + (int64_t)jj];
#         if (!m_ij) continue;

#         const int32_t j = cand[row + (int64_t)jj];
#         if (j < 0 || j >= T) {
#             if (sruntime[2]) trap_if(true);
#             continue;
#         }

#         const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;
#         const float* kg = K + jh * (int64_t)D;
#         for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
#         __syncthreads();

#         int fam_idx = 0;
#         int eff_order = 0;
#         float score_unscaled = segmented_k_score_unscaled(
#             smem_q, smem_k, K, cand, mask,
#             t, h, H, D, T, C, cand_is_per_head,
#             jj, sbounds, scaps, sstage_order,
#             s_head_lcha_on, skcfg, warp_sums,
#             &fam_idx, &eff_order
#         );

#         float score = score_unscaled * scale;
#         if (!isfinite_f(score)) { __syncthreads(); continue; }

#         float z = score - m;
#         z = fminf(z, 0.0f);
#         float p = expf_safe(z) / l;
#         if (!isfinite_f(p)) { __syncthreads(); continue; }
#         if (p < 0.0f) p = 0.0f;
#         if (p > 1.0f) p = 1.0f;

#         const float* vg = V + jh * (int64_t)DV;
#         const float v_self0 = svcfg[0];

#         int fam_s = 0;
#         int fam_e = 0;
#         float v1 = 0.0f;
#         float v2 = 0.0f;
#         if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#             fam_s = family_start(sbounds, fam_idx);
#             fam_e = family_end(sbounds, fam_idx);
#             v1 = coeff1_for_family(svcfg, fam_idx);
#             v2 = coeff2_for_family(svcfg, fam_idx);
#         }

#         float local_do_v = 0.0f;
#         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#             float vt = v_self0 * vg[dv];
#             if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#                 if (v1 != 0.0f) {
#                     vt += v1 * family_mean_at_d(
#                         V, cand, mask,
#                         t, h, H, DV, T, C, cand_is_per_head,
#                         fam_s, fam_e, dv
#                     );
#                 }
#                 if (eff_order >= 2 && v2 != 0.0f) {
#                     vt += v2 * family_second_mean_at_d(
#                         V, cand, mask,
#                         t, h, H, DV, T, C, cand_is_per_head,
#                         fam_s, fam_e, dv
#                     );
#                 }
#             }
#             local_do_v += dout[dv] * vt;
#         }

#         red[tid] = local_do_v;
#         __syncthreads();
#         for (int off = BLOCK_X / 2; off > 0; off >>= 1) {
#             if (tid < off) red[tid] += red[tid + off];
#             __syncthreads();
#         }
#         float do_v = red[0];
#         if (!isfinite_f(do_v)) { __syncthreads(); continue; }

#         local_do_out += p * do_v;
#         __syncthreads();
#     }

#     if (tid == 0) red[0] = local_do_out;
#     __syncthreads();
#     float do_out = red[0];

#     for (int d = tid; d < D; d += BLOCK_X) {
#         dQ[th * (int64_t)D + (int64_t)d] = 0.0f;
#     }
#     __syncthreads();

#     // --------------------------------------------------------
#     // Pass 2: raw grads + family accumulators
#     // --------------------------------------------------------
#     for (int jj = 0; jj < C; ++jj) {
#         const uint8_t m_ij = mask[row + (int64_t)jj];
#         if (!m_ij) continue;

#         const int32_t j = cand[row + (int64_t)jj];
#         if (j < 0 || j >= T) {
#             if (sruntime[2]) trap_if(true);
#             continue;
#         }

#         const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;
#         const float* kg = K + jh * (int64_t)D;
#         for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
#         __syncthreads();

#         int fam_idx = 0;
#         int eff_order = 0;
#         float score_unscaled = segmented_k_score_unscaled(
#             smem_q, smem_k, K, cand, mask,
#             t, h, H, D, T, C, cand_is_per_head,
#             jj, sbounds, scaps, sstage_order,
#             s_head_lcha_on, skcfg, warp_sums,
#             &fam_idx, &eff_order
#         );

#         float score = score_unscaled * scale;
#         if (!isfinite_f(score)) { __syncthreads(); continue; }

#         float z = score - m;
#         z = fminf(z, 0.0f);
#         float p = expf_safe(z) / l;
#         if (!isfinite_f(p)) { __syncthreads(); continue; }
#         if (p < 0.0f) p = 0.0f;
#         if (p > 1.0f) p = 1.0f;

#         const float* vg = V + jh * (int64_t)DV;
#         const float k_self0 = skcfg[0];
#         const float v_self0 = svcfg[0];

#         int fam_s = 0;
#         int fam_e = 0;
#         float k1 = 0.0f, k2 = 0.0f, v1 = 0.0f, v2 = 0.0f;
#         if (s_head_lcha_on && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#             fam_s = family_start(sbounds, fam_idx);
#             fam_e = family_end(sbounds, fam_idx);
#             k1 = coeff1_for_family(skcfg, fam_idx);
#             k2 = coeff2_for_family(skcfg, fam_idx);
#             if (s_use_v) {
#                 v1 = coeff1_for_family(svcfg, fam_idx);
#                 v2 = coeff2_for_family(svcfg, fam_idx);
#             }
#         }

#         // raw dV term
#         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#             atomicAdd(&dV[jh * (int64_t)DV + (int64_t)dv], p * v_self0 * dout[dv]);
#         }

#         // do_v = dOut · Vtilde_j
#         float local_do_v = 0.0f;
#         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#             float vt = v_self0 * vg[dv];
#             if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#                 if (v1 != 0.0f) {
#                     vt += v1 * family_mean_at_d(
#                         V, cand, mask,
#                         t, h, H, DV, T, C, cand_is_per_head,
#                         fam_s, fam_e, dv
#                     );
#                 }
#                 if (eff_order >= 2 && v2 != 0.0f) {
#                     vt += v2 * family_second_mean_at_d(
#                         V, cand, mask,
#                         t, h, H, DV, T, C, cand_is_per_head,
#                         fam_s, fam_e, dv
#                     );
#                 }
#             }
#             local_do_v += dout[dv] * vt;
#         }

#         red[tid] = local_do_v;
#         __syncthreads();
#         for (int off = BLOCK_X / 2; off > 0; off >>= 1) {
#             if (tid < off) red[tid] += red[tid + off];
#             __syncthreads();
#         }
#         float do_v = red[0];
#         if (!isfinite_f(do_v)) { __syncthreads(); continue; }

#         float g = p * (do_v - do_out);
#         if (!isfinite_f(g)) { __syncthreads(); continue; }

#         const float gs = g * scale;
#         if (!isfinite_f(gs)) { __syncthreads(); continue; }

#         // raw self term derivative
#         const float gs_self = gs * k_self0;
#         if (k_self0 != 0.0f && isfinite_f(gs_self)) {
#             for (int d = tid; d < D; d += BLOCK_X) {
#                 dQ[th * (int64_t)D + (int64_t)d] += gs_self * kg[d];
#             }
#             for (int d = tid; d < D; d += BLOCK_X) {
#                 atomicAdd(&dK[jh * (int64_t)D + (int64_t)d], gs_self * smem_q[d]);
#             }
#         }

#         // family accumulators
#         if (tid == 0 && s_head_lcha_on && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
#             if (k1 != 0.0f) k_acc1[fam_idx] += gs * k1;
#             if (eff_order >= 2 && k2 != 0.0f) k_acc2[fam_idx] += gs * k2;
#             if (s_use_v) {
#                 if (v1 != 0.0f) v_acc1[fam_idx] += p * v1;
#                 if (eff_order >= 2 && v2 != 0.0f) v_acc2[fam_idx] += p * v2;
#             }
#         }
#         __syncthreads();
#     }

#     // --------------------------------------------------------
#     // Pass 3A: propagate K-family gradients
#     // --------------------------------------------------------
#     for (int fam_idx = 1; fam_idx <= 3; ++fam_idx) {
#         const float S1 = k_acc1[fam_idx];
#         const float S2 = k_acc2[fam_idx];
#         if ((!isfinite_f(S1) || S1 == 0.0f) && (!isfinite_f(S2) || S2 == 0.0f)) {
#             __syncthreads();
#             continue;
#         }

#         const int fam_s = family_start(sbounds, fam_idx);
#         const int fam_e = family_end(sbounds, fam_idx);

#         // dQ contribution from first order
#         if (S1 != 0.0f) {
#             for (int d = tid; d < D; d += BLOCK_X) {
#                 float m1 = family_mean_at_d(
#                     K, cand, mask,
#                     t, h, H, D, T, C, cand_is_per_head,
#                     fam_s, fam_e, d
#                 );
#                 dQ[th * (int64_t)D + (int64_t)d] += S1 * m1;
#             }

#             const int deg1 = valid_family_degree(
#                 cand, mask, t, h, H, C, T, cand_is_per_head, fam_s, fam_e
#             );
#             if (deg1 > 0) {
#                 const float w1 = S1 / (float)deg1;
#                 const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);
#                 for (int kk = fam_s; kk < fam_e; ++kk) {
#                     if (!mask[row0 + (int64_t)kk]) continue;
#                     const int32_t j1 = cand[row0 + (int64_t)kk];
#                     if (j1 < 0 || j1 >= T) continue;
#                     const int64_t j1h = (int64_t)j1 * (int64_t)H + (int64_t)h;
#                     for (int d = tid; d < D; d += BLOCK_X) {
#                         atomicAdd(&dK[j1h * (int64_t)D + (int64_t)d], w1 * smem_q[d]);
#                     }
#                 }
#             }
#         }

#         // dQ + dK contribution from second order
#         if (S2 != 0.0f) {
#             for (int d = tid; d < D; d += BLOCK_X) {
#                 float m2 = family_second_mean_at_d(
#                     K, cand, mask,
#                     t, h, H, D, T, C, cand_is_per_head,
#                     fam_s, fam_e, d
#                 );
#                 dQ[th * (int64_t)D + (int64_t)d] += S2 * m2;
#             }

#             const int deg1 = valid_family_degree(
#                 cand, mask, t, h, H, C, T, cand_is_per_head, fam_s, fam_e
#             );
#             if (deg1 > 0) {
#                 const float inv_deg1 = 1.0f / (float)deg1;
#                 const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);

#                 for (int kk = fam_s; kk < fam_e; ++kk) {
#                     if (!mask[row0 + (int64_t)kk]) continue;
#                     const int32_t j1 = cand[row0 + (int64_t)kk];
#                     if (j1 < 0 || j1 >= T) continue;

#                     const int deg2 = valid_family_degree(
#                         cand, mask, j1, h, H, C, T, cand_is_per_head, fam_s, fam_e
#                     );
#                     if (deg2 <= 0) continue;

#                     const float w2 = S2 * inv_deg1 / (float)deg2;
#                     const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);
#                     for (int uu = fam_s; uu < fam_e; ++uu) {
#                         if (!mask[row1 + (int64_t)uu]) continue;
#                         const int32_t j2 = cand[row1 + (int64_t)uu];
#                         if (j2 < 0 || j2 >= T) continue;
#                         const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
#                         for (int d = tid; d < D; d += BLOCK_X) {
#                             atomicAdd(&dK[j2h * (int64_t)D + (int64_t)d], w2 * smem_q[d]);
#                         }
#                     }
#                 }
#             }
#         }

#         __syncthreads();
#     }

#     // --------------------------------------------------------
#     // Pass 3B: propagate V-family gradients
#     // --------------------------------------------------------
#     if (s_use_v) {
#         for (int fam_idx = 1; fam_idx <= 3; ++fam_idx) {
#             const float P1 = v_acc1[fam_idx];
#             const float P2 = v_acc2[fam_idx];
#             if ((!isfinite_f(P1) || P1 == 0.0f) && (!isfinite_f(P2) || P2 == 0.0f)) {
#                 __syncthreads();
#                 continue;
#             }

#             const int fam_s = family_start(sbounds, fam_idx);
#             const int fam_e = family_end(sbounds, fam_idx);

#             if (P1 != 0.0f) {
#                 const int deg1 = valid_family_degree(
#                     cand, mask, t, h, H, C, T, cand_is_per_head, fam_s, fam_e
#                 );
#                 if (deg1 > 0) {
#                     const float w1 = P1 / (float)deg1;
#                     const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);
#                     for (int kk = fam_s; kk < fam_e; ++kk) {
#                         if (!mask[row0 + (int64_t)kk]) continue;
#                         const int32_t j1 = cand[row0 + (int64_t)kk];
#                         if (j1 < 0 || j1 >= T) continue;
#                         const int64_t j1h = (int64_t)j1 * (int64_t)H + (int64_t)h;
#                         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#                             atomicAdd(&dV[j1h * (int64_t)DV + (int64_t)dv], w1 * dout[dv]);
#                         }
#                     }
#                 }
#             }

#             if (P2 != 0.0f) {
#                 const int deg1 = valid_family_degree(
#                     cand, mask, t, h, H, C, T, cand_is_per_head, fam_s, fam_e
#                 );
#                 if (deg1 > 0) {
#                     const float inv_deg1 = 1.0f / (float)deg1;
#                     const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);

#                     for (int kk = fam_s; kk < fam_e; ++kk) {
#                         if (!mask[row0 + (int64_t)kk]) continue;
#                         const int32_t j1 = cand[row0 + (int64_t)kk];
#                         if (j1 < 0 || j1 >= T) continue;

#                         const int deg2 = valid_family_degree(
#                             cand, mask, j1, h, H, C, T, cand_is_per_head, fam_s, fam_e
#                         );
#                         if (deg2 <= 0) continue;

#                         const float w2 = P2 * inv_deg1 / (float)deg2;
#                         const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);
#                         for (int uu = fam_s; uu < fam_e; ++uu) {
#                             if (!mask[row1 + (int64_t)uu]) continue;
#                             const int32_t j2 = cand[row1 + (int64_t)uu];
#                             if (j2 < 0 || j2 >= T) continue;
#                             const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
#                             for (int dv = tid; dv < DV; dv += BLOCK_X) {
#                                 atomicAdd(&dV[j2h * (int64_t)DV + (int64_t)dv], w2 * dout[dv]);
#                             }
#                         }
#                     }
#                 }
#             }

#             __syncthreads();
#         }
#     }
# }

# // ------------------------------------------------------------
# // cast fp32 gradients back to original dtype
# // ------------------------------------------------------------
# __global__ void cast_f32_to_f32(const float* src, float* dst, int64_t n) {
#     int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
#     if (i < n) dst[i] = src[i];
# }
# __global__ void cast_f32_to_f16(const float* src, at::Half* dst, int64_t n) {
#     int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
#     if (i < n) dst[i] = (at::Half)__float2half_rn(src[i]);
# }
# __global__ void cast_f32_to_bf16(const float* src, at::BFloat16* dst, int64_t n) {
#     int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ >= 800
#     if (i < n) dst[i] = (at::BFloat16)__float2bfloat16_rn(src[i]);
# #else
#     if (i < n) asm("trap;");
# #endif
# }

# // ============================================================================
# // C++ wrappers (hardened like old Stage-E)
# // ============================================================================

# std::vector<torch::Tensor> mh_forward(
#     torch::Tensor Q,        // [T,H,D]
#     torch::Tensor K,        // [T,H,D]
#     torch::Tensor V,        // [T,H,DV]
#     torch::Tensor cand_i32, // [T,C] or [T*H,C]
#     torch::Tensor mask_u8,  // [T,C] or [T*H,C]
#     torch::Tensor family_bounds_i32,      // [10]
#     torch::Tensor family_order_caps_i32,  // [5]
#     torch::Tensor dispatch_meta_i32,      // [10]
#     torch::Tensor head_mask_u8,           // [H]
#     torch::Tensor runtime_i32,            // [4]
#     torch::Tensor k_cfg_f32,              // [7]
#     torch::Tensor v_cfg_f32,              // [7]
#     double scale_py,
#     bool cand_is_per_head
# ) {
#     check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
#     check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
#     check_cuda(family_bounds_i32,"family_bounds");
#     check_cuda(family_order_caps_i32,"family_order_caps");
#     check_cuda(dispatch_meta_i32,"dispatch_meta");
#     check_cuda(head_mask_u8,"head_mask");
#     check_cuda(runtime_i32,"runtime_i32");
#     check_cuda(k_cfg_f32,"k_cfg");
#     check_cuda(v_cfg_f32,"v_cfg");

#     c10::cuda::CUDAGuard guard(Q.device());
#     check_same_device(Q, K, "Q", "K");
#     check_same_device(Q, V, "Q", "V");
#     check_same_device(Q, cand_i32, "Q", "cand");
#     check_same_device(Q, mask_u8, "Q", "mask");
#     check_same_device(Q, family_bounds_i32, "Q", "family_bounds");
#     check_same_device(Q, family_order_caps_i32, "Q", "family_order_caps");
#     check_same_device(Q, dispatch_meta_i32, "Q", "dispatch_meta");
#     check_same_device(Q, head_mask_u8, "Q", "head_mask");
#     check_same_device(Q, runtime_i32, "Q", "runtime_i32");
#     check_same_device(Q, k_cfg_f32, "Q", "k_cfg");
#     check_same_device(Q, v_cfg_f32, "Q", "v_cfg");

#     check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
#     check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask");
#     check_contig(family_bounds_i32,"family_bounds");
#     check_contig(family_order_caps_i32,"family_order_caps");
#     check_contig(dispatch_meta_i32,"dispatch_meta");
#     check_contig(head_mask_u8,"head_mask");
#     check_contig(runtime_i32,"runtime_i32");
#     check_contig(k_cfg_f32,"k_cfg");
#     check_contig(v_cfg_f32,"v_cfg");

#     TORCH_CHECK(Q.dim()==3 && K.dim()==3 && V.dim()==3, "Q/K/V must be [T,H,D]/[T,H,DV]");
#     TORCH_CHECK(cand_i32.dim()==2 && mask_u8.dim()==2, "cand/mask must be 2D");
#     TORCH_CHECK(cand_i32.scalar_type()==at::kInt, "cand must be int32");
#     TORCH_CHECK(mask_u8.scalar_type()==at::kByte, "mask must be uint8");
#     TORCH_CHECK(family_bounds_i32.dim()==1 && family_bounds_i32.numel()==10 && family_bounds_i32.scalar_type()==at::kInt, "family_bounds must be int32 [10]");
#     TORCH_CHECK(family_order_caps_i32.dim()==1 && family_order_caps_i32.numel()==5 && family_order_caps_i32.scalar_type()==at::kInt, "family_order_caps must be int32 [5]");
#     TORCH_CHECK(dispatch_meta_i32.dim()==1 && dispatch_meta_i32.numel()==10 && dispatch_meta_i32.scalar_type()==at::kInt, "dispatch_meta must be int32 [10]");
#     TORCH_CHECK(head_mask_u8.dim()==1 && head_mask_u8.scalar_type()==at::kByte, "head_mask must be uint8 [H]");
#     TORCH_CHECK(runtime_i32.dim()==1 && runtime_i32.numel()==4 && runtime_i32.scalar_type()==at::kInt, "runtime_i32 must be int32 [4]");
#     TORCH_CHECK(k_cfg_f32.dim()==1 && k_cfg_f32.numel()==7 && k_cfg_f32.scalar_type()==at::kFloat, "k_cfg_f32 must be float32 [7]");
#     TORCH_CHECK(v_cfg_f32.dim()==1 && v_cfg_f32.numel()==7 && v_cfg_f32.scalar_type()==at::kFloat, "v_cfg_f32 must be float32 [7]");
#     TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X");

#     check_dtype_eq(Q,K,"Q","K");
#     check_dtype_eq(Q,V,"Q","V");
#     TORCH_CHECK(Q.is_floating_point() && K.is_floating_point() && V.is_floating_point(),
#                 "Q/K/V must be floating point");
#     TORCH_CHECK(std::isfinite(scale_py), "scale must be finite");

#     const int64_t T64 = Q.size(0);
#     const int64_t H64 = Q.size(1);
#     const int64_t D64 = Q.size(2);
#     const int64_t DV64= V.size(2);
#     const int64_t C64 = cand_i32.size(1);

#     TORCH_CHECK(T64 > 0 && H64 > 0 && D64 > 0 && DV64 > 0 && C64 > 0,
#                 "T/H/D/DV/C must be positive");
#     TORCH_CHECK(K.size(0)==T64 && K.size(1)==H64 && K.size(2)==D64, "K shape mismatch");
#     TORCH_CHECK(V.size(0)==T64 && V.size(1)==H64, "V shape mismatch");
#     TORCH_CHECK(mask_u8.size(1)==C64, "mask C mismatch");
#     TORCH_CHECK(head_mask_u8.numel()==H64, "head_mask size mismatch");
#     TORCH_CHECK(D64 <= 256, "D must be <= 256");

#     TORCH_CHECK(T64 <= std::numeric_limits<int64_t>::max() / H64, "T*H would overflow int64");
#     const int64_t TH64 = T64 * H64;

#     TORCH_CHECK(T64 <= (int64_t)std::numeric_limits<int>::max(), "T too large");
#     TORCH_CHECK(H64 <= (int64_t)std::numeric_limits<int>::max(), "H too large");
#     TORCH_CHECK(D64 <= (int64_t)std::numeric_limits<int>::max(), "D too large");
#     TORCH_CHECK(DV64<= (int64_t)std::numeric_limits<int>::max(), "DV too large");
#     TORCH_CHECK(C64 <= (int64_t)std::numeric_limits<int>::max(), "C too large");
#     TORCH_CHECK(TH64 <= (int64_t)std::numeric_limits<int>::max(), "T*H too large");

#     const int64_t expected_rows = cand_is_per_head ? TH64 : T64;
#     TORCH_CHECK(cand_i32.size(0) == expected_rows, "cand rows mismatch (expected ", expected_rows, ")");
#     TORCH_CHECK(mask_u8.size(0) == expected_rows, "mask rows mismatch (expected ", expected_rows, ")");

#     auto Qf = (Q.scalar_type() == at::kFloat) ? Q : Q.to(at::kFloat);
#     auto Kf = (K.scalar_type() == at::kFloat) ? K : K.to(at::kFloat);
#     auto Vf = (V.scalar_type() == at::kFloat) ? V : V.to(at::kFloat);
#     Qf = Qf.contiguous();
#     Kf = Kf.contiguous();
#     Vf = Vf.contiguous();

#     auto out = torch::zeros({T64, H64, DV64}, Qf.options().dtype(at::kFloat));
#     auto m   = torch::empty({TH64}, Qf.options().dtype(at::kFloat));
#     auto l   = torch::empty({TH64}, Qf.options().dtype(at::kFloat));

#     const int T = (int)T64;
#     const int H = (int)H64;
#     const int D = (int)D64;
#     const int DV= (int)DV64;
#     const int C = (int)C64;

#     const float scale = (float)scale_py;
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();

#     dim3 grid((unsigned)T, (unsigned)H, 1);
#     dim3 block(BLOCK_X, 1, 1);

#     const size_t shmem = (size_t)512 * sizeof(float)
#                        + (size_t)(BLOCK_X / 32) * sizeof(float)
#                        + (size_t)V_TILE * sizeof(float);

#     fused_fwd_streaming_mh_family_full_kernel<<<grid, block, shmem, stream>>>(
#         (const float*)Qf.data_ptr(),
#         (const float*)Kf.data_ptr(),
#         (const float*)Vf.data_ptr(),
#         (const int32_t*)cand_i32.data_ptr(),
#         (const uint8_t*)mask_u8.data_ptr(),
#         (const int32_t*)family_bounds_i32.data_ptr(),
#         (const int32_t*)family_order_caps_i32.data_ptr(),
#         (const int32_t*)dispatch_meta_i32.data_ptr(),
#         (const uint8_t*)head_mask_u8.data_ptr(),
#         (const int32_t*)runtime_i32.data_ptr(),
#         (const float*)k_cfg_f32.data_ptr(),
#         (const float*)v_cfg_f32.data_ptr(),
#         (float*)m.data_ptr(),
#         (float*)l.data_ptr(),
#         (float*)out.data_ptr(),
#         T,H,D,DV,C, scale, (int)cand_is_per_head
#     );

#     auto err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_forward kernel launch failed: ", cudaGetErrorString(err));

#     return {out, m, l};
# }

# std::vector<torch::Tensor> mh_forward_frozen(
#     torch::Tensor Q,
#     torch::Tensor K,
#     torch::Tensor V,
#     torch::Tensor cand_i32,
#     torch::Tensor mask_u8,
#     torch::Tensor family_bounds_i32,
#     torch::Tensor family_order_caps_i32,
#     torch::Tensor dispatch_meta_i32,
#     torch::Tensor head_mask_u8,
#     torch::Tensor runtime_i32,
#     torch::Tensor k_cfg_f32,
#     torch::Tensor v_cfg_f32,
#     double scale_py,
#     bool cand_is_per_head
# ) {
#     check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
#     check_cuda(family_bounds_i32,"family_bounds");
#     check_cuda(family_order_caps_i32,"family_order_caps");
#     check_cuda(dispatch_meta_i32,"dispatch_meta");
#     check_cuda(head_mask_u8,"head_mask");
#     check_cuda(runtime_i32,"runtime_i32");
#     check_cuda(k_cfg_f32,"k_cfg");
#     check_cuda(v_cfg_f32,"v_cfg");

#     auto cand_frozen = cand_i32.contiguous().clone();
#     auto mask_frozen = mask_u8.contiguous().clone();
#     auto bounds_frozen = family_bounds_i32.contiguous().clone();
#     auto caps_frozen = family_order_caps_i32.contiguous().clone();
#     auto dispatch_frozen = dispatch_meta_i32.contiguous().clone();
#     auto head_mask_frozen = head_mask_u8.contiguous().clone();
#     auto runtime_frozen = runtime_i32.contiguous().clone();
#     auto kcfg_frozen = k_cfg_f32.contiguous().clone();
#     auto vcfg_frozen = v_cfg_f32.contiguous().clone();

#     auto outs = mh_forward(
#         Q, K, V,
#         cand_frozen, mask_frozen,
#         bounds_frozen, caps_frozen, dispatch_frozen,
#         head_mask_frozen, runtime_frozen,
#         kcfg_frozen, vcfg_frozen,
#         scale_py, cand_is_per_head
#     );

#     return {
#         outs[0], outs[1], outs[2],
#         cand_frozen, mask_frozen,
#         bounds_frozen, caps_frozen, dispatch_frozen,
#         head_mask_frozen, runtime_frozen,
#         kcfg_frozen, vcfg_frozen
#     };
# }

# // ============================================================
# // REPLACE ONLY THIS FUNCTION INSIDE YOUR EXISTING STAGE-E CELL
# // Keep the rest of the CUDA/C++ code VERBATIM.
# // ============================================================
# std::vector<torch::Tensor> mh_backward(
#     torch::Tensor dOut,
#     torch::Tensor Q,
#     torch::Tensor K,
#     torch::Tensor V,
#     torch::Tensor cand_i32,
#     torch::Tensor mask_u8,
#     torch::Tensor family_bounds_i32,
#     torch::Tensor family_order_caps_i32,
#     torch::Tensor dispatch_meta_i32,
#     torch::Tensor head_mask_u8,
#     torch::Tensor runtime_i32,
#     torch::Tensor k_cfg_f32,
#     torch::Tensor v_cfg_f32,
#     torch::Tensor m,
#     torch::Tensor l,
#     double scale_py,
#     bool cand_is_per_head
# ) {
#     check_cuda(dOut,"dOut"); check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
#     check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
#     check_cuda(family_bounds_i32,"family_bounds");
#     check_cuda(family_order_caps_i32,"family_order_caps");
#     check_cuda(dispatch_meta_i32,"dispatch_meta");
#     check_cuda(head_mask_u8,"head_mask");
#     check_cuda(runtime_i32,"runtime_i32");
#     check_cuda(k_cfg_f32,"k_cfg");
#     check_cuda(v_cfg_f32,"v_cfg");
#     check_cuda(m,"m"); check_cuda(l,"l");

#     c10::cuda::CUDAGuard guard(Q.device());
#     check_same_device(Q, dOut, "Q", "dOut");
#     check_same_device(Q, K, "Q", "K");
#     check_same_device(Q, V, "Q", "V");
#     check_same_device(Q, cand_i32, "Q", "cand");
#     check_same_device(Q, mask_u8, "Q", "mask");
#     check_same_device(Q, family_bounds_i32, "Q", "family_bounds");
#     check_same_device(Q, family_order_caps_i32, "Q", "family_order_caps");
#     check_same_device(Q, dispatch_meta_i32, "Q", "dispatch_meta");
#     check_same_device(Q, head_mask_u8, "Q", "head_mask");
#     check_same_device(Q, runtime_i32, "Q", "runtime_i32");
#     check_same_device(Q, k_cfg_f32, "Q", "k_cfg");
#     check_same_device(Q, v_cfg_f32, "Q", "v_cfg");
#     check_same_device(Q, m, "Q", "m");
#     check_same_device(Q, l, "Q", "l");

#     check_contig(dOut,"dOut"); check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
#     check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask");
#     check_contig(family_bounds_i32,"family_bounds");
#     check_contig(family_order_caps_i32,"family_order_caps");
#     check_contig(dispatch_meta_i32,"dispatch_meta");
#     check_contig(head_mask_u8,"head_mask");
#     check_contig(runtime_i32,"runtime_i32");
#     check_contig(k_cfg_f32,"k_cfg");
#     check_contig(v_cfg_f32,"v_cfg");
#     check_contig(m,"m"); check_contig(l,"l");

#     TORCH_CHECK(dOut.dim()==3 && Q.dim()==3 && K.dim()==3 && V.dim()==3,
#                 "dOut/Q/K/V must be [T,H,DV] / [T,H,D] / [T,H,D] / [T,H,DV]");
#     TORCH_CHECK(cand_i32.dim()==2 && mask_u8.dim()==2, "cand/mask must be 2D");
#     TORCH_CHECK(cand_i32.scalar_type()==at::kInt, "cand must be int32");
#     TORCH_CHECK(mask_u8.scalar_type()==at::kByte, "mask must be uint8");

#     TORCH_CHECK(m.scalar_type()==at::kFloat && l.scalar_type()==at::kFloat, "m/l must be float");
#     TORCH_CHECK(family_bounds_i32.dim()==1 && family_bounds_i32.numel()==10 && family_bounds_i32.scalar_type()==at::kInt,
#                 "family_bounds must be int32 [10]");
#     TORCH_CHECK(family_order_caps_i32.dim()==1 && family_order_caps_i32.numel()==5 && family_order_caps_i32.scalar_type()==at::kInt,
#                 "family_order_caps must be int32 [5]");
#     TORCH_CHECK(dispatch_meta_i32.dim()==1 && dispatch_meta_i32.numel()==10 && dispatch_meta_i32.scalar_type()==at::kInt,
#                 "dispatch_meta must be int32 [10]");
#     TORCH_CHECK(head_mask_u8.dim()==1 && head_mask_u8.scalar_type()==at::kByte,
#                 "head_mask must be uint8 [H]");
#     TORCH_CHECK(runtime_i32.dim()==1 && runtime_i32.numel()==4 && runtime_i32.scalar_type()==at::kInt,
#                 "runtime_i32 must be int32 [4]");
#     TORCH_CHECK(k_cfg_f32.dim()==1 && k_cfg_f32.numel()==7 && k_cfg_f32.scalar_type()==at::kFloat,
#                 "k_cfg_f32 must be float32 [7]");
#     TORCH_CHECK(v_cfg_f32.dim()==1 && v_cfg_f32.numel()==7 && v_cfg_f32.scalar_type()==at::kFloat,
#                 "v_cfg_f32 must be float32 [7]");
#     TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X");

#     check_dtype_eq(Q,K,"Q","K");
#     check_dtype_eq(Q,V,"Q","V");
#     check_dtype_eq(V,dOut,"V","dOut");
#     TORCH_CHECK(dOut.is_floating_point() && Q.is_floating_point() &&
#                 K.is_floating_point() && V.is_floating_point(),
#                 "dOut/Q/K/V must be floating point");
#     TORCH_CHECK(std::isfinite(scale_py), "scale must be finite");

#     const int64_t T64  = Q.size(0);
#     const int64_t H64  = Q.size(1);
#     const int64_t D64  = Q.size(2);
#     const int64_t DV64 = V.size(2);
#     const int64_t C64  = cand_i32.size(1);

#     TORCH_CHECK(T64 > 0 && H64 > 0 && D64 > 0 && DV64 > 0 && C64 > 0,
#                 "T/H/D/DV/C must be positive");
#     TORCH_CHECK(K.size(0)==T64 && K.size(1)==H64 && K.size(2)==D64, "K shape mismatch");
#     TORCH_CHECK(V.size(0)==T64 && V.size(1)==H64, "V shape mismatch");
#     TORCH_CHECK(dOut.size(0)==T64 && dOut.size(1)==H64 && dOut.size(2)==DV64, "dOut shape mismatch");
#     TORCH_CHECK(mask_u8.size(1)==C64, "mask C mismatch");
#     TORCH_CHECK(head_mask_u8.numel()==H64, "head_mask size mismatch");
#     TORCH_CHECK(D64 <= 256, "D must be <= 256");

#     TORCH_CHECK(T64 <= std::numeric_limits<int64_t>::max() / H64, "T*H would overflow int64");
#     const int64_t TH64 = T64 * H64;

#     TORCH_CHECK(T64 <= (int64_t)std::numeric_limits<int>::max(), "T too large");
#     TORCH_CHECK(H64 <= (int64_t)std::numeric_limits<int>::max(), "H too large");
#     TORCH_CHECK(D64 <= (int64_t)std::numeric_limits<int>::max(), "D too large");
#     TORCH_CHECK(DV64 <= (int64_t)std::numeric_limits<int>::max(), "DV too large");
#     TORCH_CHECK(C64 <= (int64_t)std::numeric_limits<int>::max(), "C too large");
#     TORCH_CHECK(TH64 <= (int64_t)std::numeric_limits<int>::max(), "T*H too large");
#     TORCH_CHECK(m.dim() == 1 && m.numel() == TH64, "m must be [T*H]");
#     TORCH_CHECK(l.dim() == 1 && l.numel() == TH64, "l must be [T*H]");

#     const int64_t expected_rows = cand_is_per_head ? TH64 : T64;
#     TORCH_CHECK(cand_i32.size(0) == expected_rows, "cand rows mismatch (expected ", expected_rows, ")");
#     TORCH_CHECK(mask_u8.size(0) == expected_rows, "mask rows mismatch (expected ", expected_rows, ")");

#     auto dOutf = (dOut.scalar_type() == at::kFloat) ? dOut : dOut.to(at::kFloat);
#     auto Qf    = (Q.scalar_type()    == at::kFloat) ? Q    : Q.to(at::kFloat);
#     auto Kf    = (K.scalar_type()    == at::kFloat) ? K    : K.to(at::kFloat);
#     auto Vf    = (V.scalar_type()    == at::kFloat) ? V    : V.to(at::kFloat);

#     dOutf = dOutf.contiguous();
#     Qf    = Qf.contiguous();
#     Kf    = Kf.contiguous();
#     Vf    = Vf.contiguous();

#     auto dQ_f = torch::zeros({TH64, D64},  torch::TensorOptions().dtype(torch::kFloat).device(Q.device()));
#     auto dK_f = torch::zeros({TH64, D64},  torch::TensorOptions().dtype(torch::kFloat).device(Q.device()));
#     auto dV_f = torch::zeros({TH64, DV64}, torch::TensorOptions().dtype(torch::kFloat).device(Q.device()));

#     const int T  = (int)T64;
#     const int H  = (int)H64;
#     const int D  = (int)D64;
#     const int DV = (int)DV64;
#     const int C  = (int)C64;
#     const float scale = (float)scale_py;

#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();
#     dim3 grid((unsigned)T, (unsigned)H, 1);
#     dim3 block(BLOCK_X, 1, 1);

#     fused_bwd_mh_family_full_kernel<<<grid, block, 0, stream>>>(
#         (const float*)dOutf.data_ptr(),
#         (const float*)Qf.data_ptr(),
#         (const float*)Kf.data_ptr(),
#         (const float*)Vf.data_ptr(),
#         (const int32_t*)cand_i32.data_ptr(),
#         (const uint8_t*)mask_u8.data_ptr(),
#         (const int32_t*)family_bounds_i32.data_ptr(),
#         (const int32_t*)family_order_caps_i32.data_ptr(),
#         (const int32_t*)dispatch_meta_i32.data_ptr(),
#         (const uint8_t*)head_mask_u8.data_ptr(),
#         (const int32_t*)runtime_i32.data_ptr(),
#         (const float*)k_cfg_f32.data_ptr(),
#         (const float*)v_cfg_f32.data_ptr(),
#         (const float*)m.data_ptr(),
#         (const float*)l.data_ptr(),
#         (float*)dQ_f.data_ptr(),
#         (float*)dK_f.data_ptr(),
#         (float*)dV_f.data_ptr(),
#         T, H, D, DV, C, scale, (int)cand_is_per_head
#     );

#     auto err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_backward kernel launch failed: ", cudaGetErrorString(err));

#     auto dQ = torch::empty_like(Q);
#     auto dK = torch::empty_like(K);
#     auto dV = torch::empty_like(V);

#     const int threads = 256;
#     auto launch_cast = [&](const torch::Tensor& src_f, torch::Tensor& dst) {
#         const int64_t n = dst.numel();
#         const int blocks = (int)((n + threads - 1) / threads);
#         if (dst.scalar_type() == torch::kFloat) {
#             cast_f32_to_f32<<<blocks, threads, 0, stream>>>(
#                 src_f.data_ptr<float>(), dst.data_ptr<float>(), n);
#         } else if (dst.scalar_type() == torch::kHalf) {
#             cast_f32_to_f16<<<blocks, threads, 0, stream>>>(
#                 src_f.data_ptr<float>(), (at::Half*)dst.data_ptr<at::Half>(), n);
#         } else if (dst.scalar_type() == torch::kBFloat16) {
#             cast_f32_to_bf16<<<blocks, threads, 0, stream>>>(
#                 src_f.data_ptr<float>(), (at::BFloat16*)dst.data_ptr<at::BFloat16>(), n);
#         } else {
#             TORCH_CHECK(false, "Unsupported dtype for cast");
#         }
#     };

#     launch_cast(dQ_f.view({T64, H64, D64}).contiguous(), dQ);
#     launch_cast(dK_f.view({T64, H64, D64}).contiguous(), dK);
#     launch_cast(dV_f.view({T64, H64, DV64}).contiguous(), dV);

#     err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_backward cast launch failed: ", cudaGetErrorString(err));

#     return {dQ, dK, dV};
# }

# PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
#     m.def("mh_forward", &mh_forward, "Stage-E full family-segmented LCHA forward (CUDA)");
#     m.def("mh_forward_frozen", &mh_forward_frozen, "Stage-E full family-segmented LCHA forward frozen (CUDA)");
#     m.def("mh_backward", &mh_backward, "Stage-E full family-segmented LCHA backward (CUDA)");
# }
# """

# extra_cuda_cflags = ["-O3", "--use_fast_math", "-DBLOCK_X=256", "-DV_TILE=256"]
# extra_cflags = ["-O3", "-std=c++17"]

# _build_sig = "stageE_family_segmented_lcha_full_v2_hardened"
# _build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
# ext_name = f"stageE_family_segmented_lcha_full_ext_{_build_hash}"

# stageE_family_segmented_lcha_full = load_inline(
#     name=ext_name,
#     cpp_sources="",
#     cuda_sources=CUDA_SRC,
#     functions=None,
#     extra_cuda_cflags=extra_cuda_cflags,
#     extra_cflags=extra_cflags,
#     with_cuda=True,
#     verbose=bool(FUSED_VERBOSE_BUILD),
# )

# _mh_forward_cuda = stageE_family_segmented_lcha_full.mh_forward
# _mh_forward_frozen_cuda = stageE_family_segmented_lcha_full.mh_forward_frozen
# _mh_backward_cuda = stageE_family_segmented_lcha_full.mh_backward

# # Debug-only resilience checks. Set False for speed after validation.
# STAGEE_DEBUG_FINITE_CHECKS = True


# def _stagee_assert_finite(name, x):
#     if not torch.isfinite(x).all():
#         raise ValueError(f"{name} contains NaN/Inf")


# def validate_stagee_metadata(
#     family_bounds_i32,
#     family_order_caps_i32,
#     dispatch_meta_i32,
#     runtime_i32,
#     k_cfg_f32,
#     v_cfg_f32,
#     C: int,
# ):
#     bounds = family_bounds_i32.detach().cpu().tolist()
#     caps = family_order_caps_i32.detach().cpu().tolist()
#     dispatch = dispatch_meta_i32.detach().cpu().tolist()
#     runtime = runtime_i32.detach().cpu().tolist()

#     if len(bounds) != 10:
#         raise ValueError("family_bounds_i32 must have length 10")
#     if bounds[0] != 0:
#         raise ValueError("family_bounds must start at 0")
#     if bounds[-1] > C:
#         raise ValueError("family_bounds final end exceeds C")
#     if any(x < 0 or x > C for x in bounds):
#         raise ValueError("family_bounds entries must be in [0, C]")
#     if any(bounds[i] > bounds[i + 1] for i in range(len(bounds) - 1)):
#         raise ValueError("family_bounds must be monotonic nondecreasing")
#     if not (
#         bounds[2] == bounds[1]
#         and bounds[4] == bounds[3]
#         and bounds[6] == bounds[5]
#         and bounds[8] == bounds[7]
#     ):
#         raise ValueError("family_bounds must be contiguous: next_start == previous_end")

#     if len(caps) != 5 or any(c < 0 or c > 2 for c in caps):
#         raise ValueError("family_order_caps must have 5 entries in [0, 2]")
#     if len(dispatch) != 10:
#         raise ValueError("dispatch_meta_i32 must have length 10")
#     if dispatch[4] < 0 or dispatch[4] > 2:
#         raise ValueError("dispatch_meta_i32[4] stage_order must be in [0, 2]")
#     if len(runtime) != 4:
#         raise ValueError("runtime_i32 must have length 4")
#     if not torch.isfinite(k_cfg_f32).all():
#         raise ValueError("k_cfg_f32 contains NaN/Inf")
#     if not torch.isfinite(v_cfg_f32).all():
#         raise ValueError("v_cfg_f32 contains NaN/Inf")


# def _check_finite_stagee_forward(Q, K, V, k_cfg_f32, v_cfg_f32, scale):
#     for name, x in {
#         "Q": Q,
#         "K": K,
#         "V": V,
#         "k_cfg_f32": k_cfg_f32,
#         "v_cfg_f32": v_cfg_f32,
#     }.items():
#         _stagee_assert_finite(name, x)
#     if not torch.isfinite(torch.tensor(float(scale))):
#         raise ValueError("scale contains NaN/Inf")


# def _check_finite_stagee_backward(dOut, Q, K, V, m, l, k_cfg_f32, v_cfg_f32, scale):
#     for name, x in {
#         "dOut": dOut,
#         "Q": Q,
#         "K": K,
#         "V": V,
#         "m": m,
#         "l": l,
#         "k_cfg_f32": k_cfg_f32,
#         "v_cfg_f32": v_cfg_f32,
#     }.items():
#         _stagee_assert_finite(name, x)
#     if not torch.isfinite(torch.tensor(float(scale))):
#         raise ValueError("scale contains NaN/Inf")


# def mh_forward(
#     Q, K, V,
#     cand_i32, mask_u8,
#     family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
#     head_mask_u8, runtime_i32,
#     k_cfg_f32, v_cfg_f32,
#     scale_py,
#     cand_is_per_head,
# ):
#     validate_stagee_metadata(
#         family_bounds_i32,
#         family_order_caps_i32,
#         dispatch_meta_i32,
#         runtime_i32,
#         k_cfg_f32,
#         v_cfg_f32,
#         int(cand_i32.size(1)),
#     )
#     if STAGEE_DEBUG_FINITE_CHECKS:
#         _check_finite_stagee_forward(Q, K, V, k_cfg_f32, v_cfg_f32, scale_py)
#     return _mh_forward_cuda(
#         Q, K, V,
#         cand_i32, mask_u8,
#         family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
#         head_mask_u8, runtime_i32,
#         k_cfg_f32, v_cfg_f32,
#         scale_py,
#         cand_is_per_head,
#     )


# def mh_forward_frozen(
#     Q, K, V,
#     cand_i32, mask_u8,
#     family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
#     head_mask_u8, runtime_i32,
#     k_cfg_f32, v_cfg_f32,
#     scale_py,
#     cand_is_per_head,
# ):
#     validate_stagee_metadata(
#         family_bounds_i32,
#         family_order_caps_i32,
#         dispatch_meta_i32,
#         runtime_i32,
#         k_cfg_f32,
#         v_cfg_f32,
#         int(cand_i32.size(1)),
#     )
#     if STAGEE_DEBUG_FINITE_CHECKS:
#         _check_finite_stagee_forward(Q, K, V, k_cfg_f32, v_cfg_f32, scale_py)
#     return _mh_forward_frozen_cuda(
#         Q, K, V,
#         cand_i32, mask_u8,
#         family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
#         head_mask_u8, runtime_i32,
#         k_cfg_f32, v_cfg_f32,
#         scale_py,
#         cand_is_per_head,
#     )


# def mh_backward(
#     dOut,
#     Q, K, V,
#     cand_i32, mask_u8,
#     family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
#     head_mask_u8, runtime_i32,
#     k_cfg_f32, v_cfg_f32,
#     m, l,
#     scale_py,
#     cand_is_per_head,
# ):
#     validate_stagee_metadata(
#         family_bounds_i32,
#         family_order_caps_i32,
#         dispatch_meta_i32,
#         runtime_i32,
#         k_cfg_f32,
#         v_cfg_f32,
#         int(cand_i32.size(1)),
#     )
#     if STAGEE_DEBUG_FINITE_CHECKS:
#         _check_finite_stagee_backward(dOut, Q, K, V, m, l, k_cfg_f32, v_cfg_f32, scale_py)
#     return _mh_backward_cuda(
#         dOut,
#         Q, K, V,
#         cand_i32, mask_u8,
#         family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
#         head_mask_u8, runtime_i32,
#         k_cfg_f32, v_cfg_f32,
#         m, l,
#         scale_py,
#         cand_is_per_head,
#     )

# print("Built:", ext_name)
# print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
# print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))

In [ ]:
# ============================================================
# CELL: Stage-E FULL family-segmented LCHA (hardened)
# - Keeps old FP32 bridge + frozen saved tensor pattern
# - Adds runtime gate, head mask, explicit K/V alpha0/1/2 config
# - Adds V-side harmonic path
# - Uses Stage-A metadata:
#     family_bounds_i32 [10]
#     family_order_caps_i32 [5]
#     dispatch_meta_i32 [10]
#
# Runtime tensors:
#   head_mask_u8 : [H] uint8
#   runtime_i32  : [4] int32
#       [0]=lcha_enabled
#       [1]=use_v_lcha
#       [2]=trap_on_bad_cand
#       [3]=reserved
#
# Coeff tensors (float32, CUDA, contiguous):
#   k_cfg_f32 : [7] = [self0, win1, win2, ring1, ring2, anch1, anch2]
#   v_cfg_f32 : [7] = [self0, win1, win2, ring1, ring2, anch1, anch2]
# ============================================================

import os, hashlib, torch
from torch.utils.cpp_extension import load_inline

os.environ.setdefault("TORCH_EXTENSIONS_DIR", "/content/torch_extensions")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "1"

FUSED_VERBOSE_BUILD = False

CUDA_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include <cuda_bf16.h>
#include <vector>
#include <limits>
#include <cmath>
#include <cstdint>

#ifndef BLOCK_X
#define BLOCK_X 256
#endif

#ifndef V_TILE
#define V_TILE 256
#endif

#if (BLOCK_X & (BLOCK_X - 1)) != 0
#error "BLOCK_X must be a power of two."
#endif

static inline void check_cuda(const torch::Tensor& t, const char* name) {
    TORCH_CHECK(t.is_cuda(), name, " must be CUDA");
}
static inline void check_contig(const torch::Tensor& t, const char* name) {
    TORCH_CHECK(t.is_contiguous(), name, " must be contiguous");
}
static inline void check_dtype_eq(const torch::Tensor& a, const torch::Tensor& b, const char* an, const char* bn) {
    TORCH_CHECK(a.scalar_type() == b.scalar_type(), an, " and ", bn, " dtype must match");
}
static inline void check_same_device(const torch::Tensor& a, const torch::Tensor& b,
                                     const char* an, const char* bn) {
    TORCH_CHECK(a.device() == b.device(), an, " and ", bn, " must be on the same device");
}

__device__ __forceinline__ void trap_if(bool cond) { if (cond) asm("trap;"); }
__device__ __forceinline__ float expf_safe(float x) { return expf(x); }
__device__ __forceinline__ int isfinite_f(float x) { return isfinite(x); }

// ------------------------------------------------------------
// float-only kernel path
// wrappers cast Q/K/V/dOut -> FP32 before launch
// ------------------------------------------------------------
__device__ __forceinline__ float warp_reduce_sum(float v) {
#pragma unroll
    for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(0xffffffff, v, off);
    return v;
}

template <int BLOCK>
__device__ __forceinline__ float block_reduce_sum(float v, float* __restrict__ smem_warp_sums) {
    const int lane   = (int)threadIdx.x & 31;
    const int warp   = (int)threadIdx.x >> 5;
    const int nwarps = BLOCK >> 5;

    v = warp_reduce_sum(v);
    if (lane == 0) smem_warp_sums[warp] = v;
    __syncthreads();

    if (warp == 0) {
        float out = (lane < nwarps) ? smem_warp_sums[lane] : 0.0f;
        out = warp_reduce_sum(out);
        if (lane == 0) smem_warp_sums[0] = out;
    }
    __syncthreads();
    return smem_warp_sums[0];
}

__device__ __forceinline__ float dot_vec_float(const float* __restrict__ q,
                                               const float* __restrict__ k,
                                               int D) {
    const uintptr_t qa = (uintptr_t)q;
    const uintptr_t ka = (uintptr_t)k;
    const bool aligned8 = ((qa & 0x7u) == 0u) && ((ka & 0x7u) == 0u);
    const bool even = ((D & 1) == 0);

    float acc = 0.0f;
    if (aligned8 && even) {
        const float2* q2 = reinterpret_cast<const float2*>(q);
        const float2* k2 = reinterpret_cast<const float2*>(k);
        const int n2 = D >> 1;
        for (int i2 = (int)threadIdx.x; i2 < n2; i2 += BLOCK_X) {
            float2 a = q2[i2];
            float2 b = k2[i2];
            acc += a.x * b.x + a.y * b.y;
        }
    } else {
        for (int d = (int)threadIdx.x; d < D; d += BLOCK_X) acc += q[d] * k[d];
    }
    return acc;
}

__device__ __forceinline__ int64_t cand_row_base(int t, int h, int H, int C, int cand_is_per_head) {
    return (cand_is_per_head
            ? ((int64_t)t * (int64_t)H + (int64_t)h)
            : (int64_t)t) * (int64_t)C;
}

// ------------------------------------------------------------
// family helpers
// family index layout:
//   0=self, 1=window, 2=ring, 3=anchor, 4=promoted
// family_bounds layout:
//   [self_s,self_e, win_s,win_e, ring_s,ring_e, anch_s,anch_e, prom_s,prom_e]
// cfg layout for K/V:
//   [self0, win1, win2, ring1, ring2, anch1, anch2]
// ------------------------------------------------------------
__device__ __forceinline__ int slot_to_family_idx(int slot, const int* __restrict__ bounds) {
    if (slot < bounds[1]) return 0;
    if (slot < bounds[3]) return 1;
    if (slot < bounds[5]) return 2;
    if (slot < bounds[7]) return 3;
    return 4;
}

__device__ __forceinline__ int family_start(const int* __restrict__ bounds, int fam_idx) {
    return bounds[2 * fam_idx];
}
__device__ __forceinline__ int family_end(const int* __restrict__ bounds, int fam_idx) {
    return bounds[2 * fam_idx + 1];
}
__device__ __forceinline__ int min_i(int a, int b) { return (a < b) ? a : b; }

__device__ __forceinline__ float coeff1_for_family(const float* __restrict__ cfg, int fam_idx) {
    if (fam_idx == 1) return cfg[1];
    if (fam_idx == 2) return cfg[3];
    if (fam_idx == 3) return cfg[5];
    return 0.0f;
}
__device__ __forceinline__ float coeff2_for_family(const float* __restrict__ cfg, int fam_idx) {
    if (fam_idx == 1) return cfg[2];
    if (fam_idx == 2) return cfg[4];
    if (fam_idx == 3) return cfg[6];
    return 0.0f;
}


__device__ __forceinline__ int valid_family_degree(
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    int src_t,
    int h,
    int H,
    int C,
    int TQ,   // number of candidate rows (query rows)
    int TK,   // number of K/V rows (cache length)
    int cand_is_per_head,
    int fam_s,
    int fam_e
) {
    if (src_t < 0 || src_t >= TQ) return 0;

    const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
    int deg = 0;
    for (int kk = fam_s; kk < fam_e; ++kk) {
        if (!mask[row + (int64_t)kk]) continue;
        const int32_t j = cand[row + (int64_t)kk];
        if (j < 0 || j >= TK) continue;
        deg++;
    }
    return deg;
}

// q dot mean(X_family)
__device__ __forceinline__ float dot_family_mean(
    const float* __restrict__ q_ptr,
    const float* __restrict__ X,   // [TK,H,DX]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    int src_t,
    int h,
    int H,
    int DX,
    int TQ,
    int TK,
    int C,
    int cand_is_per_head,
    int fam_s,
    int fam_e,
    float* __restrict__ warp_sums
) {
    if (src_t < 0 || src_t >= TQ) {
        float z = 0.0f;
        return block_reduce_sum<BLOCK_X>(z, warp_sums) * 0.0f;
    }

    const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
    const int deg = valid_family_degree(cand, mask, src_t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e);
    if (deg <= 0) {
        float z = 0.0f;
        return block_reduce_sum<BLOCK_X>(z, warp_sums) * 0.0f;
    }

    float local = 0.0f;
    const float w = 1.0f / (float)deg;

    for (int kk = fam_s; kk < fam_e; ++kk) {
        if (!mask[row + (int64_t)kk]) continue;
        const int32_t j2 = cand[row + (int64_t)kk];
        if (j2 < 0 || j2 >= TK) continue;
        const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
        const float* xg = X + j2h * (int64_t)DX;
        local += w * dot_vec_float(q_ptr, xg, DX);
    }
    return block_reduce_sum<BLOCK_X>(local, warp_sums);
}

// q dot second-order mean(X_family)
// NOTE: In KV-cache mode (TQ < TK), second-order expansion is only taken for
// intermediate nodes j1 that have candidate rows available (j1 < TQ). This
// preserves full-sequence behavior when TQ == TK and avoids OOB reads during decode.
__device__ __forceinline__ float dot_family_second_mean(
    const float* __restrict__ q_ptr,
    const float* __restrict__ X,   // [TK,H,DX]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    int src_t,
    int h,
    int H,
    int DX,
    int TQ,
    int TK,
    int C,
    int cand_is_per_head,
    int fam_s,
    int fam_e,
    float* __restrict__ warp_sums
) {
    if (src_t < 0 || src_t >= TQ) {
        float z = 0.0f;
        return block_reduce_sum<BLOCK_X>(z, warp_sums) * 0.0f;
    }

    const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
    const int deg1 = valid_family_degree(cand, mask, src_t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e);
    if (deg1 <= 0) {
        float z = 0.0f;
        return block_reduce_sum<BLOCK_X>(z, warp_sums) * 0.0f;
    }

    float local = 0.0f;
    const float inv_deg1 = 1.0f / (float)deg1;

    for (int kk = fam_s; kk < fam_e; ++kk) {
        if (!mask[row + (int64_t)kk]) continue;
        const int32_t j1 = cand[row + (int64_t)kk];
        if (j1 < 0 || j1 >= TK) continue;

        const int deg2 = valid_family_degree(cand, mask, j1, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e);
        if (deg2 <= 0) continue;

        const float w = inv_deg1 / (float)deg2;
        const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);

        for (int uu = fam_s; uu < fam_e; ++uu) {
            if (!mask[row1 + (int64_t)uu]) continue;
            const int32_t j2 = cand[row1 + (int64_t)uu];
            if (j2 < 0 || j2 >= TK) continue;
            const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
            const float* xg = X + j2h * (int64_t)DX;
            local += w * dot_vec_float(q_ptr, xg, DX);
        }
    }
    return block_reduce_sum<BLOCK_X>(local, warp_sums);
}

__device__ __forceinline__ float family_mean_at_d(
    const float* __restrict__ X,   // [TK,H,DX]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    int src_t,
    int h,
    int H,
    int DX,
    int TQ,
    int TK,
    int C,
    int cand_is_per_head,
    int fam_s,
    int fam_e,
    int d
) {
    if (src_t < 0 || src_t >= TQ) return 0.0f;

    const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
    int deg = 0;
    float sum = 0.0f;

    for (int kk = fam_s; kk < fam_e; ++kk) {
        if (!mask[row + (int64_t)kk]) continue;
        const int32_t j2 = cand[row + (int64_t)kk];
        if (j2 < 0 || j2 >= TK) continue;
        deg++;
        const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
        sum += X[j2h * (int64_t)DX + (int64_t)d];
    }
    return (deg > 0) ? (sum / (float)deg) : 0.0f;
}

// NOTE: same KV-cache safety rule as dot_family_second_mean.
__device__ __forceinline__ float family_second_mean_at_d(
    const float* __restrict__ X,   // [TK,H,DX]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    int src_t,
    int h,
    int H,
    int DX,
    int TQ,
    int TK,
    int C,
    int cand_is_per_head,
    int fam_s,
    int fam_e,
    int d
) {
    if (src_t < 0 || src_t >= TQ) return 0.0f;

    const int deg1 = valid_family_degree(cand, mask, src_t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e);
    if (deg1 <= 0) return 0.0f;

    const int64_t row = cand_row_base(src_t, h, H, C, cand_is_per_head);
    const float inv_deg1 = 1.0f / (float)deg1;
    float sum = 0.0f;

    for (int kk = fam_s; kk < fam_e; ++kk) {
        if (!mask[row + (int64_t)kk]) continue;
        const int32_t j1 = cand[row + (int64_t)kk];
        if (j1 < 0 || j1 >= TK) continue;

        const int deg2 = valid_family_degree(cand, mask, j1, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e);
        if (deg2 <= 0) continue;

        const float w = inv_deg1 / (float)deg2;
        const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);

        for (int uu = fam_s; uu < fam_e; ++uu) {
            if (!mask[row1 + (int64_t)uu]) continue;
            const int32_t j2 = cand[row1 + (int64_t)uu];
            if (j2 < 0 || j2 >= TK) continue;
            const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
            sum += w * X[j2h * (int64_t)DX + (int64_t)d];
        }
    }
    return sum;
}

__device__ __forceinline__ float segmented_k_score_unscaled(
    const float* __restrict__ smem_q,      // [D]
    const float* __restrict__ smem_k,      // [D]
    const float* __restrict__ K,           // [TK,H,D]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    int t,
    int h,
    int H,
    int D,
    int TQ,
    int TK,
    int C,
    int cand_is_per_head,
    int slot_jj,
    const int* __restrict__ bounds,
    const int* __restrict__ caps,
    int stage_order,
    int lcha_enabled_for_head,
    const float* __restrict__ k_cfg,
    float* __restrict__ warp_sums,
    int* family_idx_out,
    int* eff_order_out
) {
    const float qk0 = block_reduce_sum<BLOCK_X>(dot_vec_float(smem_q, smem_k, D), warp_sums);
    const float self0 = k_cfg[0];

    const int fam_idx = slot_to_family_idx(slot_jj, bounds);
    int eff_order = 0;
    if (lcha_enabled_for_head && fam_idx >= 1 && fam_idx <= 3) {
        const int fam_cap = caps[fam_idx];
        eff_order = min_i(stage_order, fam_cap);
    }

    if (family_idx_out) *family_idx_out = fam_idx;
    if (eff_order_out) *eff_order_out = eff_order;

    float score_unscaled = self0 * qk0;

    if (lcha_enabled_for_head && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
        const int fam_s = family_start(bounds, fam_idx);
        const int fam_e = family_end(bounds, fam_idx);

        const float c1 = coeff1_for_family(k_cfg, fam_idx);
        const float c2 = coeff2_for_family(k_cfg, fam_idx);

        if (c1 != 0.0f) {
            const float qk1 = dot_family_mean(
                smem_q, K, cand, mask,
                t, h, H, D, TQ, TK, C, cand_is_per_head,
                fam_s, fam_e, warp_sums
            );
            score_unscaled += c1 * qk1;
        }

        if (TQ == TK && eff_order >= 2 && c2 != 0.0f) {
            const float qk2 = dot_family_second_mean(
                smem_q, K, cand, mask,
                t, h, H, D, TQ, TK, C, cand_is_per_head,
                fam_s, fam_e, warp_sums
            );
            score_unscaled += c2 * qk2;
        }
    }

    return score_unscaled;
}

// ============================================================================
// Forward kernel
// ============================================================================

__global__ void fused_fwd_streaming_mh_family_full_kernel(
    const float* __restrict__ Q,      // [QT,H,D]
    const float* __restrict__ K,      // [KV_T,H,D]
    const float* __restrict__ V,      // [KV_T,H,DV]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    const int32_t* __restrict__ family_bounds,      // [10]
    const int32_t* __restrict__ family_order_caps,  // [5]
    const int32_t* __restrict__ dispatch_meta,      // [10]
    const uint8_t* __restrict__ head_mask,          // [H]
    const int32_t* __restrict__ runtime_i32,        // [4]
    const float* __restrict__ k_cfg,                // [7]
    const float* __restrict__ v_cfg,                // [7]
    float* __restrict__ m_out,       // [QT*H]
    float* __restrict__ l_out,       // [QT*H]
    float* __restrict__ out,         // [QT,H,DV]
    int QT, int KV_T, int H, int D, int DV, int C,
    float scale,
    int cand_is_per_head,
    int query_start,
    int q_len_per_batch,
    int kv_len_per_batch
) {
    trap_if(blockDim.x != BLOCK_X);

    const int tq = (int)blockIdx.x;
    const int h = (int)blockIdx.y;
    const int tid = (int)threadIdx.x;

    if (tq >= QT || h >= H) return;
    if (q_len_per_batch <= 0 || kv_len_per_batch <= 0) return;

    const int64_t batch_id = (int64_t)tq / (int64_t)q_len_per_batch;
    const int64_t local_q  = (int64_t)tq - batch_id * (int64_t)q_len_per_batch;
    const int64_t abs_t64  = batch_id * (int64_t)kv_len_per_batch
                           + (int64_t)query_start
                           + local_q;

    if (abs_t64 < 0 || abs_t64 >= KV_T) return;

    const int abs_t = (int)abs_t64;

    const int64_t qth = (int64_t)tq * (int64_t)H + (int64_t)h;
    const int64_t row = cand_row_base(abs_t, h, H, C, cand_is_per_head);

    __shared__ int sbounds[10];
    __shared__ int scaps[5];
    __shared__ int sruntime[4];
    __shared__ float skcfg[7];
    __shared__ float svcfg[7];
    __shared__ int sstage_order;
    __shared__ int s_head_lcha_on;
    __shared__ int s_use_v;

    if (tid < 10) sbounds[tid] = (int)family_bounds[tid];
    if (tid < 5)  scaps[tid]   = (int)family_order_caps[tid];
    if (tid < 4)  sruntime[tid]= (int)runtime_i32[tid];
    if (tid < 7)  skcfg[tid]   = k_cfg[tid];
    if (tid < 7)  svcfg[tid]   = v_cfg[tid];
    if (tid == 0) {
        sstage_order = (int)dispatch_meta[4];
        s_head_lcha_on = (sruntime[0] != 0) && (head_mask[h] != 0) && (sstage_order > 0);
        s_use_v = (sruntime[1] != 0);
    }
    __syncthreads();

    extern __shared__ unsigned char smem_raw[];
    float* smem_q = reinterpret_cast<float*>(smem_raw);
    float* smem_k = reinterpret_cast<float*>(smem_raw + (size_t)256 * sizeof(float));
    float* smem_warp = reinterpret_cast<float*>(smem_raw + (size_t)512 * sizeof(float));
    float* smem_v = reinterpret_cast<float*>(
        smem_raw + (size_t)512 * sizeof(float) + (size_t)(BLOCK_X / 32) * sizeof(float));

    const float* qg = Q + qth * (int64_t)D;
    for (int d = tid; d < D; d += BLOCK_X) smem_q[d] = qg[d];
    __syncthreads();

    for (int dv = tid; dv < DV; dv += BLOCK_X) {
        out[qth * (int64_t)DV + (int64_t)dv] = 0.0f;
    }
    __syncthreads();

    float m = -INFINITY;
    float l = 0.0f;

    for (int jj = 0; jj < C; ++jj) {
        const uint8_t m_ij = mask[row + (int64_t)jj];
        if (!m_ij) continue;

        const int32_t j = cand[row + (int64_t)jj];
        if (j < 0 || j >= KV_T) {
            if (sruntime[2]) trap_if(true);
            continue;
        }

        const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;
        const float* kg = K + jh * (int64_t)D;
        for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
        __syncthreads();

        int fam_idx = 0;
        int eff_order = 0;
        float score_unscaled = segmented_k_score_unscaled(
            smem_q, smem_k, K, cand, mask,
            abs_t, h, H, D, KV_T, KV_T, C, cand_is_per_head,
            jj, sbounds, scaps, sstage_order,
            s_head_lcha_on, skcfg, smem_warp,
            &fam_idx, &eff_order
        );

        float score = score_unscaled * scale;
        if (!isfinite_f(score)) { __syncthreads(); continue; }

        float m_new = fmaxf(m, score);
        float exp_m = (m == -INFINITY) ? 0.0f : expf_safe(m - m_new);
        float exp_s = expf_safe(score - m_new);
        if (!isfinite_f(exp_s)) exp_s = 0.0f;

        float l_new = l * exp_m + exp_s;
        if (!isfinite_f(l_new) || l_new <= 0.0f) { __syncthreads(); continue; }

        float beta  = (l * exp_m) / l_new;
        float alpha = exp_s / l_new;

        const float* vg = V + jh * (int64_t)DV;
        const float v_self0 = svcfg[0];

        int fam_s = 0;
        int fam_e = 0;
        float v1 = 0.0f;
        float v2 = 0.0f;
        if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
            fam_s = family_start(sbounds, fam_idx);
            fam_e = family_end(sbounds, fam_idx);
            v1 = coeff1_for_family(svcfg, fam_idx);
            v2 = coeff2_for_family(svcfg, fam_idx);
        }

        for (int dv0 = 0; dv0 < DV; dv0 += V_TILE) {
            const int dv = dv0 + tid;
            float vt = 0.0f;

            if (dv < DV) {
                vt = v_self0 * vg[dv];
                if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
                    if (v1 != 0.0f) {
                        vt += v1 * family_mean_at_d(
                            V, cand, mask,
                            abs_t, h, H, DV, KV_T, KV_T, C, cand_is_per_head,
                            fam_s, fam_e, dv
                        );
                    }
                    if (eff_order >= 2 && v2 != 0.0f) {
                        vt += v2 * family_second_mean_at_d(
                            V, cand, mask,
                            abs_t, h, H, DV, KV_T, KV_T, C, cand_is_per_head,
                            fam_s, fam_e, dv
                        );
                    }
                }
            }

            if (tid < V_TILE) smem_v[tid] = (dv < DV) ? vt : 0.0f;
            __syncthreads();

            if (dv < DV) {
                float old = out[qth * (int64_t)DV + (int64_t)dv];
                float upd = beta * old + alpha * smem_v[tid];
                out[qth * (int64_t)DV + (int64_t)dv] = upd;
            }
            __syncthreads();
        }

        m = m_new;
        l = l_new;
    }

    if (tid == 0) {
        m_out[qth] = m;
        l_out[qth] = l;
    }
}

// ============================================================================
// Backward kernel
// ============================================================================

__global__ void fused_bwd_mh_family_full_kernel(
    const float* __restrict__ dOut,    // [TQ,H,DV]
    const float* __restrict__ Q,       // [TQ,H,D]
    const float* __restrict__ K,       // [TK,H,D]
    const float* __restrict__ V,       // [TK,H,DV]
    const int32_t* __restrict__ cand,
    const uint8_t* __restrict__ mask,
    const int32_t* __restrict__ family_bounds,      // [10]
    const int32_t* __restrict__ family_order_caps,  // [5]
    const int32_t* __restrict__ dispatch_meta,      // [10]
    const uint8_t* __restrict__ head_mask,          // [H]
    const int32_t* __restrict__ runtime_i32,        // [4]
    const float* __restrict__ k_cfg,                // [7]
    const float* __restrict__ v_cfg,                // [7]
    const float* __restrict__ m_in,   // [TQ*H]
    const float* __restrict__ l_in,   // [TQ*H]
    float* __restrict__ dQ,           // [TQ*H,D]
    float* __restrict__ dK,           // [TK*H,D]
    float* __restrict__ dV,           // [TK*H,DV]
    int TQ, int TK, int H, int D, int DV, int C,
    float scale,
    int cand_is_per_head
) {
    trap_if(blockDim.x != BLOCK_X);

    const int t = (int)blockIdx.x;
    const int h = (int)blockIdx.y;
    const int tid = (int)threadIdx.x;
    if (t >= TQ || h >= H) return;

    const int64_t th = (int64_t)t * (int64_t)H + (int64_t)h;
    const int64_t row = cand_row_base(t, h, H, C, cand_is_per_head);

    float m = m_in[th];
    float l = l_in[th];
    if (!(isfinite_f(m)) || !(isfinite_f(l)) || l <= 0.0f) return;

    __shared__ int sbounds[10];
    __shared__ int scaps[5];
    __shared__ int sruntime[4];
    __shared__ float skcfg[7];
    __shared__ float svcfg[7];
    __shared__ int sstage_order;
    __shared__ int s_head_lcha_on;
    __shared__ int s_use_v;

    __shared__ float k_acc1[5];
    __shared__ float k_acc2[5];
    __shared__ float v_acc1[5];
    __shared__ float v_acc2[5];

    if (tid < 10) sbounds[tid] = (int)family_bounds[tid];
    if (tid < 5)  scaps[tid]   = (int)family_order_caps[tid];
    if (tid < 4)  sruntime[tid]= (int)runtime_i32[tid];
    if (tid < 7)  skcfg[tid]   = k_cfg[tid];
    if (tid < 7)  svcfg[tid]   = v_cfg[tid];
    if (tid < 5) {
        k_acc1[tid] = 0.0f;
        k_acc2[tid] = 0.0f;
        v_acc1[tid] = 0.0f;
        v_acc2[tid] = 0.0f;
    }
    if (tid == 0) {
        sstage_order = (int)dispatch_meta[4];
        s_head_lcha_on = (sruntime[0] != 0) && (head_mask[h] != 0) && (sstage_order > 0);
        s_use_v = (sruntime[1] != 0);
    }
    __syncthreads();

    __shared__ float smem_q[256];
    __shared__ float smem_k[256];
    __shared__ float red[BLOCK_X];
    __shared__ float warp_sums[BLOCK_X / 32];

    const float* qg = Q + th * (int64_t)D;
    for (int d = tid; d < D; d += BLOCK_X) smem_q[d] = qg[d];
    __syncthreads();

    const float* dout = dOut + th * (int64_t)DV;

    // --------------------------------------------------------
    // Pass 1: do_out = sum_j p_j * (dOut · Vtilde_j)
    // --------------------------------------------------------
    float local_do_out = 0.0f;

    for (int jj = 0; jj < C; ++jj) {
        const uint8_t m_ij = mask[row + (int64_t)jj];
        if (!m_ij) continue;

        const int32_t j = cand[row + (int64_t)jj];
        if (j < 0 || j >= TK) {
            if (sruntime[2]) trap_if(true);
            continue;
        }

        const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;
        const float* kg = K + jh * (int64_t)D;
        for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
        __syncthreads();

        int fam_idx = 0;
        int eff_order = 0;
        float score_unscaled = segmented_k_score_unscaled(
            smem_q, smem_k, K, cand, mask,
            t, h, H, D, TQ, TK, C, cand_is_per_head,
            jj, sbounds, scaps, sstage_order,
            s_head_lcha_on, skcfg, warp_sums,
            &fam_idx, &eff_order
        );

        float score = score_unscaled * scale;
        if (!isfinite_f(score)) { __syncthreads(); continue; }

        float z = score - m;
        z = fminf(z, 0.0f);
        float p = expf_safe(z) / l;
        if (!isfinite_f(p)) { __syncthreads(); continue; }
        if (p < 0.0f) p = 0.0f;
        if (p > 1.0f) p = 1.0f;

        const float* vg = V + jh * (int64_t)DV;
        const float v_self0 = svcfg[0];

        int fam_s = 0;
        int fam_e = 0;
        float v1 = 0.0f;
        float v2 = 0.0f;
        if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
            fam_s = family_start(sbounds, fam_idx);
            fam_e = family_end(sbounds, fam_idx);
            v1 = coeff1_for_family(svcfg, fam_idx);
            v2 = coeff2_for_family(svcfg, fam_idx);
        }

        float local_do_v = 0.0f;
        for (int dv = tid; dv < DV; dv += BLOCK_X) {
            float vt = v_self0 * vg[dv];
            if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
                if (v1 != 0.0f) {
                    vt += v1 * family_mean_at_d(
                        V, cand, mask,
                        t, h, H, DV, TQ, TK, C, cand_is_per_head,
                        fam_s, fam_e, dv
                    );
                }
                if (TQ == TK && eff_order >= 2 && v2 != 0.0f) {
                    vt += v2 * family_second_mean_at_d(
                        V, cand, mask,
                        t, h, H, DV, TQ, TK, C, cand_is_per_head,
                        fam_s, fam_e, dv
                    );
                }
            }
            local_do_v += dout[dv] * vt;
        }

        red[tid] = local_do_v;
        __syncthreads();
        for (int off = BLOCK_X / 2; off > 0; off >>= 1) {
            if (tid < off) red[tid] += red[tid + off];
            __syncthreads();
        }
        float do_v = red[0];
        if (!isfinite_f(do_v)) { __syncthreads(); continue; }

        local_do_out += p * do_v;
        __syncthreads();
    }

    if (tid == 0) red[0] = local_do_out;
    __syncthreads();
    float do_out = red[0];

    for (int d = tid; d < D; d += BLOCK_X) {
        dQ[th * (int64_t)D + (int64_t)d] = 0.0f;
    }
    __syncthreads();

    // --------------------------------------------------------
    // Pass 2: raw grads + family accumulators
    // --------------------------------------------------------
    for (int jj = 0; jj < C; ++jj) {
        const uint8_t m_ij = mask[row + (int64_t)jj];
        if (!m_ij) continue;

        const int32_t j = cand[row + (int64_t)jj];
        if (j < 0 || j >= TK) {
            if (sruntime[2]) trap_if(true);
            continue;
        }

        const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;
        const float* kg = K + jh * (int64_t)D;
        for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
        __syncthreads();

        int fam_idx = 0;
        int eff_order = 0;
        float score_unscaled = segmented_k_score_unscaled(
            smem_q, smem_k, K, cand, mask,
            t, h, H, D, TQ, TK, C, cand_is_per_head,
            jj, sbounds, scaps, sstage_order,
            s_head_lcha_on, skcfg, warp_sums,
            &fam_idx, &eff_order
        );

        float score = score_unscaled * scale;
        if (!isfinite_f(score)) { __syncthreads(); continue; }

        float z = score - m;
        z = fminf(z, 0.0f);
        float p = expf_safe(z) / l;
        if (!isfinite_f(p)) { __syncthreads(); continue; }
        if (p < 0.0f) p = 0.0f;
        if (p > 1.0f) p = 1.0f;

        const float* vg = V + jh * (int64_t)DV;
        const float k_self0 = skcfg[0];
        const float v_self0 = svcfg[0];

        int fam_s = 0;
        int fam_e = 0;
        float k1 = 0.0f, k2 = 0.0f, v1 = 0.0f, v2 = 0.0f;
        if (s_head_lcha_on && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
            fam_s = family_start(sbounds, fam_idx);
            fam_e = family_end(sbounds, fam_idx);
            k1 = coeff1_for_family(skcfg, fam_idx);
            k2 = coeff2_for_family(skcfg, fam_idx);
            if (s_use_v) {
                v1 = coeff1_for_family(svcfg, fam_idx);
                v2 = coeff2_for_family(svcfg, fam_idx);
            }
        }

        // raw dV term
        for (int dv = tid; dv < DV; dv += BLOCK_X) {
            atomicAdd(&dV[jh * (int64_t)DV + (int64_t)dv], p * v_self0 * dout[dv]);
        }

        // do_v = dOut · Vtilde_j
        float local_do_v = 0.0f;
        for (int dv = tid; dv < DV; dv += BLOCK_X) {
            float vt = v_self0 * vg[dv];
            if (s_head_lcha_on && s_use_v && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
                if (v1 != 0.0f) {
                    vt += v1 * family_mean_at_d(
                        V, cand, mask,
                        t, h, H, DV, TQ, TK, C, cand_is_per_head,
                        fam_s, fam_e, dv
                    );
                }
                if (TQ == TK && eff_order >= 2 && v2 != 0.0f) {
                    vt += v2 * family_second_mean_at_d(
                        V, cand, mask,
                        t, h, H, DV, TQ, TK, C, cand_is_per_head,
                        fam_s, fam_e, dv
                    );
                }
            }
            local_do_v += dout[dv] * vt;
        }

        red[tid] = local_do_v;
        __syncthreads();
        for (int off = BLOCK_X / 2; off > 0; off >>= 1) {
            if (tid < off) red[tid] += red[tid + off];
            __syncthreads();
        }
        float do_v = red[0];
        if (!isfinite_f(do_v)) { __syncthreads(); continue; }

        float g = p * (do_v - do_out);
        if (!isfinite_f(g)) { __syncthreads(); continue; }

        const float gs = g * scale;
        if (!isfinite_f(gs)) { __syncthreads(); continue; }

        // raw self term derivative
        const float gs_self = gs * k_self0;
        if (k_self0 != 0.0f && isfinite_f(gs_self)) {
            for (int d = tid; d < D; d += BLOCK_X) {
                dQ[th * (int64_t)D + (int64_t)d] += gs_self * kg[d];
            }
            for (int d = tid; d < D; d += BLOCK_X) {
                atomicAdd(&dK[jh * (int64_t)D + (int64_t)d], gs_self * smem_q[d]);
            }
        }

        // family accumulators
        if (tid == 0 && s_head_lcha_on && fam_idx >= 1 && fam_idx <= 3 && eff_order > 0) {
            if (k1 != 0.0f) k_acc1[fam_idx] += gs * k1;
            if (TQ == TK && eff_order >= 2 && k2 != 0.0f) k_acc2[fam_idx] += gs * k2;
            if (s_use_v) {
                if (v1 != 0.0f) v_acc1[fam_idx] += p * v1;
                if (TQ == TK && eff_order >= 2 && v2 != 0.0f) v_acc2[fam_idx] += p * v2;
            }
        }
        __syncthreads();
    }

    // --------------------------------------------------------
    // Pass 3A: propagate K-family gradients
    // --------------------------------------------------------
    for (int fam_idx = 1; fam_idx <= 3; ++fam_idx) {
        const float S1 = k_acc1[fam_idx];
        const float S2 = k_acc2[fam_idx];
        if ((!isfinite_f(S1) || S1 == 0.0f) && (!isfinite_f(S2) || S2 == 0.0f)) {
            __syncthreads();
            continue;
        }

        const int fam_s = family_start(sbounds, fam_idx);
        const int fam_e = family_end(sbounds, fam_idx);

        // dQ contribution from first order
        if (S1 != 0.0f) {
            for (int d = tid; d < D; d += BLOCK_X) {
                float m1 = family_mean_at_d(
                    K, cand, mask,
                    t, h, H, D, TQ, TK, C, cand_is_per_head,
                    fam_s, fam_e, d
                );
                dQ[th * (int64_t)D + (int64_t)d] += S1 * m1;
            }

            const int deg1 = valid_family_degree(
                cand, mask, t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e
            );
            if (deg1 > 0) {
                const float w1 = S1 / (float)deg1;
                const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);
                for (int kk = fam_s; kk < fam_e; ++kk) {
                    if (!mask[row0 + (int64_t)kk]) continue;
                    const int32_t j1 = cand[row0 + (int64_t)kk];
                    if (j1 < 0 || j1 >= TK) continue;
                    const int64_t j1h = (int64_t)j1 * (int64_t)H + (int64_t)h;
                    for (int d = tid; d < D; d += BLOCK_X) {
                        atomicAdd(&dK[j1h * (int64_t)D + (int64_t)d], w1 * smem_q[d]);
                    }
                }
            }
        }

        // dQ + dK contribution from second order
        if (TQ == TK && S2 != 0.0f) {
            for (int d = tid; d < D; d += BLOCK_X) {
                float m2 = family_second_mean_at_d(
                    K, cand, mask,
                    t, h, H, D, TQ, TK, C, cand_is_per_head,
                    fam_s, fam_e, d
                );
                dQ[th * (int64_t)D + (int64_t)d] += S2 * m2;
            }

            const int deg1 = valid_family_degree(
                cand, mask, t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e
            );
            if (deg1 > 0) {
                const float inv_deg1 = 1.0f / (float)deg1;
                const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);

                for (int kk = fam_s; kk < fam_e; ++kk) {
                    if (!mask[row0 + (int64_t)kk]) continue;
                    const int32_t j1 = cand[row0 + (int64_t)kk];
                    if (j1 < 0 || j1 >= TK) continue;

                    const int deg2 = valid_family_degree(
                        cand, mask, j1, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e
                    );
                    if (deg2 <= 0) continue;

                    const float w2 = S2 * inv_deg1 / (float)deg2;
                    const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);
                    for (int uu = fam_s; uu < fam_e; ++uu) {
                        if (!mask[row1 + (int64_t)uu]) continue;
                        const int32_t j2 = cand[row1 + (int64_t)uu];
                        if (j2 < 0 || j2 >= TK) continue;
                        const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
                        for (int d = tid; d < D; d += BLOCK_X) {
                            atomicAdd(&dK[j2h * (int64_t)D + (int64_t)d], w2 * smem_q[d]);
                        }
                    }
                }
            }
        }

        __syncthreads();
    }

    // --------------------------------------------------------
    // Pass 3B: propagate V-family gradients
    // --------------------------------------------------------
    if (s_use_v) {
        for (int fam_idx = 1; fam_idx <= 3; ++fam_idx) {
            const float P1 = v_acc1[fam_idx];
            const float P2 = v_acc2[fam_idx];
            if ((!isfinite_f(P1) || P1 == 0.0f) && (!isfinite_f(P2) || P2 == 0.0f)) {
                __syncthreads();
                continue;
            }

            const int fam_s = family_start(sbounds, fam_idx);
            const int fam_e = family_end(sbounds, fam_idx);

            if (P1 != 0.0f) {
                const int deg1 = valid_family_degree(
                    cand, mask, t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e
                );
                if (deg1 > 0) {
                    const float w1 = P1 / (float)deg1;
                    const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);
                    for (int kk = fam_s; kk < fam_e; ++kk) {
                        if (!mask[row0 + (int64_t)kk]) continue;
                        const int32_t j1 = cand[row0 + (int64_t)kk];
                        if (j1 < 0 || j1 >= TK) continue;
                        const int64_t j1h = (int64_t)j1 * (int64_t)H + (int64_t)h;
                        for (int dv = tid; dv < DV; dv += BLOCK_X) {
                            atomicAdd(&dV[j1h * (int64_t)DV + (int64_t)dv], w1 * dout[dv]);
                        }
                    }
                }
            }

            if (TQ == TK && P2 != 0.0f) {
                const int deg1 = valid_family_degree(
                    cand, mask, t, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e
                );
                if (deg1 > 0) {
                    const float inv_deg1 = 1.0f / (float)deg1;
                    const int64_t row0 = cand_row_base(t, h, H, C, cand_is_per_head);

                    for (int kk = fam_s; kk < fam_e; ++kk) {
                        if (!mask[row0 + (int64_t)kk]) continue;
                        const int32_t j1 = cand[row0 + (int64_t)kk];
                        if (j1 < 0 || j1 >= TK) continue;

                        const int deg2 = valid_family_degree(
                            cand, mask, j1, h, H, C, TQ, TK, cand_is_per_head, fam_s, fam_e
                        );
                        if (deg2 <= 0) continue;

                        const float w2 = P2 * inv_deg1 / (float)deg2;
                        const int64_t row1 = cand_row_base(j1, h, H, C, cand_is_per_head);
                        for (int uu = fam_s; uu < fam_e; ++uu) {
                            if (!mask[row1 + (int64_t)uu]) continue;
                            const int32_t j2 = cand[row1 + (int64_t)uu];
                            if (j2 < 0 || j2 >= TK) continue;
                            const int64_t j2h = (int64_t)j2 * (int64_t)H + (int64_t)h;
                            for (int dv = tid; dv < DV; dv += BLOCK_X) {
                                atomicAdd(&dV[j2h * (int64_t)DV + (int64_t)dv], w2 * dout[dv]);
                            }
                        }
                    }
                }
            }

            __syncthreads();
        }
    }
}

// ------------------------------------------------------------
// cast fp32 gradients back to original dtype
// ------------------------------------------------------------
__global__ void cast_f32_to_f32(const float* src, float* dst, int64_t n) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) dst[i] = src[i];
}
__global__ void cast_f32_to_f16(const float* src, at::Half* dst, int64_t n) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) dst[i] = (at::Half)__float2half_rn(src[i]);
}
__global__ void cast_f32_to_bf16(const float* src, at::BFloat16* dst, int64_t n) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
#if defined(__CUDA_ARCH__) && __CUDA_ARCH__ >= 800
    if (i < n) dst[i] = (at::BFloat16)__float2bfloat16_rn(src[i]);
#else
    if (i < n) asm("trap;");
#endif
}

// ============================================================================
// C++ wrappers (hardened like old Stage-E)
// ============================================================================

std::vector<torch::Tensor> mh_forward(
    torch::Tensor Q,        // [QT,H,D]
    torch::Tensor K,        // [KV_T,H,D]
    torch::Tensor V,        // [KV_T,H,DV]
    torch::Tensor cand_i32, // [KV_T,C] or [KV_T*H,C]
    torch::Tensor mask_u8,  // [KV_T,C] or [KV_T*H,C]
    torch::Tensor family_bounds_i32,      // [10]
    torch::Tensor family_order_caps_i32,  // [5]
    torch::Tensor dispatch_meta_i32,      // [10]
    torch::Tensor head_mask_u8,           // [H]
    torch::Tensor runtime_i32,            // [4]
    torch::Tensor k_cfg_f32,              // [7]
    torch::Tensor v_cfg_f32,              // [7]
    double scale_py,
    bool cand_is_per_head,
    int64_t query_start,
    int64_t q_len_per_batch,
    int64_t kv_len_per_batch
) {
    check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
    check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
    check_cuda(family_bounds_i32,"family_bounds");
    check_cuda(family_order_caps_i32,"family_order_caps");
    check_cuda(dispatch_meta_i32,"dispatch_meta");
    check_cuda(head_mask_u8,"head_mask");
    check_cuda(runtime_i32,"runtime_i32");
    check_cuda(k_cfg_f32,"k_cfg");
    check_cuda(v_cfg_f32,"v_cfg");

    c10::cuda::CUDAGuard guard(Q.device());
    check_same_device(Q, K, "Q", "K");
    check_same_device(Q, V, "Q", "V");
    check_same_device(Q, cand_i32, "Q", "cand");
    check_same_device(Q, mask_u8, "Q", "mask");
    check_same_device(Q, family_bounds_i32, "Q", "family_bounds");
    check_same_device(Q, family_order_caps_i32, "Q", "family_order_caps");
    check_same_device(Q, dispatch_meta_i32, "Q", "dispatch_meta");
    check_same_device(Q, head_mask_u8, "Q", "head_mask");
    check_same_device(Q, runtime_i32, "Q", "runtime_i32");
    check_same_device(Q, k_cfg_f32, "Q", "k_cfg");
    check_same_device(Q, v_cfg_f32, "Q", "v_cfg");

    check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
    check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask");
    check_contig(family_bounds_i32,"family_bounds");
    check_contig(family_order_caps_i32,"family_order_caps");
    check_contig(dispatch_meta_i32,"dispatch_meta");
    check_contig(head_mask_u8,"head_mask");
    check_contig(runtime_i32,"runtime_i32");
    check_contig(k_cfg_f32,"k_cfg");
    check_contig(v_cfg_f32,"v_cfg");

    TORCH_CHECK(Q.dim()==3 && K.dim()==3 && V.dim()==3,
                "Q/K/V must be [QT,H,D]/[KV_T,H,D]/[KV_T,H,DV]");
    TORCH_CHECK(cand_i32.dim()==2 && mask_u8.dim()==2, "cand/mask must be 2D");
    TORCH_CHECK(cand_i32.scalar_type()==at::kInt, "cand must be int32");
    TORCH_CHECK(mask_u8.scalar_type()==at::kByte, "mask must be uint8");
    TORCH_CHECK(family_bounds_i32.dim()==1 && family_bounds_i32.numel()==10 && family_bounds_i32.scalar_type()==at::kInt,
                "family_bounds must be int32 [10]");
    TORCH_CHECK(family_order_caps_i32.dim()==1 && family_order_caps_i32.numel()==5 && family_order_caps_i32.scalar_type()==at::kInt,
                "family_order_caps must be int32 [5]");
    TORCH_CHECK(dispatch_meta_i32.dim()==1 && dispatch_meta_i32.numel()==10 && dispatch_meta_i32.scalar_type()==at::kInt,
                "dispatch_meta must be int32 [10]");
    TORCH_CHECK(head_mask_u8.dim()==1 && head_mask_u8.scalar_type()==at::kByte,
                "head_mask must be uint8 [H]");
    TORCH_CHECK(runtime_i32.dim()==1 && runtime_i32.numel()==4 && runtime_i32.scalar_type()==at::kInt,
                "runtime_i32 must be int32 [4]");
    TORCH_CHECK(k_cfg_f32.dim()==1 && k_cfg_f32.numel()==7 && k_cfg_f32.scalar_type()==at::kFloat,
                "k_cfg_f32 must be float32 [7]");
    TORCH_CHECK(v_cfg_f32.dim()==1 && v_cfg_f32.numel()==7 && v_cfg_f32.scalar_type()==at::kFloat,
                "v_cfg_f32 must be float32 [7]");
    TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X");

    check_dtype_eq(Q,K,"Q","K");
    check_dtype_eq(Q,V,"Q","V");
    TORCH_CHECK(Q.is_floating_point() && K.is_floating_point() && V.is_floating_point(),
                "Q/K/V must be floating point");
    TORCH_CHECK(std::isfinite(scale_py), "scale must be finite");

    const int64_t QT64  = Q.size(0);
    const int64_t KVT64 = K.size(0);
    const int64_t H64   = Q.size(1);
    const int64_t D64   = Q.size(2);
    const int64_t DV64  = V.size(2);
    const int64_t C64   = cand_i32.size(1);

    TORCH_CHECK(QT64 > 0 && KVT64 > 0 && H64 > 0 && D64 > 0 && DV64 > 0 && C64 > 0,
                "QT/KV_T/H/D/DV/C must be positive");
    TORCH_CHECK(K.size(1)==H64 && K.size(2)==D64, "K shape mismatch");
    TORCH_CHECK(V.size(0)==KVT64 && V.size(1)==H64, "V shape mismatch");
    TORCH_CHECK(mask_u8.size(1)==C64, "mask C mismatch");
    TORCH_CHECK(head_mask_u8.numel()==H64, "head_mask size mismatch");
    TORCH_CHECK(D64 <= 256, "D must be <= 256");

    TORCH_CHECK(query_start >= 0, "query_start must be non-negative");
    TORCH_CHECK(q_len_per_batch > 0, "q_len_per_batch must be positive");
    TORCH_CHECK(kv_len_per_batch > 0, "kv_len_per_batch must be positive");
    TORCH_CHECK(query_start <= std::numeric_limits<int64_t>::max() - q_len_per_batch,
                "query_start + q_len_per_batch would overflow int64");
    TORCH_CHECK(query_start + q_len_per_batch <= kv_len_per_batch,
                "query_start + q_len_per_batch must be <= kv_len_per_batch");
    TORCH_CHECK(QT64 % q_len_per_batch == 0,
                "QT must be divisible by q_len_per_batch");
    TORCH_CHECK(KVT64 % kv_len_per_batch == 0,
                "KV_T must be divisible by kv_len_per_batch");
    TORCH_CHECK(QT64 / q_len_per_batch == KVT64 / kv_len_per_batch,
                "Q and K/V batch counts must match");

    TORCH_CHECK(KVT64 <= std::numeric_limits<int64_t>::max() / H64,
                "KV_T*H would overflow int64");
    const int64_t KVTH64 = KVT64 * H64;

    TORCH_CHECK(QT64 <= std::numeric_limits<int64_t>::max() / H64,
                "QT*H would overflow int64");
    const int64_t QTH64 = QT64 * H64;

    TORCH_CHECK(QT64  <= (int64_t)std::numeric_limits<int>::max(), "QT too large");
    TORCH_CHECK(KVT64 <= (int64_t)std::numeric_limits<int>::max(), "KV_T too large");
    TORCH_CHECK(H64   <= (int64_t)std::numeric_limits<int>::max(), "H too large");
    TORCH_CHECK(D64   <= (int64_t)std::numeric_limits<int>::max(), "D too large");
    TORCH_CHECK(DV64  <= (int64_t)std::numeric_limits<int>::max(), "DV too large");
    TORCH_CHECK(C64   <= (int64_t)std::numeric_limits<int>::max(), "C too large");
    TORCH_CHECK(query_start <= (int64_t)std::numeric_limits<int>::max(), "query_start too large");
    TORCH_CHECK(q_len_per_batch <= (int64_t)std::numeric_limits<int>::max(), "q_len_per_batch too large");
    TORCH_CHECK(kv_len_per_batch <= (int64_t)std::numeric_limits<int>::max(), "kv_len_per_batch too large");
    TORCH_CHECK(QTH64  <= (int64_t)std::numeric_limits<int>::max(), "QT*H too large");
    TORCH_CHECK(KVTH64 <= (int64_t)std::numeric_limits<int>::max(), "KV_T*H too large");

    const int64_t expected_rows = cand_is_per_head ? KVTH64 : KVT64;
    TORCH_CHECK(cand_i32.size(0) == expected_rows,
                "cand rows mismatch (expected ", expected_rows, ")");
    TORCH_CHECK(mask_u8.size(0) == expected_rows,
                "mask rows mismatch (expected ", expected_rows, ")");

    auto Qf = (Q.scalar_type() == at::kFloat) ? Q : Q.to(at::kFloat);
    auto Kf = (K.scalar_type() == at::kFloat) ? K : K.to(at::kFloat);
    auto Vf = (V.scalar_type() == at::kFloat) ? V : V.to(at::kFloat);
    Qf = Qf.contiguous();
    Kf = Kf.contiguous();
    Vf = Vf.contiguous();

    auto out = torch::zeros({QT64, H64, DV64}, Qf.options().dtype(at::kFloat));
    auto m   = torch::empty({QTH64}, Qf.options().dtype(at::kFloat));
    auto l   = torch::empty({QTH64}, Qf.options().dtype(at::kFloat));

    const int QT   = (int)QT64;
    const int KV_T = (int)KVT64;
    const int H    = (int)H64;
    const int D    = (int)D64;
    const int DV   = (int)DV64;
    const int C    = (int)C64;

    const float scale = (float)scale_py;
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    dim3 grid((unsigned)QT, (unsigned)H, 1);
    dim3 block(BLOCK_X, 1, 1);

    const size_t shmem = (size_t)512 * sizeof(float)
                       + (size_t)(BLOCK_X / 32) * sizeof(float)
                       + (size_t)V_TILE * sizeof(float);

    fused_fwd_streaming_mh_family_full_kernel<<<grid, block, shmem, stream>>>(
        (const float*)Qf.data_ptr(),
        (const float*)Kf.data_ptr(),
        (const float*)Vf.data_ptr(),
        (const int32_t*)cand_i32.data_ptr(),
        (const uint8_t*)mask_u8.data_ptr(),
        (const int32_t*)family_bounds_i32.data_ptr(),
        (const int32_t*)family_order_caps_i32.data_ptr(),
        (const int32_t*)dispatch_meta_i32.data_ptr(),
        (const uint8_t*)head_mask_u8.data_ptr(),
        (const int32_t*)runtime_i32.data_ptr(),
        (const float*)k_cfg_f32.data_ptr(),
        (const float*)v_cfg_f32.data_ptr(),
        (float*)m.data_ptr(),
        (float*)l.data_ptr(),
        (float*)out.data_ptr(),
        QT, KV_T, H, D, DV, C,
        scale,
        (int)cand_is_per_head,
        (int)query_start,
        (int)q_len_per_batch,
        (int)kv_len_per_batch
    );

    auto err = cudaGetLastError();
    TORCH_CHECK(err == cudaSuccess, "mh_forward kernel launch failed: ", cudaGetErrorString(err));

    return {out, m, l};
}

std::vector<torch::Tensor> mh_forward_frozen(
    torch::Tensor Q,
    torch::Tensor K,
    torch::Tensor V,
    torch::Tensor cand_i32,
    torch::Tensor mask_u8,
    torch::Tensor family_bounds_i32,
    torch::Tensor family_order_caps_i32,
    torch::Tensor dispatch_meta_i32,
    torch::Tensor head_mask_u8,
    torch::Tensor runtime_i32,
    torch::Tensor k_cfg_f32,
    torch::Tensor v_cfg_f32,
    double scale_py,
    bool cand_is_per_head,
    int64_t query_start,
    int64_t q_len_per_batch,
    int64_t kv_len_per_batch
) {
    check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
    check_cuda(family_bounds_i32,"family_bounds");
    check_cuda(family_order_caps_i32,"family_order_caps");
    check_cuda(dispatch_meta_i32,"dispatch_meta");
    check_cuda(head_mask_u8,"head_mask");
    check_cuda(runtime_i32,"runtime_i32");
    check_cuda(k_cfg_f32,"k_cfg");
    check_cuda(v_cfg_f32,"v_cfg");

    auto cand_frozen = cand_i32.contiguous().clone();
    auto mask_frozen = mask_u8.contiguous().clone();
    auto bounds_frozen = family_bounds_i32.contiguous().clone();
    auto caps_frozen = family_order_caps_i32.contiguous().clone();
    auto dispatch_frozen = dispatch_meta_i32.contiguous().clone();
    auto head_mask_frozen = head_mask_u8.contiguous().clone();
    auto runtime_frozen = runtime_i32.contiguous().clone();
    auto kcfg_frozen = k_cfg_f32.contiguous().clone();
    auto vcfg_frozen = v_cfg_f32.contiguous().clone();

    auto outs = mh_forward(
        Q, K, V,
        cand_frozen, mask_frozen,
        bounds_frozen, caps_frozen, dispatch_frozen,
        head_mask_frozen, runtime_frozen,
        kcfg_frozen, vcfg_frozen,
        scale_py,
        cand_is_per_head,
        query_start,
        q_len_per_batch,
        kv_len_per_batch
    );

    return {
        outs[0], outs[1], outs[2],
        cand_frozen, mask_frozen,
        bounds_frozen, caps_frozen, dispatch_frozen,
        head_mask_frozen, runtime_frozen,
        kcfg_frozen, vcfg_frozen
    };
}

// ============================================================
// REPLACE ONLY THIS FUNCTION INSIDE YOUR EXISTING STAGE-E CELL
// Keep the rest of the CUDA/C++ code VERBATIM.
// ============================================================
std::vector<torch::Tensor> mh_backward(
    torch::Tensor dOut,
    torch::Tensor Q,
    torch::Tensor K,
    torch::Tensor V,
    torch::Tensor cand_i32,
    torch::Tensor mask_u8,
    torch::Tensor family_bounds_i32,
    torch::Tensor family_order_caps_i32,
    torch::Tensor dispatch_meta_i32,
    torch::Tensor head_mask_u8,
    torch::Tensor runtime_i32,
    torch::Tensor k_cfg_f32,
    torch::Tensor v_cfg_f32,
    torch::Tensor m,
    torch::Tensor l,
    double scale_py,
    bool cand_is_per_head
) {
    check_cuda(dOut,"dOut"); check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
    check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
    check_cuda(family_bounds_i32,"family_bounds");
    check_cuda(family_order_caps_i32,"family_order_caps");
    check_cuda(dispatch_meta_i32,"dispatch_meta");
    check_cuda(head_mask_u8,"head_mask");
    check_cuda(runtime_i32,"runtime_i32");
    check_cuda(k_cfg_f32,"k_cfg");
    check_cuda(v_cfg_f32,"v_cfg");
    check_cuda(m,"m"); check_cuda(l,"l");

    c10::cuda::CUDAGuard guard(Q.device());
    check_same_device(Q, dOut, "Q", "dOut");
    check_same_device(Q, K, "Q", "K");
    check_same_device(Q, V, "Q", "V");
    check_same_device(Q, cand_i32, "Q", "cand");
    check_same_device(Q, mask_u8, "Q", "mask");
    check_same_device(Q, family_bounds_i32, "Q", "family_bounds");
    check_same_device(Q, family_order_caps_i32, "Q", "family_order_caps");
    check_same_device(Q, dispatch_meta_i32, "Q", "dispatch_meta");
    check_same_device(Q, head_mask_u8, "Q", "head_mask");
    check_same_device(Q, runtime_i32, "Q", "runtime_i32");
    check_same_device(Q, k_cfg_f32, "Q", "k_cfg");
    check_same_device(Q, v_cfg_f32, "Q", "v_cfg");
    check_same_device(Q, m, "Q", "m");
    check_same_device(Q, l, "Q", "l");

    check_contig(dOut,"dOut"); check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
    check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask");
    check_contig(family_bounds_i32,"family_bounds");
    check_contig(family_order_caps_i32,"family_order_caps");
    check_contig(dispatch_meta_i32,"dispatch_meta");
    check_contig(head_mask_u8,"head_mask");
    check_contig(runtime_i32,"runtime_i32");
    check_contig(k_cfg_f32,"k_cfg");
    check_contig(v_cfg_f32,"v_cfg");
    check_contig(m,"m"); check_contig(l,"l");

    TORCH_CHECK(dOut.dim()==3 && Q.dim()==3 && K.dim()==3 && V.dim()==3,
                "dOut/Q/K/V must be [TQ,H,DV] / [TQ,H,D] / [TK,H,D] / [TK,H,DV]");
    TORCH_CHECK(cand_i32.dim()==2 && mask_u8.dim()==2, "cand/mask must be 2D");
    TORCH_CHECK(cand_i32.scalar_type()==at::kInt, "cand must be int32");
    TORCH_CHECK(mask_u8.scalar_type()==at::kByte, "mask must be uint8");

    TORCH_CHECK(m.scalar_type()==at::kFloat && l.scalar_type()==at::kFloat, "m/l must be float");
    TORCH_CHECK(family_bounds_i32.dim()==1 && family_bounds_i32.numel()==10 && family_bounds_i32.scalar_type()==at::kInt,
                "family_bounds must be int32 [10]");
    TORCH_CHECK(family_order_caps_i32.dim()==1 && family_order_caps_i32.numel()==5 && family_order_caps_i32.scalar_type()==at::kInt,
                "family_order_caps must be int32 [5]");
    TORCH_CHECK(dispatch_meta_i32.dim()==1 && dispatch_meta_i32.numel()==10 && dispatch_meta_i32.scalar_type()==at::kInt,
                "dispatch_meta must be int32 [10]");
    TORCH_CHECK(head_mask_u8.dim()==1 && head_mask_u8.scalar_type()==at::kByte,
                "head_mask must be uint8 [H]");
    TORCH_CHECK(runtime_i32.dim()==1 && runtime_i32.numel()==4 && runtime_i32.scalar_type()==at::kInt,
                "runtime_i32 must be int32 [4]");
    TORCH_CHECK(k_cfg_f32.dim()==1 && k_cfg_f32.numel()==7 && k_cfg_f32.scalar_type()==at::kFloat,
                "k_cfg_f32 must be float32 [7]");
    TORCH_CHECK(v_cfg_f32.dim()==1 && v_cfg_f32.numel()==7 && v_cfg_f32.scalar_type()==at::kFloat,
                "v_cfg_f32 must be float32 [7]");
    TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X");

    check_dtype_eq(Q,K,"Q","K");
    check_dtype_eq(Q,V,"Q","V");
    check_dtype_eq(V,dOut,"V","dOut");
    TORCH_CHECK(dOut.is_floating_point() && Q.is_floating_point() &&
                K.is_floating_point() && V.is_floating_point(),
                "dOut/Q/K/V must be floating point");
    TORCH_CHECK(std::isfinite(scale_py), "scale must be finite");

    const int64_t TQ64 = Q.size(0);
    const int64_t H64  = Q.size(1);
    const int64_t D64  = Q.size(2);
    const int64_t TK64 = K.size(0);
    const int64_t DV64 = V.size(2);
    const int64_t C64  = cand_i32.size(1);

    TORCH_CHECK(TQ64 > 0 && TK64 > 0 && H64 > 0 && D64 > 0 && DV64 > 0 && C64 > 0,
                "TQ/TK/H/D/DV/C must be positive");
    TORCH_CHECK(K.size(1)==H64 && K.size(2)==D64, "K shape mismatch");
    TORCH_CHECK(V.size(0)==TK64 && V.size(1)==H64, "V shape mismatch");
    TORCH_CHECK(dOut.size(0)==TQ64 && dOut.size(1)==H64 && dOut.size(2)==DV64, "dOut shape mismatch");
    TORCH_CHECK(mask_u8.size(1)==C64, "mask C mismatch");
    TORCH_CHECK(head_mask_u8.numel()==H64, "head_mask size mismatch");
    TORCH_CHECK(D64 <= 256, "D must be <= 256");

    TORCH_CHECK(TQ64 <= std::numeric_limits<int64_t>::max() / H64, "TQ*H would overflow int64");
    TORCH_CHECK(TK64 <= std::numeric_limits<int64_t>::max() / H64, "TK*H would overflow int64");
    const int64_t TQH64 = TQ64 * H64;
    const int64_t TKH64 = TK64 * H64;

    TORCH_CHECK(TQ64 <= (int64_t)std::numeric_limits<int>::max(), "TQ too large");
    TORCH_CHECK(TK64 <= (int64_t)std::numeric_limits<int>::max(), "TK too large");
    TORCH_CHECK(H64  <= (int64_t)std::numeric_limits<int>::max(), "H too large");
    TORCH_CHECK(D64  <= (int64_t)std::numeric_limits<int>::max(), "D too large");
    TORCH_CHECK(DV64 <= (int64_t)std::numeric_limits<int>::max(), "DV too large");
    TORCH_CHECK(C64  <= (int64_t)std::numeric_limits<int>::max(), "C too large");
    TORCH_CHECK(TQH64 <= (int64_t)std::numeric_limits<int>::max(), "TQ*H too large");
    TORCH_CHECK(TKH64 <= (int64_t)std::numeric_limits<int>::max(), "TK*H too large");
    TORCH_CHECK(m.dim() == 1 && m.numel() == TQH64, "m must be [TQ*H]");
    TORCH_CHECK(l.dim() == 1 && l.numel() == TQH64, "l must be [TQ*H]");

    const int64_t expected_rows = cand_is_per_head ? TQH64 : TQ64;
    TORCH_CHECK(cand_i32.size(0) == expected_rows, "cand rows mismatch (expected ", expected_rows, ")");
    TORCH_CHECK(mask_u8.size(0) == expected_rows, "mask rows mismatch (expected ", expected_rows, ")");

    auto dOutf = (dOut.scalar_type() == at::kFloat) ? dOut : dOut.to(at::kFloat);
    auto Qf    = (Q.scalar_type()    == at::kFloat) ? Q    : Q.to(at::kFloat);
    auto Kf    = (K.scalar_type()    == at::kFloat) ? K    : K.to(at::kFloat);
    auto Vf    = (V.scalar_type()    == at::kFloat) ? V    : V.to(at::kFloat);
    dOutf = dOutf.contiguous();
    Qf = Qf.contiguous();
    Kf = Kf.contiguous();
    Vf = Vf.contiguous();

    auto dQ_f = torch::zeros({TQH64, D64}, Qf.options().dtype(at::kFloat));
    auto dK_f = torch::zeros({TKH64, D64}, Kf.options().dtype(at::kFloat));
    auto dV_f = torch::zeros({TKH64, DV64}, Vf.options().dtype(at::kFloat));

    const int TQ = (int)TQ64;
    const int TK = (int)TK64;
    const int H  = (int)H64;
    const int D  = (int)D64;
    const int DV = (int)DV64;
    const int C  = (int)C64;

    const float scale = (float)scale_py;
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    dim3 grid((unsigned)TQ, (unsigned)H, 1);
    dim3 block(BLOCK_X, 1, 1);

    fused_bwd_mh_family_full_kernel<<<grid, block, 0, stream>>>(
        (const float*)dOutf.data_ptr(),
        (const float*)Qf.data_ptr(),
        (const float*)Kf.data_ptr(),
        (const float*)Vf.data_ptr(),
        (const int32_t*)cand_i32.data_ptr(),
        (const uint8_t*)mask_u8.data_ptr(),
        (const int32_t*)family_bounds_i32.data_ptr(),
        (const int32_t*)family_order_caps_i32.data_ptr(),
        (const int32_t*)dispatch_meta_i32.data_ptr(),
        (const uint8_t*)head_mask_u8.data_ptr(),
        (const int32_t*)runtime_i32.data_ptr(),
        (const float*)k_cfg_f32.data_ptr(),
        (const float*)v_cfg_f32.data_ptr(),
        (const float*)m.data_ptr(),
        (const float*)l.data_ptr(),
        (float*)dQ_f.data_ptr(),
        (float*)dK_f.data_ptr(),
        (float*)dV_f.data_ptr(),
        TQ, TK, H, D, DV, C, scale, (int)cand_is_per_head
    );

    auto err = cudaGetLastError();
    TORCH_CHECK(err == cudaSuccess, "mh_backward kernel launch failed: ", cudaGetErrorString(err));

    auto dQ = torch::empty_like(Q);
    auto dK = torch::empty_like(K);
    auto dV = torch::empty_like(V);

    const int threads = 256;
    auto launch_cast = [&](const torch::Tensor& src_f, torch::Tensor& dst) {
        const int64_t n = dst.numel();
        const int64_t blocks64 = (n + threads - 1) / threads;
        TORCH_CHECK(blocks64 <= (int64_t)std::numeric_limits<int>::max(),
                    "cast launch grid too large");
        const int blocks = (int)blocks64;
        if (dst.scalar_type() == torch::kFloat) {
            cast_f32_to_f32<<<blocks, threads, 0, stream>>>(
                src_f.data_ptr<float>(), dst.data_ptr<float>(), n);
        } else if (dst.scalar_type() == torch::kHalf) {
            cast_f32_to_f16<<<blocks, threads, 0, stream>>>(
                src_f.data_ptr<float>(), (at::Half*)dst.data_ptr<at::Half>(), n);
        } else if (dst.scalar_type() == torch::kBFloat16) {
            cast_f32_to_bf16<<<blocks, threads, 0, stream>>>(
                src_f.data_ptr<float>(), (at::BFloat16*)dst.data_ptr<at::BFloat16>(), n);
        } else {
            TORCH_CHECK(false, "Unsupported dtype for cast");
        }
    };

    launch_cast(dQ_f.view({TQ64, H64, D64}).contiguous(), dQ);
    launch_cast(dK_f.view({TK64, H64, D64}).contiguous(), dK);
    launch_cast(dV_f.view({TK64, H64, DV64}).contiguous(), dV);

    err = cudaGetLastError();
    TORCH_CHECK(err == cudaSuccess, "mh_backward cast launch failed: ", cudaGetErrorString(err));

    return {dQ, dK, dV};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("mh_forward", &mh_forward, "Stage-E full family-segmented LCHA forward (CUDA)");
    m.def("mh_forward_frozen", &mh_forward_frozen, "Stage-E full family-segmented LCHA forward frozen (CUDA)");
    m.def("mh_backward", &mh_backward, "Stage-E full family-segmented LCHA backward (CUDA)");
}
"""

extra_cuda_cflags = ["-O3", "--use_fast_math", "-DBLOCK_X=256", "-DV_TILE=256"]
extra_cflags = ["-O3", "-std=c++17"]

_build_sig = "stageE_family_segmented_lcha_full_v2_hardened_kvcache_order2_guard"
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
ext_name = f"stageE_family_segmented_lcha_full_ext_{_build_hash}"

stageE_family_segmented_lcha_full = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=CUDA_SRC,
    functions=None,
    extra_cuda_cflags=extra_cuda_cflags,
    extra_cflags=extra_cflags,
    with_cuda=True,
    verbose=bool(FUSED_VERBOSE_BUILD),
)

_mh_forward_cuda = stageE_family_segmented_lcha_full.mh_forward
_mh_forward_frozen_cuda = stageE_family_segmented_lcha_full.mh_forward_frozen
_mh_backward_cuda = stageE_family_segmented_lcha_full.mh_backward

# Debug-only resilience checks. Set False for speed after validation.
STAGEE_DEBUG_FINITE_CHECKS = True


def _stagee_assert_finite(name, x):
    if not torch.isfinite(x).all():
        raise ValueError(f"{name} contains NaN/Inf")


def validate_stagee_metadata(
    family_bounds_i32,
    family_order_caps_i32,
    dispatch_meta_i32,
    runtime_i32,
    k_cfg_f32,
    v_cfg_f32,
    C: int,
):
    bounds = family_bounds_i32.detach().cpu().tolist()
    caps = family_order_caps_i32.detach().cpu().tolist()
    dispatch = dispatch_meta_i32.detach().cpu().tolist()
    runtime = runtime_i32.detach().cpu().tolist()

    if len(bounds) != 10:
        raise ValueError("family_bounds_i32 must have length 10")
    if bounds[0] != 0:
        raise ValueError("family_bounds must start at 0")
    if bounds[-1] > C:
        raise ValueError("family_bounds final end exceeds C")
    if any(x < 0 or x > C for x in bounds):
        raise ValueError("family_bounds entries must be in [0, C]")
    if any(bounds[i] > bounds[i + 1] for i in range(len(bounds) - 1)):
        raise ValueError("family_bounds must be monotonic nondecreasing")
    if not (
        bounds[2] == bounds[1]
        and bounds[4] == bounds[3]
        and bounds[6] == bounds[5]
        and bounds[8] == bounds[7]
    ):
        raise ValueError("family_bounds must be contiguous: next_start == previous_end")

    if len(caps) != 5:
        raise ValueError("family_order_caps_i32 must have length 5")
    if any(x < 0 or x > 2 for x in caps):
        raise ValueError("family_order_caps entries must be in {0,1,2}")

    if len(dispatch) != 10:
        raise ValueError("dispatch_meta_i32 must have length 10")
    stage_order = int(dispatch[4])
    if stage_order < 0 or stage_order > 2:
        raise ValueError("dispatch_meta_i32[4] stage_order must be in {0,1,2}")

    if len(runtime) != 4:
        raise ValueError("runtime_i32 must have length 4")
    if any(x not in (0, 1) for x in runtime[:3]):
        raise ValueError("runtime_i32[0:3] must be binary flags")

    if k_cfg_f32.numel() != 7 or v_cfg_f32.numel() != 7:
        raise ValueError("k_cfg_f32/v_cfg_f32 must have length 7")
    if STAGEE_DEBUG_FINITE_CHECKS:
        _stagee_assert_finite("k_cfg_f32", k_cfg_f32)
        _stagee_assert_finite("v_cfg_f32", v_cfg_f32)


def _check_finite_stagee_forward(Q, K, V, k_cfg_f32, v_cfg_f32, scale_py):
    _stagee_assert_finite("Q", Q)
    _stagee_assert_finite("K", K)
    _stagee_assert_finite("V", V)
    _stagee_assert_finite("k_cfg_f32", k_cfg_f32)
    _stagee_assert_finite("v_cfg_f32", v_cfg_f32)
    if not torch.isfinite(torch.tensor(float(scale_py), device=Q.device)):
        raise ValueError("scale_py must be finite")


def _check_finite_stagee_backward(dOut, Q, K, V, m, l, k_cfg_f32, v_cfg_f32, scale_py):
    _stagee_assert_finite("dOut", dOut)
    _stagee_assert_finite("Q", Q)
    _stagee_assert_finite("K", K)
    _stagee_assert_finite("V", V)
    _stagee_assert_finite("m", m)
    _stagee_assert_finite("l", l)
    _stagee_assert_finite("k_cfg_f32", k_cfg_f32)
    _stagee_assert_finite("v_cfg_f32", v_cfg_f32)
    if not torch.isfinite(torch.tensor(float(scale_py), device=Q.device)):
        raise ValueError("scale_py must be finite")


def stagee_mh_forward(
    Q, K, V,
    cand_i32, mask_u8,
    family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
    head_mask_u8, runtime_i32,
    k_cfg_f32, v_cfg_f32,
    scale_py: float,
    cand_is_per_head: bool,
    query_start: int = 0,
    q_len_per_batch: int | None = None,
    kv_len_per_batch: int | None = None,
):
    if q_len_per_batch is None:
        q_len_per_batch = int(Q.size(0))
    if kv_len_per_batch is None:
        kv_len_per_batch = int(K.size(0))

    validate_stagee_metadata(
        family_bounds_i32,
        family_order_caps_i32,
        dispatch_meta_i32,
        runtime_i32,
        k_cfg_f32,
        v_cfg_f32,
        int(cand_i32.size(1)),
    )
    if STAGEE_DEBUG_FINITE_CHECKS:
        _check_finite_stagee_forward(Q, K, V, k_cfg_f32, v_cfg_f32, scale_py)

    return _mh_forward_cuda(
        Q, K, V,
        cand_i32, mask_u8,
        family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
        head_mask_u8, runtime_i32,
        k_cfg_f32, v_cfg_f32,
        scale_py,
        cand_is_per_head,
        int(query_start),
        int(q_len_per_batch),
        int(kv_len_per_batch),
    )


def stagee_mh_forward_frozen(
    Q, K, V,
    cand_i32, mask_u8,
    family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
    head_mask_u8, runtime_i32,
    k_cfg_f32, v_cfg_f32,
    scale_py: float,
    cand_is_per_head: bool,
    query_start: int = 0,
    q_len_per_batch: int | None = None,
    kv_len_per_batch: int | None = None,
):
    if q_len_per_batch is None:
        q_len_per_batch = int(Q.size(0))
    if kv_len_per_batch is None:
        kv_len_per_batch = int(K.size(0))

    validate_stagee_metadata(
        family_bounds_i32,
        family_order_caps_i32,
        dispatch_meta_i32,
        runtime_i32,
        k_cfg_f32,
        v_cfg_f32,
        int(cand_i32.size(1)),
    )
    if STAGEE_DEBUG_FINITE_CHECKS:
        _check_finite_stagee_forward(Q, K, V, k_cfg_f32, v_cfg_f32, scale_py)

    return _mh_forward_frozen_cuda(
        Q, K, V,
        cand_i32, mask_u8,
        family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
        head_mask_u8, runtime_i32,
        k_cfg_f32, v_cfg_f32,
        scale_py,
        cand_is_per_head,
        int(query_start),
        int(q_len_per_batch),
        int(kv_len_per_batch),
    )


def stagee_mh_backward(
    dOut,
    Q, K, V,
    cand_i32, mask_u8,
    family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
    head_mask_u8, runtime_i32,
    k_cfg_f32, v_cfg_f32,
    m, l,
    scale_py,
    cand_is_per_head,
):
    validate_stagee_metadata(
        family_bounds_i32,
        family_order_caps_i32,
        dispatch_meta_i32,
        runtime_i32,
        k_cfg_f32,
        v_cfg_f32,
        int(cand_i32.size(1)),
    )
    if STAGEE_DEBUG_FINITE_CHECKS:
        _check_finite_stagee_backward(dOut, Q, K, V, m, l, k_cfg_f32, v_cfg_f32, scale_py)
    return _mh_backward_cuda(
        dOut,
        Q, K, V,
        cand_i32, mask_u8,
        family_bounds_i32, family_order_caps_i32, dispatch_meta_i32,
        head_mask_u8, runtime_i32,
        k_cfg_f32, v_cfg_f32,
        m, l,
        scale_py,
        cand_is_per_head,
    )

# ------------------------------------------------------------
# Backward-compatible aliases for StageEMHFamilyLCHAFn
# ------------------------------------------------------------
mh_forward = stagee_mh_forward
mh_forward_frozen = stagee_mh_forward_frozen
mh_backward = stagee_mh_backward


print("Built:", ext_name)
print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))

In [ ]:
import math
import torch

PROTO_ECONOMY = 0
PROTO_HYBRID = 1
PROTO_MULTISCALE = 2
PROTO_GLOBAL_REFRESH = 3

_ALLOWED_CFG_KEYS = {"self0", "win1", "win2", "ring1", "ring2", "anch1", "anch2"}


def _apply_cfg_override(cfg, override, name):
    if not override:
        return
    bad = set(override) - _ALLOWED_CFG_KEYS
    if bad:
        raise ValueError(f"{name} contains unknown keys: {sorted(bad)}")
    for k, v in override.items():
        vf = float(v)
        if not math.isfinite(vf):
            raise ValueError(f"{name}[{k}] must be finite")
        cfg[k] = vf


def _validate_same_device(ref, **tensors):
    for name, x in tensors.items():
        if x.device != ref.device:
            raise ValueError(f"{name} must be on {ref.device}, got {x.device}")


def _validate_family_bounds(family_bounds_i32, C: int):
    if family_bounds_i32.dtype != torch.int32:
        raise TypeError("family_bounds_i32 must be torch.int32")
    if family_bounds_i32.numel() != 10:
        raise ValueError("family_bounds_i32 must have 10 entries")

    b = family_bounds_i32.detach().cpu().tolist()

    if b[0] != 0:
        raise ValueError("family_bounds_i32 must start at 0")
    if any(x < 0 or x > C for x in b):
        raise ValueError("family_bounds_i32 entries must be in [0, C]")
    if any(b[i] > b[i + 1] for i in range(len(b) - 1)):
        raise ValueError("family_bounds_i32 must be monotonic")
    if not (b[2] == b[1] and b[4] == b[3] and b[6] == b[5] and b[8] == b[7]):
        raise ValueError("family_bounds_i32 must be contiguous: next_start == previous_end")


def _validate_candidate_shape(cand_i32, mask_u8, Q3, K3, cand_is_per_head: bool):
    if cand_i32.dim() != 2 or mask_u8.dim() != 2:
        raise ValueError("cand_i32 and mask_u8 must be 2D")

    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand/mask shape mismatch: {cand_i32.shape} vs {mask_u8.shape}")

    if cand_i32.dtype != torch.int32:
        raise TypeError("cand_i32 must be torch.int32")
    if mask_u8.dtype != torch.uint8:
        raise TypeError("mask_u8 must be torch.uint8")

    kv_flat = int(K3.size(0))
    H = int(Q3.size(1))
    expected_rows = kv_flat * H if cand_is_per_head else kv_flat

    if cand_i32.size(0) != expected_rows:
        raise ValueError(
            f"cand rows mismatch. Expected {expected_rows}, got {cand_i32.size(0)}. "
            "For KV-cache sparse attention, candidates must be compiled on the full flattened K/V axis."
        )


def _check_finite(name, x):
    if not torch.isfinite(x).all():
        raise ValueError(f"{name} contains NaN/Inf")


def make_lcha_head_mask(
    num_heads: int,
    frac: float = 1.0,
    policy: str = "tail",
    device=None,
):
    frac = float(max(0.0, min(1.0, frac)))
    k = int(round(num_heads * frac))
    mask = torch.zeros(num_heads, dtype=torch.uint8, device=device)
    if k <= 0:
        return mask
    if k >= num_heads:
        mask[:] = 1
        return mask

    if policy == "tail":
        mask[num_heads - k:] = 1
    elif policy == "head":
        mask[:k] = 1
    elif policy == "stride":
        idx = torch.linspace(0, num_heads - 1, steps=k, device=device).round().long().unique()
        mask[idx] = 1
    else:
        raise ValueError(f"Unknown head selection policy: {policy}")
    return mask


def resolve_protocol_lcha_cfg(
    dispatch_meta_i32: torch.Tensor,
    num_heads: int,
    device,
    lcha_enabled: bool = True,
    lcha_heads_frac: float = 1.0,
    head_policy: str = "tail",
    use_v_lcha: bool = True,
    trap_on_bad_cand: bool = False,
    k_override: dict | None = None,
    v_override: dict | None = None,
):
    """
    Returns:
      head_mask_u8 : [H] uint8 CUDA
      runtime_i32  : [4] int32 CUDA
      k_cfg_f32    : [7] float32 CUDA
      v_cfg_f32    : [7] float32 CUDA

    cfg layout:
      [self0, win1, win2, ring1, ring2, anch1, anch2]
    """
    if dispatch_meta_i32.dim() != 1 or dispatch_meta_i32.numel() != 10:
        raise ValueError("dispatch_meta_i32 must have shape [10]")
    if dispatch_meta_i32.dtype != torch.int32:
        raise TypeError("dispatch_meta_i32 must be torch.int32")

    dispatch_meta_cpu = dispatch_meta_i32.detach().cpu()
    protocol_id = int(dispatch_meta_cpu[0].item())
    stage_order = int(dispatch_meta_cpu[4].item())
    if stage_order < 0 or stage_order > 2:
        raise ValueError("dispatch_meta_i32[4] stage_order must be in [0, 2]")

    k_cfg = {
        "self0": 1.0,
        "win1":  0.0, "win2":  0.0,
        "ring1": 0.0, "ring2": 0.0,
        "anch1": 0.0, "anch2": 0.0,
    }
    v_cfg = {
        "self0": 1.0,
        "win1":  0.0, "win2":  0.0,
        "ring1": 0.0, "ring2": 0.0,
        "anch1": 0.0, "anch2": 0.0,
    }

    if protocol_id == PROTO_ECONOMY:
        pass
    elif protocol_id in (PROTO_HYBRID, PROTO_MULTISCALE):
        k_cfg.update({"win1": 0.10, "ring1": 0.15, "anch1": 0.20})
        v_cfg.update({"win1": 0.05, "ring1": 0.08, "anch1": 0.10})
    elif protocol_id == PROTO_GLOBAL_REFRESH:
        k_cfg.update({"win1": 0.10, "ring1": 0.18, "anch1": 0.22})
        v_cfg.update({"win1": 0.05, "ring1": 0.10, "anch1": 0.12})
        if stage_order >= 2:
            k_cfg.update({"ring2": 0.10, "anch2": 0.12})
            v_cfg.update({"ring2": 0.05, "anch2": 0.06})
    else:
        raise ValueError(f"Unsupported protocol_id: {protocol_id}")

    _apply_cfg_override(k_cfg, k_override, "k_override")
    _apply_cfg_override(v_cfg, v_override, "v_override")

    head_mask_u8 = make_lcha_head_mask(
        num_heads=num_heads,
        frac=lcha_heads_frac if lcha_enabled else 0.0,
        policy=head_policy,
        device=device,
    )

    runtime_i32 = torch.tensor(
        [
            1 if lcha_enabled else 0,
            1 if use_v_lcha else 0,
            1 if trap_on_bad_cand else 0,
            0,
        ],
        dtype=torch.int32,
        device=device,
    )

    k_cfg_f32 = torch.tensor(
        [
            k_cfg["self0"],
            k_cfg["win1"],  k_cfg["win2"],
            k_cfg["ring1"], k_cfg["ring2"],
            k_cfg["anch1"], k_cfg["anch2"],
        ],
        dtype=torch.float32,
        device=device,
    )

    v_cfg_f32 = torch.tensor(
        [
            v_cfg["self0"],
            v_cfg["win1"],  v_cfg["win2"],
            v_cfg["ring1"], v_cfg["ring2"],
            v_cfg["anch1"], v_cfg["anch2"],
        ],
        dtype=torch.float32,
        device=device,
    )

    return head_mask_u8.contiguous(), runtime_i32.contiguous(), k_cfg_f32.contiguous(), v_cfg_f32.contiguous()


class StageEMHFamilyLCHAFn(torch.autograd.Function):
    @staticmethod
    def forward(
        ctx,
        Q, K, V,
        cand_i32,
        mask_u8,
        family_bounds_i32,
        family_order_caps_i32,
        dispatch_meta_i32,
        scale: float,
        cand_is_per_head: bool,
        head_mask_u8,
        runtime_i32,
        k_cfg_f32,
        v_cfg_f32,
        query_start: int,
        q_len_per_batch: int,
        kv_len_per_batch: int,
    ):
        outs = mh_forward_frozen(
            Q, K, V,
            cand_i32, mask_u8,
            family_bounds_i32,
            family_order_caps_i32,
            dispatch_meta_i32,
            head_mask_u8,
            runtime_i32,
            k_cfg_f32,
            v_cfg_f32,
            float(scale),
            bool(cand_is_per_head),
            int(query_start),
            int(q_len_per_batch),
            int(kv_len_per_batch),
        )

        out, m, l = outs[0], outs[1], outs[2]
        cand_frozen = outs[3]
        mask_frozen = outs[4]
        bounds_frozen = outs[5]
        caps_frozen = outs[6]
        dispatch_frozen = outs[7]
        head_mask_frozen = outs[8]
        runtime_frozen = outs[9]
        kcfg_frozen = outs[10]
        vcfg_frozen = outs[11]

        ctx.scale = float(scale)
        ctx.cand_is_per_head = bool(cand_is_per_head)

        ctx.query_start = int(query_start)
        ctx.q_len_per_batch = int(q_len_per_batch)
        ctx.kv_len_per_batch = int(kv_len_per_batch)

        ctx.save_for_backward(
            Q, K, V,
            cand_frozen, mask_frozen,
            bounds_frozen, caps_frozen, dispatch_frozen,
            head_mask_frozen, runtime_frozen,
            kcfg_frozen, vcfg_frozen,
            m, l,
        )
        return out

    @staticmethod
    def backward(ctx, dOut):
        (
            Q, K, V,
            cand_frozen, mask_frozen,
            bounds_frozen, caps_frozen, dispatch_frozen,
            head_mask_frozen, runtime_frozen,
            kcfg_frozen, vcfg_frozen,
            m, l,
        ) = ctx.saved_tensors

        _check_finite("dOut", dOut)

        if int(ctx.query_start) != 0 or int(ctx.q_len_per_batch) != int(ctx.kv_len_per_batch):
            raise RuntimeError(
                "Sparse KV-cache backward is intentionally unsupported. "
                "Use cached sparse decoding under torch.no_grad(), or run full-sequence training."
            )

        dQ, dK, dV = mh_backward(
            dOut.contiguous(),
            Q, K, V,
            cand_frozen, mask_frozen,
            bounds_frozen, caps_frozen, dispatch_frozen,
            head_mask_frozen, runtime_frozen,
            kcfg_frozen, vcfg_frozen,
            m, l,
            float(ctx.scale),
            bool(ctx.cand_is_per_head),
        )

        return (
            dQ, dK, dV,
            None, None, None, None, None,
            None,
            None,
            None,
            None,
            None,
            None,
            None,
            None,
            None,
        )


def stagee_family_lcha_apply(
    Q, K, V,
    cand_i32,
    mask_u8,
    family_bounds_i32,
    family_order_caps_i32,
    dispatch_meta_i32,
    cand_is_per_head: bool,
    lcha_enabled: bool = True,
    lcha_heads_frac: float = 1.0,
    head_policy: str = "tail",
    use_v_lcha: bool = True,
    trap_on_bad_cand: bool = False,
    k_override: dict | None = None,
    v_override: dict | None = None,
    query_start: int = 0,
):
    # Accept either:
    #   [QT,H,D] / [KV_T,H,D] / [KV_T,H,DV]
    # or
    #   [B,QT,H,D] / [B,KV_T,H,D] / [B,KV_T,H,DV]
    if Q.dim() == 3 and K.dim() == 3 and V.dim() == 3:
        if K.size(1) != Q.size(1) or K.size(2) != Q.size(2):
            raise ValueError(f"K head/dim must match Q. Got Q={tuple(Q.shape)}, K={tuple(K.shape)}")
        if V.size(0) != K.size(0) or V.size(1) != K.size(1):
            raise ValueError(f"V first two dims must match K. Got K={tuple(K.shape)}, V={tuple(V.shape)}")

        Q3 = Q.contiguous()
        K3 = K.contiguous()
        V3 = V.contiguous()
        restore_shape = None
        q_len_per_batch = int(Q3.size(0))
        kv_len_per_batch = int(K3.size(0))

    elif Q.dim() == 4 and K.dim() == 4 and V.dim() == 4:
        if K.size(0) != Q.size(0) or K.size(2) != Q.size(2) or K.size(3) != Q.size(3):
            raise ValueError(f"K batch/head/dim must match Q. Got Q={tuple(Q.shape)}, K={tuple(K.shape)}")
        if V.size(0) != K.size(0) or V.size(1) != K.size(1) or V.size(2) != K.size(2):
            raise ValueError(f"V first three dims must match K. Got K={tuple(K.shape)}, V={tuple(V.shape)}")

        B, QT, H, D = Q.shape
        KV_T = int(K.size(1))
        DV = V.shape[-1]

        Q3 = Q.contiguous().view(B * QT, H, D)
        K3 = K.contiguous().view(B * KV_T, H, D)
        V3 = V.contiguous().view(B * KV_T, H, DV)
        restore_shape = (B, QT)
        q_len_per_batch = int(QT)
        kv_len_per_batch = int(KV_T)

    else:
        raise ValueError(
            "Expected Q/K/V either all 3D [QT,H,D]/[KV_T,H,D]/[KV_T,H,DV] "
            "or all 4D [B,QT,H,D]/[B,KV_T,H,D]/[B,KV_T,H,DV]."
        )

    query_start = int(query_start)
    if query_start < 0:
        raise ValueError(f"query_start must be non-negative. Got query_start={query_start}")
    if query_start + q_len_per_batch > kv_len_per_batch:
        raise ValueError(
            "query_start + query length cannot exceed KV length. "
            f"Got query_start={query_start}, q_len={q_len_per_batch}, kv_len={kv_len_per_batch}"
        )

    if Q3.size(0) <= 0 or Q3.size(1) <= 0 or Q3.size(2) <= 0 or K3.size(0) <= 0 or V3.size(2) <= 0:
        raise ValueError("QT/KV_T/H/D/DV must all be positive")
    if Q3.size(2) > 256:
        raise ValueError("D must be <= 256 for the current Stage-E kernel")

    _validate_same_device(
        Q3,
        K3=K3,
        V3=V3,
        cand_i32=cand_i32,
        mask_u8=mask_u8,
        family_bounds_i32=family_bounds_i32,
        family_order_caps_i32=family_order_caps_i32,
        dispatch_meta_i32=dispatch_meta_i32,
    )

    _validate_candidate_shape(cand_i32, mask_u8, Q3, K3, bool(cand_is_per_head))
    _validate_family_bounds(family_bounds_i32, int(cand_i32.size(1)))

    H = int(Q3.size(1))
    D = int(Q3.size(2))
    scale = 1.0 / math.sqrt(float(D))

    head_mask_u8, runtime_i32, k_cfg_f32, v_cfg_f32 = resolve_protocol_lcha_cfg(
        dispatch_meta_i32=dispatch_meta_i32,
        num_heads=H,
        device=Q3.device,
        lcha_enabled=lcha_enabled,
        lcha_heads_frac=lcha_heads_frac,
        head_policy=head_policy,
        use_v_lcha=use_v_lcha,
        trap_on_bad_cand=trap_on_bad_cand,
        k_override=k_override,
        v_override=v_override,
    )

    _check_finite("Q3", Q3)
    _check_finite("K3", K3)
    _check_finite("V3", V3)
    _check_finite("k_cfg_f32", k_cfg_f32)
    _check_finite("v_cfg_f32", v_cfg_f32)

    out3 = StageEMHFamilyLCHAFn.apply(
        Q3, K3, V3,
        cand_i32.contiguous(),
        mask_u8.contiguous(),
        family_bounds_i32.contiguous(),
        family_order_caps_i32.contiguous(),
        dispatch_meta_i32.contiguous(),
        float(scale),
        bool(cand_is_per_head),
        head_mask_u8,
        runtime_i32,
        k_cfg_f32,
        v_cfg_f32,
        int(query_start),
        int(q_len_per_batch),
        int(kv_len_per_batch),
    )

    if restore_shape is None:
        return out3

    B, QT = restore_shape
    return out3.view(B, QT, out3.size(1), out3.size(2))

**cublasLt QK batched + V(Stage B: Fuse Q/K/V projections with attention)**

In [ ]:
# Colab-friendly single Python cell that compiles your cublasLt Stage-B projector (Q,K,V)
# ✅ Keeps your CUDA_SRC code VERBATIM where changes are not required
# ✅ Converts ONLY the risky numerics to FP32:
#    - cublasLt outputs C in FP32 (Q/K/V are float32)
#    - ATen fallback matmuls run in FP32 (prevents fp16 overflow/Inf)
# ✅ Links -lcublasLt -lcublas
# ✅ Prints build.ninja snippets + re-runs ninja -v -j 1 on failure for full logs

import os, pathlib, subprocess, torch
from torch.utils.cpp_extension import load_inline

# ----------------------------
# Environment + build dir
# ----------------------------
os.environ.setdefault("TORCH_EXTENSIONS_DIR", "/content/torch_extensions")
TORCH_EXT_DIR = os.environ["TORCH_EXTENSIONS_DIR"]

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("cc:", torch.cuda.get_device_capability(0))

import hashlib

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# 2) Better C++ stack traces when extensions throw
os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "1"

# Toggle these as needed:
FUSED_ENABLE_HOST_LOGS   = False    # C++ host-side logs via std::cout
FUSED_ENABLE_DEVICE_LOGS = False   # CUDA printf inside kernel
FUSED_SYNC_AFTER_KERNEL  = False    # device sync after kernel for accurate failure location
FUSED_VERBOSE_BUILD      = False    # verbose compile logs

# V-tiling toggle
FUSED_USE_SMEM_V         = True    # introduce V-tiling

# ----------------------------
# CUDA_SRC (FULL CONTENT)  <-- your code with ONLY fp32-risk hardening edits
# ----------------------------
CUDA_SRC = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <cuda_runtime.h>
#include <cublasLt.h>
#include <cublas_v2.h>
#include <stdint.h>
#include <limits>
// ----------------------------
// SASA Smoke-Test Debug Macros
//   - Enable device-side printf with -DSASA_DEVICE_DEBUG
//   - Keep kernels semantically identical; logs only.
// ----------------------------
#if defined(SASA_DEVICE_DEBUG)
  #include <cstdio>
  #define SASA_DPRINTF(...) printf(__VA_ARGS__)
#else
  #define SASA_DPRINTF(...) ((void)0)
#endif

static inline void CUBLASLT_CHECK(cublasStatus_t s){
  TORCH_CHECK(s == CUBLAS_STATUS_SUCCESS, "cublasLt failed with status ", (int)s);
}
static inline void CUDA_CHECK(cudaError_t e){
  TORCH_CHECK(e == cudaSuccess, "CUDA failed: ", cudaGetErrorString(e));
}

static cublasLtHandle_t get_cublaslt_handle() {
  static cublasLtHandle_t handle = nullptr;
  static bool initialized = false;
  if (!initialized) {
    CUBLASLT_CHECK(cublasLtCreate(&handle));
    initialized = true;
  }
  return handle;
}

template <typename scalar_t>
__global__ void pack_wq_wk_kernel(
    const scalar_t* __restrict__ Wq, // [Dq, Dm]
    const scalar_t* __restrict__ Wk, // [Dq, Dm]
    scalar_t* __restrict__ Wqk,      // [2, Dq, Dm]
    int64_t n // Dq*Dm
){
  if (blockIdx.x==0 && threadIdx.x==0) { SASA_DPRINTF("[SASA_DEBUG] enter %s\n", "pack_wq_wk_kernel"); }

  int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    Wqk[i]       = Wq[i];
    Wqk[n + i]   = Wk[i];
  }
}

static inline cudaDataType_t dtype_to_cuda(torch::ScalarType t){
  if (t == at::kHalf) return CUDA_R_16F;
  if (t == at::kBFloat16) return CUDA_R_16BF;
  if (t == at::kFloat) return CUDA_R_32F;
  // should never happen due to checks
  return CUDA_R_32F;
}

static inline void set_rowmajor(cublasLtMatrixLayout_t layout){
  cublasLtOrder_t order = CUBLASLT_ORDER_ROW;
  CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
      layout, CUBLASLT_MATRIX_LAYOUT_ORDER, &order, sizeof(order)));
}

// Core helper: C = A * B^T, row-major, optionally strided-batched
// Returns true iff Lt matmul actually succeeded; false triggers caller fallback.
static bool lt_matmul_rowmajor_A_Bt(
    const void* A, const void* B, void* C,
    int64_t M, int64_t N, int64_t K,
    int64_t lda, int64_t ldb, int64_t ldc,
    cudaDataType_t Atype, cudaDataType_t Btype, cudaDataType_t Ctype,
    int batchCount,
    int64_t strideA_elems,
    int64_t strideB_elems,
    int64_t strideC_elems,
    cudaStream_t stream
){
  cublasLtHandle_t lt = get_cublaslt_handle();

  // Use FP32 accumulation for fp16/bf16, FP32 compute for fp32.
  cublasComputeType_t computeType = CUBLAS_COMPUTE_32F;

  cublasLtMatmulDesc_t opDesc;
  CUBLASLT_CHECK(cublasLtMatmulDescCreate(&opDesc, computeType, CUDA_R_32F));

  cublasOperation_t opA = CUBLAS_OP_N;
  cublasOperation_t opB = CUBLAS_OP_T;
  CUBLASLT_CHECK(cublasLtMatmulDescSetAttribute(opDesc, CUBLASLT_MATMUL_DESC_TRANSA, &opA, sizeof(opA)));
  CUBLASLT_CHECK(cublasLtMatmulDescSetAttribute(opDesc, CUBLASLT_MATMUL_DESC_TRANSB, &opB, sizeof(opB)));

  cublasLtMatrixLayout_t aLayout, bLayout, cLayout;
  // A: [M,K] row-major
  CUBLASLT_CHECK(cublasLtMatrixLayoutCreate(&aLayout, Atype, M, K, lda));
  set_rowmajor(aLayout);

  // B: [N,K] row-major (because we use B^T)
  CUBLASLT_CHECK(cublasLtMatrixLayoutCreate(&bLayout, Btype, N, K, ldb));
  set_rowmajor(bLayout);

  // C: [M,N] row-major
  CUBLASLT_CHECK(cublasLtMatrixLayoutCreate(&cLayout, Ctype, M, N, ldc));
  set_rowmajor(cLayout);

  if (batchCount > 1) {
    CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
        aLayout, CUBLASLT_MATRIX_LAYOUT_BATCH_COUNT, &batchCount, sizeof(batchCount)));
    CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
        bLayout, CUBLASLT_MATRIX_LAYOUT_BATCH_COUNT, &batchCount, sizeof(batchCount)));
    CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
        cLayout, CUBLASLT_MATRIX_LAYOUT_BATCH_COUNT, &batchCount, sizeof(batchCount)));

    CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
        aLayout, CUBLASLT_MATRIX_LAYOUT_STRIDED_BATCH_OFFSET, &strideA_elems, sizeof(strideA_elems)));
    CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
        bLayout, CUBLASLT_MATRIX_LAYOUT_STRIDED_BATCH_OFFSET, &strideB_elems, sizeof(strideB_elems)));
    CUBLASLT_CHECK(cublasLtMatrixLayoutSetAttribute(
        cLayout, CUBLASLT_MATRIX_LAYOUT_STRIDED_BATCH_OFFSET, &strideC_elems, sizeof(strideC_elems)));
  }

  cublasLtMatmulPreference_t pref;
  CUBLASLT_CHECK(cublasLtMatmulPreferenceCreate(&pref));

  // Workspace (keep modest; you can tune)
  size_t workspaceSize = 8 * 1024 * 1024;
  CUBLASLT_CHECK(cublasLtMatmulPreferenceSetAttribute(
      pref, CUBLASLT_MATMUL_PREF_MAX_WORKSPACE_BYTES, &workspaceSize, sizeof(workspaceSize)));

  cublasLtMatmulHeuristicResult_t heuristic;
  int returned = 0;
  CUBLASLT_CHECK(cublasLtMatmulAlgoGetHeuristic(
      lt, opDesc, aLayout, bLayout, cLayout, cLayout, pref, 1, &heuristic, &returned));

  // Allocate workspace (optional but helps algos)
  void* workspace = nullptr;
  if (returned > 0 && workspaceSize > 0) {
    CUDA_CHECK(cudaMalloc(&workspace, workspaceSize));
  }

  const float alpha = 1.0f;
  const float beta  = 0.0f;

  bool ok = false;

  // If heuristic not found, return false; caller must fallback.
  if (returned > 0) {
    cublasStatus_t st = cublasLtMatmul(
        lt,
        opDesc,
        &alpha,
        A, aLayout,
        B, bLayout,
        &beta,
        C, cLayout,
        C, cLayout,
        &heuristic.algo,
        workspace, workspaceSize,
        stream
    );
    ok = (st == CUBLAS_STATUS_SUCCESS);
  } else {
    printf("No cublasLt heuristic found; caller must fallback.\n");
    ok = false;
  }

  if (workspace) CUDA_CHECK(cudaFree(workspace));
  CUBLASLT_CHECK(cublasLtMatmulPreferenceDestroy(pref));
  CUBLASLT_CHECK(cublasLtMatrixLayoutDestroy(aLayout));
  CUBLASLT_CHECK(cublasLtMatrixLayoutDestroy(bLayout));
  CUBLASLT_CHECK(cublasLtMatrixLayoutDestroy(cLayout));
  CUBLASLT_CHECK(cublasLtMatmulDescDestroy(opDesc));

  return ok;
}

void proj_qkv_cuda(
    const torch::Tensor& H,     // [T, Dm]
    const torch::Tensor& Wq,    // [Dq, Dm]
    const torch::Tensor& Wk,    // [Dq, Dm]
    const torch::Tensor& Wv,    // [Dv, Dm]
    torch::Tensor& Q,           // [T, Dq]  (FP32 output)
    torch::Tensor& K,           // [T, Dq]  (FP32 output)
    torch::Tensor& V            // [T, Dv]  (FP32 output)
){
  // ---------------------------
  // HARD CHECKS
  // ---------------------------
  TORCH_CHECK(H.is_cuda() && Wq.is_cuda() && Wk.is_cuda() && Wv.is_cuda(), "H/Wq/Wk/Wv must be CUDA");
  c10::cuda::CUDAGuard guard(H.device());
  TORCH_CHECK(H.device() == Wq.device(), "H and Wq must be on same device");
  TORCH_CHECK(H.device() == Wk.device(), "H and Wk must be on same device");
  TORCH_CHECK(H.device() == Wv.device(), "H and Wv must be on same device");
  TORCH_CHECK(H.device() == Q.device(),  "H and Q must be on same device");
  TORCH_CHECK(H.device() == K.device(),  "H and K must be on same device");
  TORCH_CHECK(H.device() == V.device(),  "H and V must be on same device");
  TORCH_CHECK(H.is_contiguous(), "H must be contiguous");
  TORCH_CHECK(Wq.is_contiguous() && Wk.is_contiguous() && Wv.is_contiguous(), "Wq/Wk/Wv must be contiguous");
  TORCH_CHECK(Q.is_contiguous() && K.is_contiguous() && V.is_contiguous(), "Q/K/V must be contiguous");
  TORCH_CHECK(H.dim() == 2, "H must be [T,Dm]");
  TORCH_CHECK(Wq.dim() == 2 && Wk.dim() == 2 && Wv.dim() == 2, "Wq/Wk/Wv must be 2D");
  TORCH_CHECK(Wq.size(1) == H.size(1) && Wk.size(1) == H.size(1) && Wv.size(1) == H.size(1), "W*.size(1) must equal Dm");
  TORCH_CHECK(Wq.size(0) == Wk.size(0), "Wq/Wk output dims must match (Dq)");
  TORCH_CHECK(Q.size(0) == H.size(0) && K.size(0) == H.size(0) && V.size(0) == H.size(0), "Q/K/V T mismatch");
  TORCH_CHECK(Q.size(1) == Wq.size(0) && K.size(1) == Wk.size(0) && V.size(1) == Wv.size(0), "Q/K/V output dim mismatch");

  auto dt = H.scalar_type();
  TORCH_CHECK(dt == Wq.scalar_type() && dt == Wk.scalar_type() && dt == Wv.scalar_type(), "H and Wq/Wk/Wv must have same dtype");
  TORCH_CHECK(dt == at::kHalf || dt == at::kBFloat16 || dt == at::kFloat, "Only half/bfloat16/float supported (no double)");

  // ---- FP32 OUTPUT INVARIANT (risk hardening) ----
  TORCH_CHECK(Q.scalar_type() == at::kFloat && K.scalar_type() == at::kFloat && V.scalar_type() == at::kFloat,
              "Q/K/V must be float32 outputs for numeric safety");

  const int64_t T  = H.size(0);
  const int64_t Dm = H.size(1);
  const int64_t Dq = Wq.size(0);
  const int64_t Dv = Wv.size(0);

  TORCH_CHECK(T > 0 && Dm > 0 && Dq > 0 && Dv > 0,
              "T/Dm/Dq/Dv must be positive");
  TORCH_CHECK(Dq <= std::numeric_limits<int64_t>::max() / Dm,
              "Dq*Dm would overflow int64");
  const int64_t n = Dq * Dm;
  TORCH_CHECK(T <= std::numeric_limits<int64_t>::max() / Dq,
              "T*Dq would overflow int64");
  const int64_t TDq = T * Dq;
  TORCH_CHECK(T <= std::numeric_limits<int64_t>::max() / Dv,
              "T*Dv would overflow int64");
  const int64_t TDv = T * Dv;
  (void)TDv;

  cudaDataType_t Atype = dtype_to_cuda(dt);
  cudaDataType_t Btype = dtype_to_cuda(dt);

  // ---- FP32 OUTPUT TYPE (risk hardening) ----
  cudaDataType_t Ctype = CUDA_R_32F;

  // FIX: use CURRENT stream (matches PyTorch ops and avoids races)
  cudaStream_t stream = at::cuda::getCurrentCUDAStream();

  // ---- Pack Wq/Wk into [2, Dq, Dm] contiguous (tiny cost) ----
  auto Wqk = torch::empty({2, Dq, Dm}, H.options());
  int threads = 256;
  TORCH_CHECK(n <= (int64_t)std::numeric_limits<int>::max() * (int64_t)threads,
              "pack_wq_wk grid would exceed int block count");
  int blocks = (int)((n + threads - 1) / threads);

  AT_DISPATCH_FLOATING_TYPES_AND2(at::kHalf, at::kBFloat16, dt, "pack_wq_wk", [&](){
    pack_wq_wk_kernel<scalar_t><<<blocks, threads, 0, stream>>>(
        (const scalar_t*)Wq.data_ptr(),
        (const scalar_t*)Wk.data_ptr(),
        (scalar_t*)Wqk.data_ptr(),
        n
    );
  });
  auto pack_err = cudaGetLastError();
  TORCH_CHECK(pack_err == cudaSuccess,
              "pack_wq_wk_kernel launch failed: ", cudaGetErrorString(pack_err));

  // ---- QK: one cublasLt strided-batched matmul, batchCount=2 ----
  // A = H [T,Dm], B = Wqk [2,Dq,Dm], we compute C = A * B^T => [2,T,Dq]
  // We write into a temporary [2,T,Dq] and then split into Q,K.
  auto QK = torch::empty({2, T, Dq}, H.options().dtype(at::kFloat));

  // Row-major leading dimensions
  const int64_t lda = Dm; // A is [T,Dm]
  const int64_t ldb = Dm; // B is [Dq,Dm]
  const int64_t ldc = Dq; // C is [T,Dq]

  const int batchCount = 2;

  // ---- Strides in ELEMENTS for cublasLt strided-batch offsets ----
  const int64_t strideA = 0;        // broadcast same H for both batches
  const int64_t strideB = n;        // jump between Wq and Wk in packed buffer
  const int64_t strideC = TDq;      // jump between Q and K in output buffer

  bool ok_qk = lt_matmul_rowmajor_A_Bt(
      H.data_ptr(), Wqk.data_ptr(), QK.data_ptr(),
      T, Dq, Dm,
      lda, ldb, ldc,
      Atype, Btype, Ctype,
      batchCount,
      strideA, strideB, strideC,
      stream
  );

  // Lt failure fallback: ATen matmul (safe, correct, FP32 intermediate).
  if (!ok_qk) {
    auto Hf  = H.to(at::kFloat);
    auto Wqf = Wq.to(at::kFloat);
    auto Wkf = Wk.to(at::kFloat);
    Q.copy_(Hf.matmul(Wqf.transpose(0, 1)));
    K.copy_(Hf.matmul(Wkf.transpose(0, 1)));
  } else {
    // Slice without copy (views), then copy into Q,K tensors you allocated in C++.
    // Q,K expected contiguous [T,Dq] fp32.
    Q.copy_(QK.select(0,0));
    K.copy_(QK.select(0,1));
  }

  // ---- V: separate cublasLt matmul (different N=Dv) ----
  bool ok_v = lt_matmul_rowmajor_A_Bt(
      H.data_ptr(), Wv.data_ptr(), V.data_ptr(),
      T, Dv, Dm,
      lda, (int64_t)Dm, (int64_t)Dv,
      Atype, Btype, Ctype,
      /*batchCount=*/1,
      /*strideA=*/0, /*strideB=*/0, /*strideC=*/0,
      stream
  );

  if (!ok_v) {
    auto Hf  = H.to(at::kFloat);
    auto Wvf = Wv.to(at::kFloat);
    V.copy_(Hf.matmul(Wvf.transpose(0, 1)));
  }
}

// ============================================================================
// Minimal C++/PyBind wrapper (so Python can call it)
// ============================================================================

std::vector<torch::Tensor> proj_qkv(
    torch::Tensor H,   // [T,Dm]
    torch::Tensor Wq,  // [Dq,Dm]
    torch::Tensor Wk,  // [Dq,Dm]
    torch::Tensor Wv   // [Dv,Dm]
){
  TORCH_CHECK(H.is_cuda() && Wq.is_cuda() && Wk.is_cuda() && Wv.is_cuda(), "All inputs must be CUDA");
  c10::cuda::CUDAGuard guard(H.device());
  TORCH_CHECK(H.device() == Wq.device(), "H and Wq must be on same device");
  TORCH_CHECK(H.device() == Wk.device(), "H and Wk must be on same device");
  TORCH_CHECK(H.device() == Wv.device(), "H and Wv must be on same device");
  TORCH_CHECK(H.is_contiguous() && Wq.is_contiguous() && Wk.is_contiguous() && Wv.is_contiguous(), "Inputs must be contiguous");
  TORCH_CHECK(H.dim()==2 && Wq.dim()==2 && Wk.dim()==2 && Wv.dim()==2, "H/Wq/Wk/Wv must be 2D");
  TORCH_CHECK(Wq.size(1)==H.size(1) && Wk.size(1)==H.size(1) && Wv.size(1)==H.size(1), "W*.size(1) must match H.size(1)");

  const auto T  = H.size(0);
  const auto Dq = Wq.size(0);
  const auto Dv = Wv.size(0);

  // ---- FP32 OUTPUTS (risk hardening) ----
  auto out_opts = H.options().dtype(at::kFloat);
  auto Q = torch::empty({T, Dq}, out_opts);
  auto K = torch::empty({T, Dq}, out_opts);
  auto V = torch::empty({T, Dv}, out_opts);

  proj_qkv_cuda(H, Wq, Wk, Wv, Q, K, V);

  // Ensure kernel launch errors get surfaced promptly
  auto err = cudaGetLastError();
  TORCH_CHECK(err == cudaSuccess, "proj_qkv kernels failed: ", cudaGetErrorString(err));

  return {Q, K, V};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("proj_qkv", &proj_qkv, "Stage-B QKV projection via cublasLt (CUDA)");
}"""
# ext_name = "stageB_cublaslt_qkv_projector_ext"
import hashlib

CUDA_SRC = CUDA_SRC  # unchanged

# ---- build-hash (forces rebuild when ABI / math / linker config changes) ----
_src_hash = hashlib.md5(CUDA_SRC.encode("utf-8")).hexdigest()[:10]
_build_sig = (
    "stageB_cublaslt_qkv_projector|"
    "O3_fastmath|"
    "cxx17|"
    "cublasLt|cublas|"
    "out_fp32|"
    f"src={_src_hash}"
)
_build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]

ext_name = f"stageB_cublaslt_qkv_projector_ext_{_build_hash}"

stageB_cublaslt_qkv_projector = load_inline(
    name=ext_name,
    cpp_sources="",
    cuda_sources=CUDA_SRC,
    functions=None,          # we use PYBIND11_MODULE
    extra_cuda_cflags=[
        "-O3",
        "--use_fast_math",
    ],
    extra_cflags=["-O3", "-std=c++17"],
    extra_ldflags=["-lcublasLt", "-lcublas"],
    with_cuda=True,
    verbose=False,
)

print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))
print("detect_anomaly =", torch.is_anomaly_enabled())

proj_qkv = stageB_cublaslt_qkv_projector.proj_qkv

torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
cc: (7, 5)
CUDA_LAUNCH_BLOCKING = 0
TORCH_SHOW_CPP_STACKTRACES = 0
detect_anomaly = True


In [ ]:
import math
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict, Any, Callable

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GCCDBackboneConfig:
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: Optional[int] = None
    max_position_embeddings: int = 32768
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-6
    hidden_act: str = "silu"
    attention_dropout: float = 0.0
    resid_dropout: float = 0.0
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2
    use_bias: bool = False

    # Stage-A / Stage-E integration flags
    use_sparse_stagee: bool = True
    use_lcha: bool = True

    # default fused LCHA knobs
    lcha_enabled: bool = False
    lcha_order: int = 0
    lcha_heads_frac: float = 0.0
    lcha_use_v: bool = False
    lcha_alpha0: float = 1.0
    lcha_alpha1: float = 0.0
    lcha_alpha2: float = 0.0
    lcha_v_alpha0: float = 1.0
    lcha_v_alpha1: float = 0.0
    lcha_v_alpha2: float = 0.0

    def __post_init__(self):
        if self.num_key_value_heads is None:
            self.num_key_value_heads = self.num_attention_heads

        if self.vocab_size <= 0:
            raise ValueError("vocab_size must be > 0")
        if self.hidden_size <= 0:
            raise ValueError("hidden_size must be > 0")
        if self.intermediate_size <= 0:
            raise ValueError("intermediate_size must be > 0")
        if self.num_hidden_layers <= 0:
            raise ValueError("num_hidden_layers must be > 0")
        if self.num_attention_heads <= 0:
            raise ValueError("num_attention_heads must be > 0")
        if self.num_key_value_heads <= 0:
            raise ValueError("num_key_value_heads must be > 0")
        if self.hidden_size % self.num_attention_heads != 0:
            raise ValueError("hidden_size must be divisible by num_attention_heads")
        if self.num_attention_heads % self.num_key_value_heads != 0:
            raise ValueError("num_attention_heads must be divisible by num_key_value_heads")
        if self.max_position_embeddings <= 0:
            raise ValueError("max_position_embeddings must be > 0")
        if self.rope_theta <= 0.0:
            raise ValueError("rope_theta must be > 0")
        if self.rms_norm_eps <= 0.0:
            raise ValueError("rms_norm_eps must be > 0")
        if not (0.0 <= self.attention_dropout < 1.0):
            raise ValueError("attention_dropout must be in [0, 1)")
        if not (0.0 <= self.resid_dropout < 1.0):
            raise ValueError("resid_dropout must be in [0, 1)")
        if self.lcha_order not in (0, 1, 2):
            raise ValueError("lcha_order must be 0, 1, or 2")
        if not (0.0 <= self.lcha_heads_frac <= 1.0):
            raise ValueError("lcha_heads_frac must be in [0, 1]")


@dataclass
class GCCDModelOutput:
    logits: torch.Tensor
    last_hidden_state: torch.Tensor
    loss: Optional[torch.Tensor] = None
    hidden_states: Optional[Tuple[torch.Tensor, ...]] = None
    past_key_values: Optional[Tuple[Tuple[torch.Tensor, torch.Tensor], ...]] = None
    attn_meta: Optional[Tuple[Dict[str, Any], ...]] = None

In [ ]:
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        if hidden_size <= 0:
            raise ValueError("hidden_size must be > 0")
        if eps <= 0.0:
            raise ValueError("eps must be > 0")

        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        x_float = x.float()
        var = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x_float * torch.rsqrt(var + self.eps)
        return (self.weight.to(device=x.device, dtype=x_norm.dtype) * x_norm).to(input_dtype)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    if x.shape[-1] % 2 != 0:
        raise ValueError("RoPE head_dim must be even for rotate_half")
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def build_rope_cache(
    seq_len: int,
    head_dim: int,
    device: torch.device,
    base: float = 10000.0,
    dtype: torch.dtype = torch.float32,
) -> Tuple[torch.Tensor, torch.Tensor]:
    seq_len = int(seq_len)
    head_dim = int(head_dim)

    if seq_len < 0:
        raise ValueError("seq_len must be >= 0")
    if head_dim <= 0:
        raise ValueError("head_dim must be > 0")
    if head_dim % 2 != 0:
        raise ValueError("head_dim must be even for RoPE")
    if base <= 0.0:
        raise ValueError("RoPE base must be > 0")

    calc_dtype = torch.float32
    inv_freq = 1.0 / (
        base ** (torch.arange(0, head_dim, 2, device=device, dtype=calc_dtype) / head_dim)
    )
    t = torch.arange(seq_len, device=device, dtype=calc_dtype)
    freqs = torch.outer(t, inv_freq)  # [T, D/2]
    emb = torch.cat([freqs, freqs], dim=-1)  # [T, D]
    return emb.cos().to(dtype=dtype), emb.sin().to(dtype=dtype)


def apply_rope(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
    position_offset: int = 0,
) -> torch.Tensor:
    # x: [B, T, H, D]
    # cos/sin: [T_total, D]
    if x.dim() != 4:
        raise ValueError(f"x must be [B,T,H,D], got {tuple(x.shape)}")
    if cos.shape != sin.shape:
        raise ValueError("cos and sin must have the same shape")
    if cos.dim() != 2:
        raise ValueError(f"cos/sin must be [T_total,D], got {tuple(cos.shape)}")

    T = int(x.shape[1])
    D = int(x.shape[-1])
    position_offset = int(position_offset)

    if D % 2 != 0:
        raise ValueError("RoPE head_dim must be even")
    if cos.shape[-1] != D:
        raise ValueError(f"RoPE dim mismatch: x has D={D}, cos/sin have D={cos.shape[-1]}")
    if position_offset < 0:
        raise ValueError("position_offset must be >= 0")
    if position_offset + T > cos.shape[0]:
        raise ValueError(
            f"RoPE cache too short: need positions up to {position_offset + T}, "
            f"but cache length is {cos.shape[0]}"
        )

    cos = cos[position_offset:position_offset + T].to(device=x.device, dtype=x.dtype)
    sin = sin[position_offset:position_offset + T].to(device=x.device, dtype=x.dtype)

    cos = cos.unsqueeze(0).unsqueeze(2)  # [1, T, 1, D]
    sin = sin.unsqueeze(0).unsqueeze(2)
    return (x * cos) + (rotate_half(x) * sin)


def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    n_rep = int(n_rep)
    if x.dim() != 4:
        raise ValueError(f"x must be [B,T,H,D], got {tuple(x.shape)}")
    if n_rep <= 0:
        raise ValueError("n_rep must be > 0")

    if n_rep == 1:
        return x
    b, t, h, d = x.shape
    x = x[:, :, :, None, :].expand(b, t, h, n_rep, d)
    return x.reshape(b, t, h * n_rep, d)


class GCCDMLP(nn.Module):
    def __init__(self, cfg: GCCDBackboneConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=cfg.use_bias)
        self.up_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=cfg.use_bias)
        self.down_proj = nn.Linear(cfg.intermediate_size, cfg.hidden_size, bias=cfg.use_bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

In [ ]:
from typing import Optional, Callable, Dict, Any, Tuple

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class GCCDSparseDecoderAttention(nn.Module):
    """
    Real decoder attention block for the custom GCCD backbone.

    Two notebook hooks are injected from outside:
      1) candidate_builder_fn
      2) stagee_apply_fn

    This keeps the architecture real, but avoids hardcoding old Mistral-specific wiring.
    """

    def __init__(
        self,
        cfg: GCCDBackboneConfig,
        layer_idx: int,
        candidate_builder_fn: Optional[Callable[..., Dict[str, Any]]] = None,
        stagee_apply_fn: Optional[Callable[..., torch.Tensor]] = None,
    ):
        super().__init__()
        self.cfg = cfg
        self.layer_idx = int(layer_idx)

        self.num_heads = int(cfg.num_attention_heads)
        self.num_kv_heads = int(cfg.num_key_value_heads or cfg.num_attention_heads)

        if self.num_heads <= 0:
            raise ValueError("cfg.num_attention_heads must be > 0")
        if self.num_kv_heads <= 0:
            raise ValueError("cfg.num_key_value_heads must be > 0")
        if int(cfg.hidden_size) % self.num_heads != 0:
            raise ValueError("cfg.hidden_size must be divisible by cfg.num_attention_heads")
        if self.num_heads % self.num_kv_heads != 0:
            raise ValueError("num_attention_heads must be divisible by num_key_value_heads")

        self.head_dim = int(cfg.hidden_size) // self.num_heads
        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        self.q_proj = nn.Linear(cfg.hidden_size, self.num_heads * self.head_dim, bias=cfg.use_bias)
        self.k_proj = nn.Linear(cfg.hidden_size, self.num_kv_heads * self.head_dim, bias=cfg.use_bias)
        self.v_proj = nn.Linear(cfg.hidden_size, self.num_kv_heads * self.head_dim, bias=cfg.use_bias)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, cfg.hidden_size, bias=cfg.use_bias)

        self.attn_dropout = cfg.attention_dropout
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

        self.candidate_builder_fn = candidate_builder_fn
        self.stagee_apply_fn = stagee_apply_fn

        # protocol / LCHA knobs
        self.lcha_enabled = cfg.lcha_enabled
        self.lcha_order = cfg.lcha_order
        self.lcha_heads_frac = cfg.lcha_heads_frac
        self.lcha_use_v = cfg.lcha_use_v
        self.lcha_alpha0 = cfg.lcha_alpha0
        self.lcha_alpha1 = cfg.lcha_alpha1
        self.lcha_alpha2 = cfg.lcha_alpha2
        self.lcha_v_alpha0 = cfg.lcha_v_alpha0
        self.lcha_v_alpha1 = cfg.lcha_v_alpha1
        self.lcha_v_alpha2 = cfg.lcha_v_alpha2

        # family metadata placeholder for full paper-speed version
        self.family_meta: Optional[Dict[str, Any]] = None

    def _prepare_attention_mask(
        self,
        attention_mask: Optional[torch.Tensor],
        *,
        bsz: int,
        tq: int,
        sk: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> Optional[torch.Tensor]:
        if attention_mask is None:
            return None

        mask = attention_mask.to(device=device)

        if mask.dim() == 2:
            # [B,S]
            if mask.shape[0] != bsz or mask.shape[1] != sk:
                raise ValueError(
                    f"attention_mask [B,S] shape mismatch: got {tuple(mask.shape)}, expected [{bsz},{sk}]"
                )
            mask = mask[:, None, None, :]  # [B,1,1,S]

        elif mask.dim() == 3:
            # [B,1,S] or [B,T,S]
            if mask.shape[0] != bsz or mask.shape[-1] != sk:
                raise ValueError(
                    f"attention_mask rank-3 shape mismatch: got {tuple(mask.shape)}, expected B={bsz}, S={sk}"
                )
            if mask.shape[1] not in (1, tq):
                raise ValueError(
                    f"attention_mask rank-3 middle dim must be 1 or T={tq}, got {mask.shape[1]}"
                )
            mask = mask[:, None, :, :]  # [B,1,1,S] or [B,1,T,S]

        elif mask.dim() == 4:
            # [B,1,1,S], [B,1,T,S], or broadcastable head dim
            if mask.shape[0] != bsz or mask.shape[-1] != sk:
                raise ValueError(
                    f"attention_mask rank-4 shape mismatch: got {tuple(mask.shape)}, expected B={bsz}, S={sk}"
                )
            if mask.shape[-2] not in (1, tq):
                raise ValueError(
                    f"attention_mask query dim must be 1 or T={tq}, got {mask.shape[-2]}"
                )

        else:
            raise ValueError(f"attention_mask must have rank 2, 3, or 4, got {mask.dim()}")

        mask_value = torch.finfo(dtype).min

        if mask.dtype == torch.bool:
            # bool mask: True = keep, False = mask
            return torch.zeros_like(mask, dtype=dtype).masked_fill(~mask, mask_value)

        mask = mask.to(dtype=dtype)

        # Standard HF-style mask: 1 = keep, 0 = pad.
        # Already-additive masks usually contain negative values.
        if mask.numel() > 0 and float(mask.min().item()) >= 0.0 and float(mask.max().item()) <= 1.0:
            return (1.0 - mask) * mask_value

        # Already additive/broadcastable mask.
        return mask

    def _validate_past_key_value(
        self,
        past_key_value: Tuple[torch.Tensor, torch.Tensor],
        *,
        bsz: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> None:
        if not isinstance(past_key_value, tuple) or len(past_key_value) != 2:
            raise ValueError("past_key_value must be a tuple of (past_k, past_v)")

        past_k, past_v = past_key_value
        expected_tail = (self.num_kv_heads, self.head_dim)

        if past_k.dim() != 4 or past_v.dim() != 4:
            raise ValueError("past_k and past_v must both be [B,S,Hkv,D]")
        if past_k.shape != past_v.shape:
            raise ValueError(f"past_k and past_v shape mismatch: {tuple(past_k.shape)} vs {tuple(past_v.shape)}")
        if past_k.shape[0] != bsz:
            raise ValueError(f"past batch mismatch: got {past_k.shape[0]}, expected {bsz}")
        if tuple(past_k.shape[2:]) != expected_tail:
            raise ValueError(
                f"past KV tail shape mismatch: got {tuple(past_k.shape[2:])}, expected {expected_tail}"
            )
        if past_k.device != device or past_v.device != device:
            raise ValueError("past_key_value tensors must be on the same device as hidden_states")
        if past_k.dtype != dtype or past_v.dtype != dtype:
            raise ValueError("past_key_value tensors must have the same dtype as hidden_states")

    def _dense_fallback(
        self,
        q: torch.Tensor,  # [B,T,H,D]
        k: torch.Tensor,  # [B,S,H,D]
        v: torch.Tensor,  # [B,S,H,D]
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, tq, h, d = q.shape
        _, sk, _, _ = k.shape

        q_ = q.permute(0, 2, 1, 3)  # [B,H,T,D]
        k_ = k.permute(0, 2, 1, 3)
        v_ = v.permute(0, 2, 1, 3)

        attn = torch.matmul(q_, k_.transpose(-2, -1)) / math.sqrt(d)

        mask_value = torch.finfo(attn.dtype).min
        causal = torch.triu(
            torch.ones(tq, sk, device=q.device, dtype=torch.bool),
            diagonal=1 + (sk - tq),
        )
        attn = attn.masked_fill(causal, mask_value)

        prepared_mask = self._prepare_attention_mask(
            attention_mask,
            bsz=b,
            tq=tq,
            sk=sk,
            device=q.device,
            dtype=attn.dtype,
        )
        if prepared_mask is not None:
            attn = attn + prepared_mask

        probs = F.softmax(attn.float(), dim=-1).to(q.dtype)
        out = torch.matmul(probs, v_)  # [B,H,T,D]
        return out.permute(0, 2, 1, 3).contiguous()

    def forward(
        self,
        hidden_states: torch.Tensor,                 # [B,T,C]
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
        output_attn_meta: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Dict[str, Any]]]:
        if hidden_states.dim() != 3:
            raise ValueError(f"hidden_states must be [B,T,C], got {tuple(hidden_states.shape)}")
        if hidden_states.shape[-1] != self.cfg.hidden_size:
            raise ValueError(
                f"hidden_states last dim mismatch: got {hidden_states.shape[-1]}, expected {self.cfg.hidden_size}"
            )
        if hidden_states.shape[1] <= 0:
            raise ValueError("q_len must be > 0")

        bsz, q_len, _ = hidden_states.shape
        device = hidden_states.device
        dtype = hidden_states.dtype

        if past_key_value is not None:
            self._validate_past_key_value(
                past_key_value,
                bsz=bsz,
                device=device,
                dtype=dtype,
            )

        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_kv_heads, self.head_dim)

        past_len = 0 if past_key_value is None else int(past_key_value[0].shape[1])
        total_len = past_len + q_len

        cos, sin = build_rope_cache(
            seq_len=total_len,
            head_dim=self.head_dim,
            device=device,
            base=self.cfg.rope_theta,
            dtype=torch.float32,
        )
        cos = cos[past_len:total_len].to(device=device, dtype=dtype)
        sin = sin[past_len:total_len].to(device=device, dtype=dtype)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=1)
            v = torch.cat([past_v, v], dim=1)

        present = (k, v) if use_cache else None

        if self.num_kv_heads != self.num_heads:
            n_rep = self.num_heads // self.num_kv_heads
            k_attn = repeat_kv(k, n_rep)
            v_attn = repeat_kv(v, n_rep)
        else:
            k_attn = k
            v_attn = v

        attn_meta = None

        # ---- Sparse Stage-A + Stage-E path ----
        if self.cfg.use_sparse_stagee and self.candidate_builder_fn is not None and self.stagee_apply_fn is not None:
            stagea = self.candidate_builder_fn(
                q=q,
                k=k_attn,
                v=v_attn,
                attn_module=self,
                layer_idx=self.layer_idx,
                past_len=past_len,
                total_len=total_len,
                attention_mask=attention_mask,
            )
            if not isinstance(stagea, dict):
                raise ValueError("candidate_builder_fn must return a dict")
            if "cand_i32" not in stagea or "mask_u8" not in stagea:
                raise ValueError("candidate_builder_fn output must contain 'cand_i32' and 'mask_u8'")

            cand_i32 = stagea["cand_i32"].to(device=device, dtype=torch.int32)
            mask_u8 = stagea["mask_u8"].to(device=device, dtype=torch.uint8)
            cand_is_per_head = bool(stagea.get("cand_is_per_head", False))
            family_meta = stagea.get("family_meta", None)

            # Current minimal/full fused Stage-E-LCHA bridge.
            # stagee_apply_fn should internally call StageEMHFusedFn.apply or your final fused wrapper.
            out_thd = self.stagee_apply_fn(
                q=q,                        # [B,T,H,D]
                k=k_attn,                   # [B,S,H,D]
                v=v_attn,                   # [B,S,H,D]
                cand_i32=cand_i32,          # [T,C] or [T*H,C]
                mask_u8=mask_u8,            # [T,C] or [T*H,C]
                cand_is_per_head=cand_is_per_head,
                lcha_enabled=bool(self.lcha_enabled),
                lcha_order=int(self.lcha_order),
                lcha_heads_frac=float(self.lcha_heads_frac),
                lcha_use_v=bool(self.lcha_use_v),
                lcha_alpha0=float(self.lcha_alpha0),
                lcha_alpha1=float(self.lcha_alpha1),
                lcha_alpha2=float(self.lcha_alpha2),
                lcha_v_alpha0=float(self.lcha_v_alpha0),
                lcha_v_alpha1=float(self.lcha_v_alpha1),
                lcha_v_alpha2=float(self.lcha_v_alpha2),
                family_meta=family_meta,
                training=self.training,
            )  # expected [B,T,H,D]

            if out_thd.shape != (bsz, q_len, self.num_heads, self.head_dim):
                raise ValueError(
                    f"stagee_apply_fn output shape mismatch: got {tuple(out_thd.shape)}, "
                    f"expected {(bsz, q_len, self.num_heads, self.head_dim)}"
                )

            attn_meta = {
                "used_sparse_stagee": True,
                "used_kv_cache_fallback": False,
                "past_len": past_len,
                "q_len": q_len,
                "total_len": total_len,
                "cand_i32": cand_i32,
                "mask_u8": mask_u8,
                "cand_is_per_head": cand_is_per_head,
                "family_meta": family_meta,
              }
        else:
            out_thd = self._dense_fallback(q, k_attn, v_attn, attention_mask=attention_mask)

            attn_meta = {
                "used_sparse_stagee": False,
                "used_kv_cache_fallback": bool(past_len > 0),
                "past_len": past_len,
                "q_len": q_len,
                "total_len": total_len,
            }

        out = out_thd.reshape(bsz, q_len, self.num_heads * self.head_dim)
        out = self.o_proj(out)
        out = self.resid_dropout(out)

        if not output_attn_meta:
            attn_meta = None
        return out, present, attn_meta

In [ ]:
class GCCDDecoderLayer(nn.Module):
    def __init__(
        self,
        cfg: GCCDBackboneConfig,
        layer_idx: int,
        candidate_builder_fn: Optional[Callable[..., Dict[str, Any]]] = None,
        stagee_apply_fn: Optional[Callable[..., torch.Tensor]] = None,
    ):
        super().__init__()
        self.self_attn = GCCDSparseDecoderAttention(
            cfg=cfg,
            layer_idx=layer_idx,
            candidate_builder_fn=candidate_builder_fn,
            stagee_apply_fn=stagee_apply_fn,
        )
        self.input_layernorm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.mlp = GCCDMLP(cfg)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
        output_attn_meta: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Dict[str, Any]]]:

        if hidden_states.dim() != 3:
            raise ValueError(f"hidden_states must be [B,T,H], got {tuple(hidden_states.shape)}")

        if past_key_value is not None:
            if not (isinstance(past_key_value, tuple) and len(past_key_value) == 2):
                raise ValueError("past_key_value must be None or a tuple (past_k, past_v)")

        residual = hidden_states
        x = self.input_layernorm(hidden_states)
        attn_out, present, attn_meta = self.self_attn(
            x,
            attention_mask=attention_mask,
            past_key_value=past_key_value,
            use_cache=use_cache,
            output_attn_meta=output_attn_meta,
        )

        if attn_out.shape != residual.shape:
            raise RuntimeError(f"attn_out shape {tuple(attn_out.shape)} != residual shape {tuple(residual.shape)}")

        hidden_states = residual + attn_out

        residual = hidden_states
        x = self.post_attention_layernorm(hidden_states)
        mlp_out = self.mlp(x)

        if mlp_out.shape != residual.shape:
            raise RuntimeError(f"mlp_out shape {tuple(mlp_out.shape)} != residual shape {tuple(residual.shape)}")

        hidden_states = residual + mlp_out

        return hidden_states, present, attn_meta

In [ ]:
class GCCDDecoderModel(nn.Module):
    def __init__(
        self,
        cfg: GCCDBackboneConfig,
        candidate_builder_fn: Optional[Callable[..., Dict[str, Any]]] = None,
        stagee_apply_fn: Optional[Callable[..., torch.Tensor]] = None,
    ):
        super().__init__()
        self.cfg = cfg
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden_size, padding_idx=cfg.pad_token_id)

        self.layers = nn.ModuleList([
            GCCDDecoderLayer(
                cfg=cfg,
                layer_idx=i,
                candidate_builder_fn=candidate_builder_fn,
                stagee_apply_fn=stagee_apply_fn,
            )
            for i in range(cfg.num_hidden_layers)
        ])
        self.norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
        use_cache: bool = False,
        output_hidden_states: bool = False,
        output_attn_meta: bool = False,
    ) -> Dict[str, Any]:
        if (input_ids is None) == (inputs_embeds is None):
            raise ValueError("Provide exactly one of input_ids or inputs_embeds.")

        if input_ids is not None:
            if input_ids.dim() != 2:
                raise ValueError(f"input_ids must be [B,T], got {tuple(input_ids.shape)}")
            if input_ids.dtype not in (torch.long, torch.int64):
                raise TypeError(f"input_ids must be torch.long/int64, got {input_ids.dtype}")

        if inputs_embeds is not None:
            if inputs_embeds.dim() != 3:
                raise ValueError(f"inputs_embeds must be [B,T,H], got {tuple(inputs_embeds.shape)}")
            if inputs_embeds.size(-1) != self.cfg.hidden_size:
                raise ValueError(
                    f"inputs_embeds last dim must be hidden_size={self.cfg.hidden_size}, "
                    f"got {inputs_embeds.size(-1)}"
                )

        if inputs_embeds is None:
            hidden_states = self.embed_tokens(input_ids)
        else:
            hidden_states = inputs_embeds

        if attention_mask is not None:
            if attention_mask.size(0) != hidden_states.size(0):
                raise ValueError("attention_mask batch size must match hidden_states batch size")

        all_hidden_states = [] if output_hidden_states else None
        all_attn_meta = [] if output_attn_meta else None
        next_past_key_values = [] if use_cache else None

        if past_key_values is None:
            past_key_values = [None] * len(self.layers)
        else:
            past_key_values = list(past_key_values)
            if len(past_key_values) != len(self.layers):
                raise ValueError(
                    f"past_key_values length must equal num layers "
                    f"({len(self.layers)}), got {len(past_key_values)}"
                )

        # If your notebook uses additive attention masks for dense fallback,
        # pass them through here. Sparse Stage-E path can ignore or use them as needed.
        for i, layer in enumerate(self.layers):
            if output_hidden_states:
                all_hidden_states.append(hidden_states)

            hidden_states, present, attn_meta = layer(
                hidden_states,
                attention_mask=attention_mask,
                past_key_value=past_key_values[i],
                use_cache=use_cache,
                output_attn_meta=output_attn_meta,
            )

            if use_cache:
                next_past_key_values.append(present)
            if output_attn_meta:
                all_attn_meta.append(attn_meta)

        hidden_states = self.norm(hidden_states)

        if output_hidden_states:
            all_hidden_states.append(hidden_states)

        return {
            "last_hidden_state": hidden_states,
            "past_key_values": tuple(next_past_key_values) if use_cache else None,
            "hidden_states": tuple(all_hidden_states) if output_hidden_states else None,
            "attn_meta": tuple(all_attn_meta) if output_attn_meta else None,
        }

In [ ]:
class GCCDCausalLM(nn.Module):
    def __init__(
        self,
        cfg: GCCDBackboneConfig,
        candidate_builder_fn: Optional[Callable[..., Dict[str, Any]]] = None,
        stagee_apply_fn: Optional[Callable[..., torch.Tensor]] = None,
    ):
        super().__init__()
        self.cfg = cfg
        self.model = GCCDDecoderModel(
            cfg=cfg,
            candidate_builder_fn=candidate_builder_fn,
            stagee_apply_fn=stagee_apply_fn,
        )
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def get_input_embeddings(self) -> nn.Embedding:
        return self.model.embed_tokens

    def set_input_embeddings(self, emb: nn.Embedding) -> None:
        if not isinstance(emb, nn.Embedding):
            raise TypeError("emb must be an nn.Embedding")
        if emb.embedding_dim != self.cfg.hidden_size:
            raise ValueError(
                f"embedding_dim must equal hidden_size={self.cfg.hidden_size}, "
                f"got {emb.embedding_dim}"
            )
        self.model.embed_tokens = emb

    def tie_weights(self) -> None:
        self.lm_head.weight = self.model.embed_tokens.weight

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
        use_cache: bool = False,
        output_hidden_states: bool = False,
        output_attn_meta: bool = False,
    ) -> GCCDModelOutput:
        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_hidden_states=output_hidden_states,
            output_attn_meta=output_attn_meta,
        )

        last_hidden = out["last_hidden_state"]
        if last_hidden.size(-1) != self.cfg.hidden_size:
            raise RuntimeError(
                f"last_hidden_state last dim must be hidden_size={self.cfg.hidden_size}, "
                f"got {last_hidden.size(-1)}"
            )

        logits = self.lm_head(last_hidden)

        loss = None
        if labels is not None:
            if labels.shape != logits.shape[:2]:
                raise ValueError(
                    f"labels must have shape {tuple(logits.shape[:2])}, "
                    f"got {tuple(labels.shape)}"
                )

            labels_for_loss = labels.to(device=logits.device, dtype=torch.long)

            # Ignore padded positions if attention_mask is provided.
            if attention_mask is not None:
                labels_for_loss = labels_for_loss.masked_fill(
                    attention_mask.to(device=logits.device) == 0,
                    -100,
                )

            # Standard causal LM next-token loss:
            # token t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels_for_loss[:, 1:].contiguous()

            if shift_logits.numel() == 0:
                raise ValueError("Cannot compute causal LM loss for sequence length < 2.")

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return GCCDModelOutput(
            loss=loss,
            logits=logits,
            last_hidden_state=last_hidden,
            hidden_states=out["hidden_states"],
            past_key_values=out["past_key_values"],
            attn_meta=out["attn_meta"],
        )

In [ ]:
def _gccd_protocol_id_from_name(name: str) -> int:
    name = str(name).upper()
    if name == "ECONOMY":
        return 0
    if name == "HYBRID":
        return 1
    if name == "MULTISCALE":
        return 2
    if name == "GLOBAL_REFRESH":
        return 3
    raise ValueError(f"Unknown protocol name: {name}")


def _gccd_make_stagea_attention_mask_bn(
    *,
    q: torch.Tensor,
    k: torch.Tensor,
    attention_mask: Optional[torch.Tensor],
    total_len: int,
) -> torch.Tensor:
    if q.dim() != 4:
        raise ValueError(f"q must be [B,T,H,D], got {tuple(q.shape)}")
    if k.dim() != 4:
        raise ValueError(f"k must be [B,S,H,D], got {tuple(k.shape)}")

    bsz = int(q.size(0))
    key_len = int(k.size(1))
    total_len = int(total_len)

    if key_len != total_len:
        raise ValueError(f"k length {key_len} must equal total_len {total_len}")

    if attention_mask is None:
        return torch.ones((bsz, total_len), device=q.device, dtype=torch.uint8)

    mask = attention_mask.to(device=q.device)

    if mask.dim() != 2:
        raise ValueError(
            "Stage-A adapter currently expects attention_mask as [B,S]. "
            f"Got shape {tuple(mask.shape)}."
        )

    if mask.shape[0] != bsz:
        raise ValueError(f"attention_mask batch mismatch: got {mask.shape[0]}, expected {bsz}")

    if mask.shape[1] != total_len:
        raise ValueError(
            f"attention_mask length mismatch: got {mask.shape[1]}, expected total_len={total_len}"
        )

    if mask.dtype == torch.bool:
        return mask.to(dtype=torch.uint8).contiguous()

    # HF-style mask: 1 = keep, 0 = pad.
    return (mask > 0).to(dtype=torch.uint8).contiguous()


def _gccd_validate_stagea_dict(out: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(out, dict):
        raise TypeError("Stage-A candidate builder must return a dict")

    required = ("cand_i32", "mask_u8", "cand_is_per_head")
    missing = [k for k in required if k not in out]
    if missing:
        raise KeyError(f"Stage-A candidate builder missing required keys: {missing}")

    if not torch.is_tensor(out["cand_i32"]):
        raise TypeError("cand_i32 must be a torch.Tensor")
    if not torch.is_tensor(out["mask_u8"]):
        raise TypeError("mask_u8 must be a torch.Tensor")

    if out["cand_i32"].dtype != torch.int32:
        raise TypeError(f"cand_i32 must be torch.int32, got {out['cand_i32'].dtype}")
    if out["mask_u8"].dtype != torch.uint8:
        raise TypeError(f"mask_u8 must be torch.uint8, got {out['mask_u8'].dtype}")

    if out["cand_i32"].shape != out["mask_u8"].shape:
        raise ValueError(
            f"cand_i32 shape {tuple(out['cand_i32'].shape)} must match "
            f"mask_u8 shape {tuple(out['mask_u8'].shape)}"
        )

    if not isinstance(out["cand_is_per_head"], bool):
        raise TypeError("cand_is_per_head must be bool")

    family_meta = out.get("family_meta", None)
    if family_meta is not None:
        if not isinstance(family_meta, dict):
            raise TypeError("family_meta must be a dict when provided")

        required_meta = (
            "family_bounds_i32",
            "family_order_caps_i32",
            "dispatch_meta_i32",
        )
        missing_meta = [k for k in required_meta if k not in family_meta]
        if missing_meta:
            raise KeyError(f"family_meta missing required keys: {missing_meta}")

        for k in required_meta:
            if not torch.is_tensor(family_meta[k]):
                raise TypeError(f"family_meta['{k}'] must be a torch.Tensor")
            if family_meta[k].dtype != torch.int32:
                raise TypeError(f"family_meta['{k}'] must be torch.int32, got {family_meta[k].dtype}")

    return out

def make_stagea_adapter(existing_candidate_builder: Callable[..., Any]) -> Callable[..., Dict[str, Any]]:
    """
    Wrap the family-aware Stage-A CUDA builder.

    Converts raw output from:

      build_candidates_family_cuda(...)

    into the dict expected by GCCDSparseDecoderAttention:

      {
        "cand_i32": ...,
        "mask_u8": ...,
        "cand_is_per_head": False,
        "family_meta": {
            "valid_T": ...,
            "family_u8": ...,
            "family_bounds_i32": ...,
            "family_order_caps_i32": ...,
            "dispatch_meta_i32": ...,
        }
      }

    Supports:
      - full-sequence training/eval: past_len=0, q_len=total_len
      - KV-cache decode/chunking: past_len>0, q_len<total_len

    For KV-cache, the raw Stage-A CUDA builder still builds candidates for
    the full [B,total_len] axis;

    Wrap the family-aware Stage-A CUDA builder.

    For full-sequence training/eval:
      q_len == total_len, past_len == 0.

    For KV-cache generation:
      q_len may be 1 while total_len = past_len + q_len.

    Important:
      For sparse KV-cache, we keep the FULL candidate table over [B,total_len],
      because LCHA second-order family logic may need candidate rows for neighbor tokens.
      Stage-E receives query_start=past_len and selects the correct absolute query rows.
    """
    if not callable(existing_candidate_builder):
        raise TypeError("existing_candidate_builder must be callable")

    def _adapter(**kwargs) -> Dict[str, Any]:
        q = kwargs["q"]
        k = kwargs["k"]
        attn_module = kwargs["attn_module"]
        layer_idx = int(kwargs.get("layer_idx", getattr(attn_module, "layer_idx", 0)))
        past_len = int(kwargs.get("past_len", 0))
        total_len = int(kwargs["total_len"])
        attention_mask = kwargs.get("attention_mask", None)

        if q.dim() != 4 or k.dim() != 4:
            raise ValueError(f"q/k must be 4D [B,T,H,D]. Got q={tuple(q.shape)}, k={tuple(k.shape)}")

        bsz = int(q.size(0))
        q_len = int(q.size(1))
        key_len = int(k.size(1))

        if key_len != total_len:
            raise ValueError(f"k length {key_len} must equal total_len {total_len}")

        if past_len < 0:
            raise ValueError(f"past_len must be non-negative, got {past_len}")

        if q_len <= 0:
            raise ValueError(f"q_len must be positive, got {q_len}")

        if total_len <= 0:
            raise ValueError(f"total_len must be positive, got {total_len}")

        if past_len + q_len != total_len:
            raise ValueError(
                f"past_len + q_len must equal total_len. "
                f"Got past_len={past_len}, q_len={q_len}, total_len={total_len}."
            )

        attn_mask_bn = _gccd_make_stagea_attention_mask_bn(
            q=q,
            k=k,
            attention_mask=attention_mask,
            total_len=total_len,
        )

        protocol_name = getattr(attn_module, "_router_protocol_name", None)
        if protocol_name is None:
            protocol_name = getattr(attn_module, "_lasperht_protocol_name", "HYBRID")

        protocol_id = _gccd_protocol_id_from_name(protocol_name)

        cfg = getattr(attn_module, "cfg", None)
        num_layers = int(getattr(cfg, "num_hidden_layers", 0)) if cfg is not None else 0

        raw = existing_candidate_builder(
            attn_mask_bn=attn_mask_bn,
            w=int(getattr(attn_module, "w", 32)),
            ring_inner=int(getattr(attn_module, "ring_inner", 64)),
            ring_outer=int(getattr(attn_module, "ring_outer", 128)),
            mode=int(getattr(attn_module, "scaffold_mode", 2)),
            num_anchors=int(getattr(attn_module, "num_anchors", 4)),
            include_self=bool(getattr(attn_module, "include_self", True)),
            anchor_strategy=int(getattr(attn_module, "anchor_strategy", 1)),
            anchor_param0=int(getattr(attn_module, "anchor_param0", 0)),
            protocol_id=int(protocol_id),
            layer_idx=int(layer_idx),
            num_layers=int(num_layers),
            refresh_period=int(getattr(attn_module, "refresh_period", 0)),
        )

        if isinstance(raw, dict):
            out = dict(raw)
            family_meta = out.get("family_meta", None)
            if family_meta is None:
                family_meta = {}
            else:
                family_meta = dict(family_meta)

            family_meta["kv_cache_meta"] = {
                "past_len": past_len,
                "q_len": q_len,
                "total_len": total_len,
                "batch_size": bsz,
            }
            out["family_meta"] = family_meta

            return _gccd_validate_stagea_dict(out)

        if not isinstance(raw, (tuple, list)):
            raise TypeError(
                "Stage-A builder must return either a dict or the raw CUDA tuple/list "
                "(cand_i32, mask_u8, valid_T, family_u8, family_bounds_i32, "
                "family_order_caps_i32, dispatch_meta_i32)."
            )

        if len(raw) != 7:
            raise ValueError(f"Raw Stage-A output must have length 7, got {len(raw)}")

        cand_i32, mask_u8, valid_T, family_u8, family_bounds_i32, family_order_caps_i32, dispatch_meta_i32 = raw

        out = {
            "cand_i32": cand_i32.contiguous(),
            "mask_u8": mask_u8.contiguous(),
            "cand_is_per_head": False,
            "family_meta": {
                "valid_T": valid_T.contiguous(),
                "family_u8": family_u8.contiguous(),
                "family_bounds_i32": family_bounds_i32.contiguous(),
                "family_order_caps_i32": family_order_caps_i32.contiguous(),
                "dispatch_meta_i32": dispatch_meta_i32.contiguous(),
                "kv_cache_meta": {
                    "past_len": past_len,
                    "q_len": q_len,
                    "total_len": total_len,
                    "batch_size": bsz,
                },
            },
        }

        return _gccd_validate_stagea_dict(out)

    return _adapter


def make_stagee_adapter(existing_stagee_apply: Callable[..., torch.Tensor]) -> Callable[..., torch.Tensor]:
    """
    Wrap the family-aware Stage-E/LCHA apply path.

    Converts the GCCDSparseDecoderAttention call:

      q, k, v, cand_i32, mask_u8, family_meta

    into:

      stagee_family_lcha_apply(
          Q, K, V,
          cand_i32,
          mask_u8,
          family_bounds_i32,
          family_order_caps_i32,
          dispatch_meta_i32,
          cand_is_per_head,
          ...
      )

    Wrap the family-aware Stage-E/LCHA apply path.

    Supports full sequence and sparse KV-cache generation by passing
    query_start=past_len into stagee_family_lcha_apply(...).
    """
    if not callable(existing_stagee_apply):
        raise TypeError("existing_stagee_apply must be callable")

    def _adapter(**kwargs) -> torch.Tensor:
        q = kwargs["q"]
        k = kwargs["k"]
        v = kwargs["v"]
        cand_i32 = kwargs["cand_i32"]
        mask_u8 = kwargs["mask_u8"]
        cand_is_per_head = bool(kwargs.get("cand_is_per_head", False))
        family_meta = kwargs.get("family_meta", None)

        if family_meta is None:
            raise ValueError("family_meta is required for family-aware Stage-E/LCHA apply")
        if not isinstance(family_meta, dict):
            raise TypeError("family_meta must be a dict")

        required_meta = (
            "family_bounds_i32",
            "family_order_caps_i32",
            "dispatch_meta_i32",
        )
        missing_meta = [k_meta for k_meta in required_meta if k_meta not in family_meta]
        if missing_meta:
            raise KeyError(f"family_meta missing required keys: {missing_meta}")

        kv_cache_meta = family_meta.get("kv_cache_meta", {})
        if kv_cache_meta is None:
            kv_cache_meta = {}
        if not isinstance(kv_cache_meta, dict):
            raise TypeError("family_meta['kv_cache_meta'] must be a dict when provided")

        query_start = int(kv_cache_meta.get("past_len", 0))

        family_bounds_i32 = family_meta["family_bounds_i32"].to(device=q.device, dtype=torch.int32).contiguous()
        family_order_caps_i32 = family_meta["family_order_caps_i32"].to(device=q.device, dtype=torch.int32).contiguous()
        dispatch_meta_i32 = family_meta["dispatch_meta_i32"].to(device=q.device, dtype=torch.int32).contiguous()

        out = existing_stagee_apply(
            Q=q,
            K=k,
            V=v,
            cand_i32=cand_i32.to(device=q.device, dtype=torch.int32).contiguous(),
            mask_u8=mask_u8.to(device=q.device, dtype=torch.uint8).contiguous(),
            family_bounds_i32=family_bounds_i32,
            family_order_caps_i32=family_order_caps_i32,
            dispatch_meta_i32=dispatch_meta_i32,
            cand_is_per_head=cand_is_per_head,
            lcha_enabled=bool(kwargs.get("lcha_enabled", True)),
            lcha_heads_frac=float(kwargs.get("lcha_heads_frac", 1.0)),
            use_v_lcha=bool(kwargs.get("lcha_use_v", False)),
            trap_on_bad_cand=False,
            query_start=query_start,
        )

        if not torch.is_tensor(out):
            raise TypeError("Stage-E apply must return a torch.Tensor")

        expected_shape = q.shape
        if out.shape != expected_shape:
            raise ValueError(
                f"Stage-E output shape mismatch: got {tuple(out.shape)}, expected {tuple(expected_shape)}"
            )

        return out

    return _adapter

In [ ]:
from typing import Optional, Callable, Dict, Any

import torch
import torch.nn as nn


def build_gccd_backbone_from_existing(
    base_model: Optional[nn.Module],
    cfg: GCCDBackboneConfig,
    candidate_builder_fn: Optional[Callable[..., Dict[str, Any]]] = None,
    stagee_apply_fn: Optional[Callable[..., torch.Tensor]] = None,
    copy_token_embeddings: bool = True,
    copy_lm_head: bool = True,
) -> GCCDCausalLM:
    model = GCCDCausalLM(
        cfg=cfg,
        candidate_builder_fn=candidate_builder_fn,
        stagee_apply_fn=stagee_apply_fn,
    )

    if base_model is not None:
        # Copy token embeddings if available and shape-compatible
        if copy_token_embeddings:
            src_emb = None

            if hasattr(base_model, "get_input_embeddings"):
                src_emb = base_model.get_input_embeddings()
            elif hasattr(base_model, "model") and hasattr(base_model.model, "embed_tokens"):
                src_emb = base_model.model.embed_tokens

            if src_emb is not None and hasattr(src_emb, "weight"):
                src = src_emb.weight
                dst = model.model.embed_tokens.weight
                if src.shape == dst.shape:
                    with torch.no_grad():
                        dst.copy_(src.detach().to(device=dst.device, dtype=dst.dtype))

        # Copy LM head if available and shape-compatible
        if copy_lm_head:
            src_head = None

            if hasattr(base_model, "get_output_embeddings"):
                src_head = base_model.get_output_embeddings()
            elif hasattr(base_model, "lm_head"):
                src_head = base_model.lm_head

            if src_head is not None and hasattr(src_head, "weight"):
                src = src_head.weight
                dst = model.lm_head.weight
                if src.shape == dst.shape:
                    with torch.no_grad():
                        dst.copy_(src.detach().to(device=dst.device, dtype=dst.dtype))

    return model

In [ ]:
# ============================================================
# CELL 41.5 — Instantiate actual GCCD CausalLM model
# Put immediately AFTER build_gccd_backbone_from_existing(...)
# ============================================================

# Existing checks first
if "tokenizer" not in globals():
    raise NameError("tokenizer is not defined. Run the tokenizer cell first.")

if "GCCDBackboneConfig" not in globals():
    raise NameError("GCCDBackboneConfig is not defined. Run the GCCDBackboneConfig cell first.")

if "build_gccd_backbone_from_existing" not in globals():
    raise NameError("build_gccd_backbone_from_existing is not defined. Run the builder cell first.")

# ADD ADAPTER INSTANTIATION HERE
stagea_adapter = make_stagea_adapter(build_candidates_family_cuda)
stagee_adapter = make_stagee_adapter(stagee_family_lcha_apply)

# Then config
gccd_backbone_cfg = GCCDBackboneConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    intermediate_size=2048,
    num_hidden_layers=4,
    num_attention_heads=8,
    num_key_value_heads=8,
    max_position_embeddings=max(int(SEQ_LEN), 2048),
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0,
    bos_token_id=tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 1,
    eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 2,
    use_sparse_stagee=True,
    use_lcha=True,
    lcha_enabled=True,
    lcha_order=1,
    lcha_heads_frac=0.25,
    lcha_use_v=False,
)

# Then model
model = build_gccd_backbone_from_existing(
    base_model=None,
    cfg=gccd_backbone_cfg,
    candidate_builder_fn=stagea_adapter,
    stagee_apply_fn=stagee_adapter,
).to(device)

model.tie_weights()
model.train()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Instantiated global model:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Vocab size:", gccd_backbone_cfg.vocab_size)
print("Hidden size:", gccd_backbone_cfg.hidden_size)
print("Layers:", gccd_backbone_cfg.num_hidden_layers)
print("Params:", f"{n_params:,}")
print("Trainable params:", f"{n_trainable:,}")

In [ ]:
# ============================================================
# Runtime validation:
#   1. Sparse full-sequence forward/backward
#   2. Sparse KV-cache no_grad generation
# ============================================================

def _get_field(obj, name, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def _flatten_attn_meta(attn_meta):
    if attn_meta is None:
        return []

    flat = []

    def _rec(x):
        if x is None:
            return
        if isinstance(x, dict):
            flat.append(x)
            return
        if isinstance(x, (list, tuple)):
            for y in x:
                _rec(y)

    _rec(attn_meta)
    return flat


def _iter_gccd_sparse_attention_layers(model):
    for name, module in model.named_modules():
        if module.__class__.__name__ == "GCCDSparseDecoderAttention":
            yield name, module


def _assert_sparse_layers_wired(model):
    layers = list(_iter_gccd_sparse_attention_layers(model))

    if len(layers) == 0:
        raise RuntimeError("No GCCDSparseDecoderAttention layers found.")

    for name, layer in layers:
        if not bool(getattr(layer.cfg, "use_sparse_stagee", False)):
            raise RuntimeError(f"{name}: cfg.use_sparse_stagee=False")

        if getattr(layer, "candidate_builder_fn", None) is None:
            raise RuntimeError(f"{name}: candidate_builder_fn is None")

        if getattr(layer, "stagee_apply_fn", None) is None:
            raise RuntimeError(f"{name}: stagee_apply_fn is None")

    return layers


def _assert_attn_meta_used_sparse(outputs, *, context: str, expect_kv_cache: bool = False):
    attn_meta = _get_field(outputs, "attn_meta", None)
    metas = _flatten_attn_meta(attn_meta)

    if len(metas) == 0:
        raise RuntimeError(f"{context}: output_attn_meta=True but no attention metadata was returned.")

    sparse_flags = [bool(m.get("used_sparse_stagee", False)) for m in metas]
    if not any(sparse_flags):
        raise RuntimeError(f"{context}: no layer reported used_sparse_stagee=True.")

    fallback_flags = [bool(m.get("used_kv_cache_fallback", False)) for m in metas]
    if any(fallback_flags):
        raise RuntimeError(
            f"{context}: at least one layer used dense KV-cache fallback. "
            "Expected sparse Stage-E path."
        )

    if expect_kv_cache:
        kv_metas = [m for m in metas if int(m.get("past_len", 0)) > 0]
        if len(kv_metas) == 0:
            raise RuntimeError(f"{context}: expected cached decode metadata with past_len > 0.")

        for m in kv_metas:
            q_len = int(m.get("q_len", -1))
            total_len = int(m.get("total_len", -1))
            past_len = int(m.get("past_len", -1))

            if past_len <= 0:
                raise RuntimeError(f"{context}: invalid cached past_len={past_len}")
            if q_len <= 0:
                raise RuntimeError(f"{context}: invalid cached q_len={q_len}")
            if total_len != past_len + q_len:
                raise RuntimeError(
                    f"{context}: total_len mismatch: total_len={total_len}, "
                    f"past_len={past_len}, q_len={q_len}"
                )


def validate_sparse_full_sequence_forward_backward(
    model,
    train_loader,
    device,
    max_tokens: int = 64,
):
    layers = _assert_sparse_layers_wired(model)

    batch = next(iter(train_loader))

    input_ids = batch["input_ids"][:1, :max_tokens].to(device).contiguous()
    attention_mask = batch["attention_mask"][:1, :max_tokens].to(device).contiguous()
    labels = batch["labels"][:1, :max_tokens].to(device).contiguous()

    if input_ids.size(1) < 2:
        raise RuntimeError("Need sequence length >= 2 for causal LM loss.")

    model.train()
    model.zero_grad(set_to_none=True)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        use_cache=False,
        output_attn_meta=True,
    )

    loss = _get_field(outputs, "loss", None)
    if loss is None:
        raise RuntimeError("Full-sequence validation: model returned loss=None.")

    if not torch.isfinite(loss.detach()):
        raise RuntimeError(f"Full-sequence validation: non-finite loss: {loss.item()}")

    _assert_attn_meta_used_sparse(
        outputs,
        context="Full-sequence validation",
        expect_kv_cache=False,
    )

    loss.backward()

    bad_grads = []
    grad_tensors = 0

    for name, p in model.named_parameters():
        if p.grad is None:
            continue

        grad_tensors += 1

        if not torch.isfinite(p.grad.detach()).all():
            bad_grads.append(name)

    if bad_grads:
        raise RuntimeError(f"Full-sequence validation: non-finite gradients in {bad_grads[:10]}")

    if grad_tensors == 0:
        raise RuntimeError("Full-sequence validation: no gradients were produced.")

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("PASS: sparse full-sequence forward/backward")
    print("loss:", float(loss.detach().cpu()))
    print("validated sparse attention layers:", len(layers))
    print("gradient tensors checked:", grad_tensors)


@torch.no_grad()
def validate_sparse_kv_cache_generation(
    model,
    train_loader,
    device,
    prompt_tokens: int = 16,
):
    layers = _assert_sparse_layers_wired(model)

    batch = next(iter(train_loader))

    input_ids_full = batch["input_ids"][:1].to(device).contiguous()
    attention_mask_full = batch["attention_mask"][:1].to(device).contiguous()

    max_len = int(input_ids_full.size(1))
    prompt_tokens = int(min(prompt_tokens, max_len - 1))

    if prompt_tokens < 2:
        raise RuntimeError(
            f"Need at least 3 tokens for KV-cache validation, got sequence length {max_len}."
        )

    prompt_ids = input_ids_full[:, :prompt_tokens].contiguous()
    prompt_mask = attention_mask_full[:, :prompt_tokens].contiguous()

    next_ids = input_ids_full[:, prompt_tokens:prompt_tokens + 1].contiguous()
    next_mask = attention_mask_full[:, :prompt_tokens + 1].contiguous()

    model.eval()

    # First pass: build cache from prompt.
    out_prompt = model(
        input_ids=prompt_ids,
        attention_mask=prompt_mask,
        use_cache=True,
        output_attn_meta=True,
    )

    _assert_attn_meta_used_sparse(
        out_prompt,
        context="KV-cache prompt pass",
        expect_kv_cache=False,
    )

    past_key_values = _get_field(out_prompt, "past_key_values", None)
    if past_key_values is None:
        raise RuntimeError("KV-cache validation: prompt forward returned past_key_values=None.")

    if len(past_key_values) != model.cfg.num_hidden_layers:
        raise RuntimeError(
            f"KV-cache validation: expected {model.cfg.num_hidden_layers} cache layers, "
            f"got {len(past_key_values)}."
        )

    # Second pass: one-token cached generation.
    out_next = model(
        input_ids=next_ids,
        attention_mask=next_mask,
        past_key_values=past_key_values,
        use_cache=True,
        output_attn_meta=True,
    )

    _assert_attn_meta_used_sparse(
        out_next,
        context="KV-cache next-token pass",
        expect_kv_cache=True,
    )

    logits = _get_field(out_next, "logits", None)
    if logits is None:
        raise RuntimeError("KV-cache validation: next-token output has logits=None.")

    if tuple(logits.shape[:2]) != (1, 1):
        raise RuntimeError(
            f"KV-cache validation: expected logits shape [1,1,V], got {tuple(logits.shape)}"
        )

    next_past_key_values = _get_field(out_next, "past_key_values", None)
    if next_past_key_values is None:
        raise RuntimeError("KV-cache validation: next-token forward returned past_key_values=None.")

    expected_cache_len = prompt_tokens + 1

    for li, (k_cache, v_cache) in enumerate(next_past_key_values):
        if k_cache.shape != v_cache.shape:
            raise RuntimeError(
                f"KV-cache validation: layer {li} K/V cache shape mismatch: "
                f"{tuple(k_cache.shape)} vs {tuple(v_cache.shape)}"
            )

        if int(k_cache.shape[1]) != expected_cache_len:
            raise RuntimeError(
                f"KV-cache validation: layer {li} expected cache length {expected_cache_len}, "
                f"got {int(k_cache.shape[1])}."
            )

        if not torch.isfinite(k_cache.detach().float()).all():
            raise RuntimeError(f"KV-cache validation: layer {li} K cache has NaN/Inf.")
        if not torch.isfinite(v_cache.detach().float()).all():
            raise RuntimeError(f"KV-cache validation: layer {li} V cache has NaN/Inf.")

    if not torch.isfinite(logits.detach().float()).all():
        raise RuntimeError("KV-cache validation: next-token logits contain NaN/Inf.")

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("PASS: sparse KV-cache no_grad generation")
    print("prompt length:", prompt_tokens)
    print("next logits shape:", tuple(logits.shape))
    print("validated sparse attention layers:", len(layers))
    print("cache layers checked:", len(next_past_key_values))


validate_sparse_full_sequence_forward_backward(
    model=model,
    train_loader=train_loader,
    device=device,
    max_tokens=min(64, int(SEQ_LEN)),
)

validate_sparse_kv_cache_generation(
    model=model,
    train_loader=train_loader,
    device=device,
    prompt_tokens=min(16, int(SEQ_LEN) - 1),
)

In [ ]:
# ============================================================
# Sparse CUDA forward/backward validation
# Place immediately AFTER Cell 37 model instantiation
# ============================================================

def _iter_gccd_sparse_attention_layers(model):
    for name, module in model.named_modules():
        if module.__class__.__name__ == "GCCDSparseDecoderAttention":
            yield name, module


def validate_sparse_cuda_forward_backward(
    model,
    train_loader,
    device,
    max_tokens: int = 64,
):
    layers = list(_iter_gccd_sparse_attention_layers(model))
    if len(layers) == 0:
        raise RuntimeError("No GCCDSparseDecoderAttention layers found.")

    for name, layer in layers:
        if not bool(getattr(layer.cfg, "use_sparse_stagee", False)):
            raise RuntimeError(f"{name}: use_sparse_stagee=False")

        if getattr(layer, "candidate_builder_fn", None) is None:
            raise RuntimeError(f"{name}: candidate_builder_fn is None")

        if getattr(layer, "stagee_apply_fn", None) is None:
            raise RuntimeError(f"{name}: stagee_apply_fn is None")

    batch = next(iter(train_loader))

    input_ids = batch["input_ids"][:1, :max_tokens].to(device)
    attention_mask = batch["attention_mask"][:1, :max_tokens].to(device)
    labels = batch["labels"][:1, :max_tokens].to(device)

    if input_ids.size(1) < 2:
        raise RuntimeError("Need sequence length >= 2 for causal LM loss.")

    model.train()
    model.zero_grad(set_to_none=True)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        output_attn_meta=True,
    )

    loss = outputs.loss
    if loss is None:
        raise RuntimeError("Model returned loss=None.")

    if not torch.isfinite(loss.detach()):
        raise RuntimeError(f"Loss is non-finite: {loss.item()}")

    loss.backward()

    bad_grads = []
    checked = 0

    for name, p in model.named_parameters():
        if p.grad is not None:
            checked += 1
            if not torch.isfinite(p.grad.detach()).all():
                bad_grads.append(name)

    if bad_grads:
        raise RuntimeError(f"Non-finite gradients found in: {bad_grads[:10]}")

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("Sparse CUDA forward/backward validation PASSED")
    print("loss:", float(loss.detach().cpu()))
    print("checked gradient tensors:", checked)
    print("validated sparse attention layers:", len(layers))


validate_sparse_cuda_forward_backward(
    model=model,
    train_loader=train_loader,
    device=device,
    max_tokens=min(64, int(SEQ_LEN)),
)

In [ ]:
# ============================================================
# SASA per-forward candidate cache (shared across all layers)
# ============================================================
SASA_FORWARD_CAND_CACHE = None


def get_sasa_forward_cand_cache():
    """
    Returns the active per-forward candidate cache, or None if no cache context is active.
    Candidate-builder code should use this instead of creating persistent globals.
    """
    return SASA_FORWARD_CAND_CACHE


class SASACandCacheContext:
    """
    Usage:
      with SASACandCacheContext():
          out = model(...)
          loss = out.loss
          loss.backward()

    Keeps a dict alive across forward + backward.
    Important: backward must happen inside the context if checkpoint recompute needs the cache.
    """
    def __init__(self):
        self.cache = {}
        self._prev_cache = None
        self._entered = False

    def __enter__(self):
        global SASA_FORWARD_CAND_CACHE

        if self._entered:
            raise RuntimeError("SASACandCacheContext cannot be re-entered before exit.")

        self._prev_cache = SASA_FORWARD_CAND_CACHE
        SASA_FORWARD_CAND_CACHE = self.cache
        self._entered = True
        return self.cache

    def __exit__(self, exc_type, exc, tb):
        global SASA_FORWARD_CAND_CACHE

        SASA_FORWARD_CAND_CACHE = self._prev_cache
        self._prev_cache = None
        self._entered = False
        return False

In [ ]:
# ============================================================
# LCHA helpers (NEW CELL)
# Place after: class StageEMHFusedFn
# ============================================================
import math
import torch


def _lcha_checked_float(name: str, value) -> float:
    v = float(value)
    if not math.isfinite(v):
        raise ValueError(f"{name} contains NaN/Inf")
    return v


def _lcha_num_heads_to_use(num_heads, heads_frac):
    num_heads = int(num_heads)
    heads_frac = _lcha_checked_float("heads_frac", heads_frac)

    if num_heads <= 0:
        raise ValueError("num_heads must be > 0")

    if heads_frac <= 0.0:
        return 0
    if heads_frac >= 1.0:
        return num_heads
    return max(1, min(num_heads, int(math.ceil(float(num_heads) * heads_frac))))


def _lcha_sparse_mean_token_major(x_thd, cand_i32, mask_u8, row_chunk_size=32):
    """
    x_thd    : [T, H, D]
    cand_i32 : [T, C]
    mask_u8  : [T, C]
    returns  : [T, H, D]

    Chunked row-wise sparse neighborhood mean to avoid materializing huge [T,C,H,D].
    """
    if x_thd.dim() != 3:
        raise ValueError(f"x_thd must be [T,H,D], got {tuple(x_thd.shape)}")
    if not torch.is_floating_point(x_thd):
        raise ValueError(f"x_thd must be floating point, got {x_thd.dtype}")
    if cand_i32.dim() != 2 or mask_u8.dim() != 2:
        raise ValueError(
            f"cand_i32/mask_u8 must be [T,C], got {tuple(cand_i32.shape)} / {tuple(mask_u8.shape)}"
        )
    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand_i32.shape != mask_u8.shape: {tuple(cand_i32.shape)} vs {tuple(mask_u8.shape)}")

    T, H, D = x_thd.shape
    if cand_i32.size(0) != T:
        raise ValueError(f"cand_i32 first dim must equal T={T}, got {cand_i32.size(0)}")

    C = int(cand_i32.size(1))
    if T == 0 or C == 0:
        return x_thd

    row_chunk_size = max(1, int(row_chunk_size))
    x_thd = x_thd.contiguous()
    out = torch.empty_like(x_thd)

    cand_long = cand_i32.to(device=x_thd.device, dtype=torch.long)
    mask_bool = mask_u8.to(device=x_thd.device).to(torch.bool)

    # Validate only masked/active candidate positions.
    if bool(mask_bool.any().item()):
        active_cand = cand_long[mask_bool]
        minv = int(active_cand.min().item())
        maxv = int(active_cand.max().item())
        if minv < 0 or maxv >= T:
            raise ValueError(
                f"masked cand_i32 entries must be in [0, {T - 1}], got min={minv}, max={maxv}"
            )

    # Scrub inactive candidate slots to 0 so accidental gathers remain safe.
    cand_safe = torch.where(mask_bool, cand_long, torch.zeros_like(cand_long))

    for rs in range(0, T, row_chunk_size):
        re = min(T, rs + row_chunk_size)
        r = re - rs

        idx = cand_safe[rs:re]                                  # [r, C]
        gathered = x_thd[idx]                                   # [r, C, H, D]

        m = mask_bool[rs:re].to(dtype=x_thd.dtype).view(r, C, 1, 1)
        counts = m.sum(dim=1)                                   # [r, 1, 1]
        denom = counts.clamp_min(1.0)

        mean_val = (gathered * m).sum(dim=1) / denom             # [r, H, D]

        # If a row has no valid candidates, preserve the original row instead of zeroing it.
        has_any = (counts > 0).to(dtype=torch.bool)              # [r, 1, 1]
        out[rs:re] = torch.where(has_any, mean_val, x_thd[rs:re])

    return out


def lcha_prefilter_token_major(
    K, V,
    cand_i32, mask_u8,
    *,
    lcha_enabled=False,
    lcha_order=1,
    lcha_heads_frac=0.25,
    lcha_use_v=False,
    lcha_alpha0=1.0,
    lcha_alpha1=0.0,
    lcha_alpha2=0.0,
    lcha_v_alpha0=1.0,
    lcha_v_alpha1=0.0,
    lcha_v_alpha2=0.0,
    lcha_row_chunk_size=32,
):
    """
    K, V      : [T, H, Dk] / [T, H, Dv]
    cand_i32  : [T, C]
    mask_u8   : [T, C]

    Applies low-order harmonic filtering only to a prefix of heads for speed.
    """
    lcha_order = int(lcha_order)

    if (not bool(lcha_enabled)) or lcha_order <= 0:
        return K, V

    if lcha_order not in (1, 2):
        raise ValueError("lcha_order must be 1 or 2 when LCHA is enabled")

    if K.dim() != 3 or V.dim() != 3:
        raise ValueError(f"K/V must be [T,H,D], got {tuple(K.shape)} / {tuple(V.shape)}")
    if K.size(0) != V.size(0) or K.size(1) != V.size(1):
        raise ValueError(f"K/V must agree on [T,H], got {tuple(K.shape)} / {tuple(V.shape)}")
    if K.device != V.device:
        raise ValueError("K and V must be on the same device")
    if K.dtype != V.dtype:
        raise ValueError("K and V must have the same dtype")
    if not torch.is_floating_point(K) or not torch.is_floating_point(V):
        raise ValueError("K and V must be floating point tensors")

    T, H, Dk = K.shape
    _, Hv, Dv = V.shape
    if H != Hv:
        raise ValueError(f"K/V head count mismatch: {H} vs {Hv}")

    H_use = _lcha_num_heads_to_use(H, lcha_heads_frac)
    if H_use <= 0:
        return K, V

    row_chunk_size = max(1, int(lcha_row_chunk_size))

    lcha_alpha0 = _lcha_checked_float("lcha_alpha0", lcha_alpha0)
    lcha_alpha1 = _lcha_checked_float("lcha_alpha1", lcha_alpha1)
    lcha_alpha2 = _lcha_checked_float("lcha_alpha2", lcha_alpha2)
    lcha_v_alpha0 = _lcha_checked_float("lcha_v_alpha0", lcha_v_alpha0)
    lcha_v_alpha1 = _lcha_checked_float("lcha_v_alpha1", lcha_v_alpha1)
    lcha_v_alpha2 = _lcha_checked_float("lcha_v_alpha2", lcha_v_alpha2)

    # ---- K branch ----
    K_sel = K[:, :H_use, :].contiguous()
    PK = _lcha_sparse_mean_token_major(K_sel, cand_i32, mask_u8, row_chunk_size=row_chunk_size)

    K_mix = lcha_alpha0 * K_sel + lcha_alpha1 * PK
    if lcha_order >= 2 and abs(lcha_alpha2) > 0.0:
        P2K = _lcha_sparse_mean_token_major(PK, cand_i32, mask_u8, row_chunk_size=row_chunk_size)
        K_mix = K_mix + lcha_alpha2 * P2K

    if H_use < H:
        K_out = torch.cat([K_mix, K[:, H_use:, :]], dim=1).contiguous()
    else:
        K_out = K_mix.contiguous()

    # ---- V branch (optional) ----
    if not bool(lcha_use_v):
        return K_out, V

    V_sel = V[:, :H_use, :].contiguous()
    PV = _lcha_sparse_mean_token_major(V_sel, cand_i32, mask_u8, row_chunk_size=row_chunk_size)

    V_mix = lcha_v_alpha0 * V_sel + lcha_v_alpha1 * PV
    if lcha_order >= 2 and abs(lcha_v_alpha2) > 0.0:
        P2V = _lcha_sparse_mean_token_major(PV, cand_i32, mask_u8, row_chunk_size=row_chunk_size)
        V_mix = V_mix + lcha_v_alpha2 * P2V

    if H_use < H:
        V_out = torch.cat([V_mix, V[:, H_use:, :]], dim=1).contiguous()
    else:
        V_out = V_mix.contiguous()

    return K_out, V_out

In [ ]:
# ============================================================
# Snapshot + restore trainable params (LoRA) so sweep points are independent
#   - Run this ONCE after LoRA + SASA-CG patching is complete
#   - Then, inside the sweep loop, call restore_trainable_params_(...)
# ============================================================
import torch


def snapshot_trainable_params_(model, *, store_on_cpu: bool = True, check_finite: bool = True):
    """Return a dict of {param_name: tensor_clone} for all requires_grad params."""
    if model is None:
        raise ValueError("model must not be None")
    if not hasattr(model, "named_parameters"):
        raise TypeError("model must provide named_parameters()")

    snap = {}
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad:
                if check_finite and not torch.isfinite(p.detach().float()).all():
                    raise ValueError(f"Trainable param contains NaN/Inf before snapshot: {n}")

                t = p.detach().clone()
                if store_on_cpu:
                    t = t.cpu()
                snap[n] = t

    if len(snap) == 0:
        raise ValueError("No trainable parameters found. Check that LoRA/trainable params are enabled.")

    return snap


def restore_trainable_params_(model, snap, *, strict: bool = True, check_finite: bool = True):
    """In-place restore of requires_grad params from a snapshot dict."""
    if model is None:
        raise ValueError("model must not be None")
    if not isinstance(snap, dict) or len(snap) == 0:
        raise ValueError("snap must be a non-empty dict")

    restored = set()

    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad:
                if n not in snap:
                    raise KeyError(f"Trainable param missing in snapshot: {n}")

                src = snap[n]

                if tuple(src.shape) != tuple(p.shape):
                    raise ValueError(
                        f"Shape mismatch for {n}: snapshot {tuple(src.shape)} vs param {tuple(p.shape)}"
                    )

                if check_finite and not torch.isfinite(src.detach().float()).all():
                    raise ValueError(f"Snapshot param contains NaN/Inf: {n}")

                p.copy_(src.to(device=p.device, dtype=p.dtype))
                restored.add(n)

    if strict:
        current_trainable = {n for n, p in model.named_parameters() if p.requires_grad}
        extra_in_snapshot = set(snap.keys()) - current_trainable
        missing_restored = current_trainable - restored

        if missing_restored:
            raise KeyError(f"Trainable params not restored: {sorted(missing_restored)}")
        if extra_in_snapshot:
            raise KeyError(
                "Snapshot contains params that are no longer trainable/current: "
                f"{sorted(extra_in_snapshot)}"
            )

    # clear any stale grads
    for p in model.parameters():
        if p.grad is not None:
            p.grad = None


if "model" not in globals():
    raise NameError("model is not defined. Run model/LoRA/SASA-CG setup before taking snapshot.")

TRAINABLE_SNAPSHOT = snapshot_trainable_params_(model)
print(f"✅ Captured TRAINABLE_SNAPSHOT with {len(TRAINABLE_SNAPSHOT)} trainable tensors.")

NameError: name 'model' is not defined

In [ ]:
# ============================================================
# Stage-E ML Consistency Probe (Forward m/l must match Q,K,cand,mask)
# ============================================================
import torch, math, random


def _stagee_probe_sync(*tensors):
    if torch.cuda.is_available() and any(torch.is_tensor(t) and t.is_cuda for t in tensors):
        torch.cuda.synchronize()


@torch.no_grad()
def _stagee_row_slice(q: int, h: int, Tq: int, H: int, C: int, cand_is_per_head: bool):
    if cand_is_per_head:
        return (q * H + h) * C
    else:
        return q * C


@torch.no_grad()
def _stagee_get_ml_value(x: torch.Tensor, q: int, h: int, Tq: int, H: int, name: str) -> float:
    """
    Supports m/l stored either as [Tq*H] or [Tq,H].
    """
    if not torch.is_tensor(x):
        raise TypeError(f"{name} must be a tensor")

    if x.numel() == Tq * H:
        return float(x.reshape(-1)[q * H + h].item())

    if x.dim() == 2 and x.shape[0] == Tq and x.shape[1] == H:
        return float(x[q, h].item())

    raise ValueError(
        f"{name} has unsupported shape {tuple(x.shape)}; expected [Tq*H] or [Tq,H] "
        f"with Tq={Tq}, H={H}"
    )


@torch.no_grad()
def stagee_recompute_ml_for_rows(Q, K, cand_i32, mask_u8, scale: float, cand_is_per_head: bool, rows):
    """
    Recompute (m,l) for given (q,h) rows using the SAME semantics as Stage-E kernel:
      - ignore candidates with mask==0
      - ignore k_idx out of range
      - logits = dot(Q[q,h], K[k,h]) * scale
      - m = max(logits), l = sum(exp(logits - m))
      - if no valid logits => m=-inf, l=0

    Returns dict[(q,h)] = (m_ref, l_ref, n_valid)
    """
    if not (torch.is_tensor(Q) and torch.is_tensor(K) and torch.is_tensor(cand_i32) and torch.is_tensor(mask_u8)):
        raise TypeError("Q, K, cand_i32, and mask_u8 must be tensors")

    if not (Q.is_cuda and K.is_cuda and cand_i32.is_cuda and mask_u8.is_cuda):
        raise ValueError("Stage-E probe expects Q, K, cand_i32, and mask_u8 on CUDA")

    if Q.dim() != 3 or K.dim() != 3:
        raise ValueError(f"Q/K must be [T,H,D], got Q={tuple(Q.shape)}, K={tuple(K.shape)}")

    Tq, H, D = Q.shape
    Tk, Hk, Dk = K.shape

    if Hk != H or Dk != D:
        raise ValueError(f"Q/K shape mismatch: Q={tuple(Q.shape)}, K={tuple(K.shape)}")

    if cand_i32.dim() != 2 or mask_u8.dim() != 2:
        raise ValueError(
            f"cand_i32/mask_u8 must be [R,C], got {tuple(cand_i32.shape)} / {tuple(mask_u8.shape)}"
        )

    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand_i32/mask_u8 shape mismatch: {tuple(cand_i32.shape)} vs {tuple(mask_u8.shape)}")

    if not math.isfinite(float(scale)):
        raise ValueError("scale must be finite")

    C = int(cand_i32.shape[1])
    expected_rows = Tq * H if cand_is_per_head else Tq
    if int(cand_i32.shape[0]) != expected_rows:
        raise ValueError(
            f"cand_i32 row mismatch: got {cand_i32.shape[0]}, expected {expected_rows} "
            f"for Tq={Tq}, H={H}, cand_is_per_head={bool(cand_is_per_head)}"
        )

    Qf = Q.float()
    Kf = K.float()

    cand = cand_i32.reshape(-1)
    mask = mask_u8.reshape(-1)

    out = {}
    for (q, h) in rows:
        q = int(q)
        h = int(h)

        if q < 0 or q >= Tq:
            raise ValueError(f"q out of range: q={q}, Tq={Tq}")
        if h < 0 or h >= H:
            raise ValueError(f"h out of range: h={h}, H={H}")

        base = _stagee_row_slice(q, h, Tq, H, C, cand_is_per_head)

        c_row = cand[base:base + C].detach().cpu().tolist()
        m_row = mask[base:base + C].detach().cpu().tolist()

        logits = []
        for jj in range(C):
            if int(m_row[jj]) == 0:
                continue

            k_idx = int(c_row[jj])

            # IMPORTANT: K length is Tk, not Tq.
            if k_idx < 0 or k_idx >= Tk:
                continue

            logits.append(float(torch.dot(Qf[q, h], Kf[k_idx, h]).item() * float(scale)))

        if len(logits) == 0:
            out[(q, h)] = (-float("inf"), 0.0, 0)
        else:
            m_ref = max(logits)
            l_ref = 0.0
            for z in logits:
                l_ref += math.exp(z - m_ref)
            out[(q, h)] = (m_ref, l_ref, len(logits))

    return out


@torch.no_grad()
def stagee_probe_ml_once(
    Q,
    K,
    V,
    cand_i32,
    mask_u8,
    scale: float,
    cand_is_per_head: bool,
    n_rows: int = 8,
    seed: int = 0,
    fwd_fn=None,
):
    """
    Runs Stage-E forward, then recomputes m/l for random (q,h) rows and compares.
    Prints worst offenders.

    fwd_fn defaults to global mh_forward if available.
    """
    if fwd_fn is None:
        if "mh_forward" not in globals():
            raise NameError("mh_forward is not defined. Pass fwd_fn=... or define mh_forward first.")
        fwd_fn = globals()["mh_forward"]

    if not callable(fwd_fn):
        raise TypeError("fwd_fn must be callable")

    if Q.dim() != 3 or K.dim() != 3 or V.dim() != 3:
        raise ValueError(f"Q/K/V must be [T,H,D], got {tuple(Q.shape)} / {tuple(K.shape)} / {tuple(V.shape)}")

    Tq, H, D = Q.shape
    Tk, Hk, Dk = K.shape

    if Tq <= 0 or H <= 0:
        raise ValueError("Q must have nonzero Tq and H")
    if K.shape != V.shape:
        raise ValueError(f"K/V shape mismatch: K={tuple(K.shape)}, V={tuple(V.shape)}")
    if Hk != H or Dk != D:
        raise ValueError(f"Q/K shape mismatch: Q={tuple(Q.shape)}, K={tuple(K.shape)}")

    n_rows = int(n_rows)
    if n_rows <= 0:
        raise ValueError("n_rows must be > 0")

    _stagee_probe_sync(Q, K, V, cand_i32, mask_u8)
    Out, m, l = fwd_fn(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head))
    _stagee_probe_sync(Out, m, l)

    random.seed(int(seed))
    rows = []
    for _ in range(n_rows):
        q = random.randrange(Tq)
        h = random.randrange(H)
        rows.append((q, h))

    ref = stagee_recompute_ml_for_rows(Q, K, cand_i32, mask_u8, scale, cand_is_per_head, rows)

    worst = []
    for (q, h) in rows:
        mk = _stagee_get_ml_value(m, q, h, Tq, H, "m")
        lk = _stagee_get_ml_value(l, q, h, Tq, H, "l")
        mr, lr, nv = ref[(q, h)]

        m_ok = (
            (math.isinf(mk) and math.isinf(mr))
            or (math.isfinite(mk) and math.isfinite(mr) and abs(mk - mr) <= 1e-2 * (1.0 + abs(mr)))
        )
        l_ok = abs(lk - lr) <= 1e-2 * (1.0 + abs(lr))

        worst.append((not (m_ok and l_ok), q, h, nv, mk, mr, lk, lr))

    worst.sort(reverse=True)

    print("[Stage-E ML probe] rows checked:", rows)
    for flag, q, h, nv, mk, mr, lk, lr in worst[:min(8, len(worst))]:
        if not flag:
            continue
        print(f"  MISMATCH (q={q}, h={h}, n_valid={nv})")
        print(f"    m_kernel={mk:.6f}  m_ref={mr:.6f}")
        print(f"    l_kernel={lk:.6f}  l_ref={lr:.6f}")

    # global sanity
    bad_m_nan = int(torch.isnan(m.detach().float()).sum().item())
    bad_m_inf = int(torch.isinf(m.detach().float()).sum().item())
    bad_l_nan = int(torch.isnan(l.detach().float()).sum().item())
    bad_l_inf = int(torch.isinf(l.detach().float()).sum().item())

    print(
        f"[Stage-E ML probe] m shape={tuple(m.shape)} l shape={tuple(l.shape)} "
        f"m_nan={bad_m_nan} m_inf={bad_m_inf} l_nan={bad_l_nan} l_inf={bad_l_inf}"
    )

    return Out, m, l


# Example usage:
# _ = stagee_probe_ml_once(
#     Q, K, V, cand_i32, mask_u8,
#     scale=1.0 / math.sqrt(Q.shape[-1]),
#     cand_is_per_head=True,
#     n_rows=16,
#     seed=0,
# )

In [ ]:
# ============================================================
# Stage-E Backward Reference Check (restricted attention)
# ============================================================
import torch, math, random


def _stagee_ref_sync(*tensors):
    if torch.cuda.is_available() and any(torch.is_tensor(t) and t.is_cuda for t in tensors):
        torch.cuda.synchronize()


def _row_view(cand_i32, mask_u8, q, h, H, C, cand_is_per_head):
    if cand_is_per_head:
        row = q * H + h
    else:
        row = q
    c = cand_i32[row]      # (C,)
    m = mask_u8[row]       # (C,)
    return c, m


def _stagee_ref_max_abs(a, b):
    af = a.detach().float()
    bf = b.detach().float()

    same_inf = torch.isinf(af) & torch.isinf(bf) & (torch.sign(af) == torch.sign(bf))
    diff = (af - bf).abs()
    diff = torch.where(same_inf, torch.zeros_like(diff), diff)
    diff = torch.where(torch.isnan(diff), torch.full_like(diff, float("inf")), diff)

    return float(diff.max().item()) if diff.numel() > 0 else 0.0


def _stagee_require_kernel_fns(fwd_fn=None, bwd_fn=None):
    if fwd_fn is None:
        if "mh_forward" not in globals():
            raise NameError("mh_forward is not defined. Pass fwd_fn=... or define mh_forward first.")
        fwd_fn = globals()["mh_forward"]

    if bwd_fn is None:
        if "mh_backward" not in globals():
            raise NameError("mh_backward is not defined. Pass bwd_fn=... or define mh_backward first.")
        bwd_fn = globals()["mh_backward"]

    if not callable(fwd_fn):
        raise TypeError("fwd_fn must be callable")
    if not callable(bwd_fn):
        raise TypeError("bwd_fn must be callable")

    return fwd_fn, bwd_fn


def _stagee_validate_ref_inputs(Q, K, V, cand_i32, mask_u8, cand_is_per_head):
    if Q.dim() != 3 or K.dim() != 3 or V.dim() != 3:
        raise ValueError(f"Q/K/V must be [T,H,D], got {tuple(Q.shape)} / {tuple(K.shape)} / {tuple(V.shape)}")

    Tq, H, D = Q.shape
    Tk, Hk, Dk = K.shape
    Tv, Hv, DV = V.shape

    if Tq <= 0 or Tk <= 0:
        raise ValueError("Q and K must have nonzero sequence length")
    if H <= 0 or D <= 0:
        raise ValueError("Q must have nonzero H and D")
    if Tk != Tv or Hk != Hv:
        raise ValueError(f"K/V must agree on [T,H], got K={tuple(K.shape)} V={tuple(V.shape)}")
    if Hk != H or Dk != D:
        raise ValueError(f"Q/K shape mismatch: Q={tuple(Q.shape)} K={tuple(K.shape)}")
    if Q.device != K.device or Q.device != V.device:
        raise ValueError("Q/K/V must be on the same device")
    if cand_i32.device != Q.device or mask_u8.device != Q.device:
        raise ValueError("cand_i32/mask_u8 must be on the same device as Q/K/V")
    if cand_i32.dtype != torch.int32:
        raise ValueError(f"cand_i32 must be int32, got {cand_i32.dtype}")
    if mask_u8.dtype != torch.uint8:
        raise ValueError(f"mask_u8 must be uint8, got {mask_u8.dtype}")
    if cand_i32.dim() != 2 or mask_u8.dim() != 2:
        raise ValueError(f"cand_i32/mask_u8 must be [R,C], got {tuple(cand_i32.shape)} / {tuple(mask_u8.shape)}")
    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand_i32/mask_u8 shape mismatch: {tuple(cand_i32.shape)} vs {tuple(mask_u8.shape)}")

    expected_rows = Tq * H if cand_is_per_head else Tq
    if int(cand_i32.shape[0]) != expected_rows:
        raise ValueError(
            f"cand_i32 rows mismatch: got {cand_i32.shape[0]}, expected {expected_rows} "
            f"for Tq={Tq}, H={H}, cand_is_per_head={bool(cand_is_per_head)}"
        )


def stagee_ref_fwd(Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head):
    _stagee_validate_ref_inputs(Q, K, V, cand_i32, mask_u8, cand_is_per_head)

    Tq, H, D = Q.shape
    Tk = K.shape[0]
    DV = V.shape[-1]
    C = cand_i32.shape[1]

    Qf = Q.float()
    Kf = K.float()
    Vf = V.float()

    Out = torch.zeros((Tq, H, DV), device=Q.device, dtype=torch.float32)
    m_out = torch.empty((Tq * H,), device=Q.device, dtype=torch.float32)
    l_out = torch.empty((Tq * H,), device=Q.device, dtype=torch.float32)

    for q in range(Tq):
        for h in range(H):
            c_row, m_row = _row_view(cand_i32, mask_u8, q, h, H, C, cand_is_per_head)

            logits = []
            k_list = []
            for jj in range(C):
                if int(m_row[jj].item()) == 0:
                    continue
                k = int(c_row[jj].item())

                # IMPORTANT: valid K index is checked against Tk, not Tq.
                if k < 0 or k >= Tk:
                    continue

                z = torch.dot(Qf[q, h], Kf[k, h]) * float(scale)
                logits.append(z)
                k_list.append(k)

            ml_row = q * H + h
            if len(logits) == 0:
                m_out[ml_row] = -float("inf")
                l_out[ml_row] = 0.0
                continue

            logits_t = torch.stack(logits)  # (Nv,)
            m = torch.max(logits_t)
            ex = torch.exp(logits_t - m)
            l = torch.sum(ex)

            p = ex / l
            acc = torch.zeros((DV,), device=Q.device, dtype=torch.float32)
            for idx, k in enumerate(k_list):
                acc += p[idx] * Vf[k, h]

            Out[q, h] = acc
            m_out[ml_row] = m
            l_out[ml_row] = l

    return Out, m_out, l_out


def stagee_ref_bwd(Q, K, V, cand_i32, mask_u8, dOut, m_out, l_out, scale, cand_is_per_head):
    _stagee_validate_ref_inputs(Q, K, V, cand_i32, mask_u8, cand_is_per_head)

    Tq, H, D = Q.shape
    Tk = K.shape[0]
    DV = V.shape[-1]
    C = cand_i32.shape[1]

    if dOut.shape != (Tq, H, DV):
        raise ValueError(f"dOut shape mismatch: got {tuple(dOut.shape)}, expected {(Tq, H, DV)}")
    if m_out.numel() != Tq * H or l_out.numel() != Tq * H:
        raise ValueError("m_out/l_out must have Tq*H elements")

    Qf = Q.float()
    Kf = K.float()
    Vf = V.float()
    dOf = dOut.float()

    dQ = torch.zeros_like(Qf)
    dK = torch.zeros_like(Kf)
    dV = torch.zeros_like(Vf)

    for q in range(Tq):
        for h in range(H):
            c_row, m_row = _row_view(cand_i32, mask_u8, q, h, H, C, cand_is_per_head)
            ml_row = q * H + h
            m = float(m_out.reshape(-1)[ml_row].item())
            l = float(l_out.reshape(-1)[ml_row].item())

            if not math.isfinite(m) or not math.isfinite(l) or l <= 0.0:
                continue

            ks = []
            zs = []
            for jj in range(C):
                if int(m_row[jj].item()) == 0:
                    continue
                k = int(c_row[jj].item())

                # IMPORTANT: valid K index is checked against Tk, not Tq.
                if k < 0 or k >= Tk:
                    continue

                z = float(torch.dot(Qf[q, h], Kf[k, h]).item() * float(scale))
                ks.append(k)
                zs.append(z)

            if len(ks) == 0:
                continue

            ps = [math.exp(z - m) / l for z in zs]

            # dV accum: dV[k] += p_i * dOut
            for i, k in enumerate(ks):
                dV[k, h] += float(ps[i]) * dOf[q, h]

            # dP_i = <dOut, V[k]>
            dP = [float(torch.dot(dOf[q, h], Vf[k, h]).item()) for k in ks]
            s = 0.0
            for i in range(len(ks)):
                s += ps[i] * dP[i]

            # dLogit_i = p_i * (dP_i - s)
            dLog = [ps[i] * (dP[i] - s) for i in range(len(ks))]

            # dQ += sum_i dLog_i * K[k] * scale
            for i, k in enumerate(ks):
                dQ[q, h] += float(dLog[i] * float(scale)) * Kf[k, h]
                dK[k, h] += float(dLog[i] * float(scale)) * Qf[q, h]

    return dQ, dK, dV


@torch.no_grad()
def stagee_backward_reference_check(
    T=32,
    H=4,
    D=64,
    C=16,
    cand_is_per_head=True,
    seed=0,
    device=None,
    dtype=torch.float32,
    fwd_fn=None,
    bwd_fn=None,
    bwd_arg_order="qkv_first",  # "qkv_first" or "dout_first"
):
    fwd_fn, bwd_fn = _stagee_require_kernel_fns(fwd_fn=fwd_fn, bwd_fn=bwd_fn)

    if device is None:
        if not torch.cuda.is_available():
            raise RuntimeError("Stage-E reference check requires CUDA because mh_forward/mh_backward are CUDA kernels.")
        device = torch.device("cuda")
    else:
        device = torch.device(device)

    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA device requested, but torch.cuda.is_available() is False")

    T = int(T)
    H = int(H)
    D = int(D)
    C = int(C)

    if T <= 0 or H <= 0 or D <= 0 or C <= 0:
        raise ValueError("T, H, D, and C must all be > 0")
    if not math.isfinite(float(1.0 / math.sqrt(D))):
        raise ValueError("Invalid D for scale computation")

    torch.manual_seed(int(seed))
    random.seed(int(seed))

    Q = torch.randn((T, H, D), device=device, dtype=dtype).contiguous()
    K = torch.randn((T, H, D), device=device, dtype=dtype).contiguous()
    V = torch.randn((T, H, D), device=device, dtype=dtype).contiguous()

    rows = (T * H) if cand_is_per_head else T
    cand = torch.randint(low=0, high=T, size=(rows, C), device=device, dtype=torch.int32)
    mask = (torch.rand((rows, C), device=device) > 0.25).to(torch.uint8)

    scale = 1.0 / math.sqrt(D)

    _stagee_ref_sync(Q, K, V, cand, mask)
    Out_k, m_k, l_k = fwd_fn(Q, K, V, cand, mask, float(scale), bool(cand_is_per_head))
    _stagee_ref_sync(Out_k, m_k, l_k)

    # ---- ONE TEST: masked-slot invariance ----
    cand_safe = cand.clone()
    cand_safe[mask == 0] = 0
    Out_safe, m_safe, l_safe = fwd_fn(Q, K, V, cand_safe, mask, float(scale), bool(cand_is_per_head))

    diff_out = _stagee_ref_max_abs(Out_k, Out_safe)
    diff_m = _stagee_ref_max_abs(m_k, m_safe)
    diff_l = _stagee_ref_max_abs(l_k, l_safe)
    print("masked-slot invariance max|diff| : Out=", diff_out, " m=", diff_m, " l=", diff_l)

    dOut = torch.randn_like(Out_k).contiguous()

    if bwd_arg_order == "qkv_first":
        dQ_k, dK_k, dV_k = bwd_fn(Q, K, V, cand, mask, dOut, m_k, l_k, float(scale), bool(cand_is_per_head))
    elif bwd_arg_order == "dout_first":
        dQ_k, dK_k, dV_k = bwd_fn(dOut, Q, K, V, cand, mask, m_k, l_k, float(scale), bool(cand_is_per_head))
    else:
        raise ValueError("bwd_arg_order must be 'qkv_first' or 'dout_first'")

    _stagee_ref_sync(dQ_k, dK_k, dV_k)

    # reference
    Out_r, m_r, l_r = stagee_ref_fwd(Q, K, V, cand, mask, scale, cand_is_per_head)
    dQ_r, dK_r, dV_r = stagee_ref_bwd(Q, K, V, cand, mask, dOut, m_r, l_r, scale, cand_is_per_head)

    print("[Stage-E refcheck] max|Out_k-Out_r| =", _stagee_ref_max_abs(Out_k, Out_r))
    print("[Stage-E refcheck] max|m_k-m_r|     =", _stagee_ref_max_abs(m_k, m_r))
    print("[Stage-E refcheck] max|l_k-l_r|     =", _stagee_ref_max_abs(l_k, l_r))
    print("[Stage-E refcheck] max|dQ_k-dQ_r|   =", _stagee_ref_max_abs(dQ_k, dQ_r))
    print("[Stage-E refcheck] max|dK_k-dK_r|   =", _stagee_ref_max_abs(dK_k, dK_r))
    print("[Stage-E refcheck] max|dV_k-dV_r|   =", _stagee_ref_max_abs(dV_k, dV_r))

    # hard fail for invalid nonfinite values.
    # m may legitimately contain -inf for all-masked rows; other tensors should not.
    for name, t in [
        ("Out_k", Out_k),
        ("l_k", l_k),
        ("dQ_k", dQ_k),
        ("dK_k", dK_k),
        ("dV_k", dV_k),
    ]:
        if (~torch.isfinite(t.float())).any().item():
            raise RuntimeError(f"NONFINITE in {name}")

    m_float = m_k.float()
    if torch.isnan(m_float).any().item() or torch.isposinf(m_float).any().item():
        raise RuntimeError("Invalid NaN/+Inf in m_k")

    print("[Stage-E refcheck] completed")

    return {
        "Out_k": Out_k,
        "m_k": m_k,
        "l_k": l_k,
        "dQ_k": dQ_k,
        "dK_k": dK_k,
        "dV_k": dV_k,
        "Out_ref": Out_r,
        "m_ref": m_r,
        "l_ref": l_r,
        "dQ_ref": dQ_r,
        "dK_ref": dK_r,
        "dV_ref": dV_r,
    }


# Run:
# stagee_backward_reference_check()

In [ ]:
# ============================================================
# Stage-E Runtime Trap (find first bad (q,h) row in backward)
# ============================================================
import torch, math, random


def _stagee_trap_sync(*tensors):
    if torch.cuda.is_available() and any(torch.is_tensor(t) and t.is_cuda for t in tensors):
        torch.cuda.synchronize()


def _stagee_trap_require_fns(fwd_fn=None, bwd_fn=None):
    if fwd_fn is None:
        if "mh_forward" not in globals():
            raise NameError("mh_forward is not defined. Pass fwd_fn=... or define mh_forward first.")
        fwd_fn = globals()["mh_forward"]

    if bwd_fn is None:
        if "mh_backward" not in globals():
            raise NameError("mh_backward is not defined. Pass bwd_fn=... or define mh_backward first.")
        bwd_fn = globals()["mh_backward"]

    if not callable(fwd_fn):
        raise TypeError("fwd_fn must be callable")
    if not callable(bwd_fn):
        raise TypeError("bwd_fn must be callable")

    return fwd_fn, bwd_fn


def _stagee_trap_get_ml_value(x: torch.Tensor, q: int, h: int, Tq: int, H: int, name: str) -> float:
    """
    Supports m/l stored either as [Tq*H] or [Tq,H].
    """
    if not torch.is_tensor(x):
        raise TypeError(f"{name} must be a tensor")

    if x.numel() == Tq * H:
        return float(x.reshape(-1)[q * H + h].item())

    if x.dim() == 2 and x.shape[0] == Tq and x.shape[1] == H:
        return float(x[q, h].item())

    raise ValueError(
        f"{name} has unsupported shape {tuple(x.shape)}; expected [Tq*H] or [Tq,H] "
        f"with Tq={Tq}, H={H}"
    )


@torch.no_grad()
def _stagee_trap_recompute_ml_for_rows(
    Q,
    K,
    cand_i32,
    mask_u8,
    scale: float,
    cand_is_per_head: bool,
    rows,
):
    """
    Local recompute of m/l for selected (q,h) rows.
    Uses K length Tk, not Q length Tq.
    """
    if Q.dim() != 3 or K.dim() != 3:
        raise ValueError(f"Q/K must be [T,H,D], got Q={tuple(Q.shape)}, K={tuple(K.shape)}")

    Tq, H, D = Q.shape
    Tk, Hk, Dk = K.shape

    if Hk != H or Dk != D:
        raise ValueError(f"Q/K shape mismatch: Q={tuple(Q.shape)}, K={tuple(K.shape)}")

    if cand_i32.dim() != 2 or mask_u8.dim() != 2:
        raise ValueError(
            f"cand_i32/mask_u8 must be [R,C], got {tuple(cand_i32.shape)} / {tuple(mask_u8.shape)}"
        )

    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand_i32/mask_u8 shape mismatch: {tuple(cand_i32.shape)} vs {tuple(mask_u8.shape)}")

    C = int(cand_i32.shape[1])
    expected_rows = Tq * H if cand_is_per_head else Tq
    if int(cand_i32.shape[0]) != expected_rows:
        raise ValueError(
            f"cand_i32 row mismatch: got {cand_i32.shape[0]}, expected {expected_rows} "
            f"for Tq={Tq}, H={H}, cand_is_per_head={bool(cand_is_per_head)}"
        )

    Qf = Q.float()
    Kf = K.float()

    out = {}

    for q, h in rows:
        q = int(q)
        h = int(h)

        if q < 0 or q >= Tq:
            raise ValueError(f"q out of range: q={q}, Tq={Tq}")
        if h < 0 or h >= H:
            raise ValueError(f"h out of range: h={h}, H={H}")

        row = q * H + h if cand_is_per_head else q
        c_row = cand_i32[row].detach().cpu().tolist()
        m_row = mask_u8[row].detach().cpu().tolist()

        logits = []
        for jj in range(C):
            if int(m_row[jj]) == 0:
                continue

            k_idx = int(c_row[jj])

            # Important: valid K index is checked against Tk.
            if k_idx < 0 or k_idx >= Tk:
                continue

            logits.append(float(torch.dot(Qf[q, h], Kf[k_idx, h]).item() * float(scale)))

        if len(logits) == 0:
            out[(q, h)] = (-float("inf"), 0.0, 0)
        else:
            m_ref = max(logits)
            l_ref = 0.0
            for z in logits:
                l_ref += math.exp(z - m_ref)
            out[(q, h)] = (m_ref, l_ref, len(logits))

    return out


def _stagee_trap_nonfinite_summary(t: torch.Tensor, *, allow_neginf: bool = False):
    tf = t.detach().float()
    nan = int(torch.isnan(tf).sum().item())
    posinf = int(torch.isposinf(tf).sum().item())
    neginf = int(torch.isneginf(tf).sum().item())

    if allow_neginf:
        bad = nan + posinf
    else:
        bad = nan + posinf + neginf

    finite = tf[torch.isfinite(tf)]
    maxabs = float(finite.abs().max().item()) if finite.numel() > 0 else 0.0

    return {
        "bad": bool(bad > 0),
        "nan": nan,
        "+inf": posinf,
        "-inf": neginf,
        "finite_maxabs": maxabs,
    }


@torch.no_grad()
def stagee_find_first_bad_row(
    Q,
    K,
    V,
    cand_i32,
    mask_u8,
    scale,
    cand_is_per_head,
    dOut=None,
    max_rows_to_scan=64,
    seed=0,
    fwd_fn=None,
    bwd_fn=None,
    bwd_arg_order="qkv_first",  # "qkv_first" or "dout_first"
    grad_spike_threshold=1e3,
):
    fwd_fn, bwd_fn = _stagee_trap_require_fns(fwd_fn=fwd_fn, bwd_fn=bwd_fn)

    if Q.dim() != 3 or K.dim() != 3 or V.dim() != 3:
        raise ValueError(f"Q/K/V must be [T,H,D], got {tuple(Q.shape)} / {tuple(K.shape)} / {tuple(V.shape)}")

    Tq, H, D = Q.shape
    Tk, Hk, Dk = K.shape

    if K.shape != V.shape:
        raise ValueError(f"K/V shape mismatch: K={tuple(K.shape)}, V={tuple(V.shape)}")
    if Hk != H or Dk != D:
        raise ValueError(f"Q/K shape mismatch: Q={tuple(Q.shape)}, K={tuple(K.shape)}")
    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand_i32/mask_u8 shape mismatch: {tuple(cand_i32.shape)} vs {tuple(mask_u8.shape)}")
    if cand_i32.dtype != torch.int32:
        raise ValueError(f"cand_i32 must be int32, got {cand_i32.dtype}")
    if mask_u8.dtype != torch.uint8:
        raise ValueError(f"mask_u8 must be uint8, got {mask_u8.dtype}")
    if not math.isfinite(float(scale)):
        raise ValueError("scale must be finite")

    max_rows_to_scan = int(max_rows_to_scan)
    if max_rows_to_scan <= 0:
        raise ValueError("max_rows_to_scan must be > 0")

    _stagee_trap_sync(Q, K, V, cand_i32, mask_u8)
    Out, m, l = fwd_fn(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head))
    _stagee_trap_sync(Out, m, l)

    if dOut is None:
        dOut = torch.randn_like(Out)
    else:
        if dOut.shape != Out.shape:
            raise ValueError(f"dOut shape mismatch: got {tuple(dOut.shape)}, expected {tuple(Out.shape)}")
        dOut = dOut.to(device=Out.device, dtype=Out.dtype).contiguous()

    if bwd_arg_order == "qkv_first":
        dQ, dK, dV = bwd_fn(Q, K, V, cand_i32, mask_u8, dOut, m, l, float(scale), bool(cand_is_per_head))
    elif bwd_arg_order == "dout_first":
        dQ, dK, dV = bwd_fn(dOut, Q, K, V, cand_i32, mask_u8, m, l, float(scale), bool(cand_is_per_head))
    else:
        raise ValueError("bwd_arg_order must be 'qkv_first' or 'dout_first'")

    _stagee_trap_sync(dQ, dK, dV)

    print(
        "[Stage-E trap] nonfinite:",
        {
            "Out": _stagee_trap_nonfinite_summary(Out),
            "m": _stagee_trap_nonfinite_summary(m, allow_neginf=True),
            "l": _stagee_trap_nonfinite_summary(l),
            "dQ": _stagee_trap_nonfinite_summary(dQ),
            "dK": _stagee_trap_nonfinite_summary(dK),
            "dV": _stagee_trap_nonfinite_summary(dV),
        },
    )

    print(
        "[Stage-E trap] finite maxabs:",
        {
            "Out": _stagee_trap_nonfinite_summary(Out)["finite_maxabs"],
            "m": _stagee_trap_nonfinite_summary(m, allow_neginf=True)["finite_maxabs"],
            "l": _stagee_trap_nonfinite_summary(l)["finite_maxabs"],
            "dQ": _stagee_trap_nonfinite_summary(dQ)["finite_maxabs"],
            "dK": _stagee_trap_nonfinite_summary(dK)["finite_maxabs"],
            "dV": _stagee_trap_nonfinite_summary(dV)["finite_maxabs"],
        },
    )

    rows = [(q, h) for q in range(Tq) for h in range(H)]
    random.seed(int(seed))
    random.shuffle(rows)
    rows = rows[:min(len(rows), max_rows_to_scan)]

    suspects = []

    for (q, h) in rows:
        mk = _stagee_trap_get_ml_value(m, q, h, Tq, H, "m")
        lk = _stagee_trap_get_ml_value(l, q, h, Tq, H, "l")

        # m=-inf is allowed only if l==0.
        if math.isnan(mk) or math.isposinf(mk) or math.isnan(lk) or math.isinf(lk) or lk < 0.0:
            suspects.append((q, h, "bad_ml", mk, lk))
            continue

        if math.isneginf(mk) and lk != 0.0:
            suspects.append((q, h, "bad_empty_row_state", mk, lk))
            continue

        gq = float(dQ[q, h].float().abs().max().item())
        if (not math.isfinite(gq)) or (gq > float(grad_spike_threshold)):
            suspects.append((q, h, "bad_dQ", mk, lk))
            continue

        # Also scan dK/dV rows referenced by this candidate row.
        row = q * H + h if cand_is_per_head else q
        active = mask_u8[row].to(torch.bool)
        refs = cand_i32[row][active].detach().cpu().tolist()

        for k_idx in refs:
            k_idx = int(k_idx)
            if k_idx < 0 or k_idx >= Tk:
                continue

            gk = float(dK[k_idx, h].float().abs().max().item())
            gv = float(dV[k_idx, h].float().abs().max().item())

            if (not math.isfinite(gk)) or (gk > float(grad_spike_threshold)):
                suspects.append((q, h, f"bad_dK_ref_k={k_idx}", mk, lk))
                break

            if (not math.isfinite(gv)) or (gv > float(grad_spike_threshold)):
                suspects.append((q, h, f"bad_dV_ref_k={k_idx}", mk, lk))
                break

    if len(suspects) == 0:
        print("[Stage-E trap] no obvious bad rows in scan.")
        return Out, m, l, dQ, dK, dV

    print("[Stage-E trap] suspects (first few):")
    for s in suspects[:8]:
        print(" ", s)

    q, h, tag, *_ = suspects[0]
    ref = _stagee_trap_recompute_ml_for_rows(
        Q,
        K,
        cand_i32,
        mask_u8,
        float(scale),
        bool(cand_is_per_head),
        [(q, h)],
    )

    mr, lr, nv = ref[(q, h)]
    mk = _stagee_trap_get_ml_value(m, q, h, Tq, H, "m")
    lk = _stagee_trap_get_ml_value(l, q, h, Tq, H, "l")

    print(f"[Stage-E trap] TOP (q={q}, h={h}) tag={tag} n_valid={nv}")
    print(f"  m_kernel={mk:.6f}  m_ref={mr:.6f}")
    print(f"  l_kernel={lk:.6f}  l_ref={lr:.6f}")

    return Out, m, l, dQ, dK, dV

In [ ]:
# ============================================================
# First nonfinite gradient probe
# ============================================================
import torch


def _get_decoder_layers_for_grad_probe(model):
    """
    Tries common HF / PEFT / custom nesting paths.
    Adjust manually if your model uses a different layout.
    """
    candidate_paths = [
        ("base_model", "model", "model", "layers"),
        ("base_model", "model", "layers"),
        ("model", "model", "layers"),
        ("model", "layers"),
        ("layers",),
    ]

    for path in candidate_paths:
        obj = model
        ok = True
        for attr in path:
            if not hasattr(obj, attr):
                ok = False
                break
            obj = getattr(obj, attr)

        if ok:
            return obj, ".".join(path)

    raise AttributeError(
        "Could not find decoder layers. Tried paths: "
        + ", ".join(".".join(p) for p in candidate_paths)
    )


def _iter_tensor_grads(x):
    if torch.is_tensor(x):
        yield x
    elif isinstance(x, (tuple, list)):
        for item in x:
            yield from _iter_tensor_grads(item)
    elif isinstance(x, dict):
        for item in x.values():
            yield from _iter_tensor_grads(item)


def attach_first_nonfinite_grad_probe(model):
    layers, path_used = _get_decoder_layers_for_grad_probe(model)
    handles = []

    print(f"[grad-probe] attached to layers at: {path_used}")

    for li, layer in enumerate(layers):
        def make_hook(li):
            def hook(mod, grad_input, grad_output):
                for gi, g in enumerate(_iter_tensor_grads(grad_output)):
                    bad = ~torch.isfinite(g.detach())
                    if bool(bad.any().item()):
                        n_bad = int(bad.sum().item())
                        print(
                            f"[NONFINITE] layer={li} grad_output[{gi}] "
                            f"shape={tuple(g.shape)} dtype={g.dtype} nonfinite={n_bad}"
                        )
                        raise RuntimeError(f"First nonfinite grad at layer {li}")
            return hook

        handles.append(layer.register_full_backward_hook(make_hook(li)))

    return handles

In [ ]:
# ============================================================
# LaSPER-HT: CLEAN SINGLE-PATH NOTEBOOK BLOCK
#   - No fallback to old LaSPERDRouter
#   - Persistent prev_schedule during training
#   - One active routing path
#   - Budget-aware quota repair + cleaner projection/sampling
# ============================================================

import math
from dataclasses import dataclass, replace
from typing import List, Optional, Tuple, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW


def _infer_default_device():
    if "model" in globals() and hasattr(model, "parameters"):
        try:
            return next(model.parameters()).device
        except StopIteration:
            pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _infer_default_device()
EPS = 1e-8
AMP_ENABLED = bool(torch.cuda.is_available() and DEVICE.type == "cuda")


# ============================================================
# CELL 1: Protocol spec + basic helpers
# ============================================================
@dataclass
class ProtocolSpec:
    name: str
    w: int
    scaffold_mode: int
    ring_inner: int
    ring_outer: int
    num_anchors: int
    include_self: bool = True
    anchor_strategy: int = 0
    anchor_param0: int = 0
    quota_ratio: float = 0.25
    cost: Optional[float] = None

    # ---- LCHA controls ----
    lcha_enabled: bool = False
    lcha_order: int = 0
    lcha_heads_frac: float = 0.0
    lcha_use_v: bool = False

    # K filter coefficients
    lcha_alpha0: float = 1.0
    lcha_alpha1: float = 0.0
    lcha_alpha2: float = 0.0

    # V filter coefficients
    lcha_v_alpha0: float = 1.0
    lcha_v_alpha1: float = 0.0
    lcha_v_alpha2: float = 0.0

    # PyTorch chunk size for harmonic prefilter
    lcha_row_chunk_size: int = 32

    def __post_init__(self):
        if not isinstance(self.name, str) or len(self.name) == 0:
            raise ValueError("ProtocolSpec.name must be a non-empty string")

        self.w = int(self.w)
        self.scaffold_mode = int(self.scaffold_mode)
        self.ring_inner = int(self.ring_inner)
        self.ring_outer = int(self.ring_outer)
        self.num_anchors = int(self.num_anchors)
        self.anchor_strategy = int(self.anchor_strategy)
        self.anchor_param0 = int(self.anchor_param0)
        self.lcha_order = int(self.lcha_order)
        self.lcha_row_chunk_size = int(self.lcha_row_chunk_size)

        if self.w < 0:
            raise ValueError(f"{self.name}: w must be >= 0")
        if self.scaffold_mode not in (0, 1, 2):
            raise ValueError(f"{self.name}: scaffold_mode must be 0, 1, or 2")
        if self.ring_inner < 0 or self.ring_outer < 0:
            raise ValueError(f"{self.name}: ring_inner/ring_outer must be >= 0")
        if self.ring_outer < self.ring_inner:
            raise ValueError(f"{self.name}: ring_outer must be >= ring_inner")
        if self.num_anchors < 0:
            raise ValueError(f"{self.name}: num_anchors must be >= 0")
        if not math.isfinite(float(self.quota_ratio)) or float(self.quota_ratio) < 0.0:
            raise ValueError(f"{self.name}: quota_ratio must be finite and >= 0")
        if self.cost is not None and (not math.isfinite(float(self.cost)) or float(self.cost) < 0.0):
            raise ValueError(f"{self.name}: cost must be finite and >= 0")
        if self.lcha_order not in (0, 1, 2):
            raise ValueError(f"{self.name}: lcha_order must be 0, 1, or 2")
        if not math.isfinite(float(self.lcha_heads_frac)) or not (0.0 <= float(self.lcha_heads_frac) <= 1.0):
            raise ValueError(f"{self.name}: lcha_heads_frac must be in [0, 1]")
        if self.lcha_row_chunk_size <= 0:
            raise ValueError(f"{self.name}: lcha_row_chunk_size must be > 0")


def nominal_protocol_capacity(spec: ProtocolSpec, seq_len: int) -> int:
    seq_len = int(seq_len)
    if seq_len <= 0:
        raise ValueError("seq_len must be > 0")

    mode = int(spec.scaffold_mode)

    Cw = (2 * int(spec.w) + 1) if mode in (0, 2) else 0
    Cr = (int(spec.ring_outer) - int(spec.ring_inner) + 1) if mode in (1, 2) else 0
    Cr = max(Cr, 0)
    A = min(int(spec.num_anchors), seq_len) if int(spec.num_anchors) > 0 else 0

    # Window capacity already includes self via 2w+1.
    Cself = 1 if bool(spec.include_self) and mode not in (0, 2) else 0

    return int(Cself + Cw + Cr + A)


def ensure_protocol_costs(protocols: List[ProtocolSpec], seq_len: int) -> List[ProtocolSpec]:
    if len(protocols) == 0:
        raise ValueError("protocols must be non-empty")

    out = []
    for p in protocols:
        if not isinstance(p, ProtocolSpec):
            raise TypeError(f"Expected ProtocolSpec, got {type(p)}")

        if p.cost is None:
            out.append(replace(p, cost=float(nominal_protocol_capacity(p, seq_len))))
        else:
            out.append(p)

    return out


def quotas_from_ratios(num_layers: int, ratios: torch.Tensor) -> torch.Tensor:
    num_layers = int(num_layers)
    if num_layers <= 0:
        raise ValueError("num_layers must be > 0")

    if not torch.is_tensor(ratios):
        ratios = torch.tensor(ratios, dtype=torch.float32)

    if ratios.dim() != 1:
        raise ValueError(f"ratios must be rank-1, got shape {tuple(ratios.shape)}")
    if ratios.numel() == 0:
        raise ValueError("ratios must be non-empty")
    if not torch.isfinite(ratios.float()).all():
        raise ValueError("ratios contains NaN/Inf")
    if (ratios < 0).any():
        raise ValueError("ratios must be non-negative")

    ratios = ratios.float()
    ratio_sum = ratios.sum()
    if float(ratio_sum.item()) <= 0.0:
        raise ValueError("ratios must have positive sum")

    ratios = ratios / ratio_sum.clamp_min(EPS)

    q = torch.floor(ratios * float(num_layers))
    rem = int(num_layers - int(q.sum().item()))

    if rem > 0:
        frac = ratios * float(num_layers) - q
        order = torch.argsort(frac, descending=True)
        for i in range(rem):
            q[order[i % order.numel()]] += 1.0

    q = q.clamp_min(0.0)

    # Only enforce nonzero quota per protocol when it is mathematically possible.
    if num_layers >= int(ratios.numel()):
        zero_idx = torch.where(q == 0)[0]
        for zi in zero_idx.tolist():
            donor = int(torch.argmax(q).item())
            if q[donor] > 1:
                q[donor] -= 1.0
                q[zi] += 1.0

    if int(q.sum().item()) != num_layers:
        raise RuntimeError(f"quota repair failed: sum={int(q.sum().item())}, expected={num_layers}")

    return q


def safe_normalize(x: torch.Tensor, dim: int = -1, eps: float = 1e-6) -> torch.Tensor:
    if not torch.is_tensor(x):
        raise TypeError("x must be a torch.Tensor")
    if eps <= 0.0:
        raise ValueError("eps must be > 0")
    if not torch.isfinite(x.detach().float()).all():
        raise ValueError("safe_normalize input contains NaN/Inf")

    return x / x.norm(dim=dim, keepdim=True).clamp_min(eps)


def infer_seq_len_from_batch(batch: Dict[str, Any]) -> int:
    if not isinstance(batch, dict):
        raise TypeError("batch must be a dict")

    if "input_ids" in batch and torch.is_tensor(batch["input_ids"]):
        if batch["input_ids"].dim() < 2:
            raise ValueError("batch['input_ids'] must have shape [B,T] or higher")
        seq_len = int(batch["input_ids"].shape[1])
        if seq_len <= 0:
            raise ValueError("input_ids sequence length must be > 0")
        return seq_len

    if "labels" in batch and torch.is_tensor(batch["labels"]):
        if batch["labels"].dim() < 2:
            raise ValueError("batch['labels'] must have shape [B,T] or higher")
        seq_len = int(batch["labels"].shape[1])
        if seq_len <= 0:
            raise ValueError("labels sequence length must be > 0")
        return seq_len

    raise ValueError("Could not infer sequence length from probe batch.")

In [ ]:
# ============================================================
# CELL 2: Notebook hooks into custom GCCD sparse decoder attention
# ============================================================

def _is_gccd_sparse_attention_module(module: nn.Module) -> bool:
    """
    Detects the custom sparse decoder attention module without relying only
    on an exact class-name string.
    """
    cls_name = module.__class__.__name__

    if cls_name == "GCCDSparseDecoderAttention":
        return True

    required_attrs = (
        "candidate_builder_fn",
        "stagee_apply_fn",
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    )
    return all(hasattr(module, a) for a in required_attrs)


def find_gccd_attention_layers(model) -> List[Tuple[int, nn.Module]]:
    out = []

    if "_get_decoder_layers_any_wrapper" in globals():
        blocks = _get_decoder_layers_any_wrapper(model)
        for li, blk in enumerate(blocks):
            sa = getattr(blk, "self_attn", None)
            if sa is not None and _is_gccd_sparse_attention_module(sa):
                out.append((li, sa))
        if out:
            return out

    idx = 0
    for _, module in model.named_modules():
        if _is_gccd_sparse_attention_module(module):
            out.append((idx, module))
            idx += 1

    return out


def infer_summary_dim_from_model(model) -> int:
    layers = find_gccd_attention_layers(model)
    if len(layers) == 0:
        raise ValueError("No GCCDSparseDecoderAttention layers found.")

    _, first_layer = layers[0]

    if hasattr(first_layer, "hidden_size"):
        return int(first_layer.hidden_size)

    if hasattr(first_layer, "cfg") and hasattr(first_layer.cfg, "hidden_size"):
        return int(first_layer.cfg.hidden_size)

    raise ValueError("Could not infer hidden size from GCCD attention layer.")


def apply_protocol_to_layer(layer_module, proto: ProtocolSpec) -> None:
    # ---- sparse graph protocol fields ----
    layer_module.w = int(proto.w)
    layer_module.scaffold_mode = int(proto.scaffold_mode)
    layer_module.ring_inner = int(proto.ring_inner)
    layer_module.ring_outer = int(proto.ring_outer)
    layer_module.num_anchors = int(proto.num_anchors)
    layer_module.include_self = bool(proto.include_self)
    layer_module.anchor_strategy = int(proto.anchor_strategy)
    layer_module.anchor_param0 = int(proto.anchor_param0)
    layer_module._lasperht_protocol_name = proto.name
    layer_module._router_protocol_name = proto.name

    # ---- LCHA fields ----
    layer_module.lcha_enabled = bool(getattr(proto, "lcha_enabled", False))
    layer_module.lcha_order = int(getattr(proto, "lcha_order", 0))
    layer_module.lcha_heads_frac = float(getattr(proto, "lcha_heads_frac", 0.0))
    layer_module.lcha_use_v = bool(getattr(proto, "lcha_use_v", False))

    layer_module.lcha_alpha0 = float(getattr(proto, "lcha_alpha0", 1.0))
    layer_module.lcha_alpha1 = float(getattr(proto, "lcha_alpha1", 0.0))
    layer_module.lcha_alpha2 = float(getattr(proto, "lcha_alpha2", 0.0))

    layer_module.lcha_v_alpha0 = float(getattr(proto, "lcha_v_alpha0", 1.0))
    layer_module.lcha_v_alpha1 = float(getattr(proto, "lcha_v_alpha1", 0.0))
    layer_module.lcha_v_alpha2 = float(getattr(proto, "lcha_v_alpha2", 0.0))

    layer_module.lcha_row_chunk_size = int(getattr(proto, "lcha_row_chunk_size", 32))

    # Keep cfg synchronized when the custom attention layer owns a config object.
    if hasattr(layer_module, "cfg"):
        layer_module.cfg.lcha_enabled = layer_module.lcha_enabled
        layer_module.cfg.lcha_order = layer_module.lcha_order
        layer_module.cfg.lcha_heads_frac = layer_module.lcha_heads_frac
        layer_module.cfg.lcha_use_v = layer_module.lcha_use_v

        layer_module.cfg.lcha_alpha0 = layer_module.lcha_alpha0
        layer_module.cfg.lcha_alpha1 = layer_module.lcha_alpha1
        layer_module.cfg.lcha_alpha2 = layer_module.lcha_alpha2

        layer_module.cfg.lcha_v_alpha0 = layer_module.lcha_v_alpha0
        layer_module.cfg.lcha_v_alpha1 = layer_module.lcha_v_alpha1
        layer_module.cfg.lcha_v_alpha2 = layer_module.lcha_v_alpha2


def clear_gccd_candidate_caches(model) -> None:
    for _, layer in find_gccd_attention_layers(model):
        for cache_name in ("shared_cand_cache", "candidate_cache", "cand_cache"):
            cache = getattr(layer, cache_name, None)
            if cache is not None and hasattr(cache, "clear"):
                cache.clear()


def validate_sparse_attention_wiring(model) -> None:
    layers = find_gccd_attention_layers(model)

    if len(layers) == 0:
        raise ValueError("No GCCDSparseDecoderAttention layers found.")

    for li, layer in layers:
        if not bool(getattr(layer.cfg, "use_sparse_stagee", False)):
            raise ValueError(
                f"Layer {li}: cfg.use_sparse_stagee=False, so LaSPER-HT schedule "
                "will not affect actual attention."
            )

        if getattr(layer, "candidate_builder_fn", None) is None:
            raise ValueError(
                f"Layer {li}: candidate_builder_fn is None. Stage-A is not wired."
            )

        if getattr(layer, "stagee_apply_fn", None) is None:
            raise ValueError(
                f"Layer {li}: stagee_apply_fn is None. Stage-E is not wired."
            )


def apply_schedule_to_model(model, schedule: torch.Tensor, protocol_table: List[ProtocolSpec]) -> None:
    validate_sparse_attention_wiring(model)

    layers = find_gccd_attention_layers(model)

    if len(layers) == 0:
        raise ValueError("No GCCDSparseDecoderAttention layers found.")

    if not torch.is_tensor(schedule):
        schedule = torch.tensor(schedule, dtype=torch.long)

    schedule = schedule.detach().cpu().long().view(-1)

    if len(layers) != int(schedule.numel()):
        raise ValueError(
            f"Schedule length {int(schedule.numel())} != number of GCCD attention layers {len(layers)}"
        )

    if len(protocol_table) == 0:
        raise ValueError("protocol_table must be non-empty.")

    for i, (_, layer) in enumerate(layers):
        proto_idx = int(schedule[i].item())

        if proto_idx < 0 or proto_idx >= len(protocol_table):
            raise ValueError(
                f"Invalid protocol index {proto_idx} at layer {i}; "
                f"valid range is [0, {len(protocol_table) - 1}]"
            )

        apply_protocol_to_layer(layer, protocol_table[proto_idx])

    clear_gccd_candidate_caches(model)

    if "clear_dcsfc_runtime_state" in globals():
        clear_dcsfc_runtime_state(model)


def schedule_to_names(schedule: torch.Tensor, protocol_table: List[ProtocolSpec]) -> List[str]:
    if not torch.is_tensor(schedule):
        schedule = torch.tensor(schedule, dtype=torch.long)

    schedule = schedule.detach().cpu().long().view(-1)

    names = []
    for i in schedule.tolist():
        proto_idx = int(i)
        if proto_idx < 0 or proto_idx >= len(protocol_table):
            raise ValueError(
                f"Invalid protocol index {proto_idx}; valid range is [0, {len(protocol_table) - 1}]"
            )
        names.append(protocol_table[proto_idx].name)

    return names


@torch.no_grad()
def calibrate_protocol_costs_from_probe(
    model,
    probe_batch,
    protocol_specs,
    routing_seq_len=None,
):
    if routing_seq_len is not None:
        seq_len = int(routing_seq_len)
    else:
        if "get_batch_seq_len" in globals():
            seq_len = int(get_batch_seq_len(probe_batch))
        else:
            seq_len = infer_seq_len_from_batch(probe_batch)

    if seq_len <= 0:
        raise ValueError("routing sequence length must be > 0")

    if len(protocol_specs) == 0:
        raise ValueError("protocol_specs must be non-empty.")

    if "sanitize_protocol_table_for_seq_len" in globals():
        safe_specs = sanitize_protocol_table_for_seq_len(protocol_specs, seq_len)
    else:
        safe_specs = ensure_protocol_costs(protocol_specs, seq_len)

    if "measure_effective_budget_from_batch" not in globals():
        calibrated = []
        for p in safe_specs:
            nominal_cost = float(nominal_protocol_capacity(p, seq_len))
            calibrated.append(replace(p, cost=nominal_cost))
        return calibrated

    calibrated = []
    for p in safe_specs:
        stats = measure_effective_budget_from_batch(
            probe_batch,
            w=int(p.w),
            num_anchors=int(p.num_anchors),
            include_self=bool(p.include_self),
            scaffold_mode=int(p.scaffold_mode),
            ring_inner=int(p.ring_inner),
            ring_outer=int(p.ring_outer),
            anchor_strategy=int(p.anchor_strategy),
            anchor_param0=int(p.anchor_param0),
        )

        if "keys_per_query_mean" not in stats:
            raise KeyError("measure_effective_budget_from_batch(...) must return 'keys_per_query_mean'.")

        base_cost = float(stats["keys_per_query_mean"])
        if not math.isfinite(base_cost) or base_cost < 0.0:
            raise ValueError(f"Invalid calibrated base_cost: {base_cost}")

        lcha_mult = 1.0
        if bool(getattr(p, "lcha_enabled", False)) and int(getattr(p, "lcha_order", 0)) > 0:
            heads_frac = float(getattr(p, "lcha_heads_frac", 0.0))
            order = int(getattr(p, "lcha_order", 0))
            use_v = bool(getattr(p, "lcha_use_v", False))

            if not math.isfinite(heads_frac) or heads_frac < 0.0:
                raise ValueError(f"Invalid lcha_heads_frac for protocol {p.name}: {heads_frac}")

            # Small additive compute multiplier for harmonic prefilter.
            lcha_mult += 0.10 * heads_frac * order
            if use_v:
                lcha_mult += 0.05 * heads_frac * order

        calibrated_cost = base_cost * lcha_mult
        if not math.isfinite(calibrated_cost) or calibrated_cost < 0.0:
            raise ValueError(f"Invalid calibrated cost for protocol {p.name}: {calibrated_cost}")

        calibrated.append(replace(p, cost=calibrated_cost))

    return calibrated

In [ ]:
# ============================================================
# CELL 3: Layer summaries + deficits + usage history + budget profile
# Custom GCCD decoder path
# ============================================================
import math
from typing import Optional, Tuple, Dict, Any

import torch


def canonical_token_mask(attention_mask, batch_size: int, seqlen: int, device) -> torch.Tensor:
    batch_size = int(batch_size)
    seqlen = int(seqlen)

    if batch_size <= 0:
        raise ValueError("batch_size must be > 0")
    if seqlen <= 0:
        raise ValueError("seqlen must be > 0")

    if attention_mask is None:
        return torch.ones((batch_size, seqlen), dtype=torch.bool, device=device)

    if not torch.is_tensor(attention_mask):
        raise TypeError("attention_mask must be a torch.Tensor or None")

    m = attention_mask.to(device=device)

    if m.shape[0] != batch_size:
        raise ValueError(
            f"attention_mask batch mismatch: got {m.shape[0]}, expected {batch_size}"
        )

    if m.dim() == 2:
        # [B, S]
        if m.shape[1] < seqlen:
            raise ValueError(
                f"attention_mask sequence length too short: got {m.shape[1]}, expected at least {seqlen}"
            )
        m2 = m[:, :seqlen]

    elif m.dim() == 3 and m.size(1) == 1:
        # [B, 1, S]
        if m.shape[2] < seqlen:
            raise ValueError(
                f"attention_mask sequence length too short: got {m.shape[2]}, expected at least {seqlen}"
            )
        m2 = m[:, 0, :seqlen]

    elif m.dim() == 4 and m.size(1) == 1:
        # [B, 1, 1, S] or [B, 1, Q, S]
        if m.shape[-1] < seqlen:
            raise ValueError(
                f"attention_mask key length too short: got {m.shape[-1]}, expected at least {seqlen}"
            )

        # For [B,1,Q,S], use the final query row as the token/key visibility profile.
        if m.size(2) >= 1:
            m2 = m[:, 0, -1, :seqlen]
        else:
            raise ValueError("attention_mask query dimension must be non-empty")

    else:
        raise ValueError(
            f"Unsupported attention_mask shape {tuple(m.shape)}. "
            "Expected [B,S], [B,1,S], [B,1,1,S], or [B,1,Q,S]."
        )

    if m2.shape != (batch_size, seqlen):
        raise ValueError(
            f"canonical token mask shape mismatch: got {tuple(m2.shape)}, "
            f"expected {(batch_size, seqlen)}"
        )

    if not m2.dtype.is_floating_point:
        return m2.bool()

    # Additive mask convention: 0 = keep, -inf/negative = mask.
    if not torch.isfinite(m2).all():
        return torch.isfinite(m2)

    mn = float(m2.min().item())
    mx = float(m2.max().item())

    # Binary/probability mask convention: 1 = keep, 0 = mask.
    if mn >= 0.0 and mx <= 1.0:
        return m2 > 0.0

    # Additive finite mask convention: 0 = keep, negative = mask.
    if mx <= 0.0:
        return m2 == 0.0

    return m2 > 0.0


@torch.no_grad()
def collect_layer_summaries_from_hidden_states(
    hidden_states: Tuple[torch.Tensor, ...],
    attention_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    if hidden_states is None or len(hidden_states) == 0:
        raise ValueError("hidden_states must be a non-empty tuple/list of tensors")

    hs = list(hidden_states)

    # If embedding state is included, drop it and keep decoder-layer states only.
    if len(hs) >= 2:
        hs = hs[1:]

    if len(hs) == 0:
        raise ValueError("No layer hidden states available after removing embedding state")

    if not torch.is_tensor(hs[0]) or hs[0].dim() != 3:
        raise ValueError(f"Each hidden state must be [B,T,H], got {type(hs[0])} / {getattr(hs[0], 'shape', None)}")

    B, T, H = hs[0].shape
    device = hs[0].device
    dtype = hs[0].dtype

    if B <= 0 or T <= 0 or H <= 0:
        raise ValueError(f"Invalid hidden state shape: {tuple(hs[0].shape)}")

    for i, h in enumerate(hs):
        if not torch.is_tensor(h):
            raise TypeError(f"hidden_states[{i}] is not a tensor")
        if h.shape != hs[0].shape:
            raise ValueError(
                f"hidden_states[{i}] shape mismatch: got {tuple(h.shape)}, expected {tuple(hs[0].shape)}"
            )
        if h.device != device:
            raise ValueError("All hidden states must be on the same device")
        if not torch.isfinite(h.detach().float()).all():
            raise ValueError(f"hidden_states[{i}] contains NaN/Inf")

    token_mask = canonical_token_mask(attention_mask, B, T, device).to(dtype=torch.float32)
    denom = token_mask.sum().clamp_min(1.0)

    summaries = []
    for h in hs:
        hf = h.float()
        s = (hf * token_mask[..., None]).sum(dim=(0, 1)) / denom
        summaries.append(s)

    return torch.stack(summaries, dim=0)


def build_default_deficits(layer_summaries: torch.Tensor) -> torch.Tensor:
    if not torch.is_tensor(layer_summaries):
        raise TypeError("layer_summaries must be a torch.Tensor")
    if layer_summaries.dim() != 2:
        raise ValueError(f"layer_summaries must be [L,H], got {tuple(layer_summaries.shape)}")
    if layer_summaries.shape[0] <= 0 or layer_summaries.shape[1] <= 0:
        raise ValueError("layer_summaries must have nonzero [L,H]")
    if not torch.isfinite(layer_summaries.detach().float()).all():
        raise ValueError("layer_summaries contains NaN/Inf")

    x = safe_normalize(layer_summaries.float(), dim=-1)
    L = x.size(0)

    drift_prev = torch.zeros(L, device=x.device)
    if L > 1:
        drift_prev[1:] = 1.0 - (x[1:] * x[:-1]).sum(dim=-1)

    drift_first = 1.0 - (x * x[0:1]).sum(dim=-1)
    depth_frac = torch.linspace(0.0, 1.0, L, device=x.device)

    local_need = 1.0 - depth_frac
    global_need = depth_frac
    refresh_need = 0.5 * drift_prev + 0.5 * drift_first
    hop_need = 1.0 - torch.exp(-2.0 * depth_frac)

    deficits = torch.stack(
        [local_need, global_need, refresh_need, hop_need, drift_prev, drift_first],
        dim=-1,
    )

    if not torch.isfinite(deficits).all():
        raise ValueError("deficits contains NaN/Inf")

    return deficits


def build_usage_history(
    prev_schedule: Optional[torch.Tensor],
    num_protocols: int,
    num_layers: int,
    device,
) -> torch.Tensor:
    num_protocols = int(num_protocols)
    num_layers = int(num_layers)

    if num_protocols <= 0:
        raise ValueError("num_protocols must be > 0")
    if num_layers <= 0:
        raise ValueError("num_layers must be > 0")

    hist = torch.zeros((num_layers, num_protocols), dtype=torch.float32, device=device)

    if prev_schedule is None:
        return hist

    if not torch.is_tensor(prev_schedule):
        prev_schedule = torch.tensor(prev_schedule, dtype=torch.long, device=device)
    else:
        prev_schedule = prev_schedule.to(device=device)

    prev_schedule = prev_schedule.long().view(-1)

    counts = torch.zeros(num_protocols, dtype=torch.float32, device=device)

    for l in range(num_layers):
        hist[l] = counts

        if l < prev_schedule.numel():
            proto_idx = int(prev_schedule[l].item())
            if proto_idx < 0 or proto_idx >= num_protocols:
                raise ValueError(
                    f"prev_schedule has invalid protocol index {proto_idx} at layer {l}; "
                    f"valid range is [0, {num_protocols - 1}]"
                )
            counts[proto_idx] += 1.0

    return hist


def build_remaining_budget_profile(
    total_budget: float,
    proto_cost: torch.Tensor,
    prev_schedule: Optional[torch.Tensor],
    num_layers: int,
    device,
) -> torch.Tensor:
    total_budget = float(total_budget)
    num_layers = int(num_layers)

    if not math.isfinite(total_budget) or total_budget < 0.0:
        raise ValueError("total_budget must be finite and >= 0")
    if num_layers <= 0:
        raise ValueError("num_layers must be > 0")
    if not torch.is_tensor(proto_cost):
        proto_cost = torch.tensor(proto_cost, dtype=torch.float32, device=device)
    else:
        proto_cost = proto_cost.to(device=device, dtype=torch.float32)

    proto_cost = proto_cost.view(-1)

    if proto_cost.numel() == 0:
        raise ValueError("proto_cost must be non-empty")
    if not torch.isfinite(proto_cost).all():
        raise ValueError("proto_cost contains NaN/Inf")
    if (proto_cost < 0).any():
        raise ValueError("proto_cost must be non-negative")

    rem = torch.empty(num_layers, dtype=torch.float32, device=device)
    spent = 0.0

    if prev_schedule is None:
        rem.fill_(total_budget)
        return rem

    if not torch.is_tensor(prev_schedule):
        prev_schedule = torch.tensor(prev_schedule, dtype=torch.long, device=device)
    else:
        prev_schedule = prev_schedule.to(device=device)

    prev_schedule = prev_schedule.long().view(-1)

    for l in range(num_layers):
        rem[l] = max(total_budget - spent, 0.0)

        if l < prev_schedule.numel():
            proto_idx = int(prev_schedule[l].item())
            if proto_idx < 0 or proto_idx >= proto_cost.numel():
                raise ValueError(
                    f"prev_schedule has invalid protocol index {proto_idx} at layer {l}; "
                    f"valid range is [0, {proto_cost.numel() - 1}]"
                )
            spent += float(proto_cost[proto_idx].item())

    return rem

In [ ]:
# ============================================================
# DCSLA / landmark-anchor integration for custom GCCD decoder
#   - Keeps trainable selector/projector/score-head definitions
#   - Uses custom GCCDSparseDecoderAttention layers
#   - Injects anchors through candidate_builder_fn
#   - No Mistral-specific wiring/comments/errors
# ============================================================
import math
from dataclasses import dataclass
from typing import Optional, Dict, Tuple, List, Any

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Config
# ============================================================
@dataclass
class DCSLAConfig:
    block_size: int = 64
    heat_tau: float = 0.7

    alpha_heat: float = 1.0
    beta_bridge: float = 0.5
    gamma_deficit: float = 1.0
    delta_redundancy: float = 0.35

    landmark_dropout: float = 0.0
    use_layernorm: bool = True

    causal_only: bool = True
    refresh_every: int = 1

    util_tau_min: float = 0.20
    lambda_util: float = 1e-2
    lambda_div: float = 1e-3
    lambda_entropy: float = 1e-3

    soft_select_temp: float = 0.35
    residual_score_scale: float = 0.50

    def __post_init__(self):
        self.block_size = int(self.block_size)
        self.refresh_every = int(self.refresh_every)

        if self.block_size <= 0:
            raise ValueError("DCSLAConfig.block_size must be > 0")
        if self.refresh_every <= 0:
            raise ValueError("DCSLAConfig.refresh_every must be > 0")
        if not math.isfinite(float(self.heat_tau)) or float(self.heat_tau) < 0.0:
            raise ValueError("DCSLAConfig.heat_tau must be finite and >= 0")
        if not (0.0 <= float(self.landmark_dropout) < 1.0):
            raise ValueError("DCSLAConfig.landmark_dropout must be in [0, 1)")
        if not math.isfinite(float(self.soft_select_temp)) or float(self.soft_select_temp) <= 0.0:
            raise ValueError("DCSLAConfig.soft_select_temp must be finite and > 0")


# ============================================================
# Custom decoder layer discovery / cache clearing
# ============================================================
def _is_custom_sparse_attention_module(module: nn.Module) -> bool:
    cls_name = module.__class__.__name__

    if cls_name == "GCCDSparseDecoderAttention":
        return True

    required_attrs = (
        "candidate_builder_fn",
        "stagee_apply_fn",
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    )
    return all(hasattr(module, a) for a in required_attrs)


def find_custom_sparse_attention_layers(model) -> List[Tuple[int, nn.Module]]:
    """
    Finds custom GCCD sparse decoder attention layers.
    """
    if model is None:
        raise ValueError("model must not be None")

    out: List[Tuple[int, nn.Module]] = []

    if "_get_decoder_layers_any_wrapper" in globals():
        blocks = _get_decoder_layers_any_wrapper(model)
        for li, blk in enumerate(blocks):
            sa = getattr(blk, "self_attn", None)
            if sa is not None and _is_custom_sparse_attention_module(sa):
                out.append((li, sa))

        if out:
            return out

    idx = 0
    for _, module in model.named_modules():
        if _is_custom_sparse_attention_module(module):
            out.append((idx, module))
            idx += 1

    return out


# Compatibility alias for older notebook cells that still call find_sasa_layers.
# It now resolves custom GCCD sparse decoder attention layers.
def find_sasa_layers(model) -> List[Tuple[int, nn.Module]]:
    return find_custom_sparse_attention_layers(model)


def clear_dcsla_runtime_state(model) -> None:
    """
    Clears input-dependent landmark-anchor runtime payload from custom GCCD attention layers.
    Call this before pilot forward and after any schedule change.
    """
    for _, layer in find_custom_sparse_attention_layers(model):
        layer._dcsla_anchor_token_idx = None
        layer._dcsla_landmarks = None
        layer._dcsla_aux = None


def clear_custom_candidate_caches(model) -> None:
    """
    Clears custom decoder candidate caches after schedule/anchor changes.
    """
    for _, layer in find_custom_sparse_attention_layers(model):
        for cache_name in ("shared_cand_cache", "candidate_cache", "cand_cache"):
            cache = getattr(layer, cache_name, None)
            if cache is not None and hasattr(cache, "clear"):
                cache.clear()


# Compatibility alias for older notebook cells.
def clear_sasa_candidate_caches(model) -> None:
    clear_custom_candidate_caches(model)


# ============================================================
# Pilot forward
# ============================================================
@torch.no_grad()
def _forward_hidden_states_pilot(
    model,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
    model_forward_kwargs: Optional[Dict] = None,
):
    """
    Pilot forward used only for runtime landmark refresh.
    Runs in eval mode for stability, then restores original mode.
    """
    if model is None:
        raise ValueError("model must not be None")
    if not torch.is_tensor(input_ids):
        raise TypeError("input_ids must be a torch.Tensor")

    model_forward_kwargs = model_forward_kwargs or {}
    was_training = bool(model.training)

    model.eval()
    try:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
            **model_forward_kwargs,
        )
    finally:
        model.train(was_training)

    return outputs


def _get_hidden_states_from_outputs(outputs):
    if hasattr(outputs, "hidden_states"):
        return outputs.hidden_states
    if isinstance(outputs, dict) and "hidden_states" in outputs:
        return outputs["hidden_states"]
    raise RuntimeError("Model output does not contain hidden_states.")


# ============================================================
# Core DCSLA math helpers
# ============================================================
def _dcsla_safe_normalize(x: torch.Tensor, dim: int = -1, eps: float = 1e-6) -> torch.Tensor:
    if not torch.is_tensor(x):
        raise TypeError("x must be a torch.Tensor")
    if eps <= 0.0:
        raise ValueError("eps must be > 0")
    if not torch.isfinite(x.detach().float()).all():
        raise ValueError("_dcsla_safe_normalize input contains NaN/Inf")
    return x / x.norm(dim=dim, keepdim=True).clamp_min(eps)


def _dcsla_block_pool(
    h: torch.Tensor,            # [B,T,H]
    token_mask: torch.Tensor,   # [B,T] bool
    block_size: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
      pooled_blocks: [B,Nb,H]
      block_valid  : [B,Nb] bool
    """
    if h.dim() != 3:
        raise ValueError(f"h must be [B,T,H], got {tuple(h.shape)}")
    if token_mask.dim() != 2:
        raise ValueError(f"token_mask must be [B,T], got {tuple(token_mask.shape)}")

    B, T, H = h.shape
    if token_mask.shape != (B, T):
        raise ValueError(f"token_mask shape {tuple(token_mask.shape)} does not match [B,T]={(B,T)}")

    bs = int(block_size)
    if bs <= 0:
        raise ValueError("block_size must be > 0")

    Nb = (T + bs - 1) // bs

    pooled = []
    valid = []

    token_mask = token_mask.to(device=h.device).bool()

    for bi in range(Nb):
        s = bi * bs
        e = min((bi + 1) * bs, T)

        hb = h[:, s:e, :]
        mb = token_mask[:, s:e].float()

        denom = mb.sum(dim=1, keepdim=True).clamp_min(1.0)
        pb = (hb.float() * mb[..., None]).sum(dim=1) / denom
        pb = pb.to(dtype=h.dtype)

        vb = mb.sum(dim=1) > 0.0

        pooled.append(pb)
        valid.append(vb)

    pooled = torch.stack(pooled, dim=1)  # [B,Nb,H]
    valid = torch.stack(valid, dim=1)    # [B,Nb]
    return pooled, valid


class DCSLALandmarkProjector(nn.Module):
    """
    Landmark projector:
      a_b = Pool(H_b) + MLP(LN(Pool(H_b)))
    """
    def __init__(self, hidden_size: int, dropout: float = 0.0, use_layernorm: bool = True):
        super().__init__()

        hidden_size = int(hidden_size)
        if hidden_size <= 0:
            raise ValueError("hidden_size must be > 0")
        if not (0.0 <= float(dropout) < 1.0):
            raise ValueError("dropout must be in [0, 1)")

        self.norm = nn.LayerNorm(hidden_size) if use_layernorm else nn.Identity()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(float(dropout)),
            nn.Linear(hidden_size, hidden_size),
        )

    def forward(self, pooled_blocks: torch.Tensor) -> torch.Tensor:
        x = self.norm(pooled_blocks)
        return pooled_blocks + self.mlp(x)


def _dcsla_normalized_laplacian(W: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    if W.dim() != 2 or W.shape[0] != W.shape[1]:
        raise ValueError(f"W must be square [N,N], got {tuple(W.shape)}")
    if eps <= 0.0:
        raise ValueError("eps must be > 0")

    deg = W.sum(dim=-1).clamp_min(eps)
    inv_sqrt = deg.rsqrt()
    I = torch.eye(W.size(-1), dtype=W.dtype, device=W.device)
    return I - (inv_sqrt[:, None] * W * inv_sqrt[None, :])


@torch.no_grad()
def _dcsla_heat_centrality(L: torch.Tensor, tau: float) -> torch.Tensor:
    if L.dim() != 2 or L.shape[0] != L.shape[1]:
        raise ValueError(f"L must be square [N,N], got {tuple(L.shape)}")
    if not math.isfinite(float(tau)) or float(tau) < 0.0:
        raise ValueError("tau must be finite and >= 0")

    evals, evecs = torch.linalg.eigh(L.float())
    heat = (evecs.pow(2) * torch.exp(-float(tau) * evals)[None, :]).sum(dim=1)
    return heat.to(dtype=L.dtype)


@torch.no_grad()
def _dcsla_bridge_score(W: torch.Tensor) -> torch.Tensor:
    """
    Cheap bridge proxy:
      degree - two-hop redundancy
    """
    Wn = _dcsla_safe_normalize(W.float(), dim=-1)
    two_hop = Wn @ Wn
    return (W.float().sum(dim=-1) - two_hop.sum(dim=-1)).clamp_min(0.0).to(dtype=W.dtype)


# ============================================================
# DCSLA selector
# ============================================================
class DCSLASelector(nn.Module):
    """
    Two paths:
      1) Runtime path:
         hard landmark block selection -> representative token indices -> candidate injection
      2) Training path:
         differentiable soft-anchor summaries -> aux loss for projector + score head
    """
    def __init__(self, hidden_size: int, cfg: DCSLAConfig):
        super().__init__()

        hidden_size = int(hidden_size)
        if hidden_size <= 0:
            raise ValueError("hidden_size must be > 0")

        self.cfg = cfg
        self.hidden_size = hidden_size
        self.deficit_dim = 6

        self.projector = DCSLALandmarkProjector(
            hidden_size=hidden_size,
            dropout=float(cfg.landmark_dropout),
            use_layernorm=bool(cfg.use_layernorm),
        )

        self.deficit_proj = nn.Sequential(
            nn.Linear(self.deficit_dim, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
        )

        score_in_dim = hidden_size + hidden_size + 5
        self.score_head = nn.Sequential(
            nn.Linear(score_in_dim, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, 1),
        )

    @torch.no_grad()
    def _build_block_graph(
        self,
        landmarks_b: torch.Tensor,    # [Nb,H]
        block_valid_b: torch.Tensor,  # [Nb]
    ) -> Dict[str, torch.Tensor]:
        if landmarks_b.dim() != 2:
            raise ValueError(f"landmarks_b must be [Nb,H], got {tuple(landmarks_b.shape)}")
        if block_valid_b.dim() != 1 or block_valid_b.shape[0] != landmarks_b.shape[0]:
            raise ValueError("block_valid_b must be [Nb] and match landmarks_b")

        Nb = landmarks_b.size(0)
        if Nb <= 0:
            raise ValueError("Number of blocks must be > 0")

        block_valid_b = block_valid_b.to(device=landmarks_b.device).bool()

        lm = _dcsla_safe_normalize(landmarks_b.float(), dim=-1)
        W = (lm @ lm.T).clamp_min(0.0)

        valid2 = block_valid_b[:, None] & block_valid_b[None, :]
        W = W * valid2.float()
        W.fill_diagonal_(1.0)

        L = _dcsla_normalized_laplacian(W)
        heat = _dcsla_heat_centrality(L, tau=self.cfg.heat_tau).float()
        bridge = _dcsla_bridge_score(W).float()

        pos = torch.linspace(0.0, 1.0, Nb, device=landmarks_b.device)

        if bool(block_valid_b.any().item()):
            g_raw = (landmarks_b.float() * block_valid_b[:, None].float()).sum(dim=0, keepdim=True)
            g = _dcsla_safe_normalize(g_raw, dim=-1).squeeze(0)
        else:
            g = torch.zeros((landmarks_b.shape[-1],), dtype=torch.float32, device=landmarks_b.device)

        novelty_global = (1.0 - (lm * g[None, :]).sum(dim=-1)).clamp_min(0.0)
        novelty_prev = torch.zeros(Nb, dtype=torch.float32, device=landmarks_b.device)
        if Nb > 1:
            novelty_prev[1:] = (1.0 - (lm[1:] * lm[:-1]).sum(dim=-1)).clamp_min(0.0)

        return {
            "W": W,
            "L": L,
            "heat": heat,
            "bridge": bridge,
            "pos": pos,
            "novelty_global": novelty_global,
            "novelty_prev": novelty_prev,
        }

    @torch.no_grad()
    def _base_deficit_match(self, deficit_l: torch.Tensor, feats: Dict[str, torch.Tensor]) -> torch.Tensor:
        if deficit_l.numel() < self.deficit_dim:
            raise ValueError(f"deficit_l must have at least {self.deficit_dim} values")

        d = deficit_l.float().view(-1)
        pos = feats["pos"]
        heat = feats["heat"]
        bridge = feats["bridge"]
        novelty_prev = feats["novelty_prev"]
        novelty_global = feats["novelty_global"]

        local_pref = d[0] * (1.0 - pos)
        global_pref = d[1] * heat
        refresh_pref = d[2] * novelty_prev
        hop_pref = d[3] * bridge
        drift_prev = d[4] * novelty_prev
        drift_first = d[5] * novelty_global

        return local_pref + global_pref + refresh_pref + hop_pref + drift_prev + drift_first

    def _learned_residual_score(
        self,
        landmarks_b: torch.Tensor,    # [Nb,H]
        deficit_l: torch.Tensor,      # [6]
        feats: Dict[str, torch.Tensor],
    ) -> torch.Tensor:
        Nb = landmarks_b.size(0)

        deficit_l = deficit_l.to(device=landmarks_b.device).float().view(-1)
        if deficit_l.numel() < self.deficit_dim:
            raise ValueError(f"deficit_l must have at least {self.deficit_dim} values")

        d_emb = self.deficit_proj(deficit_l[:self.deficit_dim].unsqueeze(0)).expand(Nb, -1)

        scalar_feats = torch.stack(
            [
                feats["heat"].to(device=landmarks_b.device).float(),
                feats["bridge"].to(device=landmarks_b.device).float(),
                feats["pos"].to(device=landmarks_b.device).float(),
                feats["novelty_prev"].to(device=landmarks_b.device).float(),
                feats["novelty_global"].to(device=landmarks_b.device).float(),
            ],
            dim=-1,
        )

        score_in = torch.cat([landmarks_b.float(), d_emb, scalar_feats], dim=-1)
        residual = self.score_head(score_in).squeeze(-1)
        return float(self.cfg.residual_score_scale) * residual

    def _combined_scores(
        self,
        landmarks_b: torch.Tensor,    # [Nb,H]
        block_valid_b: torch.Tensor,  # [Nb]
        deficit_l: torch.Tensor,      # [6]
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        block_valid_b = block_valid_b.to(device=landmarks_b.device).bool()

        with torch.no_grad():
            feats = self._build_block_graph(landmarks_b.detach(), block_valid_b)
            base = (
                float(self.cfg.alpha_heat) * feats["heat"]
                + float(self.cfg.beta_bridge) * feats["bridge"]
                + float(self.cfg.gamma_deficit) * self._base_deficit_match(deficit_l, feats)
            )

        residual = self._learned_residual_score(landmarks_b, deficit_l, feats)
        raw = base.to(device=landmarks_b.device, dtype=residual.dtype) + residual
        raw = raw.masked_fill(~block_valid_b, -1e9)

        if not torch.isfinite(raw[block_valid_b]).all() if bool(block_valid_b.any().item()) else False:
            raise ValueError("DCSLA raw scores contain NaN/Inf on valid blocks")

        return raw, feats

    @torch.no_grad()
    def _greedy_select(
        self,
        landmarks_b: torch.Tensor,    # [Nb,H]
        raw_score: torch.Tensor,      # [Nb]
        block_valid_b: torch.Tensor,  # [Nb]
        A: int,
    ) -> torch.Tensor:
        A = min(int(A), int(block_valid_b.sum().item()), int(raw_score.numel()))
        if A <= 0:
            return torch.empty(0, dtype=torch.long, device=raw_score.device)

        sim = _dcsla_safe_normalize(landmarks_b.float(), dim=-1) @ _dcsla_safe_normalize(landmarks_b.float(), dim=-1).T
        sim = sim.clamp(-1.0, 1.0)

        chosen = []
        redundancy_pen = torch.zeros_like(raw_score)
        score = raw_score.masked_fill(~block_valid_b, -1e9)

        for _ in range(A):
            s = score - float(self.cfg.delta_redundancy) * redundancy_pen
            idx = int(torch.argmax(s).item())

            if not torch.isfinite(s[idx]):
                break

            chosen.append(idx)
            score[idx] = -1e9
            redundancy_pen = torch.maximum(redundancy_pen, sim[idx].to(redundancy_pen.dtype).clamp_min(0.0))

        if len(chosen) == 0:
            return torch.empty(0, dtype=torch.long, device=raw_score.device)

        return torch.tensor(chosen, dtype=torch.long, device=raw_score.device)

    @torch.no_grad()
    def select_for_layer(
        self,
        hidden_states: torch.Tensor,  # [B,T,H]
        token_mask: torch.Tensor,     # [B,T] bool
        deficit_l: torch.Tensor,      # [6]
        num_landmarks: int,
    ) -> Dict[str, torch.Tensor]:
        """
        Runtime hard-selection path used by candidate injection.
        """
        if hidden_states.dim() != 3:
            raise ValueError(f"hidden_states must be [B,T,H], got {tuple(hidden_states.shape)}")

        B, T, H = hidden_states.shape
        bs = int(self.cfg.block_size)

        if token_mask.shape != (B, T):
            raise ValueError(f"token_mask shape {tuple(token_mask.shape)} does not match {(B, T)}")

        token_mask = token_mask.to(device=hidden_states.device).bool()

        pooled, block_valid = _dcsla_block_pool(hidden_states, token_mask, bs)
        landmarks = self.projector(pooled)

        A = max(int(num_landmarks), 0)

        anchor_tok = torch.full((B, A), -1, dtype=torch.long, device=hidden_states.device)
        anchor_blk = torch.full((B, A), -1, dtype=torch.long, device=hidden_states.device)
        anchor_lm = torch.zeros((B, A, H), dtype=hidden_states.dtype, device=hidden_states.device)

        util_proxy = []
        div_proxy = []

        for b in range(B):
            raw_score, _ = self._combined_scores(
                landmarks_b=landmarks[b],
                block_valid_b=block_valid[b],
                deficit_l=deficit_l,
            )

            idx = self._greedy_select(
                landmarks_b=landmarks[b],
                raw_score=raw_score,
                block_valid_b=block_valid[b],
                A=A,
            )

            n = min(A, int(idx.numel()))
            if n == 0:
                util_proxy.append(torch.zeros((), device=hidden_states.device))
                div_proxy.append(torch.zeros((), device=hidden_states.device))
                continue

            rep_tok = []
            for ii in idx[:n].tolist():
                s = int(ii) * bs
                e = min((int(ii) + 1) * bs, T)
                center = min(s + bs // 2, T - 1)

                if bool(token_mask[b, center]):
                    rep_tok.append(center)
                else:
                    block_tv = token_mask[b, s:e]
                    if bool(block_tv.any().item()):
                        rep_tok.append(int(s + torch.argmax(block_tv.float()).item()))
                    else:
                        rep_tok.append(-1)

            rep_tok = torch.tensor(rep_tok, dtype=torch.long, device=hidden_states.device)

            anchor_tok[b, :n] = rep_tok
            anchor_blk[b, :n] = idx[:n]
            anchor_lm[b, :n] = landmarks[b, idx[:n]].to(dtype=hidden_states.dtype)

            q_valid = token_mask[b]
            if bool(q_valid.any().item()):
                qn = _dcsla_safe_normalize(hidden_states[b, q_valid].float(), dim=-1)
                an = _dcsla_safe_normalize(anchor_lm[b, :n].float(), dim=-1)
                sim = qn @ an.T
                util_proxy.append(sim.max(dim=-1).values.mean())

                if n > 1:
                    div = torch.exp((an @ an.T).clamp(max=20.0))
                    div = (div.sum() - torch.diagonal(div).sum()) / max(1, n * (n - 1))
                    div_proxy.append(div)
                else:
                    div_proxy.append(torch.zeros((), device=hidden_states.device))
            else:
                util_proxy.append(torch.zeros((), device=hidden_states.device))
                div_proxy.append(torch.zeros((), device=hidden_states.device))

        return {
            "anchor_token_idx": anchor_tok,
            "anchor_block_idx": anchor_blk,
            "landmarks": anchor_lm,
            "aux": {
                "util_proxy": torch.stack(util_proxy).mean() if len(util_proxy) else torch.zeros((), device=hidden_states.device),
                "div_proxy": torch.stack(div_proxy).mean() if len(div_proxy) else torch.zeros((), device=hidden_states.device),
            },
        }

    def trainable_aux_for_layer(
        self,
        hidden_states: torch.Tensor,  # [B,T,H]
        token_mask: torch.Tensor,     # [B,T] bool
        deficit_l: torch.Tensor,      # [6]
        num_landmarks: int,
    ) -> Dict[str, torch.Tensor]:
        """
        Differentiable training path.
        Gives gradients to projector, deficit projection, and learned score head.
        """
        if hidden_states.dim() != 3:
            raise ValueError(f"hidden_states must be [B,T,H], got {tuple(hidden_states.shape)}")

        B, T, H = hidden_states.shape
        bs = int(self.cfg.block_size)
        A = max(int(num_landmarks), 0)

        if token_mask.shape != (B, T):
            raise ValueError(f"token_mask shape {tuple(token_mask.shape)} does not match {(B, T)}")

        token_mask = token_mask.to(device=hidden_states.device).bool()

        if A == 0:
            z = hidden_states.new_zeros(())
            return {"loss": z, "util": z, "div": z, "entropy": z}

        pooled, block_valid = _dcsla_block_pool(hidden_states, token_mask, bs)
        landmarks = self.projector(pooled)

        util_list = []
        div_list = []
        ent_list = []

        for b in range(B):
            valid = block_valid[b]
            if int(valid.sum().item()) == 0:
                util_list.append(hidden_states.new_zeros(()))
                div_list.append(hidden_states.new_zeros(()))
                ent_list.append(hidden_states.new_zeros(()))
                continue

            raw_score, _ = self._combined_scores(
                landmarks_b=landmarks[b],
                block_valid_b=valid,
                deficit_l=deficit_l,
            )

            logits = raw_score.masked_fill(~valid, -1e9)
            used_detached = torch.zeros_like(logits)

            soft_anchors = []
            ent_terms = []

            for _ in range(A):
                step_logits = logits - float(self.cfg.delta_redundancy) * used_detached
                probs = F.softmax(step_logits / float(self.cfg.soft_select_temp), dim=-1)
                probs = probs * valid.float()
                probs = probs / probs.sum().clamp_min(1e-6)

                soft_anchor = probs @ landmarks[b].float()
                soft_anchors.append(soft_anchor)

                ent_terms.append(-(probs * probs.clamp_min(1e-8).log()).sum())
                used_detached = torch.maximum(used_detached, probs.detach())

            anchors = torch.stack(soft_anchors, dim=0)

            q_valid = token_mask[b]
            if bool(q_valid.any().item()):
                qn = _dcsla_safe_normalize(hidden_states[b, q_valid].float(), dim=-1)
                an = _dcsla_safe_normalize(anchors.float(), dim=-1)
                sim = qn @ an.T
                util = sim.max(dim=-1).values.mean()
            else:
                util = hidden_states.new_zeros(())

            if A > 1:
                an = _dcsla_safe_normalize(anchors.float(), dim=-1)
                div_mat = torch.exp((an @ an.T).clamp(max=20.0))
                div = (div_mat.sum() - torch.diagonal(div_mat).sum()) / max(1, A * (A - 1))
            else:
                div = hidden_states.new_zeros(())

            entropy = torch.stack(ent_terms).mean() if len(ent_terms) else hidden_states.new_zeros(())

            util_list.append(util)
            div_list.append(div)
            ent_list.append(entropy)

        util_mean = torch.stack(util_list).mean() if len(util_list) else hidden_states.new_zeros(())
        div_mean = torch.stack(div_list).mean() if len(div_list) else hidden_states.new_zeros(())
        ent_mean = torch.stack(ent_list).mean() if len(ent_list) else hidden_states.new_zeros(())

        loss = (
            float(self.cfg.lambda_util) * F.relu(float(self.cfg.util_tau_min) - util_mean)
            + float(self.cfg.lambda_div) * div_mean
            + float(self.cfg.lambda_entropy) * ent_mean
        )

        return {
            "loss": loss,
            "util": util_mean,
            "div": div_mean,
            "entropy": ent_mean,
        }


# ============================================================
# Candidate merge for custom decoder candidate_builder_fn
# ============================================================
@torch.no_grad()
def _merge_dcsla_anchors_into_candidates(
    cand_i32: torch.Tensor,
    mask_u8: torch.Tensor,
    token_mask_bn: torch.Tensor,
    extra_anchor_tok: torch.Tensor,
    causal_only: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Supports token-major candidate rows:
      cand_i32/mask_u8: [B*T, C]

    Candidate values are flattened token indices:
      flat_idx = batch_id * T + token_id
    """
    if cand_i32.dim() != 2 or mask_u8.dim() != 2:
        raise ValueError(
            f"cand_i32/mask_u8 must be [BT,C], got {tuple(cand_i32.shape)} / {tuple(mask_u8.shape)}"
        )
    if cand_i32.shape != mask_u8.shape:
        raise ValueError(f"cand_i32/mask_u8 shape mismatch: {tuple(cand_i32.shape)} vs {tuple(mask_u8.shape)}")
    if token_mask_bn.dim() != 2:
        raise ValueError(f"token_mask_bn must be [B,T], got {tuple(token_mask_bn.shape)}")
    if extra_anchor_tok.dim() != 2:
        raise ValueError(f"extra_anchor_tok must be [B,A], got {tuple(extra_anchor_tok.shape)}")

    B, T = token_mask_bn.shape
    BT, _ = cand_i32.shape
    A = int(extra_anchor_tok.size(1))
    device = cand_i32.device

    if BT != B * T:
        raise ValueError(
            f"Only token-major [B*T,C] candidates are supported here. "
            f"Got rows={BT}, expected B*T={B*T}."
        )

    if A == 0:
        return cand_i32.contiguous(), mask_u8.contiguous()

    token_mask_bn = token_mask_bn.to(device=device).bool()
    extra_anchor_tok = extra_anchor_tok.to(device=device, dtype=torch.long)

    row = torch.arange(BT, device=device)
    row_b = row // T
    row_t = row % T

    anchor_tok_row = extra_anchor_tok[row_b]
    valid = anchor_tok_row >= 0

    if bool(causal_only):
        valid = valid & (anchor_tok_row <= row_t[:, None])

    anchor_tok_clamped = anchor_tok_row.clamp(min=0, max=max(T - 1, 0))
    valid = valid & token_mask_bn[row_b, anchor_tok_clamped]

    anchor_flat = row_b[:, None] * T + anchor_tok_clamped

    active_existing = mask_u8.to(torch.bool)
    for a in range(A):
        dup = ((cand_i32 == anchor_flat[:, a:a + 1].to(dtype=cand_i32.dtype)) & active_existing).any(dim=1)
        valid[:, a] = valid[:, a] & (~dup)

    extra_cand = anchor_flat.to(dtype=cand_i32.dtype)
    extra_mask = valid.to(dtype=mask_u8.dtype)

    cand_i32 = torch.cat([cand_i32, extra_cand], dim=1).contiguous()
    mask_u8 = torch.cat([mask_u8, extra_mask], dim=1).contiguous()

    return cand_i32, mask_u8


def install_dcsla_on_model(model, selector: DCSLASelector):
    """
    Installs runtime landmark-anchor injection on custom GCCD sparse attention layers.

    Custom decoder wiring:
      - wraps each layer.candidate_builder_fn
      - appends selected anchor tokens to returned cand_i32/mask_u8
      - does not depend on old decoder-specific build_candidates_cuda methods
    """
    if model is None:
        raise ValueError("model must not be None")
    if selector is None:
        raise ValueError("selector must not be None")

    layers = find_custom_sparse_attention_layers(model)
    if len(layers) == 0:
        raise ValueError("No GCCDSparseDecoderAttention layers found.")

    for _, layer in layers:
        layer.dcsla_selector = selector
        layer._dcsla_anchor_token_idx = None
        layer._dcsla_landmarks = None
        layer._dcsla_aux = None

        if getattr(layer, "_dcsla_installed", False):
            continue

        orig_builder = getattr(layer, "candidate_builder_fn", None)
        if orig_builder is None:
            raise ValueError(
                "Custom GCCD attention layer has no candidate_builder_fn to wrap. "
                "Install/build the Stage-A candidate builder before installing landmark-anchor injection."
            )

        layer._orig_candidate_builder_fn = orig_builder

        def _wrapped_candidate_builder_fn(*args, _layer=layer, **kwargs):
            stagea = _layer._orig_candidate_builder_fn(*args, **kwargs)

            if not isinstance(stagea, dict):
                raise ValueError("candidate_builder_fn must return a dict.")
            if "cand_i32" not in stagea or "mask_u8" not in stagea:
                raise ValueError("candidate_builder_fn output must contain cand_i32 and mask_u8.")

            extra = getattr(_layer, "_dcsla_anchor_token_idx", None)
            if extra is None or int(extra.numel()) == 0:
                return stagea

            cand_i32 = stagea["cand_i32"]
            mask_u8 = stagea["mask_u8"]

            attention_mask = kwargs.get("attention_mask", None)
            q = kwargs.get("q", None)

            if q is None:
                raise ValueError("candidate_builder_fn wrapper requires q=... in kwargs to infer [B,T].")
            if q.dim() != 4:
                raise ValueError(f"q must be [B,T,H,D], got {tuple(q.shape)}")

            B = int(q.shape[0])
            T = int(kwargs.get("total_len", q.shape[1]))
            device = cand_i32.device

            if attention_mask is None:
                token_mask_bn = torch.ones((B, T), dtype=torch.bool, device=device)
            else:
                if "canonical_token_mask" not in globals():
                    raise NameError("canonical_token_mask is not defined. Run the layer-summary/mask helper cell first.")
                token_mask_bn = canonical_token_mask(attention_mask, B, T, device)

            cand_is_per_head = bool(stagea.get("cand_is_per_head", False))
            if cand_is_per_head:
                raise ValueError(
                    "Landmark-anchor merge currently supports token-major [B*T,C] candidates only. "
                    "Disable cand_is_per_head for this injection path or add a per-head merge path."
                )

            cand_i32, mask_u8 = _merge_dcsla_anchors_into_candidates(
                cand_i32=cand_i32,
                mask_u8=mask_u8,
                token_mask_bn=token_mask_bn,
                extra_anchor_tok=extra.to(device),
                causal_only=bool(_layer.dcsla_selector.cfg.causal_only),
            )

            stagea = dict(stagea)
            stagea["cand_i32"] = cand_i32
            stagea["mask_u8"] = mask_u8
            stagea["family_meta"] = dict(stagea.get("family_meta", {}) or {})
            stagea["family_meta"]["landmark_anchor_injected"] = True
            stagea["family_meta"]["num_landmark_anchors"] = int(extra.shape[1])

            return stagea

        layer.candidate_builder_fn = _wrapped_candidate_builder_fn
        layer._dcsla_installed = True

    clear_custom_candidate_caches(model)
    clear_dcsla_runtime_state(model)


# ============================================================
# Runtime controller
# ============================================================
class NotebookDCSLAController:
    """
    Runtime landmark-anchor refresh flow:
      1) clear stale runtime payload
      2) run pilot forward after the layer protocol schedule has been applied
      3) compute hard landmark anchors under that schedule
      4) actual trainable auxiliary loss is computed separately on real forward hidden states
    """
    def __init__(self, model, selector: DCSLASelector, refresh_every: int = 1):
        self.model = model
        self.selector = selector
        self.refresh_every = int(refresh_every)

        if self.refresh_every <= 0:
            raise ValueError("refresh_every must be > 0")

        self.step_idx = 0
        self.last_out = None

    @torch.no_grad()
    def refresh_after_schedule(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        deficits: Optional[torch.Tensor] = None,
        force: bool = False,
        model_forward_kwargs: Optional[Dict] = None,
    ) -> Dict[str, torch.Tensor]:
        if not torch.is_tensor(input_ids):
            raise TypeError("input_ids must be a torch.Tensor")

        model_forward_kwargs = model_forward_kwargs or {}

        do_refresh = bool(force or self.last_out is None or (self.step_idx % self.refresh_every == 0))
        self.step_idx += 1

        if not do_refresh:
            return self.last_out

        clear_dcsla_runtime_state(self.model)

        outputs = _forward_hidden_states_pilot(
            model=self.model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            model_forward_kwargs=model_forward_kwargs,
        )

        hidden_states_all = _get_hidden_states_from_outputs(outputs)

        hs = list(hidden_states_all)
        if len(hs) >= 2:
            hs = hs[1:]

        if len(hs) == 0:
            raise RuntimeError("Runtime landmark refresh could not read layer hidden states.")

        B, T, _ = hs[0].shape
        device = hs[0].device

        if "canonical_token_mask" not in globals():
            raise NameError("canonical_token_mask is not defined. Run the layer-summary/mask helper cell first.")
        if "collect_layer_summaries_from_hidden_states" not in globals():
            raise NameError("collect_layer_summaries_from_hidden_states is not defined.")
        if "build_default_deficits" not in globals():
            raise NameError("build_default_deficits is not defined.")

        token_mask = canonical_token_mask(attention_mask, B, T, device)

        layer_summaries = collect_layer_summaries_from_hidden_states(
            hidden_states_all,
            attention_mask=attention_mask,
        ).to(device)

        if deficits is None:
            deficits = build_default_deficits(layer_summaries)
        else:
            deficits = deficits.to(device)

        layers = find_custom_sparse_attention_layers(self.model)
        if len(layers) != len(hs):
            raise ValueError(
                f"Runtime landmark mismatch: {len(layers)} custom attention layers vs {len(hs)} hidden-state layers"
            )

        if deficits.dim() != 2 or deficits.shape[0] != len(layers):
            raise ValueError(
                f"deficits must be [L,D] with L={len(layers)}, got {tuple(deficits.shape)}"
            )

        util_terms = []
        div_terms = []

        selector_was_training = self.selector.training
        self.selector.eval()

        try:
            for li, (_, layer) in enumerate(layers):
                A = int(getattr(layer, "num_anchors", 0))

                if A <= 0:
                    layer._dcsla_anchor_token_idx = torch.empty((B, 0), dtype=torch.long, device=device)
                    layer._dcsla_landmarks = torch.empty((B, 0, hs[li].size(-1)), dtype=hs[li].dtype, device=device)
                    layer._dcsla_aux = {
                        "util_proxy": torch.zeros((), device=device),
                        "div_proxy": torch.zeros((), device=device),
                    }
                    util_terms.append(layer._dcsla_aux["util_proxy"])
                    div_terms.append(layer._dcsla_aux["div_proxy"])
                    continue

                out = self.selector.select_for_layer(
                    hidden_states=hs[li],
                    token_mask=token_mask,
                    deficit_l=deficits[li].to(device),
                    num_landmarks=A,
                )

                layer._dcsla_anchor_token_idx = out["anchor_token_idx"]
                layer._dcsla_landmarks = out["landmarks"]
                layer._dcsla_aux = out["aux"]

                util_terms.append(out["aux"]["util_proxy"])
                div_terms.append(out["aux"]["div_proxy"])

        finally:
            self.selector.train(selector_was_training)

        clear_custom_candidate_caches(self.model)

        self.last_out = {
            "util_proxy": torch.stack(util_terms).mean() if len(util_terms) else torch.zeros((), device=device),
            "div_proxy": torch.stack(div_terms).mean() if len(div_terms) else torch.zeros((), device=device),
            "deficits": deficits,
            "layer_summaries": layer_summaries,
        }

        return self.last_out


# ============================================================
# Trainable auxiliary loss
# ============================================================
def dcsla_aux_loss(
    model,
    selector: DCSLASelector,
    hidden_states_all,
    attention_mask: Optional[torch.Tensor],
    deficits: torch.Tensor,
    cfg: Optional[DCSLAConfig] = None,
):
    """
    Trainable landmark-anchor auxiliary loss computed on actual forward hidden states.

    Gives gradients to:
      - landmark projector
      - deficit projection
      - learned score head
    """
    if cfg is None:
        cfg = selector.cfg

    if hidden_states_all is None:
        ref = next(selector.parameters())
        z = torch.zeros((), dtype=ref.dtype, device=ref.device)
        return z, {"util": z, "div": z, "entropy": z}

    hs = list(hidden_states_all)
    if len(hs) >= 2:
        hs = hs[1:]

    if len(hs) == 0:
        ref = next(selector.parameters())
        z = torch.zeros((), dtype=ref.dtype, device=ref.device)
        return z, {"util": z, "div": z, "entropy": z}

    B, T, _ = hs[0].shape
    device = hs[0].device

    if "canonical_token_mask" not in globals():
        raise NameError("canonical_token_mask is not defined. Run the layer-summary/mask helper cell first.")

    token_mask = canonical_token_mask(attention_mask, B, T, device)

    layers = find_custom_sparse_attention_layers(model)
    if len(layers) != len(hs):
        raise ValueError(
            f"Landmark aux mismatch: {len(layers)} custom attention layers vs {len(hs)} hidden-state layers"
        )

    deficits = deficits.to(device)
    if deficits.dim() != 2 or deficits.shape[0] != len(layers):
        raise ValueError(
            f"deficits must be [L,D] with L={len(layers)}, got {tuple(deficits.shape)}"
        )

    loss_terms = []
    util_terms = []
    div_terms = []
    ent_terms = []

    for li, (_, layer) in enumerate(layers):
        A = int(getattr(layer, "num_anchors", 0))
        if A <= 0:
            continue

        out = selector.trainable_aux_for_layer(
            hidden_states=hs[li],
            token_mask=token_mask,
            deficit_l=deficits[li].to(device),
            num_landmarks=A,
        )

        loss_terms.append(out["loss"])
        util_terms.append(out["util"].detach())
        div_terms.append(out["div"].detach())
        ent_terms.append(out["entropy"].detach())

    if len(loss_terms) == 0:
        ref = next(selector.parameters())
        z = torch.zeros((), dtype=ref.dtype, device=ref.device)
        return z, {"util": z, "div": z, "entropy": z}

    loss = torch.stack(loss_terms).mean()

    stats = {
        "util": torch.stack(util_terms).mean() if len(util_terms) else loss.detach().new_zeros(()),
        "div": torch.stack(div_terms).mean() if len(div_terms) else loss.detach().new_zeros(()),
        "entropy": torch.stack(ent_terms).mean() if len(ent_terms) else loss.detach().new_zeros(()),
    }

    return loss, stats

In [ ]:
# ============================================================
# CELL 4: LaSPER-HT core math
# Custom GCCD decoder path
# ============================================================
from typing import Optional, Dict, Any

import math
import torch
import torch.nn.functional as F


def _lasper_check_finite_tensor(name: str, x: torch.Tensor) -> None:
    if not torch.is_tensor(x):
        raise TypeError(f"{name} must be a torch.Tensor")
    if not torch.isfinite(x.detach().float()).all():
        raise ValueError(f"{name} contains NaN/Inf")


def _lasper_check_matrix(name: str, x: torch.Tensor, rank: int = 2) -> None:
    if not torch.is_tensor(x):
        raise TypeError(f"{name} must be a torch.Tensor")
    if x.dim() != rank:
        raise ValueError(f"{name} must be rank-{rank}, got shape {tuple(x.shape)}")
    if any(int(s) <= 0 for s in x.shape):
        raise ValueError(f"{name} must have nonzero shape, got {tuple(x.shape)}")
    _lasper_check_finite_tensor(name, x)


def build_affinity_scores(
    layer_feat: torch.Tensor,         # [L,D]
    proto_feat: torch.Tensor,         # [K,D]
    proto_cost: torch.Tensor,         # [K]
    remaining_budget: torch.Tensor,   # [L]
    proto_usage_global: torch.Tensor, # [K]
    lambda_cost: float,
    mu_usage: float,
) -> torch.Tensor:
    """
    A_{lz} = <phi_l, psi_z> - lambda * cost(z) - mu * scarcity(z)
    """
    _lasper_check_matrix("layer_feat", layer_feat, rank=2)
    _lasper_check_matrix("proto_feat", proto_feat, rank=2)

    if layer_feat.shape[1] != proto_feat.shape[1]:
        raise ValueError(
            f"feature dim mismatch: layer_feat={tuple(layer_feat.shape)}, proto_feat={tuple(proto_feat.shape)}"
        )

    L = int(layer_feat.shape[0])
    K = int(proto_feat.shape[0])

    proto_cost = proto_cost.to(device=layer_feat.device, dtype=torch.float32).view(-1)
    remaining_budget = remaining_budget.to(device=layer_feat.device, dtype=torch.float32).view(-1)
    proto_usage_global = proto_usage_global.to(device=layer_feat.device, dtype=torch.float32).view(-1)

    if proto_cost.numel() != K:
        raise ValueError(f"proto_cost must have K={K} values, got {proto_cost.numel()}")
    if proto_usage_global.numel() != K:
        raise ValueError(f"proto_usage_global must have K={K} values, got {proto_usage_global.numel()}")
    if remaining_budget.numel() != L:
        raise ValueError(f"remaining_budget must have L={L} values, got {remaining_budget.numel()}")

    _lasper_check_finite_tensor("proto_cost", proto_cost)
    _lasper_check_finite_tensor("remaining_budget", remaining_budget)
    _lasper_check_finite_tensor("proto_usage_global", proto_usage_global)

    if (proto_cost < 0).any():
        raise ValueError("proto_cost must be non-negative")
    if (remaining_budget < 0).any():
        raise ValueError("remaining_budget must be non-negative")
    if not math.isfinite(float(lambda_cost)):
        raise ValueError("lambda_cost must be finite")
    if not math.isfinite(float(mu_usage)):
        raise ValueError("mu_usage must be finite")

    scores = layer_feat.float() @ proto_feat.float().T
    scores = scores - float(lambda_cost) * proto_cost[None, :]
    scores = scores - float(mu_usage) * proto_usage_global[None, :]

    feasible = proto_cost[None, :] <= remaining_budget[:, None]
    scores = scores.masked_fill(~feasible, -1e9)
    return scores


def build_layer_graph(
    layer_feat: torch.Tensor,
    knn: int = 0,
    self_loop_weight: float = 1.0,
) -> torch.Tensor:
    """
    Harmonic graph over layers only.
    """
    _lasper_check_matrix("layer_feat", layer_feat, rank=2)

    if not math.isfinite(float(self_loop_weight)):
        raise ValueError("self_loop_weight must be finite")

    x = safe_normalize(layer_feat.float(), dim=-1)
    W = (x @ x.T).clamp_min(0.0)

    L = int(W.size(0))
    knn = 0 if knn is None else int(knn)

    if knn < 0:
        raise ValueError("knn must be >= 0")

    if knn > 0 and knn < L:
        _, idx = torch.topk(W, k=knn, dim=-1)
        mask = torch.zeros_like(W, dtype=torch.bool)
        mask.scatter_(1, idx, True)
        W = W.masked_fill(~mask, 0.0)
        W = torch.maximum(W, W.T)

    if float(self_loop_weight) != 0.0:
        I = torch.eye(L, dtype=W.dtype, device=W.device)
        W = W + float(self_loop_weight) * I

    if not torch.isfinite(W).all():
        raise ValueError("layer graph W contains NaN/Inf")

    return W


def normalized_laplacian(W: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    if not torch.is_tensor(W):
        raise TypeError("W must be a torch.Tensor")
    if W.dim() != 2 or W.shape[0] != W.shape[1]:
        raise ValueError(f"W must be square [L,L], got {tuple(W.shape)}")
    if W.shape[0] <= 0:
        raise ValueError("W must be non-empty")
    if eps <= 0.0:
        raise ValueError("eps must be > 0")
    _lasper_check_finite_tensor("W", W)

    deg = W.sum(dim=1).clamp_min(eps)
    inv_sqrt = deg.rsqrt()
    I = torch.eye(W.size(0), dtype=W.dtype, device=W.device)
    L = I - (inv_sqrt[:, None] * W * inv_sqrt[None, :])

    if not torch.isfinite(L).all():
        raise ValueError("normalized_laplacian output contains NaN/Inf")

    return L


def sinkhorn_quota_assignment(
    scores: torch.Tensor,  # [L,K]
    quotas: torch.Tensor,  # [K], sums exactly to L
    tau: float = 0.35,
    n_iters: int = 50,
    eps: float = 1e-8,
) -> torch.Tensor:
    _lasper_check_matrix("scores", scores, rank=2)

    L, K = scores.shape

    quotas = quotas.float().to(scores.device).view(-1)
    if quotas.numel() != K:
        raise ValueError(f"quotas must have K={K} values, got {quotas.numel()}")
    _lasper_check_finite_tensor("quotas", quotas)

    if (quotas < 0).any():
        raise ValueError("quotas must be non-negative")
    if int(round(float(quotas.sum().item()))) != int(L):
        raise ValueError(f"quotas must sum to number of layers L={L}, got {float(quotas.sum().item())}")
    if not math.isfinite(float(tau)) or float(tau) <= 0.0:
        raise ValueError("tau must be finite and > 0")
    if int(n_iters) <= 0:
        raise ValueError("n_iters must be > 0")
    if eps <= 0.0:
        raise ValueError("eps must be > 0")

    feasible = torch.isfinite(scores) & (scores > -1e8)

    if not bool(feasible.any(dim=1).all().item()):
        bad_rows = torch.where(~feasible.any(dim=1))[0].detach().cpu().tolist()
        raise ValueError(f"sinkhorn has rows with no feasible protocols: {bad_rows[:8]}")

    positive_quota = quotas > 0
    if bool(positive_quota.any().item()):
        possible_cols = feasible.any(dim=0)
        impossible = positive_quota & (~possible_cols)
        if bool(impossible.any().item()):
            bad_cols = torch.where(impossible)[0].detach().cpu().tolist()
            raise ValueError(f"sinkhorn positive quota assigned to infeasible protocol columns: {bad_cols}")

    logits = scores.float() / float(tau)
    logits = logits.masked_fill(~feasible, -1e9)

    row_max = logits.max(dim=1, keepdim=True).values
    logits = logits - row_max

    Kmat = torch.exp(logits).masked_fill(~feasible, 0.0).clamp_min(0.0)

    u = torch.ones(L, dtype=torch.float32, device=scores.device)
    v = torch.ones(K, dtype=torch.float32, device=scores.device)

    row_target = torch.ones(L, dtype=torch.float32, device=scores.device)
    col_target = quotas.to(dtype=torch.float32)

    for _ in range(int(n_iters)):
        u = row_target / (Kmat @ v).clamp_min(eps)
        v = col_target / (Kmat.T @ u).clamp_min(eps)

    P = (u[:, None] * Kmat) * v[None, :]

    if not torch.isfinite(P).all():
        raise ValueError("sinkhorn output contains NaN/Inf")

    return P.to(dtype=scores.dtype)


def adjust_quotas_to_budget(
    proto_cost: torch.Tensor,     # [K]
    quota_ratio: torch.Tensor,    # [K]
    num_layers: int,
    total_budget: Optional[float] = None,
) -> torch.Tensor:
    """
    Quota repair under a budget target.
    """
    num_layers = int(num_layers)
    if num_layers <= 0:
        raise ValueError("num_layers must be > 0")

    proto_cost = proto_cost.float().view(-1)
    quota_ratio = quota_ratio.float().to(proto_cost.device).view(-1)

    if proto_cost.numel() == 0:
        raise ValueError("proto_cost must be non-empty")
    if quota_ratio.numel() != proto_cost.numel():
        raise ValueError(
            f"quota_ratio/proto_cost length mismatch: {quota_ratio.numel()} vs {proto_cost.numel()}"
        )

    _lasper_check_finite_tensor("proto_cost", proto_cost)
    _lasper_check_finite_tensor("quota_ratio", quota_ratio)

    if (proto_cost < 0).any():
        raise ValueError("proto_cost must be non-negative")
    if (quota_ratio < 0).any():
        raise ValueError("quota_ratio must be non-negative")

    q = quotas_from_ratios(num_layers, quota_ratio).to(proto_cost.device).float()

    if total_budget is None:
        return q

    total_budget = float(total_budget)
    if not math.isfinite(total_budget) or total_budget < 0.0:
        raise ValueError("total_budget must be finite and >= 0")

    cheapest = int(torch.argmin(proto_cost).item())
    min_possible = float(num_layers) * float(proto_cost[cheapest].item())

    if total_budget <= min_possible + 1e-6:
        q.zero_()
        q[cheapest] = float(num_layers)
        return q

    max_iters = max(16, num_layers * 8)
    for _ in range(max_iters):
        spend = float((q * proto_cost).sum().item())
        if spend <= total_budget + 1e-6:
            break

        donor_order = torch.argsort(proto_cost, descending=True)
        recv_order = torch.argsort(proto_cost, descending=False)

        moved = False
        for d in donor_order.tolist():
            if q[d] <= 0:
                continue
            for r in recv_order.tolist():
                if float(proto_cost[r].item()) + 1e-9 < float(proto_cost[d].item()):
                    q[d] -= 1.0
                    q[r] += 1.0
                    moved = True
                    break
            if moved:
                break

        if not moved:
            break

    if int(round(float(q.sum().item()))) != num_layers:
        raise RuntimeError(f"quota repair failed: sum={float(q.sum().item())}, expected={num_layers}")

    if (q < 0).any():
        raise RuntimeError("quota repair produced negative quota")

    return q


def hdtr_refine_assignments(
    local_scores: torch.Tensor,       # [L,K]
    layer_laplacian: torch.Tensor,    # [L,L]
    quotas: torch.Tensor,             # [K]
    feasible_mask: Optional[torch.Tensor] = None,
    tau: float = 0.35,
    eta: float = 0.20,
    hdtr_iters: int = 6,
    sinkhorn_iters: int = 50,
):
    """
    max_P <P, A> + tau * H(P) - eta * Tr(P^T L_LL P)
      s.t. row sums = 1, col sums = quotas
    """
    _lasper_check_matrix("local_scores", local_scores, rank=2)

    L, K = local_scores.shape

    if layer_laplacian.shape != (L, L):
        raise ValueError(
            f"layer_laplacian must be [L,L]={(L,L)}, got {tuple(layer_laplacian.shape)}"
        )

    _lasper_check_finite_tensor("layer_laplacian", layer_laplacian)

    quotas = quotas.float().to(local_scores.device).view(-1)
    if quotas.numel() != K:
        raise ValueError(f"quotas must have K={K} values, got {quotas.numel()}")

    if feasible_mask is None:
        feasible_mask = torch.isfinite(local_scores) & (local_scores > -1e8)
    else:
        feasible_mask = feasible_mask.to(device=local_scores.device).bool()

    if feasible_mask.shape != local_scores.shape:
        raise ValueError(
            f"feasible_mask shape mismatch: got {tuple(feasible_mask.shape)}, expected {tuple(local_scores.shape)}"
        )

    if not math.isfinite(float(tau)) or float(tau) <= 0.0:
        raise ValueError("tau must be finite and > 0")
    if not math.isfinite(float(eta)) or float(eta) < 0.0:
        raise ValueError("eta must be finite and >= 0")
    if int(hdtr_iters) < 0:
        raise ValueError("hdtr_iters must be >= 0")
    if int(sinkhorn_iters) <= 0:
        raise ValueError("sinkhorn_iters must be > 0")

    scores0 = local_scores.masked_fill(~feasible_mask, -1e9)
    P = sinkhorn_quota_assignment(scores0, quotas=quotas, tau=tau, n_iters=sinkhorn_iters)
    refined_scores = scores0

    for _ in range(int(hdtr_iters)):
        smooth_grad = 2.0 * (layer_laplacian.to(device=P.device, dtype=P.dtype) @ P)
        refined_scores = local_scores - float(eta) * smooth_grad
        refined_scores = refined_scores.masked_fill(~feasible_mask, -1e9)
        P = sinkhorn_quota_assignment(refined_scores, quotas=quotas, tau=tau, n_iters=sinkhorn_iters)

    return P, refined_scores


def hdtr_transport_objective(
    P: torch.Tensor,
    local_scores: torch.Tensor,
    layer_laplacian: torch.Tensor,
    tau: float,
    eta: float,
) -> torch.Tensor:
    _lasper_check_matrix("P", P, rank=2)

    if local_scores.shape != P.shape:
        raise ValueError(f"local_scores shape {tuple(local_scores.shape)} must match P {tuple(P.shape)}")

    L, K = P.shape
    if layer_laplacian.shape != (L, L):
        raise ValueError(f"layer_laplacian must be [L,L]={(L,L)}, got {tuple(layer_laplacian.shape)}")

    if not math.isfinite(float(tau)):
        raise ValueError("tau must be finite")
    if not math.isfinite(float(eta)):
        raise ValueError("eta must be finite")

    safe_scores = local_scores.masked_fill(~torch.isfinite(local_scores), -1e9)

    utility = (P * safe_scores).sum()
    entropy = -(P.clamp_min(EPS) * P.clamp_min(EPS).log()).sum()
    smooth = torch.trace(P.T @ layer_laplacian.to(device=P.device, dtype=P.dtype) @ P)
    return utility + float(tau) * entropy - float(eta) * smooth


def _fallback_index_under_budget(
    qleft: torch.Tensor,
    proto_cost: torch.Tensor,
    remaining: float,
) -> int:
    """
    If nothing is strictly feasible, choose the cheapest protocol among those with
    remaining quota. This minimizes overspend instead of allowing arbitrary fallback.
    """
    qleft = qleft.float()
    proto_cost = proto_cost.float().to(qleft.device)

    if qleft.numel() != proto_cost.numel():
        raise ValueError("qleft/proto_cost length mismatch")

    candidates = torch.where(qleft > 0)[0]
    if candidates.numel() == 0:
        candidates = torch.arange(len(proto_cost), device=proto_cost.device)

    if math.isfinite(float(remaining)):
        feasible = candidates[proto_cost[candidates] <= float(remaining) + 1e-6]
        if feasible.numel() > 0:
            candidates = feasible

    cheapest_local = candidates[torch.argmin(proto_cost[candidates])]
    return int(cheapest_local.item())


@torch.no_grad()
def project_transport_to_schedule(
    transport_plan: torch.Tensor,   # [L,K]
    refined_scores: torch.Tensor,   # [L,K]
    proto_cost: torch.Tensor,       # [K]
    quotas: torch.Tensor,           # [K]
    total_budget: Optional[float] = None,
    switch_penalty: float = 0.10,
) -> torch.Tensor:
    _lasper_check_matrix("transport_plan", transport_plan, rank=2)

    P = transport_plan
    L, K = P.shape
    device = P.device

    if refined_scores.shape != P.shape:
        raise ValueError("refined_scores shape must match transport_plan")

    proto_cost = proto_cost.to(device=device, dtype=torch.float32).view(-1)
    quotas = quotas.to(device=device, dtype=torch.float32).view(-1)

    if proto_cost.numel() != K:
        raise ValueError(f"proto_cost must have K={K} values, got {proto_cost.numel()}")
    if quotas.numel() != K:
        raise ValueError(f"quotas must have K={K} values, got {quotas.numel()}")

    _lasper_check_finite_tensor("proto_cost", proto_cost)
    _lasper_check_finite_tensor("quotas", quotas)

    if (proto_cost < 0).any():
        raise ValueError("proto_cost must be non-negative")
    if (quotas < 0).any():
        raise ValueError("quotas must be non-negative")
    if int(round(float(quotas.sum().item()))) != L:
        raise ValueError(f"quotas must sum to L={L}, got {float(quotas.sum().item())}")

    if total_budget is not None:
        total_budget = float(total_budget)
        if not math.isfinite(total_budget) or total_budget < 0.0:
            raise ValueError("total_budget must be finite and >= 0")

    if not math.isfinite(float(switch_penalty)):
        raise ValueError("switch_penalty must be finite")

    qleft = quotas.clone().float()
    remaining = total_budget if total_budget is not None else float("inf")

    schedule = []
    prev_idx = None

    for l in range(L):
        logits = refined_scores[l].to(device=device, dtype=torch.float32) + torch.log(P[l].float().clamp_min(EPS))

        feasible = qleft > 0
        if total_budget is not None:
            feasible = feasible & (proto_cost <= remaining + 1e-6)

        if bool(feasible.any().item()):
            logits = logits.masked_fill(~feasible, -1e9)

            if prev_idx is not None and float(switch_penalty) != 0.0 and bool(feasible[prev_idx].item()):
                logits[prev_idx] = logits[prev_idx] + float(switch_penalty)

            z = int(torch.argmax(logits).item())
        else:
            z = _fallback_index_under_budget(qleft=qleft, proto_cost=proto_cost, remaining=remaining)

        schedule.append(z)
        qleft[z] -= 1.0
        remaining -= float(proto_cost[z].item())
        prev_idx = z

    return torch.tensor(schedule, dtype=torch.long, device=device)


def sample_schedule_from_transport(
    transport_plan: torch.Tensor,   # [L,K]
    refined_scores: torch.Tensor,   # [L,K]
    proto_cost: torch.Tensor,       # [K]
    quotas: torch.Tensor,           # [K]
    total_budget: Optional[float] = None,
    switch_penalty: float = 0.10,
    temperature: float = 1.0,
    greedy: bool = False,
):
    _lasper_check_matrix("transport_plan", transport_plan, rank=2)

    P = transport_plan
    L, K = P.shape
    device = P.device

    if refined_scores.shape != P.shape:
        raise ValueError("refined_scores shape must match transport_plan")

    proto_cost = proto_cost.to(device=device, dtype=torch.float32).view(-1)
    quotas = quotas.to(device=device, dtype=torch.float32).view(-1)

    if proto_cost.numel() != K:
        raise ValueError(f"proto_cost must have K={K} values, got {proto_cost.numel()}")
    if quotas.numel() != K:
        raise ValueError(f"quotas must have K={K} values, got {quotas.numel()}")
    if int(round(float(quotas.sum().item()))) != L:
        raise ValueError(f"quotas must sum to L={L}, got {float(quotas.sum().item())}")

    _lasper_check_finite_tensor("proto_cost", proto_cost)
    _lasper_check_finite_tensor("quotas", quotas)

    if (proto_cost < 0).any():
        raise ValueError("proto_cost must be non-negative")
    if (quotas < 0).any():
        raise ValueError("quotas must be non-negative")

    if total_budget is not None:
        total_budget = float(total_budget)
        if not math.isfinite(total_budget) or total_budget < 0.0:
            raise ValueError("total_budget must be finite and >= 0")

    if not math.isfinite(float(switch_penalty)):
        raise ValueError("switch_penalty must be finite")
    if not math.isfinite(float(temperature)) or float(temperature) <= 0.0:
        raise ValueError("temperature must be finite and > 0")

    qleft = quotas.clone().float()
    remaining = total_budget if total_budget is not None else float("inf")

    chosen = []
    log_probs = []
    entropies = []

    prev_idx = None

    for l in range(L):
        logits = refined_scores[l].to(device=device, dtype=torch.float32) + torch.log(P[l].float().clamp_min(EPS))

        feasible = qleft > 0
        if total_budget is not None:
            feasible = feasible & (proto_cost <= remaining + 1e-6)

        if bool(feasible.any().item()):
            logits = logits.masked_fill(~feasible, -1e9)

            if prev_idx is not None and float(switch_penalty) != 0.0 and bool(feasible[prev_idx].item()):
                logits[prev_idx] = logits[prev_idx] + float(switch_penalty)

            logits = logits / float(max(float(temperature), 1e-4))
            probs = F.softmax(logits, dim=-1)

            if not torch.isfinite(probs).all() or float(probs.sum().item()) <= 0.0:
                raise RuntimeError("Invalid schedule sampling probabilities")

            dist = torch.distributions.Categorical(probs=probs)
            idx = torch.argmax(probs) if greedy else dist.sample()

            chosen.append(idx)
            log_probs.append(dist.log_prob(idx))
            entropies.append(dist.entropy())

            idx_int = int(idx.item())
        else:
            idx_int = _fallback_index_under_budget(qleft=qleft, proto_cost=proto_cost, remaining=remaining)
            idx = torch.tensor(idx_int, device=device, dtype=torch.long)
            chosen.append(idx)

            log_probs.append(torch.zeros((), device=device, dtype=refined_scores.dtype))
            entropies.append(torch.zeros((), device=device, dtype=refined_scores.dtype))

        qleft[idx_int] -= 1.0
        remaining -= float(proto_cost[idx_int].item())
        prev_idx = idx_int

    schedule = torch.stack(chosen).long()
    log_prob_mean = torch.stack(log_probs).mean()
    entropy_mean = torch.stack(entropies).mean()
    spent = proto_cost[schedule].sum()

    return {
        "schedule": schedule,
        "log_prob_mean": log_prob_mean,
        "entropy_mean": entropy_mean,
        "spent": spent,
    }

In [ ]:
# ============================================================
# CELL 5: Encoders + LaSPER-HT router
# Custom GCCD decoder path
# ============================================================
from typing import List, Optional, Dict, Any

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def _lasper_router_check_finite(name: str, x: torch.Tensor) -> None:
    if not torch.is_tensor(x):
        raise TypeError(f"{name} must be a torch.Tensor")
    if not torch.isfinite(x.detach().float()).all():
        raise ValueError(f"{name} contains NaN/Inf")


def _lasper_router_check_positive_int(name: str, value: int) -> int:
    value = int(value)
    if value <= 0:
        raise ValueError(f"{name} must be > 0")
    return value


class LayerStateEncoder(nn.Module):
    def __init__(self, summary_dim: int, deficit_dim: int, usage_dim: int, d_router: int = 128):
        super().__init__()

        self.summary_dim = _lasper_router_check_positive_int("summary_dim", summary_dim)
        self.deficit_dim = _lasper_router_check_positive_int("deficit_dim", deficit_dim)
        self.usage_dim = _lasper_router_check_positive_int("usage_dim", usage_dim)
        self.d_router = _lasper_router_check_positive_int("d_router", d_router)

        in_dim = self.summary_dim + self.deficit_dim + 1 + self.usage_dim

        self.net = nn.Sequential(
            nn.Linear(in_dim, self.d_router),
            nn.GELU(),
            nn.Linear(self.d_router, self.d_router),
            nn.GELU(),
            nn.Linear(self.d_router, self.d_router),
        )

    def forward(self, summaries, deficits, remaining_budget, usage_history):
        if summaries.dim() != 2:
            raise ValueError(f"summaries must be [L,H], got {tuple(summaries.shape)}")
        if deficits.dim() != 2:
            raise ValueError(f"deficits must be [L,D], got {tuple(deficits.shape)}")
        if usage_history.dim() != 2:
            raise ValueError(f"usage_history must be [L,K], got {tuple(usage_history.shape)}")

        L = int(summaries.shape[0])

        if summaries.shape[1] != self.summary_dim:
            raise ValueError(f"summaries dim mismatch: got {summaries.shape[1]}, expected {self.summary_dim}")
        if deficits.shape != (L, self.deficit_dim):
            raise ValueError(f"deficits shape mismatch: got {tuple(deficits.shape)}, expected {(L, self.deficit_dim)}")
        if usage_history.shape != (L, self.usage_dim):
            raise ValueError(f"usage_history shape mismatch: got {tuple(usage_history.shape)}, expected {(L, self.usage_dim)}")

        remaining_budget = remaining_budget.to(device=summaries.device).float().view(-1)
        if remaining_budget.numel() != L:
            raise ValueError(f"remaining_budget must have L={L} values, got {remaining_budget.numel()}")

        _lasper_router_check_finite("summaries", summaries)
        _lasper_router_check_finite("deficits", deficits)
        _lasper_router_check_finite("remaining_budget", remaining_budget)
        _lasper_router_check_finite("usage_history", usage_history)

        usage_norm = max(1.0, float(usage_history.shape[0]))

        x = torch.cat(
            [
                summaries.float(),
                deficits.float(),
                remaining_budget[:, None].float(),
                (usage_history / usage_norm).float(),
            ],
            dim=-1,
        )

        return self.net(x)


class ProtocolEncoder(nn.Module):
    def __init__(self, d_router: int = 128):
        super().__init__()

        self.d_router = _lasper_router_check_positive_int("d_router", d_router)

        self.net = nn.Sequential(
            nn.Linear(10, self.d_router),
            nn.GELU(),
            nn.Linear(self.d_router, self.d_router),
            nn.GELU(),
            nn.Linear(self.d_router, self.d_router),
        )

    def forward(self, proto_sig):
        if proto_sig.dim() != 2 or proto_sig.shape[1] != 10:
            raise ValueError(f"proto_sig must be [K,10], got {tuple(proto_sig.shape)}")
        _lasper_router_check_finite("proto_sig", proto_sig)
        return self.net(proto_sig.float())


def build_protocol_signature_table(protocol_table, device):
    if protocol_table is None or len(protocol_table) == 0:
        raise ValueError("protocol_table must be non-empty")

    rows = []

    for i, p in enumerate(protocol_table):
        if getattr(p, "cost", None) is None:
            raise ValueError(
                f"protocol_table[{i}].cost is None. "
                "Run ensure_protocol_costs(...) or calibrate_protocol_costs_from_probe(...) before building router."
            )

        row = [
            float(p.w),
            float(p.num_anchors),
            float(p.scaffold_mode),
            float(p.ring_inner),
            float(p.ring_outer),
            1.0 if p.include_self else 0.0,
            float(p.anchor_strategy),
            float(p.anchor_param0),
            float(p.cost),
            float(p.quota_ratio),
        ]

        if not all(math.isfinite(v) for v in row):
            raise ValueError(f"protocol_table[{i}] signature contains NaN/Inf")

        rows.append(row)

    x = torch.tensor(rows, dtype=torch.float32, device=device)

    denom = x.abs().amax(dim=0).clamp_min(1.0)
    return x / denom


def _repair_quotas_for_feasibility(
    quotas: torch.Tensor,
    feasible_mask: torch.Tensor,
    proto_cost: torch.Tensor,
) -> torch.Tensor:
    """
    Moves quota away from protocol columns that have no feasible layer.
    Keeps total quota unchanged.
    """
    quotas = quotas.float().clone()
    possible_cols = feasible_mask.any(dim=0)

    if bool(possible_cols.all().item()):
        return quotas

    if not bool(possible_cols.any().item()):
        raise ValueError("No feasible protocol columns exist.")

    feasible_cols = torch.where(possible_cols)[0]
    cheapest_feasible = feasible_cols[torch.argmin(proto_cost[feasible_cols])]

    impossible_cols = torch.where((quotas > 0) & (~possible_cols))[0]
    for c in impossible_cols.tolist():
        quotas[cheapest_feasible] += quotas[c]
        quotas[c] = 0.0

    return quotas


def _ensure_each_row_has_feasible_protocol(
    local_scores: torch.Tensor,
    feasible_mask: torch.Tensor,
    layer_feat: torch.Tensor,
    proto_feat: torch.Tensor,
    proto_cost: torch.Tensor,
    proto_usage_global: torch.Tensor,
    lambda_cost: float,
    mu_usage: float,
) -> tuple:
    """
    Ensures HDTR/Sinkhorn never receives an all-infeasible row.
    If a layer has no affordable protocol under remaining budget, use the cheapest protocol
    as the least-bad fallback for that row.
    """
    row_has_any = feasible_mask.any(dim=1)
    if bool(row_has_any.all().item()):
        return local_scores, feasible_mask

    cheapest = int(torch.argmin(proto_cost).item())
    bad_rows = torch.where(~row_has_any)[0]

    fallback_scores = (
        layer_feat[bad_rows].float() @ proto_feat[cheapest].float()
        - float(lambda_cost) * proto_cost[cheapest].float()
        - float(mu_usage) * proto_usage_global[cheapest].float()
    )

    local_scores = local_scores.clone()
    feasible_mask = feasible_mask.clone()

    local_scores[bad_rows, cheapest] = fallback_scores
    feasible_mask[bad_rows, cheapest] = True

    return local_scores, feasible_mask


class LaSPERHTRouter(nn.Module):
    """
    Stage 1: LaSPER local proposer
      A_{lz} = <phi_l, psi_z> - lambda cost(z) - mu scarcity(z)

    Stage 2: HDTR harmonic transport refinement
      max_P <P, A> + tau H(P) - eta Tr(P^T L_LL P)
      s.t. row-sum and quota constraints
    """
    def __init__(
        self,
        summary_dim: int,
        protocol_table: List[ProtocolSpec],
        deficit_dim: int = 6,
        d_router: int = 128,
        lambda_cost: float = 0.8,
        mu_usage: float = 0.15,
        tau: float = 0.35,
        eta_harm: float = 0.20,
        hdtr_iters: int = 6,
        sinkhorn_iters: int = 50,
        switch_penalty: float = 0.10,
        layer_knn: int = 0,
    ):
        super().__init__()

        if protocol_table is None or len(protocol_table) == 0:
            raise ValueError("protocol_table must be non-empty")

        self.protocol_table = protocol_table
        self.K = len(protocol_table)

        self.lambda_cost = float(lambda_cost)
        self.mu_usage = float(mu_usage)
        self.tau = float(tau)
        self.eta_harm = float(eta_harm)
        self.hdtr_iters = int(hdtr_iters)
        self.sinkhorn_iters = int(sinkhorn_iters)
        self.switch_penalty = float(switch_penalty)
        self.layer_knn = int(layer_knn)

        if not math.isfinite(self.lambda_cost):
            raise ValueError("lambda_cost must be finite")
        if not math.isfinite(self.mu_usage):
            raise ValueError("mu_usage must be finite")
        if not math.isfinite(self.tau) or self.tau <= 0.0:
            raise ValueError("tau must be finite and > 0")
        if not math.isfinite(self.eta_harm) or self.eta_harm < 0.0:
            raise ValueError("eta_harm must be finite and >= 0")
        if self.hdtr_iters < 0:
            raise ValueError("hdtr_iters must be >= 0")
        if self.sinkhorn_iters <= 0:
            raise ValueError("sinkhorn_iters must be > 0")
        if not math.isfinite(self.switch_penalty):
            raise ValueError("switch_penalty must be finite")
        if self.layer_knn < 0:
            raise ValueError("layer_knn must be >= 0")

        self.layer_encoder = LayerStateEncoder(summary_dim, deficit_dim, self.K, d_router=d_router)
        self.protocol_encoder = ProtocolEncoder(d_router=d_router)

        init_device = torch.device("cpu")

        proto_sig = build_protocol_signature_table(protocol_table, device=init_device)
        proto_cost = torch.tensor([float(p.cost) for p in protocol_table], dtype=torch.float32, device=init_device)
        proto_ratio = torch.tensor([float(p.quota_ratio) for p in protocol_table], dtype=torch.float32, device=init_device)

        _lasper_router_check_finite("proto_cost", proto_cost)
        _lasper_router_check_finite("proto_quota_ratio", proto_ratio)

        if (proto_cost < 0).any():
            raise ValueError("protocol costs must be non-negative")
        if (proto_ratio < 0).any():
            raise ValueError("protocol quota ratios must be non-negative")
        if float(proto_ratio.sum().item()) <= 0.0:
            raise ValueError("protocol quota ratios must have positive sum")

        self.register_buffer("proto_signature", proto_sig)
        self.register_buffer("proto_cost", proto_cost)
        self.register_buffer("proto_quota_ratio", proto_ratio)

    def forward(
        self,
        layer_summaries: torch.Tensor,        # [L,H]
        deficits: torch.Tensor,               # [L,Dd]
        remaining_budget: torch.Tensor,       # [L]
        usage_history: torch.Tensor,          # [L,K]
        total_budget: Optional[float] = None,
        quotas: Optional[torch.Tensor] = None,
    ):
        if layer_summaries.dim() != 2:
            raise ValueError(f"layer_summaries must be [L,H], got {tuple(layer_summaries.shape)}")
        if deficits.dim() != 2:
            raise ValueError(f"deficits must be [L,D], got {tuple(deficits.shape)}")
        if usage_history.dim() != 2:
            raise ValueError(f"usage_history must be [L,K], got {tuple(usage_history.shape)}")

        device = layer_summaries.device

        # Notebook-safe wiring: keep router params/buffers on the same device as summaries.
        if next(self.parameters()).device != device:
            self.to(device)

        L = int(layer_summaries.size(0))

        if L <= 0:
            raise ValueError("number of layers L must be > 0")
        if deficits.shape[0] != L:
            raise ValueError(f"deficits layer mismatch: got {deficits.shape[0]}, expected {L}")
        if usage_history.shape != (L, self.K):
            raise ValueError(f"usage_history shape mismatch: got {tuple(usage_history.shape)}, expected {(L, self.K)}")

        remaining_budget = remaining_budget.to(device=device, dtype=torch.float32).view(-1)
        if remaining_budget.numel() != L:
            raise ValueError(f"remaining_budget must have L={L} values, got {remaining_budget.numel()}")

        _lasper_router_check_finite("layer_summaries", layer_summaries)
        _lasper_router_check_finite("deficits", deficits)
        _lasper_router_check_finite("remaining_budget", remaining_budget)
        _lasper_router_check_finite("usage_history", usage_history)

        if (remaining_budget < 0).any():
            raise ValueError("remaining_budget must be non-negative")

        if total_budget is not None:
            total_budget = float(total_budget)
            if not math.isfinite(total_budget) or total_budget < 0.0:
                raise ValueError("total_budget must be finite and >= 0")

        if quotas is None:
            quotas = adjust_quotas_to_budget(
                proto_cost=self.proto_cost.to(device),
                quota_ratio=self.proto_quota_ratio.to(device),
                num_layers=L,
                total_budget=total_budget,
            )
        else:
            quotas = quotas.to(device=device, dtype=torch.float32).view(-1)

        if quotas.numel() != self.K:
            raise ValueError(f"quotas must have K={self.K} values, got {quotas.numel()}")
        _lasper_router_check_finite("quotas", quotas)

        if (quotas < 0).any():
            raise ValueError("quotas must be non-negative")
        if int(round(float(quotas.sum().item()))) != L:
            raise ValueError(f"quotas must sum to L={L}, got {float(quotas.sum().item())}")

        layer_feat = self.layer_encoder(
            layer_summaries.float(),
            deficits.float(),
            remaining_budget.float(),
            usage_history.float(),
        )

        proto_feat = self.protocol_encoder(self.proto_signature.to(device))

        proto_usage_global = (
            usage_history[-1].to(device=device, dtype=torch.float32)
            if usage_history.numel() > 0
            else torch.zeros(self.K, device=device)
        )

        # Stage 1: LaSPER local proposal.
        local_scores = build_affinity_scores(
            layer_feat=layer_feat,
            proto_feat=proto_feat,
            proto_cost=self.proto_cost.to(device),
            remaining_budget=remaining_budget,
            proto_usage_global=proto_usage_global,
            lambda_cost=self.lambda_cost,
            mu_usage=self.mu_usage,
        )

        feasible_mask = torch.isfinite(local_scores) & (local_scores > -1e8)

        local_scores, feasible_mask = _ensure_each_row_has_feasible_protocol(
            local_scores=local_scores,
            feasible_mask=feasible_mask,
            layer_feat=layer_feat,
            proto_feat=proto_feat,
            proto_cost=self.proto_cost.to(device),
            proto_usage_global=proto_usage_global,
            lambda_cost=self.lambda_cost,
            mu_usage=self.mu_usage,
        )

        quotas = _repair_quotas_for_feasibility(
            quotas=quotas,
            feasible_mask=feasible_mask,
            proto_cost=self.proto_cost.to(device),
        )

        # Stage 2: HDTR harmonic transport refinement.
        W_LL = build_layer_graph(
            layer_feat=layer_feat,
            knn=self.layer_knn,
            self_loop_weight=1.0,
        )
        L_LL = normalized_laplacian(W_LL)

        transport_plan, refined_scores = hdtr_refine_assignments(
            local_scores=local_scores,
            layer_laplacian=L_LL,
            quotas=quotas,
            feasible_mask=feasible_mask,
            tau=self.tau,
            eta=self.eta_harm,
            hdtr_iters=self.hdtr_iters,
            sinkhorn_iters=self.sinkhorn_iters,
        )

        objective = hdtr_transport_objective(
            P=transport_plan,
            local_scores=local_scores,
            layer_laplacian=L_LL,
            tau=self.tau,
            eta=self.eta_harm,
        )

        schedule = project_transport_to_schedule(
            transport_plan=transport_plan,
            refined_scores=refined_scores,
            proto_cost=self.proto_cost.to(device),
            quotas=quotas,
            total_budget=total_budget,
            switch_penalty=self.switch_penalty,
        )

        return {
            "schedule": schedule,
            "transport_plan": transport_plan,
            "local_scores": local_scores,
            "refined_scores": refined_scores,
            "layer_graph": W_LL,
            "layer_laplacian": L_LL,
            "quotas": quotas,
            "objective": objective,
            "layer_feat": layer_feat,
            "proto_feat": proto_feat,
        }

In [ ]:
# ============================================================
# CELL 6: Routing state collection
# Custom GCCD decoder path
#   - clears stale landmark runtime payload before routing-state forward
#   - clears custom candidate caches before routing-state forward
#   - runs routing-state pilot in eval mode, then restores mode
# ============================================================
from typing import Optional, Dict, Any
import math
import torch


def _lasperht_get_hidden_states(outputs):
    if hasattr(outputs, "hidden_states"):
        hs = outputs.hidden_states
    elif isinstance(outputs, dict) and "hidden_states" in outputs:
        hs = outputs["hidden_states"]
    else:
        raise RuntimeError("Model output does not contain hidden_states.")

    if hs is None or len(hs) == 0:
        raise RuntimeError("hidden_states is empty. Ensure output_hidden_states=True.")

    return hs


def _lasperht_sanitize_forward_kwargs(model_forward_kwargs: Optional[dict]) -> dict:
    kwargs = dict(model_forward_kwargs or {})

    # These are controlled by the routing pilot. Keeping user-provided copies can cause:
    # TypeError: got multiple values for keyword argument ...
    kwargs.pop("output_hidden_states", None)
    kwargs.pop("use_cache", None)

    return kwargs


@torch.no_grad()
def collect_lasperht_state(
    model,
    router: LaSPERHTRouter,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
    deficits: Optional[torch.Tensor] = None,
    total_budget: Optional[float] = None,
    prev_schedule: Optional[torch.Tensor] = None,
    model_forward_kwargs: Optional[dict] = None,
):
    if model is None:
        raise ValueError("model must not be None")
    if router is None:
        raise ValueError("router must not be None")
    if not torch.is_tensor(input_ids):
        raise TypeError("input_ids must be a torch.Tensor")
    if input_ids.dim() < 2:
        raise ValueError(f"input_ids must be at least [B,T], got {tuple(input_ids.shape)}")

    model_forward_kwargs = _lasperht_sanitize_forward_kwargs(model_forward_kwargs)

    if "clear_dcsla_runtime_state" in globals():
        clear_dcsla_runtime_state(model)

    if "clear_custom_candidate_caches" in globals():
        clear_custom_candidate_caches(model)
    elif "clear_sasa_candidate_caches" in globals():
        clear_sasa_candidate_caches(model)

    if "_forward_hidden_states_pilot" in globals():
        outputs = _forward_hidden_states_pilot(
            model=model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            model_forward_kwargs=model_forward_kwargs,
        )
    else:
        was_training = bool(model.training)
        model.eval()
        try:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
                **model_forward_kwargs,
            )
        finally:
            model.train(was_training)

    hidden_states = _lasperht_get_hidden_states(outputs)

    layer_summaries = collect_layer_summaries_from_hidden_states(
        hidden_states,
        attention_mask=attention_mask,
    ).to(input_ids.device)

    if layer_summaries.dim() != 2:
        raise ValueError(f"layer_summaries must be [L,H], got {tuple(layer_summaries.shape)}")
    if not torch.isfinite(layer_summaries.detach().float()).all():
        raise ValueError("layer_summaries contains NaN/Inf")

    L = int(layer_summaries.size(0))
    K = int(router.K)

    if L <= 0:
        raise ValueError("No layer summaries collected.")
    if K <= 0:
        raise ValueError("router.K must be > 0")

    if deficits is None:
        deficits = build_default_deficits(layer_summaries)
    else:
        deficits = deficits.to(device=layer_summaries.device, dtype=torch.float32)

    if deficits.dim() != 2 or deficits.shape[0] != L:
        raise ValueError(f"deficits must be [L,D] with L={L}, got {tuple(deficits.shape)}")
    if not torch.isfinite(deficits.detach().float()).all():
        raise ValueError("deficits contains NaN/Inf")

    proto_cost = router.proto_cost.to(layer_summaries.device)
    proto_ratio = router.proto_quota_ratio.to(layer_summaries.device)

    if proto_cost.numel() != K:
        raise ValueError(f"router.proto_cost must have K={K} values, got {proto_cost.numel()}")
    if proto_ratio.numel() != K:
        raise ValueError(f"router.proto_quota_ratio must have K={K} values, got {proto_ratio.numel()}")
    if not torch.isfinite(proto_cost.float()).all():
        raise ValueError("router.proto_cost contains NaN/Inf")
    if not torch.isfinite(proto_ratio.float()).all():
        raise ValueError("router.proto_quota_ratio contains NaN/Inf")
    if (proto_cost < 0).any():
        raise ValueError("router.proto_cost must be non-negative")
    if (proto_ratio < 0).any():
        raise ValueError("router.proto_quota_ratio must be non-negative")

    if total_budget is None:
        default_quotas = quotas_from_ratios(L, proto_ratio)
        total_budget = float((default_quotas * proto_cost).sum().item())
    else:
        total_budget = float(total_budget)
        if not math.isfinite(total_budget) or total_budget < 0.0:
            raise ValueError("total_budget must be finite and >= 0")

    usage_history = build_usage_history(
        prev_schedule=prev_schedule,
        num_protocols=K,
        num_layers=L,
        device=layer_summaries.device,
    )

    remaining_budget = build_remaining_budget_profile(
        total_budget=float(total_budget),
        proto_cost=proto_cost,
        prev_schedule=prev_schedule,
        num_layers=L,
        device=layer_summaries.device,
    )

    if usage_history.shape != (L, K):
        raise ValueError(f"usage_history shape mismatch: got {tuple(usage_history.shape)}, expected {(L, K)}")
    if remaining_budget.shape != (L,):
        raise ValueError(f"remaining_budget shape mismatch: got {tuple(remaining_budget.shape)}, expected {(L,)}")
    if not torch.isfinite(usage_history.float()).all():
        raise ValueError("usage_history contains NaN/Inf")
    if not torch.isfinite(remaining_budget.float()).all():
        raise ValueError("remaining_budget contains NaN/Inf")

    return {
        "layer_summaries": layer_summaries,
        "deficits": deficits,
        "usage_history": usage_history,
        "remaining_budget": remaining_budget,
        "total_budget": float(total_budget),
    }


@torch.no_grad()
def route_batch_lasperht(
    model,
    router: LaSPERHTRouter,
    input_ids: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None,
    deficits: Optional[torch.Tensor] = None,
    total_budget: Optional[float] = None,
    prev_schedule: Optional[torch.Tensor] = None,
    model_forward_kwargs: Optional[dict] = None,
):
    state = collect_lasperht_state(
        model=model,
        router=router,
        input_ids=input_ids,
        attention_mask=attention_mask,
        deficits=deficits,
        total_budget=total_budget,
        prev_schedule=prev_schedule,
        model_forward_kwargs=model_forward_kwargs,
    )

    router_out = router(
        layer_summaries=state["layer_summaries"],
        deficits=state["deficits"],
        remaining_budget=state["remaining_budget"],
        usage_history=state["usage_history"],
        total_budget=state["total_budget"],
    )

    if "apply_schedule_to_model" not in globals():
        raise NameError("apply_schedule_to_model is not defined. Run the custom decoder schedule-hook cell first.")

    apply_schedule_to_model(model, router_out["schedule"], router.protocol_table)

    if "clear_custom_candidate_caches" in globals():
        clear_custom_candidate_caches(model)
    elif "clear_sasa_candidate_caches" in globals():
        clear_sasa_candidate_caches(model)

    return router_out, state

In [ ]:
# ============================================================
# CELL 7: Training / evaluation helpers
# Custom GCCD decoder path
#   - refresh runtime DCSLA anchors after LaSPER schedule
#   - train DCSLA with differentiable auxiliary loss on actual hidden states
#   - no Mistral-specific wiring
# ============================================================
from dataclasses import dataclass
from typing import Optional, Dict, Any
from contextlib import nullcontext

import math
import torch
import torch.nn.functional as F
from torch.optim import AdamW


def _runtime_device(model=None, router=None):
    for obj in (model, router):
        if obj is None:
            continue
        if hasattr(obj, "parameters"):
            try:
                return next(obj.parameters()).device
            except StopIteration:
                pass
        if hasattr(obj, "buffers"):
            try:
                return next(obj.buffers()).device
            except StopIteration:
                pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _amp_enabled_for_device(device: torch.device) -> bool:
    device = torch.device(device)
    return bool(torch.cuda.is_available() and device.type == "cuda")


def _autocast_context(device: torch.device, enabled: bool):
    device = torch.device(device)
    if enabled and device.type == "cuda":
        return torch.cuda.amp.autocast(dtype=torch.float16)
    return nullcontext()


def _clear_custom_runtime_caches_if_available(model) -> None:
    if "clear_dcsla_runtime_state" in globals():
        clear_dcsla_runtime_state(model)

    if "clear_custom_candidate_caches" in globals():
        clear_custom_candidate_caches(model)
    elif "clear_sasa_candidate_caches" in globals():
        clear_sasa_candidate_caches(model)


def _get_output_loss(outputs):
    if hasattr(outputs, "loss"):
        loss = outputs.loss
    elif isinstance(outputs, dict) and "loss" in outputs:
        loss = outputs["loss"]
    elif isinstance(outputs, (tuple, list)) and len(outputs) > 0:
        loss = outputs[0]
    else:
        raise RuntimeError("Model output does not contain loss. Ensure labels=... is passed and model returns loss.")

    if loss is None:
        raise RuntimeError("Model output loss is None.")

    return loss


def _get_output_hidden_states(outputs):
    if hasattr(outputs, "hidden_states"):
        hs = outputs.hidden_states
    elif isinstance(outputs, dict) and "hidden_states" in outputs:
        hs = outputs["hidden_states"]
    else:
        raise RuntimeError("Model output does not contain hidden_states.")

    if hs is None or len(hs) == 0:
        raise RuntimeError("hidden_states is empty. Ensure output_hidden_states=True.")

    return hs


def _ensure_finite_scalar(name: str, x: torch.Tensor) -> None:
    if not torch.is_tensor(x):
        raise TypeError(f"{name} must be a tensor")
    if x.numel() != 1:
        raise ValueError(f"{name} must be scalar, got shape {tuple(x.shape)}")
    if not torch.isfinite(x.detach().float()).all():
        raise ValueError(f"{name} contains NaN/Inf")


def batch_to_device(batch, device):
    if not isinstance(batch, dict):
        raise TypeError("batch must be a dict")
    if "input_ids" not in batch:
        raise KeyError("batch must contain input_ids")

    device = torch.device(device)
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def get_labels_from_batch(batch):
    if "input_ids" not in batch:
        raise KeyError("batch must contain input_ids")

    labels = batch.get("labels", None)
    return batch["input_ids"] if labels is None else labels


@dataclass
class RouterTrainState:
    reward_ema: Optional[float] = None
    global_step: int = 0
    prev_schedule: Optional[torch.Tensor] = None


def build_default_total_budget(router, num_layers: int, device):
    num_layers = int(num_layers)
    if num_layers <= 0:
        raise ValueError("num_layers must be > 0")

    q = quotas_from_ratios(num_layers, router.proto_quota_ratio.to(device))
    budget = float((q * router.proto_cost.to(device)).sum().item())

    if not math.isfinite(budget) or budget < 0.0:
        raise ValueError(f"Invalid default total budget: {budget}")

    return budget


def _build_model_optimizer_params(model, dcsla_controller=None):
    params = []
    seen = set()

    if model is None:
        raise ValueError("model must not be None")

    for p in model.parameters():
        if p.requires_grad and id(p) not in seen:
            params.append(p)
            seen.add(id(p))

    if dcsla_controller is not None:
        if not hasattr(dcsla_controller, "selector"):
            raise TypeError("dcsla_controller must have selector")
        for p in dcsla_controller.selector.parameters():
            if p.requires_grad and id(p) not in seen:
                params.append(p)
                seen.add(id(p))

    if len(params) == 0:
        raise ValueError("No trainable model/DCSLA parameters found for optimizer.")

    return params


def _build_router_optimizer_params(router):
    if router is None:
        raise ValueError("router must not be None")

    params = [p for p in router.parameters() if p.requires_grad]
    if len(params) == 0:
        raise ValueError("No trainable router parameters found.")

    return params


def _validate_train_eval_inputs(model, router, loader_or_batch=None):
    if model is None:
        raise ValueError("model must not be None")
    if router is None:
        raise ValueError("router must not be None")
    if not hasattr(router, "protocol_table"):
        raise TypeError("router must have protocol_table")
    if not hasattr(router, "proto_cost"):
        raise TypeError("router must have proto_cost")
    if not hasattr(router, "proto_quota_ratio"):
        raise TypeError("router must have proto_quota_ratio")


def routed_eval_epoch(
    model,
    router,
    loader,
    total_budget=None,
    max_batches=None,
    dcsla_controller=None,
    dcsla_cfg: Optional[DCSLAConfig] = None,
):
    _validate_train_eval_inputs(model, router)

    device = _runtime_device(model, router)

    model.eval()
    router.eval()
    if dcsla_controller is not None:
        dcsla_controller.selector.eval()

    losses = []
    spends = []
    proto_hist = torch.zeros(len(router.protocol_table), dtype=torch.float64)
    eval_state = RouterTrainState()

    with torch.no_grad():
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= int(max_batches):
                break

            batch = batch_to_device(batch, device)
            input_ids = batch["input_ids"]
            attention_mask = batch.get("attention_mask", None)
            labels = get_labels_from_batch(batch)

            state = collect_lasperht_state(
                model=model,
                router=router,
                input_ids=input_ids,
                attention_mask=attention_mask,
                deficits=None,
                total_budget=total_budget,
                prev_schedule=eval_state.prev_schedule,
            )

            router_out = router(
                layer_summaries=state["layer_summaries"],
                deficits=state["deficits"],
                remaining_budget=state["remaining_budget"],
                usage_history=state["usage_history"],
                total_budget=state["total_budget"],
            )

            sampled = sample_schedule_from_transport(
                transport_plan=router_out["transport_plan"],
                refined_scores=router_out["refined_scores"],
                proto_cost=router.proto_cost.to(input_ids.device),
                quotas=router_out["quotas"],
                total_budget=state["total_budget"],
                switch_penalty=router.switch_penalty,
                temperature=1.0,
                greedy=True,
            )

            schedule = sampled["schedule"].detach()
            apply_schedule_to_model(model, schedule, router.protocol_table)
            _clear_custom_runtime_caches_if_available(model)

            eval_state.prev_schedule = schedule.detach().clone()

            if dcsla_controller is not None:
                dcsla_controller.refresh_after_schedule(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    deficits=state["deficits"],
                    force=True,
                )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=(dcsla_controller is not None),
                use_cache=False,
            )

            loss = _get_output_loss(outputs)

            if dcsla_controller is not None:
                hidden_states = _get_output_hidden_states(outputs)
                aux_loss, _ = dcsla_aux_loss(
                    model=model,
                    selector=dcsla_controller.selector,
                    hidden_states_all=hidden_states,
                    attention_mask=attention_mask,
                    deficits=state["deficits"],
                    cfg=dcsla_cfg,
                )
                loss = loss + aux_loss

            _ensure_finite_scalar("eval loss", loss)

            losses.append(float(loss.detach().item()))
            spends.append(float(sampled["spent"].detach().item()))

            for idx in schedule.detach().cpu().tolist():
                idx = int(idx)
                if idx < 0 or idx >= len(router.protocol_table):
                    raise ValueError(f"Invalid protocol index in schedule: {idx}")
                proto_hist[idx] += 1.0

    return {
        "loss": sum(losses) / max(len(losses), 1),
        "spend": sum(spends) / max(len(spends), 1),
        "proto_hist": (proto_hist / proto_hist.sum().clamp_min(1.0)).tolist(),
    }


def routed_train_step(
    model,
    router,
    batch,
    model_optimizer,
    router_optimizer=None,
    scaler=None,
    train_state: Optional[RouterTrainState] = None,
    total_budget=None,
    route_temperature: float = 1.0,
    entropy_coef: float = 0.01,
    aux_objective_coef: float = 0.10,
    budget_penalty_coef: float = 0.05,
    max_grad_norm_model: float = 1.0,
    max_grad_norm_router: float = 1.0,
    dcsla_controller=None,
    dcsla_cfg: Optional[DCSLAConfig] = None,
):
    _validate_train_eval_inputs(model, router)

    if train_state is None:
        train_state = RouterTrainState()

    if not math.isfinite(float(route_temperature)) or float(route_temperature) <= 0.0:
        raise ValueError("route_temperature must be finite and > 0")

    if model_optimizer is None:
        raise ValueError("model_optimizer must not be None")

    device = _runtime_device(model, router)
    amp_enabled = _amp_enabled_for_device(device)

    model.train()
    router.train()
    if dcsla_controller is not None:
        dcsla_controller.selector.train()

    batch = batch_to_device(batch, device)
    input_ids = batch["input_ids"]
    attention_mask = batch.get("attention_mask", None)
    labels = get_labels_from_batch(batch)

    state = collect_lasperht_state(
        model=model,
        router=router,
        input_ids=input_ids,
        attention_mask=attention_mask,
        deficits=None,
        total_budget=total_budget,
        prev_schedule=train_state.prev_schedule,
    )

    effective_budget = float(state["total_budget"])
    if not math.isfinite(effective_budget) or effective_budget < 0.0:
        raise ValueError(f"Invalid effective_budget: {effective_budget}")

    router_out = router(
        layer_summaries=state["layer_summaries"],
        deficits=state["deficits"],
        remaining_budget=state["remaining_budget"],
        usage_history=state["usage_history"],
        total_budget=effective_budget,
    )

    sampled = sample_schedule_from_transport(
        transport_plan=router_out["transport_plan"],
        refined_scores=router_out["refined_scores"],
        proto_cost=router.proto_cost.to(input_ids.device),
        quotas=router_out["quotas"],
        total_budget=effective_budget,
        switch_penalty=router.switch_penalty,
        temperature=route_temperature,
        greedy=False,
    )

    schedule = sampled["schedule"].detach()
    apply_schedule_to_model(model, schedule, router.protocol_table)
    _clear_custom_runtime_caches_if_available(model)

    train_state.prev_schedule = schedule.detach().clone()

    if dcsla_controller is not None:
        dcsla_controller.refresh_after_schedule(
            input_ids=input_ids,
            attention_mask=attention_mask,
            deficits=state["deficits"],
            force=False,
        )

    model_optimizer.zero_grad(set_to_none=True)

    dcsla_stats = None

    if scaler is not None and amp_enabled:
        with _autocast_context(device, enabled=True):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=(dcsla_controller is not None),
                use_cache=False,
            )

            lm_loss = _get_output_loss(outputs)

            if dcsla_controller is not None:
                hidden_states = _get_output_hidden_states(outputs)
                dcsla_loss, dcsla_stats = dcsla_aux_loss(
                    model=model,
                    selector=dcsla_controller.selector,
                    hidden_states_all=hidden_states,
                    attention_mask=attention_mask,
                    deficits=state["deficits"],
                    cfg=dcsla_cfg,
                )
                total_model_loss = lm_loss + dcsla_loss
            else:
                dcsla_loss = lm_loss.new_zeros(())
                total_model_loss = lm_loss

        _ensure_finite_scalar("lm_loss", lm_loss)
        _ensure_finite_scalar("dcsla_loss", dcsla_loss)
        _ensure_finite_scalar("total_model_loss", total_model_loss)

        scaler.scale(total_model_loss).backward()
        scaler.unscale_(model_optimizer)

        model_params = _build_model_optimizer_params(model, dcsla_controller)
        torch.nn.utils.clip_grad_norm_(model_params, float(max_grad_norm_model))

        scaler.step(model_optimizer)
        scaler.update()

    else:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=(dcsla_controller is not None),
            use_cache=False,
        )

        lm_loss = _get_output_loss(outputs)

        if dcsla_controller is not None:
            hidden_states = _get_output_hidden_states(outputs)
            dcsla_loss, dcsla_stats = dcsla_aux_loss(
                model=model,
                selector=dcsla_controller.selector,
                hidden_states_all=hidden_states,
                attention_mask=attention_mask,
                deficits=state["deficits"],
                cfg=dcsla_cfg,
            )
            total_model_loss = lm_loss + dcsla_loss
        else:
            dcsla_loss = lm_loss.new_zeros(())
            total_model_loss = lm_loss

        _ensure_finite_scalar("lm_loss", lm_loss)
        _ensure_finite_scalar("dcsla_loss", dcsla_loss)
        _ensure_finite_scalar("total_model_loss", total_model_loss)

        total_model_loss.backward()

        model_params = _build_model_optimizer_params(model, dcsla_controller)
        torch.nn.utils.clip_grad_norm_(model_params, float(max_grad_norm_model))

        model_optimizer.step()

    router_loss = None
    reward = None

    if router_optimizer is not None:
        router_optimizer.zero_grad(set_to_none=True)

        reward = -lm_loss.detach()

        overspend = F.relu(sampled["spent"].detach() - float(effective_budget))
        reward = reward - float(budget_penalty_coef) * overspend / max(float(effective_budget), 1.0)

        _ensure_finite_scalar("router reward", reward)

        if train_state.reward_ema is None:
            train_state.reward_ema = float(reward.item())

        baseline = torch.tensor(train_state.reward_ema, dtype=reward.dtype, device=reward.device)
        advantage = reward - baseline

        train_state.reward_ema = 0.95 * float(train_state.reward_ema) + 0.05 * float(reward.item())

        policy_loss = -(advantage * sampled["log_prob_mean"])
        entropy_bonus = sampled["entropy_mean"]
        aux_loss = -router_out["objective"] / max(int(router_out["transport_plan"].size(0)), 1)

        router_loss = (
            policy_loss
            - float(entropy_coef) * entropy_bonus
            + float(aux_objective_coef) * aux_loss
        )

        _ensure_finite_scalar("router_loss", router_loss)

        router_loss.backward()
        torch.nn.utils.clip_grad_norm_(router.parameters(), float(max_grad_norm_router))
        router_optimizer.step()

    train_state.global_step += 1

    metrics = {
        "lm_loss": float(lm_loss.detach().item()),
        "dcsla_loss": float(dcsla_loss.detach().item()),
        "router_loss": None if router_loss is None else float(router_loss.detach().item()),
        "reward": None if reward is None else float(reward.detach().item()),
        "spent": float(sampled["spent"].detach().item()),
        "effective_budget": effective_budget,
        "schedule": schedule.detach().cpu().tolist(),
        "schedule_names": schedule_to_names(schedule, router.protocol_table),
        "quotas": router_out["quotas"].detach().cpu().tolist(),
    }

    if dcsla_stats is not None:
        metrics["dcsla_util"] = float(dcsla_stats["util"].detach().item())
        metrics["dcsla_div"] = float(dcsla_stats["div"].detach().item())
        metrics["dcsla_entropy"] = float(dcsla_stats["entropy"].detach().item())

    return metrics


def train_lasperht(
    model,
    router,
    train_loader,
    val_loader,
    num_epochs=1,
    total_budget=None,
    lr_model=3e-4,
    lr_router=1e-4,
    weight_decay_model=0.01,
    weight_decay_router=1e-4,
    eval_every_steps=100,
    max_eval_batches=10,
    route_temperature=1.0,
    dcsla_controller=None,
    dcsla_cfg: Optional[DCSLAConfig] = None,
):
    _validate_train_eval_inputs(model, router)

    num_epochs = int(num_epochs)
    if num_epochs <= 0:
        raise ValueError("num_epochs must be > 0")

    if not math.isfinite(float(lr_model)) or float(lr_model) <= 0.0:
        raise ValueError("lr_model must be finite and > 0")
    if not math.isfinite(float(lr_router)) or float(lr_router) <= 0.0:
        raise ValueError("lr_router must be finite and > 0")

    device = _runtime_device(model, router)
    amp_enabled = _amp_enabled_for_device(device)

    model_params = _build_model_optimizer_params(model, dcsla_controller)
    router_params = _build_router_optimizer_params(router)

    model_optimizer = AdamW(
        model_params,
        lr=float(lr_model),
        weight_decay=float(weight_decay_model),
    )

    router_optimizer = AdamW(
        router_params,
        lr=float(lr_router),
        weight_decay=float(weight_decay_router),
    )

    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    train_state = RouterTrainState()
    history = []

    for epoch in range(num_epochs):
        for step, batch in enumerate(train_loader):
            metrics = routed_train_step(
                model=model,
                router=router,
                batch=batch,
                model_optimizer=model_optimizer,
                router_optimizer=router_optimizer,
                scaler=scaler,
                train_state=train_state,
                total_budget=total_budget,
                route_temperature=route_temperature,
                entropy_coef=0.01,
                aux_objective_coef=0.10,
                budget_penalty_coef=0.05,
                max_grad_norm_model=1.0,
                max_grad_norm_router=1.0,
                dcsla_controller=dcsla_controller,
                dcsla_cfg=dcsla_cfg,
            )

            if step % 10 == 0:
                msg = (
                    f"[epoch {epoch} step {step}] "
                    f"lm_loss={metrics['lm_loss']:.4f} "
                    f"dcsla_loss={metrics['dcsla_loss']:.4f} "
                    f"router_loss={metrics['router_loss'] if metrics['router_loss'] is not None else 'NA'} "
                    f"spent={metrics['spent']:.2f} "
                    f"budget={metrics['effective_budget']:.2f}"
                )

                if "dcsla_util" in metrics:
                    msg += (
                        f" util={metrics['dcsla_util']:.4f}"
                        f" div={metrics['dcsla_div']:.4f}"
                        f" ent={metrics['dcsla_entropy']:.4f}"
                    )

                print(msg)

            if (
                eval_every_steps is not None
                and int(eval_every_steps) > 0
                and train_state.global_step > 0
                and (train_state.global_step % int(eval_every_steps) == 0)
            ):
                val_metrics = routed_eval_epoch(
                    model=model,
                    router=router,
                    loader=val_loader,
                    total_budget=total_budget,
                    max_batches=max_eval_batches,
                    dcsla_controller=dcsla_controller,
                    dcsla_cfg=dcsla_cfg,
                )

                print(
                    f"  -> eval: loss={val_metrics['loss']:.4f}, "
                    f"spend={val_metrics['spend']:.2f}, "
                    f"proto_hist={val_metrics['proto_hist']}"
                )

                history.append({
                    "epoch": epoch,
                    "global_step": train_state.global_step,
                    "train_lm_loss": metrics["lm_loss"],
                    "train_dcsla_loss": metrics["dcsla_loss"],
                    "val_loss": val_metrics["loss"],
                    "val_spend": val_metrics["spend"],
                    "val_proto_hist": val_metrics["proto_hist"],
                })

    return {
        "model_optimizer": model_optimizer,
        "router_optimizer": router_optimizer,
        "train_state": train_state,
        "history": history,
    }

In [ ]:
# ============================================================
# CELL 8: Default protocol table fallback
# Custom GCCD decoder path
#   - Uses existing PROTOCOL_TABLE if already defined
#   - Defines DEFAULT_PROTOCOL_TABLE otherwise
#   - LCHA-enabled protocol menu
#   - No Mistral-specific wiring
# ============================================================
import math

if "ProtocolSpec" not in globals():
    raise NameError("ProtocolSpec is not defined. Run the ProtocolSpec cell before Cell 8.")


def _validate_protocol_table_basic(protocol_table):
    if protocol_table is None or len(protocol_table) == 0:
        raise ValueError("protocol_table must be non-empty")

    seen = set()
    ratio_sum = 0.0

    for i, p in enumerate(protocol_table):
        if not isinstance(p, ProtocolSpec):
            raise TypeError(f"protocol_table[{i}] must be ProtocolSpec, got {type(p)}")

        if not isinstance(p.name, str) or len(p.name) == 0:
            raise ValueError(f"protocol_table[{i}] has invalid name")

        if p.name in seen:
            raise ValueError(f"Duplicate protocol name: {p.name}")
        seen.add(p.name)

        if int(p.w) < 0:
            raise ValueError(f"{p.name}: w must be >= 0")

        if int(p.scaffold_mode) not in (0, 1, 2):
            raise ValueError(f"{p.name}: scaffold_mode must be 0, 1, or 2")

        if int(p.ring_inner) < 0 or int(p.ring_outer) < 0:
            raise ValueError(f"{p.name}: ring_inner/ring_outer must be >= 0")

        if int(p.ring_outer) < int(p.ring_inner):
            raise ValueError(f"{p.name}: ring_outer must be >= ring_inner")

        if int(p.num_anchors) < 0:
            raise ValueError(f"{p.name}: num_anchors must be >= 0")

        if not math.isfinite(float(p.quota_ratio)) or float(p.quota_ratio) < 0.0:
            raise ValueError(f"{p.name}: quota_ratio must be finite and >= 0")

        if p.cost is not None:
            if not math.isfinite(float(p.cost)) or float(p.cost) < 0.0:
                raise ValueError(f"{p.name}: cost must be finite and >= 0 when provided")

        if int(getattr(p, "lcha_order", 0)) not in (0, 1, 2):
            raise ValueError(f"{p.name}: lcha_order must be 0, 1, or 2")

        if not math.isfinite(float(getattr(p, "lcha_heads_frac", 0.0))):
            raise ValueError(f"{p.name}: lcha_heads_frac must be finite")

        if not (0.0 <= float(getattr(p, "lcha_heads_frac", 0.0)) <= 1.0):
            raise ValueError(f"{p.name}: lcha_heads_frac must be in [0, 1]")

        if int(getattr(p, "lcha_row_chunk_size", 32)) <= 0:
            raise ValueError(f"{p.name}: lcha_row_chunk_size must be > 0")

        ratio_sum += float(p.quota_ratio)

    if ratio_sum <= 0.0:
        raise ValueError("Sum of protocol quota_ratio values must be > 0")

    return protocol_table


DEFAULT_PROTOCOL_TABLE = [
    ProtocolSpec(
        name="HYBRID",
        w=32,
        scaffold_mode=2,
        ring_inner=64,
        ring_outer=128,
        num_anchors=4,
        include_self=True,
        anchor_strategy=1,
        anchor_param0=0,
        quota_ratio=0.35,
        cost=None,

        # ---- LCHA ----
        lcha_enabled=True,
        lcha_order=1,
        lcha_heads_frac=0.25,
        lcha_use_v=False,
        lcha_alpha0=0.90,
        lcha_alpha1=0.10,
        lcha_alpha2=0.00,
        lcha_v_alpha0=1.00,
        lcha_v_alpha1=0.00,
        lcha_v_alpha2=0.00,
        lcha_row_chunk_size=32,
    ),
    ProtocolSpec(
        name="GLOBAL_REFRESH",
        w=16,
        scaffold_mode=2,
        ring_inner=128,
        ring_outer=256,
        num_anchors=12,
        include_self=True,
        anchor_strategy=1,
        anchor_param0=0,
        quota_ratio=0.25,
        cost=None,

        # ---- LCHA ----
        lcha_enabled=True,
        lcha_order=2,
        lcha_heads_frac=0.50,
        lcha_use_v=True,
        lcha_alpha0=0.75,
        lcha_alpha1=0.20,
        lcha_alpha2=0.05,
        lcha_v_alpha0=0.90,
        lcha_v_alpha1=0.10,
        lcha_v_alpha2=0.00,
        lcha_row_chunk_size=32,
    ),
    ProtocolSpec(
        name="ECONOMY",
        w=16,
        scaffold_mode=0,
        ring_inner=0,
        ring_outer=0,
        num_anchors=0,   # local/window-only economy path
        include_self=True,
        anchor_strategy=0,
        anchor_param0=0,
        quota_ratio=0.25,
        cost=None,

        # ---- LCHA ----
        lcha_enabled=False,
        lcha_order=0,
        lcha_heads_frac=0.0,
        lcha_use_v=False,
        lcha_alpha0=1.00,
        lcha_alpha1=0.00,
        lcha_alpha2=0.00,
        lcha_v_alpha0=1.00,
        lcha_v_alpha1=0.00,
        lcha_v_alpha2=0.00,
        lcha_row_chunk_size=32,
    ),
    ProtocolSpec(
        name="MULTISCALE",
        w=24,
        scaffold_mode=2,
        ring_inner=64,
        ring_outer=256,
        num_anchors=8,
        include_self=True,
        anchor_strategy=1,
        anchor_param0=0,
        quota_ratio=0.15,
        cost=None,

        # ---- LCHA ----
        lcha_enabled=True,
        lcha_order=1,
        lcha_heads_frac=0.50,
        lcha_use_v=False,
        lcha_alpha0=0.85,
        lcha_alpha1=0.15,
        lcha_alpha2=0.00,
        lcha_v_alpha0=1.00,
        lcha_v_alpha1=0.00,
        lcha_v_alpha2=0.00,
        lcha_row_chunk_size=32,
    ),
]

_validate_protocol_table_basic(DEFAULT_PROTOCOL_TABLE)

# Fallback behavior:
# - If PROTOCOL_TABLE already exists, keep it.
# - If OVERRIDE_PROTOCOL_TABLE=True, replace it with DEFAULT_PROTOCOL_TABLE.
OVERRIDE_PROTOCOL_TABLE = bool(globals().get("OVERRIDE_PROTOCOL_TABLE", False))

if "PROTOCOL_TABLE" not in globals() or PROTOCOL_TABLE is None or OVERRIDE_PROTOCOL_TABLE:
    PROTOCOL_TABLE = DEFAULT_PROTOCOL_TABLE
else:
    PROTOCOL_TABLE = _validate_protocol_table_basic(PROTOCOL_TABLE)


def materialize_protocol_table_for_seq_len(protocol_table, seq_len: int):
    """
    Converts raw ProtocolSpec(cost=None) into ProtocolSpec(cost=float).
    Use this before constructing LaSPERHTRouter.
    """
    if "ensure_protocol_costs" not in globals():
        raise NameError("ensure_protocol_costs is not defined. Run the ProtocolSpec/helper cell first.")

    seq_len = int(seq_len)
    if seq_len <= 0:
        raise ValueError("seq_len must be > 0")

    protocol_table = _validate_protocol_table_basic(protocol_table)
    materialized = ensure_protocol_costs(protocol_table, seq_len)
    materialized = _validate_protocol_table_basic(materialized)

    for p in materialized:
        if p.cost is None:
            raise ValueError(f"{p.name}: cost is still None after materialization")

    return materialized


print("PROTOCOL_TABLE:", [p.name for p in PROTOCOL_TABLE])
print("quota_sum:", sum(float(p.quota_ratio) for p in PROTOCOL_TABLE))
print("costs_materialized:", all(p.cost is not None for p in PROTOCOL_TABLE))

In [ ]:
# ============================================================
# CELL 9: Bootstrap LaSPER-HT
# Custom GCCD decoder path
#   - infer custom decoder summary dim
#   - calibrate/materialize protocol costs
#   - build LaSPER-HT router
#   - run one routing sanity pass
#   - no Mistral-specific wiring
# ============================================================
import math
import torch


def _bootstrap_runtime_device(model):
    if model is None:
        raise ValueError("model must not be None")

    if hasattr(model, "parameters"):
        try:
            return next(model.parameters()).device
        except StopIteration:
            pass

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _bootstrap_get_loader():
    if "val_loader" in globals() and val_loader is not None:
        return val_loader, "val_loader"
    if "train_loader" in globals() and train_loader is not None:
        return train_loader, "train_loader"
    raise NameError("Neither val_loader nor train_loader is defined.")


def _bootstrap_first_batch(loader, loader_name: str):
    try:
        batch = next(iter(loader))
    except StopIteration:
        raise ValueError(f"{loader_name} is empty.")

    if not isinstance(batch, dict):
        raise TypeError(f"{loader_name} must yield dict batches.")

    if "input_ids" not in batch:
        raise KeyError(f"{loader_name} batch must contain input_ids.")

    return batch


def _bootstrap_batch_to_device(batch, device):
    return {
        k: (v.to(device) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }


def _validate_bootstrap_protocol_table(protocol_table):
    if protocol_table is None or len(protocol_table) == 0:
        raise ValueError("PROTOCOL_TABLE must be non-empty.")

    for i, p in enumerate(protocol_table):
        if not isinstance(p, ProtocolSpec):
            raise TypeError(f"PROTOCOL_TABLE[{i}] must be ProtocolSpec, got {type(p)}")

        if p.cost is None:
            raise ValueError(
                f"PROTOCOL_TABLE[{i}] / {p.name} has cost=None after calibration. "
                "Protocol costs must be materialized before building LaSPERHTRouter."
            )

        cost = float(p.cost)
        if not math.isfinite(cost) or cost < 0.0:
            raise ValueError(f"Invalid cost for protocol {p.name}: {cost}")

        ratio = float(p.quota_ratio)
        if not math.isfinite(ratio) or ratio < 0.0:
            raise ValueError(f"Invalid quota_ratio for protocol {p.name}: {ratio}")

    ratio_sum = sum(float(p.quota_ratio) for p in protocol_table)
    if ratio_sum <= 0.0:
        raise ValueError("Protocol quota_ratio values must sum to a positive value.")

    return protocol_table


# ---- required symbol checks ----
if "model" not in globals():
    raise NameError("model is not defined.")

if "PROTOCOL_TABLE" not in globals():
    raise NameError("PROTOCOL_TABLE is not defined. Run Cell 8 first.")

if "infer_summary_dim_from_model" not in globals():
    raise NameError("infer_summary_dim_from_model is not defined. Run the custom decoder hook cell first.")

if "calibrate_protocol_costs_from_probe" not in globals():
    raise NameError("calibrate_protocol_costs_from_probe is not defined.")

if "LaSPERHTRouter" not in globals():
    raise NameError("LaSPERHTRouter is not defined. Run Cell 5 first.")

if "route_batch_lasperht" not in globals():
    raise NameError("route_batch_lasperht is not defined. Run Cell 6 first.")

if "schedule_to_names" not in globals():
    raise NameError("schedule_to_names is not defined. Run the schedule hook cell first.")


DEVICE = _bootstrap_runtime_device(model)

loader, loader_name = _bootstrap_get_loader()
_probe_batch = _bootstrap_first_batch(loader, loader_name)
_probe_batch = _bootstrap_batch_to_device(_probe_batch, DEVICE)

SUMMARY_DIM = infer_summary_dim_from_model(model)
SUMMARY_DIM = int(SUMMARY_DIM)

if SUMMARY_DIM <= 0:
    raise ValueError(f"Invalid SUMMARY_DIM: {SUMMARY_DIM}")

print("SUMMARY_DIM:", SUMMARY_DIM)

PROTOCOL_TABLE = calibrate_protocol_costs_from_probe(
    model=model,
    probe_batch=_probe_batch,
    protocol_specs=PROTOCOL_TABLE,
    routing_seq_len=None,
)

PROTOCOL_TABLE = _validate_bootstrap_protocol_table(PROTOCOL_TABLE)

print("Calibrated protocol costs:")
for p in PROTOCOL_TABLE:
    print(f"  {p.name:15s} cost={float(p.cost):.4f}")

router = LaSPERHTRouter(
    summary_dim=SUMMARY_DIM,
    protocol_table=PROTOCOL_TABLE,
    deficit_dim=6,
    d_router=128,
    lambda_cost=0.8,
    mu_usage=0.15,
    tau=0.35,
    eta_harm=0.20,
    hdtr_iters=6,
    sinkhorn_iters=50,
    switch_penalty=0.10,
    layer_knn=0,   # try 4 later for a kNN layer graph
).to(DEVICE)

# ============================================================
# Routing sanity pass
# NOTE: this applies the chosen schedule to the model.
# ============================================================
batch = _bootstrap_first_batch(loader, loader_name)
batch = _bootstrap_batch_to_device(batch, DEVICE)

router_out, route_state = route_batch_lasperht(
    model=model,
    router=router,
    input_ids=batch["input_ids"],
    attention_mask=batch.get("attention_mask", None),
    deficits=None,
    total_budget=None,
    prev_schedule=None,
)

schedule = router_out["schedule"].detach()
quotas = router_out["quotas"].detach()

if schedule.numel() <= 0:
    raise ValueError("Router produced an empty schedule.")

if not torch.isfinite(router_out["objective"].detach().float()):
    raise ValueError("Router objective contains NaN/Inf.")

if "clear_custom_candidate_caches" in globals():
    clear_custom_candidate_caches(model)
elif "clear_sasa_candidate_caches" in globals():
    clear_sasa_candidate_caches(model)

print("Chosen protocols:", schedule_to_names(schedule, PROTOCOL_TABLE))
print("Quotas:", quotas.cpu().tolist())
print("Schedule ids:", schedule.cpu().tolist())
print("HDTR objective:", float(router_out["objective"].detach().item()))

NameError: name 'model' is not defined

In [ ]:
# ============================================================
# NEW CELL: Bootstrap DCSLA on top of LaSPER-routed custom GCCD decoder
# Place this after LaSPER-HT router/bootstrap cells
# ============================================================
import torch

# ---- wiring checks ----
if "model" not in globals():
    raise NameError("model is not defined.")

if "infer_summary_dim_from_model" not in globals():
    raise NameError("infer_summary_dim_from_model is not defined.")

if "DCSLAConfig" not in globals():
    raise NameError("DCSLAConfig is not defined.")

if "DCSLASelector" not in globals():
    raise NameError("DCSLASelector is not defined.")

if "install_dcsla_on_model" not in globals():
    raise NameError("install_dcsla_on_model is not defined.")

if "NotebookDCSLAController" not in globals():
    raise NameError("NotebookDCSLAController is not defined.")

if "find_custom_sparse_attention_layers" not in globals():
    raise NameError("find_custom_sparse_attention_layers is not defined.")

# Use actual model device, not stale global DEVICE.
DEVICE = next(model.parameters()).device

DCSLA_SUMMARY_DIM = int(infer_summary_dim_from_model(model))
if DCSLA_SUMMARY_DIM <= 0:
    raise ValueError(f"Invalid DCSLA_SUMMARY_DIM: {DCSLA_SUMMARY_DIM}")

print("DCSLA_SUMMARY_DIM:", DCSLA_SUMMARY_DIM)

dcsla_cfg = DCSLAConfig(
    block_size=64,
    heat_tau=0.7,
    alpha_heat=1.0,
    beta_bridge=0.5,
    gamma_deficit=1.0,
    delta_redundancy=0.35,
    landmark_dropout=0.0,
    use_layernorm=True,
    causal_only=True,
    refresh_every=1,   # safest setting first; can raise to 2/4 later
    util_tau_min=0.20,
    lambda_util=1e-2,
    lambda_div=1e-3,
    lambda_entropy=1e-3,
)

dcsla_selector = DCSLASelector(
    hidden_size=DCSLA_SUMMARY_DIM,
    cfg=dcsla_cfg,
).to(DEVICE)


custom_attention_layers = find_custom_sparse_attention_layers(model)
if len(custom_attention_layers) == 0:
    raise ValueError("No custom sparse attention layers found before DCSLA install.")

bad_layers = []
for li, layer in custom_attention_layers:
    if getattr(layer, "candidate_builder_fn", None) is None:
        bad_layers.append(li)

if bad_layers:
    raise ValueError(
        "DCSLA cannot install because candidate_builder_fn is None on layers "
        f"{bad_layers}. Re-run the adapter cell and Cell 37 model-instantiation cell. "
        "Cell 37 must pass candidate_builder_fn=stagea_adapter into build_gccd_backbone_from_existing(...)."
    )

install_dcsla_on_model(model, dcsla_selector)

custom_attention_layers = find_custom_sparse_attention_layers(model)
if len(custom_attention_layers) == 0:
    raise ValueError("DCSLA install failed: no custom sparse attention layers found.")

print(f"DCSLA installed on {len(custom_attention_layers)} custom sparse attention layers")
print("Per-layer landmark counts currently read from layer.num_anchors:")
print([int(getattr(layer, "num_anchors", 0)) for _, layer in custom_attention_layers[:12]])

dcsla_controller = NotebookDCSLAController(
    model=model,
    selector=dcsla_selector,
    refresh_every=dcsla_cfg.refresh_every,
)

NameError: name 'model' is not defined

In [ ]:
# ============================================================
# CELL 10: Example train / eval run
# Custom GCCD decoder path
#   - LaSPER-HT routing
#   - DCSLA auxiliary training
#   - No Mistral-specific wiring
# ============================================================
import math

# ---- required wiring checks ----
required_names = [
    "model",
    "router",
    "train_loader",
    "val_loader",
    "routed_eval_epoch",
    "train_lasperht",
    "dcsla_controller",
    "dcsla_cfg",
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is not defined. Run the required previous cells before Cell 10.")

if model is None:
    raise ValueError("model must not be None.")

if router is None:
    raise ValueError("router must not be None.")

if dcsla_controller is None:
    raise ValueError("dcsla_controller must not be None for this LaSPER + DCSLA run.")

if dcsla_cfg is None:
    raise ValueError("dcsla_cfg must not be None for this LaSPER + DCSLA run.")

if not hasattr(router, "protocol_table") or len(router.protocol_table) == 0:
    raise ValueError("router.protocol_table must be non-empty.")

if not hasattr(dcsla_controller, "selector"):
    raise TypeError("dcsla_controller must expose selector.")

# Example:
# total_budget = 800.0
# Or keep None to use the default quota-weighted budget.
total_budget = None

if total_budget is not None:
    total_budget = float(total_budget)
    if not math.isfinite(total_budget) or total_budget < 0.0:
        raise ValueError("total_budget must be None or a finite non-negative float.")

num_epochs = 1
lr_model = 3e-4
lr_router = 1e-4
eval_every_steps = 50
max_eval_batches = 5
route_temperature = 1.0

if num_epochs <= 0:
    raise ValueError("num_epochs must be > 0.")

if not math.isfinite(float(lr_model)) or float(lr_model) <= 0.0:
    raise ValueError("lr_model must be finite and > 0.")

if not math.isfinite(float(lr_router)) or float(lr_router) <= 0.0:
    raise ValueError("lr_router must be finite and > 0.")

if max_eval_batches is not None and int(max_eval_batches) <= 0:
    raise ValueError("max_eval_batches must be None or > 0.")

if eval_every_steps is not None and int(eval_every_steps) <= 0:
    raise ValueError("eval_every_steps must be None or > 0.")

if not math.isfinite(float(route_temperature)) or float(route_temperature) <= 0.0:
    raise ValueError("route_temperature must be finite and > 0.")


val0 = routed_eval_epoch(
    model=model,
    router=router,
    loader=val_loader,
    total_budget=total_budget,
    max_batches=max_eval_batches,
    dcsla_controller=dcsla_controller,
    dcsla_cfg=dcsla_cfg,
)
print("val_before:", val0)


train_out = train_lasperht(
    model=model,
    router=router,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=num_epochs,
    total_budget=total_budget,
    lr_model=lr_model,
    lr_router=lr_router,
    eval_every_steps=eval_every_steps,
    max_eval_batches=max_eval_batches,
    route_temperature=route_temperature,
    dcsla_controller=dcsla_controller,
    dcsla_cfg=dcsla_cfg,
)


val1 = routed_eval_epoch(
    model=model,
    router=router,
    loader=val_loader,
    total_budget=total_budget,
    max_batches=max_eval_batches,
    dcsla_controller=dcsla_controller,
    dcsla_cfg=dcsla_cfg,
)
print("val_after:", val1)

NameError: name 'model' is not defined

In [ ]:
# ============================================================
# OPTIONAL SANITY CHECK CELL
#   Uses actual LaSPER-HT router + DCSLA controller
#   Custom GCCD decoder path
# ============================================================
import torch

# ---- wiring checks ----
required_names = [
    "model",
    "router",
    "route_batch_lasperht",
    "dcsla_controller",
    "schedule_to_names",
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is not defined. Run the required previous cells first.")

if "val_loader" in globals() and val_loader is not None:
    loader = val_loader
elif "train_loader" in globals() and train_loader is not None:
    loader = train_loader
else:
    raise NameError("Neither val_loader nor train_loader is defined.")

try:
    batch = next(iter(loader))
except StopIteration:
    raise ValueError("Selected loader is empty.")

if not isinstance(batch, dict):
    raise TypeError("Loader must yield dict batches.")

if "input_ids" not in batch:
    raise KeyError("Batch must contain input_ids.")

DEVICE = next(model.parameters()).device

batch = {
    k: (v.to(DEVICE) if torch.is_tensor(v) else v)
    for k, v in batch.items()
}

router_out, route_state = route_batch_lasperht(
    model=model,
    router=router,
    input_ids=batch["input_ids"],
    attention_mask=batch.get("attention_mask", None),
    deficits=None,
    total_budget=None,
    prev_schedule=None,
)

# Do NOT call apply_schedule_to_model again.
# route_batch_lasperht(...) already applies router_out["schedule"] to the model.

dcsla_info = dcsla_controller.refresh_after_schedule(
    input_ids=batch["input_ids"],
    attention_mask=batch.get("attention_mask", None),
    deficits=route_state["deficits"],
    force=True,
)

print("Chosen protocols:", schedule_to_names(router_out["schedule"], router.protocol_table))
print("Quotas:", router_out["quotas"].detach().cpu().tolist())
print("Schedule ids:", router_out["schedule"].detach().cpu().tolist())
print("DCSLA util_proxy:", float(dcsla_info["util_proxy"].detach().item()))
print("DCSLA div_proxy :", float(dcsla_info["div_proxy"].detach().item()))

NameError: name 'train_loader' is not defined

CHECKED TILL ABOVE

In [ ]:
# sasa_layers = find_sasa_layers(model)
# for i, (_, layer) in enumerate(sasa_layers[:5]):
#     print(
#         i,
#         "w=", layer.w,
#         "anchors=", layer.num_anchors,
#         "mode=", layer.scaffold_mode,
#         "ring=", (layer.ring_inner, layer.ring_outer),
#         "astrat=", layer.anchor_strategy,
#     )

NameError: name 'model' is not defined

In [ ]:
# # Optimizer for the patched model parameters
# # Includes DCSLA selector params when dcsla_controller is active.
# optimizer = torch.optim.AdamW(
#     _build_model_optimizer_params(
#         model,
#         dcsla_controller if "dcsla_controller" in globals() else None,
#     ),
#     lr=LR if "LR" in globals() else 3e-4,
#     weight_decay=0.01,
# )

# n_total = sum(p.numel() for group in optimizer.param_groups for p in group["params"])
# print("optimizer ready")
# print("num optimizer params:", n_total)
# print("includes DCSLA selector:", bool("dcsla_controller" in globals() and dcsla_controller is not None))

NameError: name 'model' is not defined

In [ ]:
# # ============================================================
# # OPTIONAL QUICK LOSS SMOKE HELPERS
# #   Uses LaSPER router + TRAINABLE DCSLA
# # ============================================================
# @torch.no_grad()
# def eval_loss_only_routed(
#     model,
#     loader,
#     router,
#     dcsla_controller=None,
#     dcsla_cfg=None,
#     total_budget=None,
#     max_batches=None,
# ):
#     model.eval()
#     router.eval()
#     if dcsla_controller is not None:
#         dcsla_controller.selector.eval()

#     losses = []
#     prev_schedule = None

#     for bi, batch in enumerate(loader):
#         if max_batches is not None and bi >= max_batches:
#             break

#         batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
#         input_ids = batch["input_ids"]
#         attention_mask = batch.get("attention_mask", None)
#         labels = batch.get("labels", input_ids)

#         router_out, route_state = route_batch_lasperht(
#             model=model,
#             router=router,
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             deficits=None,
#             total_budget=total_budget,
#             prev_schedule=prev_schedule,
#         )

#         apply_schedule_to_model(model, router_out["schedule"], router.protocol_table)
#         prev_schedule = router_out["schedule"].detach().clone()

#         if dcsla_controller is not None:
#             dcsla_controller.refresh_after_schedule(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 deficits=route_state["deficits"],
#                 force=True,
#             )

#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels,
#             output_hidden_states=(dcsla_controller is not None),
#             use_cache=False,
#         )

#         loss = outputs.loss
#         if dcsla_controller is not None:
#             aux_loss, _ = dcsla_aux_loss(
#                 model=model,
#                 selector=dcsla_controller.selector,
#                 hidden_states_all=outputs.hidden_states,
#                 attention_mask=attention_mask,
#                 deficits=route_state["deficits"],
#                 cfg=dcsla_cfg,
#             )
#             loss = loss + aux_loss

#         losses.append(loss.detach().float().item())

#     return sum(losses) / max(len(losses), 1)


# def train_n_steps_routed(
#     model,
#     loader,
#     optimizer,
#     router,
#     dcsla_controller=None,
#     dcsla_cfg=None,
#     n_steps=20,
#     total_budget=None,
# ):
#     model.train()
#     router.eval()
#     if dcsla_controller is not None:
#         dcsla_controller.selector.train()

#     losses = []
#     it = iter(loader)
#     prev_schedule = None

#     for step in range(n_steps):
#         try:
#             batch = next(it)
#         except StopIteration:
#             it = iter(loader)
#             batch = next(it)

#         batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
#         input_ids = batch["input_ids"]
#         attention_mask = batch.get("attention_mask", None)
#         labels = batch.get("labels", input_ids)

#         with torch.no_grad():
#             router_out, route_state = route_batch_lasperht(
#                 model=model,
#                 router=router,
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 deficits=None,
#                 total_budget=total_budget,
#                 prev_schedule=prev_schedule,
#             )

#             apply_schedule_to_model(model, router_out["schedule"], router.protocol_table)
#             prev_schedule = router_out["schedule"].detach().clone()

#             if dcsla_controller is not None:
#                 dcsla_controller.refresh_after_schedule(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     deficits=route_state["deficits"],
#                     force=True,
#                 )

#         optimizer.zero_grad(set_to_none=True)

#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels,
#             output_hidden_states=(dcsla_controller is not None),
#             use_cache=False,
#         )

#         loss = outputs.loss
#         if dcsla_controller is not None:
#             aux_loss, _ = dcsla_aux_loss(
#                 model=model,
#                 selector=dcsla_controller.selector,
#                 hidden_states_all=outputs.hidden_states,
#                 attention_mask=attention_mask,
#                 deficits=route_state["deficits"],
#                 cfg=dcsla_cfg,
#             )
#             loss = loss + aux_loss

#         loss.backward()
#         optimizer.step()
#         losses.append(loss.detach().float().item())

#     return sum(losses) / max(len(losses), 1)

In [ ]:
# val_before = eval_loss_only_routed(
#     model,
#     val_loader,
#     router,
#     dcsla_controller=dcsla_controller,
#     dcsla_cfg=dcsla_cfg,
#     total_budget=None,
#     max_batches=5,
# )
# print("val_before:", val_before)

# train_loss = train_n_steps_routed(
#     model,
#     train_loader,
#     optimizer,
#     router,
#     dcsla_controller=dcsla_controller,
#     dcsla_cfg=dcsla_cfg,
#     n_steps=5,
#     total_budget=None,
# )
# print("train_loss:", train_loss)

# val_after = eval_loss_only_routed(
#     model,
#     val_loader,
#     router,
#     dcsla_controller=dcsla_controller,
#     dcsla_cfg=dcsla_cfg,
#     total_budget=None,
#     max_batches=5,
# )
# print("val_after:", val_after)

In [ ]:
# Deprecated stray cell removed on purpose.
# Keep this cell as a no-op to avoid accidental execution errors.

NameError: name 'L' is not defined

SMOKE TEST

In [ ]:
# # ============================================================
# # REAL-DATA SMOKE: cand invariant + NaN localization (Stage-E dV vs repeat_interleave)
# # ============================================================
# import torch, gc

# # ---- smoke parameters you asked for
# STAGEE_ASSERT_CAND_INVARIANTS = True
# STAGEE_ASSERT_CAND_CAUSAL     = True   # set False only if you intentionally run non-causal
# STAGEE_TRACE_DV               = True
# STAGEE_TRACE_MAX_PRINT        = 8

# SASA_HOOK_GRAD_INTO_DV_REPEAT_INTERLEAVE      = True
# SASA_HOOK_GRAD_INTO_DV_REPEAT_INTERLEAVE_ONCE = True
# _SASA_HOOK_DV_BHTD_FIRED = False

# torch.autograd.set_detect_anomaly(True)

# # ---- PATCH C (debug logs only): global run tag + backward call counter ----
# if "_STAGEE_RUN_TAG" not in globals():
#     _STAGEE_RUN_TAG = ""
# if "_STAGEE_BWD_CALL_COUNTER" not in globals():
#     _STAGEE_BWD_CALL_COUNTER = 0

# # ---- hard requirements
# req = ["model", "train_loader", "_to_device", "SASACandCacheContext"]
# missing = [x for x in req if x not in globals()]
# assert not missing, f"Missing required globals: {missing}"

# device = next(model.parameters()).device
# model.train()

# batch = next(iter(train_loader))
# batch = _to_device(batch, device)

# input_ids = batch["input_ids"]
# attention_mask = batch.get("attention_mask", None)
# labels = batch.get("labels", input_ids)

# opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4)
# scaler = torch.amp.GradScaler("cuda", enabled=False)

# opt.zero_grad(set_to_none=True)

# with SASACandCacheContext():
#     _amp_dtype = torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
#     with torch.autocast(device_type=device.type, dtype=_amp_dtype, enabled=(device.type=="cuda")):


#         out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)
#         loss = out.loss

# print("loss (fwd) =", float(loss.detach().cpu()))

# # ---- PATCH C (debug logs only): reset tag + counter right before backward ----
# globals()["_STAGEE_RUN_TAG"] = "smoke_one_backward"
# globals()["_STAGEE_BWD_CALL_COUNTER"] = 0

# loss.backward()
# print("backward done (check logs above for Stage-E dV + dv_bhtd grad hook)")



# import torch, math, traceback
# from collections import defaultdict

# # ============================================================
# # Controls
# # ============================================================
# FUSED_SYNC_AFTER_KERNEL = True

# # Run the Stage-E "truth test" (PyTorch ref vs fused) ONCE total (first mh_forward call)
# STAGEE_TRUTH_TEST_ENABLE = True
# STAGEE_TRUTH_TEST_ONCE   = True

# # For the ref compare, compute both:
# #  - ref_no_causal : uses mask only
# #  - ref_causal    : also drops cand > q_idx
# STAGEE_REF_DO_CAUSAL_COMPARE = True

# # ----------------------------
# # NEW: Stage-E fwd<->bwd consistency + ref-gradcheck (focus: backward)
# # ----------------------------
# STAGEE_BWD_CONSISTENCY_CHECK_ENABLE = True

# STAGEE_BWD_REF_GRADCHECK_ENABLE = True
# STAGEE_BWD_REF_GRADCHECK_ONCE   = True
# STAGEE_BWD_REF_GRADCHECK_HEADS  = 2     # check first Hs heads
# STAGEE_BWD_REF_GRADCHECK_D      = 64    # check first Ds dims

# # ============================================================
# # Small tensor summary helpers
# # ============================================================
# def _tinfo(x: torch.Tensor, name: str, max_abs_items=0):
#     if x is None:
#         print(f"[{name}] = None")
#         return
#     dev = str(x.device)
#     print(f"[{name}] shape={tuple(x.shape)} dtype={x.dtype} device={dev} contig={x.is_contiguous()}")
#     with torch.no_grad():
#         xf = x.detach()
#         if xf.numel() == 0:
#             print(f"[{name}] (empty)")
#             return
#         x32 = xf.float()
#         nan = int(torch.isnan(x32).sum().item())
#         inf = int(torch.isinf(x32).sum().item())
#         mx  = x32.abs().max().item()
#         mn  = x32.min().item()
#         av  = x32.mean().item()
#         print(f"[{name}] stats: mean={av:.6g} min={mn:.6g} max_abs={mx:.6g} nan={nan} inf={inf}")

# def _sync(tag=""):
#     if not FUSED_SYNC_AFTER_KERNEL:
#         return
#     torch.cuda.synchronize()
#     if tag:
#         print(f"[cuda.synchronize] {tag}")

# # ============================================================
# # Module-cycle detector (for RecursionError)
# # ============================================================
# def detect_module_cycles(root: torch.nn.Module):
#     seen = set()
#     stack = []
#     onstack = set()

#     def dfs(m):
#         mid = id(m)
#         if mid in onstack:
#             cyc = [type(x).__name__ for x in stack] + [type(m).__name__]
#             raise RuntimeError("Module cycle detected: " + " -> " + " -> ".join(cyc))
#         if mid in seen:
#             return
#         seen.add(mid)
#         onstack.add(mid)
#         stack.append(m)
#         for _, child in m.named_children():
#             dfs(child)
#         stack.pop()
#         onstack.remove(mid)

#     dfs(root)

# # ============================================================
# # Stage-E reference (PyTorch) — matches mh_forward convention
# #   Q,K,V: [T,H,D]  cand/mask: [T,C] or per-head [T*H,C]
# # ============================================================
# def _stable_softmax(scores_thc, mask_bool_thc):
#     # scores_thc: fp32 [T,H,C], mask_bool_thc: bool [T,H,C]
#     neg_inf = torch.finfo(scores_thc.dtype).min
#     s = scores_thc.masked_fill(~mask_bool_thc, neg_inf)
#     m = s.max(dim=-1, keepdim=True).values
#     all_masked = ~mask_bool_thc.any(dim=-1, keepdim=True)
#     m = torch.where(all_masked, torch.zeros_like(m), m)
#     ex = torch.exp(s - m) * mask_bool_thc.to(scores_thc.dtype)
#     z = ex.sum(dim=-1, keepdim=True)
#     z = torch.where(z == 0, torch.ones_like(z), z)
#     return ex / z

# def stagee_reference_thd(Q_thd, K_thd, V_thd, cand2d, mask2d, scale, cand_is_per_head, do_causal):
#     """
#     Returns out_ref_thd fp32: [T,H,D]
#     - do_causal drops candidates with idx > q_idx (per row t)
#     """
#     assert Q_thd.dim() == 3 and K_thd.dim() == 3 and V_thd.dim() == 3, "Q/K/V must be [T,H,D]"
#     T, H, D = Q_thd.shape
#     assert K_thd.shape[:2] == (T, H) and V_thd.shape[:2] == (T, H)
#     assert cand2d.dim() == 2 and mask2d.dim() == 2 and cand2d.shape == mask2d.shape

#     C = int(cand2d.shape[1])
#     dev = Q_thd.device

#     # Expand cand/mask to [T,H,C]
#     if bool(cand_is_per_head):
#         # [T*H, C] -> [T,H,C]
#         if int(cand2d.shape[0]) != T * H:
#             raise RuntimeError(f"cand_is_per_head=True but cand rows={int(cand2d.shape[0])} != T*H={T*H}")
#         cand_thc = cand2d.view(T, H, C).to(torch.int64)
#         mask_thc = mask2d.view(T, H, C).to(torch.uint8)
#     else:
#         # [T,C] -> [T,H,C]
#         if int(cand2d.shape[0]) != T:
#             raise RuntimeError(f"cand_is_per_head=False but cand rows={int(cand2d.shape[0])} != T={T}")
#         cand_thc = cand2d.unsqueeze(1).expand(T, H, C).to(torch.int64).contiguous()
#         mask_thc = mask2d.unsqueeze(1).expand(T, H, C).to(torch.uint8).contiguous()

#     mask_bool = (mask_thc != 0)

#     # Optional causal drop: idx <= t
#     if bool(do_causal):
#         q_idx = torch.arange(T, device=dev).view(T, 1, 1).expand(T, H, C)
#         mask_bool = mask_bool & (cand_thc <= q_idx)

#     # Clamp indices for gather (masked positions will be zeroed by mask anyway)
#     idx = cand_thc.clamp(min=0, max=T - 1)

#     # Gather K,V: want K[idx[t,h,c], h, :]
#     h_idx = torch.arange(H, device=dev).view(1, H, 1).expand(T, H, C)
#     K_g = K_thd[idx, h_idx].contiguous()  # [T,H,C,D]
#     V_g = V_thd[idx, h_idx].contiguous()  # [T,H,C,D]

#     # Scores + softmax
#     Q_fp32 = Q_thd.to(torch.float32)
#     K_fp32 = K_g.to(torch.float32)
#     scores = (Q_fp32.unsqueeze(2) * K_fp32).sum(dim=-1) * float(scale)  # [T,H,C]
#     p = _stable_softmax(scores, mask_bool)

#     out = (p.unsqueeze(-1) * V_g.to(torch.float32)).sum(dim=2)  # [T,H,D]
#     return out

# # ============================================================
# # NEW: Stage-E fwd<->bwd candidate/mask consistency + ref-gradcheck
# # ============================================================
# _STAGEE_FWD_CTX_BY_MID = {}           # keyed by id(m) from mh_forward output
# _STAGEE_BWD_GRADCHECK_DONE = False
# _STAGEE_BEST_DO_CAUSAL = None         # learned from truth test (if run)

# def _stagee_signature(Q, cand_i32, mask_u8, cand_is_per_head):
#     with torch.no_grad():
#         assert Q.dim() == 3 and cand_i32.dim() == 2 and mask_u8.dim() == 2
#         T, H, D = Q.shape
#         C = int(cand_i32.shape[1])

#         if bool(cand_is_per_head):
#             if int(cand_i32.shape[0]) != T * H:
#                 rows_ok = False
#             else:
#                 rows_ok = True
#             cand_thc = cand_i32.view(T, H, C).to(torch.int64) if rows_ok else None
#             mask_thc = mask_u8.view(T, H, C) if rows_ok else None
#             if rows_ok:
#                 maskb = (mask_thc != 0)
#                 q = torch.arange(int(T), device=Q.device, dtype=torch.int64).repeat_interleave(int(H)).view(T * H, 1) if False else None
#                 q = torch.arange(T, device=Q.device, dtype=torch.int64).view(T, 1, 1).expand(T, H, C)
#                 invalid = int((maskb & ((cand_thc < 0) | (cand_thc >= T))).sum().item())
#                 future  = int((maskb & (cand_thc > q)).sum().item())
#                 nnz     = int(maskb.sum().item())
#                 mask_sum = int(mask_thc.to(torch.int64).sum().item())
#                 cand_sum = int(cand_thc.sum().item())
#                 cand_abs_sum = int(cand_thc.abs().sum().item())
#                 cmin = int(cand_thc.min().item()) if cand_thc.numel() else 0
#                 cmax = int(cand_thc.max().item()) if cand_thc.numel() else 0
#                 mmin = int(mask_thc.min().item()) if mask_thc.numel() else 0
#                 mmax = int(mask_thc.max().item()) if mask_thc.numel() else 0
#             else:
#                 invalid = future = nnz = mask_sum = cand_sum = cand_abs_sum = 0
#                 cmin = cmax = mmin = mmax = 0
#         else:
#             rows_ok = (int(cand_i32.shape[0]) == T)
#             cand_tc = cand_i32.to(torch.int64)
#             mask_tc = mask_u8
#             maskb = (mask_tc != 0)
#             q = torch.arange(T, device=Q.device, dtype=torch.int64).view(T, 1).expand(T, C)
#             invalid = int((maskb & ((cand_tc < 0) | (cand_tc >= T))).sum().item())
#             future  = int((maskb & (cand_tc > q)).sum().item())
#             nnz     = int(maskb.sum().item())
#             mask_sum = int(mask_tc.to(torch.int64).sum().item())
#             cand_sum = int(cand_tc.sum().item())
#             cand_abs_sum = int(cand_tc.abs().sum().item())
#             cmin = int(cand_tc.min().item()) if cand_tc.numel() else 0
#             cmax = int(cand_tc.max().item()) if cand_tc.numel() else 0
#             mmin = int(mask_tc.min().item()) if mask_tc.numel() else 0
#             mmax = int(mask_tc.max().item()) if mask_tc.numel() else 0

#         sig = {
#             "T": int(T), "H": int(H), "D": int(D), "C": int(C),
#             "cand_is_per_head": bool(cand_is_per_head),
#             "cand_shape": tuple(cand_i32.shape),
#             "mask_shape": tuple(mask_u8.shape),
#             "cand_contig": bool(cand_i32.is_contiguous()),
#             "mask_contig": bool(mask_u8.is_contiguous()),
#             "cand_dtype": str(cand_i32.dtype),
#             "mask_dtype": str(mask_u8.dtype),
#             "rows_ok": bool(rows_ok),
#             "cand_sum": int(cand_sum),
#             "cand_abs_sum": int(cand_abs_sum),
#             "cand_min": int(cmin),
#             "cand_max": int(cmax),
#             "mask_sum": int(mask_sum),
#             "mask_nnz": int(nnz),
#             "mask_min": int(mmin),
#             "mask_max": int(mmax),
#             "invalid_masked": int(invalid),
#             "future_masked": int(future),
#         }
#         sig["future_frac"] = float(sig["future_masked"]) / float(max(1, sig["mask_nnz"]))
#         return sig

# def stagee_capture_fwd_ctx(m, Q, cand_i32, mask_u8, scale, cand_is_per_head):
#     try:
#         sig = _stagee_signature(Q, cand_i32, mask_u8, cand_is_per_head)
#         sig["scale"] = float(scale)
#         _STAGEE_FWD_CTX_BY_MID[id(m)] = sig
#     except Exception as e:
#         print(f"[Stage-E ctx] capture FAILED: {repr(e)}")
#         traceback.print_exc()

# def stagee_check_bwd_matches_fwd(m, Q, cand_i32, mask_u8, scale, cand_is_per_head):
#     ctx = _STAGEE_FWD_CTX_BY_MID.get(id(m), None)
#     if ctx is None:
#         print("[Stage-E ctx] WARNING: no forward ctx found for this backward (id(m) miss).")
#         return None

#     now = _stagee_signature(Q, cand_i32, mask_u8, cand_is_per_head)
#     now["scale"] = float(scale)

#     # Hard mismatches that scream “backward is using different structure than forward”
#     keys_hard = [
#         "T","H","D","C","cand_is_per_head","cand_shape","mask_shape","rows_ok",
#         "cand_dtype","mask_dtype","cand_contig","mask_contig",
#         "cand_sum","cand_abs_sum","cand_min","cand_max",
#         "mask_sum","mask_nnz","mask_min","mask_max",
#         "invalid_masked","future_masked",
#     ]
#     mism = []
#     for k in keys_hard:
#         if ctx.get(k) != now.get(k):
#             mism.append((k, ctx.get(k), now.get(k)))

#     print("\n=== [Stage-E bwd] fwd<->bwd cand/mask consistency ===")
#     print(f"[ctx] scale fwd={ctx.get('scale')} bwd={now.get('scale')}")
#     print(f"[ctx] future_frac fwd={ctx.get('future_frac'):.6f} bwd={now.get('future_frac'):.6f}")
#     print(f"[ctx] invalid_masked fwd={ctx.get('invalid_masked')} bwd={now.get('invalid_masked')}")
#     if mism:
#         print("🔥 [ctx] MISMATCH DETECTED between Stage-E forward and backward inputs:")
#         for k,a,b in mism:
#             print(f"  - {k}: fwd={a}  bwd={b}")
#     else:
#         print("[ctx] OK: backward received identical cand/mask structure & checksums as forward.")
#     return ctx

# def stagee_bwd_ref_gradcheck(
#     *,
#     dOut, Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head,
#     dQ_fused, dK_fused, dV_fused
# ):
#     global _STAGEE_BWD_GRADCHECK_DONE, _STAGEE_BEST_DO_CAUSAL
#     if not STAGEE_BWD_REF_GRADCHECK_ENABLE:
#         return
#     if STAGEE_BWD_REF_GRADCHECK_ONCE and _STAGEE_BWD_GRADCHECK_DONE:
#         return

#     try:
#         # small, deterministic slice
#         assert Q.dim() == 3
#         T, H, D = Q.shape
#         Hs = int(min(H, int(STAGEE_BWD_REF_GRADCHECK_HEADS)))
#         Ds = int(min(D, int(STAGEE_BWD_REF_GRADCHECK_D)))
#         if Hs <= 0 or Ds <= 0:
#             print("[Stage-E gradcheck] skipped (Hs or Ds <= 0)")
#             return

#         # Build sliced cand/mask consistent with head-slice
#         if bool(cand_is_per_head):
#             C = int(cand_i32.shape[1])
#             if int(cand_i32.shape[0]) != T * H:
#                 print("[Stage-E gradcheck] skipped (cand_is_per_head but cand rows != T*H)")
#                 return
#             cand_thc = cand_i32.view(T, H, C)[:, :Hs, :].contiguous().view(T * Hs, C)
#             mask_thc = mask_u8.view(T, H, C)[:, :Hs, :].contiguous().view(T * Hs, C)
#             cand_sub = cand_thc
#             mask_sub = mask_thc
#             cand_is_per_head_sub = True
#         else:
#             cand_sub = cand_i32
#             mask_sub = mask_u8
#             cand_is_per_head_sub = False

#         # Decide whether reference should enforce causal drop
#         do_causal = False
#         if _STAGEE_BEST_DO_CAUSAL is not None:
#             do_causal = bool(_STAGEE_BEST_DO_CAUSAL)
#         else:
#             # heuristic: if there are future candidates among masked, ref should drop them
#             sig = _stagee_signature(Q, cand_i32, mask_u8, cand_is_per_head)
#             do_causal = bool(sig["future_frac"] > 1e-6)

#         with torch.enable_grad():
#             Q0 = Q[:, :Hs, :Ds].detach().to(torch.float32).requires_grad_(True)
#             K0 = K[:, :Hs, :Ds].detach().to(torch.float32).requires_grad_(True)
#             V0 = V[:, :Hs, :Ds].detach().to(torch.float32).requires_grad_(True)
#             dO = dOut[:, :Hs, :Ds].detach().to(torch.float32)

#             out_ref = stagee_reference_thd(
#                 Q0, K0, V0,
#                 cand_sub, mask_sub,
#                 float(scale),
#                 bool(cand_is_per_head_sub),
#                 do_causal=bool(do_causal),
#             )
#             # vector-Jacobian product
#             loss_ref = (out_ref * dO).sum()
#             dQ_ref, dK_ref, dV_ref = torch.autograd.grad(loss_ref, (Q0, K0, V0), retain_graph=False, create_graph=False)

#         # compare against fused grads (slice)
#         dQf = dQ_fused[:, :Hs, :Ds].detach().to(torch.float32)
#         dKf = dK_fused[:, :Hs, :Ds].detach().to(torch.float32)
#         dVf = dV_fused[:, :Hs, :Ds].detach().to(torch.float32)

#         def _diff_stats(a, b):
#             diff = (a - b).abs()
#             mx = float(diff.max().item())
#             mn = float(diff.mean().item())
#             # relative: max(|a-b|)/max(1e-6, max(|ref|))
#             denom = float(a.abs().max().item())
#             rel = mx / max(1e-6, denom)
#             return mx, mn, rel, denom

#         mxQ, mnQ, relQ, denQ = _diff_stats(dQ_ref, dQf)
#         mxK, mnK, relK, denK = _diff_stats(dK_ref, dKf)
#         mxV, mnV, relV, denV = _diff_stats(dV_ref, dVf)

#         print("\n=== [Stage-E gradcheck] fused backward vs PyTorch ref backward (slice) ===")
#         print(f"[gradcheck] using do_causal={bool(do_causal)}  (learned={_STAGEE_BEST_DO_CAUSAL})  Hs={Hs} Ds={Ds}")
#         print(f"[dQ] max_abs={mxQ:.6e} mean_abs={mnQ:.6e} rel_max={relQ:.6e} ref_maxabs={denQ:.6e}")
#         print(f"[dK] max_abs={mxK:.6e} mean_abs={mnK:.6e} rel_max={relK:.6e} ref_maxabs={denK:.6e}")
#         print(f"[dV] max_abs={mxV:.6e} mean_abs={mnV:.6e} rel_max={relV:.6e} ref_maxabs={denV:.6e}")

#         _STAGEE_BWD_GRADCHECK_DONE = True

#     except Exception as e:
#         print(f"[Stage-E gradcheck] FAILED: {repr(e)}")
#         traceback.print_exc()

# # ============================================================
# # Debug wrappers for your CUDA extension call sites
# # ============================================================
# def wrap_build_candidates(build_candidates_cuda):
#     def _dbg(token_valid_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self, anchor_strategy, anchor_param0):
#         print("\n=== [Stage-A] build_candidates_cuda ===")
#         _tinfo(token_valid_bn, "token_valid_bn")
#         print(f"[Stage-A] params: w={int(w)} ring=[{int(ring_inner)},{int(ring_outer)}] mode={int(mode)} "
#               f"A={int(num_anchors)} include_self={bool(include_self)} "
#               f"anchor_strategy={int(anchor_strategy)} anchor_param0={int(anchor_param0)}")

#         out = build_candidates_cuda(
#             token_valid_bn, int(w), int(ring_inner), int(ring_outer), int(mode),
#             int(num_anchors), bool(include_self), int(anchor_strategy), int(anchor_param0)
#         )
#         _sync("after Stage-A")
#         cand_i32, mask_u8, valid_T = out
#         _tinfo(cand_i32, "cand_i32")
#         _tinfo(mask_u8, "mask_u8")
#         _tinfo(valid_T, "valid_T")
#         return out
#     return _dbg

# def wrap_proj_fwd(proj_forward):
#     def _dbg(H_flat, Wq, Wk, Wv, cos_t, sin_t):
#         print("\n=== [Stage-B] proj_forward (QKV + RoPE bridge) ===")
#         _tinfo(H_flat, "H_flat")
#         _tinfo(Wq, "Wq"); _tinfo(Wk, "Wk"); _tinfo(Wv, "Wv")
#         _tinfo(cos_t, "cos_t"); _tinfo(sin_t, "sin_t")

#         out = proj_forward(H_flat, Wq, Wk, Wv, cos_t, sin_t)
#         _sync("after Stage-B fwd")
#         Q, K, V = out
#         _tinfo(Q, "Q"); _tinfo(K, "K"); _tinfo(V, "V")
#         return out
#     return _dbg

# # global gate so we truth-test Stage-E only once
# _STAGEE_TRUTH_TEST_DONE = False

# def wrap_mh_fwd(mh_forward):
#     def _dbg(Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head):
#         global _STAGEE_TRUTH_TEST_DONE, _STAGEE_BEST_DO_CAUSAL

#         print("\n=== [Stage-E] mh_forward (fused attention) ===")
#         _tinfo(Q, "Q"); _tinfo(K, "K"); _tinfo(V, "V")
#         _tinfo(cand_i32, "cand_i32"); _tinfo(mask_u8, "mask_u8")
#         print(f"[Stage-E] scale={float(scale)} cand_is_per_head={bool(cand_is_per_head)}")

#         out = mh_forward(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head))
#         _sync("after Stage-E fwd")
#         out_thd, m, l = out
#         _tinfo(out_thd, "out_thd")
#         _tinfo(m, "m"); _tinfo(l, "l")

#         # capture forward signature keyed by id(m) so backward can verify exact same cand/mask
#         if STAGEE_BWD_CONSISTENCY_CHECK_ENABLE:
#             stagee_capture_fwd_ctx(m, Q, cand_i32, mask_u8, scale, cand_is_per_head)

#         # ------------------------------------------------------------
#         # Stage-E poison pill test (mask_u8 must fully gate invalid slots)
#         # ------------------------------------------------------------
#         STAGEE_POISON_PILL_ENABLE = bool(globals().get("STAGEE_POISON_PILL_ENABLE", True))
#         STAGEE_POISON_PILL_ONCE   = bool(globals().get("STAGEE_POISON_PILL_ONCE", True))
#         if STAGEE_POISON_PILL_ENABLE:
#             # run at most once by default (toggleable)
#             _done_flag = globals().get("_STAGEE_POISON_PILL_DONE", False)
#             do_poison = (not _done_flag) or (not STAGEE_POISON_PILL_ONCE)
#             if do_poison:
#                 try:
#                     with torch.no_grad():
#                         # cand_i32: (T,C) or (T*H,C); mask_u8 same shape
#                         bad = (mask_u8 == 0)
#                         nb = int(bad.sum().item())
#                         print(f"[Poison pill] bad(mask==0) slots = {nb}")

#                         if nb > 0:
#                             cand_poison = cand_i32.clone()

#                             # NOTE: In your pipeline Q/K/V are [T,H,D] (or equivalent),
#                             # and cand indices index into the key sequence length dimension.
#                             high = int(K.shape[0])

#                             # put random VALID indices into invalid slots; must not affect output if mask respected
#                             cand_poison[bad] = torch.randint(
#                                 low=0,
#                                 high=high,
#                                 size=(nb,),
#                                 device=cand_i32.device,
#                                 dtype=cand_i32.dtype
#                             )

#                             out2 = mh_forward(Q, K, V, cand_poison, mask_u8, float(scale), bool(cand_is_per_head))
#                             _sync("after Stage-E poison fwd")
#                             out2_thd = out2[0]

#                             poison_max_diff = (out_thd - out2_thd).abs().max().item()
#                             print("[Poison pill] poison max diff:", poison_max_diff)
#                         else:
#                             print("[Poison pill] skipped (no invalid slots where mask==0)")

#                     globals()["_STAGEE_POISON_PILL_DONE"] = True
#                 except Exception as e:
#                     print(f"[Poison pill] FAILED inside wrap_mh_fwd: {repr(e)}")
#                     traceback.print_exc()

#         # ------------------------------------------------------------
#         # Stage-E truth test INSIDE REAL PIPELINE (no fake params)
#         # ------------------------------------------------------------
#         do_truth = bool(STAGEE_TRUTH_TEST_ENABLE) and (not (_STAGEE_TRUTH_TEST_DONE and STAGEE_TRUTH_TEST_ONCE))
#         if do_truth:
#             try:
#                 # Expect Q/K/V are [T,H,D] in your pipeline
#                 if not (torch.is_tensor(Q) and Q.dim() == 3 and torch.is_tensor(cand_i32) and cand_i32.dim() == 2):
#                     print("[Truth test] skipped (unexpected shapes for Q or cand in this pipeline call)")
#                 else:
#                     T, H, D = Q.shape
#                     C = int(cand_i32.shape[1])

#                     # Probe A (within this flattened stream): future fraction among masked
#                     if bool(cand_is_per_head):
#                         if int(cand_i32.shape[0]) == T * H:
#                             cand_thc = cand_i32.view(T, H, C).to(torch.int64)
#                             mask_thc = (mask_u8.view(T, H, C) != 0)
#                             q_idx = torch.arange(T, device=Q.device, dtype=torch.int64).view(T, 1, 1).expand(T, H, C)
#                             num_masked = int(mask_thc.sum().item())
#                             num_future = int((mask_thc & (cand_thc > q_idx)).sum().item())
#                         else:
#                             num_masked = 0
#                             num_future = 0
#                     else:
#                         q_idx = torch.arange(T, device=Q.device).view(T, 1).expand(T, C)
#                         masked = (mask_u8 != 0)
#                         future = (cand_i32.to(torch.int64) > q_idx)
#                         num_masked = int(masked.sum().item())
#                         num_future = int((future & masked).sum().item())

#                     frac_future = num_future / max(1, num_masked)
#                     print(f"[Probe A] future-attend fraction among masked candidates = {frac_future:.6f}  (num_future={num_future}, num_masked={num_masked})")

#                     # Reference compares
#                     fused_fp32 = out_thd.to(torch.float32)

#                     ref_no = stagee_reference_thd(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head), do_causal=False)
#                     diff_no = (ref_no - fused_fp32).abs()
#                     print(f"[Truth test] call pattern: mh_forward(Q_thd,K_thd,V_thd,cand2d,mask2d,scale,cand_is_per_head)")
#                     mode = "per-head [T*H,C]" if bool(cand_is_per_head) else "shared [T,C]"
#                     print(f"[Truth test] cand mode used by kernel: mode={mode}  cand_is_per_head={bool(cand_is_per_head)}  cand_shape={tuple(cand_i32.shape)}  mask_shape={tuple(mask_u8.shape)}")
#                     print(f"[Truth test] max_abs(ref_no_causal - fused) = {diff_no.max().item():.6e}")
#                     print(f"[Truth test] mean_abs(ref_no_causal - fused)= {diff_no.mean().item():.6e}")

#                     best = False  # best_do_causal
#                     if bool(STAGEE_REF_DO_CAUSAL_COMPARE):
#                         ref_c = stagee_reference_thd(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head), do_causal=True)
#                         diff_c = (ref_c - fused_fp32).abs()
#                         print(f"[Truth test] max_abs(ref_causal - fused)    = {diff_c.max().item():.6e}")
#                         print(f"[Truth test] mean_abs(ref_causal - fused)   = {diff_c.mean().item():.6e}")
#                         best = bool(diff_c.mean().item() < diff_no.mean().item())
#                         _STAGEE_BEST_DO_CAUSAL = best
#                         print(f"[Truth test] learned best_do_causal={best} (based on mean_abs)")

#                     # Probe B: masked -1 usage
#                     masked_minus1 = (cand_i32.to(torch.int64) < 0) & (mask_u8 != 0)
#                     n_m1 = int(masked_minus1.sum().item())
#                     print(f"[Probe B] count(cand < 0 AND masked) = {n_m1}")
#                     if n_m1 == 0:
#                         print("[Probe B] no masked -1 candidates found; probe skipped.")

#                     print("=== DONE ===")
#                     _STAGEE_TRUTH_TEST_DONE = True
#             except Exception as e:
#                 print(f"[Truth test] FAILED inside wrap_mh_fwd: {repr(e)}")
#                 traceback.print_exc()

#         return out
#     return _dbg

# def wrap_mh_bwd(mh_backward):
#     def _dbg(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#         print("\n=== [Stage-E] mh_backward (fused attention backward) ===")
#         _tinfo(dOut, "dOut")

#         # 1) Verify backward sees EXACT same cand/mask structure as forward (keyed by id(m))
#         ctx = None
#         if STAGEE_BWD_CONSISTENCY_CHECK_ENABLE:
#             ctx = stagee_check_bwd_matches_fwd(m, Q, cand_i32, mask_u8, scale, cand_is_per_head)

#         out = mh_backward(dOut, Q, K, V, cand_i32, mask_u8, m, l, float(scale), bool(cand_is_per_head))
#         _sync("after Stage-E bwd")
#         dQ, dK, dV = out
#         _tinfo(dQ, "dQ"); _tinfo(dK, "dK"); _tinfo(dV, "dV")

#         # remove ctx entry now that backward happened (prevents unbounded growth)
#         _STAGEE_FWD_CTX_BY_MID.pop(id(m), None)

#         # 2) Direct ref-gradcheck on a small slice (this is the fastest “is backward wrong?” detector)
#         stagee_bwd_ref_gradcheck(
#             dOut=dOut, Q=Q, K=K, V=V,
#             cand_i32=cand_i32, mask_u8=mask_u8,
#             scale=scale, cand_is_per_head=cand_is_per_head,
#             dQ_fused=dQ, dK_fused=dK, dV_fused=dV
#         )

#         return out
#     return _dbg

# # ============================================================
# # Attach wrappers to every patched SASA attention layer
# # ============================================================
# def attach_sasa_debug_wrappers(model):
#     n = 0
#     for name, mod in model.named_modules():
#         if mod.__class__.__name__ == "MistralSASACGAttention_MH":
#             if getattr(mod, "_dbg_wrapped", False):
#                 continue
#             mod.build_candidates_cuda = wrap_build_candidates(mod.build_candidates_cuda)
#             mod.proj_fwd = wrap_proj_fwd(mod.proj_fwd)
#             mod.mh_fwd   = wrap_mh_fwd(mod.mh_fwd)
#             mod.mh_bwd   = wrap_mh_bwd(mod.mh_bwd)
#             mod._dbg_wrapped = True
#             n += 1
#     print(f"[attach_sasa_debug_wrappers] wrapped {n} MistralSASACGAttention_MH modules")

# # ============================================================
# # LoRA grad check (prove training is alive)
# # ============================================================
# def lora_grad_report(model, topk=12):
#     items = []
#     for n, p in model.named_parameters():
#         if ("lora_" in n.lower()) and (p.grad is not None):
#             g = p.grad.detach()
#             gn = float(g.float().norm().item())
#             gm = float(g.float().abs().max().item())
#             items.append((gn, gm, n))
#     items.sort(reverse=True, key=lambda x: x[0])
#     print("\n=== [LoRA grad report] ===")
#     if not items:
#         print("No LoRA grads found (either no LoRA params, or backward didn’t touch them).")
#         return
#     for gn, gm, n in items[:topk]:
#         print(f"{n}: grad_norm={gn:.6g} grad_maxabs={gm:.6g}")

# # ============================================================
# # NONFINITE grad probe (first layer that emits nonfinite grad_output)
# # ============================================================
# def attach_first_nonfinite_grad_probe(model):
#     layers = model.base_model.model.model.layers  # adjust if your nesting differs
#     handles = []

#     for li, layer in enumerate(layers):
#         def make_hook(li):
#             def hook(mod, grad_input, grad_output):
#                 g = grad_output[0] if isinstance(grad_output, (tuple, list)) and grad_output else grad_output
#                 if torch.is_tensor(g):
#                     bad = ~torch.isfinite(g)
#                     if bad.any():
#                         print(f"[NONFINITE] layer={li} grad_output nonfinite={int(bad.sum().item())}")
#                         raise RuntimeError(f"First nonfinite grad at layer {li}")
#             return hook
#         handles.append(layer.register_full_backward_hook(make_hook(li)))

#     return handles

# # ============================================================
# # The actual real-pipeline smoke test
# # ============================================================
# def real_pipeline_smoke_test(
#     model,
#     loader,
#     *,
#     trim_seq=256,
#     micro_bsz=1,
#     do_backward=True,
#     second_pass_cache_check=True,
# ):
#     print("\n==============================")
#     print(" REAL PIPELINE SMOKE TEST START")
#     print("==============================\n")

#     detect_module_cycles(model)
#     print("[cycle-check] OK (no module cycles)")

#     # Optional: if you have the dtype-alignment patch defined in your notebook, run it here.
#     if "patch_stageE_dtype_alignment" in globals():
#         try:
#             print("[patch] applying patch_stageE_dtype_alignment(model)")
#             globals()["patch_stageE_dtype_alignment"](model)
#         except Exception as e:
#             print(f"[patch] patch_stageE_dtype_alignment FAILED: {repr(e)}")
#             traceback.print_exc()

#     attach_sasa_debug_wrappers(model)

#     # ---- NEW: attach first-nonfinite-grad probe ----
#     _nonfinite_handles = []
#     try:
#         _nonfinite_handles = attach_first_nonfinite_grad_probe(model)
#         print(f"[nonfinite-probe] attached {len(_nonfinite_handles)} full_backward_hooks")
#     except Exception as e:
#         print(f"[nonfinite-probe] attach FAILED: {repr(e)}")
#         traceback.print_exc()

#     batch = next(iter(loader))

#     if micro_bsz is not None:
#         mb = int(micro_bsz)
#         batch = {
#             k: (v[:mb] if torch.is_tensor(v) and v.dim() > 0 and v.size(0) >= mb else v)
#             for k, v in batch.items()
#         }

#     input_ids = batch["input_ids"].cuda()
#     attention_mask = batch.get("attention_mask", None)
#     if attention_mask is not None:
#         attention_mask = attention_mask.cuda()

#     if trim_seq is not None and input_ids.shape[1] > int(trim_seq):
#         input_ids = input_ids[:, :int(trim_seq)].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :int(trim_seq)].contiguous()

#     labels = batch.get("labels", None)
#     if labels is None:
#         labels = input_ids.clone()
#     else:
#         labels = labels.cuda()
#         if trim_seq is not None and labels.shape[1] > int(trim_seq):
#             labels = labels[:, :int(trim_seq)].contiguous()

#     print("=== [Batch] ===")
#     _tinfo(input_ids, "input_ids")
#     _tinfo(attention_mask, "attention_mask")
#     _tinfo(labels, "labels")
#     _sync("after batch prep")

#     model.train()
#     opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16
#     use_scaler = False

#     scaler = torch.amp.GradScaler("cuda", enabled=False, init_scale=2**8, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000)

#     def _run_one(tag):
#         print(f"\n----- RUN: {tag} -----")

#         ctx_mgr = globals().get("SASACandCacheContext", None)
#         if ctx_mgr is None:
#             with torch.autocast(device_type=input_ids.device.type, dtype=amp_dtype, enabled=(input_ids.device.type == "cuda")):
#                 out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)
#         else:
#             with ctx_mgr():
#                 with torch.autocast(device_type=input_ids.device.type, dtype=amp_dtype, enabled=(input_ids.device.type == "cuda")):
#                     out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)

#         _sync("after model forward")

#         loss = out.loss if hasattr(out, "loss") and out.loss is not None else None
#         if loss is None:
#             raise RuntimeError("Model output has no .loss. Ensure labels are passed and model is CausalLM.")
#         print("[loss]", float(loss.item()))

#         if do_backward:
#             # ---- PATCH C (debug logs only): reset tag + counter right before backward ----
#             globals()["_STAGEE_RUN_TAG"] = str(tag)
#             globals()["_STAGEE_BWD_CALL_COUNTER"] = 0

#             opt.zero_grad(set_to_none=True)

#             if use_scaler:
#                 print("[AMP] scaler scale =", float(scaler.get_scale()))
#                 scaler.scale(loss).backward()
#             else:
#                 loss.backward()

#             _sync("after backward")
#             lora_grad_report(model)

#             if use_scaler:
#                 scaler.unscale_(opt)

#             if use_scaler:
#                 scaler.step(opt); scaler.update()
#             else:
#                 opt.step()

#             _sync("after opt.step")

#             opt.zero_grad(set_to_none=True)
#         return float(loss.item())

#     try:
#         loss1 = _run_one("pass-1")
#         if second_pass_cache_check:
#             loss2 = _run_one("pass-2")
#             print(f"\n[compare] loss1={loss1:.6g} loss2={loss2:.6g} (they can differ after opt.step; "
#                   f"disable do_backward to compare pure determinism)")
#     except Exception as e:
#         print("\n🔥 SMOKE TEST FAILED")
#         print("Exception:", repr(e))
#         traceback.print_exc()
#         raise
#     finally:
#         # ---- NEW: remove nonfinite hooks so they don't accumulate across reruns ----
#         try:
#             for h in _nonfinite_handles:
#                 h.remove()
#             if _nonfinite_handles:
#                 print(f"[nonfinite-probe] removed {len(_nonfinite_handles)} hooks")
#         except Exception as e:
#             print(f"[nonfinite-probe] remove FAILED: {repr(e)}")
#             traceback.print_exc()

#         print("\n============================")
#         print(" REAL PIPELINE SMOKE TEST END")
#         print("============================\n")

# # ---- run it on val_loader (or train_loader) ----
# real_pipeline_smoke_test(model, val_loader, trim_seq=256, micro_bsz=1, do_backward=True, second_pass_cache_check=False)


In [ ]:
# import torch, math, traceback
# from collections import defaultdict

# # -----------------------------
# # Small tensor summary helpers
# # -----------------------------
# def _tinfo(x: torch.Tensor, name: str, max_abs_items=0):
#     if x is None:
#         print(f"[{name}] = None")
#         return
#     dev = str(x.device)
#     print(f"[{name}] shape={tuple(x.shape)} dtype={x.dtype} device={dev} contig={x.is_contiguous()}")

#     # Stats (force sync-friendly, but keep it light)
#     with torch.no_grad():
#         xf = x.detach()
#         if xf.numel() == 0:
#             print(f"[{name}] (empty)")
#             return
#         # compute in fp32 to avoid fp16 overflow in stats
#         x32 = xf.float()
#         nan = int(torch.isnan(x32).sum().item())
#         inf = int(torch.isinf(x32).sum().item())
#         mx  = x32.abs().max().item()
#         mn  = x32.min().item()
#         av  = x32.mean().item()
#         print(f"[{name}] stats: mean={av:.6g} min={mn:.6g} max_abs={mx:.6g} nan={nan} inf={inf}")

# def _sync(tag=""):
#     if not FUSED_SYNC_AFTER_KERNEL:
#         return
#     torch.cuda.synchronize()
#     if tag:
#         print(f"[cuda.synchronize] {tag}")

# # -----------------------------------------
# # Module-cycle detector (for RecursionError)
# # -----------------------------------------
# def detect_module_cycles(root: torch.nn.Module):
#     seen = set()
#     stack = []
#     onstack = set()

#     def dfs(m):
#         mid = id(m)
#         if mid in onstack:
#             # cycle found: print path
#             cyc = [type(x).__name__ for x in stack] + [type(m).__name__]
#             raise RuntimeError("Module cycle detected: " + " -> ".join(cyc))
#         if mid in seen:
#             return
#         seen.add(mid)
#         onstack.add(mid)
#         stack.append(m)
#         for _, child in m.named_children():
#             dfs(child)
#         stack.pop()
#         onstack.remove(mid)

#     dfs(root)

# # ------------------------------------------------
# # Debug wrappers for your CUDA extension call sites
# # ------------------------------------------------
# def wrap_build_candidates(build_candidates_cuda):
#     def _dbg(token_valid_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self, anchor_strategy, anchor_param0):
#         print("\n=== [Stage-A] build_candidates_cuda ===")
#         _tinfo(token_valid_bn, "token_valid_bn")
#         print(f"[Stage-A] params: w={int(w)} ring=[{int(ring_inner)},{int(ring_outer)}] mode={int(mode)} "
#               f"A={int(num_anchors)} include_self={bool(include_self)} "
#               f"anchor_strategy={int(anchor_strategy)} anchor_param0={int(anchor_param0)}")

#         out = build_candidates_cuda(
#             token_valid_bn, int(w), int(ring_inner), int(ring_outer), int(mode),
#             int(num_anchors), bool(include_self), int(anchor_strategy), int(anchor_param0)
#         )
#         _sync("after Stage-A")
#         cand_i32, mask_u8, valid_T = out
#         _tinfo(cand_i32, "cand_i32")
#         _tinfo(mask_u8, "mask_u8")
#         _tinfo(valid_T, "valid_T")
#         return out
#     return _dbg

# def wrap_proj_fwd(proj_forward):
#     def _dbg(H_flat, Wq, Wk, Wv, cos_t, sin_t):
#         print("\n=== [Stage-B] proj_forward (QKV + RoPE bridge) ===")
#         _tinfo(H_flat, "H_flat")
#         _tinfo(Wq, "Wq"); _tinfo(Wk, "Wk"); _tinfo(Wv, "Wv")
#         _tinfo(cos_t, "cos_t"); _tinfo(sin_t, "sin_t")

#         out = proj_forward(H_flat, Wq, Wk, Wv, cos_t, sin_t)
#         _sync("after Stage-B fwd")
#         Q, K, V = out
#         _tinfo(Q, "Q"); _tinfo(K, "K"); _tinfo(V, "V")
#         return out
#     return _dbg

# def wrap_mh_fwd(mh_forward):
#     def _dbg(Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head):
#         print("\n=== [Stage-E] mh_forward (fused attention) ===")
#         _tinfo(Q, "Q"); _tinfo(K, "K"); _tinfo(V, "V")
#         _tinfo(cand_i32, "cand_i32"); _tinfo(mask_u8, "mask_u8")
#         print(f"[Stage-E] scale={float(scale)} cand_is_per_head={bool(cand_is_per_head)}")

#         out = mh_forward(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head))
#         _sync("after Stage-E fwd")
#         out_thd, m, l = out
#         _tinfo(out_thd, "out_thd")
#         _tinfo(m, "m"); _tinfo(l, "l")
#         return out
#     return _dbg

# def wrap_mh_bwd(mh_backward):
#     def _dbg(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#         print("\n=== [Stage-E] mh_backward (fused attention backward) ===")
#         _tinfo(dOut, "dOut")
#         out = mh_backward(dOut, Q, K, V, cand_i32, mask_u8, m, l, float(scale), bool(cand_is_per_head))
#         _sync("after Stage-E bwd")
#         dQ, dK, dV = out
#         _tinfo(dQ, "dQ"); _tinfo(dK, "dK"); _tinfo(dV, "dV")
#         return out
#     return _dbg

# # -------------------------------------------------------
# # Attach wrappers to every patched SASA attention layer
# # -------------------------------------------------------
# def attach_sasa_debug_wrappers(model):
#     n = 0
#     for name, mod in model.named_modules():
#         if mod.__class__.__name__ == "MistralSASACGAttention_MH":
#             # Wrap only once
#             if getattr(mod, "_dbg_wrapped", False):
#                 continue
#             mod.build_candidates_cuda = wrap_build_candidates(mod.build_candidates_cuda)
#             mod.proj_fwd = wrap_proj_fwd(mod.proj_fwd)
#             mod.mh_fwd   = wrap_mh_fwd(mod.mh_fwd)
#             mod.mh_bwd   = wrap_mh_bwd(mod.mh_bwd)
#             mod._dbg_wrapped = True
#             n += 1
#     print(f"[attach_sasa_debug_wrappers] wrapped {n} MistralSASACGAttention_MH modules")

# # -----------------------------------------
# # LoRA grad check (prove training is alive)
# # -----------------------------------------
# def lora_grad_report(model, topk=12):
#     items = []
#     for n, p in model.named_parameters():
#         if ("lora_" in n.lower()) and (p.grad is not None):
#             g = p.grad.detach()
#             gn = float(g.float().norm().item())
#             gm = float(g.float().abs().max().item())
#             items.append((gn, gm, n))
#     items.sort(reverse=True, key=lambda x: x[0])
#     print("\n=== [LoRA grad report] ===")
#     if not items:
#         print("No LoRA grads found (either no LoRA params, or backward didn’t touch them).")
#         return
#     for gn, gm, n in items[:topk]:
#         print(f"{n}: grad_norm={gn:.6g} grad_maxabs={gm:.6g}")

# # -----------------------------------------
# # The actual real-pipeline smoke test
# # -----------------------------------------
# def real_pipeline_smoke_test(
#     model,
#     loader,
#     *,
#     trim_seq=256,
#     micro_bsz=1,
#     do_backward=True,
#     second_pass_cache_check=True,
# ):
#     print("\n==============================")
#     print(" REAL PIPELINE SMOKE TEST START")
#     print("==============================\n")

#     # 0) catch module cycles early
#     detect_module_cycles(model)
#     print("[cycle-check] OK (no module cycles)")

#     # 1) attach wrappers (logs + synchronizes around kernels)
#     attach_sasa_debug_wrappers(model)

#     # 2) grab a real batch
#     batch = next(iter(loader))

#     if micro_bsz is not None:
#         mb = int(micro_bsz)
#         batch = {
#             k: (v[:mb] if torch.is_tensor(v) and v.dim() > 0 and v.size(0) >= mb else v)
#             for k, v in batch.items()
#         }

#     # common HF batch keys
#     input_ids = batch["input_ids"].cuda()
#     attention_mask = batch.get("attention_mask", None)
#     if attention_mask is not None:
#         attention_mask = attention_mask.cuda()

#     # trim (keep RoPE + masking active)
#     if trim_seq is not None and input_ids.shape[1] > int(trim_seq):
#         input_ids = input_ids[:, :int(trim_seq)].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :int(trim_seq)].contiguous()

#     labels = batch.get("labels", None)
#     if labels is None:
#         labels = input_ids.clone()
#     else:
#         labels = labels.cuda()
#         if trim_seq is not None and labels.shape[1] > int(trim_seq):
#             labels = labels[:, :int(trim_seq)].contiguous()

#     print("=== [Batch] ===")
#     _tinfo(input_ids, "input_ids")
#     _tinfo(attention_mask, "attention_mask")
#     _tinfo(labels, "labels")
#     _sync("after batch prep")

#     # 3) forward/backward in a per-forward cache context (your notebook has this)
#     model.train()  # if this RecursionErrors, cycle-check should have caught why
#     opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

#     def _run_one(tag):
#         print(f"\n----- RUN: {tag} -----")
#         opt.zero_grad(set_to_none=True)

#         # Your notebook defines SASACandCacheContext; use it if available
#         ctx_mgr = globals().get("SASACandCacheContext", None)
#         if ctx_mgr is None:
#             # fall back: no context manager
#             out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)
#         else:
#             with ctx_mgr():
#                 out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)

#         _sync("after model forward")

#         loss = out.loss if hasattr(out, "loss") and out.loss is not None else None
#         if loss is None:
#             raise RuntimeError("Model output has no .loss. Ensure labels are passed and model is CausalLM.")
#         print("[loss]", float(loss.item()))

#         if do_backward:
#             loss.backward()
#             _sync("after backward")
#             lora_grad_report(model)
#             opt.step()
#             _sync("after opt.step")
#         return float(loss.item())

#     try:
#         loss1 = _run_one("pass-1")
#         if second_pass_cache_check:
#             # pass-2 should re-use candidates (if reuse is enabled) and behave deterministically
#             loss2 = _run_one("pass-2")
#             print(f"\n[compare] loss1={loss1:.6g} loss2={loss2:.6g} (they can differ after opt.step; "
#                   f"disable do_backward to compare pure determinism)")
#     except Exception as e:
#         print("\n🔥 SMOKE TEST FAILED")
#         print("Exception:", repr(e))
#         traceback.print_exc()
#         raise
#     finally:
#         print("\n============================")
#         print(" REAL PIPELINE SMOKE TEST END")
#         print("============================\n")

# # ---- run it on val_loader (or train_loader) ----
# real_pipeline_smoke_test(model, val_loader, trim_seq=256, do_backward=True, second_pass_cache_check=False)


OLD SMOKE TEST FOR SASA-CG.... ORIGINAL

In [ ]:
# # ============================================================
# # Budget sweep (Ablation 1): quality–cost frontier at matched measured budget
# #   Sweeps w (window) and logs:
# #     - measured keys/query + density (Stage-A probe)
# #     - eval ppl + throughput
# #     - eval peak memory
# # Produces: df_budget
# # ============================================================
# import pandas as pd
# import gc, torch

# BUDGET_TRAIN_STEPS = 100
# BUDGET_EVAL_STEPS  = 50
# BUDGET_LR          = 1e-4
# BUDGET_GRAD_ACCUM  = 1

# # Fixed scaffold for budget sweep (window-only frontier)
# BUDGET_SCAFFOLD_MODE = 0
# BUDGET_RING_INNER = 0
# BUDGET_RING_OUTER = 0

# # Anchor knobs (keep fixed for a "pure" budget sweep)
# BUDGET_ANCHOR_STRATEGY = 0
# BUDGET_ANCHOR_PARAM0   = 0

# # Probe batch for measuring effective budget (keys/query)
# _budget_probe_batch = next(iter(val_loader))
# _probe_attn = _budget_probe_batch.get("attention_mask", None)
# if _probe_attn is None:
#     _probe_attn = torch.ones_like(_budget_probe_batch["input_ids"])
# _probe_N = int(_probe_attn.shape[1])

# RECS_BUDGET = []

# for ww in BUDGET_W_LIST:
#     ww = int(ww)
#     restore_trainable_params_(model, TRAINABLE_SNAPSHOT)

#     # print("\n" + "="*80)
#     # print(
#     #     f"BUDGET_SWEEP w={ww} | mode={BUDGET_SCAFFOLD_MODE} ring=[{BUDGET_RING_INNER}..{BUDGET_RING_OUTER}] "
#     #     f"(anchors={int(SWEEP_NUM_ANCHORS)}, include_self={bool(SWEEP_INCLUDE_SELF)}) "
#     #     f"anchor_strategy={int(BUDGET_ANCHOR_STRATEGY)} anchor_param0={int(BUDGET_ANCHOR_PARAM0)}"
#     # )

#     # update all SASA layers
#     set_sasacg_params(
#         model,
#         w=ww,
#         scaffold_mode=int(BUDGET_SCAFFOLD_MODE),
#         ring_inner=int(BUDGET_RING_INNER),
#         ring_outer=int(BUDGET_RING_OUTER),
#         num_anchors=int(SWEEP_NUM_ANCHORS),
#         include_self=bool(SWEEP_INCLUDE_SELF),
#         anchor_strategy=int(BUDGET_ANCHOR_STRATEGY),
#         anchor_param0=int(BUDGET_ANCHOR_PARAM0),
#         verbose=True,
#     )

#     # cleanup between runs (keeps Colab stable)
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()

#     # measure effective keys/query under the real attention mask
#     budget_stats = measure_effective_budget_from_batch(
#         _budget_probe_batch,
#         w=ww,
#         ring_inner=int(BUDGET_RING_INNER),
#         ring_outer=int(BUDGET_RING_OUTER),
#         scaffold_mode=int(BUDGET_SCAFFOLD_MODE),
#         num_anchors=int(SWEEP_NUM_ANCHORS),
#         include_self=bool(SWEEP_INCLUDE_SELF),
#         anchor_strategy=int(BUDGET_ANCHOR_STRATEGY),
#         anchor_param0=int(BUDGET_ANCHOR_PARAM0),
#     )
#     print("budget:", budget_stats)

#     # optional short train (usually keep 0 for frontier-on-inference)
#     if BUDGET_TRAIN_STEPS and BUDGET_TRAIN_STEPS > 0:
#         train_stats = train_n_steps(
#             model,
#             train_loader,
#             steps=int(BUDGET_TRAIN_STEPS),
#             lr=float(BUDGET_LR),
#             grad_accum=int(BUDGET_GRAD_ACCUM),
#             max_grad_norm=0.5,   # or 1.0 while debugging
#           )
#     else:
#         train_stats = {
#             "train_tokens": 0,
#             "train_runtime_sec": 0.0,
#             "train_tokens_per_sec": None,
#             "peak_alloc_gb": None,
#             "peak_reserved_gb": None,
#         }

#     # eval (quality point)
#     _reset_cuda_peak()
#     eval_stats = eval_loss_only(model, val_loader, max_steps=int(BUDGET_EVAL_STEPS))

#     # record
#     rec = {}
#     rec.update({
#         "run_name": f"budget_w{ww}__A{int(SWEEP_NUM_ANCHORS)}__m{int(BUDGET_SCAFFOLD_MODE)}",
#         "scaffold_mode": int(BUDGET_SCAFFOLD_MODE),
#         "w": ww,
#         "ring_inner": int(BUDGET_RING_INNER),
#         "ring_outer": int(BUDGET_RING_OUTER),
#         "num_anchors": int(SWEEP_NUM_ANCHORS),
#         "include_self": bool(SWEEP_INCLUDE_SELF),
#         "anchor_strategy": int(BUDGET_ANCHOR_STRATEGY),
#         "anchor_param0": int(BUDGET_ANCHOR_PARAM0),
#         "N_probe": int(_probe_N),
#     })
#     rec.update(budget_stats)
#     rec.update({f"train_{k}": v for k, v in train_stats.items() if k not in ("peak_alloc_gb", "peak_reserved_gb")})
#     rec.update({f"eval_{k}": v for k, v in eval_stats.items()})

#     # convenience fields for plotting (matches scaffold sweep schema)
#     rec["tokens_per_sec"] = rec["eval_eval_tokens_per_sec"]
#     rec["eval_ppl"] = rec["eval_eval_ppl"]
#     rec["eval_loss"] = rec["eval_eval_loss"]
#     rec["peak_alloc_gb_eval"] = rec["eval_peak_alloc_gb"]
#     rec["peak_reserved_gb_eval"] = rec["eval_peak_reserved_gb"]

#     RECS_BUDGET.append(rec)

#     print("REC:", {
#         "run": rec["run_name"],
#         "keys_mean": rec.get("keys_per_query_mean", None),
#         "ppl": rec["eval_ppl"],
#         "tps": rec["tokens_per_sec"],
#         "peak_gb": rec["peak_alloc_gb_eval"],
#     })

# df_budget = pd.DataFrame(RECS_BUDGET).sort_values("keys_per_query_mean").reset_index(drop=True)
# df_budget


Streaming output truncated to the last 5000 lines.
[Stage-E] scale=0.08838834764831843 cand_is_per_head=False
[cuda.synchronize] after Stage-E fwd
[out_thd] shape=(1024, 32, 128) dtype=torch.float32 device=cuda:0 contig=True
[out_thd] stats: mean=-0.00100184 min=-2.05016 max_abs=2.05016 nan=0 inf=0
[m] shape=(32768,) dtype=torch.float32 device=cuda:0 contig=True
[m] stats: mean=0.5365 min=-16.0359 max_abs=16.0359 nan=0 inf=0
[l] shape=(32768,) dtype=torch.float32 device=cuda:0 contig=True
[l] stats: mean=2.38131 min=1 max_abs=7.7502 nan=0 inf=0

=== [Stage-A] build_candidates_cuda ===
[token_valid_bn] shape=(4, 256) dtype=torch.int64 device=cuda:0 contig=True
[token_valid_bn] stats: mean=1 min=1 max_abs=1 nan=0 inf=0
[Stage-A] params: w=7 ring=[0,0] mode=0 A=0 include_self=True anchor_strategy=0 anchor_param0=0
[cuda.synchronize] after Stage-A
[cand_i32] shape=(1024, 15) dtype=torch.int32 device=cuda:0 contig=True
[cand_i32] stats: mean=267.681 min=-1 max_abs=1023 nan=0 inf=0
[mask_u8]

KeyboardInterrupt: 

In [ ]:
# # ============================================================
# # REAL BACKWARD NaN TRACER (find first bad SASA CUDA backward)
# #   - wraps Stage-E mh_bwd + Stage-B proj_bwd across ALL SASA layers
# #   - runs ONE real CE backward micro-step
# #   - prints FIRST layer/stage where non-finites appear (inputs or outputs)
# #   - PATCHES ADDED:
# #       (1) Stage-B output shapes + output maxabs (esp to verify dWq finite under FP32)
# #       (2) layer-31 Stage-B input dtype+maxabs (existing)
# #       (3) layer-29 Stage-E input dtype+maxabs (new)
# #       (4) TRUE FP32 discriminator for Stage-B: cast proj_bwd inputs to float32
# #       (5) Optional TRUE FP32 discriminator for Stage-E: cast mh_bwd inputs to float32 (may fail if kernel fp16-only)
# #       (6) LOSS SCALING A/B: backward on (loss * loss_scale) to test overflow vs kernel bug
# #       (7) STOMP CHECK: snapshot selected weights before/after forward and backward
# # ============================================================
# import torch, gc, math
# from collections import defaultdict, OrderedDict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext", "_get_decoder_layers_any_wrapper"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_trace] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _isfinite_all(x):
#     return bool(torch.isfinite(x).all().item())

# def _nonfinite_count(x):
#     return int((~torch.isfinite(x)).sum().item())

# def _maxabs(x):
#     return float(x.detach().float().abs().max().item()) if (torch.is_tensor(x) and x.numel()) else 0.0

# def _check_trainables_finite(m, name_contains=None, max_show=30):
#     bad = []
#     total = 0
#     for n,p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         total += 1
#         if not _isfinite_all(p.detach()):
#             bad.append((n, tuple(p.shape), str(p.dtype), _nonfinite_count(p.detach()), _maxabs(p.detach())))
#     if bad:
#         print(f"[nan_trace] ❌ NON-FINITE TRAINABLE PARAMS already: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ trainable params finite: {total} tensors" + (f" (filtered by '{name_contains}')" if name_contains else ""))
#     return True

# def _get_sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for li, l in enumerate(layers):
#         sa = getattr(l, "self_attn", None)
#         if sa is not None and sa.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((li, l, sa))
#     return sasa

# # ----------------------------
# # STOMP helpers
# # ----------------------------
# def _safe_getattr_chain(obj, chain, default=None):
#     cur = obj
#     for k in chain:
#         if not hasattr(cur, k):
#             return default
#         cur = getattr(cur, k)
#     return cur

# def _select_stomp_tensors(model, layer_ids=(31, 30, 0)):
#     """
#     Pick a few weights that must NEVER change during forward/backward (before opt.step()).
#     Returns OrderedDict[name -> tensor_ref].
#     """
#     out = OrderedDict()
#     layers = _get_decoder_layers_any_wrapper(model)

#     for li in layer_ids:
#         if li < 0 or li >= len(layers):
#             continue
#         l = layers[li]
#         sa = getattr(l, "self_attn", None)
#         if sa is None:
#             continue
#         old = getattr(sa, "old", None)
#         if old is None:
#             continue

#         q_proj = getattr(old, "q_proj", None)
#         if q_proj is not None and hasattr(q_proj, "weight") and torch.is_tensor(q_proj.weight):
#             out[f"layers.{li}.self_attn.old.q_proj.weight"] = q_proj.weight

#         k_proj = getattr(old, "k_proj", None)
#         if k_proj is not None and hasattr(k_proj, "weight") and torch.is_tensor(k_proj.weight):
#             out[f"layers.{li}.self_attn.old.k_proj.weight"] = k_proj.weight

#     emb = None
#     if hasattr(model, "get_input_embeddings"):
#         emb = model.get_input_embeddings()
#     if emb is not None and hasattr(emb, "weight") and torch.is_tensor(emb.weight):
#         out["input_embeddings.weight"] = emb.weight

#     lm_head = _safe_getattr_chain(model, ["lm_head"], default=None)
#     if lm_head is not None and hasattr(lm_head, "weight") and torch.is_tensor(lm_head.weight):
#         out["lm_head.weight"] = lm_head.weight

#     return out

# def _snapshot_selected(tensor_map):
#     snap = {}
#     for name, t in tensor_map.items():
#         try:
#             snap[name] = t.detach().clone()
#         except Exception:
#             pass
#     return snap

# def _report_stomp(before_snap, tensor_map, tag, max_show=20):
#     changed = []
#     for name, t in tensor_map.items():
#         if name not in before_snap:
#             continue
#         try:
#             cur = t.detach()
#             ref = before_snap[name]
#             if cur.shape != ref.shape:
#                 changed.append((name, "shape_changed", tuple(ref.shape), tuple(cur.shape)))
#                 continue
#             if not torch.equal(cur, ref):
#                 d = (cur.float() - ref.float())
#                 changed.append((name, float(d.abs().max().item()), float(d.abs().mean().item()), str(cur.dtype)))
#         except Exception:
#             changed.append((name, "compare_failed"))
#     if changed:
#         print(f"[nan_trace] ❌ STOMP DETECTED ({tag}): {len(changed)} tensors changed (showing up to {max_show})")
#         for row in changed[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ stomp check OK ({tag}): no tracked weights changed")
#     return True

# def install_sasa_backward_sentinels(
#     model,
#     stop_on_first=True,
#     verbose=True,
#     force_fp32_proj_bwd=False,   # Stage-B FP32 discriminator
#     force_fp32_mh_bwd=False,     # Stage-E FP32 discriminator (may fail if kernel fp16-only)
#     print_layer31_input_stats=True,
#     print_layer29_input_stats=True,
# ):
#     """
#     Wrap mh_bwd and proj_bwd for each SASA layer to detect non-finites in:
#       - Stage-E backward outputs: dQ,dK,dV
#       - Stage-B backward outputs: dH, dWq, dWk, dWv
#     """
#     sasa = _get_sasa_layers(model)
#     if not sasa:
#         raise RuntimeError("[nan_trace] No SASA layers found.")

#     events = []
#     orig = {"mh_bwd": {}, "proj_bwd": {}}
#     call_counts = defaultdict(int)

#     def _emit(tag, li, msg):
#         s = f"[nan_trace] ❌ {tag} @ layer {li}: {msg}"
#         events.append(s)
#         print(s)
#         if stop_on_first:
#             raise RuntimeError(s)

#     for li, layer, attn in sasa:
#         # ---- wrap Stage-E mh_bwd ----
#         if hasattr(attn, "mh_bwd") and callable(attn.mh_bwd):
#             orig_mh = attn.mh_bwd
#             orig["mh_bwd"][li] = orig_mh

#             def _make_mh_wrap(li, fn):
#                 def mh_bwd_wrap(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#                     call_counts[("mh_bwd", li)] += 1

#                     # PATCH: layer-29 input magnitude stats (to distinguish overflow vs bug)
#                     if print_layer29_input_stats and int(li) == 29:
#                         try:
#                             print(
#                                 f"[nan_trace] Stage-E INPUT STATS @ layer {li}: "
#                                 f"dOut {getattr(dOut,'dtype',None)} maxabs={_maxabs(dOut):.3e} | "
#                                 f"Q {getattr(Q,'dtype',None)} maxabs={_maxabs(Q):.3e} | "
#                                 f"K {getattr(K,'dtype',None)} maxabs={_maxabs(K):.3e} | "
#                                 f"V {getattr(V,'dtype',None)} maxabs={_maxabs(V):.3e} | "
#                                 f"m {getattr(m,'dtype',None)} maxabs={_maxabs(m):.3e} | "
#                                 f"l {getattr(l,'dtype',None)} maxabs={_maxabs(l):.3e} | "
#                                 f"scale={float(scale) if not torch.is_tensor(scale) else float(scale.detach().float().cpu().item())}"
#                             )
#                         except Exception:
#                             pass

#                     # finiteness checks on inputs
#                     if torch.is_tensor(dOut) and (not _isfinite_all(dOut)):
#                         _emit("Stage-E INPUT", li, f"dOut nonfinite (bad={_nonfinite_count(dOut)}) maxabs={_maxabs(dOut):.3e}")
#                     for nm, t in [("Q",Q),("K",K),("V",V),("m",m),("l",l)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     # PATCH: optional FP32 mh_bwd attempt (may fail if extension doesn't support fp32)
#                     if bool(force_fp32_mh_bwd):
#                         try:
#                             dOut_ = dOut.contiguous().float()
#                             Q_    = Q.contiguous().float()
#                             K_    = K.contiguous().float()
#                             V_    = V.contiguous().float()
#                             m_    = m.contiguous().float()
#                             l_    = l.contiguous().float()
#                             out = fn(dOut_, Q_, K_, V_, cand_i32, mask_u8, m_, l_, scale, cand_is_per_head)
#                         except Exception as e:
#                             print(f"[nan_trace] Stage-E FP32 call failed @ layer {li}: {type(e).__name__}: {e}")
#                             raise
#                     else:
#                         out = fn(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head)

#                     outs = out if isinstance(out, (tuple, list)) else (out,)

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return mh_bwd_wrap
#             attn.mh_bwd = _make_mh_wrap(li, orig_mh)

#         # ---- wrap Stage-B proj_bwd ----
#         if hasattr(attn, "proj_bwd") and callable(attn.proj_bwd):
#             orig_pb = attn.proj_bwd
#             orig["proj_bwd"][li] = orig_pb

#             def _make_proj_wrap(li, fn):
#                 def proj_bwd_wrap(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t):
#                     call_counts[("proj_bwd", li)] += 1

#                     # layer-31 input stats (existing)
#                     if print_layer31_input_stats and int(li) == 31:
#                         try:
#                             dq_dtype = str(getattr(dQ, "dtype", None))
#                             h_dtype  = str(getattr(H_flat, "dtype", None))
#                             print(f"[nan_trace] Stage-B INPUT STATS @ layer {li}: dQ dtype={dq_dtype} maxabs={_maxabs(dQ):.3e} | H dtype={h_dtype} maxabs={_maxabs(H_flat):.3e}")
#                         except Exception:
#                             pass

#                     # input finiteness checks
#                     for nm, t in [("dQ",dQ),("dK",dK),("dV",dV),("H",H_flat),("cos",cos_t),("sin",sin_t)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     # Stage-B FP32 discriminator
#                     if bool(force_fp32_proj_bwd):
#                         try:
#                             dQ_     = dQ.contiguous().float()
#                             dK_     = dK.contiguous().float()
#                             dV_     = dV.contiguous().float()
#                             H_flat_ = H_flat.contiguous().float()
#                             Wq_     = Wq.contiguous().float()
#                             Wk_     = Wk_eff.contiguous().float()
#                             Wv_     = Wv_eff.contiguous().float()
#                             cos_    = cos_t.contiguous().float()
#                             sin_    = sin_t.contiguous().float()
#                             out = fn(dQ_, dK_, dV_, H_flat_, Wq_, Wk_, Wv_, cos_, sin_)
#                         except Exception as e:
#                             print(f"[nan_trace] Stage-B FP32 call failed @ layer {li}: {type(e).__name__}: {e}")
#                             raise
#                     else:
#                         out = fn(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t)

#                     outs = out if isinstance(out, (tuple, list)) else (out,)

#                     # output shapes + maxabs (so you can verify dWq finite under FP32 mode)
#                     try:
#                         shapes = []
#                         maxabs = []
#                         for t in outs:
#                             if torch.is_tensor(t):
#                                 shapes.append(tuple(t.shape))
#                                 maxabs.append(_maxabs(t))
#                             else:
#                                 shapes.append(type(t).__name__)
#                                 maxabs.append(float("nan"))
#                         print(f"[nan_trace] Stage-B OUTPUT SHAPES @ layer {li}: {shapes}")
#                         print(f"[nan_trace] Stage-B OUTPUT MAXABS  @ layer {li}: {[f'{x:.3e}' for x in maxabs]}")
#                     except Exception:
#                         pass

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return proj_bwd_wrap
#             attn.proj_bwd = _make_proj_wrap(li, orig_pb)

#     if verbose:
#         print(
#             f"[nan_trace] installed sentinels on {len(sasa)} SASA layers "
#             f"({'stop_on_first' if stop_on_first else 'collect_all'}). "
#             f"force_fp32_proj_bwd={bool(force_fp32_proj_bwd)} "
#             f"force_fp32_mh_bwd={bool(force_fp32_mh_bwd)}"
#         )
#     return orig, events, call_counts

# def uninstall_sasa_backward_sentinels(model, orig):
#     sasa = _get_sasa_layers(model)
#     for li, layer, attn in sasa:
#         if li in orig.get("mh_bwd", {}):
#             attn.mh_bwd = orig["mh_bwd"][li]
#         if li in orig.get("proj_bwd", {}):
#             attn.proj_bwd = orig["proj_bwd"][li]
#     print("[nan_trace] restored original mh_bwd/proj_bwd")

# def run_one_real_backward_trace(
#     w=128,
#     max_seq=256,
#     anomaly=False,
#     only_lora=True,
#     autocast_on=True,
#     force_fp32_proj_bwd=False,
#     force_fp32_mh_bwd=False,
#     loss_scale=1.0,     # PATCH: loss scaling A/B
#     stomp_check=True,
# ):
#     device = next(model.parameters()).device
#     model.train()

#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     ok_params = _check_trainables_finite(model, name_contains=("lora_" if only_lora else None))
#     if not ok_params:
#         print("[nan_trace] >>> Your trainables are already non-finite. Reload model / re-init LoRA before debugging backward.")
#         return

#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels    = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     set_sasacg_params(model, w=int(w), verbose=True)
#     model.zero_grad(set_to_none=True)

#     tracked = _select_stomp_tensors(model, layer_ids=(31, 30, 0))
#     snap0 = _snapshot_selected(tracked) if stomp_check else None

#     orig, events, call_counts = install_sasa_backward_sentinels(
#         model,
#         stop_on_first=True,
#         verbose=True,
#         force_fp32_proj_bwd=bool(force_fp32_proj_bwd),
#         force_fp32_mh_bwd=bool(force_fp32_mh_bwd),
#         print_layer31_input_stats=True,
#         print_layer29_input_stats=True,
#     )

#     torch.autograd.set_detect_anomaly(bool(anomaly))

#     try:
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         with SASACandCacheContext():
#             with torch.autocast(
#                 device_type="cuda",
#                 dtype=amp_dtype,
#                 enabled=(torch.cuda.is_available() and bool(autocast_on)),
#             ):
#                 out = model(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     labels=labels,
#                     use_cache=False,
#                     past_key_values=None,
#                 )
#                 loss = out.loss

#         if stomp_check:
#             _report_stomp(snap0, tracked, tag="after_forward")

#         loss_ok = _isfinite_all(loss.detach())
#         logits_ok = _isfinite_all(out.logits.detach())
#         print(
#             f"[nan_trace] forward: loss={float(loss.detach().float().cpu().item()):.6f} "
#             f"loss_ok={loss_ok} | logits_ok={logits_ok} logits_maxabs={_maxabs(out.logits.detach()):.3e} "
#             f"| loss_scale={float(loss_scale):.3e}"
#         )
#         if not (loss_ok and logits_ok):
#             print("[nan_trace] >>> Forward already non-finite. Backward trace is meaningless until forward is fixed.")
#             return

#         # PATCH: loss scaling A/B
#         (loss * float(loss_scale)).backward()

#         print("[nan_trace] ✅ backward completed without any sentinel detecting non-finites.")
#         bad = 0
#         total = 0
#         for n,p in _iter_trainables_named(model):
#             if only_lora and ("lora_" not in n):
#                 continue
#             if p.grad is None:
#                 continue
#             total += 1
#             if not _isfinite_all(p.grad):
#                 bad += 1
#         print(f"[nan_trace] grad check (filtered={'lora_' if only_lora else 'all'}): nonfinite={bad}/{total}")

#     finally:
#         if stomp_check and snap0 is not None:
#             _report_stomp(snap0, tracked, tag="after_backward_or_crash")
#         uninstall_sasa_backward_sentinels(model, orig)
#         torch.autograd.set_detect_anomaly(False)

# # ============================================================
# # RUN MATRIX (add ONLY what matters next)
# #   1) Baseline autocast OFF (repro)
# #   2) Stage-B FP32 discriminator (you ran; keep)
# #   3) Loss-scale A/B under Stage-B FP32 (overflow test for Stage-E)
# #   4) Optional: attempt Stage-E FP32 call (may error if kernel fp16-only)
# # ============================================================

# def _run_case(name, **kwargs):
#     print("\n" + "="*90)
#     print(f"[nan_trace] RUN CASE: {name} | {kwargs}")
#     print("="*90)
#     try:
#         run_one_real_backward_trace(**kwargs)
#     except Exception as e:
#         print(f"[nan_trace] ({name}) stopped with exception: {type(e).__name__}: {e}")

# # 1) Baseline autocast OFF (repro)
# _run_case(
#     "baseline_autocast_OFF",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=False,
#     force_fp32_mh_bwd=False,
#     loss_scale=1.0,
#     stomp_check=True,
# )

# # 2) Stage-B FP32 discriminator (your key case)
# _run_case(
#     "stageB_force_FP32_inputs",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=True,
#     force_fp32_mh_bwd=False,
#     loss_scale=1.0,
#     stomp_check=True,
# )

# # 3) Loss-scale A/B under Stage-B FP32 (if this fixes Stage-E INF, it's overflow)
# _run_case(
#     "stageB_FP32 + loss_scale_1e-2",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=True,
#     force_fp32_mh_bwd=False,
#     loss_scale=1e-2,
#     stomp_check=True,
# )

# # 4) Optional: try forcing Stage-E FP32 too (may fail if kernel doesn't support fp32)
# _run_case(
#     "stageB_FP32 + stageE_try_FP32",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=True,
#     force_fp32_mh_bwd=True,
#     loss_scale=1.0,
#     stomp_check=True,
# )


In [ ]:
# # ============================================================
# # FULL TRAIN-STEP NaN AUDIT (real train loop path)
# #   - runs ONE micro-step with your real forward (SASA + LoRA + CE loss)
# #   - checks: forward logits/loss finiteness
# #   - checks: backward grad finiteness (lists first offenders)
# #   - checks: optimizer step effect on params (detects NaN injection)
# #   - optional: anomaly detection to pinpoint the first bad backward op
# # ============================================================
# import torch, math, gc
# from collections import OrderedDict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_audit] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _tensor_nonfinite_stats(x: torch.Tensor):
#     # returns (is_all_finite, nan_count, inf_count, max_abs)
#     if x is None:
#         return (True, 0, 0, 0.0)
#     finite = torch.isfinite(x)
#     all_ok = bool(finite.all().item())
#     nan_ct = int(torch.isnan(x).sum().item())
#     inf_ct = int(torch.isinf(x).sum().item())
#     max_abs = float(x.detach().float().abs().max().item()) if x.numel() else 0.0
#     return all_ok, nan_ct, inf_ct, max_abs

# def _snapshot_params(m, name_contains=None, max_items=999999):
#     snap = OrderedDict()
#     for n, p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         snap[n] = p.detach().clone()
#         if len(snap) >= max_items:
#             break
#     return snap

# def _count_nonfinite_trainables(m, only_grad=False, max_show=20):
#     bad = []
#     total = 0
#     for n, p in _iter_trainables_named(m):
#         total += 1
#         x = p.grad if only_grad else p.detach()
#         ok, nan_ct, inf_ct, max_abs = _tensor_nonfinite_stats(x)
#         if not ok:
#             bad.append((n, tuple(p.shape), str(p.dtype), nan_ct, inf_ct, max_abs))
#     if bad:
#         print(f"[nan_audit] ❌ non-finite {'grads' if only_grad else 'params'}: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#     else:
#         print(f"[nan_audit] ✅ all trainable {'grads' if only_grad else 'params'} finite: {total} tensors")
#     return bad

# @torch.no_grad()
# def _check_forward_finite(out):
#     # out: HF CausalLMOutputWithCrossAttentions-like
#     loss = out.loss
#     logits = out.logits
#     loss_ok, loss_nan, loss_inf, loss_maxabs = _tensor_nonfinite_stats(loss.detach())
#     log_ok, log_nan, log_inf, log_maxabs = _tensor_nonfinite_stats(logits.detach())
#     return {
#         "loss_ok": loss_ok, "loss_nan": loss_nan, "loss_inf": loss_inf, "loss": float(loss.detach().float().cpu().item()),
#         "logits_ok": log_ok, "logits_nan": log_nan, "logits_inf": log_inf, "logits_maxabs": log_maxabs,
#     }

# def one_step_nan_audit(
#     w=128,
#     max_seq=256,
#     lr=3e-4,
#     grad_accum=1,
#     max_grad_norm=1.0,
#     anomaly=False,
#     print_top_grad_norms=8,
#     only_lora=True,
# ):
#     """
#     If only_lora=True, audit focuses on LoRA trainables (faster).
#     It still checks global forward finiteness.
#     """

#     device = next(model.parameters()).device
#     model.train()

#     # choose autocast dtype like your train loop
#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     # grab one batch
#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)

#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     # truncate for speed/determinism
#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     # apply w
#     set_sasacg_params(model, w=int(w), verbose=True)

#     # make optimizer over trainables
#     params = [(n,p) for (n,p) in _iter_trainables_named(model)]
#     if only_lora:
#         params = [(n,p) for (n,p) in params if "lora_" in n]
#     if not params:
#         raise RuntimeError("[nan_audit] No trainable params matched (only_lora=True may be filtering everything).")

#     opt = torch.optim.AdamW([p for _,p in params], lr=float(lr), betas=(0.9, 0.95), weight_decay=0.0)

#     # clear grads
#     opt.zero_grad(set_to_none=True)

#     # snapshot params before
#     snap_before = _snapshot_params(model, name_contains="lora_" if only_lora else None)

#     # optional anomaly
#     if anomaly:
#         torch.autograd.set_detect_anomaly(True)

#     # ----- forward -----
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()

#     with SASACandCacheContext():
#         with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
#             out = model(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 labels=labels,
#                 use_cache=False,
#                 past_key_values=None,
#             )
#             loss = out.loss
#             loss = loss / float(grad_accum)

#     fchk = _check_forward_finite(out)
#     print(f"[nan_audit] forward: loss={fchk['loss']:.6f} loss_ok={fchk['loss_ok']} | logits_ok={fchk['logits_ok']} logits_maxabs={fchk['logits_maxabs']:.3e}")
#     if not fchk["loss_ok"] or not fchk["logits_ok"]:
#         print("[nan_audit] >>> NaNs/INFs already in FORWARD. Stop here; backward will poison everything.")
#         _count_nonfinite_trainables(model, only_grad=False, max_show=20)
#         return {"where": "forward_nonfinite", "forward": fchk}

#     # ----- backward -----
#     loss.backward()

#     # grad checks
#     bad_grads = _count_nonfinite_trainables(model, only_grad=True, max_show=30)

#     # print some grad norms (helps detect explosion even if finite)
#     # (only for the filtered params list used in optimizer)
#     gnorms = []
#     for n,p in params:
#         if p.grad is None:
#             continue
#         g = p.grad.detach()
#         gnorms.append((float(g.float().norm().item()), n))
#     gnorms.sort(reverse=True)
#     if gnorms:
#         print("[nan_audit] top grad norms:")
#         for v,n in gnorms[:print_top_grad_norms]:
#             print(f"   {v:.3e}  {n}")

#     if bad_grads:
#         print("[nan_audit] >>> NaNs/INFs introduced in BACKWARD. Do NOT step optimizer.")
#         return {"where": "backward_nonfinite", "forward": fchk, "bad_grads": bad_grads[:30]}

#     # ----- grad clip (like your loop) -----
#     if max_grad_norm is not None:
#         torch.nn.utils.clip_grad_norm_([p for _,p in params], float(max_grad_norm))

#     # ----- optimizer step -----
#     opt.step()
#     opt.zero_grad(set_to_none=True)

#     # param checks after
#     bad_params_after = _count_nonfinite_trainables(model, only_grad=False, max_show=30)

#     # compare deltas (sanity)
#     deltas = []
#     for n,p in params:
#         if n in snap_before:
#             d = (p.detach() - snap_before[n]).float()
#             deltas.append((float(d.abs().mean().item()), float(d.abs().max().item()), n))
#     deltas.sort(reverse=True)
#     print("[nan_audit] top param mean|Δ| after step:")
#     for mean_d, max_d, n in deltas[:8]:
#         print(f"   mean|Δ|={mean_d:.3e} max|Δ|={max_d:.3e}  {n}")

#     if bad_params_after:
#         print("[nan_audit] >>> Params became non-finite ONLY AFTER opt.step(). That means grads were finite BUT optimizer math/state went bad (rare) or some grads were silently non-finite.")
#         return {"where": "optimizer_step_nonfinite", "forward": fchk, "bad_params_after": bad_params_after[:30]}

#     print("[nan_audit] ✅ one-step looks numerically healthy.")
#     return {"where": "ok", "forward": fchk}

# # --- run it ---
# # If your weights already got poisoned earlier, reload model first before trusting this.
# res = one_step_nan_audit(w=128, max_seq=256, lr=3e-4, anomaly=False, only_lora=True)
# res


In [ ]:
# # ============================================================
# # NA N / INF SCAN: which trainable params are already broken?
# # ============================================================
# import torch

# def scan_trainables_for_nonfinite(model, max_show=30):
#     bad = []
#     total = 0
#     for name, p in model.named_parameters():
#         if not p.requires_grad:
#             continue
#         total += 1
#         x = p.detach()
#         finite = torch.isfinite(x)
#         if not bool(finite.all().item()):
#             nan_ct = int(torch.isnan(x).sum().item())
#             inf_ct = int(torch.isinf(x).sum().item())
#             bad.append((name, tuple(x.shape), str(x.dtype), nan_ct, inf_ct))

#     print(f"[nan_scan] trainable tensors: {total}")
#     if not bad:
#         print("[nan_scan] ✅ no NaN/Inf in trainables")
#         return

#     print(f"[nan_scan] ❌ non-finite trainables: {len(bad)} (showing up to {max_show})")
#     for row in bad[:max_show]:
#         print("  ", row)

# scan_trainables_for_nonfinite(model)


[nan_scan] trainable tensors: 128
[nan_scan] ✅ no NaN/Inf in trainables


In [ ]:
# # ============================================================
# # TRACE SMOKE: where does invariance enter?
# #   - logs Stage-A: keys_mean/keys_p95/mask_sum/valid_sum (inside forward)
# #   - logs Stage-E: out sample digest (inside mh_fwd)
# #   - logs o_proj: input/output digest
# #   - logs self_attn output digest
# #   - logs logits slice max-diff across w
# # ============================================================
# import torch, gc

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _find_first_sasa_layer(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     for li, l in enumerate(layers):
#         if getattr(l, "self_attn", None) is not None and l.self_attn.__class__.__name__ == "MistralSASACGAttention_MH":
#             return li, l
#     return None, None

# @torch.no_grad()
# def trace_w_effect(w1=16, w2=128):
#     li, layer0 = _find_first_sasa_layer(model)
#     if layer0 is None:
#         raise RuntimeError("[trace] No SASA layers found.")
#     attn0 = layer0.self_attn

#     model.eval()
#     device = next(model.parameters()).device

#     # fixed batch
#     batch = next(iter(val_loader))
#     batch = _to_device(batch, device)

#     # recorders
#     rec = {"run1": {}, "run2": {}}

#     # --------- wrap Stage-A (first layer only) ---------
#     _orig_stageA = attn0.build_candidates_cuda
#     def _stageA_dbg(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self, anchor_strategy=0, anchor_param0=0):
#         cand_i32, mask_u8, valid_T = _orig_stageA(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self, anchor_strategy, anchor_param0)

#         # cheap-but-informative stats
#         # mask_u8: [Ttot, C]  (or [Ttot*H,C] if per-head, but yours is token-level here)
#         Ttot = int(mask_u8.size(0))
#         C = int(mask_u8.size(1))
#         keys = mask_u8.to(torch.int32).sum(dim=1)  # [Ttot]
#         keys_mean = float(keys.float().mean().item())
#         # quantile is fine here (Ttot ~ seq_len*bsz)
#         keys_p95 = float(torch.quantile(keys.float(), 0.95).item()) if keys.numel() > 0 else 0.0

#         valid_sum = None
#         if valid_T is not None:
#             valid_sum = int(valid_T.to(torch.int32).sum().item())

#         # stash into a temp slot chosen by current w (we fill it outside)
#         _stageA_dbg._last = {
#             "w_arg": int(w),
#             "C": C,
#             "mask_sum": int(mask_u8.to(torch.int32).sum().item()),
#             "keys_mean": keys_mean,
#             "keys_p95": keys_p95,
#             "valid_sum": valid_sum,
#             "cand_row0_head": cand_i32[0, :min(16, C)].detach().cpu().tolist() if cand_i32.numel() > 0 else [],
#         }
#         return cand_i32, mask_u8, valid_T
#     _stageA_dbg._last = None

#     # --------- wrap Stage-E (mh_fwd) ---------
#     _orig_mh_fwd = attn0.mh_fwd
#     def _mh_fwd_dbg(Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head):
#         out, m, l = _orig_mh_fwd(Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head)

#         # tiny digest
#         osamp = out[0, 0, :min(8, out.size(-1))].detach().float().cpu().tolist()
#         _mh_fwd_dbg._last = {
#             "out_shape": tuple(out.shape),
#             "out_samp": osamp,
#             "out_absmean_samp": float(torch.tensor(osamp).abs().mean().item()) if len(osamp) else 0.0,
#         }
#         return out, m, l
#     _mh_fwd_dbg._last = None

#     # --------- hooks: o_proj + self_attn output ---------
#     oproj_last = {}
#     attn_out_last = {}

#     def _hook_oproj(mod, inp, out):
#         x = inp[0]
#         xs = x[0, :min(2, x.size(0)), :min(8, x.size(-1))].detach().float().cpu()
#         ys = out[0, :min(2, out.size(0)), :min(8, out.size(-1))].detach().float().cpu()
#         oproj_last.clear()
#         oproj_last.update({
#             "in_absmean": float(xs.abs().mean().item()),
#             "out_absmean": float(ys.abs().mean().item()),
#             "out_samp_row0": ys[0].tolist() if ys.numel() else [],
#         })

#     def _hook_attn(mod, inp, out):
#         # out is (attn_output, None)
#         ao = out[0] if isinstance(out, (tuple, list)) else out
#         s = ao[0, :min(2, ao.size(0)), :min(8, ao.size(-1))].detach().float().cpu()
#         attn_out_last.clear()
#         attn_out_last.update({
#             "attn_absmean": float(s.abs().mean().item()),
#             "attn_samp_row0": s[0].tolist() if s.numel() else [],
#         })

#     h1 = attn0.old.o_proj.register_forward_hook(_hook_oproj)
#     h2 = attn0.register_forward_hook(_hook_attn)

#     # --------- apply wrappers ---------
#     attn0.build_candidates_cuda = _stageA_dbg
#     attn0.mh_fwd = _mh_fwd_dbg

#     def _run(tag, ww):
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         set_sasacg_params(model, w=int(ww), verbose=True)

#         # forward
#         out = model(
#             input_ids=batch["input_ids"],
#             attention_mask=batch.get("attention_mask", None),
#             labels=batch.get("labels", batch["input_ids"]),
#             use_cache=False,
#             past_key_values=None,
#         )

#         loss = float(out.loss.detach().float().cpu().item())
#         logits = out.logits.detach()

#         # logits slice fingerprint (stronger than mean/std)
#         sl = logits[:, :min(2, logits.size(1)), :min(128, logits.size(2))].float().cpu()
#         fp = {
#             "loss": loss,
#             "logits_absmean": float(sl.abs().mean().item()),
#             "logits_samp0": sl[0, 0, :8].tolist(),
#         }

#         # attach the stage stats captured during forward
#         fp["stageA"] = dict(_stageA_dbg._last) if _stageA_dbg._last is not None else None
#         fp["stageE"] = dict(_mh_fwd_dbg._last) if _mh_fwd_dbg._last is not None else None
#         fp["oproj"]  = dict(oproj_last) if oproj_last else None
#         fp["attn0"]  = dict(attn_out_last) if attn_out_last else None
#         fp["logits_slice"] = sl  # keep for diff

#         rec[tag] = fp

#     try:
#         _run("run1", w1)
#         _run("run2", w2)

#         # compute logits slice max diff
#         d = (rec["run2"]["logits_slice"] - rec["run1"]["logits_slice"]).abs().max().item()

#         print("\n" + "="*90)
#         print("[trace] SUMMARY")
#         for tag in ["run1", "run2"]:
#             r = rec[tag]
#             print(f"\n[{tag}] loss={r['loss']:.9f} | logits_absmean={r['logits_absmean']:.6f} | logits_samp0={r['logits_samp0']}")
#             print(f"  stageA={r['stageA']}")
#             print(f"  stageE={r['stageE']}")
#             print(f"  oproj={r['oproj']}")
#             print(f"  attn0={r['attn0']}")

#         print(f"\n[trace] logits_slice max|diff| between w={w1} and w={w2}: {d:.9e}")

#         print("\n" + "="*90)
#         print("[trace] HOW TO READ THIS")
#         print("A) If stageA.keys_mean changes but stageE.out_samp doesn't -> Stage-E ignoring cand/mask (or reading wrong).")
#         print("B) If stageE.out_samp changes but oproj/out or attn0 doesn't -> o_proj (or later) is flattening differences.")
#         print("C) If attn0 changes but logits max-diff ~0 -> downstream layers are overwriting it (unlikely if all layers SASA).")
#         print("D) If stageA.keys_mean does NOT change -> Stage-A is effectively clamped/masked in real forward.")
#     finally:
#         # restore
#         attn0.build_candidates_cuda = _orig_stageA
#         attn0.mh_fwd = _orig_mh_fwd
#         h1.remove(); h2.remove()
#         print("\n[trace] restored hooks/wrappers.")

# # Run it
# trace_w_effect(w1=16, w2=128)


In [ ]:
# # ============================================================
# # Frontier plots (KDD-friendly) — Scaffold structure sweep
# #   - Labels include: mode, w, ring_inner, ring_outer
# #   - Budget–quality plot uses x = keys_per_query_mean
# #   - Keeps your "no explicit color" style (marker encodes type)
# # ============================================================
# import matplotlib.pyplot as plt

# def _mode_name(m: int) -> str:
#     m = int(m)
#     if m == 0: return "win"
#     if m == 1: return "ring"
#     if m == 2: return "win+ring"
#     return f"mode{m}"

# def _scaffold_label(r) -> str:
#     return f"{_mode_name(r['scaffold_mode'])} (m={int(r['scaffold_mode'])}, w={int(r['w'])}, r=[{int(r['ring_inner'])},{int(r['ring_outer'])}])"

# def _scaffold_marker(m: int) -> str:
#     m = int(m)
#     if m == 0: return "o"  # window
#     if m == 1: return "s"  # ring
#     if m == 2: return "^"  # win+ring
#     return "x"


ORIGIINAL ELEMENT

In [ ]:
# # Colab-friendly single Python cell that compiles your C++/CUDA as a PyTorch extension.
# # ✅ Keeps your CUDA/C++ code + comments VERBATIM (it is embedded exactly as-is below).
# # ✅ Only adds the Python wrapper + build glue so it runs in Colab.

# import os, torch
# import hashlib
# import os, torch
# from torch.utils.cpp_extension import load_inline
# import hashlib

# os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
# # 2) Better C++ stack traces when extensions throw
# os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "0"

# # # Toggle these as needed:
# FUSED_ENABLE_HOST_LOGS   = False    # C++ host-side logs via std::cout
# FUSED_ENABLE_DEVICE_LOGS = False   # CUDA printf inside kernel
# FUSED_SYNC_AFTER_KERNEL  = False    # device sync after kernel for accurate failure location
# FUSED_VERBOSE_BUILD      = False    # verbose compile logs

# # V-tiling toggle
# FUSED_USE_SMEM_V         = True    # introduce V-tiling

# # from torch.utils.cpp_extension import load_inline

# # ----------------------------
# # Your code (VERBATIM) as a CUDA source string
# # ----------------------------
# CUDA_SRC = r"""
# // ============================================================
# // Candidate Builder (CUDA) for SASA-CG / GISO-style neighbor attention
# //
# // Matches your notebook semantics (build_candidate_tensor_flat_batched_fast):
# //  - Window candidates: j = i+off for off in [-w..w]
# //      * INVALID if out-of-range (NO clamping -> avoids edge duplicates)
# //      * INVALID if not causal (j > i)
# //      * INVALID if candidate is padding (attn_mask[b,j]==0)
# //      * INVALID if include_self==false and off==0
# //  - Anchor candidates: A = min(num_anchors, N) if num_anchors>0 else 0
# //      * anchors_local[a] = floor(a * N / A)  (equiv to floor(linspace(0,N,A+1)[:-1]))
# //      * keep only if outside window [max(i-w,0), min(i+w,N-1)]
# //      * and causal (anchor <= i)
# //      * and candidate valid (attn_mask[b,anchor]!=0)
# //  - If query is padding (attn_mask[b,i]==0): entire row invalid
# //
# // Inputs:
# //   attn_mask_bn: [B,N] on CUDA. Any dtype ok; nonzero means valid.
# // Args:
# //   w, num_anchors, include_self
# //
# // Outputs:
# //   cand_i32: [T, Cmax] int32, -1 invalid
# //   mask_u8 : [T, Cmax] uint8, 0/1 validity
# //   valid_T : [T] uint8, 0/1 for query validity
# //
# // Compile as a PyTorch extension (C++ + CUDA).
# // ============================================================

# #include <torch/extension.h>
# #include <ATen/cuda/CUDAContext.h>
# #include <cuda.h>
# #include <cuda_runtime.h>
# #include <vector>
# #include <cstdint>
# // ----------------------------
# // SASA Smoke-Test Debug Macros
# //   - Enable device-side printf with -DSASA_DEVICE_DEBUG
# //   - Keep kernels semantically identical; logs only.
# // ----------------------------
# #if defined(SASA_DEVICE_DEBUG)
#   #include <cstdio>
#   #define SASA_DPRINTF(...) printf(__VA_ARGS__)
# #else
#   #define SASA_DPRINTF(...) ((void)0)
# #endif

# #define CHECK_CUDA(x) TORCH_CHECK((x).is_cuda(), #x " must be a CUDA tensor")
# #define CHECK_CONTIGUOUS(x) TORCH_CHECK((x).is_contiguous(), #x " must be contiguous")
# #define CHECK_2D(x) TORCH_CHECK((x).dim() == 2, #x " must be 2D [B,N]")

# // ------------------------------------------------------------
# // Kernel: assumes mask_bn_u8 is uint8 [B,N] with 0/1
# // ------------------------------------------------------------
# __global__ void build_candidates_kernel_u8(
#     const uint8_t* __restrict__ mask_bn_u8, // [B,N]
#     int B,
#     int N,
#     int w,
#     int ring_inner, // ring distance lower bound (>=1 when enabled)
#     int ring_outer, // ring distance upper bound (>= ring_inner when enabled)
#     int mode,       // 0=window-only, 1=ring-only, 2=window+ring
#     int A,          // already min(num_anchors, N), could be 0
#     bool include_self,
#     int anchor_strategy, // NEW: 0=uniform,1=periodic,2=boundary,3=global_tokens
#     int anchor_param0,   // NEW: strategy-specific param (period / leftK / start)
#     int Cw,         // window slots (2*w+1) or 0 if disabled
#     int Cr,         // ring slots (ring_outer-ring_inner+1) or 0 if disabled
#     int Cmax,       // Cw + Cr + A
#     int32_t* __restrict__ cand_out, // [T,Cmax]
#     uint8_t* __restrict__ ok_out,   // [T,Cmax]
#     uint8_t* __restrict__ valid_T   // [T]
# ) {
#     int t = (int)blockIdx.x; // one row per block
#     int T = B * N;
#     if (t >= T) return;

#     int b = t / N;
#     int i = t - b * N;

#     // Query validity (single-writer to avoid benign race / sanitizer warnings)
#     __shared__ uint8_t qv;
#     if (threadIdx.x == 0) {
#         qv = mask_bn_u8[b * N + i] ? 1 : 0;
#         valid_T[t] = qv;
#     }
#     // Broadcast q_valid to all threads in block
#     //q_valid = __syncthreads_or((int)q_valid) ? 1 : 0;
#     __syncthreads();
#     uint8_t q_valid = qv;

#     // If query is padding -> entire row invalid
#     if (!q_valid) {
#         const int64_t base = (int64_t)t * (int64_t)Cmax; // 64-bit safe
#         for (int c = threadIdx.x; c < Cmax; c += blockDim.x) {
#             cand_out[base + (int64_t)c] = -1;
#             ok_out[base + (int64_t)c] = 0;
#         }
#         return;
#     }

#     const int64_t base = (int64_t)t * (int64_t)Cmax; // 64-bit safe

#     // =========================================================
#     // PATCHED: Window + Ring + Anchors scaffold builder
#     //   - Window candidates [0..Cw)
#     //   - Ring candidates   [Cw..Cw+Cr)
#     //   - Anchors           [Cw+Cr..Cmax)
#     // =========================================================

#     // Track whether ANY valid key exists in this row (stability hardening)
#     int any_ok = 0;

#     // Window candidates [0..Cw)
#     if (Cw > 0) {
#       for (int c = threadIdx.x; c < Cw; c += blockDim.x) {
#           int off = c - w;
#           int j = i + off;

#           bool in_range = (j >= 0) && (j < N);
#           bool causal   = (j <= i);
#           bool self_ok  = include_self || (off != 0);

#           if (!in_range || !causal || !self_ok) {
#               cand_out[base + (int64_t)c] = -1;
#               ok_out[base + (int64_t)c] = 0;
#           } else {
#               uint8_t cand_valid = mask_bn_u8[b * N + j] ? 1 : 0;
#               if (!cand_valid) {
#                   cand_out[base + (int64_t)c] = -1;
#                   ok_out[base + (int64_t)c] = 0;
#               } else {
#                   cand_out[base + (int64_t)c] = (int32_t)(b * N + j);
#                   ok_out[base + (int64_t)c] = 1;
#                   any_ok = 1;
#               }
#           }
#       }
#     }

#     // Ring candidates [Cw..Cw+Cr)
#     if (Cr > 0) {
#       for (int c = threadIdx.x; c < Cr; c += blockDim.x) {
#           int d = ring_inner + c;   // distance in [ring_inner..ring_outer]
#           int j = i - d;

#           int out_c = Cw + c;

#           // FIX #1: prevent overlap duplicates with window when both are enabled.
#           // Overlap happens for backward distances d <= w (because window includes j=i-d).
#           bool overlaps_window = (Cw > 0) && (d <= w);

#           bool in_range = (j >= 0) && (j < N);
#           bool causal   = true;     // j = i - d always <= i
#           bool self_ok  = include_self || (d != 0);

#           if (overlaps_window || !in_range || !causal || !self_ok) {
#               cand_out[base + (int64_t)out_c] = -1;
#               ok_out[base + (int64_t)out_c] = 0;
#           } else {
#               uint8_t cand_valid = mask_bn_u8[b * N + j] ? 1 : 0;
#               if (!cand_valid) {
#                   cand_out[base + (int64_t)out_c] = -1;
#                   ok_out[base + (int64_t)out_c] = 0;
#               } else {
#                   cand_out[base + (int64_t)out_c] = (int32_t)(b * N + j);
#                   ok_out[base + (int64_t)out_c] = 1;
#                   any_ok = 1;
#               }
#           }
#       }
#     }

#     // Anchor candidates [Cscaf..Cmax)
#     int Cscaf = Cw + Cr;

#     if (A == 0) {
#       for (int c = Cscaf + threadIdx.x; c < Cmax; c += blockDim.x) {
#           cand_out[base + (int64_t)c] = -1;
#           ok_out[base + (int64_t)c] = 0;
#       }
#       // (no return) — allow stability fallback below
#     } else {

#       // window forbidden interval
#       int lo_win = i - w; if (lo_win < 0) lo_win = 0;
#       int hi_win = i; if (hi_win > (N - 1)) hi_win = N - 1;

#       // ring forbidden interval
#       int lo_ring = i - ring_outer; if (lo_ring < 0) lo_ring = 0;
#       int hi_ring = i - ring_inner; if (hi_ring < 0) hi_ring = -1;

#       for (int c = Cscaf + threadIdx.x; c < Cmax; c += blockDim.x) {
#           int a_idx = c - Cscaf; // 0..A-1

#           // ------------------------------------------------------------
#           // ANCHOR ABLATION: placement strategy
#           //   0: uniform (original)
#           //   1: periodic (param0=period; if <=0 uses N/A)
#           //   2: boundary-biased (param0=leftK; if <=0 uses A/2)
#           //   3: global tokens style (param0=start; if <=0 uses 0)
#           // ------------------------------------------------------------
#           int a_local = 0;
#           if (anchor_strategy == 1) {
#               int p = (anchor_param0 > 0) ? anchor_param0 : (N / (A > 0 ? A : 1));
#               if (p < 1) p = 1;
#               a_local = a_idx * p;
#               if (a_local > (N - 1)) a_local = (N - 1);
#           } else if (anchor_strategy == 2) {
#               int k = (anchor_param0 > 0) ? anchor_param0 : (A / 2);
#               if (k < 0) k = 0;
#               if (k > A) k = A;
#               if (a_idx < k) a_local = a_idx;
#               else a_local = N - (A - a_idx);
#               if (a_local < 0) a_local = 0;
#               if (a_local > (N - 1)) a_local = (N - 1);
#           } else if (anchor_strategy == 3) {
#               int start = (anchor_param0 > 0) ? anchor_param0 : 0;
#               if (start < 0) start = 0;
#               a_local = start + a_idx;
#               if (a_local > (N - 1)) a_local = (N - 1);
#           } else {
#               a_local = (int)(((long long)a_idx * (long long)N) / (long long)A);
#           }

#           bool causal = (a_local <= i);
#           bool self_ok = include_self || (a_local != i);

#           bool in_window = (Cw > 0) && (a_local >= lo_win && a_local <= hi_win);
#           bool in_ring   = (Cr > 0) && (a_local >= lo_ring && a_local <= hi_ring);

#           if (!causal || !self_ok || in_window || in_ring) {
#               cand_out[base + (int64_t)c] = -1;
#               ok_out[base + (int64_t)c] = 0;
#           } else {
#               uint8_t cand_valid = mask_bn_u8[b * N + a_local] ? 1 : 0;
#               if (!cand_valid) {
#                   cand_out[base + (int64_t)c] = -1;
#                   ok_out[base + (int64_t)c] = 0;
#               } else {
#                   cand_out[base + (int64_t)c] = (int32_t)(b * N + a_local);
#                   ok_out[base + (int64_t)c] = 1;
#                   any_ok = 1;
#               }
#           }
#       }
#     }

#     // =========================================================
#     // Stability hardening:
#     //   Guarantee at least one valid key per non-padding query.
#     //   If row is empty, fallback to self in slot 0.
#     // =========================================================
#     int any_ok_all = __syncthreads_or(any_ok);
#     if (any_ok_all == 0) {
#         if (threadIdx.x == 0) {
#             cand_out[base + 0] = (int32_t)(b * N + i);
#             ok_out[base + 0]   = 1;
#         }
#     }
# }

# // ------------------------------------------------------------
# // C++ launcher / binding
# // ------------------------------------------------------------
# std::vector<torch::Tensor> build_candidates_cuda(
#     torch::Tensor attn_mask_bn, // [B,N], any dtype, CUDA
#     int64_t w,
#     int64_t ring_inner,
#     int64_t ring_outer,
#     int64_t mode,        // 0=window-only, 1=ring-only, 2=window+ring
#     int64_t num_anchors,
#     bool include_self,
#     int64_t anchor_strategy, // NEW
#     int64_t anchor_param0    // NEW
# ) {
#     CHECK_CUDA(attn_mask_bn);
#     CHECK_CONTIGUOUS(attn_mask_bn);
#     CHECK_2D(attn_mask_bn);

#     TORCH_CHECK(w >= 0, "w must be >= 0");
#     TORCH_CHECK(num_anchors >= 0, "num_anchors must be >= 0");
#     TORCH_CHECK(mode == 0 || mode == 1 || mode == 2, "mode must be 0 (window), 1 (ring), or 2 (window+ring)");

#     // FIX #2: For causal attention, include_self must be true to avoid empty candidate set at i=0.
#     // Without self, token 0 has no valid causal keys (and many kernels will NaN).
#     TORCH_CHECK(include_self, "include_self must be true for causal scaffolds (prevents empty candidate rows at i=0).");

#     const int64_t B64 = attn_mask_bn.size(0);
#     const int64_t N64 = attn_mask_bn.size(1);
#     TORCH_CHECK(B64 > 0 && N64 > 0, "attn_mask_bn must be non-empty");

#     TORCH_CHECK(B64 <= INT32_MAX && N64 <= INT32_MAX, "B and N must fit in int32");

#     // =========================
#     // PATCH 1: overflow/range-hardening for w/Cw/Cmax
#     // =========================
#     TORCH_CHECK(w <= N64, "w must be <= sequence length N");

#     // Window slots only if mode includes window
#     int64_t Cw64 = 0;
#     if (mode == 0 || mode == 2) {
#         Cw64 = 2 * w + 1;
#         TORCH_CHECK(Cw64 > 0, "Cw must be positive (overflow?)");
#     }

#     // Ring slots only if mode includes ring
#     int64_t Cr64 = 0;
#     if (mode == 1 || mode == 2) {
#         TORCH_CHECK(ring_inner >= 1, "ring_inner must be >= 1 when ring is enabled");
#         TORCH_CHECK(ring_outer >= ring_inner, "ring_outer must be >= ring_inner when ring is enabled");
#         TORCH_CHECK(ring_outer <= (N64 - 1), "ring_outer must be <= N-1 when ring is enabled");
#         Cr64 = (ring_outer - ring_inner + 1);
#         TORCH_CHECK(Cr64 > 0, "Cr must be positive (overflow?)");
#     } else {
#         // ring disabled -> ignore provided values
#         ring_inner = 0;
#         ring_outer = 0;
#     }

#     // FIX #3: ring-only requires anchors (otherwise early tokens have zero valid ring keys).
#     if (mode == 1) {
#         TORCH_CHECK(num_anchors > 0, "ring-only requires num_anchors > 0 to avoid empty candidate sets for early tokens.");
#     }

#     int64_t A64 = 0;
#     if (num_anchors > 0) A64 = std::min<int64_t>(num_anchors, N64);

#     const int64_t Cmax64 = Cw64 + Cr64 + A64;
#     TORCH_CHECK(Cmax64 > 0, "Cmax must be positive (overflow?)");
#     TORCH_CHECK(Cmax64 <= (int64_t)INT32_MAX, "Cmax must fit in int32");

#     TORCH_CHECK(w <= (int64_t)INT32_MAX, "w too large for int32");
#     TORCH_CHECK(ring_inner <= (int64_t)INT32_MAX, "ring_inner too large for int32");
#     TORCH_CHECK(ring_outer <= (int64_t)INT32_MAX, "ring_outer too large for int32");
#     TORCH_CHECK(mode <= (int64_t)INT32_MAX, "mode too large for int32");

#     const int ww = (int)w;
#     const int rin = (int)ring_inner;
#     const int rout = (int)ring_outer;
#     const int imod = (int)mode;

#     const int Cw = (int)Cw64;
#     const int Cr = (int)Cr64;
#     const int A  = (int)A64;
#     const int Cmax = (int)Cmax64;
#     // =========================

#     const int B = (int)B64;
#     const int N = (int)N64;

#     const int64_t T64 = B64 * N64;
#     TORCH_CHECK(T64 <= INT32_MAX, "T=B*N must fit in int32 for this kernel");
#     const int T = (int)T64;

#     // Defensive: total elements must fit in int64 (PyTorch indexing uses int64 anyway)
#     const int64_t total_elems = T64 * Cmax64;
#     TORCH_CHECK(total_elems > 0, "T*Cmax must be positive (overflow?)");

#     // =========================
#     // PATCH 2: deterministic mask_u8 conversion with uint8 fast-path
#     // =========================
#     torch::Tensor mask_u8;
#     // Binarize explicitly: (mask != 0) to avoid float/half truncation edge cases.
#     mask_u8 = attn_mask_bn.ne(0).to(torch::kUInt8).contiguous();
#     TORCH_CHECK(mask_u8.is_contiguous(), "mask_u8 must be contiguous");
#     // =========================

#     auto opts_i32 = torch::TensorOptions().dtype(torch::kInt32).device(attn_mask_bn.device());
#     auto opts_u8  = torch::TensorOptions().dtype(torch::kUInt8).device(attn_mask_bn.device());

#     torch::Tensor cand = torch::empty({T, Cmax}, opts_i32);
#     torch::Tensor ok   = torch::empty({T, Cmax}, opts_u8);
#     torch::Tensor valid_T = torch::empty({T}, opts_u8);

#     // =========================
#     // PATCH 3: choose threads based on Cmax (optional micro-opt; semantics unchanged)
#     // =========================
#     int threads = 256;
#     if (Cmax <= 128) threads = 128;
#     if (Cmax <= 64)  threads = 64;
#     const dim3 block(threads);
#     const dim3 grid((unsigned int)T);
#     // =========================

#     // Stream correctness: respect PyTorch current stream
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();

#     build_candidates_kernel_u8<<<grid, block, 0, stream>>>(
#         (const uint8_t*)mask_u8.data_ptr<uint8_t>(),
#         B, N, ww,
#         rin, rout, imod,
#         A, include_self,
#         (int)anchor_strategy, (int)anchor_param0,
#         Cw, Cr, Cmax,
#         cand.data_ptr<int32_t>(),
#         ok.data_ptr<uint8_t>(),
#         valid_T.data_ptr<uint8_t>()
#     );

#     C10_CUDA_KERNEL_LAUNCH_CHECK();

#     return {cand, ok, valid_T};
# }

# PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
#     m.def("build_candidates_cuda", &build_candidates_cuda,
#           "Build candidates [T,Cmax] + mask [T,Cmax] + valid_T [T] (CUDA)");
# }"""



# # ext_name = "stageA_cuda_candidate_builder_ext"

# # Base flags
# extra_cuda_cflags = ["-O3", "--use_fast_math"]
# extra_cflags = ["-O3"]

# # # Compile-time macros so the C++ sees your Python toggles
# extra_cuda_cflags += [
#     f"-DFUSED_ENABLE_HOST_LOGS={(1 if FUSED_ENABLE_HOST_LOGS else 0)}",
#     f"-DFUSED_ENABLE_DEVICE_LOGS={(1 if FUSED_ENABLE_DEVICE_LOGS else 0)}",
#     f"-DFUSED_SYNC_AFTER_KERNEL={(1 if FUSED_SYNC_AFTER_KERNEL else 0)}",
#     f"-DFUSED_USE_SMEM_V={(1 if FUSED_USE_SMEM_V else 0)}",
# ]

# # Force rebuild when toggles change (torch extensions cache by name)
# _build_sig = (
#     f"stageA_cuda_candidate_builder|"
#     f"h{int(FUSED_ENABLE_HOST_LOGS)}"
#     f"d{int(FUSED_ENABLE_DEVICE_LOGS)}"
#     f"s{int(FUSED_SYNC_AFTER_KERNEL)}"
#     f"v{int(FUSED_USE_SMEM_V)}"
#     f"|O3_fastmath"
#     f"|anchor_strategy_v1"  # NEW: ensures cache bust when anchor logic changes
# )
# _build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
# ext_name = f"stageA_cuda_candidate_builder_ext_{_build_hash}"

# stageA_cuda_candidate_builder = load_inline(
#     name=ext_name,
#     cpp_sources="",
#     cuda_sources=CUDA_SRC,
#     functions=None,          # we use PYBIND11_MODULE
#     extra_cuda_cflags=extra_cuda_cflags,
#     extra_cflags=extra_cflags,
#     with_cuda=True,
#     verbose=bool(FUSED_VERBOSE_BUILD),
# )

# # Raw extension entry (expects NEW signature with anchor_strategy + anchor_param0)
# build_candidates_cuda_raw = stageA_cuda_candidate_builder.build_candidates_cuda
# def build_candidates_cuda(
#     attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self,
#     anchor_strategy=0, anchor_param0=0,
# ):
#     """
#     Backward-compatible wrapper for Stage-A candidate builder.
#     - Old call sites can pass the 7-arg scaffold signature.
#     - New call sites can additionally pass anchor_strategy/anchor_param0.
#     """
#     return build_candidates_cuda_raw(
#         attn_mask_bn,
#         int(w),
#         int(ring_inner),
#         int(ring_outer),
#         int(mode),
#         int(num_anchors),
#         bool(include_self),
#         int(anchor_strategy),
#         int(anchor_param0),
#     )   # Option B

# print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
# print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))
# print("detect_anomaly =", torch.is_anomaly_enabled())
# # # Backward-compatible Python wrapper:
# # # - Old call sites that pass 7 args still work.
# # # - New ablation call sites can pass anchor_strategy/anchor_param0.
# # def build_candidates_cuda(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self,
# #                           anchor_strategy=0, anchor_param0=0):
# #     return build_candidates_cuda_raw(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self,
# #                                      anchor_strategy, anchor_param0)

# # # NOTE: wrapper signature:
# # # build_candidates_cuda(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self,
# # #                       anchor_strategy=0, anchor_param0=0)


In [ ]:
# import os, shutil
# import torch
# from torch.utils.cpp_extension import load_inline

# import hashlib

# os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
# # 2) Better C++ stack traces when extensions throw
# os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "0"

# # Toggle these as needed:
# FUSED_ENABLE_HOST_LOGS   = False    # C++ host-side logs via std::cout
# FUSED_ENABLE_DEVICE_LOGS = False   # CUDA printf inside kernel
# FUSED_SYNC_AFTER_KERNEL  = False    # device sync after kernel for accurate failure location
# FUSED_VERBOSE_BUILD      = False    # verbose compile logs

# # V-tiling toggle
# FUSED_USE_SMEM_V         = True    # introduce V-tiling

# STAGEB_MH_SRC = r"""
# #include <torch/extension.h>
# #include <ATen/cuda/CUDAContext.h>
# #include <cuda.h>
# #include <cuda_runtime.h>
# #include <cuda_fp16.h>   // <-- ADDED (for half)
# #include <cuda_bf16.h>
# #include <c10/cuda/CUDAGuard.h>
# #include <vector>
# #include <stdint.h>
# #include <cublas_v2.h>

# #define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be CUDA")
# #define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
# #define CHECK_DTYPE(x, dt) TORCH_CHECK(x.scalar_type() == dt, #x " wrong dtype")

# // -----------------------------------------
# // dtype helpers
# // -----------------------------------------
# template<typename T> __device__ __forceinline__ float to_f32(T x);
# template<> __device__ __forceinline__ float to_f32<float>(float x){ return x; }
# template<> __device__ __forceinline__ float to_f32<half>(half x){ return __half2float(x); }
# template<> __device__ __forceinline__ float to_f32<__nv_bfloat16>(__nv_bfloat16 x){
# #if defined(__CUDA_ARCH__) && (__CUDA_ARCH__ < 800)
#     asm("trap;");
# #endif
#     return __bfloat162float(x);
# }

# template<typename T> __device__ __forceinline__ T from_f32(float x);
# template<> __device__ __forceinline__ float from_f32<float>(float x){ return x; }
# template<> __device__ __forceinline__ half from_f32<half>(float x){ return __float2half_rn(x); }
# template<> __device__ __forceinline__ __nv_bfloat16 from_f32<__nv_bfloat16>(float x){
# #if defined(__CUDA_ARCH__) && (__CUDA_ARCH__ < 800)
#     asm("trap;");
# #endif
#     return __float2bfloat16_rn(x);
# }

# // -----------------------------------------
# // RoPE (forward/backward) on [T,H,D]
# // We assume full rotary on D (head_dim), and D is even.
# // cos/sin are float tensors shaped [T, D/2] (token-major).
# // -----------------------------------------
# template<typename scalar_t>
# __global__ void rope_fwd_kernel(
#     scalar_t* __restrict__ Q,   // [T,H,D]
#     scalar_t* __restrict__ K,   // [T,H,D]
#     const float* __restrict__ cos, // [T, D/2]
#     const float* __restrict__ sin, // [T, D/2]
#     int T, int H, int D
# ){
#     int t = (int)blockIdx.x;
#     int h = (int)blockIdx.y;
#     int tid = (int)threadIdx.x;
#     if (t >= T || h >= H) return;

#     int halfD = D >> 1;
#     // each thread handles multiple pairs
#     for (int i = tid; i < halfD; i += (int)blockDim.x){
#         int d0 = i;
#         int d1 = i + halfD;

#         float c = cos[t * halfD + i];
#         float s = sin[t * halfD + i];

#         // Q
#         scalar_t* q = Q + ((int64_t)t * H + h) * D;
#         float q0 = to_f32<scalar_t>(q[d0]);
#         float q1 = to_f32<scalar_t>(q[d1]);
#         float rq0 = q0 * c - q1 * s;
#         float rq1 = q0 * s + q1 * c;
#         q[d0] = from_f32<scalar_t>(rq0);
#         q[d1] = from_f32<scalar_t>(rq1);

#         // K
#         scalar_t* k = K + ((int64_t)t * H + h) * D;
#         float k0 = to_f32<scalar_t>(k[d0]);
#         float k1 = to_f32<scalar_t>(k[d1]);
#         float rk0 = k0 * c - k1 * s;
#         float rk1 = k0 * s + k1 * c;
#         k[d0] = from_f32<scalar_t>(rk0);
#         k[d1] = from_f32<scalar_t>(rk1);
#     }
# }

# template<typename scalar_t>
# __global__ void rope_bwd_kernel(
#     scalar_t* __restrict__ dQ,  // [T,H,D]
#     scalar_t* __restrict__ dK,  // [T,H,D]
#     const float* __restrict__ cos, // [T, D/2]
#     const float* __restrict__ sin, // [T, D/2]
#     int T, int H, int D
# ){
#     // Backward for rotation:
#     // [x'; y'] = [ x*c - y*s ; x*s + y*c ]
#     // gradient wrt [x;y] given [dx';dy']:
#     // dx = dx'*c + dy'*s
#     // dy = -dx'*s + dy'*c
#     int t = (int)blockIdx.x;
#     int h = (int)blockIdx.y;
#     int tid = (int)threadIdx.x;
#     if (t >= T || h >= H) return;

#     int halfD = D >> 1;
#     for (int i = tid; i < halfD; i += (int)blockDim.x){
#         int d0 = i;
#         int d1 = i + halfD;

#         float c = cos[t * halfD + i];
#         float s = sin[t * halfD + i];

#         scalar_t* dq = dQ + ((int64_t)t * H + h) * D;
#         float dxp = to_f32<scalar_t>(dq[d0]);
#         float dyp = to_f32<scalar_t>(dq[d1]);
#         float dx  = dxp * c + dyp * s;
#         float dy  = -dxp * s + dyp * c;
#         dq[d0] = from_f32<scalar_t>(dx);
#         dq[d1] = from_f32<scalar_t>(dy);

#         scalar_t* dk = dK + ((int64_t)t * H + h) * D;
#         float kxp = to_f32<scalar_t>(dk[d0]);
#         float kyp = to_f32<scalar_t>(dk[d1]);
#         float kx  = kxp * c + kyp * s;
#         float ky  = -kxp * s + kyp * c;
#         dk[d0] = from_f32<scalar_t>(kx);
#         dk[d1] = from_f32<scalar_t>(ky);
#     }
# }

# // -----------------------------------------
# // GEMM helpers (cublasGemmEx).
# // Shapes:
# //   H:  [T, Dm] row-major
# //   W:  [O, Dm] row-major
# //   Y = H @ W^T -> [T, O] row-major
# // -----------------------------------------
# static void gemm_rowmajor_A_Bt(
#     cublasHandle_t handle,
#     const void* A, const void* B, void* C,
#     int T, int Dm, int O,
#     cudaDataType_t aType, cudaDataType_t bType, cudaDataType_t cType,
#     cublasComputeType_t computeType,
#     cudaStream_t stream
# ){
#     // We compute C[T,O] = A[T,Dm] * (B[O,Dm])^T -> B^T [Dm,O]
#     // cublas assumes column-major by default; easiest trick: treat row-major matrices as transposed column-major.
#     // RowMajor(A[T,Dm]) == ColMajor(A^T[Dm,T])
#     // RowMajor(C[T,O]) == ColMajor(C^T[O,T])
#     // Want: C^T[O,T] = B[O,Dm] * A^T[Dm,T]
#     // So call GEMM: (O x Dm) * (Dm x T) -> (O x T)
#     const float alpha = 1.0f;
#     const float beta  = 0.0f;

#     cublasSetStream(handle, stream);

#     // lda for B (colmajor) = O, but B is row-major [O,Dm] -> colmajor [Dm,O] if transposed.
#     // Here we pass B as-is as "A" in gemm (colmajor) with opN: it should be laid out as colmajor (O x Dm).
#     // Our B is row-major (O x Dm), which equals colmajor (Dm x O) — not same.
#     // So we use opT on B and interpret underlying as colmajor.
#     // More stable approach: use cublas with CUBLAS_OP_T / OP_N with correct leading dimensions in row-major mapping.
#     //
#     // We'll use cublasGemmEx on the transformed equation:
#     // C^T(O,T) = B(O,Dm) * A^T(Dm,T)
#     // Treat B as colmajor with lda=O (but our B is rowmajor) => mismatch.
#     //
#     // To avoid layout headaches, we require contiguous and use cublasLt in the final version.
#     // For now, use ATen matmul fallback if you want perfect correctness.
#     TORCH_CHECK(false, "gemm_rowmajor_A_Bt called but cublas row-major mapping is disabled in this minimal build. Use the ATen fallback path below or swap to your existing cublasLt helper.");
# }

# // -----------------------------------------
# // Forward/Backward interface
# // We provide two modes:
# //   - fast path: use your existing cublasLt helper (swap in here)
# //   - safe path: ATen matmul (always correct, slower)
# // -----------------------------------------

# std::vector<torch::Tensor> stageb_mh_project_forward(
#     torch::Tensor H,   // [T, Dm]
#     torch::Tensor Wq,  // [H*D, Dm]
#     torch::Tensor Wk,  // [H*D, Dm]
#     torch::Tensor Wv,  // [H*DV, Dm]
#     torch::Tensor cos, // [T, D/2] float
#     torch::Tensor sin  // [T, D/2] float
# ){
#     CHECK_CUDA(H); CHECK_CUDA(Wq); CHECK_CUDA(Wk); CHECK_CUDA(Wv);
#     CHECK_CONTIGUOUS(H); CHECK_CONTIGUOUS(Wq); CHECK_CONTIGUOUS(Wk); CHECK_CONTIGUOUS(Wv);
#     CHECK_CUDA(cos); CHECK_CUDA(sin);
#     CHECK_CONTIGUOUS(cos); CHECK_CONTIGUOUS(sin);
#     TORCH_CHECK(cos.scalar_type() == torch::kFloat && sin.scalar_type() == torch::kFloat, "cos/sin must be float32");

#     // ---- PATCH: lock device + make dtype invariants explicit ----
#     c10::cuda::CUDAGuard guard(H.device());
#     TORCH_CHECK(H.is_floating_point() && Wq.is_floating_point() && Wk.is_floating_point() && Wv.is_floating_point(),
#                 "H/Wq/Wk/Wv must be floating point");

#     auto T = (int)H.size(0);
#     auto Dm = (int)H.size(1);
#     TORCH_CHECK(Wq.size(1) == Dm && Wk.size(1) == Dm && Wv.size(1) == Dm, "Wq/Wk/Wv second dim must match Dm");

#     int qk_out = (int)Wq.size(0);
#     int v_out  = (int)Wv.size(0);
#     TORCH_CHECK(qk_out == (int)Wk.size(0), "Wq/Wk out dims mismatch");
#     TORCH_CHECK(qk_out > 0 && v_out > 0, "invalid out dims");

#     // Infer H and D from qk_out and cos shape: qk_out = Hh * D
#     int halfD = (int)cos.size(1);
#     int D = halfD * 2;
#     TORCH_CHECK(D % 2 == 0, "D must be even");
#     TORCH_CHECK(cos.size(0) == T && sin.size(0) == T && sin.size(1) == halfD, "cos/sin shape must be [T, D/2]");
#     TORCH_CHECK(qk_out % D == 0, "Wq out dim must be multiple of head_dim (D)");
#     int Hh = qk_out / D;
#     TORCH_CHECK(v_out % Hh == 0, "Wv out dim must be multiple of n_heads");
#     int DV = v_out / Hh;

#     // ---- FP32 compute (cast inputs to fp32; outputs STAY fp32; no downcast) ----
#     auto Hf  = H.to(torch::kFloat).contiguous();
#     auto Wqf = Wq.to(torch::kFloat).contiguous();
#     auto Wkf = Wk.to(torch::kFloat).contiguous();
#     auto Wvf = Wv.to(torch::kFloat).contiguous();

#     // ---- SAFE fallback projections using ATen matmul ----
#     // H: [T,Dm], Wq: [Hh*D,Dm] -> H @ Wq.T => [T,Hh*D]
#     auto Qflat = torch::matmul(Hf, Wqf.t());
#     auto Kflat = torch::matmul(Hf, Wkf.t());
#     auto Vflat = torch::matmul(Hf, Wvf.t());

#     auto Q = Qflat.view({T, Hh, D}).contiguous();
#     auto K = Kflat.view({T, Hh, D}).contiguous();
#     auto V = Vflat.view({T, Hh, DV}).contiguous();

#     // apply RoPE (in-place) in fp32
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();
#     const dim3 grid(T, Hh);
#     const dim3 block(128);

#     rope_fwd_kernel<float><<<grid, block, 0, stream>>>(
#         (float*)Q.data_ptr(), (float*)K.data_ptr(),
#         (const float*)cos.data_ptr(), (const float*)sin.data_ptr(),
#         T, Hh, D
#     );

#     C10_CUDA_KERNEL_LAUNCH_CHECK();

#     // outputs remain FP32 (Stage-E must be made compatible to consume FP32 Q/K/V)
#     return {Q, K, V};
# }

# std::vector<torch::Tensor> stageb_mh_project_backward(
#     torch::Tensor dQ,  // [T,H,D]
#     torch::Tensor dK,  // [T,H,D]
#     torch::Tensor dV,  // [T,H,DV]
#     torch::Tensor H,   // [T,Dm]
#     torch::Tensor Wq,  // [H*D, Dm]
#     torch::Tensor Wk,  // [H*D, Dm]
#     torch::Tensor Wv,  // [H*DV, Dm]
#     torch::Tensor cos, // [T, D/2] float
#     torch::Tensor sin  // [T, D/2] float
# ){
#     CHECK_CUDA(dQ); CHECK_CUDA(dK); CHECK_CUDA(dV);
#     CHECK_CUDA(H); CHECK_CUDA(Wq); CHECK_CUDA(Wk); CHECK_CUDA(Wv);
#     CHECK_CUDA(cos); CHECK_CUDA(sin);
#     CHECK_CONTIGUOUS(dQ); CHECK_CONTIGUOUS(dK); CHECK_CONTIGUOUS(dV);
#     CHECK_CONTIGUOUS(H); CHECK_CONTIGUOUS(Wq); CHECK_CONTIGUOUS(Wk); CHECK_CONTIGUOUS(Wv);
#     CHECK_CONTIGUOUS(cos); CHECK_CONTIGUOUS(sin);
#     TORCH_CHECK(cos.scalar_type() == torch::kFloat && sin.scalar_type() == torch::kFloat, "cos/sin must be float32");

#     // ---- PATCH: lock device + make dtype invariants explicit ----
#     c10::cuda::CUDAGuard guard(H.device());
#     TORCH_CHECK(dQ.is_floating_point() && dK.is_floating_point() && dV.is_floating_point() &&
#                 H.is_floating_point() && Wq.is_floating_point() && Wk.is_floating_point() && Wv.is_floating_point(),
#                 "dQ/dK/dV/H/Wq/Wk/Wv must be floating point");

#     auto T = (int)H.size(0);
#     auto Dm = (int)H.size(1);

#     // ---- PATCH: mirror forward dim check for W second dim ----
#     TORCH_CHECK(Wq.size(1) == Dm && Wk.size(1) == Dm && Wv.size(1) == Dm, "Wq/Wk/Wv second dim must match Dm");

#     TORCH_CHECK(dQ.size(0) == T && dK.size(0) == T && dV.size(0) == T, "dQ/dK/dV T mismatch");
#     int Hh = (int)dQ.size(1);
#     int D  = (int)dQ.size(2);
#     int DV = (int)dV.size(2);
#     TORCH_CHECK(dK.size(1) == Hh && dK.size(2) == D, "dK shape mismatch");
#     TORCH_CHECK(dV.size(1) == Hh, "dV shape mismatch");
#     TORCH_CHECK((int)Wq.size(0) == Hh * D && (int)Wk.size(0) == Hh * D && (int)Wv.size(0) == Hh * DV, "W dims mismatch");

#     // ---- FP32 compute (cast inputs to fp32; grads STAY fp32; no downcast) ----
#     auto Hf  = H.to(torch::kFloat).contiguous();
#     auto Wqf = Wq.to(torch::kFloat).contiguous();
#     auto Wkf = Wk.to(torch::kFloat).contiguous();
#     auto Wvf = Wv.to(torch::kFloat).contiguous();

#     auto dQf = dQ.to(torch::kFloat).contiguous();
#     auto dKf = dK.to(torch::kFloat).contiguous();
#     auto dVf = dV.to(torch::kFloat).contiguous();

#     // ---- PATCH: avoid in-place mutation of caller-provided dQ/dK ----
#     auto dQ_work = dQf.clone();
#     auto dK_work = dKf.clone();

#     // Apply RoPE backward to dQ/dK in-place (work tensors) in fp32
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();
#     const dim3 grid(T, Hh);
#     const dim3 block(128);

#     rope_bwd_kernel<float><<<grid, block, 0, stream>>>(
#         (float*)dQ_work.data_ptr(), (float*)dK_work.data_ptr(),
#         (const float*)cos.data_ptr(), (const float*)sin.data_ptr(),
#         T, Hh, D
#     );

#     C10_CUDA_KERNEL_LAUNCH_CHECK();

#     // Flatten head dims for GEMMs
#     auto dQflat = dQ_work.reshape({T, Hh * D}).contiguous();
#     auto dKflat = dK_work.reshape({T, Hh * D}).contiguous();
#     auto dVflat = dVf.reshape({T, Hh * DV}).contiguous();

#     // ---- SAFE fallback grads using ATen matmul ----
#     // dW = dOut^T @ H  where dOut is [T,O] => dW [O,Dm]
#     auto dWq = torch::matmul(dQflat.t(), Hf);
#     auto dWk = torch::matmul(dKflat.t(), Hf);
#     auto dWv = torch::matmul(dVflat.t(), Hf);

#     // dH = dQ @ Wq + dK @ Wk + dV @ Wv  (W are [O,Dm], so W^T is [Dm,O], but we need [T,Dm]
#     auto dH = torch::matmul(dQflat, Wqf) + torch::matmul(dKflat, Wkf) + torch::matmul(dVflat, Wvf);

#     // grads remain FP32 (Stage-E + upstream must be made compatible to consume FP32 grads where applicable)
#     return {dH, dWq, dWk, dWv};
# }

# PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
#     m.def("proj_forward",  &stageb_mh_project_forward,  "StageB-MH project forward (QKV + RoPE)");
#     m.def("proj_backward", &stageb_mh_project_backward, "StageB-MH project backward (RoPE-bwd + GEMM grads)");
# }
# """

# # ext_name = "stageB_MH_projector_rope_stageE_bridge_ext"
# import hashlib

# # ---- build-hash (forces rebuild when code / flags / linkage change) ----
# _src_hash = hashlib.md5(STAGEB_MH_SRC.encode("utf-8")).hexdigest()[:10]
# _build_sig = (
#     "stageB_MH_projector_rope_stageE_bridge|"
#     "O3_fastmath|"
#     "cxx17|"
#     "cublas|"
#     f"src={_src_hash}"
# )
# _build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]
# ext_name = f"stageB_MH_projector_rope_stageE_bridge_ext_{_build_hash}"

# stageB_MH_projector_rope_stageE_bridge = load_inline(
#     name=ext_name,
#     cpp_sources="",
#     cuda_sources=STAGEB_MH_SRC,
#     functions=None,          # we use PYBIND11_MODULE
#     extra_cuda_cflags=["-O3", "--use_fast_math"],
#     extra_cflags=["-O3", "-std=c++17"],
#     with_cuda=True,
#     verbose=False,
#     extra_ldflags=["-lcublas"],
# )

# print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
# print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))
# print("detect_anomaly =", torch.is_anomaly_enabled())

# proj_forward  = stageB_MH_projector_rope_stageE_bridge.proj_forward
# proj_backward = stageB_MH_projector_rope_stageE_bridge.proj_backward


In [ ]:
# # Colab-friendly single Python cell that compiles your Stage-E (MH fused sparse attention)
# # ✅ Keeps your CUDA/C++ code + comments VERBATIM (except the minimal FP32-compat changes below)
# # ✅ Makes Stage-E compatible with Stage-B that returns FP32 Q/K/V (no downcast)
# # ✅ Risky numerics stay FP32: dot/softmax/out-accum all operate on FP32 tensors in the wrapper
# # ✅ The ONE minimal sm75 compile fix is preserved:
# #    __nv_bfloat16 z; z.x=0; return z;  -->  __nv_bfloat16 z{}; return z;
# #
# # ✅ PATCH A/B/C (Backward stability):
# #   PATCH A: FIX do_out reduction bug (was multiplying do_out by BLOCK_X => exploding grads)
# #   PATCH B: clamp (score - m) <= 0 before exp to prevent overflow if forward/backward mismatch
# #   PATCH C: isfinite(l)/isfinite(p)/isfinite(do_v)/isfinite(g) guards to prevent NaN/INF poisoning
# #
# # ✅ NEW (Cand integrity / mutation fix path):
# #   Adds mh_forward_frozen(...) that returns {out, m, l, cand_frozen, mask_frozen}
# #   where cand/mask are cloned+contiguous inside the C++ wrapper (never mutated in-place).
# #   This lets your Python autograd wrapper save the *frozen* tensors for backward, preventing
# #   the “forward saw -1 but backward saw 0” mismatch caused by external in-place clamping or
# #   buffer reuse between forward/backward.

# import os, re, math, pathlib, subprocess, torch
# from torch.utils.cpp_extension import load_inline

# # ----------------------------
# # Environment + build dir
# # ----------------------------
# os.environ.setdefault("TORCH_EXTENSIONS_DIR", "/content/torch_extensions")
# TORCH_EXT_DIR = os.environ["TORCH_EXTENSIONS_DIR"]

# print("torch:", torch.__version__)
# print("cuda available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("gpu:", torch.cuda.get_device_name(0))
#     print("cc:", torch.cuda.get_device_capability(0))

# import hashlib

# os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
# # 2) Better C++ stack traces when extensions throw
# os.environ["TORCH_SHOW_CPP_STACKTRACES"] = "0"
# # Toggle these as needed:
# FUSED_ENABLE_HOST_LOGS   = False    # C++ host-side logs via std::cout
# FUSED_ENABLE_DEVICE_LOGS = False   # CUDA printf inside kernel
# FUSED_SYNC_AFTER_KERNEL  = False    # device sync after kernel for accurate failure location
# FUSED_VERBOSE_BUILD      = False    # verbose compile logs

# # V-tiling toggle
# FUSED_USE_SMEM_V         = True    # introduce V-tiling

# # ----------------------------
# # Your code (VERBATIM) as a CUDA source string
# # ----------------------------
# CUDA_SRC = r"""
# // ============================================================================
# // Stage E: Multi-head fused sparse attention (streaming softmax, MH)
# // Exports: mh_forward, mh_backward
# //
# // Q,K: [T,H,D]   (float/half/bf16)
# // V  : [T,H,DV]  (float/half/bf16)  (must match Q/K dtype for simplicity)
# // cand/mask: [T,C] or [T*H,C] (controlled by cand_is_per_head)
# // m,l: [T*H] float (saved for backward)
# //
# // This revision patches the remaining *real* bugs/hazards called out in audit:
# //  - FIX #1 (forward): smem_v write now explicitly guards (dv < DV) at write site
# //  - FIX #2 (forward): exp_s / l_new finite guards to avoid INF/NaN propagating into alpha/beta
# //  - PATCH A/B/C (backward): fixes do_out scaling bug + clamps/guards to prevent exploding grads
# //  - Minor hardening: clearer invariants + comments (no behavior change)
# //
# // Notes:
# //  - Backward shared-memory tree reductions are correct (just slower than warp reductions).
# //    Not changed here because user asked “patch remaining bugs” (correctness first).
# //
# // ============================================================================
# // FP32 COMPATIBILITY NOTE (Stage-B bridge):
# //   Stage-B now returns FP32 Q/K/V (no downcast) for stability.
# //   To keep Stage-E stable + compatible, the *wrappers* below cast Q/K/V/dOut to FP32
# //   and run ONLY the float kernels. This ensures the risky pieces (softmax + out updates)
# //   never round-trip through FP16/BF16 during the loop.
# //   - Output of mh_forward is FP32.
# //   - Backward returns grads cast back to the *original* dtype for compatibility,
# //     but when driven by Stage-B FP32, grads stay FP32.
# //
# //   Kernels remain templated and unchanged; only wrapper dispatch/casting changes.
# // ============================================================================

# #include <torch/extension.h>
# #include <ATen/cuda/CUDAContext.h>
# #include <cuda.h>
# #include <cuda_runtime.h>
# #include <cuda_fp16.h>
# #include <cuda_bf16.h>
# #include <vector>
# #include <limits>
# #include <type_traits>
# // ----------------------------
# // SASA Smoke-Test Debug Macros
# //   - Enable device-side printf with -DSASA_DEVICE_DEBUG
# //   - Keep kernels semantically identical; logs only.
# // ----------------------------
# #if defined(SASA_DEVICE_DEBUG)
#   #include <cstdio>
#   #define SASA_DPRINTF(...) printf(__VA_ARGS__)
# #else
#   #define SASA_DPRINTF(...) ((void)0)
# #endif

# #ifndef BLOCK_X
# #define BLOCK_X 256
# #endif

# #ifndef V_TILE
# #define V_TILE 256
# #endif

# #if (BLOCK_X & (BLOCK_X - 1)) != 0
# #error "BLOCK_X must be a power of two."
# #endif

# static inline void check_cuda(const torch::Tensor& t, const char* name) {
#     TORCH_CHECK(t.is_cuda(), name, " must be CUDA");
# }
# static inline void check_contig(const torch::Tensor& t, const char* name) {
#     TORCH_CHECK(t.is_contiguous(), name, " must be contiguous");
# }
# static inline void check_dtype_eq(const torch::Tensor& a, const torch::Tensor& b, const char* an, const char* bn) {
#     TORCH_CHECK(a.scalar_type() == b.scalar_type(), an, " and ", bn, " dtype must match");
# }

# __device__ __forceinline__ void trap_if(bool cond) { if (cond) asm("trap;"); }
# __device__ __forceinline__ float expf_safe(float x) { return expf(x); }
# __device__ __forceinline__ int isfinite_f(float x) { return isfinite(x); }

# // -----------------------------
# // dtype helpers
# // -----------------------------
# template <typename T>
# __device__ __forceinline__ float to_f32(T x);

# template <>
# __device__ __forceinline__ float to_f32<float>(float x) { return x; }

# template <>
# __device__ __forceinline__ float to_f32<half>(half x) { return __half2float(x); }

# template <>
# __device__ __forceinline__ float to_f32<__nv_bfloat16>(__nv_bfloat16 x) {
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ >= 800
#     return __bfloat162float(x);
# #else
#     asm("trap;");
#     return 0.0f;
# #endif
# }

# template <typename T>
# __device__ __forceinline__ T from_f32(float x);

# template <>
# __device__ __forceinline__ float from_f32<float>(float x) { return x; }

# template <>
# __device__ __forceinline__ half from_f32<half>(float x) { return __float2half_rn(x); }

# template <>
# __device__ __forceinline__ __nv_bfloat16 from_f32<__nv_bfloat16>(float x) {
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ >= 800
#     return __float2bfloat16_rn(x);
# #else
#     asm("trap;");
#     // ---- MINIMAL FIX (compile on sm75): no guarantee of public `.x` field ----
#     __nv_bfloat16 z{};
#     return z;
# #endif
# }

# // -----------------------------
# // warp / block reduce
# // -----------------------------
# __device__ __forceinline__ float warp_reduce_sum(float v) {
# #pragma unroll
#     for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(0xffffffff, v, off);
#     return v;
# }

# template <int BLOCK>
# __device__ __forceinline__ float block_reduce_sum(float v, float* __restrict__ smem_warp_sums) {
#     const int lane   = (int)threadIdx.x & 31;
#     const int warp   = (int)threadIdx.x >> 5;
#     const int nwarps = BLOCK >> 5;

#     // reduce within warp
#     v = warp_reduce_sum(v);
#     if (lane == 0) smem_warp_sums[warp] = v;
#     __syncthreads();

#     // warp 0 reduces warp-sums, then writes a single scalar for the whole block
#     if (warp == 0) {
#         float out = (lane < nwarps) ? smem_warp_sums[lane] : 0.0f;
#         out = warp_reduce_sum(out);
#         if (lane == 0) smem_warp_sums[0] = out;
#     }
#     __syncthreads();

#     // broadcast via shared memory to all threads
#     return smem_warp_sums[0];
# }


# // -----------------------------
# // vectorized dot (safe fallback)
# // -----------------------------
# __device__ __forceinline__ float dot_vec_float(const float* __restrict__ q,
#                                               const float* __restrict__ k,
#                                               int D) {
#     const uintptr_t qa = (uintptr_t)q;
#     const uintptr_t ka = (uintptr_t)k;
#     const bool aligned8 = ((qa & 0x7u) == 0u) && ((ka & 0x7u) == 0u);
#     const bool even = ((D & 1) == 0);

#     float acc = 0.0f;
#     if (aligned8 && even) {
#         const float2* q2 = reinterpret_cast<const float2*>(q);
#         const float2* k2 = reinterpret_cast<const float2*>(k);
#         const int n2 = D >> 1;
#         for (int i2 = (int)threadIdx.x; i2 < n2; i2 += BLOCK_X) {
#             float2 a = q2[i2];
#             float2 b = k2[i2];
#             acc += a.x * b.x + a.y * b.y;
#         }
#     } else {
#         for (int d = (int)threadIdx.x; d < D; d += BLOCK_X) acc += q[d] * k[d];
#     }
#     return acc;
# }

# __device__ __forceinline__ float dot_vec_half(const half* __restrict__ q,
#                                              const half* __restrict__ k,
#                                              int D) {
#     const uintptr_t qa = (uintptr_t)q;
#     const uintptr_t ka = (uintptr_t)k;
#     const bool aligned4 = ((qa & 0x3u) == 0u) && ((ka & 0x3u) == 0u);
#     const bool even = ((D & 1) == 0);

#     float acc = 0.0f;
#     if (aligned4 && even) {
#         const half2* q2 = reinterpret_cast<const half2*>(q);
#         const half2* k2 = reinterpret_cast<const half2*>(k);
#         const int n2 = D >> 1;
#         for (int i2 = (int)threadIdx.x; i2 < n2; i2 += BLOCK_X) {
#             float2 af = __half22float2(q2[i2]);
#             float2 bf = __half22float2(k2[i2]);
#             acc += af.x * bf.x + af.y * bf.y;
#         }
#     } else {
#         for (int d = (int)threadIdx.x; d < D; d += BLOCK_X) {
#             acc += __half2float(q[d]) * __half2float(k[d]);
#         }
#     }
#     return acc;
# }

# __device__ __forceinline__ float dot_vec_bf16(const __nv_bfloat16* __restrict__ q,
#                                              const __nv_bfloat16* __restrict__ k,
#                                              int D) {
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ >= 800
#     const uintptr_t qa = (uintptr_t)q;
#     const uintptr_t ka = (uintptr_t)k;
#     const bool aligned4 = ((qa & 0x3u) == 0u) && ((ka & 0x3u) == 0u);
#     const bool even = ((D & 1) == 0);

#     float acc = 0.0f;
#     if (aligned4 && even) {
#         const __nv_bfloat162* q2 = reinterpret_cast<const __nv_bfloat162*>(q);
#         const __nv_bfloat162* k2 = reinterpret_cast<const __nv_bfloat162*>(k);
#         const int n2 = D >> 1;
#         for (int i2 = (int)threadIdx.x; i2 < n2; i2 += BLOCK_X) {
#             float2 af = __bfloat1622float2(q2[i2]);
#             float2 bf = __bfloat1622float2(k2[i2]);
#             acc += af.x * bf.x + af.y * bf.y;
#         }
#     } else {
#         for (int d = (int)threadIdx.x; d < D; d += BLOCK_X) {
#             acc += __bfloat162float(q[d]) * __bfloat162float(k[d]);
#         }
#     }
#     return acc;
# #else
#     asm("trap;");
#     return 0.0f;
# #endif
# }

# template <typename QK_T>
# __device__ __forceinline__ float dot_vec(const QK_T* __restrict__ q,
#                                         const QK_T* __restrict__ k,
#                                         int D) {
#     if constexpr (std::is_same<QK_T, float>::value) return dot_vec_float((const float*)q, (const float*)k, D);
#     if constexpr (std::is_same<QK_T, half>::value)  return dot_vec_half((const half*)q, (const half*)k, D);
#     return dot_vec_bf16((const __nv_bfloat16*)q, (const __nv_bfloat16*)k, D);
# }

# // cand/mask row base
# __device__ __forceinline__ int64_t cand_row_base(int t, int h, int H, int C, int cand_is_per_head) {
#     return (cand_is_per_head ? (int64_t)(t * H + h) : (int64_t)t) * (int64_t)C;
# }

# // ============================================================================
# // Forward kernel
# // ============================================================================

# template <typename QK_T, typename V_T>
# __global__ void fused_fwd_streaming_mh_kernel(
#     const QK_T* __restrict__ Q,     // [T,H,D]
#     const QK_T* __restrict__ K,     // [T,H,D]
#     const V_T*  __restrict__ V,     // [T,H,DV]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     float* __restrict__ m_out,      // [T*H]
#     float* __restrict__ l_out,      // [T*H]
#     V_T* __restrict__ out,          // [T,H,DV]
#     int T, int H, int D, int DV, int C,
#     float scale,
#     int cand_is_per_head
# ) {
#   if (blockIdx.x==0 && threadIdx.x==0) { SASA_DPRINTF("[SASA_DEBUG] enter %s\n", "fused_fwd_streaming_mh_kernel"); }

#     // Mislaunch should never be silent.
#     trap_if(blockDim.x != BLOCK_X); // blockDim.x must equal BLOCK_X

#     if constexpr (std::is_same<QK_T, __nv_bfloat16>::value || std::is_same<V_T, __nv_bfloat16>::value) {
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ < 800
#         asm("trap;");
# #endif
#     }

#     const int t = (int)blockIdx.x;
#     const int h = (int)blockIdx.y;
#     const int tid = (int)threadIdx.x;
#     if (t >= T || h >= H) return;

#     const int64_t th = (int64_t)t * (int64_t)H + (int64_t)h;
#     const int64_t row = cand_row_base(t, h, H, C, cand_is_per_head);

#     // dynamic shared:
#     //   smem_q: 256 * QK_T
#     //   smem_k: 256 * QK_T
#     //   smem_warp: (BLOCK_X/32) floats
#     //   smem_v: V_TILE floats
#     extern __shared__ unsigned char smem_raw[];
#     QK_T* smem_q = reinterpret_cast<QK_T*>(smem_raw);
#     QK_T* smem_k = reinterpret_cast<QK_T*>(smem_raw + (size_t)256 * sizeof(QK_T));
#     float* smem_warp = reinterpret_cast<float*>(smem_raw + (size_t)512 * sizeof(QK_T));
#     float* smem_v = reinterpret_cast<float*>(
#         smem_raw + (size_t)512 * sizeof(QK_T) + (size_t)(BLOCK_X / 32) * sizeof(float));

#     // load Q once
#     const QK_T* qg = Q + th * (int64_t)D;
#     for (int d = tid; d < D; d += BLOCK_X) smem_q[d] = qg[d];
#     __syncthreads();

#     // init out
#     for (int dv = tid; dv < DV; dv += BLOCK_X) out[th * (int64_t)DV + (int64_t)dv] = from_f32<V_T>(0.0f);
#     __syncthreads();

#     float m = -INFINITY;
#     float l = 0.0f;

#     for (int jj = 0; jj < C; ++jj) {
#       // ---- PATCH: mask must gate any use of cand index ----
#       const uint8_t m_ij = mask[row + (int64_t)jj];
#       if (!m_ij) continue;

#       const int32_t j = cand[row + (int64_t)jj];

#       // ---- PATCH: explicit clamp/guard for invalid indices (e.g., -1) ----
#       if (j < 0 || j >= T) {
#         #if defined(SASA_DEVICE_DEBUG)
#         if (blockIdx.x == 0 && blockIdx.y == 0 && threadIdx.x == 0) {
#             SASA_DPRINTF("[SASA_DEBUG] invalid cand j=%d at (t=%d,h=%d,jj=%d) mask=1\n", j, t, h, jj);
#         }
#         #endif
#         continue;
#       }

#       const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;

#     // ... keep the rest VERBATIM ...


#         // load K into shared
#         const QK_T* kg = K + jh * (int64_t)D;
#         for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
#         __syncthreads();

#         float partial = dot_vec<QK_T>(smem_q, smem_k, D);
#         float score = block_reduce_sum<BLOCK_X>(partial, smem_warp) * scale;
#         if (!isfinite_f(score)) { __syncthreads(); continue; }

#         float m_new = fmaxf(m, score);
#         float exp_m = (m == -INFINITY) ? 0.0f : expf_safe(m - m_new);

#         // ---- FIX #2: finite guards around exp / l_new ----
#         float exp_s = expf_safe(score - m_new);
#         if (!isfinite_f(exp_s)) exp_s = 0.0f; // avoid INF/NaN feeding l_new

#         float l_new = l * exp_m + exp_s;
#         if (!isfinite_f(l_new) || l_new <= 0.0f) {
#             // If this happens, safest action is to skip this candidate update
#             // (prevents alpha/beta NaNs from poisoning out).
#             __syncthreads();
#             continue;
#         }

#         float beta  = (l * exp_m) / l_new;
#         float alpha = exp_s / l_new;

#         const V_T* vg = V + jh * (int64_t)DV;

#         // V tiling: smem_v[k] holds v[dv0 + k] for k in [0..V_TILE-1]
#         // Always write all entries (zero-fill) so DV<V_TILE is safe.
#         for (int dv0 = 0; dv0 < DV; dv0 += V_TILE) {
#             const int dv = dv0 + tid;

#             // ---- FIX #1: explicit (dv < DV) guard at the write site ----
#             float vval = 0.0f;
#             if (dv < DV) vval = to_f32<V_T>(vg[dv]);
#             if (tid < V_TILE) smem_v[tid] = (dv < DV) ? vval : 0.0f;
#             __syncthreads();

#             if (dv < DV) {
#                 float old = to_f32<V_T>(out[th * (int64_t)DV + (int64_t)dv]);
#                 float upd = beta * old + alpha * smem_v[tid];
#                 out[th * (int64_t)DV + (int64_t)dv] = from_f32<V_T>(upd);
#             }
#             __syncthreads();
#         }

#         m = m_new;
#         l = l_new;
#     }

#     if (tid == 0) {
#         m_out[th] = m;
#         l_out[th] = l;
#     }
# }

# // ============================================================================
# // Backward kernel
# // ============================================================================

# template <typename QK_T, typename V_T>
# __global__ void fused_bwd_mh_kernel(
#     const V_T* __restrict__ dOut,   // [T,H,DV]
#     const QK_T* __restrict__ Q,     // [T,H,D]
#     const QK_T* __restrict__ K,     // [T,H,D]
#     const V_T*  __restrict__ V,     // [T,H,DV]
#     const int32_t* __restrict__ cand,
#     const uint8_t* __restrict__ mask,
#     const float* __restrict__ m_in, // [T*H]
#     const float* __restrict__ l_in, // [T*H]
#     float* __restrict__ dQ,         // [T*H,D] fp32
#     float* __restrict__ dK,         // [T*H,D] fp32
#     float* __restrict__ dV,         // [T*H,DV] fp32
#     int T, int H, int D, int DV, int C,
#     float scale,
#     int cand_is_per_head
# ) {
#     trap_if(blockDim.x != BLOCK_X); // mislaunch: blockDim.x must equal BLOCK_X

#     if constexpr (std::is_same<QK_T, __nv_bfloat16>::value || std::is_same<V_T, __nv_bfloat16>::value) {
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ < 800
#         asm("trap;");
# #endif
#     }

#     const int t = (int)blockIdx.x;
#     const int h = (int)blockIdx.y;
#     const int tid = (int)threadIdx.x;
#     if (t >= T || h >= H) return;

#     const int64_t th = (int64_t)t * (int64_t)H + (int64_t)h;
#     const int64_t row = cand_row_base(t, h, H, C, cand_is_per_head);

#     float m = m_in[th];
#     float l = l_in[th];
#     if (!(isfinite_f(m)) || !(isfinite_f(l)) || l <= 0.0f) return;

#     __shared__ QK_T smem_q[256];
#     __shared__ QK_T smem_k[256];
#     __shared__ float red[BLOCK_X];
#     __shared__ float warp_sums[BLOCK_X / 32];

#     const QK_T* qg = Q + th * (int64_t)D;
#     for (int d = tid; d < D; d += BLOCK_X) smem_q[d] = qg[d];
#     __syncthreads();

#     const V_T* dout = dOut + th * (int64_t)DV;

#     // Compute do_out = Σ_j p_j (dOut·V_j)
#     float local_do_out = 0.0f;

#     for (int jj = 0; jj < C; ++jj) {
#         const uint8_t m_ij = mask[row + (int64_t)jj];
#         if (!m_ij) continue;

#         const int32_t j = cand[row + (int64_t)jj];
#         if (j < 0 || j >= T) continue;

#         const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;

#         const QK_T* kg = K + jh * (int64_t)D;
#         for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
#         __syncthreads();

#         float partial = dot_vec<QK_T>(smem_q, smem_k, D);
#         float score = block_reduce_sum<BLOCK_X>(partial, warp_sums) * scale;
#         if (!isfinite_f(score)) { __syncthreads(); continue; }

#         float z = score - m;
#         // PATCH B: guard against any forward/backward mismatch that could make (score - m) > 0
#         // (in exact math, m is the max score, so z should be <= 0).
#         z = fminf(z, 0.0f);
#         float p = expf_safe(z) / l;
#         // PATCH C: harden against NaN/INF p poisoning atomics
#         if (!isfinite_f(p)) { __syncthreads(); continue; }
#         if (p < 0.0f) p = 0.0f;
#         if (p > 1.0f) p = 1.0f;

#         const V_T* vg = V + jh * (int64_t)DV;
#         float local_do_v = 0.0f;
#         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#             local_do_v += to_f32<V_T>(dout[dv]) * to_f32<V_T>(vg[dv]);
#         }

#         red[tid] = local_do_v;
#         __syncthreads();
#         for (int off = BLOCK_X / 2; off > 0; off >>= 1) {
#             if (tid < off) red[tid] += red[tid + off];
#             __syncthreads();
#         }
#         float do_v = red[0];
#         if (!isfinite_f(do_v)) { __syncthreads(); continue; }

#         local_do_out += p * do_v;
#         __syncthreads();
#     }

#     // PATCH A: local_do_out is already a full scalar Σ_j p_j (dOut·V_j).
#     // Every thread computes the same value here because `score` (hence p) and `do_v` are block-broadcast scalars.
#     // Reducing across threads would multiply by BLOCK_X and explode gradients.
#     if (tid == 0) red[0] = local_do_out;
#     __syncthreads();
#     float do_out = red[0];

#     // dQ init (safe because one block per (t,h))
#     for (int d = tid; d < D; d += BLOCK_X) {
#         dQ[th * (int64_t)D + (int64_t)d] = 0.0f;
#     }
#     __syncthreads();

#     // Main loop
#     for (int jj = 0; jj < C; ++jj) {
#         const uint8_t m_ij = mask[row + (int64_t)jj];
#         if (!m_ij) continue;

#         const int32_t j = cand[row + (int64_t)jj];
#         if (j < 0 || j >= T) continue;

#         const int64_t jh = (int64_t)j * (int64_t)H + (int64_t)h;

#         const QK_T* kg = K + jh * (int64_t)D;
#         for (int d = tid; d < D; d += BLOCK_X) smem_k[d] = kg[d];
#         __syncthreads();

#         float partial = dot_vec<QK_T>(smem_q, smem_k, D);
#         float score = block_reduce_sum<BLOCK_X>(partial, warp_sums) * scale;
#         if (!isfinite_f(score)) { __syncthreads(); continue; }

#         float z = score - m;
#         // PATCH B: guard against any forward/backward mismatch that could make (score - m) > 0
#         // (in exact math, m is the max score, so z should be <= 0).
#         z = fminf(z, 0.0f);
#         float p = expf_safe(z) / l;
#         // PATCH C: harden against NaN/INF p poisoning atomics
#         if (!isfinite_f(p)) { __syncthreads(); continue; }
#         if (p < 0.0f) p = 0.0f;
#         if (p > 1.0f) p = 1.0f;

#         // dV: float atomics only
#         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#             atomicAdd(&dV[jh * (int64_t)DV + (int64_t)dv], p * to_f32<V_T>(dout[dv]));
#         }

#         // do_v = dOut · V_j
#         const V_T* vg = V + jh * (int64_t)DV;
#         float local_do_v = 0.0f;
#         for (int dv = tid; dv < DV; dv += BLOCK_X) {
#             local_do_v += to_f32<V_T>(dout[dv]) * to_f32<V_T>(vg[dv]);
#         }

#         red[tid] = local_do_v;
#         __syncthreads();
#         for (int off = BLOCK_X / 2; off > 0; off >>= 1) {
#             if (tid < off) red[tid] += red[tid + off];
#             __syncthreads();
#         }
#         float do_v = red[0];
#         if (!isfinite_f(do_v)) { __syncthreads(); continue; }

#         float g = p * (do_v - do_out);
#         if (!isfinite_f(g)) { __syncthreads(); continue; }

#         // Chain rule through score = scale * dot(Q, K)
#         float gs = g * scale;
#         if (!isfinite_f(gs)) { __syncthreads(); continue; }

#         // dQ: no atomics needed
#         for (int d = tid; d < D; d += BLOCK_X) {
#           dQ[th * (int64_t)D + (int64_t)d] += gs * to_f32<QK_T>(kg[d]);
#         }

#         // dK: float atomics only
#         for (int d = tid; d < D; d += BLOCK_X) {
#           atomicAdd(&dK[jh * (int64_t)D + (int64_t)d], gs * to_f32<QK_T>(smem_q[d]));
#         }
#         __syncthreads();
#     }
# }


# // -----------------------------
# // Cast fp32 gradients back to dtype
# // -----------------------------
# __global__ void cast_f32_to_f32(const float* src, float* dst, int64_t n) {
#     int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
#     if (i < n) dst[i] = src[i];
# }
# __global__ void cast_f32_to_f16(const float* src, at::Half* dst, int64_t n) {
#     int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
#     if (i < n) dst[i] = (at::Half)__float2half_rn(src[i]);
# }
# __global__ void cast_f32_to_bf16(const float* src, at::BFloat16* dst, int64_t n) {
#     int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
# #if defined(__CUDA_ARCH__) && __CUDA_ARCH__ >= 800
#     if (i < n) dst[i] = (at::BFloat16)__float2bfloat16_rn(src[i]);
# #else
#     if (i < n) asm("trap;");
# #endif
# }

# // ============================================================================
# // C++ wrappers
# // ============================================================================

# std::vector<torch::Tensor> mh_forward(
#     torch::Tensor Q,        // [T,H,D]
#     torch::Tensor K,        // [T,H,D]
#     torch::Tensor V,        // [T,H,DV]
#     torch::Tensor cand_i32, // [T,C] or [T*H,C]
#     torch::Tensor mask_u8,  // [T,C] or [T*H,C]
#     double scale_py,        // e.g. 1/sqrt(D)
#     bool cand_is_per_head   // true if cand/mask are [T*H,C]
# ) {
#     check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
#     check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
#     check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
#     check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask");

#     TORCH_CHECK(Q.dim()==3 && K.dim()==3 && V.dim()==3, "Q/K/V must be [T,H,D] / [T,H,DV]");
#     TORCH_CHECK(cand_i32.dim()==2 && mask_u8.dim()==2, "cand/mask must be 2D");
#     TORCH_CHECK(cand_i32.scalar_type()==at::kInt, "cand must be int32");
#     TORCH_CHECK(mask_u8.scalar_type()==at::kByte, "mask must be uint8");
#     TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X (got V_TILE=", V_TILE, ", BLOCK_X=", BLOCK_X, ")");

#     // keep original invariants
#     check_dtype_eq(Q,K,"Q","K");
#     check_dtype_eq(Q,V,"Q","V");

#     const int64_t T64 = Q.size(0);
#     const int64_t H64 = Q.size(1);
#     const int64_t D64 = Q.size(2);
#     const int64_t DV64= V.size(2);
#     TORCH_CHECK(K.size(0)==T64 && K.size(1)==H64 && K.size(2)==D64, "K shape mismatch");
#     TORCH_CHECK(V.size(0)==T64 && V.size(1)==H64, "V shape mismatch");

#     const int64_t C64 = cand_i32.size(1);
#     TORCH_CHECK(mask_u8.size(1)==C64, "mask C mismatch");
#     TORCH_CHECK(D64 <= 256, "D must be <= 256 for this shared-tiling kernel. D=", D64);

#     TORCH_CHECK(T64 <= (int64_t)std::numeric_limits<int>::max(), "T too large");
#     TORCH_CHECK(H64 <= (int64_t)std::numeric_limits<int>::max(), "H too large");
#     TORCH_CHECK(D64 <= (int64_t)std::numeric_limits<int>::max(), "D too large");
#     TORCH_CHECK(DV64<= (int64_t)std::numeric_limits<int>::max(), "DV too large");
#     TORCH_CHECK(C64 <= (int64_t)std::numeric_limits<int>::max(), "C too large");

#     const int T = (int)T64;
#     const int H = (int)H64;
#     const int D = (int)D64;
#     const int DV= (int)DV64;
#     const int C = (int)C64;

#     const int64_t expected_rows = cand_is_per_head ? (T64 * H64) : T64;
#     TORCH_CHECK(cand_i32.size(0) == expected_rows, "cand rows mismatch (expected ", expected_rows, ")");
#     TORCH_CHECK(mask_u8.size(0) == expected_rows, "mask rows mismatch (expected ", expected_rows, ")");

#     // ============================
#     // FP32 stability bridge:
#     //   Cast Q/K/V -> float and run ONLY float kernels.
#     //   Output is FP32 (compatible with Stage-B FP32 outputs).
#     // ============================
#     auto Qf = (Q.scalar_type() == at::kFloat) ? Q : Q.to(at::kFloat);
#     auto Kf = (K.scalar_type() == at::kFloat) ? K : K.to(at::kFloat);
#     auto Vf = (V.scalar_type() == at::kFloat) ? V : V.to(at::kFloat);
#     Qf = Qf.contiguous();
#     Kf = Kf.contiguous();
#     Vf = Vf.contiguous();

#     auto out = torch::zeros({T64, H64, DV64}, Vf.options().dtype(at::kFloat));
#     auto m   = torch::empty({T64 * H64}, Q.options().dtype(at::kFloat));
#     auto l   = torch::empty({T64 * H64}, Q.options().dtype(at::kFloat));

#     const float scale = (float)scale_py;
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();

#     dim3 grid((unsigned)T, (unsigned)H, 1);
#     dim3 block(BLOCK_X, 1, 1);

#     // Shared memory size must match the instantiated kernel types.
#     // Here we ALWAYS launch <float,float>, so use sizeof(float).
#     const size_t shmem = (size_t)512 * sizeof(float)
#                        + (size_t)(BLOCK_X/32) * sizeof(float)
#                        + (size_t)V_TILE * sizeof(float);

#     fused_fwd_streaming_mh_kernel<float,float><<<grid, block, shmem, stream>>>(
#         (const float*)Qf.data_ptr(),
#         (const float*)Kf.data_ptr(),
#         (const float*)Vf.data_ptr(),
#         (const int32_t*)cand_i32.data_ptr(),
#         (const uint8_t*)mask_u8.data_ptr(),
#         (float*)m.data_ptr(),
#         (float*)l.data_ptr(),
#         (float*)out.data_ptr(),
#         T,H,D,DV,C, scale, (int)cand_is_per_head
#     );

#     auto err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_forward kernel launch failed: ", cudaGetErrorString(err));

#     return {out, m, l};
# }

# // ============================================================================
# // NEW: forward that "freezes" cand/mask (clone+contig) and returns them.
# //   This is the clean way to prevent forward/backward cand mismatch caused by:
# //     - external in-place clamping (-1 -> 0)
# //     - buffer reuse / aliasing between forward and backward
# //   Your Python autograd wrapper should save cand_frozen/mask_frozen instead of the originals.
# // ============================================================================

# std::vector<torch::Tensor> mh_forward_frozen(
#     torch::Tensor Q,        // [T,H,D]
#     torch::Tensor K,        // [T,H,D]
#     torch::Tensor V,        // [T,H,DV]
#     torch::Tensor cand_i32, // [T,C] or [T*H,C]
#     torch::Tensor mask_u8,  // [T,C] or [T*H,C]
#     double scale_py,        // e.g. 1/sqrt(D)
#     bool cand_is_per_head   // true if cand/mask are [T*H,C]
# ) {
#     check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
#     check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask");
#     check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
#     check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask");

#     TORCH_CHECK(Q.dim()==3 && K.dim()==3 && V.dim()==3, "Q/K/V must be [T,H,D] / [T,H,DV]");
#     TORCH_CHECK(cand_i32.dim()==2 && mask_u8.dim()==2, "cand/mask must be 2D");
#     TORCH_CHECK(cand_i32.scalar_type()==at::kInt, "cand must be int32");
#     TORCH_CHECK(mask_u8.scalar_type()==at::kByte, "mask must be uint8");
#     TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X (got V_TILE=", V_TILE, ", BLOCK_X=", BLOCK_X, ")");

#     // keep original invariants
#     check_dtype_eq(Q,K,"Q","K");
#     check_dtype_eq(Q,V,"Q","V");

#     const int64_t T64 = Q.size(0);
#     const int64_t H64 = Q.size(1);
#     const int64_t D64 = Q.size(2);
#     const int64_t DV64= V.size(2);
#     TORCH_CHECK(K.size(0)==T64 && K.size(1)==H64 && K.size(2)==D64, "K shape mismatch");
#     TORCH_CHECK(V.size(0)==T64 && V.size(1)==H64, "V shape mismatch");

#     const int64_t C64 = cand_i32.size(1);
#     TORCH_CHECK(mask_u8.size(1)==C64, "mask C mismatch");
#     TORCH_CHECK(D64 <= 256, "D must be <= 256 for this shared-tiling kernel. D=", D64);

#     TORCH_CHECK(T64 <= (int64_t)std::numeric_limits<int>::max(), "T too large");
#     TORCH_CHECK(H64 <= (int64_t)std::numeric_limits<int>::max(), "H too large");
#     TORCH_CHECK(D64 <= (int64_t)std::numeric_limits<int>::max(), "D too large");
#     TORCH_CHECK(DV64<= (int64_t)std::numeric_limits<int>::max(), "DV too large");
#     TORCH_CHECK(C64 <= (int64_t)std::numeric_limits<int>::max(), "C too large");

#     const int T = (int)T64;
#     const int H = (int)H64;
#     const int D = (int)D64;
#     const int DV= (int)DV64;
#     const int C = (int)C64;

#     const int64_t expected_rows = cand_is_per_head ? (T64 * H64) : T64;
#     TORCH_CHECK(cand_i32.size(0) == expected_rows, "cand rows mismatch (expected ", expected_rows, ")");
#     TORCH_CHECK(mask_u8.size(0) == expected_rows, "mask rows mismatch (expected ", expected_rows, ")");

#     // ---- FREEZE (no in-place ops on inputs): clone+contig so saved tensors can't be mutated later ----
#     auto cand_frozen = cand_i32.contiguous().clone();
#     auto mask_frozen = mask_u8.contiguous().clone();

#     // ============================
#     // FP32 stability bridge:
#     // ============================
#     auto Qf = (Q.scalar_type() == at::kFloat) ? Q : Q.to(at::kFloat);
#     auto Kf = (K.scalar_type() == at::kFloat) ? K : K.to(at::kFloat);
#     auto Vf = (V.scalar_type() == at::kFloat) ? V : V.to(at::kFloat);
#     Qf = Qf.contiguous();
#     Kf = Kf.contiguous();
#     Vf = Vf.contiguous();

#     auto out = torch::zeros({T64, H64, DV64}, Vf.options().dtype(at::kFloat));
#     auto m   = torch::empty({T64 * H64}, Q.options().dtype(at::kFloat));
#     auto l   = torch::empty({T64 * H64}, Q.options().dtype(at::kFloat));

#     const float scale = (float)scale_py;
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();

#     dim3 grid((unsigned)T, (unsigned)H, 1);
#     dim3 block(BLOCK_X, 1, 1);

#     const size_t shmem = (size_t)512 * sizeof(float)
#                        + (size_t)(BLOCK_X/32) * sizeof(float)
#                        + (size_t)V_TILE * sizeof(float);

#     fused_fwd_streaming_mh_kernel<float,float><<<grid, block, shmem, stream>>>(
#         (const float*)Qf.data_ptr(),
#         (const float*)Kf.data_ptr(),
#         (const float*)Vf.data_ptr(),
#         (const int32_t*)cand_frozen.data_ptr(),
#         (const uint8_t*)mask_frozen.data_ptr(),
#         (float*)m.data_ptr(),
#         (float*)l.data_ptr(),
#         (float*)out.data_ptr(),
#         T,H,D,DV,C, scale, (int)cand_is_per_head
#     );

#     auto err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_forward_frozen kernel launch failed: ", cudaGetErrorString(err));

#     return {out, m, l, cand_frozen, mask_frozen};
# }

# std::vector<torch::Tensor> mh_backward(
#     torch::Tensor dOut,     // [T,H,DV] same dtype as Q/V
#     torch::Tensor Q,        // [T,H,D]
#     torch::Tensor K,        // [T,H,D]
#     torch::Tensor V,        // [T,H,DV]
#     torch::Tensor cand_i32, // [T,C] or [T*H,C]
#     torch::Tensor mask_u8,  // [T,C] or [T*H,C]
#     torch::Tensor m,        // [T*H] float
#     torch::Tensor l,        // [T*H] float
#     double scale_py,
#     bool cand_is_per_head
# ) {
#     check_cuda(dOut,"dOut"); check_cuda(Q,"Q"); check_cuda(K,"K"); check_cuda(V,"V");
#     check_cuda(cand_i32,"cand"); check_cuda(mask_u8,"mask"); check_cuda(m,"m"); check_cuda(l,"l");
#     check_contig(dOut,"dOut"); check_contig(Q,"Q"); check_contig(K,"K"); check_contig(V,"V");
#     check_contig(cand_i32,"cand"); check_contig(mask_u8,"mask"); check_contig(m,"m"); check_contig(l,"l");

#     TORCH_CHECK(m.scalar_type()==at::kFloat && l.scalar_type()==at::kFloat, "m/l must be float");
#     TORCH_CHECK(V_TILE == BLOCK_X, "This kernel requires V_TILE == BLOCK_X (got V_TILE=", V_TILE, ", BLOCK_X=", BLOCK_X, ")");

#     // keep original invariants
#     check_dtype_eq(Q,K,"Q","K");
#     check_dtype_eq(Q,V,"Q","V");
#     check_dtype_eq(Q,dOut,"Q","dOut");

#     const int64_t T64 = Q.size(0);
#     const int64_t H64 = Q.size(1);
#     const int64_t D64 = Q.size(2);
#     const int64_t DV64= V.size(2);
#     const int64_t C64 = cand_i32.size(1);

#     TORCH_CHECK(D64 <= 256, "D must be <= 256 for this kernel");

#     const int T = (int)T64;
#     const int H = (int)H64;
#     const int D = (int)D64;
#     const int DV= (int)DV64;
#     const int C = (int)C64;

#     // ============================
#     // FP32 stability bridge:
#     //   Cast dOut/Q/K/V -> float and run ONLY float kernels.
#     //   Accumulated grads are FP32 buffers (already).
#     //   Then cast back to original dtype for compatibility.
#     //   (When driven by Stage-B FP32, original dtype is float so this is a no-op cast.)
#     // ============================
#     auto dOutf = (dOut.scalar_type() == at::kFloat) ? dOut : dOut.to(at::kFloat);
#     auto Qf    = (Q.scalar_type()    == at::kFloat) ? Q    : Q.to(at::kFloat);
#     auto Kf    = (K.scalar_type()    == at::kFloat) ? K    : K.to(at::kFloat);
#     auto Vf    = (V.scalar_type()    == at::kFloat) ? V    : V.to(at::kFloat);

#     dOutf = dOutf.contiguous();
#     Qf    = Qf.contiguous();
#     Kf    = Kf.contiguous();
#     Vf    = Vf.contiguous();

#     auto dQ_f = torch::zeros({T64*H64, D64},  torch::TensorOptions().dtype(torch::kFloat).device(Q.device()));
#     auto dK_f = torch::zeros({T64*H64, D64},  torch::TensorOptions().dtype(torch::kFloat).device(Q.device()));
#     auto dV_f = torch::zeros({T64*H64, DV64}, torch::TensorOptions().dtype(torch::kFloat).device(Q.device()));

#     const float scale = (float)scale_py;
#     cudaStream_t stream = at::cuda::getCurrentCUDAStream();

#     dim3 grid((unsigned)T, (unsigned)H, 1);
#     dim3 block(BLOCK_X, 1, 1);

#     fused_bwd_mh_kernel<float,float><<<grid, block, 0, stream>>>(
#         (const float*)dOutf.data_ptr(),
#         (const float*)Qf.data_ptr(),
#         (const float*)Kf.data_ptr(),
#         (const float*)Vf.data_ptr(),
#         (const int32_t*)cand_i32.data_ptr(),
#         (const uint8_t*)mask_u8.data_ptr(),
#         (const float*)m.data_ptr(),
#         (const float*)l.data_ptr(),
#         (float*)dQ_f.data_ptr(),
#         (float*)dK_f.data_ptr(),
#         (float*)dV_f.data_ptr(),
#         T,H,D,DV,C, scale, (int)cand_is_per_head
#     );

#     auto err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_backward kernel launch failed: ", cudaGetErrorString(err));

#     // Return grads in the original dtype (compat),
#     // but when Stage-B feeds FP32, these will stay FP32.
#     auto dQ = torch::empty_like(Q);
#     auto dK = torch::empty_like(K);
#     auto dV = torch::empty_like(V);

#     const int threads = 256;
#     auto launch_cast = [&](const torch::Tensor& src_f, torch::Tensor& dst) {
#         const int64_t n = dst.numel();
#         const int blocks = (int)((n + threads - 1) / threads);
#         if (dst.scalar_type() == torch::kFloat) {
#             cast_f32_to_f32<<<blocks, threads, 0, stream>>>(
#                 src_f.data_ptr<float>(), dst.data_ptr<float>(), n);
#         } else if (dst.scalar_type() == torch::kHalf) {
#             cast_f32_to_f16<<<blocks, threads, 0, stream>>>(
#                 src_f.data_ptr<float>(), (at::Half*)dst.data_ptr<at::Half>(), n);
#         } else if (dst.scalar_type() == torch::kBFloat16) {
#             cast_f32_to_bf16<<<blocks, threads, 0, stream>>>(
#                 src_f.data_ptr<float>(), (at::BFloat16*)dst.data_ptr<at::BFloat16>(), n);
#         } else {
#             TORCH_CHECK(false, "Unsupported dtype for cast");
#         }
#     };

#     launch_cast(dQ_f.view({T64, H64, D64}).contiguous(), dQ);
#     launch_cast(dK_f.view({T64, H64, D64}).contiguous(), dK);
#     launch_cast(dV_f.view({T64, H64, DV64}).contiguous(), dV);

#     err = cudaGetLastError();
#     TORCH_CHECK(err == cudaSuccess, "mh_backward cast launch failed: ", cudaGetErrorString(err));

#     return {dQ, dK, dV};
# }

# PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
#     m.def("mh_forward",  &mh_forward,  "Stage-E multi-head fused sparse attention forward (CUDA)");
#     m.def("mh_forward_frozen",  &mh_forward_frozen,  "Stage-E forward that returns frozen cand/mask (CUDA)");
#     m.def("mh_backward", &mh_backward, "Stage-E multi-head fused sparse attention backward (CUDA)");
# }"""
# # ext_name = "stageE_multi_head_fusion_ext"

# CUDA_SRC = CUDA_SRC  # unchanged

# # ---- build-hash (forces rebuild when flags / kernel config / fp32-bridge changes) ----
# _build_sig = (
#     "stageE_multi_head_fusion|"
#     "BLOCK_X=256|"
#     "V_TILE=256|"
#     "cxx17|"
#     "fp32_bridge_wrappers_v2_doout_p_clamps|"
#     "add_forward_frozen_cand_mask_v1"
# )
# _build_hash = hashlib.md5(_build_sig.encode("utf-8")).hexdigest()[:10]

# ext_name = f"stageE_multi_head_fusion_ext_{_build_hash}"

# stageE_multi_head_fusion = load_inline(
#     name=ext_name,
#     cpp_sources="",
#     cuda_sources=CUDA_SRC,
#     functions=None,          # we use PYBIND11_MODULE
#     extra_cuda_cflags=[
#       "-O3",
#       "-DBLOCK_X=256",
#       "-DV_TILE=256",
#       "-DSASA_DEVICE_DEBUG",       # enables SASA_DPRINTF
#       "-DSASA_TRAP_ON_BAD_CAND",   # makes invalid cand trigger trap_if(true)
#     ],
#     extra_cflags=["-O3", "-std=c++17"],
#     with_cuda=True,
#     verbose=False,
# )

# print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))
# print("TORCH_SHOW_CPP_STACKTRACES =", os.environ.get("TORCH_SHOW_CPP_STACKTRACES"))
# print("detect_anomaly =", torch.is_anomaly_enabled())

# mh_forward         = stageE_multi_head_fusion.mh_forward
# mh_forward_frozen  = stageE_multi_head_fusion.mh_forward_frozen
# mh_backward        = stageE_multi_head_fusion.mh_backward


In [ ]:
# import torch, gc

# def _to_device(batch, device):
#     out = {}
#     for k,v in batch.items():
#         out[k] = v.to(device) if torch.is_tensor(v) else v
#     return out

# @torch.no_grad()
# def _snap_lora(model):
#     snap = {}
#     for n,p in model.named_parameters():
#         if p.requires_grad and ("lora_" in n):
#             snap[n] = p.detach().clone()
#     return snap

# @torch.no_grad()
# def _max_lora_delta(model, snap):
#     md = 0.0
#     worst = None
#     for n,p in model.named_parameters():
#         if n in snap:
#             d = (p.detach() - snap[n]).abs().max().item()
#             if d > md:
#                 md = float(d); worst = n
#     return md, worst

# def backward_param_mutation_check(w=128, max_seq=256):
#     device = next(model.parameters()).device
#     model.train()
#     model.zero_grad(set_to_none=True)

#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"][:, :max_seq].contiguous()
#     labels    = batch.get("labels", input_ids)[:, :max_seq].contiguous()
#     attn_mask = batch.get("attention_mask", None)
#     if attn_mask is not None:
#         attn_mask = attn_mask[:, :max_seq].contiguous()

#     set_sasacg_params(model, w=int(w), verbose=False)

#     snap = _snap_lora(model)

#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()

#     with SASACandCacheContext():
#         out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels, use_cache=False)
#         loss = out.loss

#     loss.backward()  # <-- NO opt.step()

#     md, worst = _max_lora_delta(model, snap)
#     print(f"[stomp_check] max |Δ| in LoRA params during backward (should be 0): {md:.3e}  worst={worst}")
#     return md, worst

# backward_param_mutation_check(w=128, max_seq=256)


RuntimeError: Q and dOut dtype must match

In [ ]:
# # ============================================================
# # SMOKE TEST (compatible with your notebook + your Stage-A shapes)
# #   - no dependency on Cell 34/35 being run
# #   - uses your actual model forward path (Stage-A + Stage-E)
# #   - isolates: (1) params not propagating, (2) Stage-A ignoring w,
# #               (3) Stage-E not called / ignoring candidates, (4) true invariance
# # ============================================================
# import torch, math, numpy as np

# # ----------------------------
# # Minimal fallbacks (only if missing in current runtime)
# # ----------------------------
# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _get_budget_defaults():
#     mode  = int(globals().get("BUDGET_SCAFFOLD_MODE", 0))
#     rin   = int(globals().get("BUDGET_RING_INNER", 0))
#     rout  = int(globals().get("BUDGET_RING_OUTER", 0))
#     A     = int(globals().get("SWEEP_NUM_ANCHORS", 0))
#     inc_s = bool(globals().get("SWEEP_INCLUDE_SELF", True))
#     astr  = int(globals().get("BUDGET_ANCHOR_STRATEGY", 0))
#     ap0   = int(globals().get("BUDGET_ANCHOR_PARAM0", 0))
#     return mode, rin, rout, A, inc_s, astr, ap0

# # ----------------------------
# # Hard requirements (fail fast with exact missing names)
# # ----------------------------
# _required = ["model", "val_loader", "build_candidates_cuda", "SASACandCacheContext"]
# _missing = [x for x in _required if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[smoke] Missing required notebook names: {_missing}")

# if "set_sasacg_params" not in globals() or "_get_decoder_layers_any_wrapper" not in globals():
#     raise RuntimeError("[smoke] You need the helper cell that defines _get_decoder_layers_any_wrapper + set_sasacg_params (your Cell 33).")

# # ----------------------------
# # Helper: find SASA layers
# # ----------------------------
# def _sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for i, layer in enumerate(layers):
#         if getattr(layer, "self_attn", None) is None:
#             continue
#         if layer.self_attn.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((i, layer))
#     return layers, sasa

# # ----------------------------
# # Helper: Stage-A budget probe (minimal; matches your Stage-A outputs)
# #   Stage-A returns ok_u8 shape [T, Cmax] where T = B*N
# # ----------------------------
# @torch.no_grad()
# def _budget_probe_from_batch(batch, *, w, scaffold_mode, ring_inner, ring_outer, num_anchors,
#                             include_self, anchor_strategy, anchor_param0):
#     device = next(model.parameters()).device
#     if "attention_mask" in batch and batch["attention_mask"] is not None:
#         attn_mask = batch["attention_mask"].to(device)
#     else:
#         attn_mask = torch.ones_like(batch["input_ids"], device=device)

#     # ensure [B,N]
#     if attn_mask.dim() != 2:
#         attn_mask = attn_mask.view(attn_mask.shape[0], -1)

#     B, N = int(attn_mask.shape[0]), int(attn_mask.shape[1])

#     cand_i32, ok_u8, valid_T = build_candidates_cuda(
#         attn_mask,
#         int(w),
#         int(ring_inner),
#         int(ring_outer),
#         int(scaffold_mode),
#         int(num_anchors),
#         bool(include_self),
#         int(anchor_strategy),
#         int(anchor_param0),
#     )

#     Cmax = int(ok_u8.shape[-1])
#     T = B * N
#     assert int(ok_u8.shape[0]) == T, f"[smoke] ok_u8.shape[0]={ok_u8.shape[0]} but B*N={T}"

#     ok = ok_u8.view(B, N, Cmax).to(torch.float32)
#     k = ok.sum(dim=-1)  # [B,N] keys per query

#     tv = attn_mask.to(torch.bool)
#     kval = k[tv]
#     if kval.numel() == 0:
#         return dict(Cmax=Cmax, keys_mean=float("nan"), keys_p95=float("nan"), density=float("nan"))

#     kval_cpu = kval.detach().float().cpu()
#     keys_mean = float(kval_cpu.mean().item())
#     keys_p95  = float(torch.quantile(kval_cpu, 0.95).item())
#     density   = float(keys_mean / float(N))
#     return dict(Cmax=Cmax, keys_mean=keys_mean, keys_p95=keys_p95, density=density)

# # ----------------------------
# # Helper: eval forward fingerprint (single batch, deterministic)
# # ----------------------------
# @torch.no_grad()
# def _eval_loss_and_fp(batch):
#     device = next(model.parameters()).device
#     b = _to_device(batch, device)
#     input_ids = b["input_ids"]
#     attention_mask = b.get("attention_mask", None)
#     labels = b.get("labels", input_ids)

#     model.eval()
#     with SASACandCacheContext():
#         out = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels,
#             use_cache=False,
#             past_key_values=None,
#         )

#     loss = float(out.loss.detach().float().cpu().item())
#     lg = out.logits.detach()
#     sl = lg[:, :min(4, lg.shape[1]), :min(32, lg.shape[2])].float().cpu()
#     fp = (float(sl.mean().item()), float(sl.abs().mean().item()), float(sl.std(unbiased=False).item()))
#     return loss, fp

# # ----------------------------
# # The smoke test
# # ----------------------------
# def smoke_test_invariance(w1=16, w2=128):
#     mode, rin, rout, A, inc_s, astr, ap0 = _get_budget_defaults()

#     layers, sasa = _sasa_layers(model)
#     print(f"[smoke] SASA layers found: {len(sasa)} / total decoder layers: {len(layers)}")
#     if len(sasa) == 0:
#         raise RuntimeError("[smoke] No MistralSASACGAttention_MH layers found -> patch did not land on active decoder layers.")

#     batch = next(iter(val_loader))

#     # ---- Instrument Stage-A + Stage-E inside the REAL model forward ----
#     stageA_seen = []
#     stageE_calls = 0
#     saved = []

#     def _wrap_stageA(layer_idx, origA):
#         def _A(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self,
#                anchor_strategy=0, anchor_param0=0):
#             cand_i32, ok_u8, valid_T = origA(
#                 attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self,
#                 anchor_strategy, anchor_param0
#             )
#             stageA_seen.append((int(layer_idx), int(w), int(ok_u8.shape[-1])))
#             return cand_i32, ok_u8, valid_T
#         return _A

#     def _wrap_stageE(origE):
#         def _E(Q, K, V, cand_i32, mask_u8, scale, cand_is_per_head):
#             nonlocal stageE_calls
#             stageE_calls += 1
#             return origE(Q, K, V, cand_i32, mask_u8, float(scale), bool(cand_is_per_head))
#         return _E

#     try:
#         for layer_idx, layer in sasa:
#             attn = layer.self_attn
#             saved.append((attn, attn.build_candidates_cuda, attn.mh_fwd))
#             attn.build_candidates_cuda = _wrap_stageA(layer_idx, attn.build_candidates_cuda)
#             attn.mh_fwd = _wrap_stageE(attn.mh_fwd)

#         # ----------------------------
#         # RUN 1 (w1)
#         # ----------------------------
#         print("\n" + "="*80)
#         print(f"[smoke] RUN 1: setting w={w1} (mode={mode}, ring=[{rin}..{rout}], A={A}, include_self={inc_s}, anchor_strategy={astr}, anchor_param0={ap0})")
#         set_sasacg_params(
#             model,
#             w=int(w1),
#             scaffold_mode=int(mode),
#             ring_inner=int(rin),
#             ring_outer=int(rout),
#             num_anchors=int(A),
#             include_self=bool(inc_s),
#             anchor_strategy=int(astr),
#             anchor_param0=int(ap0),
#             verbose=True,
#         )

#         # verify propagation onto an active SASA layer
#         att0 = sasa[0][1].self_attn
#         print(f"[smoke] first SASA layer params: w={int(att0.w)} mode={int(att0.scaffold_mode)} rin={int(att0.ring_inner)} rout={int(att0.ring_outer)} A={int(att0.num_anchors)}")

#         b1 = _budget_probe_from_batch(batch, w=int(w1), scaffold_mode=int(mode), ring_inner=int(rin), ring_outer=int(rout),
#                                       num_anchors=int(A), include_self=bool(inc_s), anchor_strategy=int(astr), anchor_param0=int(ap0))
#         loss1, fp1 = _eval_loss_and_fp(batch)
#         seen1 = stageA_seen[:]
#         calls1 = stageE_calls

#         # reset logs for run2
#         stageA_seen.clear()
#         stageE_calls = 0

#         # ----------------------------
#         # RUN 2 (w2)
#         # ----------------------------
#         print("\n" + "="*80)
#         print(f"[smoke] RUN 2: setting w={w2} (mode={mode}, ring=[{rin}..{rout}], A={A}, include_self={inc_s}, anchor_strategy={astr}, anchor_param0={ap0})")
#         set_sasacg_params(
#             model,
#             w=int(w2),
#             scaffold_mode=int(mode),
#             ring_inner=int(rin),
#             ring_outer=int(rout),
#             num_anchors=int(A),
#             include_self=bool(inc_s),
#             anchor_strategy=int(astr),
#             anchor_param0=int(ap0),
#             verbose=True,
#         )

#         att0 = sasa[0][1].self_attn
#         print(f"[smoke] first SASA layer params: w={int(att0.w)} mode={int(att0.scaffold_mode)} rin={int(att0.ring_inner)} rout={int(att0.ring_outer)} A={int(att0.num_anchors)}")

#         b2 = _budget_probe_from_batch(batch, w=int(w2), scaffold_mode=int(mode), ring_inner=int(rin), ring_outer=int(rout),
#                                       num_anchors=int(A), include_self=bool(inc_s), anchor_strategy=int(astr), anchor_param0=int(ap0))
#         loss2, fp2 = _eval_loss_and_fp(batch)
#         seen2 = stageA_seen[:]
#         calls2 = stageE_calls

#     finally:
#         # restore original callables
#         for attn, origA, origE in saved:
#             attn.build_candidates_cuda = origA
#             attn.mh_fwd = origE

#     # ----------------------------
#     # Report
#     # ----------------------------
#     print("\n" + "="*80)
#     print("[smoke] Stage-A budget probe (same batch):")
#     print(f"  w={w1}: Cmax={b1['Cmax']} keys_mean={b1['keys_mean']:.3f} keys_p95={b1['keys_p95']:.3f} density={b1['density']:.6f}")
#     print(f"  w={w2}: Cmax={b2['Cmax']} keys_mean={b2['keys_mean']:.3f} keys_p95={b2['keys_p95']:.3f} density={b2['density']:.6f}")

#     print("\n[smoke] Model eval outputs (same batch):")
#     print(f"  loss(w={w1})={loss1:.6f} fp={fp1}")
#     print(f"  loss(w={w2})={loss2:.6f} fp={fp2}")
#     print(f"  Δloss={abs(loss2-loss1):.6e}")

#     def _summ(seen):
#         if not seen:
#             return "[]"
#         # show a compact summary: unique w observed + a few samples
#         ws = sorted(list({x[1] for x in seen}))
#         head = seen[:6]
#         return f"unique_w={ws}, head={head}"

#     print("\n[smoke] Stage-A observed inside model forward (layer_idx, w, Cmax):")
#     print(f"  run1: {_summ(seen1)}")
#     print(f"  run2: {_summ(seen2)}")

#     print("\n[smoke] Stage-E call count inside forward:")
#     print(f"  run1 Stage-E calls={calls1}")
#     print(f"  run2 Stage-E calls={calls2}")

#     # ----------------------------
#     # Diagnosis logic (the whole point)
#     # ----------------------------
#     print("\n" + "="*80)
#     print("[smoke] DIAGNOSIS:")
#     if not seen1 or not seen2:
#         print("  - Stage-A was not called inside model forward -> you are NOT on the SASA path (patch not active or dense fallback).")
#         return

#     if (int(w1) not in [x[1] for x in seen1]) or (int(w2) not in [x[1] for x in seen2]):
#         print("  - Stage-A did NOT see the requested w values inside forward -> params are not reaching the active SASA layers.")
#         print("    (This would make your sweep effectively constant-w => constant PPL.)")

#     if (abs(b1["keys_mean"] - b2["keys_mean"]) < 1e-9) and (b1["Cmax"] == b2["Cmax"]):
#         print("  - Stage-A budget does NOT change with w on the same batch -> Stage-A is ignoring w OR attention_mask is constant/degenerate.")
#     else:
#         print("  - Stage-A budget changes with w -> Stage-A is behaving (candidate budget really changes).")

#     if calls1 == 0 or calls2 == 0:
#         print("  - Stage-E was not called -> you are not executing the fused sparse attention path.")
#     else:
#         print("  - Stage-E is being called -> fused sparse attention path is active.")

#     if (abs(loss2-loss1) < 1e-12) and all(abs(a-b) < 1e-12 for a,b in zip(fp1, fp2)):
#         print("  - Outputs are IDENTICAL across w (loss + logits fingerprint).")
#         print("    If Stage-A budget changes, that strongly suggests Stage-E is effectively ignoring cand/mask (or cand/mask behave like dense).")
#     else:
#         print("  - Outputs change across w on the same batch -> invariance is NOT in attention math.")
#         print("    Then constant PPL in your sweep is coming from sweep wiring (e.g., not updating w before eval, reusing stale results, etc.).")

# # run
# smoke_test_invariance(w1=16, w2=128)


In [ ]:
# Clean sort for annotations
df_plot = df.sort_values("keys_per_query_mean").reset_index(drop=True)

# ----------------------------
# Plot 1: Budget–quality (PPL vs measured keys/query) — log PPL
# ----------------------------
plt.figure()

for m in sorted(df_plot["scaffold_mode"].unique()):
    sub = df_plot[df_plot["scaffold_mode"] == m]
    plt.scatter(sub["keys_per_query_mean"], sub["eval_ppl"], marker=_scaffold_marker(m), label=_mode_name(m))

for _, r in df_plot.iterrows():
    plt.annotate(_scaffold_label(r),
                 (r["keys_per_query_mean"], r["eval_ppl"]),
                 fontsize=8)

plt.yscale("log")
plt.xlabel("Measured keys/query (mean)")
plt.ylabel("Perplexity (PPL, log scale)")
plt.title("Scaffold structure sweep — Budget vs Quality")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ----------------------------
# Plot 2: Quality–cost (PPL vs Throughput) — log PPL
# ----------------------------
df_plot2 = df.sort_values("tokens_per_sec").reset_index(drop=True)

plt.figure()

for m in sorted(df_plot2["scaffold_mode"].unique()):
    sub = df_plot2[df_plot2["scaffold_mode"] == m]
    plt.scatter(sub["tokens_per_sec"], sub["eval_ppl"], marker=_scaffold_marker(m), label=_mode_name(m))

for _, r in df_plot2.iterrows():
    plt.annotate(_scaffold_label(r),
                 (r["tokens_per_sec"], r["eval_ppl"]),
                 fontsize=8)

plt.yscale("log")
plt.xlabel("Eval tokens/sec")
plt.ylabel("Perplexity (PPL, log scale)")
plt.title("Scaffold structure sweep — Quality vs Cost")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# # ============================================================
# # REAL BACKWARD NaN TRACER (find first bad SASA CUDA backward)
# #   - wraps Stage-E mh_bwd + Stage-B proj_bwd across ALL SASA layers
# #   - runs ONE real CE backward micro-step
# #   - prints FIRST layer/stage where non-finites appear (inputs or outputs)
# # ============================================================
# import torch, gc, math
# from collections import defaultdict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext", "_get_decoder_layers_any_wrapper"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_trace] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _isfinite_all(x):
#     return bool(torch.isfinite(x).all().item())

# def _nonfinite_count(x):
#     return int((~torch.isfinite(x)).sum().item())

# def _maxabs(x):
#     return float(x.detach().float().abs().max().item()) if x.numel() else 0.0

# def _check_trainables_finite(m, name_contains=None, max_show=30):
#     bad = []
#     total = 0
#     for n,p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         total += 1
#         if not _isfinite_all(p.detach()):
#             bad.append((n, tuple(p.shape), str(p.dtype), _nonfinite_count(p.detach()), _maxabs(p.detach())))
#     if bad:
#         print(f"[nan_trace] ❌ NON-FINITE TRAINABLE PARAMS already: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ trainable params finite: {total} tensors" + (f" (filtered by '{name_contains}')" if name_contains else ""))
#     return True

# def _get_sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for li, l in enumerate(layers):
#         sa = getattr(l, "self_attn", None)
#         if sa is not None and sa.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((li, l, sa))
#     return sasa

# def install_sasa_backward_sentinels(model, stop_on_first=True, verbose=True):
#     """
#     Wrap mh_bwd and proj_bwd for each SASA layer to detect non-finites in:
#       - Stage-E backward outputs: dQ,dK,dV
#       - Stage-B backward outputs: dH, dWq, dWk, dWv
#     Also checks key inputs (dOut, Q/K/V, m/l, dQ/dK/dV, etc.)
#     """
#     sasa = _get_sasa_layers(model)
#     if not sasa:
#         raise RuntimeError("[nan_trace] No SASA layers found.")

#     events = []
#     orig = {"mh_bwd": {}, "proj_bwd": {}}
#     call_counts = defaultdict(int)

#     def _emit(tag, li, msg):
#         s = f"[nan_trace] ❌ {tag} @ layer {li}: {msg}"
#         events.append(s)
#         print(s)
#         if stop_on_first:
#             raise RuntimeError(s)

#     for li, layer, attn in sasa:
#         # ---- wrap Stage-E mh_bwd ----
#         if hasattr(attn, "mh_bwd") and callable(attn.mh_bwd):
#             orig_mh = attn.mh_bwd
#             orig["mh_bwd"][li] = orig_mh

#             def _make_mh_wrap(li, fn):
#                 def mh_bwd_wrap(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#                     call_counts[("mh_bwd", li)] += 1
#                     # check inputs (cheap but decisive)
#                     if torch.is_tensor(dOut) and (not _isfinite_all(dOut)):
#                         _emit("Stage-E INPUT", li, f"dOut nonfinite (bad={_nonfinite_count(dOut)}) maxabs={_maxabs(dOut):.3e}")
#                     for nm, t in [("Q",Q),("K",K),("V",V),("m",m),("l",l)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head)

#                     # out should be (dQ,dK,dV) but be robust
#                     if isinstance(out, (tuple, list)):
#                         outs = out
#                     else:
#                         outs = (out,)

#                     # check outputs
#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return mh_bwd_wrap
#             attn.mh_bwd = _make_mh_wrap(li, orig_mh)

#         # ---- wrap Stage-B proj_bwd ----
#         if hasattr(attn, "proj_bwd") and callable(attn.proj_bwd):
#             orig_pb = attn.proj_bwd
#             orig["proj_bwd"][li] = orig_pb

#             def _make_proj_wrap(li, fn):
#                 def proj_bwd_wrap(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t):
#                     call_counts[("proj_bwd", li)] += 1

#                     for nm, t in [("dQ",dQ),("dK",dK),("dV",dV),("H",H_flat),("cos",cos_t),("sin",sin_t)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t)

#                     if isinstance(out, (tuple, list)):
#                         outs = out
#                     else:
#                         outs = (out,)

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return proj_bwd_wrap
#             attn.proj_bwd = _make_proj_wrap(li, orig_pb)

#     if verbose:
#         print(f"[nan_trace] installed sentinels on {len(sasa)} SASA layers "
#               f"({'stop_on_first' if stop_on_first else 'collect_all'}).")
#     return orig, events, call_counts

# def uninstall_sasa_backward_sentinels(model, orig):
#     sasa = _get_sasa_layers(model)
#     for li, layer, attn in sasa:
#         if li in orig.get("mh_bwd", {}):
#             attn.mh_bwd = orig["mh_bwd"][li]
#         if li in orig.get("proj_bwd", {}):
#             attn.proj_bwd = orig["proj_bwd"][li]
#     print("[nan_trace] restored original mh_bwd/proj_bwd")

# def run_one_real_backward_trace(
#     w=128,
#     max_seq=256,
#     anomaly=False,
#     only_lora=True,
# ):
#     device = next(model.parameters()).device
#     model.train()

#     # AMP like your loop
#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     # verify params are sane BEFORE we do anything
#     ok_params = _check_trainables_finite(model, name_contains=("lora_" if only_lora else None))
#     if not ok_params:
#         print("[nan_trace] >>> Your trainables are already non-finite. Reload model / re-init LoRA before debugging backward.")
#         return

#     # batch
#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels    = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     # set SASA params
#     set_sasacg_params(model, w=int(w), verbose=True)

#     # clear grads
#     model.zero_grad(set_to_none=True)

#     # install sentinels
#     orig, events, call_counts = install_sasa_backward_sentinels(model, stop_on_first=True, verbose=True)

#     # anomaly option
#     torch.autograd.set_detect_anomaly(bool(anomaly))

#     try:
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         with SASACandCacheContext():
#             with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
#                 out = model(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     labels=labels,
#                     use_cache=False,
#                     past_key_values=None,
#                 )
#                 loss = out.loss

#         # forward sanity
#         loss_ok = _isfinite_all(loss.detach())
#         logits_ok = _isfinite_all(out.logits.detach())
#         print(f"[nan_trace] forward: loss={float(loss.detach().float().cpu().item()):.6f} loss_ok={loss_ok} | logits_ok={logits_ok} logits_maxabs={_maxabs(out.logits.detach()):.3e}")
#         if not (loss_ok and logits_ok):
#             print("[nan_trace] >>> Forward already non-finite. Backward trace is meaningless until forward is fixed.")
#             return

#         # backward (this is where we expect to catch the first NaN)
#         loss.backward()

#         # if we reach here, no sentinel fired
#         print("[nan_trace] ✅ backward completed without any Stage-B/Stage-E sentinel detecting non-finites.")
#         # still check LoRA grads quickly
#         bad = 0
#         total = 0
#         for n,p in _iter_trainables_named(model):
#             if only_lora and ("lora_" not in n):
#                 continue
#             if p.grad is None:
#                 continue
#             total += 1
#             if not _isfinite_all(p.grad):
#                 bad += 1
#         print(f"[nan_trace] grad check (filtered={'lora_' if only_lora else 'all'}): nonfinite={bad}/{total}")

#     finally:
#         uninstall_sasa_backward_sentinels(model, orig)
#         # restore anomaly setting to default off
#         torch.autograd.set_detect_anomaly(False)

# # ---- RUN ----
# run_one_real_backward_trace(w=128, max_seq=256, anomaly=False, only_lora=True)


In [ ]:
# # ============================================================
# # SMOKE TEST (Notebook-compatible): verify the REAL sweep wiring
# #   - Confirms eval_loss_only actually runs under the requested w
# #   - Confirms Stage-A sees that w (inside forward) and reports Cmax
# #   - Confirms Stage-E (mh_fwd) is hit (sparse path alive)
# #   - Optional: confirms 1 train step changes trainable params
# # ============================================================
# import torch, gc

# # ----------------------------
# # Hard requirements (fail fast with exact missing names)
# # ----------------------------
# _required = [
#     "model", "train_loader", "val_loader",
#     "_get_decoder_layers_any_wrapper", "set_sasacg_params",
#     "eval_loss_only", "train_n_steps",
# ]
# _missing = [x for x in _required if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[smoke_sweep] Missing required names in notebook: {_missing}")

# # _to_device is defined in your Cell 35; if you run this before that cell, define a compatible fallback.
# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _find_first_sasa_layer(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     for li, l in enumerate(layers):
#         if getattr(l, "self_attn", None) is not None and l.self_attn.__class__.__name__ == "MistralSASACGAttention_MH":
#             return li, l, layers
#     return None, None, layers

# @torch.no_grad()
# def _loss_on_fixed_batch(model, batch):
#     device = next(model.parameters()).device
#     b = _to_device(batch, device)
#     input_ids = b["input_ids"]
#     attention_mask = b.get("attention_mask", None)
#     labels = b.get("labels", input_ids)

#     out = model(
#         input_ids=input_ids,
#         attention_mask=attention_mask,
#         labels=labels,
#         use_cache=False,
#         past_key_values=None,
#     )
#     return float(out.loss.detach().float().cpu().item())

# def smoke_check_budget_wiring(w1=16, w2=128, eval_steps=2, do_one_train_step=True):
#     li, layer0, layers = _find_first_sasa_layer(model)
#     if layer0 is None:
#         raise RuntimeError("[smoke_sweep] No SASA layers found (patch did not land on the real decoder layers).")

#     print(f"[smoke_sweep] SASA layers found: {sum(1 for l in layers if l.self_attn.__class__.__name__=='MistralSASACGAttention_MH')} / total={len(layers)}")
#     print(f"[smoke_sweep] Using first SASA layer index={li}")

#     fixed_val_batch   = next(iter(val_loader))
#     fixed_train_batch = next(iter(train_loader))

#     attn0 = layer0.self_attn

#     # --- instrument ONLY the first SASA layer (keeps it light & unambiguous) ---
#     stageA_seen = []  # (w, Cmax)
#     stageE_calls = 0

#     _orig_stageA = attn0.build_candidates_cuda
#     _orig_mh_fwd = attn0.mh_fwd

#     def _stageA_dbg(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self, anchor_strategy=0, anchor_param0=0):
#         cand_i32, ok_u8, valid_T = _orig_stageA(attn_mask_bn, w, ring_inner, ring_outer, mode, num_anchors, include_self, anchor_strategy, anchor_param0)
#         # record a small amount only
#         if len(stageA_seen) < 8:
#             stageA_seen.append((int(w), int(ok_u8.shape[1])))
#         return cand_i32, ok_u8, valid_T

#     def _stageE_dbg(*args, **kwargs):
#         nonlocal stageE_calls
#         stageE_calls += 1
#         return _orig_mh_fwd(*args, **kwargs)

#     attn0.build_candidates_cuda = _stageA_dbg
#     attn0.mh_fwd = _stageE_dbg

#     try:
#         for ww, tag in [(w1, "RUN1"), (w2, "RUN2")]:
#             gc.collect()
#             if torch.cuda.is_available():
#                 torch.cuda.empty_cache()
#                 torch.cuda.synchronize()

#             stageA_seen.clear()
#             stageE_calls = 0

#             print("\n" + "="*90)
#             print(f"[smoke_sweep] {tag}: set w={int(ww)} then evaluate")

#             set_sasacg_params(model, w=int(ww), verbose=True)
#             print(f"[smoke_sweep] first SASA layer now has w={int(attn0.w)} mode={int(getattr(attn0,'scaffold_mode',0))} rin={int(getattr(attn0,'ring_inner',0))} rout={int(getattr(attn0,'ring_outer',0))} A={int(getattr(attn0,'num_anchors',0))}")

#             # (A) fixed-batch loss: ultra-sensitive sanity check
#             loss_fixed = _loss_on_fixed_batch(model, fixed_val_batch)

#             # (B) the real eval loop you use in sweeps
#             if "_reset_cuda_peak" in globals():
#                 _reset_cuda_peak()
#             ev = eval_loss_only(model, val_loader, max_steps=int(eval_steps))

#             print(f"[smoke_sweep] Stage-A observed inside forward (first layer): {stageA_seen if stageA_seen else 'NO CALLS'}")
#             print(f"[smoke_sweep] Stage-E calls observed (first layer): {stageE_calls}")
#             print(f"[smoke_sweep] fixed-batch loss={loss_fixed:.6f} | eval_loss={ev['eval_loss']:.6f} | eval_ppl={ev['eval_ppl']:.3f}")

#         if do_one_train_step:
#             print("\n" + "="*90)
#             print("[smoke_sweep] TRAIN SANITY: 1 real train step should move trainable params")

#             params = [p for p in model.parameters() if p.requires_grad]
#             if not params:
#                 raise RuntimeError("[smoke_sweep] No trainable params found (requires_grad=False everywhere).")

#             with torch.no_grad():
#                 p0 = float(sum(p.detach().float().abs().mean().item() for p in params) / max(1, len(params)))

#             set_sasacg_params(model, w=int(w1), verbose=False)
#             _ = train_n_steps(model, train_loader, steps=1, lr=3e-4, grad_accum=1)

#             with torch.no_grad():
#                 p1 = float(sum(p.detach().float().abs().mean().item() for p in params) / max(1, len(params)))

#             print(f"[smoke_sweep] mean(|trainable_param|): before={p0:.6e} after={p1:.6e} Δ={abs(p1-p0):.6e}")

#     finally:
#         # restore
#         attn0.build_candidates_cuda = _orig_stageA
#         attn0.mh_fwd = _orig_mh_fwd
#         print("\n[smoke_sweep] hooks restored.")

# # Run with two very different budgets
# smoke_check_budget_wiring(w1=16, w2=128, eval_steps=2, do_one_train_step=True)


In [ ]:
# # ============================================================
# # REAL BACKWARD NaN TRACER (find first bad SASA CUDA backward)
# #   - wraps Stage-E mh_bwd + Stage-B proj_bwd across ALL SASA layers
# #   - runs ONE real CE backward micro-step
# #   - prints FIRST layer/stage where non-finites appear (inputs or outputs)
# #   - PATCHES ADDED:
# #       (1) try/except so autocast OFF run executes even if ON fails
# #       (2) print Stage-B output shapes
# #       (3) print dtype+maxabs for dQ and H_flat at layer 31
# #       (4) TRUE FP32 discriminator: optionally cast proj_bwd inputs to float32
# #       (5) STOMP CHECK: snapshot selected weights before/after forward and backward
# # ============================================================
# import torch, gc, math
# from collections import defaultdict, OrderedDict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext", "_get_decoder_layers_any_wrapper"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_trace] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _isfinite_all(x):
#     return bool(torch.isfinite(x).all().item())

# def _nonfinite_count(x):
#     return int((~torch.isfinite(x)).sum().item())

# def _maxabs(x):
#     return float(x.detach().float().abs().max().item()) if x.numel() else 0.0

# def _check_trainables_finite(m, name_contains=None, max_show=30):
#     bad = []
#     total = 0
#     for n,p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         total += 1
#         if not _isfinite_all(p.detach()):
#             bad.append((n, tuple(p.shape), str(p.dtype), _nonfinite_count(p.detach()), _maxabs(p.detach())))
#     if bad:
#         print(f"[nan_trace] ❌ NON-FINITE TRAINABLE PARAMS already: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ trainable params finite: {total} tensors" + (f" (filtered by '{name_contains}')" if name_contains else ""))
#     return True

# def _get_sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for li, l in enumerate(layers):
#         sa = getattr(l, "self_attn", None)
#         if sa is not None and sa.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((li, l, sa))
#     return sasa

# # ----------------------------
# # STOMP helpers
# # ----------------------------
# def _safe_getattr_chain(obj, chain, default=None):
#     cur = obj
#     for k in chain:
#         if not hasattr(cur, k):
#             return default
#         cur = getattr(cur, k)
#     return cur

# def _select_stomp_tensors(model, layer_ids=(31, 30, 0)):
#     """
#     Pick a few weights that must NEVER change during forward/backward (before opt.step()).
#     Returns OrderedDict[name -> tensor_ref].
#     """
#     out = OrderedDict()
#     layers = _get_decoder_layers_any_wrapper(model)

#     # SASA q_proj weights at selected layers (if present)
#     for li in layer_ids:
#         if li < 0 or li >= len(layers):
#             continue
#         l = layers[li]
#         sa = getattr(l, "self_attn", None)
#         if sa is None:
#             continue
#         old = getattr(sa, "old", None)
#         if old is None:
#             continue
#         q_proj = getattr(old, "q_proj", None)
#         if q_proj is not None and hasattr(q_proj, "weight") and torch.is_tensor(q_proj.weight):
#             out[f"layers.{li}.self_attn.old.q_proj.weight"] = q_proj.weight

#         # also grab something unrelated-ish within same layer
#         k_proj = getattr(old, "k_proj", None)
#         if k_proj is not None and hasattr(k_proj, "weight") and torch.is_tensor(k_proj.weight):
#             out[f"layers.{li}.self_attn.old.k_proj.weight"] = k_proj.weight

#     # Embedding + lm_head if they exist
#     emb = None
#     if hasattr(model, "get_input_embeddings"):
#         emb = model.get_input_embeddings()
#     if emb is not None and hasattr(emb, "weight") and torch.is_tensor(emb.weight):
#         out["input_embeddings.weight"] = emb.weight

#     lm_head = _safe_getattr_chain(model, ["lm_head"], default=None)
#     if lm_head is not None and hasattr(lm_head, "weight") and torch.is_tensor(lm_head.weight):
#         out["lm_head.weight"] = lm_head.weight

#     return out

# def _snapshot_selected(tensor_map):
#     snap = {}
#     for name, t in tensor_map.items():
#         try:
#             snap[name] = t.detach().clone()
#         except Exception:
#             pass
#     return snap

# def _report_stomp(before_snap, tensor_map, tag, max_show=20):
#     """
#     Compare current tensor values to snapshots and report changes.
#     """
#     changed = []
#     for name, t in tensor_map.items():
#         if name not in before_snap:
#             continue
#         try:
#             cur = t.detach()
#             ref = before_snap[name]
#             if cur.shape != ref.shape:
#                 changed.append((name, "shape_changed", tuple(ref.shape), tuple(cur.shape)))
#                 continue
#             # exact compare first (fast). If not exact, quantify.
#             if not torch.equal(cur, ref):
#                 d = (cur.float() - ref.float())
#                 changed.append((name, float(d.abs().max().item()), float(d.abs().mean().item()), str(cur.dtype)))
#         except Exception:
#             changed.append((name, "compare_failed"))
#     if changed:
#         print(f"[nan_trace] ❌ STOMP DETECTED ({tag}): {len(changed)} tensors changed (showing up to {max_show})")
#         for row in changed[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ stomp check OK ({tag}): no tracked weights changed")
#     return True

# def install_sasa_backward_sentinels(
#     model,
#     stop_on_first=True,
#     verbose=True,
#     force_fp32_proj_bwd=False,       # PATCH: true FP32 discriminator
#     print_layer31_input_stats=True,  # keep your existing layer-31 stats
# ):
#     """
#     Wrap mh_bwd and proj_bwd for each SASA layer to detect non-finites in:
#       - Stage-E backward outputs: dQ,dK,dV
#       - Stage-B backward outputs: dH, dWq, dWk, dWv
#     Also checks key inputs (dOut, Q/K/V, m/l, dQ/dK/dV, etc.)
#     """
#     sasa = _get_sasa_layers(model)
#     if not sasa:
#         raise RuntimeError("[nan_trace] No SASA layers found.")

#     events = []
#     orig = {"mh_bwd": {}, "proj_bwd": {}}
#     call_counts = defaultdict(int)

#     def _emit(tag, li, msg):
#         s = f"[nan_trace] ❌ {tag} @ layer {li}: {msg}"
#         events.append(s)
#         print(s)
#         if stop_on_first:
#             raise RuntimeError(s)

#     for li, layer, attn in sasa:
#         # ---- wrap Stage-E mh_bwd ----
#         if hasattr(attn, "mh_bwd") and callable(attn.mh_bwd):
#             orig_mh = attn.mh_bwd
#             orig["mh_bwd"][li] = orig_mh

#             def _make_mh_wrap(li, fn):
#                 def mh_bwd_wrap(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#                     call_counts[("mh_bwd", li)] += 1
#                     if torch.is_tensor(dOut) and (not _isfinite_all(dOut)):
#                         _emit("Stage-E INPUT", li, f"dOut nonfinite (bad={_nonfinite_count(dOut)}) maxabs={_maxabs(dOut):.3e}")
#                     for nm, t in [("Q",Q),("K",K),("V",V),("m",m),("l",l)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head)

#                     outs = out if isinstance(out, (tuple, list)) else (out,)
#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return mh_bwd_wrap
#             attn.mh_bwd = _make_mh_wrap(li, orig_mh)

#         # ---- wrap Stage-B proj_bwd ----
#         if hasattr(attn, "proj_bwd") and callable(attn.proj_bwd):
#             orig_pb = attn.proj_bwd
#             orig["proj_bwd"][li] = orig_pb

#             def _make_proj_wrap(li, fn):
#                 def proj_bwd_wrap(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t):
#                     call_counts[("proj_bwd", li)] += 1

#                     # PATCH: print dtype + maxabs of dQ and H_flat for layer 31
#                     if print_layer31_input_stats and int(li) == 31:
#                         try:
#                             dq_dtype = str(getattr(dQ, "dtype", None))
#                             h_dtype  = str(getattr(H_flat, "dtype", None))
#                             dq_ma = _maxabs(dQ) if torch.is_tensor(dQ) else float("nan")
#                             h_ma  = _maxabs(H_flat) if torch.is_tensor(H_flat) else float("nan")
#                             print(f"[nan_trace] Stage-B INPUT STATS @ layer {li}: dQ dtype={dq_dtype} maxabs={dq_ma:.3e} | H dtype={h_dtype} maxabs={h_ma:.3e}")
#                         except Exception:
#                             pass

#                     # input finiteness checks
#                     for nm, t in [("dQ",dQ),("dK",dK),("dV",dV),("H",H_flat),("cos",cos_t),("sin",sin_t)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     # PATCH: TRUE FP32 discriminator (attempt to call proj_bwd in FP32)
#                     if bool(force_fp32_proj_bwd):
#                         try:
#                             dQ_     = dQ.contiguous().float()
#                             dK_     = dK.contiguous().float()
#                             dV_     = dV.contiguous().float()
#                             H_flat_ = H_flat.contiguous().float()
#                             Wq_     = Wq.contiguous().float()
#                             Wk_     = Wk_eff.contiguous().float()
#                             Wv_     = Wv_eff.contiguous().float()
#                             cos_    = cos_t.contiguous().float()
#                             sin_    = sin_t.contiguous().float()
#                             out = fn(dQ_, dK_, dV_, H_flat_, Wq_, Wk_, Wv_, cos_, sin_)
#                         except Exception as e:
#                             print(f"[nan_trace] Stage-B FP32 call failed @ layer {li}: {type(e).__name__}: {e}")
#                             raise
#                     else:
#                         out = fn(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t)

#                     outs = out if isinstance(out, (tuple, list)) else (out,)

#                     # PATCH: print output shapes BEFORE finiteness checks
#                     try:
#                         shapes = []
#                         for t in outs:
#                             if torch.is_tensor(t):
#                                 shapes.append(tuple(t.shape))
#                             else:
#                                 shapes.append(type(t).__name__)
#                         print(f"[nan_trace] Stage-B OUTPUT SHAPES @ layer {li}: {shapes}")
#                     except Exception:
#                         pass

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return proj_bwd_wrap
#             attn.proj_bwd = _make_proj_wrap(li, orig_pb)

#     if verbose:
#         print(f"[nan_trace] installed sentinels on {len(sasa)} SASA layers "
#               f"({'stop_on_first' if stop_on_first else 'collect_all'}). "
#               f"force_fp32_proj_bwd={bool(force_fp32_proj_bwd)}")
#     return orig, events, call_counts

# def uninstall_sasa_backward_sentinels(model, orig):
#     sasa = _get_sasa_layers(model)
#     for li, layer, attn in sasa:
#         if li in orig.get("mh_bwd", {}):
#             attn.mh_bwd = orig["mh_bwd"][li]
#         if li in orig.get("proj_bwd", {}):
#             attn.proj_bwd = orig["proj_bwd"][li]
#     print("[nan_trace] restored original mh_bwd/proj_bwd")

# def run_one_real_backward_trace(
#     w=128,
#     max_seq=256,
#     anomaly=False,
#     only_lora=True,
#     autocast_on=True,
#     force_fp32_proj_bwd=False,  # PATCH: true FP32 discriminator
#     stomp_check=True,           # PATCH: stomp check
# ):
#     device = next(model.parameters()).device
#     model.train()

#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     # verify params are sane BEFORE we do anything
#     ok_params = _check_trainables_finite(model, name_contains=("lora_" if only_lora else None))
#     if not ok_params:
#         print("[nan_trace] >>> Your trainables are already non-finite. Reload model / re-init LoRA before debugging backward.")
#         return

#     # batch
#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels    = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     # set SASA params
#     set_sasacg_params(model, w=int(w), verbose=True)

#     # clear grads
#     model.zero_grad(set_to_none=True)

#     # STOMP: select + snapshot tracked weights
#     tracked = _select_stomp_tensors(model, layer_ids=(31, 30, 0))
#     snap0 = _snapshot_selected(tracked) if stomp_check else None

#     # install sentinels
#     orig, events, call_counts = install_sasa_backward_sentinels(
#         model,
#         stop_on_first=True,
#         verbose=True,
#         force_fp32_proj_bwd=bool(force_fp32_proj_bwd),
#         print_layer31_input_stats=True,
#     )

#     torch.autograd.set_detect_anomaly(bool(anomaly))

#     err = None
#     try:
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         with SASACandCacheContext():
#             with torch.autocast(
#                 device_type="cuda",
#                 dtype=amp_dtype,
#                 enabled=(torch.cuda.is_available() and bool(autocast_on)),
#             ):
#                 out = model(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     labels=labels,
#                     use_cache=False,
#                     past_key_values=None,
#                 )
#                 loss = out.loss

#         # STOMP: did forward mutate weights?
#         if stomp_check:
#             _report_stomp(snap0, tracked, tag="after_forward")

#         # forward sanity
#         loss_ok = _isfinite_all(loss.detach())
#         logits_ok = _isfinite_all(out.logits.detach())
#         print(f"[nan_trace] forward: loss={float(loss.detach().float().cpu().item()):.6f} loss_ok={loss_ok} | logits_ok={logits_ok} logits_maxabs={_maxabs(out.logits.detach()):.3e}")
#         if not (loss_ok and logits_ok):
#             print("[nan_trace] >>> Forward already non-finite. Backward trace is meaningless until forward is fixed.")
#             return

#         # backward
#         loss.backward()

#         print("[nan_trace] ✅ backward completed without any Stage-B/Stage-E sentinel detecting non-finites.")
#         bad = 0
#         total = 0
#         for n,p in _iter_trainables_named(model):
#             if only_lora and ("lora_" not in n):
#                 continue
#             if p.grad is None:
#                 continue
#             total += 1
#             if not _isfinite_all(p.grad):
#                 bad += 1
#         print(f"[nan_trace] grad check (filtered={'lora_' if only_lora else 'all'}): nonfinite={bad}/{total}")

#     except Exception as e:
#         err = e
#         raise
#     finally:
#         # STOMP: did backward mutate weights? (even if backward crashed)
#         if stomp_check and snap0 is not None:
#             _report_stomp(snap0, tracked, tag="after_backward_or_crash")

#         uninstall_sasa_backward_sentinels(model, orig)
#         torch.autograd.set_detect_anomaly(False)

# # ============================================================
# # RUN MATRIX
# #   A) baseline autocast ON (real training)
# #   B) baseline autocast OFF (still likely fp16 inside model)
# #   C) TRUE FP32 discriminator: force proj_bwd inputs to float32
# # ============================================================

# def _run_case(name, **kwargs):
#     print("\n" + "="*90)
#     print(f"[nan_trace] RUN CASE: {name} | {kwargs}")
#     print("="*90)
#     try:
#         run_one_real_backward_trace(**kwargs)
#     except Exception as e:
#         print(f"[nan_trace] ({name}) stopped with exception: {type(e).__name__}: {e}")

# # A) autocast ON
# _run_case(
#     "autocast_ON",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=True,
#     force_fp32_proj_bwd=False,
#     stomp_check=True,
# )

# # B) autocast OFF
# _run_case(
#     "autocast_OFF",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=False,
#     stomp_check=True,
# )

# # C) TRUE FP32 proj_bwd discriminator (autocast OFF + cast proj_bwd inputs to FP32)
# _run_case(
#     "proj_bwd_force_FP32_inputs",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=True,
#     stomp_check=True,
# )


In [ ]:
# # ============================================================
# # REAL BACKWARD NaN TRACER (find first bad SASA CUDA backward)
# #   - wraps Stage-E mh_bwd + Stage-B proj_bwd across ALL SASA layers
# #   - runs ONE real CE backward micro-step
# #   - prints FIRST layer/stage where non-finites appear (inputs or outputs)
# #
# # NEW PATCHES (on top of your current version):
# #   (A) Stage-B overflow proof @ layer 30:
# #       - prints fp16 overflow bound ~ S*max|H|*max|dQ|
# #       - computes small fp32 64x64 sub-block of dWq = H^T dQ
# #   (B) Stage-E pinpoint @ layer 29:
# #       - prints keys/query stats from cand_i32/mask_u8
# #       - prints l stats (min, count l<=0, count l<eps)
# #       - when Stage-E output has nonfinites, prints first few (row,col) with l[row] and keys[row]
# #   (C) Debug-only clamp for l: clamp_l_eps (A/B test for "l==0" / denom issues)
# #   (D) Loss-scale sweep under Stage-B FP32 mode
# # ============================================================
# import torch, gc, math
# from collections import defaultdict, OrderedDict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext", "_get_decoder_layers_any_wrapper"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_trace] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _isfinite_all(x):
#     return bool(torch.isfinite(x).all().item())

# def _nonfinite_count(x):
#     return int((~torch.isfinite(x)).sum().item())

# def _maxabs(x):
#     return float(x.detach().float().abs().max().item()) if (torch.is_tensor(x) and x.numel()) else 0.0

# def _check_trainables_finite(m, name_contains=None, max_show=30):
#     bad = []
#     total = 0
#     for n,p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         total += 1
#         if not _isfinite_all(p.detach()):
#             bad.append((n, tuple(p.shape), str(p.dtype), _nonfinite_count(p.detach()), _maxabs(p.detach())))
#     if bad:
#         print(f"[nan_trace] ❌ NON-FINITE TRAINABLE PARAMS already: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ trainable params finite: {total} tensors" + (f" (filtered by '{name_contains}')" if name_contains else ""))
#     return True

# def _get_sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for li, l in enumerate(layers):
#         sa = getattr(l, "self_attn", None)
#         if sa is not None and sa.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((li, l, sa))
#     return sasa

# # ----------------------------
# # STOMP helpers
# # ----------------------------
# def _safe_getattr_chain(obj, chain, default=None):
#     cur = obj
#     for k in chain:
#         if not hasattr(cur, k):
#             return default
#         cur = getattr(cur, k)
#     return cur

# def _select_stomp_tensors(model, layer_ids=(31, 30, 0)):
#     out = OrderedDict()
#     layers = _get_decoder_layers_any_wrapper(model)

#     for li in layer_ids:
#         if li < 0 or li >= len(layers):
#             continue
#         l = layers[li]
#         sa = getattr(l, "self_attn", None)
#         if sa is None:
#             continue
#         old = getattr(sa, "old", None)
#         if old is None:
#             continue

#         q_proj = getattr(old, "q_proj", None)
#         if q_proj is not None and hasattr(q_proj, "weight") and torch.is_tensor(q_proj.weight):
#             out[f"layers.{li}.self_attn.old.q_proj.weight"] = q_proj.weight

#         k_proj = getattr(old, "k_proj", None)
#         if k_proj is not None and hasattr(k_proj, "weight") and torch.is_tensor(k_proj.weight):
#             out[f"layers.{li}.self_attn.old.k_proj.weight"] = k_proj.weight

#     emb = None
#     if hasattr(model, "get_input_embeddings"):
#         emb = model.get_input_embeddings()
#     if emb is not None and hasattr(emb, "weight") and torch.is_tensor(emb.weight):
#         out["input_embeddings.weight"] = emb.weight

#     lm_head = _safe_getattr_chain(model, ["lm_head"], default=None)
#     if lm_head is not None and hasattr(lm_head, "weight") and torch.is_tensor(lm_head.weight):
#         out["lm_head.weight"] = lm_head.weight

#     return out

# def _snapshot_selected(tensor_map):
#     snap = {}
#     for name, t in tensor_map.items():
#         try:
#             snap[name] = t.detach().clone()
#         except Exception:
#             pass
#     return snap

# def _report_stomp(before_snap, tensor_map, tag, max_show=20):
#     changed = []
#     for name, t in tensor_map.items():
#         if name not in before_snap:
#             continue
#         try:
#             cur = t.detach()
#             ref = before_snap[name]
#             if cur.shape != ref.shape:
#                 changed.append((name, "shape_changed", tuple(ref.shape), tuple(cur.shape)))
#                 continue
#             if not torch.equal(cur, ref):
#                 d = (cur.float() - ref.float())
#                 changed.append((name, float(d.abs().max().item()), float(d.abs().mean().item()), str(cur.dtype)))
#         except Exception:
#             changed.append((name, "compare_failed"))
#     if changed:
#         print(f"[nan_trace] ❌ STOMP DETECTED ({tag}): {len(changed)} tensors changed (showing up to {max_show})")
#         for row in changed[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ stomp check OK ({tag}): no tracked weights changed")
#     return True

# # ----------------------------
# # Candidate + l diagnostics (Stage-E)
# # ----------------------------
# def _keys_per_query(cand_i32, mask_u8):
#     """
#     Best-effort keys/query from (cand_i32, mask_u8).
#     Works for typical [S, C] layouts.
#     """
#     if not torch.is_tensor(cand_i32):
#         return None
#     if cand_i32.ndim != 2:
#         return None
#     if torch.is_tensor(mask_u8) and mask_u8.ndim == 2 and mask_u8.shape == cand_i32.shape:
#         valid = (mask_u8 != 0)
#     else:
#         # common convention: invalid candidates are negative
#         valid = (cand_i32 >= 0)
#     return valid.sum(dim=1)

# def install_sasa_backward_sentinels(
#     model,
#     stop_on_first=True,
#     verbose=True,
#     force_fp32_proj_bwd=False,
#     print_layer31_input_stats=True,
#     print_layer29_input_stats=True,
#     clamp_l_eps=None,              # DEBUG A/B: clamp l to eps before Stage-E bwd
#     stageE_debug_rows=5,           # number of nonfinite indices to print
#     stageB_overflow_probe=True,    # enable Stage-B overflow proof at layer 30
# ):
#     sasa = _get_sasa_layers(model)
#     if not sasa:
#         raise RuntimeError("[nan_trace] No SASA layers found.")

#     events = []
#     orig = {"mh_bwd": {}, "proj_bwd": {}}
#     call_counts = defaultdict(int)

#     def _emit(tag, li, msg):
#         s = f"[nan_trace] ❌ {tag} @ layer {li}: {msg}"
#         events.append(s)
#         print(s)
#         if stop_on_first:
#             raise RuntimeError(s)

#     for li, layer, attn in sasa:
#         # ---- wrap Stage-E mh_bwd ----
#         if hasattr(attn, "mh_bwd") and callable(attn.mh_bwd):
#             orig_mh = attn.mh_bwd
#             orig["mh_bwd"][li] = orig_mh

#             def _make_mh_wrap(li, fn):
#                 def mh_bwd_wrap(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#                     call_counts[("mh_bwd", li)] += 1

#                     # layer-29: print candidate stats + l stats
#                     kpq = None
#                     if print_layer29_input_stats and int(li) == 29:
#                         try:
#                             kpq = _keys_per_query(cand_i32, mask_u8)
#                             if kpq is not None:
#                                 k_min = int(kpq.min().item())
#                                 k_max = int(kpq.max().item())
#                                 k_mean = float(kpq.float().mean().item())
#                                 k_zeros = int((kpq == 0).sum().item())
#                                 print(f"[nan_trace] Stage-E CAND STATS @ layer {li}: keys/query min={k_min} mean={k_mean:.2f} max={k_max} zeros={k_zeros}/{int(kpq.numel())}")
#                             l_min = float(l.detach().float().min().item()) if torch.is_tensor(l) else float("nan")
#                             l_max = float(l.detach().float().max().item()) if torch.is_tensor(l) else float("nan")
#                             l_le0 = int((l <= 0).sum().item()) if torch.is_tensor(l) else -1
#                             l_lt_eps = int((l < 1e-12).sum().item()) if torch.is_tensor(l) else -1
#                             print(
#                                 f"[nan_trace] Stage-E INPUT STATS @ layer {li}: "
#                                 f"dOut {getattr(dOut,'dtype',None)} maxabs={_maxabs(dOut):.3e} | "
#                                 f"Q {getattr(Q,'dtype',None)} maxabs={_maxabs(Q):.3e} | "
#                                 f"K {getattr(K,'dtype',None)} maxabs={_maxabs(K):.3e} | "
#                                 f"V {getattr(V,'dtype',None)} maxabs={_maxabs(V):.3e} | "
#                                 f"m {getattr(m,'dtype',None)} maxabs={_maxabs(m):.3e} | "
#                                 f"l {getattr(l,'dtype',None)} min={l_min:.3e} max={l_max:.3e} (<=0:{l_le0}, <1e-12:{l_lt_eps}) | "
#                                 f"scale={float(scale) if not torch.is_tensor(scale) else float(scale.detach().float().cpu().item())}"
#                             )
#                         except Exception:
#                             pass

#                     # finiteness checks on inputs
#                     if torch.is_tensor(dOut) and (not _isfinite_all(dOut)):
#                         _emit("Stage-E INPUT", li, f"dOut nonfinite (bad={_nonfinite_count(dOut)}) maxabs={_maxabs(dOut):.3e}")
#                     for nm, t in [("Q",Q),("K",K),("V",V),("m",m),("l",l)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     # DEBUG A/B: clamp l to eps (only changes what you pass to kernel)
#                     l_use = l
#                     if clamp_l_eps is not None and torch.is_tensor(l_use):
#                         eps = float(clamp_l_eps)
#                         try:
#                             n_clamped = int((l_use < eps).sum().item())
#                             if int(li) == 29 and n_clamped > 0:
#                                 print(f"[nan_trace] Stage-E DEBUG @ layer {li}: clamping l to eps={eps:.1e} for {n_clamped} entries")
#                             l_use = l_use.clamp_min(eps)
#                         except Exception:
#                             pass

#                     out = fn(dOut, Q, K, V, cand_i32, mask_u8, m, l_use, scale, cand_is_per_head)
#                     outs = out if isinstance(out, (tuple, list)) else (out,)

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             # print a few concrete offender indices for layer 29
#                             if int(li) == 29:
#                                 try:
#                                     bad_idx = torch.nonzero(~torch.isfinite(t), as_tuple=False)
#                                     if bad_idx.numel() > 0:
#                                         print(f"[nan_trace] Stage-E OUTPUT NONFINITE SAMPLE @ layer {li} output[{j}] (showing up to {stageE_debug_rows})")
#                                         for r in bad_idx[:stageE_debug_rows]:
#                                             rr = int(r[0].item()) if r.numel() >= 1 else -1
#                                             cc = int(r[1].item()) if r.numel() >= 2 else -1
#                                             lv = float(l_use[rr].detach().float().item()) if (torch.is_tensor(l_use) and l_use.ndim == 1 and rr >= 0 and rr < l_use.numel()) else float("nan")
#                                             kv = int(kpq[rr].item()) if (kpq is not None and rr >= 0 and rr < kpq.numel()) else -1
#                                             print(f"   idx=({rr},{cc}) l={lv:.3e} keys={kv}")
#                                 except Exception:
#                                     pass
#                             _emit("Stage-E OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return mh_bwd_wrap
#             attn.mh_bwd = _make_mh_wrap(li, orig_mh)

#         # ---- wrap Stage-B proj_bwd ----
#         if hasattr(attn, "proj_bwd") and callable(attn.proj_bwd):
#             orig_pb = attn.proj_bwd
#             orig["proj_bwd"][li] = orig_pb

#             def _make_proj_wrap(li, fn):
#                 def proj_bwd_wrap(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t):
#                     call_counts[("proj_bwd", li)] += 1

#                     if print_layer31_input_stats and int(li) == 31:
#                         try:
#                             dq_dtype = str(getattr(dQ, "dtype", None))
#                             h_dtype  = str(getattr(H_flat, "dtype", None))
#                             print(f"[nan_trace] Stage-B INPUT STATS @ layer {li}: dQ dtype={dq_dtype} maxabs={_maxabs(dQ):.3e} | H dtype={h_dtype} maxabs={_maxabs(H_flat):.3e}")
#                         except Exception:
#                             pass

#                     for nm, t in [("dQ",dQ),("dK",dK),("dV",dV),("H",H_flat),("cos",cos_t),("sin",sin_t)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     # Stage-B overflow proof @ layer 30 when running baseline (fp16 path)
#                     if stageB_overflow_probe and (int(li) == 30) and (not bool(force_fp32_proj_bwd)):
#                         try:
#                             S = int(H_flat.shape[0])
#                             h_ma = _maxabs(H_flat)
#                             dq_ma = _maxabs(dQ)
#                             bound = float(S) * float(h_ma) * float(dq_ma)
#                             print(f"[nan_trace] Stage-B OVERFLOW PROBE @ layer {li}: S={S} max|H|={h_ma:.3e} max|dQ|={dq_ma:.3e} => bound~{bound:.3e} (fp16 max≈6.55e4)")
#                             # fp32 64x64 sub-block of dWq = H^T dQ
#                             D = int(Wq.shape[0])
#                             step = max(1, D // 64)
#                             cols = torch.arange(0, D, step, device=H_flat.device)[:64]
#                             sub = (H_flat[:, cols].float().T @ dQ[:, cols].float())
#                             print(f"[nan_trace] Stage-B dWq FP32 sub(64x64) maxabs={float(sub.abs().max().item()):.3e}")
#                         except Exception:
#                             pass

#                     # Stage-B FP32 discriminator
#                     if bool(force_fp32_proj_bwd):
#                         dQ_     = dQ.contiguous().float()
#                         dK_     = dK.contiguous().float()
#                         dV_     = dV.contiguous().float()
#                         H_flat_ = H_flat.contiguous().float()
#                         Wq_     = Wq.contiguous().float()
#                         Wk_     = Wk_eff.contiguous().float()
#                         Wv_     = Wv_eff.contiguous().float()
#                         cos_    = cos_t.contiguous().float()
#                         sin_    = sin_t.contiguous().float()
#                         out = fn(dQ_, dK_, dV_, H_flat_, Wq_, Wk_, Wv_, cos_, sin_)
#                     else:
#                         out = fn(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t)

#                     outs = out if isinstance(out, (tuple, list)) else (out,)

#                     try:
#                         shapes = []
#                         maxabs = []
#                         for t in outs:
#                             if torch.is_tensor(t):
#                                 shapes.append(tuple(t.shape))
#                                 maxabs.append(_maxabs(t))
#                             else:
#                                 shapes.append(type(t).__name__)
#                                 maxabs.append(float("nan"))
#                         print(f"[nan_trace] Stage-B OUTPUT SHAPES @ layer {li}: {shapes}")
#                         print(f"[nan_trace] Stage-B OUTPUT MAXABS  @ layer {li}: {[f'{x:.3e}' for x in maxabs]}")
#                     except Exception:
#                         pass

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return proj_bwd_wrap
#             attn.proj_bwd = _make_proj_wrap(li, orig_pb)

#     if verbose:
#         print(
#             f"[nan_trace] installed sentinels on {len(sasa)} SASA layers "
#             f"({'stop_on_first' if stop_on_first else 'collect_all'}). "
#             f"force_fp32_proj_bwd={bool(force_fp32_proj_bwd)} clamp_l_eps={clamp_l_eps}"
#         )
#     return orig, events, call_counts

# def uninstall_sasa_backward_sentinels(model, orig):
#     sasa = _get_sasa_layers(model)
#     for li, layer, attn in sasa:
#         if li in orig.get("mh_bwd", {}):
#             attn.mh_bwd = orig["mh_bwd"][li]
#         if li in orig.get("proj_bwd", {}):
#             attn.proj_bwd = orig["proj_bwd"][li]
#     print("[nan_trace] restored original mh_bwd/proj_bwd")

# def run_one_real_backward_trace(
#     w=128,
#     max_seq=256,
#     anomaly=False,
#     only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=False,
#     loss_scale=1.0,
#     stomp_check=True,
#     clamp_l_eps=None,
# ):
#     device = next(model.parameters()).device
#     model.train()

#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     ok_params = _check_trainables_finite(model, name_contains=("lora_" if only_lora else None))
#     if not ok_params:
#         print("[nan_trace] >>> Your trainables are already non-finite. Reload model / re-init LoRA before debugging backward.")
#         return

#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels    = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     set_sasacg_params(model, w=int(w), verbose=True)
#     model.zero_grad(set_to_none=True)

#     tracked = _select_stomp_tensors(model, layer_ids=(31, 30, 0))
#     snap0 = _snapshot_selected(tracked) if stomp_check else None

#     orig, events, call_counts = install_sasa_backward_sentinels(
#         model,
#         stop_on_first=True,
#         verbose=True,
#         force_fp32_proj_bwd=bool(force_fp32_proj_bwd),
#         clamp_l_eps=clamp_l_eps,
#     )

#     torch.autograd.set_detect_anomaly(bool(anomaly))

#     try:
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         with SASACandCacheContext():
#             with torch.autocast(
#                 device_type="cuda",
#                 dtype=amp_dtype,
#                 enabled=(torch.cuda.is_available() and bool(autocast_on)),
#             ):
#                 out = model(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     labels=labels,
#                     use_cache=False,
#                     past_key_values=None,
#                 )
#                 loss = out.loss

#         if stomp_check:
#             _report_stomp(snap0, tracked, tag="after_forward")

#         loss_ok = _isfinite_all(loss.detach())
#         logits_ok = _isfinite_all(out.logits.detach())
#         print(
#             f"[nan_trace] forward: loss={float(loss.detach().float().cpu().item()):.6f} "
#             f"loss_ok={loss_ok} | logits_ok={logits_ok} logits_maxabs={_maxabs(out.logits.detach()):.3e} "
#             f"| loss_scale={float(loss_scale):.3e} | clamp_l_eps={clamp_l_eps}"
#         )
#         if not (loss_ok and logits_ok):
#             print("[nan_trace] >>> Forward already non-finite. Backward trace is meaningless until forward is fixed.")
#             return

#         (loss * float(loss_scale)).backward()

#         print("[nan_trace] ✅ backward completed without any sentinel detecting non-finites.")
#         bad = 0
#         total = 0
#         for n,p in _iter_trainables_named(model):
#             if only_lora and ("lora_" not in n):
#                 continue
#             if p.grad is None:
#                 continue
#             total += 1
#             if not _isfinite_all(p.grad):
#                 bad += 1
#         print(f"[nan_trace] grad check (filtered={'lora_' if only_lora else 'all'}): nonfinite={bad}/{total}")

#     finally:
#         if stomp_check and snap0 is not None:
#             _report_stomp(snap0, tracked, tag="after_backward_or_crash")
#         uninstall_sasa_backward_sentinels(model, orig)
#         torch.autograd.set_detect_anomaly(False)

# def _run_case(name, **kwargs):
#     print("\n" + "="*90)
#     print(f"[nan_trace] RUN CASE: {name} | {kwargs}")
#     print("="*90)
#     try:
#         run_one_real_backward_trace(**kwargs)
#     except Exception as e:
#         print(f"[nan_trace] ({name}) stopped with exception: {type(e).__name__}: {e}")

# # ============================================================
# # RUNS THAT MATTER NOW
# #   1) Baseline (repro Stage-B fp16 overflow)
# #   2) Stage-B FP32 discriminator (exposes Stage-E issue)
# #   3) Loss-scale sweep under Stage-B FP32
# #   4) Debug A/B: clamp l under Stage-B FP32 (tests l==0 denom issue)
# # ============================================================

# _run_case(
#     "baseline_autocast_OFF",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=False,
#     loss_scale=1.0,
#     stomp_check=True,
#     clamp_l_eps=None,
# )

# _run_case(
#     "stageB_force_FP32_inputs",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=True,
#     loss_scale=1.0,
#     stomp_check=True,
#     clamp_l_eps=None,
# )

# for s in [1e-2, 1e-3, 1e-4]:
#     _run_case(
#         f"stageB_FP32 + loss_scale_{s:.0e}",
#         w=128, max_seq=256, anomaly=False, only_lora=True,
#         autocast_on=False,
#         force_fp32_proj_bwd=True,
#         loss_scale=s,
#         stomp_check=True,
#         clamp_l_eps=None,
#     )

# _run_case(
#     "stageB_FP32 + loss_scale_1e-2 + clamp_l",
#     w=128, max_seq=256, anomaly=False, only_lora=True,
#     autocast_on=False,
#     force_fp32_proj_bwd=True,
#     loss_scale=1e-2,
#     stomp_check=True,
#     clamp_l_eps=1e-6,
# )


In [ ]:
# # ============================================================
# # REAL BACKWARD NaN TRACER (find first bad SASA CUDA backward)
# #   - wraps Stage-E mh_bwd + Stage-B proj_bwd across ALL SASA layers
# #   - runs ONE real CE backward micro-step
# #   - prints FIRST layer/stage where non-finites appear (inputs or outputs)
# # ============================================================
# import torch, gc, math
# from collections import defaultdict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext", "_get_decoder_layers_any_wrapper"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_trace] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _isfinite_all(x):
#     return bool(torch.isfinite(x).all().item())

# def _nonfinite_count(x):
#     return int((~torch.isfinite(x)).sum().item())

# def _maxabs(x):
#     return float(x.detach().float().abs().max().item()) if x.numel() else 0.0

# def _check_trainables_finite(m, name_contains=None, max_show=30):
#     bad = []
#     total = 0
#     for n,p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         total += 1
#         if not _isfinite_all(p.detach()):
#             bad.append((n, tuple(p.shape), str(p.dtype), _nonfinite_count(p.detach()), _maxabs(p.detach())))
#     if bad:
#         print(f"[nan_trace] ❌ NON-FINITE TRAINABLE PARAMS already: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ trainable params finite: {total} tensors" + (f" (filtered by '{name_contains}')" if name_contains else ""))
#     return True

# def _get_sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for li, l in enumerate(layers):
#         sa = getattr(l, "self_attn", None)
#         if sa is not None and sa.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((li, l, sa))
#     return sasa

# def install_sasa_backward_sentinels(model, stop_on_first=True, verbose=True):
#     """
#     Wrap mh_bwd and proj_bwd for each SASA layer to detect non-finites in:
#       - Stage-E backward outputs: dQ,dK,dV
#       - Stage-B backward outputs: dH, dWq, dWk, dWv
#     Also checks key inputs (dOut, Q/K/V, m/l, dQ/dK/dV, etc.)
#     """
#     sasa = _get_sasa_layers(model)
#     if not sasa:
#         raise RuntimeError("[nan_trace] No SASA layers found.")

#     events = []
#     orig = {"mh_bwd": {}, "proj_bwd": {}}
#     call_counts = defaultdict(int)

#     def _emit(tag, li, msg):
#         s = f"[nan_trace] ❌ {tag} @ layer {li}: {msg}"
#         events.append(s)
#         print(s)
#         if stop_on_first:
#             raise RuntimeError(s)

#     for li, layer, attn in sasa:
#         # ---- wrap Stage-E mh_bwd ----
#         if hasattr(attn, "mh_bwd") and callable(attn.mh_bwd):
#             orig_mh = attn.mh_bwd
#             orig["mh_bwd"][li] = orig_mh

#             def _make_mh_wrap(li, fn):
#                 def mh_bwd_wrap(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#                     call_counts[("mh_bwd", li)] += 1
#                     # check inputs (cheap but decisive)
#                     if torch.is_tensor(dOut) and (not _isfinite_all(dOut)):
#                         _emit("Stage-E INPUT", li, f"dOut nonfinite (bad={_nonfinite_count(dOut)}) maxabs={_maxabs(dOut):.3e}")
#                     for nm, t in [("Q",Q),("K",K),("V",V),("m",m),("l",l)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head)

#                     # out should be (dQ,dK,dV) but be robust
#                     if isinstance(out, (tuple, list)):
#                         outs = out
#                     else:
#                         outs = (out,)

#                     # check outputs
#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return mh_bwd_wrap
#             attn.mh_bwd = _make_mh_wrap(li, orig_mh)

#         # ---- wrap Stage-B proj_bwd ----
#         if hasattr(attn, "proj_bwd") and callable(attn.proj_bwd):
#             orig_pb = attn.proj_bwd
#             orig["proj_bwd"][li] = orig_pb

#             def _make_proj_wrap(li, fn):
#                 def proj_bwd_wrap(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t):
#                     call_counts[("proj_bwd", li)] += 1

#                     for nm, t in [("dQ",dQ),("dK",dK),("dV",dV),("H",H_flat),("cos",cos_t),("sin",sin_t)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t)

#                     if isinstance(out, (tuple, list)):
#                         outs = out
#                     else:
#                         outs = (out,)

#                     # ---- PATCH: print output shapes BEFORE finiteness checks ----
#                     try:
#                         shapes = []
#                         for t in outs:
#                             if torch.is_tensor(t):
#                                 shapes.append(tuple(t.shape))
#                             else:
#                                 shapes.append(type(t).__name__)
#                         print(f"[nan_trace] Stage-B OUTPUT SHAPES @ layer {li}: {shapes}")
#                     except Exception:
#                         pass
#                     # ----------------------------------------------------------

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return proj_bwd_wrap
#             attn.proj_bwd = _make_proj_wrap(li, orig_pb)

#     if verbose:
#         print(f"[nan_trace] installed sentinels on {len(sasa)} SASA layers "
#               f"({'stop_on_first' if stop_on_first else 'collect_all'}).")
#     return orig, events, call_counts

# def uninstall_sasa_backward_sentinels(model, orig):
#     sasa = _get_sasa_layers(model)
#     for li, layer, attn in sasa:
#         if li in orig.get("mh_bwd", {}):
#             attn.mh_bwd = orig["mh_bwd"][li]
#         if li in orig.get("proj_bwd", {}):
#             attn.proj_bwd = orig["proj_bwd"][li]
#     print("[nan_trace] restored original mh_bwd/proj_bwd")

# def run_one_real_backward_trace(
#     w=128,
#     max_seq=256,
#     anomaly=False,
#     only_lora=True,
#     autocast_on=True,   # PATCH: allow toggling autocast ON/OFF
# ):
#     device = next(model.parameters()).device
#     model.train()

#     # AMP like your loop
#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     # verify params are sane BEFORE we do anything
#     ok_params = _check_trainables_finite(model, name_contains=("lora_" if only_lora else None))
#     if not ok_params:
#         print("[nan_trace] >>> Your trainables are already non-finite. Reload model / re-init LoRA before debugging backward.")
#         return

#     # batch
#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels    = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     # set SASA params
#     set_sasacg_params(model, w=int(w), verbose=True)

#     # clear grads
#     model.zero_grad(set_to_none=True)

#     # install sentinels
#     orig, events, call_counts = install_sasa_backward_sentinels(model, stop_on_first=True, verbose=True)

#     # anomaly option
#     torch.autograd.set_detect_anomaly(bool(anomaly))

#     try:
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         with SASACandCacheContext():
#             # ---- PATCH: run with autocast ON/OFF ----
#             with torch.autocast(
#                 device_type="cuda",
#                 dtype=amp_dtype,
#                 enabled=(torch.cuda.is_available() and bool(autocast_on)),
#             ):
#                 out = model(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     labels=labels,
#                     use_cache=False,
#                     past_key_values=None,
#                 )
#                 loss = out.loss

#         # forward sanity
#         loss_ok = _isfinite_all(loss.detach())
#         logits_ok = _isfinite_all(out.logits.detach())
#         print(f"[nan_trace] forward: loss={float(loss.detach().float().cpu().item()):.6f} loss_ok={loss_ok} | logits_ok={logits_ok} logits_maxabs={_maxabs(out.logits.detach()):.3e}")
#         if not (loss_ok and logits_ok):
#             print("[nan_trace] >>> Forward already non-finite. Backward trace is meaningless until forward is fixed.")
#             return

#         # backward (this is where we expect to catch the first NaN)
#         loss.backward()

#         # if we reach here, no sentinel fired
#         print("[nan_trace] ✅ backward completed without any Stage-B/Stage-E sentinel detecting non-finites.")
#         # still check LoRA grads quickly
#         bad = 0
#         total = 0
#         for n,p in _iter_trainables_named(model):
#             if only_lora and ("lora_" not in n):
#                 continue
#             if p.grad is None:
#                 continue
#             total += 1
#             if not _isfinite_all(p.grad):
#                 bad += 1
#         print(f"[nan_trace] grad check (filtered={'lora_' if only_lora else 'all'}): nonfinite={bad}/{total}")

#     finally:
#         uninstall_sasa_backward_sentinels(model, orig)
#         # restore anomaly setting to default off
#         torch.autograd.set_detect_anomaly(False)

# # ---- RUN ----
# # PATCH: run twice (autocast ON then OFF)
# run_one_real_backward_trace(w=128, max_seq=256, anomaly=False, only_lora=True, autocast_on=True)
# run_one_real_backward_trace(w=128, max_seq=256, anomaly=False, only_lora=True, autocast_on=False)


In [ ]:
# # ============================================================
# # REAL BACKWARD NaN TRACER (find first bad SASA CUDA backward)
# #   - wraps Stage-E mh_bwd + Stage-B proj_bwd across ALL SASA layers
# #   - runs ONE real CE backward micro-step
# #   - prints FIRST layer/stage where non-finites appear (inputs or outputs)
# # ============================================================
# import torch, gc, math
# from collections import defaultdict

# # ---------- hard requirements ----------
# _req = ["model", "train_loader", "set_sasacg_params", "SASACandCacheContext", "_get_decoder_layers_any_wrapper"]
# _missing = [x for x in _req if x not in globals()]
# if _missing:
#     raise RuntimeError(f"[nan_trace] Missing required names: {_missing}")

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# def _iter_trainables_named(m):
#     for n, p in m.named_parameters():
#         if p.requires_grad:
#             yield n, p

# def _isfinite_all(x):
#     return bool(torch.isfinite(x).all().item())

# def _nonfinite_count(x):
#     return int((~torch.isfinite(x)).sum().item())

# def _maxabs(x):
#     return float(x.detach().float().abs().max().item()) if x.numel() else 0.0

# def _check_trainables_finite(m, name_contains=None, max_show=30):
#     bad = []
#     total = 0
#     for n,p in _iter_trainables_named(m):
#         if name_contains is not None and (name_contains not in n):
#             continue
#         total += 1
#         if not _isfinite_all(p.detach()):
#             bad.append((n, tuple(p.shape), str(p.dtype), _nonfinite_count(p.detach()), _maxabs(p.detach())))
#     if bad:
#         print(f"[nan_trace] ❌ NON-FINITE TRAINABLE PARAMS already: {len(bad)} / {total} (showing up to {max_show})")
#         for row in bad[:max_show]:
#             print("   ", row)
#         return False
#     print(f"[nan_trace] ✅ trainable params finite: {total} tensors" + (f" (filtered by '{name_contains}')" if name_contains else ""))
#     return True

# def _get_sasa_layers(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     sasa = []
#     for li, l in enumerate(layers):
#         sa = getattr(l, "self_attn", None)
#         if sa is not None and sa.__class__.__name__ == "MistralSASACGAttention_MH":
#             sasa.append((li, l, sa))
#     return sasa

# def install_sasa_backward_sentinels(model, stop_on_first=True, verbose=True):
#     """
#     Wrap mh_bwd and proj_bwd for each SASA layer to detect non-finites in:
#       - Stage-E backward outputs: dQ,dK,dV
#       - Stage-B backward outputs: dH, dWq, dWk, dWv
#     Also checks key inputs (dOut, Q/K/V, m/l, dQ/dK/dV, etc.)
#     """
#     sasa = _get_sasa_layers(model)
#     if not sasa:
#         raise RuntimeError("[nan_trace] No SASA layers found.")

#     events = []
#     orig = {"mh_bwd": {}, "proj_bwd": {}}
#     call_counts = defaultdict(int)

#     def _emit(tag, li, msg):
#         s = f"[nan_trace] ❌ {tag} @ layer {li}: {msg}"
#         events.append(s)
#         print(s)
#         if stop_on_first:
#             raise RuntimeError(s)

#     for li, layer, attn in sasa:
#         # ---- wrap Stage-E mh_bwd ----
#         if hasattr(attn, "mh_bwd") and callable(attn.mh_bwd):
#             orig_mh = attn.mh_bwd
#             orig["mh_bwd"][li] = orig_mh

#             def _make_mh_wrap(li, fn):
#                 def mh_bwd_wrap(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head):
#                     call_counts[("mh_bwd", li)] += 1
#                     # check inputs (cheap but decisive)
#                     if torch.is_tensor(dOut) and (not _isfinite_all(dOut)):
#                         _emit("Stage-E INPUT", li, f"dOut nonfinite (bad={_nonfinite_count(dOut)}) maxabs={_maxabs(dOut):.3e}")
#                     for nm, t in [("Q",Q),("K",K),("V",V),("m",m),("l",l)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dOut, Q, K, V, cand_i32, mask_u8, m, l, scale, cand_is_per_head)

#                     # out should be (dQ,dK,dV) but be robust
#                     if isinstance(out, (tuple, list)):
#                         outs = out
#                     else:
#                         outs = (out,)

#                     # check outputs
#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-E OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return mh_bwd_wrap
#             attn.mh_bwd = _make_mh_wrap(li, orig_mh)

#         # ---- wrap Stage-B proj_bwd ----
#         if hasattr(attn, "proj_bwd") and callable(attn.proj_bwd):
#             orig_pb = attn.proj_bwd
#             orig["proj_bwd"][li] = orig_pb

#             def _make_proj_wrap(li, fn):
#                 def proj_bwd_wrap(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t):
#                     call_counts[("proj_bwd", li)] += 1

#                     # ---- PATCH: for layer 31, print dtype + maxabs of dQ and H_flat (before any finiteness checks) ----
#                     if int(li) == 31:
#                         try:
#                             dq_dtype = str(getattr(dQ, "dtype", None))
#                             h_dtype  = str(getattr(H_flat, "dtype", None))
#                             dq_ma = _maxabs(dQ) if torch.is_tensor(dQ) else float("nan")
#                             h_ma  = _maxabs(H_flat) if torch.is_tensor(H_flat) else float("nan")
#                             print(f"[nan_trace] Stage-B INPUT STATS @ layer {li}: dQ dtype={dq_dtype} maxabs={dq_ma:.3e} | H dtype={h_dtype} maxabs={h_ma:.3e}")
#                         except Exception:
#                             pass
#                     # -----------------------------------------------------------------------------------------------

#                     for nm, t in [("dQ",dQ),("dK",dK),("dV",dV),("H",H_flat),("cos",cos_t),("sin",sin_t)]:
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B INPUT", li, f"{nm} nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")

#                     out = fn(dQ, dK, dV, H_flat, Wq, Wk_eff, Wv_eff, cos_t, sin_t)

#                     if isinstance(out, (tuple, list)):
#                         outs = out
#                     else:
#                         outs = (out,)

#                     # ---- already-present PATCH: print output shapes BEFORE finiteness checks ----
#                     try:
#                         shapes = []
#                         for t in outs:
#                             if torch.is_tensor(t):
#                                 shapes.append(tuple(t.shape))
#                             else:
#                                 shapes.append(type(t).__name__)
#                         print(f"[nan_trace] Stage-B OUTPUT SHAPES @ layer {li}: {shapes}")
#                     except Exception:
#                         pass
#                     # ----------------------------------------------------------

#                     for j, t in enumerate(outs):
#                         if torch.is_tensor(t) and (not _isfinite_all(t)):
#                             _emit("Stage-B OUTPUT", li, f"output[{j}] nonfinite (bad={_nonfinite_count(t)}) maxabs={_maxabs(t):.3e}")
#                     return out
#                 return proj_bwd_wrap
#             attn.proj_bwd = _make_proj_wrap(li, orig_pb)

#     if verbose:
#         print(f"[nan_trace] installed sentinels on {len(sasa)} SASA layers "
#               f"({'stop_on_first' if stop_on_first else 'collect_all'}).")
#     return orig, events, call_counts

# def uninstall_sasa_backward_sentinels(model, orig):
#     sasa = _get_sasa_layers(model)
#     for li, layer, attn in sasa:
#         if li in orig.get("mh_bwd", {}):
#             attn.mh_bwd = orig["mh_bwd"][li]
#         if li in orig.get("proj_bwd", {}):
#             attn.proj_bwd = orig["proj_bwd"][li]
#     print("[nan_trace] restored original mh_bwd/proj_bwd")

# def run_one_real_backward_trace(
#     w=128,
#     max_seq=256,
#     anomaly=False,
#     only_lora=True,
#     autocast_on=True,   # PATCH: allow toggling autocast ON/OFF
# ):
#     device = next(model.parameters()).device
#     model.train()

#     # AMP like your loop
#     use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#     amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     # verify params are sane BEFORE we do anything
#     ok_params = _check_trainables_finite(model, name_contains=("lora_" if only_lora else None))
#     if not ok_params:
#         print("[nan_trace] >>> Your trainables are already non-finite. Reload model / re-init LoRA before debugging backward.")
#         return

#     # batch
#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)
#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     labels = batch.get("labels", input_ids)

#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         labels    = labels[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()

#     # set SASA params
#     set_sasacg_params(model, w=int(w), verbose=True)

#     # clear grads
#     model.zero_grad(set_to_none=True)

#     # install sentinels
#     orig, events, call_counts = install_sasa_backward_sentinels(model, stop_on_first=True, verbose=True)

#     # anomaly option
#     torch.autograd.set_detect_anomaly(bool(anomaly))

#     try:
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#         with SASACandCacheContext():
#             # ---- PATCH: run with autocast ON/OFF ----
#             with torch.autocast(
#                 device_type="cuda",
#                 dtype=amp_dtype,
#                 enabled=(torch.cuda.is_available() and bool(autocast_on)),
#             ):
#                 out = model(
#                     input_ids=input_ids,
#                     attention_mask=attention_mask,
#                     labels=labels,
#                     use_cache=False,
#                     past_key_values=None,
#                 )
#                 loss = out.loss

#         # forward sanity
#         loss_ok = _isfinite_all(loss.detach())
#         logits_ok = _isfinite_all(out.logits.detach())
#         print(f"[nan_trace] forward: loss={float(loss.detach().float().cpu().item()):.6f} loss_ok={loss_ok} | logits_ok={logits_ok} logits_maxabs={_maxabs(out.logits.detach()):.3e}")
#         if not (loss_ok and logits_ok):
#             print("[nan_trace] >>> Forward already non-finite. Backward trace is meaningless until forward is fixed.")
#             return

#         # backward (this is where we expect to catch the first NaN)
#         loss.backward()

#         # if we reach here, no sentinel fired
#         print("[nan_trace] ✅ backward completed without any Stage-B/Stage-E sentinel detecting non-finites.")
#         # still check LoRA grads quickly
#         bad = 0
#         total = 0
#         for n,p in _iter_trainables_named(model):
#             if only_lora and ("lora_" not in n):
#                 continue
#             if p.grad is None:
#                 continue
#             total += 1
#             if not _isfinite_all(p.grad):
#                 bad += 1
#         print(f"[nan_trace] grad check (filtered={'lora_' if only_lora else 'all'}): nonfinite={bad}/{total}")

#     finally:
#         uninstall_sasa_backward_sentinels(model, orig)
#         # restore anomaly setting to default off
#         torch.autograd.set_detect_anomaly(False)

# # ---- RUN ----
# # PATCH: run twice (autocast ON then OFF) and continue even if ON run throws
# try:
#     run_one_real_backward_trace(w=128, max_seq=256, anomaly=False, only_lora=True, autocast_on=True)
# except Exception as e:
#     print(f"[nan_trace] (autocast ON) stopped with exception: {type(e).__name__}: {e}")

# try:
#     run_one_real_backward_trace(w=128, max_seq=256, anomaly=False, only_lora=True, autocast_on=False)
# except Exception as e:
#     print(f"[nan_trace] (autocast OFF) stopped with exception: {type(e).__name__}: {e}")


In [ ]:
# # ============================================================
# # Scaffold sweep (Ablation 2: structure knob sweep)
# #   - window-only vs ring-only vs window+ring
# #   - plot/compare vs measured keys_per_query_mean (NOT nominal w)
# # ============================================================
# import pandas as pd
# import gc, torch

# # ---- sweep knobs ----
# SWEEP_TRAIN_STEPS = 0        # set >0 if you want to retrain per scaffold (expensive)
# SWEEP_EVAL_STEPS  = 50
# SWEEP_LR          = 3e-4
# SWEEP_GRAD_ACCUM  = 1

# # IMPORTANT: new Stage-A enforces include_self == True
# SWEEP_INCLUDE_SELF = True

# # Grab one batch for budget measurement (mask-aware)
# _budget_probe_batch = next(iter(val_loader))

# # ----------------------------
# # Ablation 2: scaffold configs
# #   mode: 0=window, 1=ring, 2=window+ring
# # NOTE:
# #   - ring-only (mode=1) requires num_anchors > 0 (per your TORCH_CHECK)
# #   - ring_outer must be <= N-1 (N from probe batch)
# # ----------------------------
# _probe_N = int(_budget_probe_batch["attention_mask"].shape[1]) if (
#     isinstance(_budget_probe_batch, dict) and "attention_mask" in _budget_probe_batch and _budget_probe_batch["attention_mask"] is not None
# ) else int(_budget_probe_batch["input_ids"].shape[1])

# def _clamp_ring_outer(r_out: int, N: int) -> int:
#     return int(min(int(r_out), int(N) - 1))

# SCAFFOLD_SWEEP = [
#   # window-only sweep
#   {"name":"win_w32",            "mode":0, "w":32,  "r_in":0,    "r_out":0},
#   {"name":"win_w64",            "mode":0, "w":64,  "r_in":0,    "r_out":0},
#   {"name":"win_w128",           "mode":0, "w":128, "r_in":0,    "r_out":0},

#   # ring-only sweep (no local edges) — requires anchors > 0
#   {"name":"ring_32_64",         "mode":1, "w":0,   "r_in":32,   "r_out":_clamp_ring_outer(64,  _probe_N)},
#   {"name":"ring_64_128",        "mode":1, "w":0,   "r_in":64,   "r_out":_clamp_ring_outer(128, _probe_N)},
#   {"name":"ring_128_255",       "mode":1, "w":0,   "r_in":128,  "r_out":_clamp_ring_outer(255, _probe_N)},

#   # window+ring sweep (local + long edges)
#   {"name":"win32_ring64_128",   "mode":2, "w":32,  "r_in":64,   "r_out":_clamp_ring_outer(128, _probe_N)},
#   {"name":"win64_ring128_255",  "mode":2, "w":64,  "r_in":128,  "r_out":_clamp_ring_outer(255, _probe_N)},
# ]

# RECS = []

# for cfg in SCAFFOLD_SWEEP:
#     print("\n" + "="*80)

#     mode = int(cfg["mode"])
#     ww   = int(cfg["w"])
#     rin  = int(cfg["r_in"])
#     rout = int(cfg["r_out"])

#     # enforce ring-only anchor requirement from Stage-A
#     if mode == 1 and int(SWEEP_NUM_ANCHORS) <= 0:
#         raise RuntimeError("ring-only (mode=1) requires SWEEP_NUM_ANCHORS > 0 (per Stage-A TORCH_CHECK).")

#     print(
#         f"SCAFFOLD {cfg['name']} | mode={mode} w={ww} ring=[{rin}..{rout}] "
#         f"(anchors={int(SWEEP_NUM_ANCHORS)}, include_self={bool(SWEEP_INCLUDE_SELF)})"
#     )

#     # set SASA params on all layers
#     set_sasacg_params(
#         model,
#         w=ww,
#         scaffold_mode=mode,
#         ring_inner=rin,
#         ring_outer=rout,
#         num_anchors=int(SWEEP_NUM_ANCHORS),
#         include_self=bool(SWEEP_INCLUDE_SELF),
#         verbose=True,
#     )

#     # measure actual keys/query & density (+ window/ring/anchor breakdown, distances)
#     budget_stats = measure_effective_budget_from_batch(
#         _budget_probe_batch,
#         w=ww,
#         ring_inner=rin,
#         ring_outer=rout,
#         scaffold_mode=mode,
#         num_anchors=int(SWEEP_NUM_ANCHORS),
#         include_self=bool(SWEEP_INCLUDE_SELF),
#     )
#     print("budget:", budget_stats)

#     # optional short train (usually keep 0 for frontier-on-inference)
#     if SWEEP_TRAIN_STEPS and SWEEP_TRAIN_STEPS > 0:
#         train_stats = train_n_steps(
#             model,
#             train_loader,
#             steps=int(SWEEP_TRAIN_STEPS),
#             lr=float(SWEEP_LR),
#             grad_accum=int(SWEEP_GRAD_ACCUM),
#         )
#     else:
#         train_stats = {
#             "train_tokens": 0,
#             "train_runtime_sec": 0.0,
#             "train_tokens_per_sec": None,
#             "peak_alloc_gb": None,
#             "peak_reserved_gb": None,
#         }

#     # eval (quality point)
#     _reset_cuda_peak()
#     eval_stats = eval_loss_only(model, val_loader, max_steps=int(SWEEP_EVAL_STEPS))

#     # record
#     rec = {}
#     rec.update({
#         "scaffold_name": cfg["name"],
#         "scaffold_mode": mode,
#         "w": ww,
#         "ring_inner": rin,
#         "ring_outer": rout,
#         "num_anchors": int(SWEEP_NUM_ANCHORS),
#         "include_self": bool(SWEEP_INCLUDE_SELF),
#     })
#     rec.update(budget_stats)
#     rec.update({f"train_{k}": v for k, v in train_stats.items() if k not in ("peak_alloc_gb", "peak_reserved_gb")})
#     rec.update({f"eval_{k}": v for k, v in eval_stats.items()})

#     # convenience fields for plotting
#     rec["tokens_per_sec"] = rec["eval_eval_tokens_per_sec"]
#     rec["eval_ppl"] = rec["eval_eval_ppl"]
#     rec["eval_loss"] = rec["eval_eval_loss"]
#     rec["peak_alloc_gb_eval"] = rec["eval_peak_alloc_gb"]
#     rec["peak_reserved_gb_eval"] = rec["eval_peak_reserved_gb"]

#     RECS.append(rec)

#     # compact summary emphasizing measured budget (not nominal w)
#     print("REC:", {
#         "name": rec["scaffold_name"],
#         "keys_mean": rec.get("keys_per_query_mean", None),
#         "win_keys_mean": rec.get("keys_per_query_window_mean", None),
#         "ring_keys_mean": rec.get("keys_per_query_ring_mean", None),
#         "anc_keys_mean": rec.get("keys_per_query_anchor_mean", None),
#         "mean_dist_win": rec.get("mean_distance_window", None),
#         "mean_dist_ring": rec.get("mean_distance_ring", None),
#         "ppl": rec["eval_ppl"],
#         "tps": rec["tokens_per_sec"],
#         "peak_gb": rec["peak_alloc_gb_eval"],
#     })

# df = pd.DataFrame(RECS).sort_values("keys_per_query_mean").reset_index(drop=True)
# df


In [ ]:
# # ============================================================
# # KERNEL BACKWARD NAN PROBE (Stage-B vs Stage-E) — CORRECTED
# #   - FIX: removes stale forward_pre_hooks left behind by crashed runs
# #   - robustly captures hidden_states even if passed via kwargs
# #   - optional fix_empty_rows to test "empty softmax row" hypothesis
# # ============================================================
# import torch, gc, math

# # ----------------------------
# # 0) Purge stale crashed hooks from earlier attempts
# # ----------------------------
# def _purge_stale_prehooks(root, names=("_prehook",)):
#     removed = 0
#     for m in root.modules():
#         if hasattr(m, "_forward_pre_hooks") and isinstance(m._forward_pre_hooks, dict):
#             for hid, fn in list(m._forward_pre_hooks.items()):
#                 if getattr(fn, "__name__", None) in names:
#                     try:
#                         del m._forward_pre_hooks[hid]
#                         removed += 1
#                     except Exception:
#                         pass
#     print(f"[nan_probe] purged stale forward_pre_hooks named {names}: removed={removed}")

# _purge_stale_prehooks(model, names=("_prehook",))

# # ----------------------------
# # 1) Helpers
# # ----------------------------
# def _find_first_sasa_layer(model):
#     layers = _get_decoder_layers_any_wrapper(model)
#     for li, l in enumerate(layers):
#         if getattr(l, "self_attn", None) is not None and l.self_attn.__class__.__name__ == "MistralSASACGAttention_MH":
#             return li, l, layers
#     return None, None, layers

# if "_to_device" not in globals():
#     def _to_device(batch, device):
#         out = {}
#         for k, v in batch.items():
#             out[k] = v.to(device) if torch.is_tensor(v) else v
#         return out

# # ----------------------------
# # 2) The probe
# # ----------------------------
# def kernel_backward_nan_probe(w=128, max_seq=1024, force_dtype=None, fix_empty_rows=False):
#     # sanity: required symbols
#     req = ["StageBMHProjectRoPEFn", "StageEMHFusedFn", "rope_cos_sin_token_major_strict"]
#     missing = [x for x in req if x not in globals()]
#     if missing:
#         raise RuntimeError(f"[nan_probe] Missing required names: {missing}")

#     li, layer0, layers = _find_first_sasa_layer(model)
#     if layer0 is None:
#         raise RuntimeError("[nan_probe] No SASA layers found.")
#     attn0 = layer0.self_attn

#     model.train()
#     device = next(model.parameters()).device

#     batch = next(iter(train_loader))
#     batch = _to_device(batch, device)

#     input_ids = batch["input_ids"]
#     attention_mask = batch.get("attention_mask", None)
#     B, N = input_ids.shape
#     if N > max_seq:
#         input_ids = input_ids[:, :max_seq].contiguous()
#         if attention_mask is not None:
#             attention_mask = attention_mask[:, :max_seq].contiguous()
#         B, N = input_ids.shape

#     # ------------------------------------------------------------
#     # (A) Capture hidden_states entering attn0.forward (kwargs-safe)
#     # ------------------------------------------------------------
#     cap = {}
#     _orig_forward = attn0.forward

#     def _forward_wrap(*args, **kwargs):
#         hs = None
#         if len(args) > 0:
#             hs = args[0]
#         else:
#             for k in ("hidden_states", "x", "hidden_state"):
#                 if k in kwargs:
#                     hs = kwargs[k]
#                     break
#         if ("hs" not in cap) and (hs is not None):
#             cap["hs"] = hs.detach()
#         return _orig_forward(*args, **kwargs)

#     attn0.forward = _forward_wrap
#     try:
#         with torch.no_grad():
#             set_sasacg_params(model, w=int(w), verbose=False)
#             _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
#     finally:
#         attn0.forward = _orig_forward

#     if "hs" not in cap:
#         raise RuntimeError("[nan_probe] Failed to capture hidden_states entering first SASA layer.")

#     hs = cap["hs"]  # [B,N,H]
#     hs_ok = bool(torch.isfinite(hs).all().item())
#     print(f"[nan_probe] captured hs shape={tuple(hs.shape)} finite? {hs_ok}")
#     if not hs_ok:
#         # If your model already got poisoned by NaN LoRA weights, hs can be non-finite.
#         # This keeps the probe meaningful for kernel debugging.
#         hs = torch.nan_to_num(hs, nan=0.0, posinf=0.0, neginf=0.0)

#     # ------------------------------------------------------------
#     # RoPE cos/sin (Stage-B contract)
#     # ------------------------------------------------------------
#     position_ids = torch.arange(N, device=device, dtype=torch.long).unsqueeze(0).expand(B, N)
#     cos_t, sin_t = rope_cos_sin_token_major_strict(
#         position_ids,
#         head_dim=int(attn0.head_dim),
#         device=device,
#         theta=float(getattr(getattr(attn0.old, "config", None), "rope_theta", 10000.0)),
#     )

#     # token-valid mask for Stage-A
#     if attention_mask is None:
#         token_valid_bn = torch.ones((B, N), device=device, dtype=torch.int64)
#     else:
#         token_valid_bn = attention_mask.to(device=device, dtype=torch.int64)

#     # ------------------------------------------------------------
#     # Stage-A
#     # ------------------------------------------------------------
#     cand_i32, ok_u8, valid_T = attn0.build_candidates_cuda(
#         token_valid_bn,
#         int(w),
#         int(getattr(attn0, "ring_inner", 0)),
#         int(getattr(attn0, "ring_outer", 0)),
#         int(getattr(attn0, "scaffold_mode", 0)),
#         int(getattr(attn0, "num_anchors", 0)),
#         bool(getattr(attn0, "include_self", True)),
#         int(getattr(attn0, "anchor_strategy", 0)),
#         int(getattr(attn0, "anchor_param0", 0)),
#     )

#     keys = ok_u8.to(torch.int32).sum(dim=1)
#     zeros = int((keys == 0).sum().item())
#     print(f"[nan_probe] Stage-A: T={ok_u8.size(0)} C={ok_u8.size(1)} "
#           f"keys_mean={keys.float().mean().item():.3f} keys_p95={torch.quantile(keys.float(),0.95).item():.1f} "
#           f"zero-key-queries={zeros}")

#     # Optional: enforce >=1 key per row (tests empty-softmax NaN hypothesis)
#     if fix_empty_rows and zeros > 0:
#         with torch.no_grad():
#             ridx = (keys == 0).nonzero(as_tuple=False).squeeze(1)
#             tok = (ridx % int(N)).to(device=cand_i32.device, dtype=cand_i32.dtype)
#             cand_i32[ridx, 0] = tok
#             ok_u8[ridx, 0] = 1
#         keys2 = ok_u8.to(torch.int32).sum(dim=1)
#         zeros2 = int((keys2 == 0).sum().item())
#         print(f"[nan_probe] Stage-A: after fix_empty_rows -> zero-key-queries={zeros2}")

#     # ------------------------------------------------------------
#     # Stage-B backward probe
#     # ------------------------------------------------------------
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()

#     if force_dtype is not None:
#         compute_dtype = force_dtype
#     else:
#         use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#         compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

#     H = hs.to(dtype=compute_dtype).contiguous().reshape(B * N, -1).requires_grad_(True)

#     Wq = attn0.old.q_proj.weight.detach().to(dtype=compute_dtype).contiguous()
#     Wk = attn0.old.k_proj.weight.detach().to(dtype=compute_dtype).contiguous()
#     Wv = attn0.old.v_proj.weight.detach().to(dtype=compute_dtype).contiguous()

#     Q, K, V = StageBMHProjectRoPEFn.apply(
#         H, Wq, Wk, Wv,
#         cos_t.float(), sin_t.float(),
#         attn0.proj_fwd, attn0.proj_bwd
#     )

#     lossB = (Q.float().square().mean() + K.float().square().mean() + V.float().square().mean())
#     lossB.backward()

#     gH_ok = (H.grad is not None) and bool(torch.isfinite(H.grad).all().item())
#     print(f"[nan_probe] Stage-B backward: loss={float(lossB.detach().cpu().item()):.6f} | dH finite? {gH_ok}")
#     if H.grad is not None and not gH_ok:
#         bad = (~torch.isfinite(H.grad)).sum().item()
#         print(f"[nan_probe] Stage-B backward NONFINITES in dH: {int(bad)} / {H.grad.numel()}")

#     # ------------------------------------------------------------
#     # Stage-E backward probe (isolated)
#     # ------------------------------------------------------------
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()

#     Q2 = Q.detach().requires_grad_(True)
#     K2 = K.detach().requires_grad_(True)
#     V2 = V.detach().requires_grad_(True)

#     scale = 1.0 / math.sqrt(float(attn0.head_dim))

#     out = StageEMHFusedFn.apply(
#         Q2, K2, V2,
#         cand_i32, ok_u8,
#         float(scale),
#         False,          # cand_is_per_head
#         attn0.mh_fwd,
#         attn0.mh_bwd
#     )

#     out_ok = bool(torch.isfinite(out).all().item())
#     print(f"[nan_probe] Stage-E forward finite? {out_ok}")
#     if not out_ok:
#         bad = (~torch.isfinite(out)).sum().item()
#         print(f"[nan_probe] Stage-E forward NONFINITES in out: {int(bad)} / {out.numel()}")

#     lossE = out.float().square().mean()
#     lossE.backward()

#     gQ_ok = (Q2.grad is not None) and bool(torch.isfinite(Q2.grad).all().item())
#     gK_ok = (K2.grad is not None) and bool(torch.isfinite(K2.grad).all().item())
#     gV_ok = (V2.grad is not None) and bool(torch.isfinite(V2.grad).all().item())

#     print(f"[nan_probe] Stage-E backward: loss={float(lossE.detach().cpu().item()):.6f} | dQ finite? {gQ_ok} | dK finite? {gK_ok} | dV finite? {gV_ok}")
#     if not (gQ_ok and gK_ok and gV_ok):
#         for nm, g in [("dQ", Q2.grad), ("dK", K2.grad), ("dV", V2.grad)]:
#             if g is None:
#                 print(f"[nan_probe] {nm} is None")
#                 continue
#             bad = (~torch.isfinite(g)).sum().item()
#             if bad:
#                 print(f"[nan_probe] Stage-E backward NONFINITES in {nm}: {int(bad)} / {g.numel()}")

# # Run it
# kernel_backward_nan_probe(w=128, max_seq=1024, fix_empty_rows=False)
# # If you see non-finites in Stage-E backward and zero-key-queries>0, run:
# # kernel_backward_nan_probe(w=128, max_seq=1024, fix_empty_rows=True)


In [ ]:
# ----------------------------
# Plot 3: Memory vs measured budget — marker encodes scaffold type
# ----------------------------
plt.figure()

for m in sorted(df_plot["scaffold_mode"].unique()):
    sub = df_plot[df_plot["scaffold_mode"] == m]
    plt.scatter(sub["keys_per_query_mean"], sub["peak_alloc_gb_eval"], marker=_scaffold_marker(m), label=_mode_name(m))

for _, r in df_plot.iterrows():
    plt.annotate(_scaffold_label(r),
                 (r["keys_per_query_mean"], r["peak_alloc_gb_eval"]),
                 fontsize=8)

plt.xlabel("Measured keys/query (mean)")
plt.ylabel("Peak alloc (GB) during eval")
plt.title("Scaffold structure sweep — Memory vs Budget")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Optional: frontier line (sorted) on Quality–Cost plot
#   (keeps labels scaffold-aware)
# ----------------------------
plt.figure(figsize=(6.5, 4.5))
plt.scatter(df_plot2["tokens_per_sec"], df_plot2["eval_ppl"])
plt.plot(df_plot2["tokens_per_sec"], df_plot2["eval_ppl"])

for _, r in df_plot2.iterrows():
    plt.annotate(
        f"{_mode_name(r['scaffold_mode'])} k≈{r['keys_per_query_mean']:.0f} w={int(r['w'])} r=[{int(r['ring_inner'])},{int(r['ring_outer'])}]",
        (r["tokens_per_sec"], r["eval_ppl"]),
        fontsize=8
    )

plt.yscale("log")
plt.xlabel("Eval tokens/sec")
plt.ylabel("Perplexity (log scale)")
plt.title("Quality–Cost Frontier (Scaffold sweep, N=256)")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# plt.figure(figsize=(6.5,4.5))
# dfk = df.sort_values("keys_per_query_mean")

# plt.scatter(dfk["keys_per_query_mean"], dfk["eval_ppl"])
# plt.plot(dfk["keys_per_query_mean"], dfk["eval_ppl"])
# for _, r in dfk.iterrows():
#     plt.annotate(f"w={int(r['w'])}", (r["keys_per_query_mean"], r["eval_ppl"]), fontsize=8)

# plt.yscale("log")
# plt.xlabel("Measured keys/query (mean)")
# plt.ylabel("Perplexity (log scale)")
# plt.title("Quality vs Effective Budget (N=256)")
# plt.grid(True, which="both", alpha=0.3)
# plt.tight_layout()
# plt.show()


In [ ]:
# plt.figure(figsize=(6.5,4.5))
# plt.scatter(df["keys_per_query_mean"], df["peak_alloc_gb_eval"])
# for _, r in df.iterrows():
#     plt.annotate(f"w={int(r['w'])}", (r["keys_per_query_mean"], r["peak_alloc_gb_eval"]), fontsize=8)

# plt.ylim(df["peak_alloc_gb_eval"].min() - 0.1, df["peak_alloc_gb_eval"].max() + 0.1)
# plt.xlabel("Measured keys/query (mean)")
# plt.ylabel("Peak alloc during eval (GB)")
# plt.title("Memory vs Effective Budget (N=256)")
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()


In [ ]:
# from matplotlib.ticker import FormatStrFormatter


# df_frontier = df.sort_values("tokens_per_sec").reset_index(drop=True)
# df_budget   = df.sort_values("keys_per_query_mean").reset_index(drop=True)

# # ============================================================
# # Paper-grade plots (KDD-friendly) — with label offsets to avoid overlaps
# # ============================================================
# import matplotlib.pyplot as plt
# import numpy as np

# # -------------------------------
# # Robust geometry helpers
# # -------------------------------
# def _seg_intersects_rect(p1, p2, rect):
#     x1, y1 = p1
#     x2, y2 = p2
#     xmin, ymin, xmax, ymax = rect

#     # Quick reject: bounding boxes don't overlap
#     if max(x1, x2) < xmin or min(x1, x2) > xmax or max(y1, y2) < ymin or min(y1, y2) > ymax:
#         return False

#     # If either endpoint is inside rect => intersection
#     if (xmin <= x1 <= xmax and ymin <= y1 <= ymax) or (xmin <= x2 <= xmax and ymin <= y2 <= ymax):
#         return True

#     def ccw(A, B, C):
#         return (C[1]-A[1])*(B[0]-A[0]) > (B[1]-A[1])*(C[0]-A[0])

#     def seg_intersect(A, B, C, D):
#         return (ccw(A, C, D) != ccw(B, C, D)) and (ccw(A, B, C) != ccw(A, B, D))

#     A = (x1, y1)
#     B = (x2, y2)

#     R1 = (xmin, ymin); R2 = (xmax, ymin); R3 = (xmax, ymax); R4 = (xmin, ymax)

#     return (
#         seg_intersect(A, B, R1, R2) or
#         seg_intersect(A, B, R2, R3) or
#         seg_intersect(A, B, R3, R4) or
#         seg_intersect(A, B, R4, R1)
#     )

# def _bbox_rect_in_display(text_obj, renderer):
#     bb = text_obj.get_window_extent(renderer=renderer)
#     return (bb.x0, bb.y0, bb.x1, bb.y1)

# def _rect_inside_rect(inner, outer, pad=0.0):
#     ix0, iy0, ix1, iy1 = inner
#     ox0, oy0, ox1, oy1 = outer
#     return (ix0 >= ox0 + pad and iy0 >= oy0 + pad and ix1 <= ox1 - pad and iy1 <= oy1 - pad)

# def _rects_overlap(a, b, pad=0.0):
#     ax0, ay0, ax1, ay1 = a
#     bx0, by0, bx1, by1 = b
#     return not (ax1 < bx0 + pad or ax0 > bx1 - pad or ay1 < by0 + pad or ay0 > by1 - pad)

# # -------------------------------
# # Plot (with auto label placement)
# # -------------------------------
# dfF = df.sort_values("tokens_per_sec").reset_index(drop=True)

# fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=160)

# ax.plot(dfF["tokens_per_sec"], dfF["eval_ppl"])
# ax.scatter(dfF["tokens_per_sec"], dfF["eval_ppl"])
# ax.set_yscale("log")

# ax.set_xlabel("Eval tokens/sec")
# ax.set_ylabel("Perplexity (log scale)")
# ax.set_title("Quality–Cost Frontier (Budget sweep, N=256)")
# ax.grid(True, which="both", alpha=0.25)

# # Breathing room (right margin for right-side labels)
# xmin, xmax = float(dfF["tokens_per_sec"].min()), float(dfF["tokens_per_sec"].max())
# ax.set_xlim(xmin - 25, xmax + 150)

# ymin, ymax = float(dfF["eval_ppl"].min()), float(dfF["eval_ppl"].max())
# ax.set_ylim(ymin * 0.85, ymax * 1.25)

# fig.canvas.draw()
# renderer = fig.canvas.get_renderer()

# # polyline in display coords
# pts_data = list(zip(dfF["tokens_per_sec"].astype(float).tolist(), dfF["eval_ppl"].astype(float).tolist()))
# pts_disp = [ax.transData.transform(p) for p in pts_data]
# segments = list(zip(pts_disp[:-1], pts_disp[1:]))

# # axes rect in display coords
# axbb = ax.get_window_extent(renderer=renderer)
# ax_rect = (axbb.x0, axbb.y0, axbb.x1, axbb.y1)

# label_bbox = dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.92)

# # ---- make labels compact (w next to k) ----
# def make_label(k, w):
#     return f"k≈{k:.0f} (w={w})"

# # ---- preferred per-w offsets (tried FIRST) ----
# # These implement your requested tweaks:
# #   - w=256 label lower/closer to its point
# #   - w=8 and w=16 separated vertically to avoid overlap
# preferred = {
#     256: [(18,  2), (18, -6), (26,  2), (26, -6)],   # push down vs earlier
#     128: [(18,  6), (26,  6), (18, -4), (26, -4)],
#     64:  [(18,  8), (26,  8), (18, -6), (26, -6)],
#     32:  [(18,  8), (26,  8), (18, -6), (26, -6)],
#     16:  [(18, -10), (26, -10), (18,  10), (26,  10)],  # force below to split from w=8
#     8:   [(18,  14), (26,  14), (18,  -6), (26,  -6)],  # force above
# }

# # fallback offsets (all right-side)
# fallback = [
#     (18,  10), (26,  14), (34,  18), (44,  22), (56,  26),
#     (18,  -8), (26, -12), (34, -16), (44, -20), (56, -24),
#     (70,  10), (70, -10), (90, 14), (90, -14),
# ]

# placed_rects = []  # store accepted text rects so labels don't overlap each other

# # Place labels: right side + no line intersection + no label overlap
# for (x, y), r in zip(pts_data, dfF.itertuples(index=False)):
#     w = int(getattr(r, "w"))
#     k = float(getattr(r, "keys_per_query_mean"))
#     label = make_label(k, w)

#     # candidate list: preferred first, then fallback
#     cand_list = preferred.get(w, []) + fallback

#     chosen = None
#     for dx, dy in cand_list:
#         txt = ax.annotate(
#             label,
#             xy=(x, y),
#             xytext=(dx, dy),
#             textcoords="offset points",
#             fontsize=9,
#             ha="left",
#             va="bottom",
#             bbox=label_bbox
#         )

#         fig.canvas.draw()
#         renderer = fig.canvas.get_renderer()
#         rect = _bbox_rect_in_display(txt, renderer)

#         inside = _rect_inside_rect(rect, ax_rect, pad=2.0)
#         intersects_line = any(_seg_intersects_rect(p1, p2, rect) for (p1, p2) in segments)
#         overlaps_other_labels = any(_rects_overlap(rect, rr, pad=2.0) for rr in placed_rects)

#         if inside and (not intersects_line) and (not overlaps_other_labels):
#             chosen = txt
#             placed_rects.append(rect)
#             break

#         txt.remove()

#     # last-resort fallback (still right-side; may overlap if everything fails)
#     if chosen is None:
#         txt = ax.annotate(
#             label, xy=(x, y),
#             xytext=(90, 14),
#             textcoords="offset points",
#             fontsize=9, ha="left", va="bottom",
#             bbox=label_bbox
#         )
#         fig.canvas.draw()
#         placed_rects.append(_bbox_rect_in_display(txt, fig.canvas.get_renderer()))

# # Knee annotation (keep as-is; compact)
# rows128 = dfF[dfF["w"] == 128]
# if len(rows128) > 0:
#     r128 = rows128.iloc[0]
#     ax.annotate(
#         "knee",
#         xy=(float(r128["tokens_per_sec"]), float(r128["eval_ppl"])),
#         xytext=(60, 18),
#         textcoords="offset points",
#         fontsize=9,
#         bbox=label_bbox,
#         arrowprops=dict(arrowstyle="->", lw=1.0)
#     )

# fig.tight_layout()
# plt.show()





# # ------------------------------------------
# # 2) Quality vs Effective Budget (log PPL)
# # ------------------------------------------
# plt.figure(figsize=(6.5, 4.5))
# plt.scatter(df_budget["keys_per_query_mean"], df_budget["eval_ppl"])
# plt.plot(df_budget["keys_per_query_mean"], df_budget["eval_ppl"])
# plt.yscale("log")

# for _, r in df_budget.iterrows():
#     w = int(r["w"])
#     plt.annotate(
#         f"w={w}",
#         (r["keys_per_query_mean"], r["eval_ppl"]),
#         textcoords="offset points",
#         xytext=(8, -2),
#         fontsize=8,
#         ha="left",
#         va="bottom"
#     )

# plt.xlabel("Measured keys/query (mean)")
# plt.ylabel("Perplexity (log scale)")
# plt.title("Quality vs Effective Budget (N=256)")
# plt.grid(True, which="both", alpha=0.3)
# plt.tight_layout()
# # plt.savefig("quality_vs_budget_N256.pdf", bbox_inches="tight")  # optional
# plt.show()

# # ------------------------------------------
# # 3) Memory vs Effective Budget (tight y-axis)
# # ------------------------------------------
# plt.figure(figsize=(6.5, 4.5))
# plt.scatter(df_budget["keys_per_query_mean"], df_budget["peak_alloc_gb_eval"])

# for _, r in df_budget.iterrows():
#     w = int(r["w"])
#     plt.annotate(
#         f"w={w}",
#         (r["keys_per_query_mean"], r["peak_alloc_gb_eval"]),
#         textcoords="offset points",
#         xytext=(8, -2),
#         fontsize=8,
#         ha="left",
#         va="bottom"
#     )

# ymin = float(df_budget["peak_alloc_gb_eval"].min())
# ymax = float(df_budget["peak_alloc_gb_eval"].max())
# pad = max(0.01, 0.1 * (ymax - ymin))
# plt.ylim(ymin - pad, ymax + pad)

# plt.xlabel("Measured keys/query (mean)")
# plt.ylabel("Peak alloc during eval (GB)")
# plt.title("Memory vs Effective Budget (N=256)")
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# # plt.savefig("memory_vs_budget_N256.pdf", bbox_inches="tight")  # optional
# plt.show()